In [1]:
# ============================================================
# PROJECT 9 — CELL 0
# NEW-COLAB RUNTIME BOOTSTRAP AND COMPLETION-STATE VALIDATION
#
# This cell:
# - mounts Google Drive
# - validates Projects 1–8 as COMPLETE_AND_FROZEN
# - validates Project 8's permanent checkpoint
# - verifies the TCP-CI archive MD5
# - restores the dataset into /content when required
# - writes a Project 9 runtime-bootstrap record
#
# It does NOT modify:
# - the completion registry
# - Projects 1–8
# - any frozen result package
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import tarfile

import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

THESIS_TOTAL_PROJECTS = 25
EXPECTED_COMPLETED_PROJECTS = list(range(1, 9))
EXPECTED_COMPLETION_STATUS = "COMPLETE_AND_FROZEN"

PROJECT8_NAME = "optimatika@ojAlgo"
PROJECT8_SLUG = "optimatika__ojAlgo"

EXPECTED_PROJECT8_RAW_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_PROJECT8_PACKAGE_SHA256 = (
    "1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607"
)

EXPECTED_PROJECT8_STEP11D_STATUS = (
    "PASS_PROJECT_8_COMPLETION_REGISTRY_UPDATED_AND_FINAL_FREEZE_RECORDED"
)

EXPECTED_ARCHIVE_MD5 = (
    "728804085c757ff5357aa165b4b6384f"
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT8_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_08_selection_checkpoint.json"
)

PROJECT8_STEP11D_STATUS_PATH = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT8_SLUG
    / "ojalgo_step11d_status.json"
)

DATA_ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

PROJECT9_SELECTION_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_09_selection"
)

BOOTSTRAP_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_runtime_bootstrap.json"
)


print("=" * 100)
print("=== PROJECT 9 NEW-NOTEBOOK RUNTIME BOOTSTRAP ===")
print("=" * 100)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(path, algorithm, chunk_size=8 * 1024 * 1024):
    digest = hashlib.new(algorithm)

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def safe_extract_tar(archive_path, destination):
    archive_path = Path(archive_path)
    destination = Path(destination).resolve()

    with tarfile.open(
        archive_path,
        mode="r:*",
    ) as archive:

        for member in archive.getmembers():
            member_destination = (
                destination
                / member.name
            ).resolve()

            try:
                member_destination.relative_to(
                    destination
                )

            except ValueError:
                raise RuntimeError(
                    "Unsafe archive path detected:\n"
                    f"{member.name}"
                )

        try:
            archive.extractall(
                destination,
                filter="data",
            )

        except TypeError:
            archive.extractall(
                destination
            )


def locate_dataset_root():
    direct_candidates = [
        Path(
            "/content/TCP-CI-main-dataset/datasets"
        ),
        Path(
            "/content/TCP-CI-main-dataset-main/datasets"
        ),
        Path(
            "/content/TCP-CI-main/datasets"
        ),
    ]

    for candidate in direct_candidates:
        if candidate.exists() and candidate.is_dir():
            return candidate

    discovered = []

    for candidate in Path("/content").rglob("datasets"):
        if not candidate.is_dir():
            continue

        parent_text = str(
            candidate.parent
        ).lower()

        if (
            "tcp-ci" in parent_text
            or "tcp_ci" in parent_text
        ):
            discovered.append(candidate)

    if not discovered:
        return None

    return sorted(
        discovered,
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )[0]


# ------------------------------------------------------------
# 4. VALIDATE REQUIRED PERMANENT FILES
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    PROJECT8_CHECKPOINT_PATH,
    PROJECT8_STEP11D_STATUS_PATH,
    DATA_ARCHIVE_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required permanent files are missing:\n"
        + "\n".join(missing_paths)
    )


# ------------------------------------------------------------
# 5. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}

missing_registry_columns = (
    required_registry_columns
    - set(registry.columns)
)

if missing_registry_columns:
    raise RuntimeError(
        "Registry is missing required columns:\n"
        f"{sorted(missing_registry_columns)}"
    )

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if len(registry) != 8:
    raise AssertionError(
        "Completion registry must contain exactly eight rows.\n"
        f"Actual rows: {len(registry)}"
    )

if set(project_numbers) != set(
    EXPECTED_COMPLETED_PROJECTS
):
    raise AssertionError(
        "Registry does not contain exactly Projects 1–8.\n"
        f"Actual: {sorted(project_numbers.tolist())}"
    )

if project_numbers.duplicated().any():
    raise AssertionError(
        "Duplicate project numbers exist in the registry."
    )

if not registry["Status"].eq(
    EXPECTED_COMPLETION_STATUS
).all():
    raise AssertionError(
        "Not all Projects 1–8 are COMPLETE_AND_FROZEN."
    )

project8_registry = registry[
    project_numbers.eq(8)
]

if len(project8_registry) != 1:
    raise AssertionError(
        "Registry must contain exactly one Project 8 row."
    )

project8_registry_row = project8_registry.iloc[0]

if project8_registry_row["Project"] != PROJECT8_NAME:
    raise AssertionError(
        "Project 8 name differs in the registry."
    )

if project8_registry_row["ProjectSlug"] != PROJECT8_SLUG:
    raise AssertionError(
        "Project 8 slug differs in the registry."
    )


# ------------------------------------------------------------
# 6. VALIDATE PROJECT 8 CHECKPOINT AND STATUS
# ------------------------------------------------------------

project8_checkpoint = json.loads(
    PROJECT8_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

project8_step11d_status = json.loads(
    PROJECT8_STEP11D_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    int(project8_checkpoint.get("ProjectNumber", -1))
    != 8
):
    raise AssertionError(
        "Project 8 checkpoint project number differs."
    )

if (
    project8_checkpoint.get("Project")
    != PROJECT8_NAME
):
    raise AssertionError(
        "Project 8 checkpoint identity differs."
    )

if (
    project8_checkpoint.get("Status")
    != EXPECTED_COMPLETION_STATUS
):
    raise AssertionError(
        "Project 8 checkpoint is not COMPLETE_AND_FROZEN."
    )

if (
    project8_checkpoint.get("RawAuditRootSHA256")
    != EXPECTED_PROJECT8_RAW_SHA256
):
    raise AssertionError(
        "Project 8 raw freeze SHA-256 differs."
    )

if (
    project8_checkpoint.get(
        "FrozenFinalPackageRootSHA256"
    )
    != EXPECTED_PROJECT8_PACKAGE_SHA256
):
    raise AssertionError(
        "Project 8 package freeze SHA-256 differs."
    )

if (
    project8_step11d_status.get("Status")
    != EXPECTED_PROJECT8_STEP11D_STATUS
):
    raise AssertionError(
        "Project 8 Step 11D status differs."
    )


# ------------------------------------------------------------
# 7. VERIFY ARCHIVE AND RESTORE DATASET
# ------------------------------------------------------------

archive_md5 = calculate_hash(
    DATA_ARCHIVE_PATH,
    "md5",
)

if archive_md5 != EXPECTED_ARCHIVE_MD5:
    raise AssertionError(
        "TCP-CI archive MD5 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_MD5}\n"
        f"Actual:   {archive_md5}"
    )

dataset_root = locate_dataset_root()

dataset_extracted_now = False

if dataset_root is None:
    print("\nRuntime dataset not found.")
    print("Extracting TCP-CI archive into /content...")

    safe_extract_tar(
        DATA_ARCHIVE_PATH,
        Path("/content"),
    )

    dataset_extracted_now = True
    dataset_root = locate_dataset_root()

if dataset_root is None:
    raise FileNotFoundError(
        "TCP-CI dataset root could not be located "
        "after archive extraction."
    )


# ------------------------------------------------------------
# 8. VALIDATE THESIS SCOPE
# ------------------------------------------------------------

project_directories = sorted([
    path
    for path in dataset_root.iterdir()
    if path.is_dir()
])

if len(project_directories) != THESIS_TOTAL_PROJECTS:
    raise AssertionError(
        "TCP-CI project count differs from the frozen "
        "25-project thesis scope.\n"
        f"Expected: {THESIS_TOTAL_PROJECTS}\n"
        f"Actual:   {len(project_directories)}\n"
        f"Dataset root: {dataset_root}"
    )

completed_projects = len(registry)
remaining_projects = (
    THESIS_TOTAL_PROJECTS
    - completed_projects
)

if remaining_projects != 17:
    raise AssertionError(
        "Remaining project count differs.\n"
        f"Expected: 17\n"
        f"Actual:   {remaining_projects}"
    )


# ------------------------------------------------------------
# 9. WRITE BOOTSTRAP STATUS
# ------------------------------------------------------------

bootstrap_payload = {
    "Status":
        "PASS_PROJECT_9_NEW_NOTEBOOK_RUNTIME_BOOTSTRAP",

    "DatasetRoot":
        str(dataset_root),

    "DatasetExtractedNow":
        dataset_extracted_now,

    "ArchivePath":
        str(DATA_ARCHIVE_PATH),

    "ArchiveMD5":
        archive_md5,

    "ThesisTotalProjects":
        THESIS_TOTAL_PROJECTS,

    "CompletedProjects":
        completed_projects,

    "RemainingProjects":
        remaining_projects,

    "CompletedProjectNumbers":
        EXPECTED_COMPLETED_PROJECTS,

    "Project8Status":
        EXPECTED_COMPLETION_STATUS,

    "Project8RawRootSHA256":
        EXPECTED_PROJECT8_RAW_SHA256,

    "Project8PackageRootSHA256":
        EXPECTED_PROJECT8_PACKAGE_SHA256,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    bootstrap_payload,
)


# ------------------------------------------------------------
# 10. RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("=== PROJECT 9 RUNTIME BOOTSTRAP RESULT ===")
print("=" * 100)

print("\nThesis scope:")

print(
    "Total projects:",
    THESIS_TOTAL_PROJECTS,
)

print(
    "Completed:",
    completed_projects,
)

print(
    "Remaining:",
    remaining_projects,
)

print(
    "Current project:",
    PROJECT_NUMBER if "PROJECT_NUMBER" in globals() else 9,
)

print("\nCompletion state:")

print(
    "Registry projects:",
    sorted(project_numbers.tolist()),
)

print(
    "COMPLETE_AND_FROZEN:",
    int(
        registry["Status"]
        .eq(EXPECTED_COMPLETION_STATUS)
        .sum()
    ),
)

print(
    "Project 8 confirmed:",
    True,
)

print("\nRuntime dataset:")

print(
    "Archive MD5:",
    archive_md5,
)

print(
    "Dataset root:",
    dataset_root,
)

print(
    "TCP-CI project directories:",
    len(project_directories),
)

print(
    "Extracted during this cell:",
    dataset_extracted_now,
)

print("\nBootstrap record:")

print(
    BOOTSTRAP_STATUS_PATH
)

print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print(
    "\nSTATUS:",
    "PASS_PROJECT_9_NEW_NOTEBOOK_RUNTIME_BOOTSTRAP",
)

print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== PROJECT 9 NEW-NOTEBOOK RUNTIME BOOTSTRAP ===

Runtime dataset not found.
Extracting TCP-CI archive into /content...


FileNotFoundError: TCP-CI dataset root could not be located after archive extraction.

In [2]:
# ============================================================
# PROJECT 9 — CELL 0 V2
# NEW-COLAB RUNTIME BOOTSTRAP AND ROBUST DATASET RESTORE
#
# Fix:
# - does not assume the archive contains a folder named
#   exactly "datasets"
# - searches for the directory containing the 25 TCP-CI
#   project folders
# - extracts into a dedicated runtime-only directory
#
# This cell does NOT modify:
# - completion registry
# - Projects 1–8
# - frozen result packages
# - raw experiment outputs
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

import hashlib
import json
import os
import shutil
import tarfile

import pandas as pd


# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():

    drive.mount(
        "/content/drive"
    )

else:

    print(
        "Google Drive is already mounted."
    )


# ------------------------------------------------------------
# 2. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

THESIS_TOTAL_PROJECTS = 25

EXPECTED_COMPLETED_PROJECTS = list(
    range(1, 9)
)

EXPECTED_COMPLETION_STATUS = (
    "COMPLETE_AND_FROZEN"
)

PROJECT8_NAME = (
    "optimatika@ojAlgo"
)

PROJECT8_SLUG = (
    "optimatika__ojAlgo"
)

EXPECTED_PROJECT8_RAW_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_PROJECT8_PACKAGE_SHA256 = (
    "1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607"
)

EXPECTED_PROJECT8_STEP11D_STATUS = (
    "PASS_PROJECT_8_COMPLETION_REGISTRY_UPDATED_AND_FINAL_FREEZE_RECORDED"
)

EXPECTED_ARCHIVE_MD5 = (
    "728804085c757ff5357aa165b4b6384f"
)

BOOTSTRAP_PASS_STATUS = (
    "PASS_PROJECT_9_NEW_NOTEBOOK_RUNTIME_BOOTSTRAP"
)


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT8_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_08_selection_checkpoint.json"
)

PROJECT8_STEP11D_STATUS_PATH = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT8_SLUG
    / "ojalgo_step11d_status.json"
)

DATA_ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

# Dedicated runtime-only extraction directory.
# It is safe to recreate after every Colab disconnect.

RUNTIME_RESTORE_DIR = Path(
    "/content/project9_tcpci_runtime"
)

PROJECT9_SELECTION_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_09_selection"
)

BOOTSTRAP_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_runtime_bootstrap.json"
)


print("=" * 104)
print("=== PROJECT 9 CELL 0 V2: ROBUST NEW-NOTEBOOK RUNTIME BOOTSTRAP ===")
print("=" * 104)


# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def safe_extract_tar(
    archive_path,
    destination,
):
    archive_path = Path(
        archive_path
    )

    destination = Path(
        destination
    ).resolve()

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tarfile.open(
        archive_path,
        mode="r:*",
    ) as archive:

        members = archive.getmembers()

        for member in members:

            member_destination = (
                destination
                / member.name
            ).resolve()

            try:

                member_destination.relative_to(
                    destination
                )

            except ValueError:

                raise RuntimeError(
                    "Unsafe archive member detected:\n"
                    f"{member.name}"
                )

        try:

            archive.extractall(
                destination,
                filter="data",
            )

        except TypeError:

            archive.extractall(
                destination
            )


def supported_table_file_count(
    directory,
):
    supported_suffixes = (
        ".csv",
        ".csv.gz",
        ".parquet",
    )

    return sum(
        1
        for path in directory.rglob("*")
        if (
            path.is_file()
            and path.name.lower().endswith(
                supported_suffixes
            )
        )
    )


def project_directories_under(
    candidate_root,
):
    if (
        not candidate_root.exists()
        or not candidate_root.is_dir()
    ):

        return []

    return sorted([
        child
        for child in candidate_root.iterdir()
        if (
            child.is_dir()
            and "@" in child.name
        )
    ])


def is_valid_dataset_root(
    candidate_root,
):
    """
    A valid root must contain exactly the 25 project folders,
    including the already completed Project 8 source folder.
    """

    project_directories = (
        project_directories_under(
            candidate_root
        )
    )

    if len(
        project_directories
    ) != THESIS_TOTAL_PROJECTS:

        return False

    project_names = {
        path.name
        for path in project_directories
    }

    if PROJECT8_NAME not in project_names:

        return False

    # Validate several projects have actual table content.

    sampled_projects = (
        project_directories[:3]
        + project_directories[-3:]
    )

    for project_directory in sampled_projects:

        if supported_table_file_count(
            project_directory
        ) < 3:

            return False

    return True


def find_dataset_roots(
    search_root,
    maximum_depth=7,
):
    """
    Search directories without descending into Google Drive
    or deeply traversing the project table files.
    """

    search_root = Path(
        search_root
    )

    if not search_root.exists():

        return []

    discovered = []

    search_root_depth = len(
        search_root.parts
    )

    for current_root, directories, _ in os.walk(
        search_root
    ):

        current_path = Path(
            current_root
        )

        current_depth = (
            len(
                current_path.parts
            )
            - search_root_depth
        )

        # Do not walk into mounted Drive during runtime search.

        directories[:] = [
            directory
            for directory in directories
            if directory not in {
                "drive",
                ".ipynb_checkpoints",
                "__MACOSX",
            }
        ]

        if current_depth > maximum_depth:

            directories[:] = []
            continue

        if is_valid_dataset_root(
            current_path
        ):

            discovered.append(
                current_path
            )

            # No reason to descend into the 25 projects.
            directories[:] = []

    return sorted(
        set(
            discovered
        ),
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )


def locate_existing_dataset_root():
    """
    Check known historical paths first, then perform a
    constrained runtime search.
    """

    direct_candidates = [
        Path(
            "/content/TCP-CI-main-dataset/datasets"
        ),

        Path(
            "/content/TCP-CI-main-dataset"
        ),

        Path(
            "/content/TCP-CI-main-dataset-main/datasets"
        ),

        Path(
            "/content/TCP-CI-main-dataset-main"
        ),

        Path(
            "/content/TCP-CI-main/datasets"
        ),

        Path(
            "/content/TCP-CI-main"
        ),

        RUNTIME_RESTORE_DIR,

        RUNTIME_RESTORE_DIR
        / "datasets",
    ]

    for candidate in direct_candidates:

        if is_valid_dataset_root(
            candidate
        ):

            return candidate

    discovered = find_dataset_roots(
        Path(
            "/content"
        ),
        maximum_depth=6,
    )

    if discovered:

        return discovered[0]

    return None


def archive_structure_preview(
    archive_path,
    limit=40,
):
    """
    Return a compact preview for diagnostics without
    extracting the archive.
    """

    with tarfile.open(
        archive_path,
        mode="r:*",
    ) as archive:

        member_names = [
            member.name
            for member in archive.getmembers()
            if member.name
        ]

    top_prefixes = []

    for member_name in member_names:

        parts = Path(
            member_name
        ).parts

        if not parts:
            continue

        prefix = "/".join(
            parts[:3]
        )

        if prefix not in top_prefixes:

            top_prefixes.append(
                prefix
            )

        if len(
            top_prefixes
        ) >= limit:

            break

    return {
        "MemberCount":
            len(
                member_names
            ),

        "Preview":
            top_prefixes,
    }


# ------------------------------------------------------------
# 5. VALIDATE PERMANENT INPUTS
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    PROJECT8_CHECKPOINT_PATH,
    PROJECT8_STEP11D_STATUS_PATH,
    DATA_ARCHIVE_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required permanent files are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


# ------------------------------------------------------------
# 6. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}

missing_registry_columns = (
    required_registry_columns
    - set(
        registry.columns
    )
)

if missing_registry_columns:

    raise RuntimeError(
        "Completion registry is missing columns:\n"
        f"{sorted(missing_registry_columns)}"
    )

project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)

if len(
    registry
) != 8:

    raise AssertionError(
        "Completion registry must contain eight rows.\n"
        f"Actual: {len(registry)}"
    )

if set(
    project_numbers
) != set(
    EXPECTED_COMPLETED_PROJECTS
):

    raise AssertionError(
        "Registry does not contain exactly Projects 1–8.\n"
        f"Actual: {sorted(project_numbers.tolist())}"
    )

if project_numbers.duplicated().any():

    raise AssertionError(
        "Duplicate project numbers exist in the registry."
    )

if not registry[
    "Status"
].eq(
    EXPECTED_COMPLETION_STATUS
).all():

    raise AssertionError(
        "Projects 1–8 are not all COMPLETE_AND_FROZEN."
    )

project8_registry_rows = registry[
    project_numbers.eq(8)
]

if len(
    project8_registry_rows
) != 1:

    raise AssertionError(
        "Exactly one Project 8 registry row is required."
    )

project8_registry_row = (
    project8_registry_rows.iloc[0]
)

if (
    project8_registry_row[
        "Project"
    ]
    != PROJECT8_NAME
):

    raise AssertionError(
        "Project 8 name differs in the registry."
    )

if (
    project8_registry_row[
        "ProjectSlug"
    ]
    != PROJECT8_SLUG
):

    raise AssertionError(
        "Project 8 slug differs in the registry."
    )


# ------------------------------------------------------------
# 7. VALIDATE PROJECT 8 PERMANENT FREEZE
# ------------------------------------------------------------

project8_checkpoint = json.loads(
    PROJECT8_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

project8_step11d_status = json.loads(
    PROJECT8_STEP11D_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    int(
        project8_checkpoint.get(
            "ProjectNumber",
            -1,
        )
    )
    != 8
):

    raise AssertionError(
        "Project 8 checkpoint number differs."
    )

if (
    project8_checkpoint.get(
        "Project"
    )
    != PROJECT8_NAME
):

    raise AssertionError(
        "Project 8 checkpoint identity differs."
    )

if (
    project8_checkpoint.get(
        "Status"
    )
    != EXPECTED_COMPLETION_STATUS
):

    raise AssertionError(
        "Project 8 checkpoint is not COMPLETE_AND_FROZEN."
    )

if (
    project8_checkpoint.get(
        "RawAuditRootSHA256"
    )
    != EXPECTED_PROJECT8_RAW_SHA256
):

    raise AssertionError(
        "Project 8 raw freeze SHA-256 differs."
    )

if (
    project8_checkpoint.get(
        "FrozenFinalPackageRootSHA256"
    )
    != EXPECTED_PROJECT8_PACKAGE_SHA256
):

    raise AssertionError(
        "Project 8 package freeze SHA-256 differs."
    )

if (
    project8_step11d_status.get(
        "Status"
    )
    != EXPECTED_PROJECT8_STEP11D_STATUS
):

    raise AssertionError(
        "Project 8 Step 11D status differs."
    )


# ------------------------------------------------------------
# 8. VERIFY THE ARCHIVE
# ------------------------------------------------------------

archive_md5 = calculate_hash(
    DATA_ARCHIVE_PATH,
    "md5",
)

if archive_md5 != EXPECTED_ARCHIVE_MD5:

    raise AssertionError(
        "TCP-CI archive MD5 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_MD5}\n"
        f"Actual:   {archive_md5}"
    )

archive_preview = archive_structure_preview(
    DATA_ARCHIVE_PATH
)

print("\nArchive validation:")

print(
    "Archive:",
    DATA_ARCHIVE_PATH,
)

print(
    "Archive MD5:",
    archive_md5,
)

print(
    "Archive members:",
    archive_preview[
        "MemberCount"
    ],
)

print(
    "Archive structure preview:"
)

for item in archive_preview[
    "Preview"
][:15]:

    print(
        " -",
        item,
    )


# ------------------------------------------------------------
# 9. LOCATE OR RESTORE THE RUNTIME DATASET
# ------------------------------------------------------------

dataset_root = (
    locate_existing_dataset_root()
)

dataset_extracted_now = False

if dataset_root is None:

    print(
        "\nNo valid 25-project runtime dataset was found."
    )

    print(
        "Extracting archive into dedicated runtime directory:"
    )

    print(
        RUNTIME_RESTORE_DIR
    )

    if RUNTIME_RESTORE_DIR.exists():

        shutil.rmtree(
            RUNTIME_RESTORE_DIR
        )

    RUNTIME_RESTORE_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

    safe_extract_tar(
        archive_path=(
            DATA_ARCHIVE_PATH
        ),

        destination=(
            RUNTIME_RESTORE_DIR
        ),
    )

    dataset_extracted_now = True

    discovered_roots = find_dataset_roots(
        RUNTIME_RESTORE_DIR,
        maximum_depth=8,
    )

    if discovered_roots:

        dataset_root = (
            discovered_roots[0]
        )

if dataset_root is None:

    extracted_directories = sorted([
        str(path.relative_to(
            RUNTIME_RESTORE_DIR
        ))
        for path in (
            RUNTIME_RESTORE_DIR.rglob("*")
            if RUNTIME_RESTORE_DIR.exists()
            else []
        )
        if (
            path.is_dir()
            and len(
                path.relative_to(
                    RUNTIME_RESTORE_DIR
                ).parts
            ) <= 4
        )
    ])[:100]

    print(
        "\nRuntime extraction directory preview:"
    )

    for item in extracted_directories:

        print(
            " -",
            item,
        )

    raise FileNotFoundError(
        "The archive was extracted, but no directory "
        "containing exactly 25 TCP-CI project folders "
        "could be identified.\n"
        "The archive and extraction previews above can "
        "be used for precise diagnosis."
    )


# ------------------------------------------------------------
# 10. VALIDATE THE LOCATED DATASET ROOT
# ------------------------------------------------------------

project_directories = (
    project_directories_under(
        dataset_root
    )
)

project_names = [
    path.name
    for path in project_directories
]

if len(
    project_directories
) != THESIS_TOTAL_PROJECTS:

    raise AssertionError(
        "Located dataset root does not contain exactly "
        "25 project folders.\n"
        f"Root: {dataset_root}\n"
        f"Actual: {len(project_directories)}"
    )

if PROJECT8_NAME not in set(
    project_names
):

    raise AssertionError(
        "Located dataset root does not contain "
        f"{PROJECT8_NAME}."
    )

completed_projects = len(
    registry
)

remaining_projects = (
    THESIS_TOTAL_PROJECTS
    - completed_projects
)

if remaining_projects != 17:

    raise AssertionError(
        "Remaining project count differs.\n"
        f"Expected: 17\n"
        f"Actual: {remaining_projects}"
    )


# ------------------------------------------------------------
# 11. WRITE SUCCESSFUL BOOTSTRAP RECORD
# ------------------------------------------------------------

bootstrap_payload = {
    "Status":
        BOOTSTRAP_PASS_STATUS,

    "ProjectNumber":
        PROJECT_NUMBER,

    "DatasetRoot":
        str(
            dataset_root
        ),

    "RuntimeRestoreDirectory":
        str(
            RUNTIME_RESTORE_DIR
        ),

    "DatasetExtractedNow":
        dataset_extracted_now,

    "ArchivePath":
        str(
            DATA_ARCHIVE_PATH
        ),

    "ArchiveMD5":
        archive_md5,

    "ArchiveMemberCount":
        archive_preview[
            "MemberCount"
        ],

    "ThesisTotalProjects":
        THESIS_TOTAL_PROJECTS,

    "CompletedProjects":
        completed_projects,

    "RemainingProjects":
        remaining_projects,

    "CompletedProjectNumbers":
        EXPECTED_COMPLETED_PROJECTS,

    "ProjectDirectories":
        project_names,

    "Project8Status":
        EXPECTED_COMPLETION_STATUS,

    "Project8RawRootSHA256":
        EXPECTED_PROJECT8_RAW_SHA256,

    "Project8PackageRootSHA256":
        EXPECTED_PROJECT8_PACKAGE_SHA256,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    bootstrap_payload,
)


# ------------------------------------------------------------
# 12. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 104)
print("=== PROJECT 9 CELL 0 V2 RESULT ===")
print("=" * 104)

print("\nThesis scope:")

print(
    "Total projects:",
    THESIS_TOTAL_PROJECTS,
)

print(
    "Completed:",
    completed_projects,
)

print(
    "Remaining:",
    remaining_projects,
)

print(
    "Current project:",
    PROJECT_NUMBER,
)

print("\nCompletion state:")

print(
    "Registry projects:",
    sorted(
        project_numbers.tolist()
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    int(
        registry[
            "Status"
        ].eq(
            EXPECTED_COMPLETION_STATUS
        ).sum()
    ),
)

print(
    "Project 8 confirmed:",
    True,
)

print("\nRuntime dataset:")

print(
    "Dataset root:",
    dataset_root,
)

print(
    "TCP-CI project directories:",
    len(
        project_directories
    ),
)

print(
    "Extracted during this cell:",
    dataset_extracted_now,
)

print(
    "First five projects:",
    project_names[:5],
)

print(
    "Last five projects:",
    project_names[-5:],
)

print("\nBootstrap record:")

print(
    BOOTSTRAP_STATUS_PATH
)

print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print(
    "\nSTATUS:",
    BOOTSTRAP_PASS_STATUS,
)

print("=" * 104)

Google Drive is already mounted.
=== PROJECT 9 CELL 0 V2: ROBUST NEW-NOTEBOOK RUNTIME BOOTSTRAP ===

Archive validation:
Archive: /content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz
Archive MD5: 728804085c757ff5357aa165b4b6384f
Archive members: 176
Archive structure preview:
 - datasets
 - datasets/yamcs@Yamcs
 - datasets/yamcs@Yamcs/builds.csv
 - datasets/yamcs@Yamcs/id_map.csv
 - datasets/yamcs@Yamcs/entity_change_history.csv
 - datasets/yamcs@Yamcs/dataset.csv
 - datasets/yamcs@Yamcs/exe.csv
 - datasets/yamcs@Yamcs/contributors.csv
 - datasets/jcabi@jcabi-github
 - datasets/jcabi@jcabi-github/builds.csv
 - datasets/jcabi@jcabi-github/id_map.csv
 - datasets/jcabi@jcabi-github/entity_change_history.csv
 - datasets/jcabi@jcabi-github/dataset.csv
 - datasets/jcabi@jcabi-github/exe.csv
 - datasets/jcabi@jcabi-github/contributors.csv


=== PROJECT 9 CELL 0 V2 RESULT ===

Thesis scope:
Total projects: 25
Completed: 8
Remaining: 17
Current project: 9

Completion sta

In [3]:
# ============================================================
# PROJECT 9 — STEP 1A
# REMAINING-PROJECT DISCOVERY, ELIGIBILITY AND RANKING
#
# This cell:
# - reads the successful bootstrap record
# - excludes Projects 1–8
# - scans the 17 unfinished TCP-CI projects
# - uses canonical timestamp-ascending / Build-ID-descending
#   chronology
# - applies the fixed chronological 75%/25% partition
# - measures raw and model-ready failure support
# - ranks eligible candidates
# - saves progress after every project
#
# This is provisional selection only.
#
# It does NOT:
# - freeze Project 9's identity
# - modify the completion registry
# - modify Projects 1–8
# - train any models
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS AND PATHS
# ------------------------------------------------------------

PROJECT_NUMBER = 9
THESIS_TOTAL_PROJECTS = 25
EXPECTED_COMPLETED_PROJECTS = list(range(1, 9))
EXPECTED_COMPLETION_STATUS = "COMPLETE_AND_FROZEN"

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT9_SELECTION_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_09_selection"
)

BOOTSTRAP_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_runtime_bootstrap.json"
)

PROGRESS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_scan_progress.csv"
)

CANDIDATE_INVENTORY_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_inventory.csv"
)

ELIGIBLE_CANDIDATES_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_eligible_candidates_ranked.csv"
)

PROVISIONAL_SELECTION_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_provisional_selection.json"
)

STEP1A_REPORT_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_step1a_status.json"
)

CSV_CHUNK_SIZE = 250_000


print("=" * 104)
print("=== PROJECT 9 STEP 1A: CANDIDATE DISCOVERY AND RANKING ===")
print("=" * 104)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def calculate_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def normalise_name(value):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def project_slug(project_name):
    return str(project_name).replace(
        "@",
        "__",
    )


def strip_table_suffix(file_name):
    lower_name = str(file_name).lower()

    for suffix in [
        ".csv.gz",
        ".parquet",
        ".csv",
        ".gz",
    ]:
        if lower_name.endswith(suffix):
            return lower_name[:-len(suffix)]

    return lower_name


def table_files(project_directory):
    supported_suffixes = (
        ".csv",
        ".csv.gz",
        ".parquet",
    )

    return sorted([
        path
        for path in project_directory.rglob("*")
        if (
            path.is_file()
            and path.name.lower().endswith(
                supported_suffixes
            )
        )
    ])


def read_header(path):
    path = Path(path)

    if path.name.lower().endswith(
        ".parquet"
    ):
        return list(
            pd.read_parquet(
                path
            ).columns
        )

    return list(
        pd.read_csv(
            path,
            nrows=0,
        ).columns
    )


def detect_column(
    columns,
    candidates,
    required=True,
):
    lookup = {
        normalise_name(column):
            column
        for column in columns
    }

    for candidate in candidates:
        key = normalise_name(candidate)

        if key in lookup:
            return lookup[key]

    if required:
        raise RuntimeError(
            "Required column not found.\n"
            f"Candidates: {candidates}\n"
            f"Available: {columns}"
        )

    return None


def canonical_identifier(series):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if numeric.notna().mean() >= 0.95:
        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:
            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def verdict_failure_mask(series):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if numeric.notna().mean() >= 0.95:
        return numeric.fillna(0).ne(0)

    text = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    pass_values = {
        "",
        "0",
        "pass",
        "passed",
        "success",
        "successful",
        "ok",
    }

    failure_values = {
        "1",
        "2",
        "fail",
        "failed",
        "failure",
        "error",
        "errored",
    }

    unknown_mask = ~text.isin(
        pass_values | failure_values
    )

    if unknown_mask.any():
        raise RuntimeError(
            "Unrecognised verdict values: "
            f"{sorted(text[unknown_mask].unique())[:20]}"
        )

    return text.isin(failure_values)


def read_columns(path, columns):
    path = Path(path)

    if path.name.lower().endswith(
        ".parquet"
    ):
        yield pd.read_parquet(
            path,
            columns=columns,
        )

        return

    for chunk in pd.read_csv(
        path,
        usecols=columns,
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ):
        yield chunk


def resolve_project_files(project_directory):
    files = table_files(
        project_directory
    )

    if not files:
        raise RuntimeError(
            "No supported table files found."
        )

    metadata = []

    for path in files:
        columns = read_header(path)

        metadata.append({
            "Path":
                path,

            "Base":
                normalise_name(
                    strip_table_suffix(
                        path.name
                    )
                ),

            "Columns":
                columns,

            "ColumnCount":
                len(columns),

            "NormalisedColumns":
                {
                    normalise_name(column)
                    for column in columns
                },
        })

    assignments = {
        "BuildsPath":
            None,

        "ExecutionsPath":
            None,

        "DatasetPath":
            None,

        "IDMapPath":
            None,

        "EntityHistoryPath":
            None,
    }

    exact_aliases = {
        "BuildsPath": {
            "build",
            "builds",
            "buildhistory",
        },

        "ExecutionsPath": {
            "exe",
            "execution",
            "executions",
            "testexecution",
            "testexecutions",
        },

        "DatasetPath": {
            "dataset",
            "mldataset",
            "modeldataset",
        },
    }

    used_paths = set()

    for role, aliases in exact_aliases.items():
        matches = [
            item
            for item in metadata
            if item["Base"] in aliases
        ]

        if matches:
            selected = sorted(
                matches,
                key=lambda item: (
                    len(item["Path"].name),
                    item["Path"].name.lower(),
                ),
            )[0]

            assignments[role] = selected["Path"]
            used_paths.add(selected["Path"])

    id_map_matches = [
        item
        for item in metadata
        if (
            "idmap" in item["Base"]
            or item["Base"] in {
                "testmap",
                "mapping",
            }
        )
    ]

    if id_map_matches:
        selected = sorted(
            id_map_matches,
            key=lambda item: (
                len(item["Path"].name),
                item["Path"].name.lower(),
            ),
        )[0]

        assignments["IDMapPath"] = selected["Path"]
        used_paths.add(selected["Path"])

    entity_matches = [
        item
        for item in metadata
        if (
            "entityhistory" in item["Base"]
            or (
                "entity" in item["Base"]
                and "history" in item["Base"]
            )
        )
    ]

    if entity_matches:
        selected = sorted(
            entity_matches,
            key=lambda item: (
                len(item["Path"].name),
                item["Path"].name.lower(),
            ),
        )[0]

        assignments[
            "EntityHistoryPath"
        ] = selected["Path"]

        used_paths.add(selected["Path"])

    def score(item, role):
        columns = item["Columns"]
        normalised_columns = (
            item["NormalisedColumns"]
        )

        has_build = any(
            name in normalised_columns
            for name in {
                "build",
                "buildid",
            }
        )

        has_test = any(
            name in normalised_columns
            for name in {
                "test",
                "testid",
                "testname",
            }
        )

        has_verdict = any(
            name in normalised_columns
            for name in {
                "verdict",
                "result",
                "outcome",
                "label",
            }
        )

        has_started_at = any(
            name in normalised_columns
            for name in {
                "startedat",
                "starttime",
                "buildstartedat",
                "timestamp",
            }
        )

        if role == "BuildsPath":
            return (
                20 * has_build
                + 30 * has_started_at
                - 10 * has_verdict
                - 5 * has_test
            )

        if role == "ExecutionsPath":
            return (
                20 * has_build
                + 20 * has_test
                + 25 * has_verdict
                + 10 * (
                    item["ColumnCount"] <= 20
                )
                - 10 * (
                    item["ColumnCount"] > 40
                )
            )

        if role == "DatasetPath":
            return (
                20 * has_build
                + 20 * has_test
                + 25 * has_verdict
                + min(
                    item["ColumnCount"],
                    100,
                )
                + 20 * (
                    item["ColumnCount"] > 20
                )
            )

        if role == "IDMapPath":
            return (
                30 * (
                    "idmap" in item["Base"]
                )
                + 10 * has_test
                + 10 * (
                    item["ColumnCount"] <= 5
                )
            )

        if role == "EntityHistoryPath":
            return (
                30 * (
                    "entityhistory"
                    in item["Base"]
                )
                + 10 * (
                    "entity" in item["Base"]
                )
                + 10 * (
                    "history" in item["Base"]
                )
            )

        return 0

    for role in assignments:
        if assignments[role] is not None:
            continue

        available = [
            item
            for item in metadata
            if item["Path"] not in used_paths
        ]

        scored = sorted(
            [
                (
                    score(item, role),
                    item,
                )
                for item in available
            ],
            key=lambda pair: (
                pair[0],
                -len(pair[1]["Path"].name),
                pair[1]["Path"].name.lower(),
            ),
            reverse=True,
        )

        if scored and scored[0][0] > 0:
            assignments[role] = (
                scored[0][1]["Path"]
            )

            used_paths.add(
                scored[0][1]["Path"]
            )

    return assignments


def partition_counts(
    path,
    build_column,
    verdict_column,
    training_builds,
    evaluation_builds,
):
    result = {
        "Rows":
            0,

        "TrainingRows":
            0,

        "EvaluationRows":
            0,

        "UnlinkedRows":
            0,

        "TrainingFailures":
            0,

        "TrainingPasses":
            0,

        "EvaluationFailures":
            0,

        "EvaluationPasses":
            0,

        "FailingTrainingBuilds":
            set(),

        "FailingEvaluationBuilds":
            set(),
    }

    for chunk in read_columns(
        path,
        [
            build_column,
            verdict_column,
        ],
    ):
        build_keys = canonical_identifier(
            chunk[build_column]
        )

        failures = verdict_failure_mask(
            chunk[verdict_column]
        )

        training_mask = build_keys.isin(
            training_builds
        )

        evaluation_mask = build_keys.isin(
            evaluation_builds
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        result["Rows"] += len(chunk)

        result["TrainingRows"] += int(
            training_mask.sum()
        )

        result["EvaluationRows"] += int(
            evaluation_mask.sum()
        )

        result["UnlinkedRows"] += int(
            (~linked_mask).sum()
        )

        result["TrainingFailures"] += int(
            (
                training_mask
                & failures
            ).sum()
        )

        result["TrainingPasses"] += int(
            (
                training_mask
                & ~failures
            ).sum()
        )

        result["EvaluationFailures"] += int(
            (
                evaluation_mask
                & failures
            ).sum()
        )

        result["EvaluationPasses"] += int(
            (
                evaluation_mask
                & ~failures
            ).sum()
        )

        result[
            "FailingTrainingBuilds"
        ].update(
            build_keys[
                training_mask
                & failures
            ].tolist()
        )

        result[
            "FailingEvaluationBuilds"
        ].update(
            build_keys[
                evaluation_mask
                & failures
            ].tolist()
        )

    result["FailingTrainingBuilds"] = len(
        result["FailingTrainingBuilds"]
    )

    result["FailingEvaluationBuilds"] = len(
        result["FailingEvaluationBuilds"]
    )

    return result


def inspect_candidate(
    project_directory,
    completed_names,
    completed_slugs,
):
    started = time.perf_counter()

    project_name = project_directory.name
    slug = project_slug(project_name)

    record = {
        "Project":
            project_name,

        "ProjectSlug":
            slug,

        "SourceDirectory":
            str(project_directory),

        "AlreadyCompleted":
            (
                project_name in completed_names
                or slug in completed_slugs
            ),

        "BuildsPath":
            "",

        "ExecutionsPath":
            "",

        "DatasetPath":
            "",

        "IDMapPath":
            "",

        "EntityHistoryPath":
            "",

        "RequiredFilesPresent":
            False,

        "Builds":
            np.nan,

        "TrainingBuilds":
            np.nan,

        "EvaluationBuilds":
            np.nan,

        "RawExecutionRows":
            np.nan,

        "RawTrainingRows":
            np.nan,

        "RawEvaluationRows":
            np.nan,

        "RawUnlinkedRows":
            np.nan,

        "RawTrainFailures":
            np.nan,

        "RawTrainPasses":
            np.nan,

        "RawEvaluationFailures":
            np.nan,

        "RawEvaluationPasses":
            np.nan,

        "RawFailingTrainBuilds":
            np.nan,

        "RawFailingEvaluationBuilds":
            np.nan,

        "ModelReadyRows":
            np.nan,

        "ModelTrainingRows":
            np.nan,

        "ModelEvaluationRows":
            np.nan,

        "ModelUnlinkedRows":
            np.nan,

        "ModelTrainFailures":
            np.nan,

        "ModelTrainPasses":
            np.nan,

        "ModelEvaluationFailures":
            np.nan,

        "ModelEvaluationPasses":
            np.nan,

        "ModelFailingTrainBuilds":
            np.nan,

        "ModelFailingEvaluationBuilds":
            np.nan,

        "ProtocolEligible":
            False,

        "EligibilityReason":
            "",

        "InspectionStatus":
            "PENDING",

        "InspectionError":
            "",

        "ElapsedSeconds":
            np.nan,
    }

    if record["AlreadyCompleted"]:
        record[
            "InspectionStatus"
        ] = "SKIPPED_COMPLETED"

        record[
            "EligibilityReason"
        ] = "Already COMPLETE_AND_FROZEN"

        record[
            "ElapsedSeconds"
        ] = (
            time.perf_counter()
            - started
        )

        return record

    try:
        resolved = resolve_project_files(
            project_directory
        )

        for role, path in resolved.items():
            record[role] = (
                str(path)
                if path is not None
                else ""
            )

        missing_roles = [
            role
            for role, path in resolved.items()
            if path is None
        ]

        if missing_roles:
            record[
                "InspectionStatus"
            ] = "INELIGIBLE"

            record[
                "EligibilityReason"
            ] = (
                "Missing required files: "
                + ", ".join(missing_roles)
            )

            record[
                "ElapsedSeconds"
            ] = (
                time.perf_counter()
                - started
            )

            return record

        record["RequiredFilesPresent"] = True

        builds_path = resolved["BuildsPath"]
        exe_path = resolved["ExecutionsPath"]
        dataset_path = resolved["DatasetPath"]

        builds_columns = read_header(
            builds_path
        )

        build_id_column = detect_column(
            builds_columns,
            [
                "Build",
                "BuildID",
                "BuildId",
                "build_id",
            ],
        )

        started_at_column = detect_column(
            builds_columns,
            [
                "started_at",
                "StartedAt",
                "BuildStartedAt",
                "StartTime",
                "Timestamp",
            ],
        )

        if builds_path.name.lower().endswith(
            ".parquet"
        ):
            builds = pd.read_parquet(
                builds_path,
                columns=[
                    build_id_column,
                    started_at_column,
                ],
            )
        else:
            builds = pd.read_csv(
                builds_path,
                usecols=[
                    build_id_column,
                    started_at_column,
                ],
                low_memory=False,
            )

        builds = builds.rename(
            columns={
                build_id_column:
                    "Build",

                started_at_column:
                    "StartedAt",
            }
        )

        builds["BuildKey"] = (
            canonical_identifier(
                builds["Build"]
            )
        )

        builds["StartedAtParsed"] = (
            pd.to_datetime(
                builds["StartedAt"],
                errors="coerce",
                utc=True,
            )
        )

        timestamp_failures = int(
            builds[
                "StartedAtParsed"
            ].isna().sum()
        )

        if timestamp_failures:
            raise RuntimeError(
                "Unparseable build timestamps: "
                f"{timestamp_failures}"
            )

        builds = builds.drop_duplicates(
            subset=["BuildKey"],
            keep="first",
        )

        numeric_build_ids = pd.to_numeric(
            builds["BuildKey"],
            errors="coerce",
        )

        if numeric_build_ids.notna().all():
            builds["BuildTieOrder"] = (
                numeric_build_ids
            )
        else:
            builds["BuildTieOrder"] = (
                builds["BuildKey"]
                .astype(str)
            )

        builds = (
            builds.sort_values(
                [
                    "StartedAtParsed",
                    "BuildTieOrder",
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        build_count = len(builds)
        training_build_count = int(
            math.floor(
                0.75 * build_count
            )
        )

        evaluation_build_count = (
            build_count
            - training_build_count
        )

        training_build_keys = set(
            builds.iloc[
                :training_build_count
            ]["BuildKey"]
        )

        evaluation_build_keys = set(
            builds.iloc[
                training_build_count:
            ]["BuildKey"]
        )

        record["Builds"] = build_count
        record[
            "TrainingBuilds"
        ] = training_build_count
        record[
            "EvaluationBuilds"
        ] = evaluation_build_count

        exe_columns = read_header(
            exe_path
        )

        exe_build_column = detect_column(
            exe_columns,
            [
                "Build",
                "BuildID",
                "BuildId",
                "build_id",
            ],
        )

        exe_verdict_column = detect_column(
            exe_columns,
            [
                "Verdict",
                "Result",
                "Outcome",
                "Label",
                "Status",
            ],
        )

        raw_counts = partition_counts(
            exe_path,
            exe_build_column,
            exe_verdict_column,
            training_build_keys,
            evaluation_build_keys,
        )

        dataset_columns = read_header(
            dataset_path
        )

        dataset_build_column = detect_column(
            dataset_columns,
            [
                "Build",
                "BuildID",
                "BuildId",
                "build_id",
            ],
        )

        dataset_verdict_column = detect_column(
            dataset_columns,
            [
                "Verdict",
                "Result",
                "Outcome",
                "Label",
                "Status",
            ],
        )

        model_counts = partition_counts(
            dataset_path,
            dataset_build_column,
            dataset_verdict_column,
            training_build_keys,
            evaluation_build_keys,
        )

        record.update({
            "RawExecutionRows":
                raw_counts["Rows"],

            "RawTrainingRows":
                raw_counts["TrainingRows"],

            "RawEvaluationRows":
                raw_counts["EvaluationRows"],

            "RawUnlinkedRows":
                raw_counts["UnlinkedRows"],

            "RawTrainFailures":
                raw_counts["TrainingFailures"],

            "RawTrainPasses":
                raw_counts["TrainingPasses"],

            "RawEvaluationFailures":
                raw_counts[
                    "EvaluationFailures"
                ],

            "RawEvaluationPasses":
                raw_counts[
                    "EvaluationPasses"
                ],

            "RawFailingTrainBuilds":
                raw_counts[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_counts[
                    "FailingEvaluationBuilds"
                ],

            "ModelReadyRows":
                model_counts["Rows"],

            "ModelTrainingRows":
                model_counts["TrainingRows"],

            "ModelEvaluationRows":
                model_counts["EvaluationRows"],

            "ModelUnlinkedRows":
                model_counts["UnlinkedRows"],

            "ModelTrainFailures":
                model_counts["TrainingFailures"],

            "ModelTrainPasses":
                model_counts["TrainingPasses"],

            "ModelEvaluationFailures":
                model_counts[
                    "EvaluationFailures"
                ],

            "ModelEvaluationPasses":
                model_counts[
                    "EvaluationPasses"
                ],

            "ModelFailingTrainBuilds":
                model_counts[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_counts[
                    "FailingEvaluationBuilds"
                ],
        })

        reasons = []

        if build_count < 4:
            reasons.append(
                "Fewer than four chronological builds"
            )

        if training_build_count < 1:
            reasons.append(
                "Empty training-build partition"
            )

        if evaluation_build_count < 1:
            reasons.append(
                "Empty evaluation-build partition"
            )

        if raw_counts["TrainingFailures"] < 1:
            reasons.append(
                "No raw training failures"
            )

        if raw_counts["TrainingPasses"] < 1:
            reasons.append(
                "No raw training passes"
            )

        if raw_counts["EvaluationFailures"] < 1:
            reasons.append(
                "No raw evaluation failures"
            )

        if model_counts["TrainingFailures"] < 1:
            reasons.append(
                "No model-ready training failures"
            )

        if model_counts["TrainingPasses"] < 1:
            reasons.append(
                "No model-ready training passes"
            )

        if model_counts[
            "EvaluationFailures"
        ] < 1:
            reasons.append(
                "No model-ready evaluation failures"
            )

        if model_counts[
            "EvaluationRows"
        ] < 1:
            reasons.append(
                "No model-ready evaluation rows"
            )

        if reasons:
            record[
                "ProtocolEligible"
            ] = False

            record[
                "EligibilityReason"
            ] = "; ".join(reasons)

            record[
                "InspectionStatus"
            ] = "INELIGIBLE"

        else:
            record[
                "ProtocolEligible"
            ] = True

            record[
                "EligibilityReason"
            ] = "Essential protocol requirements satisfied"

            record[
                "InspectionStatus"
            ] = "ELIGIBLE"

    except Exception as error:
        record[
            "InspectionStatus"
        ] = "ERROR"

        record[
            "EligibilityReason"
        ] = "Inspection error"

        record[
            "InspectionError"
        ] = (
            f"{type(error).__name__}: {error}"
        )

    record["ElapsedSeconds"] = (
        time.perf_counter()
        - started
    )

    return record


# ------------------------------------------------------------
# 3. VALIDATE BOOTSTRAP AND REGISTRY
# ------------------------------------------------------------

required_paths = [
    BOOTSTRAP_STATUS_PATH,
    REGISTRY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Step 1A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

bootstrap = json.loads(
    BOOTSTRAP_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    bootstrap.get("Status")
    != "PASS_PROJECT_9_NEW_NOTEBOOK_RUNTIME_BOOTSTRAP"
):
    raise AssertionError(
        "Project 9 bootstrap has not passed."
    )

dataset_root = Path(
    bootstrap["DatasetRoot"]
)

if not dataset_root.exists():
    raise FileNotFoundError(
        "Runtime dataset root is missing. "
        "Rerun Cell 0 first."
    )

registry_sha256_before = (
    calculate_sha256(
        REGISTRY_PATH
    )
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if (
    len(registry) != 8
    or set(project_numbers)
    != set(EXPECTED_COMPLETED_PROJECTS)
):
    raise AssertionError(
        "Completion registry does not contain "
        "exactly Projects 1–8."
    )

if not registry["Status"].eq(
    EXPECTED_COMPLETION_STATUS
).all():
    raise AssertionError(
        "Projects 1–8 are not all "
        "COMPLETE_AND_FROZEN."
    )

completed_names = set(
    registry["Project"].astype(str)
)

completed_slugs = set(
    registry["ProjectSlug"].astype(str)
)


# ------------------------------------------------------------
# 4. DISCOVER THE 25 PROJECT DIRECTORIES
# ------------------------------------------------------------

project_directories = sorted([
    path
    for path in dataset_root.iterdir()
    if path.is_dir()
])

if len(project_directories) != THESIS_TOTAL_PROJECTS:
    raise AssertionError(
        "Dataset does not contain the frozen "
        "25-project thesis scope.\n"
        f"Actual: {len(project_directories)}"
    )

remaining_directories = [
    path
    for path in project_directories
    if (
        path.name not in completed_names
        and project_slug(path.name)
        not in completed_slugs
    )
]

if len(remaining_directories) != 17:
    raise AssertionError(
        "Remaining project-directory count differs.\n"
        f"Expected: 17\n"
        f"Actual:   {len(remaining_directories)}"
    )

print("\nCompletion state:")

print(
    "Completed projects:",
    len(registry),
)

print(
    "Remaining project directories:",
    len(remaining_directories),
)

print(
    "Registry SHA-256:",
    registry_sha256_before,
)


# ------------------------------------------------------------
# 5. LOAD RESUMABLE PROGRESS
# ------------------------------------------------------------

PROJECT9_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if PROGRESS_PATH.exists():
    progress = (
        pd.read_csv(
            PROGRESS_PATH,
            low_memory=False,
        )
    )

    completed_scan_names = set(
        progress.loc[
            progress["InspectionStatus"]
            .ne("ERROR"),
            "Project",
        ].astype(str)
    )

    progress_records = (
        progress.to_dict("records")
    )

    print("\nResuming saved scan progress:")

    print(
        "Existing completed candidate scans:",
        len(completed_scan_names),
    )

else:
    completed_scan_names = set()
    progress_records = []

    print("\nNo saved scan progress found.")
    print("Starting the 17-project scan.")


# ------------------------------------------------------------
# 6. INSPECT EACH REMAINING PROJECT
# ------------------------------------------------------------

for index, project_directory in enumerate(
    remaining_directories,
    start=1,
):
    if (
        project_directory.name
        in completed_scan_names
    ):
        print(
            f"[{index:02d}/17] "
            f"Skipping saved result: "
            f"{project_directory.name}"
        )

        continue

    print(
        f"[{index:02d}/17] "
        f"Inspecting: "
        f"{project_directory.name}"
    )

    result = inspect_candidate(
        project_directory,
        completed_names,
        completed_slugs,
    )

    progress_records = [
        row
        for row in progress_records
        if row.get("Project")
        != project_directory.name
    ]

    progress_records.append(result)

    progress_frame = (
        pd.DataFrame(
            progress_records
        )
        .sort_values(
            "Project",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    atomic_write_csv(
        PROGRESS_PATH,
        progress_frame,
    )

    print(
        "    Status:",
        result["InspectionStatus"],
    )

    print(
        "    Builds:",
        result["Builds"],
        "| Model rows:",
        result["ModelReadyRows"],
        "| Model eval failures:",
        result[
            "ModelEvaluationFailures"
        ],
        "| Time:",
        round(
            float(
                result["ElapsedSeconds"]
            ),
            2,
        ),
        "seconds",
    )


# ------------------------------------------------------------
# 7. FINALISE INVENTORY
# ------------------------------------------------------------

candidate_inventory = (
    pd.read_csv(
        PROGRESS_PATH,
        low_memory=False,
    )
    .sort_values(
        "Project",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(candidate_inventory) != 17:
    raise AssertionError(
        "Candidate inventory does not contain "
        "all 17 unfinished projects.\n"
        f"Actual: {len(candidate_inventory)}"
    )

eligible_candidates = (
    candidate_inventory[
        candidate_inventory[
            "ProtocolEligible"
        ].eq(True)
    ]
    .copy()
)

if eligible_candidates.empty:
    display(candidate_inventory)

    raise RuntimeError(
        "No protocol-eligible Project 9 "
        "candidate was found."
    )

eligible_candidates[
    "MinimumEvaluationFailureSupport"
] = eligible_candidates[
    [
        "RawEvaluationFailures",
        "ModelEvaluationFailures",
    ]
].min(axis=1)

eligible_candidates[
    "MinimumTrainingFailureSupport"
] = eligible_candidates[
    [
        "RawTrainFailures",
        "ModelTrainFailures",
    ]
].min(axis=1)

eligible_candidates[
    "RuntimeCohortRows"
] = eligible_candidates[
    "ModelReadyRows"
]

eligible_candidates = (
    eligible_candidates
    .sort_values(
        [
            "MinimumEvaluationFailureSupport",
            "ModelFailingEvaluationBuilds",
            "MinimumTrainingFailureSupport",
            "ModelFailingTrainBuilds",
            "RuntimeCohortRows",
            "Project",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(eligible_candidates) + 1,
    ),
)

provisional_candidate = (
    eligible_candidates.iloc[0]
)


# ------------------------------------------------------------
# 8. REGISTRY IMMUTABILITY
# ------------------------------------------------------------

registry_sha256_after = (
    calculate_sha256(
        REGISTRY_PATH
    )
)

if (
    registry_sha256_after
    != registry_sha256_before
):
    raise AssertionError(
        "Completion registry changed during "
        "Project 9 candidate discovery."
    )


# ------------------------------------------------------------
# 9. WRITE FINAL STEP 1A OUTPUTS
# ------------------------------------------------------------

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    candidate_inventory,
)

atomic_write_csv(
    ELIGIBLE_CANDIDATES_PATH,
    eligible_candidates,
)

provisional_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionStatus":
        "PROVISIONAL_NOT_FROZEN",

    "CandidateRank":
        int(
            provisional_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        provisional_candidate[
            "Project"
        ],

    "ProjectSlug":
        provisional_candidate[
            "ProjectSlug"
        ],

    "SourceDirectory":
        provisional_candidate[
            "SourceDirectory"
        ],

    "BuildsPath":
        provisional_candidate[
            "BuildsPath"
        ],

    "ExecutionsPath":
        provisional_candidate[
            "ExecutionsPath"
        ],

    "DatasetPath":
        provisional_candidate[
            "DatasetPath"
        ],

    "IDMapPath":
        provisional_candidate[
            "IDMapPath"
        ],

    "EntityHistoryPath":
        provisional_candidate[
            "EntityHistoryPath"
        ],

    "Metrics": {
        "Builds":
            int(
                provisional_candidate[
                    "Builds"
                ]
            ),

        "TrainingBuilds":
            int(
                provisional_candidate[
                    "TrainingBuilds"
                ]
            ),

        "EvaluationBuilds":
            int(
                provisional_candidate[
                    "EvaluationBuilds"
                ]
            ),

        "RawExecutionRows":
            int(
                provisional_candidate[
                    "RawExecutionRows"
                ]
            ),

        "ModelReadyRows":
            int(
                provisional_candidate[
                    "ModelReadyRows"
                ]
            ),

        "RawTrainFailures":
            int(
                provisional_candidate[
                    "RawTrainFailures"
                ]
            ),

        "RawEvaluationFailures":
            int(
                provisional_candidate[
                    "RawEvaluationFailures"
                ]
            ),

        "ModelTrainFailures":
            int(
                provisional_candidate[
                    "ModelTrainFailures"
                ]
            ),

        "ModelEvaluationFailures":
            int(
                provisional_candidate[
                    "ModelEvaluationFailures"
                ]
            ),

        "ModelFailingEvaluationBuilds":
            int(
                provisional_candidate[
                    "ModelFailingEvaluationBuilds"
                ]
            ),
    },

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "GeneratedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_payload,
)

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        "PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE",

    "DatasetRoot":
        str(dataset_root),

    "ThesisTotalProjects":
        THESIS_TOTAL_PROJECTS,

    "CompletedProjects":
        8,

    "RemainingProjects":
        17,

    "CandidatesInspected":
        len(candidate_inventory),

    "EligibleCandidates":
        len(eligible_candidates),

    "IneligibleCandidates":
        int(
            candidate_inventory[
                "InspectionStatus"
            ].eq("INELIGIBLE").sum()
        ),

    "InspectionErrors":
        int(
            candidate_inventory[
                "InspectionStatus"
            ].eq("ERROR").sum()
        ),

    "ProvisionalProject":
        provisional_candidate[
            "Project"
        ],

    "ProvisionalProjectSlug":
        provisional_candidate[
            "ProjectSlug"
        ],

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)

atomic_write_json(
    STEP1A_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Status":
            "PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE",

        "EligibleCandidates":
            len(eligible_candidates),

        "ProvisionalProject":
            provisional_candidate[
                "Project"
            ],

        "ProvisionalProjectSlug":
            provisional_candidate[
                "ProjectSlug"
            ],

        "CompletionRegistryModified":
            False,

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    },
)


# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "ModelReadyRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]

print("\nRanked eligible Project 9 candidates:")

display(
    eligible_candidates[
        display_columns
    ]
)

inspection_errors = candidate_inventory[
    candidate_inventory[
        "InspectionStatus"
    ].eq("ERROR")
]

if not inspection_errors.empty:
    print("\nInspection errors:")

    display(
        inspection_errors[
            [
                "Project",
                "InspectionError",
            ]
        ]
    )


print("\n")
print("=" * 104)
print("=== PROJECT 9 STEP 1A RESULT ===")
print("=" * 104)

print("\nThesis progress:")

print(
    "Total projects:",
    THESIS_TOTAL_PROJECTS,
)

print(
    "Completed:",
    8,
)

print(
    "Remaining before Project 9:",
    17,
)

print("\nCandidate discovery:")

print(
    "Candidates inspected:",
    len(candidate_inventory),
)

print(
    "Protocol-eligible candidates:",
    len(eligible_candidates),
)

print(
    "Inspection errors:",
    len(inspection_errors),
)

print("\nProvisional Project 9 candidate:")

print(
    "Candidate rank:",
    int(
        provisional_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    provisional_candidate[
        "Project"
    ],
)

print(
    "Project slug:",
    provisional_candidate[
        "ProjectSlug"
    ],
)

print(
    "Source directory:",
    provisional_candidate[
        "SourceDirectory"
    ],
)

print(
    "Builds:",
    int(
        provisional_candidate[
            "Builds"
        ]
    ),
)

print(
    "Training / evaluation builds:",
    int(
        provisional_candidate[
            "TrainingBuilds"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "EvaluationBuilds"
        ]
    ),
)

print(
    "Raw execution rows:",
    int(
        provisional_candidate[
            "RawExecutionRows"
        ]
    ),
)

print(
    "Model-ready rows:",
    int(
        provisional_candidate[
            "ModelReadyRows"
        ]
    ),
)

print(
    "Raw train / evaluation failures:",
    int(
        provisional_candidate[
            "RawTrainFailures"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "RawEvaluationFailures"
        ]
    ),
)

print(
    "Model train / evaluation failures:",
    int(
        provisional_candidate[
            "ModelTrainFailures"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "ModelEvaluationFailures"
        ]
    ),
)

print("\nSaved outputs:")

print(
    PROGRESS_PATH
)

print(
    CANDIDATE_INVENTORY_PATH
)

print(
    ELIGIBLE_CANDIDATES_PATH
)

print(
    PROVISIONAL_SELECTION_PATH
)

print(
    STEP1A_REPORT_PATH
)

print(
    STEP1A_STATUS_PATH
)

print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)

print(
    "\nSTATUS:",
    "PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE",
)

print("=" * 104)

=== PROJECT 9 STEP 1A: CANDIDATE DISCOVERY AND RANKING ===

Completion state:
Completed projects: 8
Remaining project directories: 17
Registry SHA-256: a442446f3ca6207b31213fc422fb0c883c96608a8e6a3907da969e326808bee7

No saved scan progress found.
Starting the 17-project scan.
[01/17] Inspecting: EMResearch@EvoMaster
    Status: ERROR
    Builds: nan | Model rows: nan | Model eval failures: nan | Time: 0.05 seconds
[02/17] Inspecting: Graylog2@graylog2-server
    Status: ERROR
    Builds: nan | Model rows: nan | Model eval failures: nan | Time: 0.17 seconds
[03/17] Inspecting: JMRI@JMRI
    Status: ERROR
    Builds: nan | Model rows: nan | Model eval failures: nan | Time: 0.05 seconds
[04/17] Inspecting: SonarSource@sonarqube
    Status: ERROR
    Builds: nan | Model rows: nan | Model eval failures: nan | Time: 0.24 seconds
[05/17] Inspecting: apache@curator
    Status: ERROR
    Builds: nan | Model rows: nan | Model eval failures: nan | Time: 0.15 seconds
[06/17] Inspecting: apache@lo

,Project,ProjectSlug,SourceDirectory,AlreadyCompleted,BuildsPath,ExecutionsPath,DatasetPath,IDMapPath,EntityHistoryPath,RequiredFilesPresent,...,ModelTrainPasses,ModelEvaluationFailures,ModelEvaluationPasses,ModelFailingTrainBuilds,ModelFailingEvaluationBuilds,ProtocolEligible,EligibilityReason,InspectionStatus,InspectionError,ElapsedSeconds
0,EMResearch@EvoMaster,EMResearch__EvoMaster,/content/datasets/EMResearch@EvoMaster,False,/content/datasets/EMResearch@EvoMaster/builds.csv,/content/datasets/EMResearch@EvoMaster/exe.csv,/content/datasets/EMResearch@EvoMaster/dataset...,/content/datasets/EMResearch@EvoMaster/id_map.csv,/content/datasets/EMResearch@EvoMaster/entity_...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.045684
1,Graylog2@graylog2-server,Graylog2__graylog2-server,/content/datasets/Graylog2@graylog2-server,False,/content/datasets/Graylog2@graylog2-server/bui...,/content/datasets/Graylog2@graylog2-server/exe...,/content/datasets/Graylog2@graylog2-server/dat...,/content/datasets/Graylog2@graylog2-server/id_...,/content/datasets/Graylog2@graylog2-server/ent...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.173498
2,JMRI@JMRI,JMRI__JMRI,/content/datasets/JMRI@JMRI,False,/content/datasets/JMRI@JMRI/builds.csv,/content/datasets/JMRI@JMRI/exe.csv,/content/datasets/JMRI@JMRI/dataset.csv,/content/datasets/JMRI@JMRI/id_map.csv,/content/datasets/JMRI@JMRI/entity_change_hist...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.048844
3,SonarSource@sonarqube,SonarSource__sonarqube,/content/datasets/SonarSource@sonarqube,False,/content/datasets/SonarSource@sonarqube/builds...,/content/datasets/SonarSource@sonarqube/exe.csv,/content/datasets/SonarSource@sonarqube/datase...,/content/datasets/SonarSource@sonarqube/id_map...,/content/datasets/SonarSource@sonarqube/entity...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.244085
4,apache@curator,apache__curator,/content/datasets/apache@curator,False,/content/datasets/apache@curator/builds.csv,/content/datasets/apache@curator/exe.csv,/content/datasets/apache@curator/dataset.csv,/content/datasets/apache@curator/id_map.csv,/content/datasets/apache@curator/entity_change...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.152402
5,apache@logging-log4j2,apache__logging-log4j2,/content/datasets/apache@logging-log4j2,False,/content/datasets/apache@logging-log4j2/builds...,/content/datasets/apache@logging-log4j2/exe.csv,/content/datasets/apache@logging-log4j2/datase...,/content/datasets/apache@logging-log4j2/id_map...,/content/datasets/apache@logging-log4j2/entity...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.069282
6,apache@rocketmq,apache__rocketmq,/content/datasets/apache@rocketmq,False,/content/datasets/apache@rocketmq/builds.csv,/content/datasets/apache@rocketmq/exe.csv,/content/datasets/apache@rocketmq/dataset.csv,/content/datasets/apache@rocketmq/id_map.csv,/content/datasets/apache@rocketmq/entity_chang...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.109962
7,apache@shardingsphere,apache__shardingsphere,/content/datasets/apache@shardingsphere,False,/content/datasets/apache@shardingsphere/builds...,/content/datasets/apache@shardingsphere/exe.csv,/content/datasets/apache@shardingsphere/datase...,/content/datasets/apache@shardingsphere/id_map...,/content/datasets/apache@shardingsphere/entity...,True,...,NaN,NaN,NaN,NaN,NaN,False,Inspection error,ERROR,RuntimeError: Required column not found.\nCand...,0.141197
8,apache@sling,apache__sling,/content/datasets/apache@sling,False,/content/datasets/apache@sling/builds.csv,/content/datasets/apache@sling/exe.csv,/

RuntimeError: No protocol-eligible Project 9 candidate was found.

In [4]:
# ============================================================
# PROJECT 9 — CELL 1 V2
# COMPLETE CANDIDATE DISCOVERY, ELIGIBILITY AND RANKING
#
# Fixes:
# - builds.csv build identifier: id
# - builds.csv chronology: started_at
# - exe.csv build identifier: build
# - exe.csv verdict: test_result
# - dataset.csv model fields: Build and Verdict
# - records the exact resolved schema for every project
# - uses a new V2 progress file, ignoring failed V1 progress
# - saves progress after every candidate
#
# This cell DOES NOT:
# - modify Projects 1–8
# - modify the completion registry
# - freeze Project 9
# - train any model
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9
THESIS_TOTAL_PROJECTS = 25

EXPECTED_COMPLETED_PROJECTS = list(
    range(1, 9)
)

EXPECTED_COMPLETION_STATUS = (
    "COMPLETE_AND_FROZEN"
)

EXPECTED_BOOTSTRAP_STATUS = (
    "PASS_PROJECT_9_NEW_NOTEBOOK_RUNTIME_BOOTSTRAP"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE"
)

CSV_CHUNK_SIZE = 250_000


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT9_SELECTION_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_09_selection"
)

BOOTSTRAP_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_runtime_bootstrap.json"
)


# The failed V1 progress remains untouched.

FAILED_V1_PROGRESS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_scan_progress.csv"
)


# V2 uses a separate progress file.

PROGRESS_V2_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_scan_progress_v2.csv"
)

SCHEMA_AUDIT_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_inventory.csv"
)

ELIGIBLE_CANDIDATES_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_eligible_candidates_ranked.csv"
)

INELIGIBLE_CANDIDATES_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_ineligible_candidates.csv"
)

INSPECTION_ERRORS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_inspection_errors.csv"
)

PROVISIONAL_SELECTION_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_provisional_selection.json"
)

STEP1A_REPORT_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_step1a_status.json"
)


print("=" * 106)
print("=== PROJECT 9 CELL 1 V2: CANDIDATE DISCOVERY, ELIGIBILITY AND RANKING ===")
print("=" * 106)


# ------------------------------------------------------------
# 3. FILE HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def normalise_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def project_slug(
    project_name,
):
    return str(
        project_name
    ).replace(
        "@",
        "__",
    )


def read_header(
    path,
):
    path = Path(path)

    if path.name.lower().endswith(
        ".parquet"
    ):

        return list(
            pd.read_parquet(
                path
            ).columns
        )

    return list(
        pd.read_csv(
            path,
            nrows=0,
        ).columns
    )


def read_full_selected_columns(
    path,
    columns,
):
    path = Path(path)

    if path.name.lower().endswith(
        ".parquet"
    ):

        return pd.read_parquet(
            path,
            columns=columns,
        )

    return pd.read_csv(
        path,
        usecols=columns,
        low_memory=False,
    )


def iterate_selected_columns(
    path,
    columns,
):
    path = Path(path)

    if path.name.lower().endswith(
        ".parquet"
    ):

        yield pd.read_parquet(
            path,
            columns=columns,
        )

        return

    for chunk in pd.read_csv(
        path,
        usecols=columns,
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ):

        yield chunk


def detect_column(
    columns,
    candidates,
    role,
):
    lookup = {
        normalise_name(column):
            column
        for column in columns
    }

    for candidate in candidates:

        candidate_key = normalise_name(
            candidate
        )

        if candidate_key in lookup:

            return lookup[
                candidate_key
            ]

    raise RuntimeError(
        "Required column not found.\n"
        f"Role: {role}\n"
        f"Candidates: {candidates}\n"
        f"Available columns: {columns}"
    )


# ------------------------------------------------------------
# 4. DATA NORMALISATION
# ------------------------------------------------------------

def canonical_identifier(
    series,
):
    """
    Convert numerically equivalent identifiers to the same
    string representation.

    Examples:
        123, 123.0 and '123' -> '123'
    """

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    numeric_coverage = float(
        numeric.notna().mean()
    )


    if numeric_coverage >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()


        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )


    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def failure_mask(
    verdict_series,
):
    """
    TCP-CI verdict convention:
        0 = pass
        nonzero = failure subtype
    """

    numeric = pd.to_numeric(
        verdict_series,
        errors="coerce",
    )

    numeric_coverage = float(
        numeric.notna().mean()
    )


    if numeric_coverage >= 0.95:

        return numeric.fillna(
            0
        ).ne(0)


    text = (
        verdict_series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )


    pass_values = {
        "",
        "0",
        "pass",
        "passed",
        "success",
        "successful",
        "ok",
    }


    failure_values = {
        "1",
        "2",
        "3",
        "fail",
        "failed",
        "failure",
        "error",
        "errored",
        "exception",
        "assertion",
        "unknown_failure",
    }


    known_mask = text.isin(
        pass_values
        | failure_values
    )


    if not known_mask.all():

        unknown_values = sorted(
            text[
                ~known_mask
            ].unique()
        )[:20]

        raise RuntimeError(
            "Unrecognised verdict values:\n"
            f"{unknown_values}"
        )


    return text.isin(
        failure_values
    )


# ------------------------------------------------------------
# 5. PROJECT SOURCE-SCHEMA RESOLUTION
# ------------------------------------------------------------

def resolve_standard_project_files(
    project_directory,
):
    """
    The 25-project archive uses the same five standard
    source filenames.
    """

    expected_files = {
        "BuildsPath":
            project_directory
            / "builds.csv",

        "ExecutionsPath":
            project_directory
            / "exe.csv",

        "DatasetPath":
            project_directory
            / "dataset.csv",

        "IDMapPath":
            project_directory
            / "id_map.csv",

        "EntityHistoryPath":
            project_directory
            / "entity_change_history.csv",
    }


    missing = [
        role
        for role, path in expected_files.items()
        if not path.exists()
    ]


    if missing:

        raise FileNotFoundError(
            "Standard TCP-CI project files are missing.\n"
            f"Project: {project_directory.name}\n"
            f"Missing roles: {missing}"
        )


    return expected_files


def resolve_project_schema(
    project_directory,
):
    files = resolve_standard_project_files(
        project_directory
    )


    builds_columns = read_header(
        files[
            "BuildsPath"
        ]
    )


    execution_columns = read_header(
        files[
            "ExecutionsPath"
        ]
    )


    dataset_columns = read_header(
        files[
            "DatasetPath"
        ]
    )


    build_id_column = detect_column(
        builds_columns,
        [
            "id",
            "build",
            "Build",
            "build_id",
            "BuildID",
            "BuildId",
            "tr_build_id",
            "travisBuildId",
            "travis_build_id",
        ],
        role=(
            "builds.csv build identifier"
        ),
    )


    started_at_column = detect_column(
        builds_columns,
        [
            "started_at",
            "StartedAt",
            "gh_build_started_at",
            "build_started_at",
            "BuildStartedAt",
            "start_time",
            "StartTime",
            "started",
            "timestamp",
            "Timestamp",
        ],
        role=(
            "builds.csv chronological timestamp"
        ),
    )


    execution_build_column = detect_column(
        execution_columns,
        [
            "build",
            "Build",
            "build_id",
            "BuildID",
            "BuildId",
            "travisBuildId",
            "travis_build_id",
            "tr_build_id",
        ],
        role=(
            "exe.csv build identifier"
        ),
    )


    execution_verdict_column = detect_column(
        execution_columns,
        [
            "test_result",
            "TestResult",
            "testResult",
            "verdict",
            "Verdict",
            "result",
            "Result",
            "outcome",
            "Outcome",
            "label",
            "Label",
        ],
        role=(
            "exe.csv verdict"
        ),
    )


    dataset_build_column = detect_column(
        dataset_columns,
        [
            "Build",
            "build",
            "build_id",
            "BuildID",
            "BuildId",
            "travisBuildId",
            "travis_build_id",
            "tr_build_id",
        ],
        role=(
            "dataset.csv build identifier"
        ),
    )


    dataset_verdict_column = detect_column(
        dataset_columns,
        [
            "Verdict",
            "verdict",
            "test_result",
            "TestResult",
            "testResult",
            "result",
            "Result",
            "outcome",
            "Outcome",
            "label",
            "Label",
        ],
        role=(
            "dataset.csv verdict"
        ),
    )


    return {
        **files,

        "BuildsColumns":
            builds_columns,

        "ExecutionsColumns":
            execution_columns,

        "DatasetColumns":
            dataset_columns,

        "BuildIDColumn":
            build_id_column,

        "StartedAtColumn":
            started_at_column,

        "ExecutionBuildColumn":
            execution_build_column,

        "ExecutionVerdictColumn":
            execution_verdict_column,

        "DatasetBuildColumn":
            dataset_build_column,

        "DatasetVerdictColumn":
            dataset_verdict_column,
    }


# ------------------------------------------------------------
# 6. BUILD CHRONOLOGY
# ------------------------------------------------------------

def construct_build_chronology(
    builds_path,
    build_id_column,
    started_at_column,
):
    builds = read_full_selected_columns(
        builds_path,
        [
            build_id_column,
            started_at_column,
        ],
    )


    builds = builds.rename(
        columns={
            build_id_column:
                "BuildOriginal",

            started_at_column:
                "StartedAtOriginal",
        }
    )


    builds[
        "BuildKey"
    ] = canonical_identifier(
        builds[
            "BuildOriginal"
        ]
    )


    builds[
        "StartedAt"
    ] = pd.to_datetime(
        builds[
            "StartedAtOriginal"
        ],
        errors="coerce",
        utc=True,
    )


    missing_build_ids = int(
        builds[
            "BuildKey"
        ].eq("").sum()
    )


    timestamp_parse_failures = int(
        builds[
            "StartedAt"
        ].isna().sum()
    )


    if missing_build_ids:

        raise RuntimeError(
            "builds.csv contains missing build identifiers.\n"
            f"Rows: {missing_build_ids}"
        )


    if timestamp_parse_failures:

        examples = (
            builds.loc[
                builds[
                    "StartedAt"
                ].isna(),
                "StartedAtOriginal",
            ]
            .astype(str)
            .head(10)
            .tolist()
        )

        raise RuntimeError(
            "builds.csv contains unparseable timestamps.\n"
            f"Rows: {timestamp_parse_failures}\n"
            f"Examples: {examples}"
        )


    duplicate_timestamp_variants = (
        builds
        .groupby(
            "BuildKey"
        )[
            "StartedAt"
        ]
        .nunique()
    )


    conflicting_builds = (
        duplicate_timestamp_variants[
            duplicate_timestamp_variants.gt(1)
        ]
    )


    if not conflicting_builds.empty:

        raise RuntimeError(
            "Duplicate build IDs have conflicting timestamps.\n"
            f"Examples: "
            f"{conflicting_builds.head(10).to_dict()}"
        )


    builds = (
        builds.drop_duplicates(
            subset=[
                "BuildKey",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )


    numeric_build_ids = pd.to_numeric(
        builds[
            "BuildKey"
        ],
        errors="coerce",
    )


    if numeric_build_ids.notna().all():

        builds[
            "BuildTieOrder"
        ] = numeric_build_ids

    else:

        builds[
            "BuildTieOrder"
        ] = (
            builds[
                "BuildKey"
            ]
            .astype(str)
        )


    # Frozen chronology:
    #   started_at ascending
    #   build ID descending for equal timestamps

    builds = (
        builds.sort_values(
            [
                "StartedAt",
                "BuildTieOrder",
            ],
            ascending=[
                True,
                False,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    builds[
        "BuildOrder"
    ] = np.arange(
        1,
        len(builds) + 1,
    )


    if not builds[
        "BuildKey"
    ].is_unique:

        raise AssertionError(
            "Canonical build chronology is not unique."
        )


    return (
        builds,
        timestamp_parse_failures,
    )


# ------------------------------------------------------------
# 7. PARTITION COUNTS
# ------------------------------------------------------------

def count_partition_rows(
    path,
    build_column,
    verdict_column,
    training_build_keys,
    evaluation_build_keys,
):
    result = {
        "Rows":
            0,

        "TrainingRows":
            0,

        "EvaluationRows":
            0,

        "UnlinkedRows":
            0,

        "TrainingFailures":
            0,

        "TrainingPasses":
            0,

        "EvaluationFailures":
            0,

        "EvaluationPasses":
            0,

        "FailingTrainingBuilds":
            set(),

        "FailingEvaluationBuilds":
            set(),

        "ObservedVerdicts":
            set(),
    }


    for chunk in iterate_selected_columns(
        path,
        [
            build_column,
            verdict_column,
        ],
    ):

        build_keys = canonical_identifier(
            chunk[
                build_column
            ]
        )


        failures = failure_mask(
            chunk[
                verdict_column
            ]
        )


        numeric_verdicts = pd.to_numeric(
            chunk[
                verdict_column
            ],
            errors="coerce",
        )


        result[
            "ObservedVerdicts"
        ].update(
            numeric_verdicts
            .dropna()
            .astype(float)
            .unique()
            .tolist()
        )


        training_mask = build_keys.isin(
            training_build_keys
        )


        evaluation_mask = build_keys.isin(
            evaluation_build_keys
        )


        linked_mask = (
            training_mask
            | evaluation_mask
        )


        result[
            "Rows"
        ] += int(
            len(chunk)
        )


        result[
            "TrainingRows"
        ] += int(
            training_mask.sum()
        )


        result[
            "EvaluationRows"
        ] += int(
            evaluation_mask.sum()
        )


        result[
            "UnlinkedRows"
        ] += int(
            (
                ~linked_mask
            ).sum()
        )


        result[
            "TrainingFailures"
        ] += int(
            (
                training_mask
                & failures
            ).sum()
        )


        result[
            "TrainingPasses"
        ] += int(
            (
                training_mask
                & ~failures
            ).sum()
        )


        result[
            "EvaluationFailures"
        ] += int(
            (
                evaluation_mask
                & failures
            ).sum()
        )


        result[
            "EvaluationPasses"
        ] += int(
            (
                evaluation_mask
                & ~failures
            ).sum()
        )


        result[
            "FailingTrainingBuilds"
        ].update(
            build_keys.loc[
                training_mask
                & failures
            ].tolist()
        )


        result[
            "FailingEvaluationBuilds"
        ].update(
            build_keys.loc[
                evaluation_mask
                & failures
            ].tolist()
        )


    result[
        "FailingTrainingBuilds"
    ] = len(
        result[
            "FailingTrainingBuilds"
        ]
    )


    result[
        "FailingEvaluationBuilds"
    ] = len(
        result[
            "FailingEvaluationBuilds"
        ]
    )


    result[
        "ObservedVerdicts"
    ] = ",".join(
        str(value)
        for value in sorted(
            result[
                "ObservedVerdicts"
            ]
        )
    )


    return result


# ------------------------------------------------------------
# 8. INSPECT ONE CANDIDATE
# ------------------------------------------------------------

def inspect_candidate(
    project_directory,
):
    start_time = time.perf_counter()

    project_name = (
        project_directory.name
    )

    slug = project_slug(
        project_name
    )


    record = {
        "Project":
            project_name,

        "ProjectSlug":
            slug,

        "SourceDirectory":
            str(
                project_directory
            ),

        "BuildsPath":
            "",

        "ExecutionsPath":
            "",

        "DatasetPath":
            "",

        "IDMapPath":
            "",

        "EntityHistoryPath":
            "",

        "BuildIDColumn":
            "",

        "StartedAtColumn":
            "",

        "ExecutionBuildColumn":
            "",

        "ExecutionVerdictColumn":
            "",

        "DatasetBuildColumn":
            "",

        "DatasetVerdictColumn":
            "",

        "BuildColumnCount":
            np.nan,

        "ExecutionColumnCount":
            np.nan,

        "DatasetColumnCount":
            np.nan,

        "Builds":
            np.nan,

        "TrainingBuilds":
            np.nan,

        "EvaluationBuilds":
            np.nan,

        "RawExecutionRows":
            np.nan,

        "RawTrainingRows":
            np.nan,

        "RawEvaluationRows":
            np.nan,

        "RawUnlinkedRows":
            np.nan,

        "RawTrainFailures":
            np.nan,

        "RawTrainPasses":
            np.nan,

        "RawEvaluationFailures":
            np.nan,

        "RawEvaluationPasses":
            np.nan,

        "RawFailingTrainBuilds":
            np.nan,

        "RawFailingEvaluationBuilds":
            np.nan,

        "RawObservedVerdicts":
            "",

        "ModelReadyRows":
            np.nan,

        "ModelTrainingRows":
            np.nan,

        "ModelEvaluationRows":
            np.nan,

        "ModelUnlinkedRows":
            np.nan,

        "ModelTrainFailures":
            np.nan,

        "ModelTrainPasses":
            np.nan,

        "ModelEvaluationFailures":
            np.nan,

        "ModelEvaluationPasses":
            np.nan,

        "ModelFailingTrainBuilds":
            np.nan,

        "ModelFailingEvaluationBuilds":
            np.nan,

        "ModelObservedVerdicts":
            "",

        "TimestampParseFailures":
            np.nan,

        "ProtocolEligible":
            False,

        "EligibilityReason":
            "",

        "InspectionStatus":
            "PENDING",

        "InspectionError":
            "",

        "ElapsedSeconds":
            np.nan,
    }


    try:

        schema = resolve_project_schema(
            project_directory
        )


        for path_role in [
            "BuildsPath",
            "ExecutionsPath",
            "DatasetPath",
            "IDMapPath",
            "EntityHistoryPath",
        ]:

            record[
                path_role
            ] = str(
                schema[
                    path_role
                ]
            )


        for column_role in [
            "BuildIDColumn",
            "StartedAtColumn",
            "ExecutionBuildColumn",
            "ExecutionVerdictColumn",
            "DatasetBuildColumn",
            "DatasetVerdictColumn",
        ]:

            record[
                column_role
            ] = schema[
                column_role
            ]


        record[
            "BuildColumnCount"
        ] = len(
            schema[
                "BuildsColumns"
            ]
        )


        record[
            "ExecutionColumnCount"
        ] = len(
            schema[
                "ExecutionsColumns"
            ]
        )


        record[
            "DatasetColumnCount"
        ] = len(
            schema[
                "DatasetColumns"
            ]
        )


        (
            chronology,
            timestamp_parse_failures,
        ) = construct_build_chronology(
            builds_path=(
                schema[
                    "BuildsPath"
                ]
            ),

            build_id_column=(
                schema[
                    "BuildIDColumn"
                ]
            ),

            started_at_column=(
                schema[
                    "StartedAtColumn"
                ]
            ),
        )


        number_of_builds = int(
            len(
                chronology
            )
        )


        training_build_count = int(
            math.floor(
                0.75
                * number_of_builds
            )
        )


        evaluation_build_count = int(
            number_of_builds
            - training_build_count
        )


        training_build_keys = set(
            chronology.iloc[
                :training_build_count
            ][
                "BuildKey"
            ]
        )


        evaluation_build_keys = set(
            chronology.iloc[
                training_build_count:
            ][
                "BuildKey"
            ]
        )


        if (
            training_build_keys
            & evaluation_build_keys
        ):

            raise AssertionError(
                "Training and evaluation build sets overlap."
            )


        raw_counts = count_partition_rows(
            path=(
                schema[
                    "ExecutionsPath"
                ]
            ),

            build_column=(
                schema[
                    "ExecutionBuildColumn"
                ]
            ),

            verdict_column=(
                schema[
                    "ExecutionVerdictColumn"
                ]
            ),

            training_build_keys=(
                training_build_keys
            ),

            evaluation_build_keys=(
                evaluation_build_keys
            ),
        )


        model_counts = count_partition_rows(
            path=(
                schema[
                    "DatasetPath"
                ]
            ),

            build_column=(
                schema[
                    "DatasetBuildColumn"
                ]
            ),

            verdict_column=(
                schema[
                    "DatasetVerdictColumn"
                ]
            ),

            training_build_keys=(
                training_build_keys
            ),

            evaluation_build_keys=(
                evaluation_build_keys
            ),
        )


        record.update({
            "Builds":
                number_of_builds,

            "TrainingBuilds":
                training_build_count,

            "EvaluationBuilds":
                evaluation_build_count,

            "TimestampParseFailures":
                timestamp_parse_failures,

            "RawExecutionRows":
                raw_counts[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_counts[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_counts[
                    "EvaluationRows"
                ],

            "RawUnlinkedRows":
                raw_counts[
                    "UnlinkedRows"
                ],

            "RawTrainFailures":
                raw_counts[
                    "TrainingFailures"
                ],

            "RawTrainPasses":
                raw_counts[
                    "TrainingPasses"
                ],

            "RawEvaluationFailures":
                raw_counts[
                    "EvaluationFailures"
                ],

            "RawEvaluationPasses":
                raw_counts[
                    "EvaluationPasses"
                ],

            "RawFailingTrainBuilds":
                raw_counts[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_counts[
                    "FailingEvaluationBuilds"
                ],

            "RawObservedVerdicts":
                raw_counts[
                    "ObservedVerdicts"
                ],

            "ModelReadyRows":
                model_counts[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_counts[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_counts[
                    "EvaluationRows"
                ],

            "ModelUnlinkedRows":
                model_counts[
                    "UnlinkedRows"
                ],

            "ModelTrainFailures":
                model_counts[
                    "TrainingFailures"
                ],

            "ModelTrainPasses":
                model_counts[
                    "TrainingPasses"
                ],

            "ModelEvaluationFailures":
                model_counts[
                    "EvaluationFailures"
                ],

            "ModelEvaluationPasses":
                model_counts[
                    "EvaluationPasses"
                ],

            "ModelFailingTrainBuilds":
                model_counts[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_counts[
                    "FailingEvaluationBuilds"
                ],

            "ModelObservedVerdicts":
                model_counts[
                    "ObservedVerdicts"
                ],
        })


        eligibility_reasons = []


        if number_of_builds < 4:

            eligibility_reasons.append(
                "Fewer than four chronological builds"
            )


        if training_build_count < 1:

            eligibility_reasons.append(
                "Empty training-build partition"
            )


        if evaluation_build_count < 1:

            eligibility_reasons.append(
                "Empty evaluation-build partition"
            )


        if raw_counts[
            "TrainingRows"
        ] < 1:

            eligibility_reasons.append(
                "No raw training rows"
            )


        if raw_counts[
            "EvaluationRows"
        ] < 1:

            eligibility_reasons.append(
                "No raw evaluation rows"
            )


        if raw_counts[
            "TrainingFailures"
        ] < 1:

            eligibility_reasons.append(
                "No raw training failures"
            )


        if raw_counts[
            "TrainingPasses"
        ] < 1:

            eligibility_reasons.append(
                "No raw training passes"
            )


        if raw_counts[
            "EvaluationFailures"
        ] < 1:

            eligibility_reasons.append(
                "No raw evaluation failures"
            )


        if model_counts[
            "TrainingRows"
        ] < 1:

            eligibility_reasons.append(
                "No model-ready training rows"
            )


        if model_counts[
            "EvaluationRows"
        ] < 1:

            eligibility_reasons.append(
                "No model-ready evaluation rows"
            )


        if model_counts[
            "TrainingFailures"
        ] < 1:

            eligibility_reasons.append(
                "No model-ready training failures"
            )


        if model_counts[
            "TrainingPasses"
        ] < 1:

            eligibility_reasons.append(
                "No model-ready training passes"
            )


        if model_counts[
            "EvaluationFailures"
        ] < 1:

            eligibility_reasons.append(
                "No model-ready evaluation failures"
            )


        if eligibility_reasons:

            record[
                "ProtocolEligible"
            ] = False

            record[
                "EligibilityReason"
            ] = "; ".join(
                eligibility_reasons
            )

            record[
                "InspectionStatus"
            ] = "INELIGIBLE"

        else:

            record[
                "ProtocolEligible"
            ] = True

            record[
                "EligibilityReason"
            ] = (
                "Essential fixed-holdout protocol "
                "requirements satisfied"
            )

            record[
                "InspectionStatus"
            ] = "ELIGIBLE"


    except Exception as error:

        record[
            "ProtocolEligible"
        ] = False

        record[
            "EligibilityReason"
        ] = "Inspection error"

        record[
            "InspectionStatus"
        ] = "ERROR"

        record[
            "InspectionError"
        ] = (
            f"{type(error).__name__}: {error}"
        )


    record[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - start_time
    )


    return record


# ------------------------------------------------------------
# 9. VALIDATE BOOTSTRAP
# ------------------------------------------------------------

required_paths = [
    BOOTSTRAP_STATUS_PATH,
    REGISTRY_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Project 9 inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


bootstrap = json.loads(
    BOOTSTRAP_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    bootstrap.get(
        "Status"
    )
    != EXPECTED_BOOTSTRAP_STATUS
):

    raise AssertionError(
        "Project 9 Cell 0 V2 has not passed."
    )


dataset_root = Path(
    bootstrap[
        "DatasetRoot"
    ]
)


if not dataset_root.exists():

    raise FileNotFoundError(
        "Runtime dataset root is missing.\n"
        "Rerun Project 9 Cell 0 V2."
    )


# ------------------------------------------------------------
# 10. VALIDATE REGISTRY
# ------------------------------------------------------------

registry_sha256_before = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}


missing_registry_columns = (
    required_registry_columns
    - set(
        registry.columns
    )
)


if missing_registry_columns:

    raise RuntimeError(
        "Registry is missing required columns:\n"
        f"{sorted(missing_registry_columns)}"
    )


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if len(
    registry
) != 8:

    raise AssertionError(
        "Registry must contain exactly eight projects."
    )


if set(
    registry_project_numbers
) != set(
    EXPECTED_COMPLETED_PROJECTS
):

    raise AssertionError(
        "Registry does not contain exactly Projects 1–8."
    )


if registry_project_numbers.duplicated().any():

    raise AssertionError(
        "Duplicate project numbers exist in the registry."
    )


if not registry[
    "Status"
].eq(
    EXPECTED_COMPLETION_STATUS
).all():

    raise AssertionError(
        "Projects 1–8 are not all COMPLETE_AND_FROZEN."
    )


completed_project_names = set(
    registry[
        "Project"
    ].astype(str)
)


completed_project_slugs = set(
    registry[
        "ProjectSlug"
    ].astype(str)
)


# ------------------------------------------------------------
# 11. DISCOVER REMAINING PROJECTS
# ------------------------------------------------------------

all_project_directories = sorted([
    path
    for path in dataset_root.iterdir()
    if (
        path.is_dir()
        and "@" in path.name
    )
])


if len(
    all_project_directories
) != THESIS_TOTAL_PROJECTS:

    raise AssertionError(
        "Runtime dataset does not contain 25 projects.\n"
        f"Actual: {len(all_project_directories)}"
    )


remaining_project_directories = [
    path
    for path in all_project_directories
    if (
        path.name
        not in completed_project_names
        and project_slug(
            path.name
        )
        not in completed_project_slugs
    )
]


if len(
    remaining_project_directories
) != 17:

    raise AssertionError(
        "Remaining project count differs.\n"
        f"Expected: 17\n"
        f"Actual: {len(remaining_project_directories)}"
    )


print("\nCompletion state:")

print(
    "Completed projects:",
    len(
        registry
    ),
)

print(
    "Remaining project directories:",
    len(
        remaining_project_directories
    ),
)

print(
    "Dataset root:",
    dataset_root,
)

print(
    "Registry SHA-256:",
    registry_sha256_before,
)


# ------------------------------------------------------------
# 12. LOAD VALID V2 PROGRESS
# ------------------------------------------------------------

PROJECT9_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


progress_records = []
completed_v2_projects = set()


if PROGRESS_V2_PATH.exists():

    existing_progress = pd.read_csv(
        PROGRESS_V2_PATH,
        low_memory=False,
    )


    if not existing_progress.empty:

        # Successful or conclusively ineligible V2 rows may
        # be reused. ERROR rows are always rescanned.

        reusable_mask = (
            existing_progress[
                "InspectionStatus"
            ].isin(
                [
                    "ELIGIBLE",
                    "INELIGIBLE",
                ]
            )
        )


        reusable_progress = (
            existing_progress.loc[
                reusable_mask
            ]
            .copy()
        )


        progress_records = (
            reusable_progress
            .to_dict(
                "records"
            )
        )


        completed_v2_projects = set(
            reusable_progress[
                "Project"
            ].astype(str)
        )


    print("\nResuming V2 scan progress:")

    print(
        "Reusable candidate scans:",
        len(
            completed_v2_projects
        ),
    )


else:

    print("\nNo V2 scan progress found.")
    print(
        "Starting a fresh corrected 17-project scan."
    )


if FAILED_V1_PROGRESS_PATH.exists():

    print(
        "\nFailed V1 progress retained for audit:"
    )

    print(
        FAILED_V1_PROGRESS_PATH
    )


# ------------------------------------------------------------
# 13. SCAN ALL 17 REMAINING PROJECTS
# ------------------------------------------------------------

for position, project_directory in enumerate(
    remaining_project_directories,
    start=1,
):

    project_name = (
        project_directory.name
    )


    if project_name in completed_v2_projects:

        print(
            f"[{position:02d}/17] "
            f"Reusing V2 result: {project_name}"
        )

        continue


    print(
        f"[{position:02d}/17] "
        f"Inspecting: {project_name}"
    )


    result = inspect_candidate(
        project_directory
    )


    # Replace an earlier V2 row for the same project.

    progress_records = [
        row
        for row in progress_records
        if str(
            row.get(
                "Project"
            )
        ) != project_name
    ]


    progress_records.append(
        result
    )


    progress_frame = (
        pd.DataFrame(
            progress_records
        )
        .sort_values(
            "Project",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    atomic_write_csv(
        PROGRESS_V2_PATH,
        progress_frame,
    )


    print(
        "    Status:",
        result[
            "InspectionStatus"
        ],
    )


    print(
        "    Resolved columns:",
        (
            result[
                "BuildIDColumn"
            ],
            result[
                "StartedAtColumn"
            ],
            result[
                "ExecutionBuildColumn"
            ],
            result[
                "ExecutionVerdictColumn"
            ],
            result[
                "DatasetBuildColumn"
            ],
            result[
                "DatasetVerdictColumn"
            ],
        ),
    )


    print(
        "    Builds:",
        result[
            "Builds"
        ],
        "| Model rows:",
        result[
            "ModelReadyRows"
        ],
        "| Model eval failures:",
        result[
            "ModelEvaluationFailures"
        ],
        "| Seconds:",
        round(
            float(
                result[
                    "ElapsedSeconds"
                ]
            ),
            2,
        ),
    )


    if (
        result[
            "InspectionStatus"
        ]
        == "ERROR"
    ):

        print(
            "    Error:",
            result[
                "InspectionError"
            ],
        )


# ------------------------------------------------------------
# 14. FINALISE CANDIDATE INVENTORY
# ------------------------------------------------------------

candidate_inventory = (
    pd.read_csv(
        PROGRESS_V2_PATH,
        low_memory=False,
    )
    .sort_values(
        "Project",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    candidate_inventory
) != 17:

    raise AssertionError(
        "V2 candidate inventory does not contain "
        "all 17 unfinished projects.\n"
        f"Actual rows: {len(candidate_inventory)}"
    )


candidate_inventory[
    "ProtocolEligible"
] = (
    candidate_inventory[
        "ProtocolEligible"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)


inspection_errors = (
    candidate_inventory[
        candidate_inventory[
            "InspectionStatus"
        ].eq(
            "ERROR"
        )
    ]
    .copy()
)


schema_audit_columns = [
    "Project",
    "ProjectSlug",
    "BuildsPath",
    "ExecutionsPath",
    "DatasetPath",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "BuildColumnCount",
    "ExecutionColumnCount",
    "DatasetColumnCount",
    "InspectionStatus",
    "InspectionError",
]


schema_audit = (
    candidate_inventory[
        schema_audit_columns
    ]
    .copy()
)


atomic_write_csv(
    SCHEMA_AUDIT_PATH,
    schema_audit,
)


# Every remaining project must now have been inspected
# successfully or conclusively classified as ineligible.

if not inspection_errors.empty:

    atomic_write_csv(
        INSPECTION_ERRORS_PATH,
        inspection_errors,
    )


    print("\nCandidate inspection errors:")

    display(
        inspection_errors[
            [
                "Project",
                "BuildIDColumn",
                "StartedAtColumn",
                "ExecutionBuildColumn",
                "ExecutionVerdictColumn",
                "DatasetBuildColumn",
                "DatasetVerdictColumn",
                "InspectionError",
            ]
        ]
    )


    raise RuntimeError(
        "PROJECT 9 CELL 1 V2 DID NOT COMPLETE.\n"
        f"{len(inspection_errors)} project(s) still have "
        "source inspection errors.\n"
        "The corrected progress was preserved."
    )


# ------------------------------------------------------------
# 15. RANK ELIGIBLE CANDIDATES
# ------------------------------------------------------------

eligible_candidates = (
    candidate_inventory[
        candidate_inventory[
            "ProtocolEligible"
        ]
    ]
    .copy()
)


ineligible_candidates = (
    candidate_inventory[
        ~candidate_inventory[
            "ProtocolEligible"
        ]
    ]
    .copy()
)


if eligible_candidates.empty:

    display(
        candidate_inventory
    )

    raise RuntimeError(
        "No protocol-eligible Project 9 candidate "
        "was found after successful schema resolution."
    )


eligible_candidates[
    "MinimumEvaluationFailureSupport"
] = (
    eligible_candidates[
        [
            "RawEvaluationFailures",
            "ModelEvaluationFailures",
        ]
    ]
    .min(
        axis=1
    )
)


eligible_candidates[
    "MinimumTrainingFailureSupport"
] = (
    eligible_candidates[
        [
            "RawTrainFailures",
            "ModelTrainFailures",
        ]
    ]
    .min(
        axis=1
    )
)


# Ranking is only for operational execution order.
# All 25 projects remain in the thesis scope.

eligible_candidates = (
    eligible_candidates
    .sort_values(
        [
            "MinimumEvaluationFailureSupport",
            "ModelFailingEvaluationBuilds",
            "MinimumTrainingFailureSupport",
            "ModelFailingTrainBuilds",
            "ModelReadyRows",
            "Project",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(
            eligible_candidates
        ) + 1,
    ),
)


provisional_candidate = (
    eligible_candidates.iloc[0]
)


# ------------------------------------------------------------
# 16. VERIFY REGISTRY IMMUTABILITY
# ------------------------------------------------------------

registry_sha256_after = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


if (
    registry_sha256_after
    != registry_sha256_before
):

    raise AssertionError(
        "Completion registry changed during candidate scan."
    )


# ------------------------------------------------------------
# 17. WRITE FINAL STEP 1A OUTPUTS
# ------------------------------------------------------------

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    candidate_inventory,
)


atomic_write_csv(
    ELIGIBLE_CANDIDATES_PATH,
    eligible_candidates,
)


atomic_write_csv(
    INELIGIBLE_CANDIDATES_PATH,
    ineligible_candidates,
)


if INSPECTION_ERRORS_PATH.exists():

    INSPECTION_ERRORS_PATH.unlink()


provisional_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionStatus":
        "PROVISIONAL_NOT_FROZEN",

    "CandidateRank":
        int(
            provisional_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        provisional_candidate[
            "Project"
        ],

    "ProjectSlug":
        provisional_candidate[
            "ProjectSlug"
        ],

    "SourceDirectory":
        provisional_candidate[
            "SourceDirectory"
        ],

    "SourceFiles": {
        "Builds":
            provisional_candidate[
                "BuildsPath"
            ],

        "Executions":
            provisional_candidate[
                "ExecutionsPath"
            ],

        "Dataset":
            provisional_candidate[
                "DatasetPath"
            ],

        "IDMap":
            provisional_candidate[
                "IDMapPath"
            ],

        "EntityHistory":
            provisional_candidate[
                "EntityHistoryPath"
            ],
    },

    "ResolvedSchema": {
        "BuildIDColumn":
            provisional_candidate[
                "BuildIDColumn"
            ],

        "StartedAtColumn":
            provisional_candidate[
                "StartedAtColumn"
            ],

        "ExecutionBuildColumn":
            provisional_candidate[
                "ExecutionBuildColumn"
            ],

        "ExecutionVerdictColumn":
            provisional_candidate[
                "ExecutionVerdictColumn"
            ],

        "DatasetBuildColumn":
            provisional_candidate[
                "DatasetBuildColumn"
            ],

        "DatasetVerdictColumn":
            provisional_candidate[
                "DatasetVerdictColumn"
            ],
    },

    "CandidateMetrics": {
        "Builds":
            int(
                provisional_candidate[
                    "Builds"
                ]
            ),

        "TrainingBuilds":
            int(
                provisional_candidate[
                    "TrainingBuilds"
                ]
            ),

        "EvaluationBuilds":
            int(
                provisional_candidate[
                    "EvaluationBuilds"
                ]
            ),

        "RawExecutionRows":
            int(
                provisional_candidate[
                    "RawExecutionRows"
                ]
            ),

        "RawTrainingRows":
            int(
                provisional_candidate[
                    "RawTrainingRows"
                ]
            ),

        "RawEvaluationRows":
            int(
                provisional_candidate[
                    "RawEvaluationRows"
                ]
            ),

        "RawTrainFailures":
            int(
                provisional_candidate[
                    "RawTrainFailures"
                ]
            ),

        "RawEvaluationFailures":
            int(
                provisional_candidate[
                    "RawEvaluationFailures"
                ]
            ),

        "ModelReadyRows":
            int(
                provisional_candidate[
                    "ModelReadyRows"
                ]
            ),

        "ModelTrainingRows":
            int(
                provisional_candidate[
                    "ModelTrainingRows"
                ]
            ),

        "ModelEvaluationRows":
            int(
                provisional_candidate[
                    "ModelEvaluationRows"
                ]
            ),

        "ModelTrainFailures":
            int(
                provisional_candidate[
                    "ModelTrainFailures"
                ]
            ),

        "ModelEvaluationFailures":
            int(
                provisional_candidate[
                    "ModelEvaluationFailures"
                ]
            ),

        "ModelFailingEvaluationBuilds":
            int(
                provisional_candidate[
                    "ModelFailingEvaluationBuilds"
                ]
            ),
    },

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "GeneratedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "DatasetRoot":
        str(
            dataset_root
        ),

    "ThesisTotalProjects":
        THESIS_TOTAL_PROJECTS,

    "CompletedProjects":
        8,

    "RemainingProjects":
        17,

    "CandidatesInspected":
        len(
            candidate_inventory
        ),

    "EligibleCandidates":
        len(
            eligible_candidates
        ),

    "IneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalProject":
        provisional_candidate[
            "Project"
        ],

    "ProvisionalProjectSlug":
        provisional_candidate[
            "ProjectSlug"
        ],

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)


atomic_write_json(
    STEP1A_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Status":
            STEP1A_PASS_STATUS,

        "CandidatesInspected":
            len(
                candidate_inventory
            ),

        "EligibleCandidates":
            len(
                eligible_candidates
            ),

        "IneligibleCandidates":
            len(
                ineligible_candidates
            ),

        "InspectionErrors":
            0,

        "ProvisionalProject":
            provisional_candidate[
                "Project"
            ],

        "ProvisionalProjectSlug":
            provisional_candidate[
                "ProjectSlug"
            ],

        "CompletionRegistryModified":
            False,

        "Projects1To8Modified":
            False,

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    },
)


# ------------------------------------------------------------
# 18. DISPLAY RANKED RESULTS
# ------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]


print("\nRanked eligible Project 9 candidates:")

display(
    eligible_candidates[
        ranked_display_columns
    ]
)


if not ineligible_candidates.empty:

    print("\nProtocol-ineligible remaining projects:")

    display(
        ineligible_candidates[
            [
                "Project",
                "Builds",
                "RawTrainFailures",
                "RawEvaluationFailures",
                "ModelTrainFailures",
                "ModelEvaluationFailures",
                "EligibilityReason",
            ]
        ]
    )


print("\nResolved TCP-CI schemas:")

display(
    schema_audit[
        [
            "Project",
            "BuildIDColumn",
            "StartedAtColumn",
            "ExecutionBuildColumn",
            "ExecutionVerdictColumn",
            "DatasetBuildColumn",
            "DatasetVerdictColumn",
            "InspectionStatus",
        ]
    ]
)


# ------------------------------------------------------------
# 19. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 106)
print("=== PROJECT 9 CELL 1 V2 RESULT ===")
print("=" * 106)

print("\nThesis progress:")

print(
    "Total projects:",
    THESIS_TOTAL_PROJECTS,
)

print(
    "Completed:",
    8,
)

print(
    "Remaining before Project 9:",
    17,
)


print("\nCandidate discovery:")

print(
    "Candidates inspected:",
    len(
        candidate_inventory
    ),
)

print(
    "Protocol-eligible candidates:",
    len(
        eligible_candidates
    ),
)

print(
    "Protocol-ineligible candidates:",
    len(
        ineligible_candidates
    ),
)

print(
    "Inspection errors:",
    len(
        inspection_errors
    ),
)


print("\nProvisional Project 9 candidate:")

print(
    "Candidate rank:",
    int(
        provisional_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    provisional_candidate[
        "Project"
    ],
)

print(
    "Project slug:",
    provisional_candidate[
        "ProjectSlug"
    ],
)

print(
    "Source directory:",
    provisional_candidate[
        "SourceDirectory"
    ],
)


print("\nResolved schema:")

print(
    "builds.csv build ID:",
    provisional_candidate[
        "BuildIDColumn"
    ],
)

print(
    "builds.csv timestamp:",
    provisional_candidate[
        "StartedAtColumn"
    ],
)

print(
    "exe.csv build:",
    provisional_candidate[
        "ExecutionBuildColumn"
    ],
)

print(
    "exe.csv verdict:",
    provisional_candidate[
        "ExecutionVerdictColumn"
    ],
)

print(
    "dataset.csv build:",
    provisional_candidate[
        "DatasetBuildColumn"
    ],
)

print(
    "dataset.csv verdict:",
    provisional_candidate[
        "DatasetVerdictColumn"
    ],
)


print("\nCandidate dimensions:")

print(
    "Builds:",
    int(
        provisional_candidate[
            "Builds"
        ]
    ),
)

print(
    "Training / evaluation builds:",
    int(
        provisional_candidate[
            "TrainingBuilds"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "EvaluationBuilds"
        ]
    ),
)

print(
    "Raw execution rows:",
    int(
        provisional_candidate[
            "RawExecutionRows"
        ]
    ),
)

print(
    "Raw training / evaluation rows:",
    int(
        provisional_candidate[
            "RawTrainingRows"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "RawEvaluationRows"
        ]
    ),
)

print(
    "Raw train / evaluation failures:",
    int(
        provisional_candidate[
            "RawTrainFailures"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "RawEvaluationFailures"
        ]
    ),
)

print(
    "Model-ready rows:",
    int(
        provisional_candidate[
            "ModelReadyRows"
        ]
    ),
)

print(
    "Model training / evaluation rows:",
    int(
        provisional_candidate[
            "ModelTrainingRows"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "ModelEvaluationRows"
        ]
    ),
)

print(
    "Model train / evaluation failures:",
    int(
        provisional_candidate[
            "ModelTrainFailures"
        ]
    ),
    "/",
    int(
        provisional_candidate[
            "ModelEvaluationFailures"
        ]
    ),
)

print(
    "Model failing evaluation builds:",
    int(
        provisional_candidate[
            "ModelFailingEvaluationBuilds"
        ]
    ),
)


print("\nSaved outputs:")

for output_path in [
    PROGRESS_V2_PATH,
    SCHEMA_AUDIT_PATH,
    CANDIDATE_INVENTORY_PATH,
    ELIGIBLE_CANDIDATES_PATH,
    INELIGIBLE_CANDIDATES_PATH,
    PROVISIONAL_SELECTION_PATH,
    STEP1A_REPORT_PATH,
    STEP1A_STATUS_PATH,
]:

    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP1A_PASS_STATUS,
)

print("=" * 106)

=== PROJECT 9 CELL 1 V2: CANDIDATE DISCOVERY, ELIGIBILITY AND RANKING ===

Completion state:
Completed projects: 8
Remaining project directories: 17
Dataset root: /content/datasets
Registry SHA-256: a442446f3ca6207b31213fc422fb0c883c96608a8e6a3907da969e326808bee7

No V2 scan progress found.
Starting a fresh corrected 17-project scan.

Failed V1 progress retained for audit:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/project_09_selection/project_09_candidate_scan_progress.csv
[01/17] Inspecting: EMResearch@EvoMaster
    Status: ELIGIBLE
    Resolved columns: ('id', 'started_at', 'build', 'verdict', 'Build', 'Verdict')
    Builds: 583 | Model rows: 14460 | Model eval failures: 68 | Seconds: 1.2
[02/17] Inspecting: Graylog2@graylog2-server
    Status: INELIGIBLE
    Resolved columns: ('id', 'started_at', 'build', 'verdict', 'Build', 'Verdict')
    Builds: 3668 | Model rows: 4822 | Model eval failures: 0 | Seconds: 1.86
[03/17] Inspecting: JMRI@JMRI
    Status: ELIGIBLE
   

,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,RawExecutionRows,RawTrainFailures,RawEvaluationFailures,RawFailingEvaluationBuilds,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainFailures,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,822,616,206,472765,1447,665,30,79383,60880,18503,1427,665,30,0,0
1,2,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,408,306,102,47094,65,213,27,8706,6095,2611,63,213,27,0,0
2,3,apache@shardingsphere,apache__shardingsphere,1049,786,263,833541,1193,171,25,91042,78035,13007,1188,171,25,0,0
3,4,zolyfarkas@spf4j,zolyfarkas__spf4j,587,440,147,68787,297,101,91,26762,18406,8356,296,101,91,0,0
4,5,jcabi@jcabi-github,jcabi__jcabi-github,809,606,203,140526,91,78,44,10437,2357,8080,90,78,44,0,0
5,6,JMRI@JMRI,JMRI__JMRI,1481,1110,371,6469640,240,73,24,410395,303251,107144,239,73,24,0,0
6,7,EMResearch@EvoMaster,EMResearch__EvoMaster,583,437,146,59155,286,68,41,14460,9907,4553,284,68,41,0,0
7,8,apache@sling,apache__sling,1403,1052,351,265459,767,49,48,113175,107157,6018,765,49,48,0,0
8,9,yamcs@Yamcs,yamcs__Yamcs,504,378,126,58101,147,45,10,7533,6452,1081,145,45,10,0,0
9,10,apache@logging-log4j2,apache__logging-log4j2,441,330,111,240253,208,40,39,117968,95812,22156,207,40,39,0,0



Protocol-ineligible remaining projects:


,Project,Builds,RawTrainFailures,RawEvaluationFailures,ModelTrainFailures,ModelEvaluationFailures,EligibilityReason
1,Graylog2@graylog2-server,3668,280,0,279,0,No raw evaluation failures; No model-ready eva...



Resolved TCP-CI schemas:


,Project,BuildIDColumn,StartedAtColumn,ExecutionBuildColumn,ExecutionVerdictColumn,DatasetBuildColumn,DatasetVerdictColumn,InspectionStatus
0,EMResearch@EvoMaster,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
1,Graylog2@graylog2-server,id,started_at,build,verdict,Build,Verdict,INELIGIBLE
2,JMRI@JMRI,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
3,SonarSource@sonarqube,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
4,apache@curator,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
5,apache@logging-log4j2,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
6,apache@rocketmq,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
7,apache@shardingsphere,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
8,apache@sling,id,started_at,build,verdict,Build,Verdict,ELIGIBLE
9,camunda@camunda-bpm-platform,id,started_at,build,verdict,Build,Verdict,ELIGIBLE




=== PROJECT 9 CELL 1 V2 RESULT ===

Thesis progress:
Total projects: 25
Completed: 8
Remaining before Project 9: 17

Candidate discovery:
Candidates inspected: 17
Protocol-eligible candidates: 16
Protocol-ineligible candidates: 1
Inspection errors: 0

Provisional Project 9 candidate:
Candidate rank: 1
Project: camunda@camunda-bpm-platform
Project slug: camunda__camunda-bpm-platform
Source directory: /content/datasets/camunda@camunda-bpm-platform

Resolved schema:
builds.csv build ID: id
builds.csv timestamp: started_at
exe.csv build: build
exe.csv verdict: verdict
dataset.csv build: Build
dataset.csv verdict: Verdict

Candidate dimensions:
Builds: 822
Training / evaluation builds: 616 / 206
Raw execution rows: 472765
Raw training / evaluation rows: 313984 / 158781
Raw train / evaluation failures: 1447 / 665
Model-ready rows: 79383
Model training / evaluation rows: 60880 / 18503
Model train / evaluation failures: 1427 / 665
Model failing evaluation builds: 30

Saved outputs:
/content/

In [5]:
# ============================================================
# PROJECT 9 — CELL 2 / STEP 1B
# PERMANENT SELECTION LOCK, SOURCE FREEZE AND SPLIT PREFLIGHT
#
# Selected project:
#   camunda@camunda-bpm-platform
#
# This cell:
# - validates the successful Project 9 candidate scan
# - confirms Camunda is ranked first
# - hashes and freezes all five source files
# - reconstructs canonical build chronology
# - freezes the chronological 75%/25% split
# - verifies all raw and model-ready row counts
# - identifies the fixed scored evaluation-build cohort
# - creates project_09_selection_checkpoint.json
#
# This cell does NOT:
# - modify the completion registry
# - modify Projects 1–8
# - modify the source dataset
# - generate noisy labels
# - train models
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. FROZEN PROJECT IDENTITY
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = (
    "camunda"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

COMPLETION_STATUS = (
    "COMPLETE_AND_FROZEN"
)


# ------------------------------------------------------------
# 2. FROZEN PROJECT 9 DIMENSIONS
# ------------------------------------------------------------

EXPECTED_CANDIDATE_RANK = 1

EXPECTED_BUILDS = 822
EXPECTED_TRAINING_BUILDS = 616
EXPECTED_EVALUATION_BUILDS = 206

EXPECTED_RAW_ROWS = 472765
EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_RAW_EVALUATION_ROWS = 158781
EXPECTED_RAW_TRAIN_FAILURES = 1447
EXPECTED_RAW_EVALUATION_FAILURES = 665
EXPECTED_RAW_FAILING_EVALUATION_BUILDS = 30

EXPECTED_MODEL_ROWS = 79383
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503
EXPECTED_MODEL_TRAIN_FAILURES = 1427
EXPECTED_MODEL_EVALUATION_FAILURES = 665
EXPECTED_MODEL_FAILING_EVALUATION_BUILDS = 30

EXPECTED_RAW_UNLINKED_ROWS = 0
EXPECTED_MODEL_UNLINKED_ROWS = 0

TRAINING_FRACTION = 0.75
EVALUATION_FRACTION = 0.25


# ------------------------------------------------------------
# 3. FROZEN SOURCE SCHEMA
# ------------------------------------------------------------

EXPECTED_BUILD_ID_COLUMN = "id"
EXPECTED_STARTED_AT_COLUMN = "started_at"

EXPECTED_EXECUTION_BUILD_COLUMN = "build"
EXPECTED_EXECUTION_VERDICT_COLUMN = "verdict"

EXPECTED_DATASET_BUILD_COLUMN = "Build"
EXPECTED_DATASET_VERDICT_COLUMN = "Verdict"


# ------------------------------------------------------------
# 4. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT9_SELECTION_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / "project_09_selection"
)

BOOTSTRAP_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_runtime_bootstrap.json"
)

STEP1A_STATUS_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_step1a_status.json"
)

PROVISIONAL_SELECTION_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_provisional_selection.json"
)

CANDIDATE_INVENTORY_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_candidate_inventory.csv"
)

ELIGIBLE_CANDIDATES_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_eligible_candidates_ranked.csv"
)

SCHEMA_AUDIT_PATH = (
    PROJECT9_SELECTION_DIR
    / "project_09_source_schema_audit.csv"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

SELECTION_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_preflight"
)

SOURCE_MANIFEST_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_file_manifest.csv"
)

BUILD_CHRONOLOGY_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_canonical_build_chronology.csv.gz"
)

FIXED_SPLIT_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_fixed_build_split.csv.gz"
)

SCORED_EVALUATION_BUILDS_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_scored_evaluation_builds.csv"
)

SELECTION_VALIDATION_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_validation.csv"
)

SELECTION_REPORT_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_report.json"
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

PROJECT9_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)


print("=" * 108)
print("=== PROJECT 9 CELL 2 / STEP 1B: SELECTION LOCK, SOURCE FREEZE AND SPLIT PREFLIGHT ===")
print("=" * 108)


# ------------------------------------------------------------
# 5. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativeName",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            str(
                row.RelativeName
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(
                    row.SizeBytes
                )
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:

        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):

        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):

        return value.isoformat()

    try:

        if pd.isna(value):

            return None

    except Exception:

        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if compression == "gzip":

        temporary_path = path.with_name(
            path.name + ".tmp.gz"
        )

    else:

        temporary_path = path.with_name(
            path.name + ".tmp"
        )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


def normalise_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    numeric_coverage = float(
        numeric.notna().mean()
    )

    if numeric_coverage >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def failure_mask(
    verdict_series,
):
    numeric = pd.to_numeric(
        verdict_series,
        errors="coerce",
    )

    numeric_coverage = float(
        numeric.notna().mean()
    )

    if numeric_coverage >= 0.95:

        return numeric.fillna(
            0
        ).ne(0)

    text = (
        verdict_series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    passing_values = {
        "",
        "0",
        "pass",
        "passed",
        "success",
        "successful",
        "ok",
    }

    failure_values = {
        "1",
        "2",
        "3",
        "fail",
        "failed",
        "failure",
        "error",
        "errored",
        "exception",
        "assertion",
    }

    unknown_mask = ~text.isin(
        passing_values
        | failure_values
    )

    if unknown_mask.any():

        raise RuntimeError(
            "Unknown verdict values detected:\n"
            f"{sorted(text[unknown_mask].unique())[:20]}"
        )

    return text.isin(
        failure_values
    )


def detect_column(
    columns,
    candidates,
    required=True,
):
    lookup = {
        normalise_name(
            column
        ):
            column
        for column in columns
    }

    for candidate in candidates:

        candidate_key = normalise_name(
            candidate
        )

        if candidate_key in lookup:

            return lookup[
                candidate_key
            ]

    if required:

        raise RuntimeError(
            "Required column was not found.\n"
            f"Candidates: {candidates}\n"
            f"Available: {columns}"
        )

    return None


def build_level_counts(
    dataframe,
    build_column,
    verdict_column,
    row_prefix,
):
    working = dataframe[
        [
            build_column,
            verdict_column,
        ]
    ].copy()

    working[
        "BuildKey"
    ] = canonical_identifier(
        working[
            build_column
        ]
    )

    working[
        "IsFailure"
    ] = failure_mask(
        working[
            verdict_column
        ]
    )

    counts = (
        working
        .groupby(
            "BuildKey",
            as_index=False,
        )
        .agg(
            Rows=(
                "BuildKey",
                "size",
            ),

            Failures=(
                "IsFailure",
                "sum",
            ),
        )
    )

    counts[
        "Passes"
    ] = (
        counts[
            "Rows"
        ]
        - counts[
            "Failures"
        ]
    )

    counts = counts.rename(
        columns={
            "Rows":
                f"{row_prefix}Rows",

            "Failures":
                f"{row_prefix}Failures",

            "Passes":
                f"{row_prefix}Passes",
        }
    )

    return (
        working,
        counts,
    )


# ------------------------------------------------------------
# 6. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    STEP1A_STATUS_PATH,
    PROVISIONAL_SELECTION_PATH,
    CANDIDATE_INVENTORY_PATH,
    ELIGIBLE_CANDIDATES_PATH,
    SCHEMA_AUDIT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required Project 9 Step 1B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


# ------------------------------------------------------------
# 7. VALIDATE BOOTSTRAP AND STEP 1A
# ------------------------------------------------------------

bootstrap_status = json.loads(
    BOOTSTRAP_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step1a_status = json.loads(
    STEP1A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

provisional_selection = json.loads(
    PROVISIONAL_SELECTION_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    bootstrap_status.get(
        "Status"
    )
    != "PASS_PROJECT_9_NEW_NOTEBOOK_RUNTIME_BOOTSTRAP"
):

    raise AssertionError(
        "Project 9 Cell 0 V2 has not passed."
    )

if (
    step1a_status.get(
        "Status"
    )
    != EXPECTED_SELECTION_STATUS
):

    raise AssertionError(
        "Project 9 Cell 1 V2 has not passed."
    )

if (
    step1a_status.get(
        "ProvisionalProject"
    )
    != PROJECT_NAME
):

    raise AssertionError(
        "Step 1A provisional project differs."
    )

if (
    step1a_status.get(
        "ProvisionalProjectSlug"
    )
    != PROJECT_SLUG
):

    raise AssertionError(
        "Step 1A provisional project slug differs."
    )

if (
    provisional_selection.get(
        "Project"
    )
    != PROJECT_NAME
):

    raise AssertionError(
        "Provisional-selection project differs."
    )

if int(
    provisional_selection.get(
        "CandidateRank",
        -1,
    )
) != EXPECTED_CANDIDATE_RANK:

    raise AssertionError(
        "Provisional candidate is not ranked first."
    )


# ------------------------------------------------------------
# 8. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)

if len(
    registry
) != 8:

    raise AssertionError(
        "Registry must contain exactly Projects 1–8."
    )

if set(
    project_numbers
) != set(
    range(1, 9)
):

    raise AssertionError(
        "Registry does not contain exactly Projects 1–8."
    )

if project_numbers.duplicated().any():

    raise AssertionError(
        "Duplicate project numbers exist."
    )

if not registry[
    "Status"
].eq(
    COMPLETION_STATUS
).all():

    raise AssertionError(
        "Projects 1–8 are not all COMPLETE_AND_FROZEN."
    )

if (
    registry[
        "Project"
    ].eq(
        PROJECT_NAME
    ).any()
    or
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
):

    raise AssertionError(
        "Project 9 is already present in the registry."
    )


# ------------------------------------------------------------
# 9. VALIDATE RANKED CANDIDATE
# ------------------------------------------------------------

eligible_candidates = pd.read_csv(
    ELIGIBLE_CANDIDATES_PATH,
    low_memory=False,
)

candidate_inventory = pd.read_csv(
    CANDIDATE_INVENTORY_PATH,
    low_memory=False,
)

if len(
    eligible_candidates
) != 16:

    raise AssertionError(
        "Expected 16 protocol-eligible candidates."
    )

ranked_first = (
    eligible_candidates
    .sort_values(
        "CandidateRank",
        kind="mergesort",
    )
    .iloc[0]
)

if int(
    ranked_first[
        "CandidateRank"
    ]
) != EXPECTED_CANDIDATE_RANK:

    raise AssertionError(
        "Ranked-first candidate does not have rank 1."
    )

if (
    ranked_first[
        "Project"
    ]
    != PROJECT_NAME
):

    raise AssertionError(
        "Ranked-first candidate is not Camunda."
    )

if (
    ranked_first[
        "ProjectSlug"
    ]
    != PROJECT_SLUG
):

    raise AssertionError(
        "Ranked-first candidate slug differs."
    )


expected_candidate_values = {
    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAINING_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "RawExecutionRows":
        EXPECTED_RAW_ROWS,

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "RawEvaluationRows":
        EXPECTED_RAW_EVALUATION_ROWS,

    "RawTrainFailures":
        EXPECTED_RAW_TRAIN_FAILURES,

    "RawEvaluationFailures":
        EXPECTED_RAW_EVALUATION_FAILURES,

    "RawFailingEvaluationBuilds":
        EXPECTED_RAW_FAILING_EVALUATION_BUILDS,

    "ModelReadyRows":
        EXPECTED_MODEL_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ModelTrainFailures":
        EXPECTED_MODEL_TRAIN_FAILURES,

    "ModelEvaluationFailures":
        EXPECTED_MODEL_EVALUATION_FAILURES,

    "ModelFailingEvaluationBuilds":
        EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,

    "RawUnlinkedRows":
        EXPECTED_RAW_UNLINKED_ROWS,

    "ModelUnlinkedRows":
        EXPECTED_MODEL_UNLINKED_ROWS,
}

for column, expected_value in (
    expected_candidate_values.items()
):

    actual_value = int(
        ranked_first[
            column
        ]
    )

    if actual_value != expected_value:

        raise AssertionError(
            "Ranked-candidate metric differs.\n"
            f"Column: {column}\n"
            f"Expected: {expected_value}\n"
            f"Actual: {actual_value}"
        )


# ------------------------------------------------------------
# 10. RESOLVE SOURCE DIRECTORY AND FILES
# ------------------------------------------------------------

dataset_root = Path(
    bootstrap_status[
        "DatasetRoot"
    ]
)

source_directory = (
    dataset_root
    / PROJECT_NAME
)

source_files = {
    "builds.csv":
        source_directory
        / "builds.csv",

    "exe.csv":
        source_directory
        / "exe.csv",

    "dataset.csv":
        source_directory
        / "dataset.csv",

    "id_map.csv":
        source_directory
        / "id_map.csv",

    "entity_change_history.csv":
        source_directory
        / "entity_change_history.csv",
}

missing_source_files = [
    str(path)
    for path in source_files.values()
    if not path.exists()
]

if missing_source_files:

    raise FileNotFoundError(
        "Project 9 source files are missing:\n"
        + "\n".join(
            missing_source_files
        )
    )


# ------------------------------------------------------------
# 11. FREEZE SOURCE FILE HASHES
# ------------------------------------------------------------

source_manifest_records = []

for relative_name, path in (
    source_files.items()
):

    source_manifest_records.append({
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "RelativeName":
            relative_name,

        "RuntimePath":
            str(path),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_hash(
                path
            ),
    })

source_manifest = pd.DataFrame(
    source_manifest_records
)

source_file_count = len(
    source_manifest
)

source_total_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)

source_root_sha256 = (
    root_inventory_hash(
        source_manifest
    )
)

source_hashes_before = {
    row.RelativeName:
        row.SHA256
    for row in source_manifest.itertuples(
        index=False
    )
}


# ------------------------------------------------------------
# 12. READ AND VALIDATE SOURCE SCHEMAS
# ------------------------------------------------------------

builds_columns = list(
    pd.read_csv(
        source_files[
            "builds.csv"
        ],
        nrows=0,
    ).columns
)

execution_columns = list(
    pd.read_csv(
        source_files[
            "exe.csv"
        ],
        nrows=0,
    ).columns
)

dataset_columns = list(
    pd.read_csv(
        source_files[
            "dataset.csv"
        ],
        nrows=0,
    ).columns
)

for expected_column, columns, file_name in [
    (
        EXPECTED_BUILD_ID_COLUMN,
        builds_columns,
        "builds.csv",
    ),
    (
        EXPECTED_STARTED_AT_COLUMN,
        builds_columns,
        "builds.csv",
    ),
    (
        EXPECTED_EXECUTION_BUILD_COLUMN,
        execution_columns,
        "exe.csv",
    ),
    (
        EXPECTED_EXECUTION_VERDICT_COLUMN,
        execution_columns,
        "exe.csv",
    ),
    (
        EXPECTED_DATASET_BUILD_COLUMN,
        dataset_columns,
        "dataset.csv",
    ),
    (
        EXPECTED_DATASET_VERDICT_COLUMN,
        dataset_columns,
        "dataset.csv",
    ),
]:

    if expected_column not in columns:

        raise AssertionError(
            "Frozen source column is missing.\n"
            f"File: {file_name}\n"
            f"Column: {expected_column}"
        )

execution_test_column = detect_column(
    execution_columns,
    [
        "test",
        "Test",
        "test_id",
        "TestID",
        "TestId",
    ],
)

dataset_test_column = detect_column(
    dataset_columns,
    [
        "Test",
        "test",
        "test_id",
        "TestID",
        "TestId",
    ],
)


# ------------------------------------------------------------
# 13. CONSTRUCT CANONICAL BUILD CHRONOLOGY
# ------------------------------------------------------------

builds = pd.read_csv(
    source_files[
        "builds.csv"
    ],
    usecols=[
        EXPECTED_BUILD_ID_COLUMN,
        EXPECTED_STARTED_AT_COLUMN,
    ],
    low_memory=False,
)

builds = builds.rename(
    columns={
        EXPECTED_BUILD_ID_COLUMN:
            "BuildOriginal",

        EXPECTED_STARTED_AT_COLUMN:
            "StartedAtOriginal",
    }
)

builds[
    "Build"
] = canonical_identifier(
    builds[
        "BuildOriginal"
    ]
)

builds[
    "StartedAt"
] = pd.to_datetime(
    builds[
        "StartedAtOriginal"
    ],
    errors="coerce",
    utc=True,
)

missing_build_ids = int(
    builds[
        "Build"
    ].eq("").sum()
)

timestamp_parse_failures = int(
    builds[
        "StartedAt"
    ].isna().sum()
)

if missing_build_ids:

    raise AssertionError(
        "Missing build identifiers were found."
    )

if timestamp_parse_failures:

    raise AssertionError(
        "Unparseable build timestamps were found."
    )

timestamp_variants = (
    builds
    .groupby(
        "Build"
    )[
        "StartedAt"
    ]
    .nunique()
)

conflicting_duplicate_builds = int(
    timestamp_variants.gt(1).sum()
)

if conflicting_duplicate_builds:

    raise AssertionError(
        "Duplicate builds have conflicting timestamps."
    )

builds = (
    builds.drop_duplicates(
        subset=[
            "Build",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

numeric_build_ids = pd.to_numeric(
    builds[
        "Build"
    ],
    errors="coerce",
)

if numeric_build_ids.notna().all():

    builds[
        "BuildTieOrder"
    ] = numeric_build_ids

else:

    builds[
        "BuildTieOrder"
    ] = (
        builds[
            "Build"
        ].astype(str)
    )

build_chronology = (
    builds.sort_values(
        [
            "StartedAt",
            "BuildTieOrder",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

build_chronology[
    "BuildOrder"
] = np.arange(
    1,
    len(
        build_chronology
    ) + 1,
)

build_chronology = (
    build_chronology[
        [
            "Build",
            "BuildOriginal",
            "StartedAt",
            "BuildOrder",
        ]
    ]
)

if len(
    build_chronology
) != EXPECTED_BUILDS:

    raise AssertionError(
        "Canonical build count differs."
    )

if not build_chronology[
    "Build"
].is_unique:

    raise AssertionError(
        "Canonical build identifiers are not unique."
    )


# ------------------------------------------------------------
# 14. FREEZE THE 75% / 25% BUILD SPLIT
# ------------------------------------------------------------

training_build_count = int(
    math.floor(
        TRAINING_FRACTION
        * len(
            build_chronology
        )
    )
)

evaluation_build_count = int(
    len(
        build_chronology
    )
    - training_build_count
)

if (
    training_build_count
    != EXPECTED_TRAINING_BUILDS
):

    raise AssertionError(
        "Training-build count differs."
    )

if (
    evaluation_build_count
    != EXPECTED_EVALUATION_BUILDS
):

    raise AssertionError(
        "Evaluation-build count differs."
    )

fixed_split = (
    build_chronology.copy()
)

fixed_split[
    "Partition"
] = np.where(
    fixed_split[
        "BuildOrder"
    ].le(
        training_build_count
    ),
    "TRAINING",
    "EVALUATION",
)

training_build_keys = set(
    fixed_split.loc[
        fixed_split[
            "Partition"
        ].eq(
            "TRAINING"
        ),
        "Build",
    ]
)

evaluation_build_keys = set(
    fixed_split.loc[
        fixed_split[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "Build",
    ]
)

if (
    training_build_keys
    & evaluation_build_keys
):

    raise AssertionError(
        "Training and evaluation build sets overlap."
    )


# ------------------------------------------------------------
# 15. LOAD RAW AND MODEL-READY COHORTS
# ------------------------------------------------------------

raw_source = pd.read_csv(
    source_files[
        "exe.csv"
    ],
    usecols=[
        EXPECTED_EXECUTION_BUILD_COLUMN,
        execution_test_column,
        EXPECTED_EXECUTION_VERDICT_COLUMN,
    ],
    low_memory=False,
)

model_source = pd.read_csv(
    source_files[
        "dataset.csv"
    ],
    usecols=[
        EXPECTED_DATASET_BUILD_COLUMN,
        dataset_test_column,
        EXPECTED_DATASET_VERDICT_COLUMN,
    ],
    low_memory=False,
)

raw_source[
    "BuildKey"
] = canonical_identifier(
    raw_source[
        EXPECTED_EXECUTION_BUILD_COLUMN
    ]
)

model_source[
    "BuildKey"
] = canonical_identifier(
    model_source[
        EXPECTED_DATASET_BUILD_COLUMN
    ]
)

raw_source[
    "IsFailure"
] = failure_mask(
    raw_source[
        EXPECTED_EXECUTION_VERDICT_COLUMN
    ]
)

model_source[
    "IsFailure"
] = failure_mask(
    model_source[
        EXPECTED_DATASET_VERDICT_COLUMN
    ]
)

raw_source[
    "Partition"
] = np.select(
    [
        raw_source[
            "BuildKey"
        ].isin(
            training_build_keys
        ),

        raw_source[
            "BuildKey"
        ].isin(
            evaluation_build_keys
        ),
    ],
    [
        "TRAINING",
        "EVALUATION",
    ],
    default="UNLINKED",
)

model_source[
    "Partition"
] = np.select(
    [
        model_source[
            "BuildKey"
        ].isin(
            training_build_keys
        ),

        model_source[
            "BuildKey"
        ].isin(
            evaluation_build_keys
        ),
    ],
    [
        "TRAINING",
        "EVALUATION",
    ],
    default="UNLINKED",
)


# ------------------------------------------------------------
# 16. RECOMPUTE ALL FROZEN COUNTS
# ------------------------------------------------------------

actual_counts = {
    "Builds":
        len(
            build_chronology
        ),

    "TrainingBuilds":
        len(
            training_build_keys
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_keys
        ),

    "RawRows":
        len(
            raw_source
        ),

    "RawTrainingRows":
        int(
            raw_source[
                "Partition"
            ].eq(
                "TRAINING"
            ).sum()
        ),

    "RawEvaluationRows":
        int(
            raw_source[
                "Partition"
            ].eq(
                "EVALUATION"
            ).sum()
        ),

    "RawUnlinkedRows":
        int(
            raw_source[
                "Partition"
            ].eq(
                "UNLINKED"
            ).sum()
        ),

    "RawTrainFailures":
        int(
            (
                raw_source[
                    "Partition"
                ].eq(
                    "TRAINING"
                )
                &
                raw_source[
                    "IsFailure"
                ]
            ).sum()
        ),

    "RawEvaluationFailures":
        int(
            (
                raw_source[
                    "Partition"
                ].eq(
                    "EVALUATION"
                )
                &
                raw_source[
                    "IsFailure"
                ]
            ).sum()
        ),

    "ModelRows":
        len(
            model_source
        ),

    "ModelTrainingRows":
        int(
            model_source[
                "Partition"
            ].eq(
                "TRAINING"
            ).sum()
        ),

    "ModelEvaluationRows":
        int(
            model_source[
                "Partition"
            ].eq(
                "EVALUATION"
            ).sum()
        ),

    "ModelUnlinkedRows":
        int(
            model_source[
                "Partition"
            ].eq(
                "UNLINKED"
            ).sum()
        ),

    "ModelTrainFailures":
        int(
            (
                model_source[
                    "Partition"
                ].eq(
                    "TRAINING"
                )
                &
                model_source[
                    "IsFailure"
                ]
            ).sum()
        ),

    "ModelEvaluationFailures":
        int(
            (
                model_source[
                    "Partition"
                ].eq(
                    "EVALUATION"
                )
                &
                model_source[
                    "IsFailure"
                ]
            ).sum()
        ),
}

expected_counts = {
    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAINING_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "RawRows":
        EXPECTED_RAW_ROWS,

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "RawEvaluationRows":
        EXPECTED_RAW_EVALUATION_ROWS,

    "RawUnlinkedRows":
        EXPECTED_RAW_UNLINKED_ROWS,

    "RawTrainFailures":
        EXPECTED_RAW_TRAIN_FAILURES,

    "RawEvaluationFailures":
        EXPECTED_RAW_EVALUATION_FAILURES,

    "ModelRows":
        EXPECTED_MODEL_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ModelUnlinkedRows":
        EXPECTED_MODEL_UNLINKED_ROWS,

    "ModelTrainFailures":
        EXPECTED_MODEL_TRAIN_FAILURES,

    "ModelEvaluationFailures":
        EXPECTED_MODEL_EVALUATION_FAILURES,
}

for check_name, expected_value in (
    expected_counts.items()
):

    actual_value = int(
        actual_counts[
            check_name
        ]
    )

    if actual_value != expected_value:

        raise AssertionError(
            "Frozen project count differs.\n"
            f"Check: {check_name}\n"
            f"Expected: {expected_value}\n"
            f"Actual: {actual_value}"
        )


# ------------------------------------------------------------
# 17. BUILD-LEVEL RAW AND MODEL COUNTS
# ------------------------------------------------------------

raw_working, raw_build_counts = (
    build_level_counts(
        dataframe=raw_source,
        build_column="BuildKey",
        verdict_column=(
            EXPECTED_EXECUTION_VERDICT_COLUMN
        ),
        row_prefix="Raw",
    )
)

model_working, model_build_counts = (
    build_level_counts(
        dataframe=model_source,
        build_column="BuildKey",
        verdict_column=(
            EXPECTED_DATASET_VERDICT_COLUMN
        ),
        row_prefix="Model",
    )
)

fixed_split = (
    fixed_split
    .merge(
        raw_build_counts,
        left_on="Build",
        right_on="BuildKey",
        how="left",
        validate="one_to_one",
    )
    .drop(
        columns=[
            "BuildKey",
        ]
    )
    .merge(
        model_build_counts,
        left_on="Build",
        right_on="BuildKey",
        how="left",
        validate="one_to_one",
    )
    .drop(
        columns=[
            "BuildKey",
        ]
    )
)

count_columns = [
    "RawRows",
    "RawFailures",
    "RawPasses",
    "ModelRows",
    "ModelFailures",
    "ModelPasses",
]

for column in count_columns:

    fixed_split[
        column
    ] = (
        fixed_split[
            column
        ]
        .fillna(0)
        .astype(int)
    )

fixed_split[
    "RawScoredEvaluationBuild"
] = (
    fixed_split[
        "Partition"
    ].eq(
        "EVALUATION"
    )
    &
    fixed_split[
        "RawFailures"
    ].gt(0)
)

fixed_split[
    "ModelScoredEvaluationBuild"
] = (
    fixed_split[
        "Partition"
    ].eq(
        "EVALUATION"
    )
    &
    fixed_split[
        "ModelFailures"
    ].gt(0)
)

fixed_split[
    "ScoredEvaluationBuild"
] = (
    fixed_split[
        "RawScoredEvaluationBuild"
    ]
    &
    fixed_split[
        "ModelScoredEvaluationBuild"
    ]
)

raw_scored_builds = set(
    fixed_split.loc[
        fixed_split[
            "RawScoredEvaluationBuild"
        ],
        "Build",
    ]
)

model_scored_builds = set(
    fixed_split.loc[
        fixed_split[
            "ModelScoredEvaluationBuild"
        ],
        "Build",
    ]
)

if raw_scored_builds != model_scored_builds:

    raise AssertionError(
        "Raw and model-ready failing evaluation-build "
        "sets differ."
    )

if len(
    raw_scored_builds
) != EXPECTED_MODEL_FAILING_EVALUATION_BUILDS:

    raise AssertionError(
        "Scored evaluation-build count differs."
    )

scored_evaluation_builds = (
    fixed_split.loc[
        fixed_split[
            "ScoredEvaluationBuild"
        ],
        [
            "Build",
            "BuildOrder",
            "StartedAt",
            "RawRows",
            "RawFailures",
            "ModelRows",
            "ModelFailures",
        ],
    ]
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

scored_evaluation_rows = int(
    scored_evaluation_builds[
        "ModelRows"
    ].sum()
)

scored_evaluation_failures = int(
    scored_evaluation_builds[
        "ModelFailures"
    ].sum()
)

if (
    scored_evaluation_failures
    != EXPECTED_MODEL_EVALUATION_FAILURES
):

    raise AssertionError(
        "Scored evaluation failures differ."
    )


# ------------------------------------------------------------
# 18. MODEL-READY BUILD/TEST UNIQUENESS
# ------------------------------------------------------------

model_source[
    "TestKey"
] = canonical_identifier(
    model_source[
        dataset_test_column
    ]
)

model_duplicate_build_test_rows = int(
    model_source.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)

if model_duplicate_build_test_rows:

    raise AssertionError(
        "Model-ready dataset contains duplicate "
        "Build/Test instances.\n"
        f"Rows: {model_duplicate_build_test_rows}"
    )


# ------------------------------------------------------------
# 19. VALIDATION TABLE
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1A status",

        "Expected":
            EXPECTED_SELECTION_STATUS,

        "Actual":
            step1a_status[
                "Status"
            ],

        "Pass":
            step1a_status[
                "Status"
            ]
            == EXPECTED_SELECTION_STATUS,
    },

    {
        "Check":
            "Selected project",

        "Expected":
            PROJECT_NAME,

        "Actual":
            ranked_first[
                "Project"
            ],

        "Pass":
            ranked_first[
                "Project"
            ]
            == PROJECT_NAME,
    },

    {
        "Check":
            "Candidate rank",

        "Expected":
            EXPECTED_CANDIDATE_RANK,

        "Actual":
            int(
                ranked_first[
                    "CandidateRank"
                ]
            ),

        "Pass":
            int(
                ranked_first[
                    "CandidateRank"
                ]
            )
            == EXPECTED_CANDIDATE_RANK,
    },

    {
        "Check":
            "Source files",

        "Expected":
            5,

        "Actual":
            source_file_count,

        "Pass":
            source_file_count == 5,
    },

    {
        "Check":
            "Canonical builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                build_chronology
            ),

        "Pass":
            len(
                build_chronology
            )
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Training builds",

        "Expected":
            EXPECTED_TRAINING_BUILDS,

        "Actual":
            len(
                training_build_keys
            ),

        "Pass":
            len(
                training_build_keys
            )
            == EXPECTED_TRAINING_BUILDS,
    },

    {
        "Check":
            "Evaluation builds",

        "Expected":
            EXPECTED_EVALUATION_BUILDS,

        "Actual":
            len(
                evaluation_build_keys
            ),

        "Pass":
            len(
                evaluation_build_keys
            )
            == EXPECTED_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Raw rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                raw_source
            ),

        "Pass":
            len(
                raw_source
            )
            == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Raw training rows",

        "Expected":
            EXPECTED_RAW_TRAINING_ROWS,

        "Actual":
            actual_counts[
                "RawTrainingRows"
            ],

        "Pass":
            actual_counts[
                "RawTrainingRows"
            ]
            == EXPECTED_RAW_TRAINING_ROWS,
    },

    {
        "Check":
            "Raw evaluation rows",

        "Expected":
            EXPECTED_RAW_EVALUATION_ROWS,

        "Actual":
            actual_counts[
                "RawEvaluationRows"
            ],

        "Pass":
            actual_counts[
                "RawEvaluationRows"
            ]
            == EXPECTED_RAW_EVALUATION_ROWS,
    },

    {
        "Check":
            "Raw evaluation failures",

        "Expected":
            EXPECTED_RAW_EVALUATION_FAILURES,

        "Actual":
            actual_counts[
                "RawEvaluationFailures"
            ],

        "Pass":
            actual_counts[
                "RawEvaluationFailures"
            ]
            == EXPECTED_RAW_EVALUATION_FAILURES,
    },

    {
        "Check":
            "Model rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                model_source
            ),

        "Pass":
            len(
                model_source
            )
            == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            actual_counts[
                "ModelTrainingRows"
            ],

        "Pass":
            actual_counts[
                "ModelTrainingRows"
            ]
            == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            actual_counts[
                "ModelEvaluationRows"
            ],

        "Pass":
            actual_counts[
                "ModelEvaluationRows"
            ]
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Check":
            "Model evaluation failures",

        "Expected":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "Actual":
            actual_counts[
                "ModelEvaluationFailures"
            ],

        "Pass":
            actual_counts[
                "ModelEvaluationFailures"
            ]
            == EXPECTED_MODEL_EVALUATION_FAILURES,
    },

    {
        "Check":
            "Raw unlinked rows",

        "Expected":
            0,

        "Actual":
            actual_counts[
                "RawUnlinkedRows"
            ],

        "Pass":
            actual_counts[
                "RawUnlinkedRows"
            ] == 0,
    },

    {
        "Check":
            "Model unlinked rows",

        "Expected":
            0,

        "Actual":
            actual_counts[
                "ModelUnlinkedRows"
            ],

        "Pass":
            actual_counts[
                "ModelUnlinkedRows"
            ] == 0,
    },

    {
        "Check":
            "Scored evaluation builds",

        "Expected":
            EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,

        "Actual":
            len(
                scored_evaluation_builds
            ),

        "Pass":
            len(
                scored_evaluation_builds
            )
            == EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Raw/model scored build sets equal",

        "Expected":
            True,

        "Actual":
            raw_scored_builds
            == model_scored_builds,

        "Pass":
            raw_scored_builds
            == model_scored_builds,
    },

    {
        "Check":
            "Scored evaluation failures",

        "Expected":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "Actual":
            scored_evaluation_failures,

        "Pass":
            scored_evaluation_failures
            == EXPECTED_MODEL_EVALUATION_FAILURES,
    },

    {
        "Check":
            "Model duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows == 0,
    },

    {
        "Check":
            "Timestamp parse failures",

        "Expected":
            0,

        "Actual":
            timestamp_parse_failures,

        "Pass":
            timestamp_parse_failures == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]

selection_validation = pd.DataFrame(
    validation_records
)

failed_checks = (
    selection_validation[
        ~selection_validation[
            "Pass"
        ]
    ]
    .copy()
)

print("\nSelection and split validation:")

display(
    selection_validation
)

if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 1B DID NOT PASS.\n"
        "Project 9 has not been locked."
    )


# ------------------------------------------------------------
# 20. WRITE FROZEN SELECTION OUTPUTS
# ------------------------------------------------------------

SELECTION_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_write_csv(
    SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    BUILD_CHRONOLOGY_PATH,
    build_chronology,
    compression="gzip",
)

atomic_write_csv(
    FIXED_SPLIT_PATH,
    fixed_split,
    compression="gzip",
)

atomic_write_csv(
    SCORED_EVALUATION_BUILDS_PATH,
    scored_evaluation_builds,
)

atomic_write_csv(
    SELECTION_VALIDATION_PATH,
    selection_validation,
)


# ------------------------------------------------------------
# 21. VERIFY SOURCE AND REGISTRY IMMUTABILITY
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_hash(
            path
        )
    for relative_name, path in (
        source_files.items()
    )
}

source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)

if not source_files_unchanged:

    raise AssertionError(
        "A source file changed during Step 1B."
    )

registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)

if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 1B."
    )


# ------------------------------------------------------------
# 22. WRITE PERMANENT PROJECT 9 CHECKPOINT
# ------------------------------------------------------------

selection_evidence_paths = [
    BOOTSTRAP_STATUS_PATH,
    STEP1A_STATUS_PATH,
    PROVISIONAL_SELECTION_PATH,
    CANDIDATE_INVENTORY_PATH,
    ELIGIBLE_CANDIDATES_PATH,
    SCHEMA_AUDIT_PATH,
    SOURCE_MANIFEST_PATH,
    BUILD_CHRONOLOGY_PATH,
    FIXED_SPLIT_PATH,
    SCORED_EVALUATION_BUILDS_PATH,
    SELECTION_VALIDATION_PATH,
]

selection_evidence = []

for path in selection_evidence_paths:

    selection_evidence.append({
        "Path":
            str(path),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_hash(
                path
            ),
    })

checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "SelectionStatus":
        "SELECTED_SOURCE_FROZEN_SPLIT_VALIDATED",

    "CandidateRank":
        EXPECTED_CANDIDATE_RANK,

    "SourceDirectory":
        str(
            source_directory
        ),

    "ArchivePath":
        bootstrap_status.get(
            "ArchivePath"
        ),

    "ArchiveMD5":
        bootstrap_status.get(
            "ArchiveMD5"
        ),

    "SourceFiles":
        {
            row.RelativeName: {
                "RuntimePath":
                    row.RuntimePath,

                "SizeBytes":
                    int(
                        row.SizeBytes
                    ),

                "SHA256":
                    row.SHA256,
            }
            for row in source_manifest.itertuples(
                index=False
            )
        },

    "SourceFileCount":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "ResolvedSchema": {
        "BuildsBuildID":
            EXPECTED_BUILD_ID_COLUMN,

        "BuildsStartedAt":
            EXPECTED_STARTED_AT_COLUMN,

        "ExecutionBuild":
            EXPECTED_EXECUTION_BUILD_COLUMN,

        "ExecutionTest":
            execution_test_column,

        "ExecutionVerdict":
            EXPECTED_EXECUTION_VERDICT_COLUMN,

        "DatasetBuild":
            EXPECTED_DATASET_BUILD_COLUMN,

        "DatasetTest":
            dataset_test_column,

        "DatasetVerdict":
            EXPECTED_DATASET_VERDICT_COLUMN,
    },

    "Chronology": {
        "PrimaryOrder":
            "started_at ascending",

        "EqualTimestampTieBreak":
            "build ID descending",

        "CanonicalBuilds":
            EXPECTED_BUILDS,

        "TimestampParseFailures":
            timestamp_parse_failures,

        "ConflictingDuplicateBuilds":
            conflicting_duplicate_builds,

        "ChronologyFile":
            str(
                BUILD_CHRONOLOGY_PATH
            ),
    },

    "FixedSplit": {
        "Type":
            "chronological fixed holdout",

        "TrainingFraction":
            TRAINING_FRACTION,

        "EvaluationFraction":
            EVALUATION_FRACTION,

        "TrainingBuilds":
            EXPECTED_TRAINING_BUILDS,

        "EvaluationBuilds":
            EXPECTED_EVALUATION_BUILDS,

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "RawEvaluationRows":
            EXPECTED_RAW_EVALUATION_ROWS,

        "ModelTrainingRows":
            EXPECTED_MODEL_TRAINING_ROWS,

        "ModelEvaluationRows":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "FixedSplitFile":
            str(
                FIXED_SPLIT_PATH
            ),
    },

    "FailureSupport": {
        "RawTrainingFailures":
            EXPECTED_RAW_TRAIN_FAILURES,

        "RawEvaluationFailures":
            EXPECTED_RAW_EVALUATION_FAILURES,

        "ModelTrainingFailures":
            EXPECTED_MODEL_TRAIN_FAILURES,

        "ModelEvaluationFailures":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "ScoredEvaluationBuilds":
            EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,

        "ScoredEvaluationRows":
            scored_evaluation_rows,

        "ScoredEvaluationFailures":
            scored_evaluation_failures,

        "ScoredEvaluationBuildsFile":
            str(
                SCORED_EVALUATION_BUILDS_PATH
            ),
    },

    "FixedInstanceDesign": {
        "RawTrainingCohortRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "ModelTrainingCohortRows":
            EXPECTED_MODEL_TRAINING_ROWS,

        "ModelEvaluationCohortRows":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "ModelBuildTestDuplicates":
            model_duplicate_build_test_rows,

        "RawUnlinkedRows":
            EXPECTED_RAW_UNLINKED_ROWS,

        "ModelUnlinkedRows":
            EXPECTED_MODEL_UNLINKED_ROWS,
    },

    "SelectionEvidence":
        selection_evidence,

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "ValidationChecks":
        len(
            selection_validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "LockedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "Status":
        STEP1B_PASS_STATUS,
}

atomic_write_json(
    PROJECT9_CHECKPOINT_PATH,
    checkpoint_payload,
)


# ------------------------------------------------------------
# 23. WRITE REPORT AND STATUS
# ------------------------------------------------------------

selection_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP1B_PASS_STATUS,

    "CandidateRank":
        EXPECTED_CANDIDATE_RANK,

    "SourceFileCount":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAINING_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "RawRows":
        EXPECTED_RAW_ROWS,

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "RawEvaluationRows":
        EXPECTED_RAW_EVALUATION_ROWS,

    "RawTrainingFailures":
        EXPECTED_RAW_TRAIN_FAILURES,

    "RawEvaluationFailures":
        EXPECTED_RAW_EVALUATION_FAILURES,

    "ModelRows":
        EXPECTED_MODEL_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ModelTrainingFailures":
        EXPECTED_MODEL_TRAIN_FAILURES,

    "ModelEvaluationFailures":
        EXPECTED_MODEL_EVALUATION_FAILURES,

    "ScoredEvaluationBuilds":
        len(
            scored_evaluation_builds
        ),

    "ScoredEvaluationRows":
        scored_evaluation_rows,

    "ScoredEvaluationFailures":
        scored_evaluation_failures,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            selection_validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "Checkpoint":
        str(
            PROJECT9_CHECKPOINT_PATH
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

atomic_write_json(
    SELECTION_REPORT_PATH,
    selection_report,
)

step1b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP1B_PASS_STATUS,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAINING_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ScoredEvaluationBuilds":
        len(
            scored_evaluation_builds
        ),

    "ScoredEvaluationRows":
        scored_evaluation_rows,

    "ScoredEvaluationFailures":
        scored_evaluation_failures,

    "Checkpoint":
        str(
            PROJECT9_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            PROJECT9_CHECKPOINT_PATH
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

atomic_write_json(
    STEP1B_STATUS_PATH,
    step1b_status,
)


# ------------------------------------------------------------
# 24. FINAL READBACK
# ------------------------------------------------------------

required_outputs = [
    SOURCE_MANIFEST_PATH,
    BUILD_CHRONOLOGY_PATH,
    FIXED_SPLIT_PATH,
    SCORED_EVALUATION_BUILDS_PATH,
    SELECTION_VALIDATION_PATH,
    SELECTION_REPORT_PATH,
    PROJECT9_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:

    raise RuntimeError(
        "Step 1B outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )

final_checkpoint = json.loads(
    PROJECT9_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

final_status = json.loads(
    STEP1B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    final_checkpoint.get(
        "Status"
    )
    != STEP1B_PASS_STATUS
):

    raise AssertionError(
        "Project 9 checkpoint status differs."
    )

if (
    final_status.get(
        "Status"
    )
    != STEP1B_PASS_STATUS
):

    raise AssertionError(
        "Project 9 Step 1B status differs."
    )


# ------------------------------------------------------------
# 25. DISPLAY FROZEN SCORED BUILD COHORT
# ------------------------------------------------------------

print("\nFrozen scored evaluation-build cohort:")

display(
    scored_evaluation_builds
)


# ------------------------------------------------------------
# 26. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 108)
print("=== PROJECT 9 CELL 2 / STEP 1B RESULT ===")
print("=" * 108)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    EXPECTED_CANDIDATE_RANK,
)


print("\nSource freeze:")

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_total_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)

print(
    "Source files unchanged:",
    source_files_unchanged,
)


print("\nCanonical chronology and split:")

print(
    "Canonical builds:",
    len(
        build_chronology
    ),
)

print(
    "Training builds:",
    len(
        training_build_keys
    ),
)

print(
    "Evaluation builds:",
    len(
        evaluation_build_keys
    ),
)

print(
    "Timestamp parse failures:",
    timestamp_parse_failures,
)

print(
    "Conflicting duplicate builds:",
    conflicting_duplicate_builds,
)


print("\nFixed source cohorts:")

print(
    "Raw rows:",
    len(
        raw_source
    ),
)

print(
    "Raw training / evaluation rows:",
    actual_counts[
        "RawTrainingRows"
    ],
    "/",
    actual_counts[
        "RawEvaluationRows"
    ],
)

print(
    "Raw training / evaluation failures:",
    actual_counts[
        "RawTrainFailures"
    ],
    "/",
    actual_counts[
        "RawEvaluationFailures"
    ],
)

print(
    "Model rows:",
    len(
        model_source
    ),
)

print(
    "Model training / evaluation rows:",
    actual_counts[
        "ModelTrainingRows"
    ],
    "/",
    actual_counts[
        "ModelEvaluationRows"
    ],
)

print(
    "Model training / evaluation failures:",
    actual_counts[
        "ModelTrainFailures"
    ],
    "/",
    actual_counts[
        "ModelEvaluationFailures"
    ],
)


print("\nScored evaluation cohort:")

print(
    "Scored evaluation builds:",
    len(
        scored_evaluation_builds
    ),
)

print(
    "Scored evaluation rows:",
    scored_evaluation_rows,
)

print(
    "Scored evaluation failures:",
    scored_evaluation_failures,
)

print(
    "Raw/model scored-build sets equal:",
    raw_scored_builds
    == model_scored_builds,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        selection_validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nPermanent selection checkpoint:")

print(
    PROJECT9_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        PROJECT9_CHECKPOINT_PATH
    ),
)


print("\nSaved outputs:")

for output_path in required_outputs:

    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 108)

=== PROJECT 9 CELL 2 / STEP 1B: SELECTION LOCK, SOURCE FREEZE AND SPLIT PREFLIGHT ===

Selection and split validation:


,Check,Expected,Actual,Pass
0,Step 1A status,PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE,PASS_PROJECT_9_CANDIDATE_DISCOVERY_COMPLETE,True
1,Selected project,camunda@camunda-bpm-platform,camunda@camunda-bpm-platform,True
2,Candidate rank,1,1,True
3,Source files,5,5,True
4,Canonical builds,822,822,True
5,Training builds,616,616,True
6,Evaluation builds,206,206,True
7,Raw rows,472765,472765,True
8,Raw training rows,313984,313984,True
9,Raw evaluation rows,158781,158781,True



Frozen scored evaluation-build cohort:


,Build,BuildOrder,StartedAt,RawRows,RawFailures,ModelRows,ModelFailures
0,121149283,617,2016-04-06 13:25:02+00:00,551,2,551,2
1,121159550,618,2016-04-06 14:05:06+00:00,551,2,551,2
2,121163441,619,2016-04-06 14:19:14+00:00,551,2,551,2
3,121166609,620,2016-04-06 14:30:31+00:00,551,2,551,2
4,121169737,621,2016-04-06 14:42:09+00:00,551,2,551,2
5,122449871,642,2016-04-12 07:51:49+00:00,557,1,557,1
6,123035563,669,2016-04-14 12:16:52+00:00,48,7,48,7
7,123038249,670,2016-04-14 12:29:42+00:00,558,4,558,4
8,123081030,672,2016-04-14 15:01:51+00:00,559,1,559,1
9,123104273,674,2016-04-14 16:28:25+00:00,785,99,785,99




=== PROJECT 9 CELL 2 / STEP 1B RESULT ===

Project identity:
Project number: 9
Project: camunda@camunda-bpm-platform
Project slug: camunda__camunda-bpm-platform
Candidate rank: 1

Source freeze:
Source files: 5
Source bytes: 105119189
Source root SHA-256: 65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07
Source files unchanged: True

Canonical chronology and split:
Canonical builds: 822
Training builds: 616
Evaluation builds: 206
Timestamp parse failures: 0
Conflicting duplicate builds: 0

Fixed source cohorts:
Raw rows: 472765
Raw training / evaluation rows: 313984 / 158781
Raw training / evaluation failures: 1447 / 665
Model rows: 79383
Model training / evaluation rows: 60880 / 18503
Model training / evaluation failures: 1427 / 665

Scored evaluation cohort:
Scored evaluation builds: 30
Scored evaluation rows: 18503
Scored evaluation failures: 665
Raw/model scored-build sets equal: True

Validation:
Checks: 23
Failed checks: 0

Permanent selection checkpoint:
/conten

In [6]:
# ============================================================
# PROJECT 9 — CELL 3 / STEP 2A
# SOURCE SCHEMA, FEATURE SET AND JOIN-STRUCTURE AUDIT
#
# PROJECT: camunda@camunda-bpm-platform
#
# This cell:
# - validates the permanent Project 9 selection checkpoint
# - independently rehashes all five frozen source files
# - audits every source column and sample dtype
# - freezes the 19 REC feature names
# - separates 13 verdict-dependent and 6 independent REC fields
# - validates model-ready Build/Test uniqueness
# - audits raw-to-model Build/Test coverage
# - audits verdict and duration consistency
# - inspects id_map.csv and entity_change_history.csv structure
#
# This cell does NOT:
# - reconstruct REC features
# - inject noise
# - train models
# - alter the source dataset
# - modify the registry
# - modify Projects 1–8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = (
    "camunda"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

STEP2A_PASS_STATUS = (
    "PASS_PROJECT_9_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)

EXPECTED_MODEL_ROWS = 79383
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

EXPECTED_RAW_ROWS = 472765
EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_RAW_EVALUATION_ROWS = 158781

EXPECTED_BUILDS = 822
EXPECTED_TRAINING_BUILDS = 616
EXPECTED_EVALUATION_BUILDS = 206

RECENT_WINDOW = 6


# ------------------------------------------------------------
# 2. FROZEN REC FEATURE SET
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    & set(VERDICT_INDEPENDENT_REC_FEATURES)
):

    raise AssertionError(
        "Dependent and independent REC sets overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):

    raise AssertionError(
        "REC dependency classes do not cover all 19 features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

SELECTION_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_preflight"
)

FIXED_SPLIT_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_fixed_build_split.csv.gz"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

COLUMN_INVENTORY_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_column_inventory.csv"
)

TABLE_SUMMARY_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_table_summary.csv"
)

KEY_COVERAGE_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_test_key_coverage.csv"
)

JOIN_AUDIT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_model_join_audit.csv"
)

FEATURE_CLASSIFICATION_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_feature_classification.csv"
)

RELATION_SCHEMA_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_relation_schema_audit.csv"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)


print("=" * 108)
print("=== PROJECT 9 CELL 3 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE AUDIT ===")
print("=" * 108)


# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativeName",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            str(row.RelativeName).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(int(row.SizeBytes)).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(row.SHA256)
        )

        digest.update(b"\n")

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def normalise_name(value):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(numeric.notna().mean()) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def detect_column(
    columns,
    candidates,
    role,
    required=True,
):
    lookup = {
        normalise_name(column):
            column
        for column in columns
    }

    for candidate in candidates:

        key = normalise_name(
            candidate
        )

        if key in lookup:

            return lookup[key]

    if required:

        raise RuntimeError(
            "Required source column not found.\n"
            f"Role: {role}\n"
            f"Candidates: {candidates}\n"
            f"Available: {columns}"
        )

    return None


def read_sample(path, rows=5000):

    return pd.read_csv(
        path,
        nrows=rows,
        low_memory=False,
    )


def column_inventory(
    table_name,
    path,
):
    header = pd.read_csv(
        path,
        nrows=0,
    )

    sample = read_sample(
        path,
        rows=5000,
    )

    records = []

    for position, column in enumerate(
        header.columns,
        start=1,
    ):

        series = sample[column]

        records.append({
            "Table":
                table_name,

            "Path":
                str(path),

            "ColumnPosition":
                position,

            "Column":
                column,

            "NormalisedColumn":
                normalise_name(column),

            "SampleDtype":
                str(series.dtype),

            "SampleRows":
                len(series),

            "SampleNonMissing":
                int(series.notna().sum()),

            "SampleUnique":
                int(series.nunique(dropna=True)),

            "ExampleValues":
                " | ".join(
                    series
                    .dropna()
                    .astype(str)
                    .drop_duplicates()
                    .head(5)
                    .tolist()
                ),
        })

    return pd.DataFrame(records)


def serialise_columns(columns):

    return json.dumps(
        list(columns)
    )


# ------------------------------------------------------------
# 5. VALIDATE CHECKPOINT AND REGISTRY
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    PROJECT_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FIXED_SPLIT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required Step 2A inputs are missing:\n"
        + "\n".join(missing_paths)
    )


checkpoint = json.loads(
    PROJECT_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step1b_status = json.loads(
    STEP1B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    checkpoint.get("Status")
    != EXPECTED_STEP1B_STATUS
):

    raise AssertionError(
        "Project 9 checkpoint status differs."
    )


if (
    step1b_status.get("Status")
    != EXPECTED_STEP1B_STATUS
):

    raise AssertionError(
        "Project 9 Step 1B status differs."
    )


if checkpoint.get("Project") != PROJECT_NAME:

    raise AssertionError(
        "Project checkpoint identity differs."
    )


if (
    checkpoint.get("SourceRootSHA256")
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Recorded source-root SHA-256 differs."
    )


registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if (
    len(registry) != 8
    or set(project_numbers)
    != set(range(1, 9))
):

    raise AssertionError(
        "Completion registry does not contain exactly Projects 1–8."
    )

if project_numbers.eq(PROJECT_NUMBER).any():

    raise AssertionError(
        "Project 9 is already present in the registry."
    )


# ------------------------------------------------------------
# 6. REVALIDATE THE FIVE FROZEN SOURCE FILES
# ------------------------------------------------------------

source_file_records = []

for relative_name, metadata in (
    checkpoint["SourceFiles"].items()
):

    path = Path(
        metadata["RuntimePath"]
    )

    if not path.exists():

        raise FileNotFoundError(
            "Frozen Project 9 source file is missing:\n"
            f"{path}"
        )

    actual_size = int(
        path.stat().st_size
    )

    actual_sha256 = calculate_sha256(
        path
    )

    if actual_size != int(
        metadata["SizeBytes"]
    ):

        raise AssertionError(
            "Frozen source-file size differs.\n"
            f"File: {relative_name}"
        )

    if actual_sha256 != metadata[
        "SHA256"
    ]:

        raise AssertionError(
            "Frozen source-file SHA-256 differs.\n"
            f"File: {relative_name}"
        )

    source_file_records.append({
        "RelativeName":
            relative_name,

        "Path":
            str(path),

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


source_manifest = pd.DataFrame(
    source_file_records
)

source_root_sha256 = root_inventory_hash(
    source_manifest
)

if (
    source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Recalculated source-root SHA-256 differs."
    )


source_paths = {
    row.RelativeName:
        Path(row.Path)
    for row in source_manifest.itertuples(
        index=False
    )
}


# ------------------------------------------------------------
# 7. CREATE COMPLETE COLUMN INVENTORY
# ------------------------------------------------------------

column_inventory_frames = []

for table_name in [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
]:

    column_inventory_frames.append(
        column_inventory(
            table_name,
            source_paths[table_name],
        )
    )


source_column_inventory = pd.concat(
    column_inventory_frames,
    ignore_index=True,
)


table_summary_records = []

for table_name, path in (
    source_paths.items()
):

    columns = list(
        pd.read_csv(
            path,
            nrows=0,
        ).columns
    )

    row_count = sum(
        len(chunk)
        for chunk in pd.read_csv(
            path,
            chunksize=250000,
            low_memory=False,
        )
    )

    table_summary_records.append({
        "Table":
            table_name,

        "Path":
            str(path),

        "Rows":
            int(row_count),

        "Columns":
            len(columns),

        "ColumnNames":
            serialise_columns(columns),

        "SizeBytes":
            int(path.stat().st_size),

        "SHA256":
            calculate_sha256(path),
    })


source_table_summary = pd.DataFrame(
    table_summary_records
)


# ------------------------------------------------------------
# 8. RESOLVE EXACT KEY COLUMNS
# ------------------------------------------------------------

builds_columns = list(
    pd.read_csv(
        source_paths["builds.csv"],
        nrows=0,
    ).columns
)

exe_columns = list(
    pd.read_csv(
        source_paths["exe.csv"],
        nrows=0,
    ).columns
)

dataset_columns = list(
    pd.read_csv(
        source_paths["dataset.csv"],
        nrows=0,
    ).columns
)

id_map_columns = list(
    pd.read_csv(
        source_paths["id_map.csv"],
        nrows=0,
    ).columns
)

entity_history_columns = list(
    pd.read_csv(
        source_paths[
            "entity_change_history.csv"
        ],
        nrows=0,
    ).columns
)


builds_build_column = detect_column(
    builds_columns,
    [
        "id",
        "build",
        "Build",
    ],
    "builds.csv build identifier",
)

builds_timestamp_column = detect_column(
    builds_columns,
    [
        "started_at",
        "StartedAt",
    ],
    "builds.csv timestamp",
)

exe_build_column = detect_column(
    exe_columns,
    [
        "build",
        "Build",
    ],
    "exe.csv build identifier",
)

exe_test_column = detect_column(
    exe_columns,
    [
        "test",
        "Test",
        "test_id",
        "TestID",
    ],
    "exe.csv test identifier",
)

exe_verdict_column = detect_column(
    exe_columns,
    [
        "verdict",
        "Verdict",
        "test_result",
    ],
    "exe.csv verdict",
)

exe_duration_column = detect_column(
    exe_columns,
    [
        "duration",
        "Duration",
        "execution_time",
        "ExecutionTime",
    ],
    "exe.csv duration",
)

exe_job_column = detect_column(
    exe_columns,
    [
        "job",
        "Job",
        "job_id",
        "JobID",
    ],
    "exe.csv job identifier",
    required=False,
)


dataset_build_column = detect_column(
    dataset_columns,
    [
        "Build",
        "build",
    ],
    "dataset.csv build identifier",
)

dataset_test_column = detect_column(
    dataset_columns,
    [
        "Test",
        "test",
        "test_id",
        "TestID",
    ],
    "dataset.csv test identifier",
)

dataset_verdict_column = detect_column(
    dataset_columns,
    [
        "Verdict",
        "verdict",
    ],
    "dataset.csv verdict",
)

dataset_duration_column = detect_column(
    dataset_columns,
    [
        "Duration",
        "duration",
        "ExecutionTime",
        "execution_time",
    ],
    "dataset.csv duration",
    required=False,
)


# ------------------------------------------------------------
# 9. VALIDATE THE 19 REC COLUMNS
# ------------------------------------------------------------

missing_rec_columns = [
    column
    for column in REC_FEATURE_COLUMNS
    if column not in dataset_columns
]

unexpected_rec_columns = [
    column
    for column in dataset_columns
    if (
        column.startswith("REC_")
        and column not in REC_FEATURE_COLUMNS
    )
]

if missing_rec_columns:

    raise AssertionError(
        "dataset.csv is missing frozen REC fields:\n"
        + "\n".join(missing_rec_columns)
    )

if unexpected_rec_columns:

    raise AssertionError(
        "Unexpected REC fields were detected:\n"
        + "\n".join(unexpected_rec_columns)
    )


feature_classification_records = []

identifier_columns = {
    dataset_build_column,
    dataset_test_column,
    dataset_verdict_column,
}

if dataset_duration_column is not None:
    identifier_columns.add(
        dataset_duration_column
    )


for column in dataset_columns:

    if column in REC_FEATURE_COLUMNS:

        if column in (
            VERDICT_DEPENDENT_REC_FEATURES
        ):

            feature_class = (
                "REC_VERDICT_DEPENDENT"
            )

        else:

            feature_class = (
                "REC_VERDICT_INDEPENDENT"
            )

        model_predictor_candidate = True

    elif column in identifier_columns:

        feature_class = (
            "IDENTIFIER_LABEL_OR_COST"
        )

        model_predictor_candidate = False

    else:

        feature_class = (
            "NON_REC_PREDICTOR_CANDIDATE"
        )

        model_predictor_candidate = True


    feature_classification_records.append({
        "Column":
            column,

        "FeatureClass":
            feature_class,

        "ModelPredictorCandidate":
            model_predictor_candidate,

        "VerdictDependent":
            column in (
                VERDICT_DEPENDENT_REC_FEATURES
            ),

        "VerdictIndependent":
            column in (
                VERDICT_INDEPENDENT_REC_FEATURES
            ),
    })


feature_classification = pd.DataFrame(
    feature_classification_records
)


# ------------------------------------------------------------
# 10. LOAD FROZEN SPLIT AND CORE TABLES
# ------------------------------------------------------------

fixed_split = pd.read_csv(
    FIXED_SPLIT_PATH,
    low_memory=False,
)

fixed_split[
    "BuildKey"
] = canonical_identifier(
    fixed_split["Build"]
)

training_build_keys = set(
    fixed_split.loc[
        fixed_split["Partition"].eq(
            "TRAINING"
        ),
        "BuildKey",
    ]
)

evaluation_build_keys = set(
    fixed_split.loc[
        fixed_split["Partition"].eq(
            "EVALUATION"
        ),
        "BuildKey",
    ]
)


if len(fixed_split) != EXPECTED_BUILDS:

    raise AssertionError(
        "Frozen split build count differs."
    )

if len(training_build_keys) != (
    EXPECTED_TRAINING_BUILDS
):

    raise AssertionError(
        "Frozen training-build count differs."
    )

if len(evaluation_build_keys) != (
    EXPECTED_EVALUATION_BUILDS
):

    raise AssertionError(
        "Frozen evaluation-build count differs."
    )


exe_usecols = [
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]

if exe_job_column is not None:
    exe_usecols.append(exe_job_column)


raw_data = pd.read_csv(
    source_paths["exe.csv"],
    usecols=exe_usecols,
    low_memory=False,
)

model_data = pd.read_csv(
    source_paths["dataset.csv"],
    low_memory=False,
)


raw_data["BuildKey"] = (
    canonical_identifier(
        raw_data[exe_build_column]
    )
)

raw_data["TestKey"] = (
    canonical_identifier(
        raw_data[exe_test_column]
    )
)

model_data["BuildKey"] = (
    canonical_identifier(
        model_data[dataset_build_column]
    )
)

model_data["TestKey"] = (
    canonical_identifier(
        model_data[dataset_test_column]
    )
)


raw_data["Partition"] = np.select(
    [
        raw_data["BuildKey"].isin(
            training_build_keys
        ),
        raw_data["BuildKey"].isin(
            evaluation_build_keys
        ),
    ],
    [
        "TRAINING",
        "EVALUATION",
    ],
    default="UNLINKED",
)

model_data["Partition"] = np.select(
    [
        model_data["BuildKey"].isin(
            training_build_keys
        ),
        model_data["BuildKey"].isin(
            evaluation_build_keys
        ),
    ],
    [
        "TRAINING",
        "EVALUATION",
    ],
    default="UNLINKED",
)


# ------------------------------------------------------------
# 11. KEY UNIQUENESS AND COVERAGE
# ------------------------------------------------------------

model_duplicate_rows = int(
    model_data.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)

if model_duplicate_rows != 0:

    raise AssertionError(
        "Model-ready Build/Test rows are not unique."
    )


raw_key_summary = (
    raw_data
    .groupby(
        [
            "BuildKey",
            "TestKey",
        ],
        as_index=False,
    )
    .agg(
        RawRows=(
            "BuildKey",
            "size",
        ),

        RawVerdictVariants=(
            exe_verdict_column,
            "nunique",
        ),

        RawDurationVariants=(
            exe_duration_column,
            "nunique",
        ),

        RawFirstVerdict=(
            exe_verdict_column,
            "first",
        ),

        RawFirstDuration=(
            exe_duration_column,
            "first",
        ),
    )
)


model_key_frame = (
    model_data[
        [
            "BuildKey",
            "TestKey",
            dataset_verdict_column,
        ]
        + (
            [dataset_duration_column]
            if dataset_duration_column
            is not None
            else []
        )
    ]
    .copy()
)


model_raw_join = (
    model_key_frame
    .merge(
        raw_key_summary,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


model_keys_missing_from_raw = int(
    model_raw_join["_merge"]
    .ne("both")
    .sum()
)

model_keys_with_multiple_raw_rows = int(
    model_raw_join[
        "RawRows"
    ].fillna(0).gt(1).sum()
)

model_keys_with_raw_verdict_variants = int(
    model_raw_join[
        "RawVerdictVariants"
    ].fillna(0).gt(1).sum()
)

model_keys_with_raw_duration_variants = int(
    model_raw_join[
        "RawDurationVariants"
    ].fillna(0).gt(1).sum()
)


model_verdict_numeric = pd.to_numeric(
    model_raw_join[
        dataset_verdict_column
    ],
    errors="coerce",
)

raw_verdict_numeric = pd.to_numeric(
    model_raw_join[
        "RawFirstVerdict"
    ],
    errors="coerce",
)

verdict_comparable = (
    model_raw_join["_merge"].eq("both")
    &
    model_raw_join[
        "RawVerdictVariants"
    ].eq(1)
)

verdict_mismatches = int(
    (
        verdict_comparable
        &
        ~np.isclose(
            model_verdict_numeric,
            raw_verdict_numeric,
            rtol=0,
            atol=0,
            equal_nan=True,
        )
    ).sum()
)


duration_mismatches = np.nan

if dataset_duration_column is not None:

    model_duration_numeric = pd.to_numeric(
        model_raw_join[
            dataset_duration_column
        ],
        errors="coerce",
    )

    raw_duration_numeric = pd.to_numeric(
        model_raw_join[
            "RawFirstDuration"
        ],
        errors="coerce",
    )

    duration_comparable = (
        model_raw_join["_merge"].eq("both")
        &
        model_raw_join[
            "RawDurationVariants"
        ].eq(1)
    )

    duration_mismatches = int(
        (
            duration_comparable
            &
            ~np.isclose(
                model_duration_numeric,
                raw_duration_numeric,
                rtol=1e-9,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


# ------------------------------------------------------------
# 12. PARTITION AND ROW-COUNT AUDIT
# ------------------------------------------------------------

key_coverage_records = [
    {
        "Table":
            "exe.csv",

        "Rows":
            len(raw_data),

        "UniqueBuilds":
            raw_data[
                "BuildKey"
            ].nunique(),

        "UniqueTests":
            raw_data[
                "TestKey"
            ].nunique(),

        "UniqueBuildTestKeys":
            raw_data[
                [
                    "BuildKey",
                    "TestKey",
                ]
            ].drop_duplicates().shape[0],

        "DuplicateBuildTestRows":
            int(
                raw_data.duplicated(
                    subset=[
                        "BuildKey",
                        "TestKey",
                    ],
                    keep=False,
                ).sum()
            ),

        "TrainingRows":
            int(
                raw_data[
                    "Partition"
                ].eq("TRAINING").sum()
            ),

        "EvaluationRows":
            int(
                raw_data[
                    "Partition"
                ].eq("EVALUATION").sum()
            ),

        "UnlinkedRows":
            int(
                raw_data[
                    "Partition"
                ].eq("UNLINKED").sum()
            ),
    },

    {
        "Table":
            "dataset.csv",

        "Rows":
            len(model_data),

        "UniqueBuilds":
            model_data[
                "BuildKey"
            ].nunique(),

        "UniqueTests":
            model_data[
                "TestKey"
            ].nunique(),

        "UniqueBuildTestKeys":
            model_data[
                [
                    "BuildKey",
                    "TestKey",
                ]
            ].drop_duplicates().shape[0],

        "DuplicateBuildTestRows":
            model_duplicate_rows,

        "TrainingRows":
            int(
                model_data[
                    "Partition"
                ].eq("TRAINING").sum()
            ),

        "EvaluationRows":
            int(
                model_data[
                    "Partition"
                ].eq("EVALUATION").sum()
            ),

        "UnlinkedRows":
            int(
                model_data[
                    "Partition"
                ].eq("UNLINKED").sum()
            ),
    },
]


key_coverage = pd.DataFrame(
    key_coverage_records
)


join_audit = pd.DataFrame([
    {
        "Check":
            "Model Build/Test keys",

        "Value":
            len(model_data),
    },

    {
        "Check":
            "Model keys found in raw executions",

        "Value":
            int(
                model_raw_join[
                    "_merge"
                ].eq("both").sum()
            ),
    },

    {
        "Check":
            "Model keys missing from raw executions",

        "Value":
            model_keys_missing_from_raw,
    },

    {
        "Check":
            "Model keys with multiple raw rows",

        "Value":
            model_keys_with_multiple_raw_rows,
    },

    {
        "Check":
            "Model keys with raw verdict variants",

        "Value":
            model_keys_with_raw_verdict_variants,
    },

    {
        "Check":
            "Model keys with raw duration variants",

        "Value":
            model_keys_with_raw_duration_variants,
    },

    {
        "Check":
            "Comparable verdict mismatches",

        "Value":
            verdict_mismatches,
    },

    {
        "Check":
            "Comparable duration mismatches",

        "Value":
            duration_mismatches,
    },
])


# ------------------------------------------------------------
# 13. RELATION-TABLE SCHEMA AUDIT
# ------------------------------------------------------------

id_map_sample = read_sample(
    source_paths["id_map.csv"],
    rows=20,
)

entity_history_sample = read_sample(
    source_paths[
        "entity_change_history.csv"
    ],
    rows=20,
)


relation_schema = pd.DataFrame([
    {
        "Table":
            "id_map.csv",

        "Rows":
            int(
                source_table_summary.loc[
                    source_table_summary[
                        "Table"
                    ].eq("id_map.csv"),
                    "Rows",
                ].iloc[0]
            ),

        "Columns":
            len(id_map_columns),

        "ColumnNames":
            serialise_columns(
                id_map_columns
            ),

        "SampleJSON":
            id_map_sample.head(5)
            .to_json(
                orient="records"
            ),
    },

    {
        "Table":
            "entity_change_history.csv",

        "Rows":
            int(
                source_table_summary.loc[
                    source_table_summary[
                        "Table"
                    ].eq(
                        "entity_change_history.csv"
                    ),
                    "Rows",
                ].iloc[0]
            ),

        "Columns":
            len(entity_history_columns),

        "ColumnNames":
            serialise_columns(
                entity_history_columns
            ),

        "SampleJSON":
            entity_history_sample.head(5)
            .to_json(
                orient="records"
            ),
    },
])


# ------------------------------------------------------------
# 14. VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status["Status"],

        "Pass":
            step1b_status["Status"]
            == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Frozen source files",

        "Expected":
            5,

        "Actual":
            len(source_manifest),

        "Pass":
            len(source_manifest) == 5,
    },

    {
        "Check":
            "REC feature count",

        "Expected":
            19,

        "Actual":
            len(REC_FEATURE_COLUMNS),

        "Pass":
            len(REC_FEATURE_COLUMNS) == 19,
    },

    {
        "Check":
            "Verdict-dependent REC features",

        "Expected":
            13,

        "Actual":
            len(
                VERDICT_DEPENDENT_REC_FEATURES
            ),

        "Pass":
            len(
                VERDICT_DEPENDENT_REC_FEATURES
            ) == 13,
    },

    {
        "Check":
            "Verdict-independent REC features",

        "Expected":
            6,

        "Actual":
            len(
                VERDICT_INDEPENDENT_REC_FEATURES
            ),

        "Pass":
            len(
                VERDICT_INDEPENDENT_REC_FEATURES
            ) == 6,
    },

    {
        "Check":
            "Dataset rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(model_data),

        "Pass":
            len(model_data)
            == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Dataset training rows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            int(
                model_data[
                    "Partition"
                ].eq("TRAINING").sum()
            ),

        "Pass":
            int(
                model_data[
                    "Partition"
                ].eq("TRAINING").sum()
            )
            == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Check":
            "Dataset evaluation rows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            int(
                model_data[
                    "Partition"
                ].eq("EVALUATION").sum()
            ),

        "Pass":
            int(
                model_data[
                    "Partition"
                ].eq("EVALUATION").sum()
            )
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Check":
            "Raw rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(raw_data),

        "Pass":
            len(raw_data)
            == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Raw training rows",

        "Expected":
            EXPECTED_RAW_TRAINING_ROWS,

        "Actual":
            int(
                raw_data[
                    "Partition"
                ].eq("TRAINING").sum()
            ),

        "Pass":
            int(
                raw_data[
                    "Partition"
                ].eq("TRAINING").sum()
            )
            == EXPECTED_RAW_TRAINING_ROWS,
    },

    {
        "Check":
            "Raw evaluation rows",

        "Expected":
            EXPECTED_RAW_EVALUATION_ROWS,

        "Actual":
            int(
                raw_data[
                    "Partition"
                ].eq("EVALUATION").sum()
            ),

        "Pass":
            int(
                raw_data[
                    "Partition"
                ].eq("EVALUATION").sum()
            )
            == EXPECTED_RAW_EVALUATION_ROWS,
    },

    {
        "Check":
            "Model duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            model_duplicate_rows,

        "Pass":
            model_duplicate_rows == 0,
    },

    {
        "Check":
            "Model Build/Test keys missing from raw",

        "Expected":
            0,

        "Actual":
            model_keys_missing_from_raw,

        "Pass":
            model_keys_missing_from_raw == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation["Pass"]
].copy()


print("\nStep 2A validation:")

display(validation)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(failed_checks)

    raise RuntimeError(
        "PROJECT 9 STEP 2A DID NOT PASS."
    )


# ------------------------------------------------------------
# 15. VERIFY IMMUTABILITY
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_sha256(path)
    for relative_name, path
    in source_paths.items()
}

source_hashes_before = {
    row.RelativeName:
        row.SHA256
    for row in source_manifest.itertuples(
        index=False
    )
}

if source_hashes_after != (
    source_hashes_before
):

    raise AssertionError(
        "A frozen source file changed during Step 2A."
    )


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)

if (
    registry_sha256_after
    != registry_sha256_before
):

    raise AssertionError(
        "Completion registry changed during Step 2A."
    )


# ------------------------------------------------------------
# 16. WRITE OUTPUTS
# ------------------------------------------------------------

SCHEMA_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_write_csv(
    COLUMN_INVENTORY_PATH,
    source_column_inventory,
)

atomic_write_csv(
    TABLE_SUMMARY_PATH,
    source_table_summary,
)

atomic_write_csv(
    KEY_COVERAGE_PATH,
    key_coverage,
)

atomic_write_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_write_csv(
    FEATURE_CLASSIFICATION_PATH,
    feature_classification,
)

atomic_write_csv(
    RELATION_SCHEMA_PATH,
    relation_schema,
)


report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "SourceFiles":
        len(source_manifest),

    "SourceTables": {
        row.Table: {
            "Rows":
                int(row.Rows),

            "Columns":
                int(row.Columns),

            "ColumnNames":
                json.loads(
                    row.ColumnNames
                ),
        }
        for row in (
            source_table_summary
            .itertuples(index=False)
        )
    },

    "ResolvedColumns": {
        "BuildsBuild":
            builds_build_column,

        "BuildsTimestamp":
            builds_timestamp_column,

        "ExecutionBuild":
            exe_build_column,

        "ExecutionTest":
            exe_test_column,

        "ExecutionVerdict":
            exe_verdict_column,

        "ExecutionDuration":
            exe_duration_column,

        "ExecutionJob":
            exe_job_column,

        "DatasetBuild":
            dataset_build_column,

        "DatasetTest":
            dataset_test_column,

        "DatasetVerdict":
            dataset_verdict_column,

        "DatasetDuration":
            dataset_duration_column,
    },

    "RECFeatureProtocol": {
        "RecentWindow":
            RECENT_WINDOW,

        "AllRECFeatures":
            REC_FEATURE_COLUMNS,

        "VerdictDependent":
            VERDICT_DEPENDENT_REC_FEATURES,

        "VerdictIndependent":
            VERDICT_INDEPENDENT_REC_FEATURES,
    },

    "KeyAudit": {
        "ModelDuplicateBuildTestRows":
            model_duplicate_rows,

        "ModelKeysMissingFromRaw":
            model_keys_missing_from_raw,

        "ModelKeysWithMultipleRawRows":
            model_keys_with_multiple_raw_rows,

        "ModelKeysWithRawVerdictVariants":
            model_keys_with_raw_verdict_variants,

        "ModelKeysWithRawDurationVariants":
            model_keys_with_raw_duration_variants,

        "ComparableVerdictMismatches":
            verdict_mismatches,

        "ComparableDurationMismatches":
            json_safe(duration_mismatches),
    },

    "IDMapColumns":
        id_map_columns,

    "EntityChangeHistoryColumns":
        entity_history_columns,

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_checks),

    "SourceFilesUnchanged":
        True,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2A_REPORT_PATH,
    report,
)


status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "RECFeatures":
        len(REC_FEATURE_COLUMNS),

    "VerdictDependentRECFeatures":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatures":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "ModelRows":
        len(model_data),

    "ModelTrainingRows":
        int(
            model_data[
                "Partition"
            ].eq("TRAINING").sum()
        ),

    "ModelEvaluationRows":
        int(
            model_data[
                "Partition"
            ].eq("EVALUATION").sum()
        ),

    "ModelDuplicateBuildTestRows":
        model_duplicate_rows,

    "ModelKeysMissingFromRaw":
        model_keys_missing_from_raw,

    "FailedValidationChecks":
        len(failed_checks),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2A_STATUS_PATH,
    status,
)


# ------------------------------------------------------------
# 17. DISPLAY STRUCTURE NEEDED FOR STEP 2B
# ------------------------------------------------------------

print("\nSource table summary:")

display(
    source_table_summary[
        [
            "Table",
            "Rows",
            "Columns",
            "SizeBytes",
        ]
    ]
)


print("\nBuild/Test key coverage:")

display(key_coverage)


print("\nRaw-to-model join audit:")

display(join_audit)


print("\nid_map.csv sample:")

display(id_map_sample)


print("\nentity_change_history.csv sample:")

display(entity_history_sample)


print("\nNon-REC dataset columns:")

display(
    feature_classification[
        ~feature_classification[
            "Column"
        ].isin(REC_FEATURE_COLUMNS)
    ]
)


# ------------------------------------------------------------
# 18. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 108)
print("=== PROJECT 9 CELL 3 / STEP 2A RESULT ===")
print("=" * 108)

print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nFeature protocol:")

print(
    "REC features:",
    len(REC_FEATURE_COLUMNS),
)

print(
    "Verdict-dependent REC features:",
    len(
        VERDICT_DEPENDENT_REC_FEATURES
    ),
)

print(
    "Verdict-independent REC features:",
    len(
        VERDICT_INDEPENDENT_REC_FEATURES
    ),
)

print(
    "Recent window:",
    RECENT_WINDOW,
)


print("\nCore row structure:")

print(
    "Raw rows:",
    len(raw_data),
)

print(
    "Model rows:",
    len(model_data),
)

print(
    "Model duplicate Build/Test rows:",
    model_duplicate_rows,
)

print(
    "Model keys missing from raw:",
    model_keys_missing_from_raw,
)

print(
    "Model keys with multiple raw rows:",
    model_keys_with_multiple_raw_rows,
)

print(
    "Comparable verdict mismatches:",
    verdict_mismatches,
)

print(
    "Comparable duration mismatches:",
    duration_mismatches,
)


print("\nRelation tables:")

print(
    "id_map.csv columns:",
    id_map_columns,
)

print(
    "entity_change_history.csv columns:",
    entity_history_columns,
)


print("\nValidation:")

print(
    "Checks:",
    len(validation),
)

print(
    "Failed checks:",
    len(failed_checks),
)


print("\nSaved outputs:")

for output_path in [
    COLUMN_INVENTORY_PATH,
    TABLE_SUMMARY_PATH,
    KEY_COVERAGE_PATH,
    JOIN_AUDIT_PATH,
    FEATURE_CLASSIFICATION_PATH,
    RELATION_SCHEMA_PATH,
    STEP2A_REPORT_PATH,
    STEP2A_STATUS_PATH,
]:

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP2A_PASS_STATUS,
)

print("=" * 108)

=== PROJECT 9 CELL 3 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE AUDIT ===

Step 2A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_...,PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_...,True
1,Source root SHA-256,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,True
2,Frozen source files,5,5,True
3,REC feature count,19,19,True
4,Verdict-dependent REC features,13,13,True
5,Verdict-independent REC features,6,6,True
6,Dataset rows,79383,79383,True
7,Dataset training rows,60880,60880,True
8,Dataset evaluation rows,18503,18503,True
9,Raw rows,472765,472765,True



Source table summary:


,Table,Rows,Columns,SizeBytes
0,builds.csv,822,3,72316
1,exe.csv,472765,5,14847830
2,dataset.csv,79383,154,61556683
3,id_map.csv,50190,2,4967668
4,entity_change_history.csv,267783,8,23674692



Build/Test key coverage:


...


entity_change_history.csv sample:


,EntityId,AddedLines,DeletedLines,Contributor,BugFix,Commit,CommitDate,MergeCommit
0,1,8,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
1,517,239,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
2,518,65,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
3,519,61,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
4,520,156,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
5,521,186,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
6,522,72,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
7,523,39,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
8,524,54,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False
9,525,64,0,1,0,83a92c062a4b641a6d9203cf84b9195c609e118f,2010-06-18 12:54:32+00:00,False



Non-REC dataset columns:


,Column,FeatureClass,ModelPredictorCandidate,VerdictDependent,VerdictIndependent
0,Build,IDENTIFIER_LABEL_OR_COST,False,False,False
1,Test,IDENTIFIER_LABEL_OR_COST,False,False,False
2,TES_COM_CountDeclFunction,NON_REC_PREDICTOR_CANDIDATE,True,False,False
3,TES_COM_CountLine,NON_REC_PREDICTOR_CANDIDATE,True,False,False
4,TES_COM_CountLineBlank,NON_REC_PREDICTOR_CANDIDATE,True,False,False
...,...,...,...,...,...
149,COD_COV_PRO_IMP_MinorContributorCount,NON_REC_PREDICTOR_CANDIDATE,True,False,False
150,COD_COV_PRO_IMP_OwnersExperience,NON_REC_PREDICTOR_CANDIDATE,True,False,False
151,COD_COV_PRO_IMP_AllCommitersExperience,NON_REC_PREDICTOR_CANDIDATE,True,False,False
152,DET_COV_C_Faults,NON_REC_PREDICTOR_CANDIDATE,True,False,False




=== PROJECT 9 CELL 3 / STEP 2A RESULT ===

Project:
camunda@camunda-bpm-platform
Source root SHA-256: 65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07

Feature protocol:
REC features: 19
Verdict-dependent REC features: 13
Verdict-independent REC features: 6
Recent window: 6

Core row structure:
Raw rows: 472765
Model rows: 79383
Model duplicate Build/Test rows: 0
Model keys missing from raw: 0
Model keys with multiple raw rows: 0
Comparable verdict mismatches: 0
Comparable duration mismatches: 0

Relation tables:
id_map.csv columns: ['key', 'value']
entity_change_history.csv columns: ['EntityId', 'AddedLines', 'DeletedLines', 'Contributor', 'BugFix', 'Commit', 'CommitDate', 'MergeCommit']

Validation:
Checks: 15
Failed checks: 0

Saved outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/camunda__camunda-bpm-platform/camunda_schema_preflight/camunda_source_column_inventory.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/camunda__camunda

In [7]:
# ============================================================
# PROJECT 9 — CELL 4 / STEP 2B
# CLEAN REC RECONSTRUCTION AND CLEAN-ANCHOR FREEZE
#
# PROJECT: camunda@camunda-bpm-platform
#
# This cell:
# - validates Steps 1B and 2A
# - independently revalidates the five frozen source files
# - resolves builds to commits and changed source entities
# - reconstructs all 19 REC features from clean exe.csv history
# - uses only executions before each current Build/Test row
# - validates the reconstruction against dataset.csv
# - freezes per-row clean-anchor offsets
#
# Frozen noisy-feature formula for later steps:
#
#   noisy_feature =
#       original_clean_feature
#       + (
#           direct_noisy_reconstruction
#           - direct_clean_reconstruction
#         )
#
# Equivalent:
#
#   noisy_feature =
#       direct_noisy_reconstruction
#       + clean_anchor_offset
#
# This guarantees exact reproduction at 0% noise while allowing
# verdict-dependent REC values to change under corrupted history.
#
# This cell does NOT:
# - inject noise
# - train models
# - alter source files
# - alter Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from IPython.display import display

import hashlib
import json
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = (
    "camunda"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_9_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

STEP2B_PASS_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)

EXPECTED_BUILDS = 822
EXPECTED_RAW_ROWS = 472765
EXPECTED_MODEL_ROWS = 79383

EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

RECENT_WINDOW = 6

COMPARISON_RTOL = 1e-9
COMPARISON_ATOL = 1e-9
OFFSET_ZERO_ATOL = 1e-12


# ------------------------------------------------------------
# 2. FROZEN REC FEATURE PROTOCOL
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


FILE_HISTORY_REC_FEATURES = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


NON_FILE_HISTORY_REC_FEATURES = [
    column
    for column in REC_FEATURE_COLUMNS
    if column not in FILE_HISTORY_REC_FEATURES
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "The frozen REC protocol must contain 19 features."
    )


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:
    raise AssertionError(
        "The frozen protocol must contain 13 "
        "verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:
    raise AssertionError(
        "The frozen protocol must contain six "
        "verdict-independent REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    & set(VERDICT_INDEPENDENT_REC_FEATURES)
):
    raise AssertionError(
        "REC feature dependency classes overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "REC dependency classes do not cover all features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT_SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

SELECTION_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_preflight"
)

FIXED_SPLIT_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_fixed_build_split.csv.gz"
)

REC_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_rec_preflight"
)

BUILD_COMMIT_TOKEN_PROFILE_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_commit_token_profile.csv"
)

COMMIT_MATCHING_AUDIT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_commit_matching_audit.csv"
)

BUILD_ENTITY_MAP_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_summary.json"
)

CLEAN_REC_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_reconstructed.parquet"
)

CLEAN_REC_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_anchor_offsets.parquet"
)

REC_COMPARISON_SUMMARY_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_comparison_summary.csv"
)

REC_MISMATCH_EXAMPLES_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_mismatch_examples.csv"
)

ANCHOR_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_anchor_validation.csv"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)


print("=" * 110)
print("=== PROJECT 9 CELL 4 / STEP 2B: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===")
print("=" * 110)


# ------------------------------------------------------------
# 4. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativeName",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            str(row.RelativeName).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(int(row.SizeBytes)).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(row.SHA256)
        )

        digest.update(b"\n")

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if compression == "gzip":

        temporary_path = path.with_name(
            path.name + ".tmp.gz"
        )

    else:

        temporary_path = path.with_name(
            path.name + ".tmp"
        )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def normalise_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(numeric.notna().mean()) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def detect_column(
    columns,
    candidates,
    role,
    required=True,
):
    lookup = {
        normalise_name(column):
            column
        for column in columns
    }

    for candidate in candidates:

        key = normalise_name(
            candidate
        )

        if key in lookup:

            return lookup[key]

    if required:

        raise RuntimeError(
            "Required column not found.\n"
            f"Role: {role}\n"
            f"Candidates: {candidates}\n"
            f"Available: {columns}"
        )

    return None


def normalise_commit(
    value,
):
    if pd.isna(value):
        return ""

    return str(value).strip().lower()


def parse_commit_tokens(
    value,
):
    """
    Extract hexadecimal Git commit tokens from builds.csv.

    Supports:
    - one full commit hash;
    - abbreviated hashes;
    - comma/semicolon/space separated hashes;
    - JSON-like lists.
    """

    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text:
        return []

    tokens = re.findall(
        r"(?i)(?<![0-9a-f])[0-9a-f]{7,40}(?![0-9a-f])",
        text,
    )

    if not tokens:

        rough_tokens = re.split(
            r"[\s,;|\[\]\(\)\"']+",
            text,
        )

        tokens = [
            token
            for token in rough_tokens
            if re.fullmatch(
                r"(?i)[0-9a-f]{7,40}",
                token,
            )
        ]

    output = []
    seen = set()

    for token in tokens:

        token = token.lower()

        if token not in seen:

            seen.add(token)
            output.append(token)

    return output


def finite_max_absolute_difference(
    values,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    finite = numeric[
        np.isfinite(numeric)
    ]

    if len(finite) == 0:
        return np.nan

    return float(
        np.max(finite)
    )


# ------------------------------------------------------------
# 5. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    PROJECT_SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FIXED_SPLIT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required Step 2B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_checkpoint = json.loads(
    PROJECT_SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step1b_status = json.loads(
    STEP1B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_status = json.loads(
    STEP2A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step1b_status.get("Status")
    != EXPECTED_STEP1B_STATUS
):

    raise AssertionError(
        "Project 9 Step 1B has not passed."
    )


if (
    step2a_status.get("Status")
    != EXPECTED_STEP2A_STATUS
):

    raise AssertionError(
        "Project 9 Step 2A has not passed."
    )


if (
    selection_checkpoint.get("Project")
    != PROJECT_NAME
):

    raise AssertionError(
        "Project 9 checkpoint identity differs."
    )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Project 9 source-root SHA-256 differs."
    )


registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if (
    len(registry) != 8
    or set(project_numbers)
    != set(range(1, 9))
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the registry."
    )


# ------------------------------------------------------------
# 6. REVALIDATE FROZEN SOURCE FILES
# ------------------------------------------------------------

source_records = []

for relative_name, metadata in (
    selection_checkpoint[
        "SourceFiles"
    ].items()
):

    source_path = Path(
        metadata[
            "RuntimePath"
        ]
    )

    if not source_path.exists():

        raise FileNotFoundError(
            "Frozen source file is missing:\n"
            f"{source_path}"
        )

    actual_size = int(
        source_path.stat().st_size
    )

    actual_sha256 = calculate_sha256(
        source_path
    )

    if actual_size != int(
        metadata[
            "SizeBytes"
        ]
    ):

        raise AssertionError(
            "Frozen source-file size differs.\n"
            f"File: {relative_name}"
        )

    if actual_sha256 != metadata[
        "SHA256"
    ]:

        raise AssertionError(
            "Frozen source-file SHA-256 differs.\n"
            f"File: {relative_name}"
        )

    source_records.append({
        "RelativeName":
            relative_name,

        "Path":
            str(source_path),

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


source_manifest = pd.DataFrame(
    source_records
)

source_root_sha256 = root_inventory_hash(
    source_manifest
)

if (
    source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Recalculated source-root SHA-256 differs."
    )


source_paths = {
    row.RelativeName:
        Path(row.Path)
    for row in source_manifest.itertuples(
        index=False
    )
}


source_hashes_before = {
    row.RelativeName:
        row.SHA256
    for row in source_manifest.itertuples(
        index=False
    )
}


# ------------------------------------------------------------
# 7. LOAD RESOLVED SOURCE SCHEMA
# ------------------------------------------------------------

resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


builds_build_column = (
    resolved_columns[
        "BuildsBuild"
    ]
)

builds_timestamp_column = (
    resolved_columns[
        "BuildsTimestamp"
    ]
)

exe_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

exe_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

exe_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

exe_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

exe_job_column = (
    resolved_columns.get(
        "ExecutionJob"
    )
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


builds_columns = list(
    pd.read_csv(
        source_paths[
            "builds.csv"
        ],
        nrows=0,
    ).columns
)


build_commit_column = detect_column(
    builds_columns,
    [
        "commits",
        "commit",
        "commit_sha",
        "commit_hash",
        "commit_ids",
        "sha",
        "revision",
    ],
    role=(
        "builds.csv commit field"
    ),
)


entity_columns = list(
    pd.read_csv(
        source_paths[
            "entity_change_history.csv"
        ],
        nrows=0,
    ).columns
)


entity_id_column = detect_column(
    entity_columns,
    [
        "EntityId",
        "entity_id",
        "EntityID",
        "id",
    ],
    role=(
        "entity_change_history.csv entity identifier"
    ),
)


entity_commit_column = detect_column(
    entity_columns,
    [
        "Commit",
        "commit",
        "CommitHash",
        "commit_hash",
        "sha",
    ],
    role=(
        "entity_change_history.csv commit identifier"
    ),
)


id_map_columns = list(
    pd.read_csv(
        source_paths[
            "id_map.csv"
        ],
        nrows=0,
    ).columns
)


id_map_key_column = detect_column(
    id_map_columns,
    [
        "key",
        "path",
        "entity",
        "name",
    ],
    role=(
        "id_map.csv entity path"
    ),
)


id_map_value_column = detect_column(
    id_map_columns,
    [
        "value",
        "id",
        "EntityId",
        "entity_id",
    ],
    role=(
        "id_map.csv entity identifier"
    ),
)


print("\nResolved reconstruction schema:")

print(
    "Build commit column:",
    build_commit_column,
)

print(
    "Entity-history commit column:",
    entity_commit_column,
)

print(
    "Entity-history ID column:",
    entity_id_column,
)

print(
    "ID-map key / value:",
    id_map_key_column,
    "/",
    id_map_value_column,
)


# ------------------------------------------------------------
# 8. LOAD FROZEN BUILD CHRONOLOGY
# ------------------------------------------------------------

fixed_split = pd.read_csv(
    FIXED_SPLIT_PATH,
    low_memory=False,
)

if len(
    fixed_split
) != EXPECTED_BUILDS:

    raise AssertionError(
        "Frozen build split does not contain 822 builds."
    )


fixed_split[
    "BuildKey"
] = canonical_identifier(
    fixed_split[
        "Build"
    ]
)


fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


if not fixed_split[
    "BuildKey"
].is_unique:

    raise AssertionError(
        "Frozen build chronology is not unique."
    )


build_order_map = (
    fixed_split
    .set_index(
        "BuildKey"
    )[
        "BuildOrder"
    ]
    .to_dict()
)


global_build_position = {
    build_key:
        int(build_order) - 1
    for build_key, build_order
    in build_order_map.items()
}


# ------------------------------------------------------------
# 9. BUILD → COMMIT TOKEN PROFILE
# ------------------------------------------------------------

builds_commit_data = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    usecols=[
        builds_build_column,
        builds_timestamp_column,
        build_commit_column,
    ],
    low_memory=False,
)


builds_commit_data[
    "BuildKey"
] = canonical_identifier(
    builds_commit_data[
        builds_build_column
    ]
)


builds_commit_data[
    "BuildOrder"
] = builds_commit_data[
    "BuildKey"
].map(
    build_order_map
)


if builds_commit_data[
    "BuildOrder"
].isna().any():

    raise AssertionError(
        "A builds.csv row could not be mapped to the "
        "frozen chronology."
    )


builds_commit_data[
    "CommitTokens"
] = builds_commit_data[
    build_commit_column
].apply(
    parse_commit_tokens
)


builds_commit_data[
    "CommitTokenCount"
] = builds_commit_data[
    "CommitTokens"
].apply(
    len
)


build_commit_token_profile = (
    builds_commit_data[
        [
            "BuildKey",
            "BuildOrder",
            builds_timestamp_column,
            build_commit_column,
            "CommitTokenCount",
            "CommitTokens",
        ]
    ]
    .copy()
)


build_commit_token_profile[
    "CommitTokens"
] = build_commit_token_profile[
    "CommitTokens"
].apply(
    lambda values:
        ",".join(values)
)


build_rows_without_commit_tokens = int(
    build_commit_token_profile[
        "CommitTokenCount"
    ].eq(0).sum()
)


multi_commit_builds = int(
    build_commit_token_profile[
        "CommitTokenCount"
    ].gt(1).sum()
)


build_commit_token_rows = []


for row in builds_commit_data.itertuples(
    index=False
):

    build_key = getattr(
        row,
        "BuildKey"
    )

    build_order = int(
        getattr(
            row,
            "BuildOrder"
        )
    )

    commit_tokens = getattr(
        row,
        "CommitTokens"
    )

    for token_position, token in enumerate(
        commit_tokens,
        start=1,
    ):

        build_commit_token_rows.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "TokenPosition":
                token_position,

            "CommitToken":
                token,
        })


build_commit_tokens = pd.DataFrame(
    build_commit_token_rows,
    columns=[
        "BuildKey",
        "BuildOrder",
        "TokenPosition",
        "CommitToken",
    ],
)


# ------------------------------------------------------------
# 10. LOAD AND NORMALISE ENTITY HISTORY
# ------------------------------------------------------------

entity_history = pd.read_csv(
    source_paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    "EntityId"
] = pd.to_numeric(
    entity_history[
        entity_id_column
    ],
    errors="coerce",
).astype("Int64")


entity_history[
    "CommitNormalised"
] = entity_history[
    entity_commit_column
].apply(
    normalise_commit
)


invalid_entity_rows = int(
    (
        entity_history[
            "EntityId"
        ].isna()
        |
        entity_history[
            "CommitNormalised"
        ].eq("")
    ).sum()
)


entity_history_valid = (
    entity_history.dropna(
        subset=[
            "EntityId",
        ]
    )
    .loc[
        lambda frame:
            frame[
                "CommitNormalised"
            ].ne("")
    ]
    .copy()
)


entity_history_valid[
    "EntityId"
] = entity_history_valid[
    "EntityId"
].astype(int)


unique_history_commits = sorted(
    entity_history_valid[
        "CommitNormalised"
    ].unique().tolist()
)


history_commit_set = set(
    unique_history_commits
)


# ------------------------------------------------------------
# 11. MATCH BUILD COMMIT TOKENS TO ENTITY-HISTORY COMMITS
# ------------------------------------------------------------

commit_match_records = []


for row in build_commit_tokens.itertuples(
    index=False
):

    token = row.CommitToken

    matched_commit = None
    match_type = None
    candidate_count = 0


    if token in history_commit_set:

        matched_commit = token
        match_type = (
            "EXACT_NORMALISED_COMMIT_TOKEN"
        )
        candidate_count = 1

    else:

        prefix_candidates = [
            commit_value
            for commit_value in unique_history_commits
            if (
                commit_value.startswith(token)
                or token.startswith(
                    commit_value
                )
            )
        ]

        candidate_count = len(
            prefix_candidates
        )

        if candidate_count == 1:

            matched_commit = (
                prefix_candidates[0]
            )

            match_type = (
                "UNIQUE_PREFIX_COMMIT_TOKEN"
            )

        elif candidate_count == 0:

            match_type = (
                "UNMATCHED_COMMIT_TOKEN"
            )

        else:

            match_type = (
                "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
            )


    commit_match_records.append({
        "BuildKey":
            row.BuildKey,

        "BuildOrder":
            int(
                row.BuildOrder
            ),

        "TokenPosition":
            int(
                row.TokenPosition
            ),

        "CommitToken":
            token,

        "MatchedCommit":
            matched_commit,

        "MatchType":
            match_type,

        "CandidateCount":
            candidate_count,
    })


commit_matching_audit = pd.DataFrame(
    commit_match_records,
    columns=[
        "BuildKey",
        "BuildOrder",
        "TokenPosition",
        "CommitToken",
        "MatchedCommit",
        "MatchType",
        "CandidateCount",
    ],
)


exact_commit_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "EXACT_NORMALISED_COMMIT_TOKEN"
    ).sum()
)


prefix_commit_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX_COMMIT_TOKEN"
    ).sum()
)


unmatched_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNMATCHED_COMMIT_TOKEN"
    ).sum()
)


ambiguous_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
    ).sum()
)


if ambiguous_commit_tokens:

    print("\nAmbiguous commit mappings:")

    display(
        commit_matching_audit[
            commit_matching_audit[
                "MatchType"
            ].eq(
                "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
            )
        ].head(30)
    )

    raise RuntimeError(
        "Ambiguous commit-prefix mappings were found."
    )


matched_commit_tokens = (
    commit_matching_audit[
        commit_matching_audit[
            "MatchedCommit"
        ].notna()
    ]
    .copy()
)


matched_builds = set(
    matched_commit_tokens[
        "BuildKey"
    ]
)


builds_with_no_matched_commit = int(
    (
        ~build_commit_token_profile[
            "BuildKey"
        ].isin(
            matched_builds
        )
    ).sum()
)


total_commit_tokens = int(
    len(
        commit_matching_audit
    )
)


matched_commit_token_count = int(
    matched_commit_tokens[
        "CommitToken"
    ].count()
)


commit_token_coverage_percent = (
    100.0
    if total_commit_tokens == 0
    else
    100.0
    * matched_commit_token_count
    / total_commit_tokens
)


# ------------------------------------------------------------
# 12. CONSTRUCT BUILD ↔ ENTITY MAPPING
# ------------------------------------------------------------

entity_commit_pairs = (
    entity_history_valid[
        [
            "CommitNormalised",
            "EntityId",
        ]
    ]
    .drop_duplicates()
)


build_entity_map = (
    matched_commit_tokens[
        [
            "BuildKey",
            "BuildOrder",
            "CommitToken",
            "MatchedCommit",
            "MatchType",
        ]
    ]
    .merge(
        entity_commit_pairs,
        left_on="MatchedCommit",
        right_on="CommitNormalised",
        how="left",
        validate="many_to_many",
    )
    .drop(
        columns=[
            "CommitNormalised",
        ]
    )
)


build_entity_map = (
    build_entity_map.dropna(
        subset=[
            "EntityId",
        ]
    )
    .copy()
)


build_entity_map[
    "EntityId"
] = build_entity_map[
    "EntityId"
].astype(int)


build_entity_map = (
    build_entity_map.drop_duplicates(
        subset=[
            "BuildKey",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "BuildOrder",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


builds_with_mapped_entities = int(
    build_entity_map[
        "BuildKey"
    ].nunique()
)


mapped_entity_ids = set(
    build_entity_map[
        "EntityId"
    ].astype(int)
)


id_map = pd.read_csv(
    source_paths[
        "id_map.csv"
    ],
    usecols=[
        id_map_key_column,
        id_map_value_column,
    ],
    low_memory=False,
)


id_map[
    "EntityId"
] = pd.to_numeric(
    id_map[
        id_map_value_column
    ],
    errors="coerce",
).astype("Int64")


valid_id_map = id_map.dropna(
    subset=[
        "EntityId",
    ]
).copy()


valid_id_map[
    "EntityId"
] = valid_id_map[
    "EntityId"
].astype(int)


id_map_entity_ids = set(
    valid_id_map[
        "EntityId"
    ]
)


entity_history_ids = set(
    entity_history_valid[
        "EntityId"
    ].astype(int)
)


history_entity_ids_missing_from_id_map = (
    entity_history_ids
    - id_map_entity_ids
)


id_map_entity_ids_missing_from_history = (
    id_map_entity_ids
    - entity_history_ids
)


changed_entities_by_build = {
    build_key:
        set(
            group[
                "EntityId"
            ].astype(int)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


clean_changed_entities_by_build = {
    build_key:
        changed_entities_by_build.get(
            build_key,
            set(),
        )
    for build_key in fixed_split[
        "BuildKey"
    ]
}


entity_changed_builds = {
    int(entity_id):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for entity_id, group
    in build_entity_map.groupby(
        "EntityId",
        sort=False,
    )
}


print("\nCommit/entity mapping summary:")

print(
    "Build commit tokens:",
    total_commit_tokens,
)

print(
    "Exact matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    prefix_commit_matches,
)

print(
    "Unmatched tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous tokens:",
    ambiguous_commit_tokens,
)

print(
    "Token coverage percent:",
    commit_token_coverage_percent,
)

print(
    "Build rows without commit tokens:",
    build_rows_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


# ------------------------------------------------------------
# 13. LOAD CLEAN EXECUTION HISTORY
# ------------------------------------------------------------

execution_usecols = [
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]

if (
    exe_job_column is not None
    and exe_job_column not in execution_usecols
):

    execution_usecols.append(
        exe_job_column
    )


execution_history = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=execution_usecols,
    low_memory=False,
)


if len(
    execution_history
) != EXPECTED_RAW_ROWS:

    raise AssertionError(
        "Clean execution-history row count differs."
    )


execution_history[
    "BuildKey"
] = canonical_identifier(
    execution_history[
        exe_build_column
    ]
)


execution_history[
    "TestKey"
] = canonical_identifier(
    execution_history[
        exe_test_column
    ]
)


execution_history[
    "BuildOrder"
] = execution_history[
    "BuildKey"
].map(
    build_order_map
)


if execution_history[
    "BuildOrder"
].isna().any():

    raise AssertionError(
        "Some execution builds could not be mapped "
        "to the frozen chronology."
    )


execution_history[
    "Verdict"
] = pd.to_numeric(
    execution_history[
        exe_verdict_column
    ],
    errors="raise",
).astype(int)


execution_history[
    "Duration"
] = pd.to_numeric(
    execution_history[
        exe_duration_column
    ],
    errors="coerce",
)


invalid_durations = int(
    (
        ~np.isfinite(
            execution_history[
                "Duration"
            ].to_numpy(
                dtype=float
            )
        )
        |
        execution_history[
            "Duration"
        ].lt(0)
    ).sum()
)


if invalid_durations:

    raise AssertionError(
        "Execution history contains invalid durations.\n"
        f"Rows: {invalid_durations}"
    )


raw_duplicate_build_test_rows = int(
    execution_history.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if raw_duplicate_build_test_rows:

    raise AssertionError(
        "Raw execution history contains duplicate "
        "Build/Test rows."
    )


sort_columns = [
    "BuildOrder",
]

if exe_job_column is not None:

    sort_columns.append(
        exe_job_column
    )

sort_columns.append(
    "TestKey"
)


execution_history = (
    execution_history.sort_values(
        sort_columns,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. LOAD REQUESTED MODEL-READY ROWS
# ------------------------------------------------------------

dataset_usecols = (
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
    + REC_FEATURE_COLUMNS
)


dataset_rec = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    usecols=dataset_usecols,
    low_memory=False,
)


if len(
    dataset_rec
) != EXPECTED_MODEL_ROWS:

    raise AssertionError(
        "Model-ready dataset row count differs."
    )


dataset_rec[
    "BuildKey"
] = canonical_identifier(
    dataset_rec[
        dataset_build_column
    ]
)


dataset_rec[
    "TestKey"
] = canonical_identifier(
    dataset_rec[
        dataset_test_column
    ]
)


model_duplicate_build_test_rows = int(
    dataset_rec.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if model_duplicate_build_test_rows:

    raise AssertionError(
        "Model-ready Build/Test rows are not unique."
    )


requested_pairs_by_test = {
    test_key:
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in dataset_rec.groupby(
        "TestKey",
        sort=False,
    )
}


requested_pair_count = int(
    len(
        dataset_rec
    )
)


# ------------------------------------------------------------
# 15. REC RECONSTRUCTION HELPERS
# ------------------------------------------------------------

def calculate_max_test_file_rate(
    target_builds,
    current_changed_entities,
):
    """
    Reproduce TCP-CI's maximum changed-file overlap rate.

    - No prior failure/transition builds: -1
    - Prior target builds but no entity overlap: 0
    - Otherwise: maximum overlap frequency divided by
      number of unique target builds.
    """

    if len(target_builds) == 0:

        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:

        changed_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )

        overlap_count = len(
            changed_builds.intersection(
                target_builds
            )
        )

        if overlap_count > maximum_frequency:

            maximum_frequency = (
                overlap_count
            )

    if maximum_frequency == 0:

        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_clean_rec_features(
    history,
):
    """
    Reconstruct all 19 REC features using only executions
    before each requested current Build/Test execution.

    The implementation maintains cumulative state per test
    to avoid repeatedly slicing the full execution history.
    """

    reconstructed_records = []

    tests_processed = 0
    requested_rows_reconstructed = 0

    total_tests = int(
        history[
            "TestKey"
        ].nunique()
    )


    for test_key, test_history in history.groupby(
        "TestKey",
        sort=False,
    ):

        tests_processed += 1

        requested_builds = (
            requested_pairs_by_test.get(
                str(test_key)
            )
        )

        if not requested_builds:

            continue


        test_history = (
            test_history.sort_values(
                [
                    "BuildOrder",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        first_build_key = str(
            test_history.iloc[0][
                "BuildKey"
            ]
        )


        if first_build_key not in (
            global_build_position
        ):

            raise AssertionError(
                "First test build is absent from the "
                "global chronology."
            )


        prior_count = 0

        prior_duration_sum = 0.0
        prior_duration_max = -np.inf

        prior_failure_count = 0
        prior_assertion_count = 0
        prior_exception_count = 0
        prior_transition_count = 0

        last_failure_position = None
        last_transition_position = None

        previous_verdict = None
        previous_duration = None

        recent_history = deque(
            maxlen=RECENT_WINDOW
        )

        failure_builds = set()
        transition_builds = set()


        for row in test_history.itertuples(
            index=False
        ):

            current_build = str(
                row.BuildKey
            )

            current_verdict = int(
                row.Verdict
            )

            current_duration = float(
                row.Duration
            )


            if current_build in requested_builds:

                record = {
                    "BuildKey":
                        current_build,

                    "TestKey":
                        str(
                            test_key
                        ),
                }


                if prior_count == 0:

                    for feature in (
                        REC_FEATURE_COLUMNS
                    ):

                        record[
                            feature
                        ] = -1.0

                    record[
                        "REC_Age"
                    ] = 0.0


                else:

                    current_global_position = (
                        global_build_position[
                            current_build
                        ]
                    )

                    first_global_position = (
                        global_build_position[
                            first_build_key
                        ]
                    )

                    age = (
                        current_global_position
                        - first_global_position
                    )


                    if last_failure_position is None:

                        last_failure_age = -1.0

                    else:

                        last_failure_age = float(
                            prior_count
                            - 1
                            - last_failure_position
                        )


                    if last_transition_position is None:

                        last_transition_age = -1.0

                    else:

                        last_transition_age = float(
                            prior_count
                            - 1
                            - last_transition_position
                        )


                    recent_rows = list(
                        recent_history
                    )

                    recent_length = len(
                        recent_rows
                    )

                    recent_durations = np.asarray(
                        [
                            item[
                                "Duration"
                            ]
                            for item in recent_rows
                        ],
                        dtype=float,
                    )

                    recent_verdicts = np.asarray(
                        [
                            item[
                                "Verdict"
                            ]
                            for item in recent_rows
                        ],
                        dtype=int,
                    )

                    recent_transitions = np.asarray(
                        [
                            item[
                                "Transition"
                            ]
                            for item in recent_rows
                        ],
                        dtype=int,
                    )


                    recent_fail_rate = float(
                        np.count_nonzero(
                            recent_verdicts != 0
                        )
                        / recent_length
                    )

                    recent_assert_rate = float(
                        np.count_nonzero(
                            recent_verdicts == 2
                        )
                        / recent_length
                    )

                    recent_exc_rate = float(
                        np.count_nonzero(
                            recent_verdicts == 1
                        )
                        / recent_length
                    )

                    recent_transition_rate = float(
                        np.count_nonzero(
                            recent_transitions == 1
                        )
                        / recent_length
                    )


                    total_fail_rate = float(
                        prior_failure_count
                        / prior_count
                    )

                    total_assert_rate = float(
                        prior_assertion_count
                        / prior_count
                    )

                    total_exc_rate = float(
                        prior_exception_count
                        / prior_count
                    )

                    total_transition_rate = float(
                        prior_transition_count
                        / prior_count
                    )


                    current_changed_entities = (
                        clean_changed_entities_by_build.get(
                            current_build,
                            set(),
                        )
                    )


                    max_file_fail_rate = (
                        calculate_max_test_file_rate(
                            target_builds=(
                                failure_builds
                            ),

                            current_changed_entities=(
                                current_changed_entities
                            ),
                        )
                    )


                    max_file_transition_rate = (
                        calculate_max_test_file_rate(
                            target_builds=(
                                transition_builds
                            ),

                            current_changed_entities=(
                                current_changed_entities
                            ),
                        )
                    )


                    record.update({
                        "REC_Age":
                            float(age),

                        "REC_LastFailureAge":
                            last_failure_age,

                        "REC_LastTransitionAge":
                            last_transition_age,

                        "REC_RecentAvgExeTime":
                            float(
                                recent_durations.mean()
                            ),

                        "REC_RecentMaxExeTime":
                            float(
                                recent_durations.max()
                            ),

                        "REC_RecentFailRate":
                            recent_fail_rate,

                        "REC_RecentAssertRate":
                            recent_assert_rate,

                        "REC_RecentExcRate":
                            recent_exc_rate,

                        "REC_RecentTransitionRate":
                            recent_transition_rate,

                        "REC_TotalAvgExeTime":
                            float(
                                prior_duration_sum
                                / prior_count
                            ),

                        "REC_TotalMaxExeTime":
                            float(
                                prior_duration_max
                            ),

                        "REC_TotalFailRate":
                            total_fail_rate,

                        "REC_TotalAssertRate":
                            total_assert_rate,

                        "REC_TotalExcRate":
                            total_exc_rate,

                        "REC_TotalTransitionRate":
                            total_transition_rate,

                        "REC_LastVerdict":
                            float(
                                previous_verdict
                            ),

                        "REC_LastExeTime":
                            float(
                                previous_duration
                            ),

                        "REC_MaxTestFileFailRate":
                            max_file_fail_rate,

                        "REC_MaxTestFileTransitionRate":
                            max_file_transition_rate,
                    })


                reconstructed_records.append(
                    record
                )

                requested_rows_reconstructed += 1


            current_transition = (
                0
                if previous_verdict is None
                else int(
                    current_verdict
                    != previous_verdict
                )
            )


            if current_verdict != 0:

                prior_failure_count += 1

                last_failure_position = (
                    prior_count
                )

                failure_builds.add(
                    current_build
                )


            if current_verdict == 2:

                prior_assertion_count += 1


            if current_verdict == 1:

                prior_exception_count += 1


            if current_transition == 1:

                prior_transition_count += 1

                last_transition_position = (
                    prior_count
                )

                transition_builds.add(
                    current_build
                )


            prior_duration_sum += (
                current_duration
            )

            prior_duration_max = max(
                prior_duration_max,
                current_duration,
            )


            recent_history.append({
                "BuildKey":
                    current_build,

                "Verdict":
                    current_verdict,

                "Duration":
                    current_duration,

                "Transition":
                    current_transition,
            })


            previous_verdict = (
                current_verdict
            )

            previous_duration = (
                current_duration
            )

            prior_count += 1


        if (
            tests_processed % 100 == 0
            or tests_processed == total_tests
        ):

            print(
                "REC reconstruction progress:",
                tests_processed,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                requested_rows_reconstructed,
            )


    reconstructed = pd.DataFrame(
        reconstructed_records
    )


    return reconstructed


# ------------------------------------------------------------
# 16. RUN CLEAN RECONSTRUCTION
# ------------------------------------------------------------

print("\nReconstructing all 19 clean REC features...")

reconstruction_started = (
    time.perf_counter()
)


clean_rec_reconstructed_keys = (
    reconstruct_clean_rec_features(
        execution_history
    )
)


reconstruction_seconds = float(
    time.perf_counter()
    - reconstruction_started
)


reconstructed_row_count = int(
    len(
        clean_rec_reconstructed_keys
    )
)


reconstructed_duplicate_rows = int(
    clean_rec_reconstructed_keys.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if reconstructed_row_count != (
    requested_pair_count
):

    raise AssertionError(
        "Clean REC reconstruction row count differs.\n"
        f"Expected: {requested_pair_count}\n"
        f"Actual: {reconstructed_row_count}"
    )


if reconstructed_duplicate_rows:

    raise AssertionError(
        "Clean REC reconstruction contains duplicate "
        "Build/Test rows."
    )


print("\nClean REC reconstruction completed:")

print(
    "Requested rows:",
    requested_pair_count,
)

print(
    "Reconstructed rows:",
    reconstructed_row_count,
)

print(
    "Duplicate reconstructed rows:",
    reconstructed_duplicate_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


# ------------------------------------------------------------
# 17. ALIGN RECONSTRUCTION WITH ORIGINAL DATASET
# ------------------------------------------------------------

comparison = (
    dataset_rec[
        [
            dataset_build_column,
            dataset_test_column,
            "BuildKey",
            "TestKey",
        ]
        + REC_FEATURE_COLUMNS
    ]
    .merge(
        clean_rec_reconstructed_keys,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_Original",
            "_Reconstructed",
        ),
        indicator=True,
    )
)


missing_reconstructed_rows = int(
    comparison[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


if missing_reconstructed_rows:

    raise AssertionError(
        "Some model-ready rows are missing from the "
        "clean REC reconstruction."
    )


comparison_summary_records = []
mismatch_example_records = []


clean_rec_reconstructed = (
    dataset_rec[
        [
            dataset_build_column,
            dataset_test_column,
            "BuildKey",
            "TestKey",
        ]
    ]
    .copy()
)


clean_anchor_offsets = (
    dataset_rec[
        [
            dataset_build_column,
            dataset_test_column,
            "BuildKey",
            "TestKey",
        ]
    ]
    .copy()
)


total_direct_mismatch_values = 0
total_nonzero_offset_values = 0
total_anchored_mismatch_values = 0


for feature in REC_FEATURE_COLUMNS:

    original_column = (
        f"{feature}_Original"
    )

    reconstructed_column = (
        f"{feature}_Reconstructed"
    )


    original_values = pd.to_numeric(
        comparison[
            original_column
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    reconstructed_values = pd.to_numeric(
        comparison[
            reconstructed_column
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=COMPARISON_RTOL,
        atol=COMPARISON_ATOL,
        equal_nan=True,
    )


    offsets = (
        original_values
        - reconstructed_values
    )


    anchored_values = (
        reconstructed_values
        + offsets
    )


    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=COMPARISON_RTOL,
        atol=COMPARISON_ATOL,
        equal_nan=True,
    )


    zero_offset_mask = np.isclose(
        offsets,
        0.0,
        rtol=0,
        atol=OFFSET_ZERO_ATOL,
        equal_nan=False,
    )


    direct_mismatches = int(
        (
            ~direct_match_mask
        ).sum()
    )


    nonzero_offsets = int(
        (
            ~zero_offset_mask
            &
            np.isfinite(
                offsets
            )
        ).sum()
    )


    anchored_mismatches = int(
        (
            ~anchored_match_mask
        ).sum()
    )


    total_direct_mismatch_values += (
        direct_mismatches
    )

    total_nonzero_offset_values += (
        nonzero_offsets
    )

    total_anchored_mismatch_values += (
        anchored_mismatches
    )


    clean_rec_reconstructed[
        feature
    ] = reconstructed_values


    clean_anchor_offsets[
        feature
    ] = offsets


    absolute_differences = np.abs(
        offsets
    )


    comparison_summary_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in (
                    VERDICT_DEPENDENT_REC_FEATURES
                )
                else
                "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in (
                FILE_HISTORY_REC_FEATURES
            ),

        "Rows":
            len(
                comparison
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            direct_mismatches,

        "NonZeroAnchorOffsets":
            nonzero_offsets,

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            anchored_mismatches,

        "MaximumAbsoluteDirectDifference":
            finite_max_absolute_difference(
                absolute_differences
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.nanmean(
                    absolute_differences
                )
            ),
    })


    mismatch_positions = np.flatnonzero(
        ~direct_match_mask
    )[:10]


    for position in mismatch_positions:

        mismatch_example_records.append({
            "Feature":
                feature,

            "FeatureClass":
                (
                    "VERDICT_DEPENDENT"
                    if feature in (
                        VERDICT_DEPENDENT_REC_FEATURES
                    )
                    else
                    "VERDICT_INDEPENDENT"
                ),

            "Build":
                comparison.iloc[
                    position
                ][
                    dataset_build_column
                ],

            "Test":
                comparison.iloc[
                    position
                ][
                    dataset_test_column
                ],

            "BuildKey":
                comparison.iloc[
                    position
                ][
                    "BuildKey"
                ],

            "TestKey":
                comparison.iloc[
                    position
                ][
                    "TestKey"
                ],

            "Original":
                original_values[
                    position
                ],

            "DirectReconstruction":
                reconstructed_values[
                    position
                ],

            "AnchorOffset":
                offsets[
                    position
                ],

            "AnchoredValue":
                anchored_values[
                    position
                ],
        })


rec_comparison_summary = pd.DataFrame(
    comparison_summary_records
)


rec_mismatch_examples = pd.DataFrame(
    mismatch_example_records,
    columns=[
        "Feature",
        "FeatureClass",
        "Build",
        "Test",
        "BuildKey",
        "TestKey",
        "Original",
        "DirectReconstruction",
        "AnchorOffset",
        "AnchoredValue",
    ],
)


rows_with_any_nonzero_offset = int(
    (
        ~np.isclose(
            clean_anchor_offsets[
                REC_FEATURE_COLUMNS
            ].to_numpy(
                dtype=float
            ),
            0.0,
            rtol=0,
            atol=OFFSET_ZERO_ATOL,
            equal_nan=False,
        )
    ).any(
        axis=1
    ).sum()
)


dependent_direct_mismatch_values = int(
    rec_comparison_summary.loc[
        rec_comparison_summary[
            "Feature"
        ].isin(
            VERDICT_DEPENDENT_REC_FEATURES
        ),
        "DirectMismatchingRows",
    ].sum()
)


independent_direct_mismatch_values = int(
    rec_comparison_summary.loc[
        rec_comparison_summary[
            "Feature"
        ].isin(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_feature_direct_mismatch_values = int(
    rec_comparison_summary.loc[
        rec_comparison_summary[
            "Feature"
        ].isin(
            FILE_HISTORY_REC_FEATURES
        ),
        "DirectMismatchingRows",
    ].sum()
)


# ------------------------------------------------------------
# 18. VALIDATE CLEAN-ANCHOR FORMULA
# ------------------------------------------------------------

anchor_validation_records = []


for feature in REC_FEATURE_COLUMNS:

    direct_values = pd.to_numeric(
        clean_rec_reconstructed[
            feature
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    offsets = pd.to_numeric(
        clean_anchor_offsets[
            feature
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    original_values = pd.to_numeric(
        dataset_rec[
            feature
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    reconstructed_clean_at_zero = (
        direct_values
        + offsets
    )


    match_mask = np.isclose(
        reconstructed_clean_at_zero,
        original_values,
        rtol=COMPARISON_RTOL,
        atol=COMPARISON_ATOL,
        equal_nan=True,
    )


    anchor_validation_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in (
                    VERDICT_DEPENDENT_REC_FEATURES
                )
                else
                "VERDICT_INDEPENDENT"
            ),

        "Rows":
            len(
                original_values
            ),

        "MatchingRows":
            int(
                match_mask.sum()
            ),

        "MismatchingRows":
            int(
                (
                    ~match_mask
                ).sum()
            ),

        "Pass":
            bool(
                match_mask.all()
            ),
    })


anchor_validation = pd.DataFrame(
    anchor_validation_records
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


if failed_anchor_features:

    print("\nFailed anchor features:")

    display(
        anchor_validation[
            ~anchor_validation[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "The clean-anchor formula did not reproduce "
        "dataset.csv exactly."
    )


# ------------------------------------------------------------
# 19. OVERALL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status[
                "Status"
            ],

        "Pass":
            step1b_status[
                "Status"
            ]
            == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Step 2A passed",

        "Expected":
            EXPECTED_STEP2A_STATUS,

        "Actual":
            step2a_status[
                "Status"
            ],

        "Pass":
            step2a_status[
                "Status"
            ]
            == EXPECTED_STEP2A_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Frozen source files",

        "Expected":
            5,

        "Actual":
            len(
                source_manifest
            ),

        "Pass":
            len(
                source_manifest
            ) == 5,
    },

    {
        "Check":
            "Canonical builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                fixed_split
            ),

        "Pass":
            len(
                fixed_split
            )
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Raw execution rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                execution_history
            ),

        "Pass":
            len(
                execution_history
            )
            == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Model-ready rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                dataset_rec
            ),

        "Pass":
            len(
                dataset_rec
            )
            == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Raw duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows
            == 0,
    },

    {
        "Check":
            "Model duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows
            == 0,
    },

    {
        "Check":
            "Reconstructed REC rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            reconstructed_row_count,

        "Pass":
            reconstructed_row_count
            == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Duplicate reconstructed rows",

        "Expected":
            0,

        "Actual":
            reconstructed_duplicate_rows,

        "Pass":
            reconstructed_duplicate_rows
            == 0,
    },

    {
        "Check":
            "Missing reconstructed rows",

        "Expected":
            0,

        "Actual":
            missing_reconstructed_rows,

        "Pass":
            missing_reconstructed_rows
            == 0,
    },

    {
        "Check":
            "REC features reconstructed",

        "Expected":
            19,

        "Actual":
            len(
                REC_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                REC_FEATURE_COLUMNS
            ) == 19,
    },

    {
        "Check":
            "Failed clean-anchor features",

        "Expected":
            0,

        "Actual":
            failed_anchor_features,

        "Pass":
            failed_anchor_features
            == 0,
    },

    {
        "Check":
            "Anchored mismatch values",

        "Expected":
            0,

        "Actual":
            total_anchored_mismatch_values,

        "Pass":
            total_anchored_mismatch_values
            == 0,
    },

    {
        "Check":
            "Ambiguous commit tokens",

        "Expected":
            0,

        "Actual":
            ambiguous_commit_tokens,

        "Pass":
            ambiguous_commit_tokens
            == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 2B validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 2B DID NOT PASS."
    )


# ------------------------------------------------------------
# 20. VERIFY SOURCE AND REGISTRY IMMUTABILITY
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}


source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)


if not source_files_unchanged:

    raise AssertionError(
        "A frozen source file changed during Step 2B."
    )


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 2B."
    )


# ------------------------------------------------------------
# 21. WRITE RECONSTRUCTION OUTPUTS
# ------------------------------------------------------------

REC_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    build_commit_token_profile,
)


atomic_write_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    commit_matching_audit,
)


atomic_write_csv(
    BUILD_ENTITY_MAP_PATH,
    build_entity_map,
    compression="gzip",
)


atomic_write_parquet(
    CLEAN_REC_RECONSTRUCTED_PATH,
    clean_rec_reconstructed,
)


atomic_write_parquet(
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    clean_anchor_offsets,
)


atomic_write_csv(
    REC_COMPARISON_SUMMARY_PATH,
    rec_comparison_summary,
)


atomic_write_csv(
    REC_MISMATCH_EXAMPLES_PATH,
    rec_mismatch_examples,
)


atomic_write_csv(
    ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


entity_mapping_summary = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "BuildCommitColumn":
        build_commit_column,

    "EntityHistoryCommitColumn":
        entity_commit_column,

    "EntityHistoryIdColumn":
        entity_id_column,

    "IdMapKeyColumn":
        id_map_key_column,

    "IdMapValueColumn":
        id_map_value_column,

    "BuildRows":
        len(
            builds_commit_data
        ),

    "BuildRowsWithoutCommitTokens":
        build_rows_without_commit_tokens,

    "MultiCommitBuilds":
        multi_commit_builds,

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixCommitMatches":
        prefix_commit_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "MatchedCommitTokenRows":
        matched_commit_token_count,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithNoMatchedCommit":
        builds_with_no_matched_commit,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "MappedEntityIds":
        len(
            mapped_entity_ids
        ),

    "EntityHistoryUniqueIds":
        len(
            entity_history_ids
        ),

    "IdMapUniqueIds":
        len(
            id_map_entity_ids
        ),

    "HistoryEntityIdsMissingFromIdMap":
        len(
            history_entity_ids_missing_from_id_map
        ),

    "IdMapEntityIdsMissingFromHistory":
        len(
            id_map_entity_ids_missing_from_history
        ),

    "InvalidEntityHistoryRows":
        invalid_entity_rows,

    "GeneratedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    ENTITY_MAPPING_SUMMARY_PATH,
    entity_mapping_summary,
)


# ------------------------------------------------------------
# 22. HASH FROZEN RECONSTRUCTION ARTEFACTS
# ------------------------------------------------------------

reconstruction_artifact_paths = [
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    CLEAN_REC_RECONSTRUCTED_PATH,
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    REC_COMPARISON_SUMMARY_PATH,
    REC_MISMATCH_EXAMPLES_PATH,
    ANCHOR_VALIDATION_PATH,
]


reconstruction_artifacts = []


for path in reconstruction_artifact_paths:

    reconstruction_artifacts.append({
        "Path":
            str(path),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


# ------------------------------------------------------------
# 23. WRITE REPORT AND CHECKPOINT
# ------------------------------------------------------------

report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatures":
        REC_FEATURE_COLUMNS,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "CommitEntityMapping": {
        "BuildCommitColumn":
            build_commit_column,

        "CommitMatchingPolicy":
            (
                "EXACT_NORMALISED_COMMIT_TOKEN; "
                "UNIQUE_PREFIX_FALLBACK"
            ),

        "BuildCommitTokenRows":
            total_commit_tokens,

        "ExactMatches":
            exact_commit_matches,

        "UniquePrefixMatches":
            prefix_commit_matches,

        "UnmatchedTokens":
            unmatched_commit_tokens,

        "AmbiguousTokens":
            ambiguous_commit_tokens,

        "CommitTokenCoveragePercent":
            commit_token_coverage_percent,

        "BuildsWithNoMatchedCommit":
            builds_with_no_matched_commit,

        "BuildsWithMappedEntities":
            builds_with_mapped_entities,

        "BuildEntityRows":
            len(
                build_entity_map
            ),
    },

    "CleanReconstruction": {
        "RawHistoryRows":
            len(
                execution_history
            ),

        "RequestedRows":
            requested_pair_count,

        "ReconstructedRows":
            reconstructed_row_count,

        "DuplicateReconstructedRows":
            reconstructed_duplicate_rows,

        "MissingReconstructedRows":
            missing_reconstructed_rows,

        "ReconstructionSeconds":
            reconstruction_seconds,

        "DirectMismatchingFeatureValues":
            total_direct_mismatch_values,

        "VerdictDependentDirectMismatches":
            dependent_direct_mismatch_values,

        "VerdictIndependentDirectMismatches":
            independent_direct_mismatch_values,

        "FileHistoryDirectMismatches":
            file_feature_direct_mismatch_values,

        "NonZeroAnchorOffsetValues":
            total_nonzero_offset_values,

        "RowsWithAnyNonZeroAnchorOffset":
            rows_with_any_nonzero_offset,

        "AnchoredMismatchingFeatureValues":
            total_anchored_mismatch_values,
    },

    "CleanAnchorPolicy": {
        "Formula":
            (
                "original_clean_feature + "
                "(direct_noisy_reconstruction - "
                "direct_clean_reconstruction)"
            ),

        "EquivalentFormula":
            (
                "direct_noisy_reconstruction + "
                "clean_anchor_offset"
            ),

        "ZeroNoiseExactReproduction":
            total_anchored_mismatch_values == 0,

        "FailedAnchorFeatures":
            failed_anchor_features,
    },

    "ReconstructionArtifacts":
        reconstruction_artifacts,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_REPORT_PATH,
    report,
)


rec_checkpoint = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatureCount":
        len(
            REC_FEATURE_COLUMNS
        ),

    "VerdictDependentRECFeatureCount":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatureCount":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "CleanRECReconstructed":
        str(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECReconstructedSHA256":
        calculate_sha256(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECAnchorOffsets":
        str(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "CleanRECAnchorOffsetsSHA256":
        calculate_sha256(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "BuildEntityMapSHA256":
        calculate_sha256(
            BUILD_ENTITY_MAP_PATH
        ),

    "DirectMismatchingFeatureValues":
        total_direct_mismatch_values,

    "NonZeroAnchorOffsetValues":
        total_nonzero_offset_values,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_offset,

    "AnchoredMismatchingFeatureValues":
        total_anchored_mismatch_values,

    "CleanAnchorFormula":
        (
            "original_clean_feature + "
            "(direct_noisy_reconstruction - "
            "direct_clean_reconstruction)"
        ),

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    REC_CHECKPOINT_PATH,
    rec_checkpoint,
)


status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "SourceRootSHA256":
        source_root_sha256,

    "RawHistoryRows":
        len(
            execution_history
        ),

    "ModelRows":
        len(
            dataset_rec
        ),

    "ReconstructedRows":
        reconstructed_row_count,

    "RECFeatures":
        len(
            REC_FEATURE_COLUMNS
        ),

    "DirectMismatchingFeatureValues":
        total_direct_mismatch_values,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_offset,

    "AnchoredMismatchingFeatureValues":
        total_anchored_mismatch_values,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "ReconstructionSeconds":
        reconstruction_seconds,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            REC_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_STATUS_PATH,
    status,
)


# ------------------------------------------------------------
# 24. FINAL READBACK
# ------------------------------------------------------------

required_outputs = (
    reconstruction_artifact_paths
    + [
        STEP2B_REPORT_PATH,
        REC_CHECKPOINT_PATH,
        STEP2B_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]


if missing_outputs:

    raise RuntimeError(
        "Step 2B outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


final_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_status.get(
        "Status"
    )
    != STEP2B_PASS_STATUS
):

    raise AssertionError(
        "Final Step 2B status differs."
    )


# ------------------------------------------------------------
# 25. DISPLAY AUDIT RESULTS
# ------------------------------------------------------------

print("\nClean REC feature comparison:")

display(
    rec_comparison_summary
)


print("\nClean-anchor validation:")

display(
    anchor_validation
)


if not rec_mismatch_examples.empty:

    print("\nExample direct-reconstruction differences:")

    display(
        rec_mismatch_examples.head(
            50
        )
    )


print("\nCommit matching audit summary:")

display(
    commit_matching_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


# ------------------------------------------------------------
# 26. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 110)
print("=== PROJECT 9 CELL 4 / STEP 2B RESULT ===")
print("=" * 110)

print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nCommit and entity mapping:")

print(
    "Build commit column:",
    build_commit_column,
)

print(
    "Build commit-token rows:",
    total_commit_tokens,
)

print(
    "Exact commit matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    prefix_commit_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_commit_tokens,
)

print(
    "Commit-token coverage:",
    commit_token_coverage_percent,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


print("\nClean REC reconstruction:")

print(
    "Raw history rows:",
    len(
        execution_history
    ),
)

print(
    "Requested model-ready rows:",
    requested_pair_count,
)

print(
    "Reconstructed rows:",
    reconstructed_row_count,
)

print(
    "Duplicate reconstructed rows:",
    reconstructed_duplicate_rows,
)

print(
    "Missing reconstructed rows:",
    missing_reconstructed_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


print("\nDirect reconstruction comparison:")

print(
    "Direct mismatching feature values:",
    total_direct_mismatch_values,
)

print(
    "Verdict-dependent direct mismatches:",
    dependent_direct_mismatch_values,
)

print(
    "Verdict-independent direct mismatches:",
    independent_direct_mismatch_values,
)

print(
    "File-history direct mismatches:",
    file_feature_direct_mismatch_values,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_offset,
)

print(
    "Non-zero anchor-offset values:",
    total_nonzero_offset_values,
)


print("\nClean-anchor validation:")

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatching feature values:",
    total_anchored_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    total_anchored_mismatch_values == 0,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nFrozen REC checkpoint:")

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        REC_CHECKPOINT_PATH
    ),
)


print("\nSaved outputs:")

for output_path in required_outputs:

    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP2B_PASS_STATUS,
)

print("=" * 110)

=== PROJECT 9 CELL 4 / STEP 2B: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===

Resolved reconstruction schema:
Build commit column: commits
Entity-history commit column: Commit
Entity-history ID column: EntityId
ID-map key / value: key / value

Commit/entity mapping summary:
Build commit tokens: 1052
Exact matches: 1052
Unique-prefix matches: 0
Unmatched tokens: 0
Ambiguous tokens: 0
Token coverage percent: 100.0
Build rows without commit tokens: 0
Builds with no matched commit: 0
Builds with mapped entities: 822
Build/entity rows: 13145

Reconstructing all 19 clean REC features...
REC reconstruction progress: 100 / 1021 tests | reconstructed rows: 14739
REC reconstruction progress: 200 / 1021 tests | reconstructed rows: 28584
REC reconstruction progress: 300 / 1021 tests | reconstructed rows: 38600
REC reconstruction progress: 400 / 1021 tests | reconstructed rows: 47450
REC reconstruction progress: 500 / 1021 tests | reconstructed rows: 53110
REC reconstruction progress: 600 / 1021 

AttributeError: 'numpy.ndarray' object has no attribute 'to_numpy'

In [8]:
# ============================================================
# PROJECT 9 — CELL 4 / STEP 2B V2
# CLEAN REC RECONSTRUCTION AND CLEAN-ANCHOR FREEZE
#
# PROJECT: camunda@camunda-bpm-platform
#
# V2 fixes:
# - accepts NumPy arrays in difference-summary helpers
# - explicitly preserves dataset.csv row order during joins
# - validates that all original/reconstructed REC values
#   are finite before freezing offsets
# - writes outputs only after every validation passes
#
# This cell:
# - validates Steps 1B and 2A
# - revalidates all five frozen source files
# - reconstructs all 19 clean REC features
# - freezes clean-anchor offsets
# - guarantees exact clean-data reproduction at 0% noise
#
# This cell does NOT:
# - inject noise
# - train models
# - modify source data
# - modify the completion registry
# - modify Projects 1–8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from IPython.display import display

import hashlib
import json
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = (
    "camunda"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_AND_SPLIT_VALIDATED"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_9_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_AUDITED"
)

STEP2B_PASS_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)

EXPECTED_BUILDS = 822
EXPECTED_RAW_ROWS = 472765
EXPECTED_MODEL_ROWS = 79383

EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

RECENT_WINDOW = 6

COMPARISON_RTOL = 1e-9
COMPARISON_ATOL = 1e-9
OFFSET_ZERO_ATOL = 1e-12


# ------------------------------------------------------------
# 2. FROZEN REC FEATURE PROTOCOL
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


FILE_HISTORY_REC_FEATURES = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "The frozen REC protocol must contain 19 features."
    )


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:
    raise AssertionError(
        "The frozen protocol must contain 13 "
        "verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:
    raise AssertionError(
        "The frozen protocol must contain six "
        "verdict-independent REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    & set(VERDICT_INDEPENDENT_REC_FEATURES)
):
    raise AssertionError(
        "REC dependency classes overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):

    raise AssertionError(
        "REC dependency classes do not cover "
        "all 19 REC features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT_SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP1B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step1b_status.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

SELECTION_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_preflight"
)

FIXED_SPLIT_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_fixed_build_split.csv.gz"
)

REC_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_rec_preflight"
)

BUILD_COMMIT_TOKEN_PROFILE_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_commit_token_profile.csv"
)

COMMIT_MATCHING_AUDIT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_commit_matching_audit.csv"
)

BUILD_ENTITY_MAP_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_summary.json"
)

CLEAN_REC_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_reconstructed.parquet"
)

CLEAN_REC_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_anchor_offsets.parquet"
)

REC_COMPARISON_SUMMARY_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_comparison_summary.csv"
)

REC_MISMATCH_EXAMPLES_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_mismatch_examples.csv"
)

ANCHOR_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)


print("=" * 112)
print("=== PROJECT 9 CELL 4 / STEP 2B V2: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===")
print("=" * 112)


# ------------------------------------------------------------
# 4. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativeName",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            str(row.RelativeName).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if compression == "gzip":

        temporary_path = path.with_name(
            path.name + ".tmp.gz"
        )

    else:

        temporary_path = path.with_name(
            path.name + ".tmp"
        )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def normalise_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def detect_column(
    columns,
    candidates,
    role,
    required=True,
):
    lookup = {
        normalise_name(column):
            column
        for column in columns
    }

    for candidate in candidates:

        key = normalise_name(
            candidate
        )

        if key in lookup:

            return lookup[key]

    if required:

        raise RuntimeError(
            "Required column not found.\n"
            f"Role: {role}\n"
            f"Candidates: {candidates}\n"
            f"Available: {columns}"
        )

    return None


def normalise_commit(
    value,
):
    if pd.isna(value):
        return ""

    return str(
        value
    ).strip().lower()


def parse_commit_tokens(
    value,
):
    if pd.isna(value):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"(?i)(?<![0-9a-f])[0-9a-f]{7,40}(?![0-9a-f])",
        text,
    )

    if not tokens:

        rough_tokens = re.split(
            r"[\s,;|\[\]\(\)\"']+",
            text,
        )

        tokens = [
            token
            for token in rough_tokens
            if re.fullmatch(
                r"(?i)[0-9a-f]{7,40}",
                token,
            )
        ]

    output = []
    seen = set()

    for token in tokens:

        token = token.lower()

        if token not in seen:

            seen.add(token)
            output.append(token)

    return output


def as_float_array(
    values,
):
    """
    Robustly accept:
    - pandas Series
    - pandas Index
    - Python lists
    - NumPy arrays
    """

    if isinstance(
        values,
        pd.Series,
    ):

        return pd.to_numeric(
            values,
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

    if isinstance(
        values,
        pd.Index,
    ):

        return pd.to_numeric(
            values.to_series(
                index=np.arange(
                    len(values)
                )
            ),
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

    array = np.asarray(
        values
    )

    if np.issubdtype(
        array.dtype,
        np.number,
    ):

        return array.astype(
            float,
            copy=False,
        )

    return pd.to_numeric(
        pd.Series(
            array.ravel()
        ),
        errors="coerce",
    ).to_numpy(
        dtype=float
    ).reshape(
        array.shape
    )


def finite_max_absolute_difference(
    values,
):
    """
    V2 fix: works directly with both pandas and NumPy values.
    """

    numeric = as_float_array(
        values
    )

    finite = numeric[
        np.isfinite(
            numeric
        )
    ]

    if finite.size == 0:
        return np.nan

    return float(
        np.max(
            np.abs(
                finite
            )
        )
    )


def finite_mean_absolute_difference(
    values,
):
    numeric = as_float_array(
        values
    )

    finite = numeric[
        np.isfinite(
            numeric
        )
    ]

    if finite.size == 0:
        return np.nan

    return float(
        np.mean(
            np.abs(
                finite
            )
        )
    )


# ------------------------------------------------------------
# 5. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    PROJECT_SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FIXED_SPLIT_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 2B V2 inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_checkpoint = json.loads(
    PROJECT_SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)


step1b_status = json.loads(
    STEP1B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step2a_status = json.loads(
    STEP2A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):

    raise AssertionError(
        "Project 9 Step 1B has not passed."
    )


if (
    step2a_status.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
):

    raise AssertionError(
        "Project 9 Step 2A has not passed."
    )


if (
    selection_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
):

    raise AssertionError(
        "Project 9 checkpoint identity differs."
    )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Project 9 source-root SHA-256 differs."
    )


registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != 8
    or set(project_numbers)
    != set(range(1, 9))
):

    raise AssertionError(
        "Completion registry must contain exactly "
        "Projects 1–8."
    )


if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the registry."
    )


# ------------------------------------------------------------
# 6. REVALIDATE FROZEN SOURCE FILES
# ------------------------------------------------------------

source_records = []


for relative_name, metadata in (
    selection_checkpoint[
        "SourceFiles"
    ].items()
):

    source_path = Path(
        metadata[
            "RuntimePath"
        ]
    )

    if not source_path.exists():

        raise FileNotFoundError(
            "Frozen source file is missing:\n"
            f"{source_path}"
        )

    actual_size = int(
        source_path.stat().st_size
    )

    actual_sha256 = calculate_sha256(
        source_path
    )

    if actual_size != int(
        metadata[
            "SizeBytes"
        ]
    ):

        raise AssertionError(
            "Frozen source-file size differs.\n"
            f"File: {relative_name}"
        )

    if actual_sha256 != metadata[
        "SHA256"
    ]:

        raise AssertionError(
            "Frozen source-file SHA-256 differs.\n"
            f"File: {relative_name}"
        )

    source_records.append({
        "RelativeName":
            relative_name,

        "Path":
            str(source_path),

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


source_manifest = pd.DataFrame(
    source_records
)


source_root_sha256 = root_inventory_hash(
    source_manifest
)


if (
    source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Recalculated source-root SHA-256 differs."
    )


source_paths = {
    row.RelativeName:
        Path(row.Path)
    for row in source_manifest.itertuples(
        index=False
    )
}


source_hashes_before = {
    row.RelativeName:
        row.SHA256
    for row in source_manifest.itertuples(
        index=False
    )
}


# ------------------------------------------------------------
# 7. RESOLVE SOURCE SCHEMA
# ------------------------------------------------------------

resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


builds_build_column = (
    resolved_columns[
        "BuildsBuild"
    ]
)

builds_timestamp_column = (
    resolved_columns[
        "BuildsTimestamp"
    ]
)

exe_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

exe_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

exe_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

exe_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

exe_job_column = (
    resolved_columns.get(
        "ExecutionJob"
    )
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


builds_columns = list(
    pd.read_csv(
        source_paths[
            "builds.csv"
        ],
        nrows=0,
    ).columns
)


build_commit_column = detect_column(
    builds_columns,
    [
        "commits",
        "commit",
        "commit_sha",
        "commit_hash",
        "commit_ids",
        "sha",
        "revision",
    ],
    role="builds.csv commit field",
)


entity_columns = list(
    pd.read_csv(
        source_paths[
            "entity_change_history.csv"
        ],
        nrows=0,
    ).columns
)


entity_id_column = detect_column(
    entity_columns,
    [
        "EntityId",
        "entity_id",
        "EntityID",
        "id",
    ],
    role=(
        "entity_change_history.csv entity identifier"
    ),
)


entity_commit_column = detect_column(
    entity_columns,
    [
        "Commit",
        "commit",
        "CommitHash",
        "commit_hash",
        "sha",
    ],
    role=(
        "entity_change_history.csv commit identifier"
    ),
)


id_map_columns = list(
    pd.read_csv(
        source_paths[
            "id_map.csv"
        ],
        nrows=0,
    ).columns
)


id_map_key_column = detect_column(
    id_map_columns,
    [
        "key",
        "path",
        "entity",
        "name",
    ],
    role="id_map.csv entity path",
)


id_map_value_column = detect_column(
    id_map_columns,
    [
        "value",
        "id",
        "EntityId",
        "entity_id",
    ],
    role="id_map.csv entity identifier",
)


print("\nResolved reconstruction schema:")

print(
    "Build commit column:",
    build_commit_column,
)

print(
    "Entity-history commit column:",
    entity_commit_column,
)

print(
    "Entity-history ID column:",
    entity_id_column,
)

print(
    "ID-map key / value:",
    id_map_key_column,
    "/",
    id_map_value_column,
)


# ------------------------------------------------------------
# 8. LOAD FROZEN BUILD CHRONOLOGY
# ------------------------------------------------------------

fixed_split = pd.read_csv(
    FIXED_SPLIT_PATH,
    low_memory=False,
)


if len(
    fixed_split
) != EXPECTED_BUILDS:

    raise AssertionError(
        "Frozen build split does not contain 822 builds."
    )


fixed_split[
    "BuildKey"
] = canonical_identifier(
    fixed_split[
        "Build"
    ]
)


fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


if not fixed_split[
    "BuildKey"
].is_unique:

    raise AssertionError(
        "Frozen build chronology is not unique."
    )


build_order_map = (
    fixed_split
    .set_index(
        "BuildKey"
    )[
        "BuildOrder"
    ]
    .to_dict()
)


global_build_position = {
    build_key:
        int(build_order) - 1
    for build_key, build_order
    in build_order_map.items()
}


# ------------------------------------------------------------
# 9. BUILD → COMMIT TOKEN PROFILE
# ------------------------------------------------------------

builds_commit_data = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    usecols=[
        builds_build_column,
        builds_timestamp_column,
        build_commit_column,
    ],
    low_memory=False,
)


builds_commit_data[
    "BuildKey"
] = canonical_identifier(
    builds_commit_data[
        builds_build_column
    ]
)


builds_commit_data[
    "BuildOrder"
] = builds_commit_data[
    "BuildKey"
].map(
    build_order_map
)


if builds_commit_data[
    "BuildOrder"
].isna().any():

    raise AssertionError(
        "A builds.csv row could not be mapped "
        "to the frozen chronology."
    )


builds_commit_data[
    "CommitTokens"
] = builds_commit_data[
    build_commit_column
].apply(
    parse_commit_tokens
)


builds_commit_data[
    "CommitTokenCount"
] = builds_commit_data[
    "CommitTokens"
].apply(
    len
)


build_commit_token_profile = (
    builds_commit_data[
        [
            "BuildKey",
            "BuildOrder",
            builds_timestamp_column,
            build_commit_column,
            "CommitTokenCount",
            "CommitTokens",
        ]
    ]
    .copy()
)


build_commit_token_profile[
    "CommitTokens"
] = build_commit_token_profile[
    "CommitTokens"
].apply(
    lambda values:
        ",".join(values)
)


build_rows_without_commit_tokens = int(
    build_commit_token_profile[
        "CommitTokenCount"
    ].eq(0).sum()
)


multi_commit_builds = int(
    build_commit_token_profile[
        "CommitTokenCount"
    ].gt(1).sum()
)


build_commit_token_rows = []


for row in builds_commit_data.itertuples(
    index=False
):

    build_key = str(
        row.BuildKey
    )

    build_order = int(
        row.BuildOrder
    )

    commit_tokens = row.CommitTokens

    for token_position, token in enumerate(
        commit_tokens,
        start=1,
    ):

        build_commit_token_rows.append({
            "BuildKey":
                build_key,

            "BuildOrder":
                build_order,

            "TokenPosition":
                token_position,

            "CommitToken":
                token,
        })


build_commit_tokens = pd.DataFrame(
    build_commit_token_rows,
    columns=[
        "BuildKey",
        "BuildOrder",
        "TokenPosition",
        "CommitToken",
    ],
)


# ------------------------------------------------------------
# 10. LOAD ENTITY HISTORY
# ------------------------------------------------------------

entity_history = pd.read_csv(
    source_paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    "EntityId"
] = pd.to_numeric(
    entity_history[
        entity_id_column
    ],
    errors="coerce",
).astype("Int64")


entity_history[
    "CommitNormalised"
] = entity_history[
    entity_commit_column
].apply(
    normalise_commit
)


invalid_entity_rows = int(
    (
        entity_history[
            "EntityId"
        ].isna()
        |
        entity_history[
            "CommitNormalised"
        ].eq("")
    ).sum()
)


entity_history_valid = (
    entity_history.dropna(
        subset=[
            "EntityId",
        ]
    )
    .loc[
        lambda frame:
            frame[
                "CommitNormalised"
            ].ne("")
    ]
    .copy()
)


entity_history_valid[
    "EntityId"
] = entity_history_valid[
    "EntityId"
].astype(int)


unique_history_commits = sorted(
    entity_history_valid[
        "CommitNormalised"
    ].unique().tolist()
)


history_commit_set = set(
    unique_history_commits
)


# ------------------------------------------------------------
# 11. MATCH COMMITS
# ------------------------------------------------------------

commit_match_records = []


for row in build_commit_tokens.itertuples(
    index=False
):

    token = str(
        row.CommitToken
    )

    matched_commit = None
    match_type = None
    candidate_count = 0


    if token in history_commit_set:

        matched_commit = token

        match_type = (
            "EXACT_NORMALISED_COMMIT_TOKEN"
        )

        candidate_count = 1


    else:

        prefix_candidates = [
            commit_value
            for commit_value in unique_history_commits
            if (
                commit_value.startswith(
                    token
                )
                or token.startswith(
                    commit_value
                )
            )
        ]


        candidate_count = len(
            prefix_candidates
        )


        if candidate_count == 1:

            matched_commit = (
                prefix_candidates[0]
            )

            match_type = (
                "UNIQUE_PREFIX_COMMIT_TOKEN"
            )


        elif candidate_count == 0:

            match_type = (
                "UNMATCHED_COMMIT_TOKEN"
            )


        else:

            match_type = (
                "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
            )


    commit_match_records.append({
        "BuildKey":
            str(row.BuildKey),

        "BuildOrder":
            int(row.BuildOrder),

        "TokenPosition":
            int(row.TokenPosition),

        "CommitToken":
            token,

        "MatchedCommit":
            matched_commit,

        "MatchType":
            match_type,

        "CandidateCount":
            candidate_count,
    })


commit_matching_audit = pd.DataFrame(
    commit_match_records,
    columns=[
        "BuildKey",
        "BuildOrder",
        "TokenPosition",
        "CommitToken",
        "MatchedCommit",
        "MatchType",
        "CandidateCount",
    ],
)


exact_commit_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "EXACT_NORMALISED_COMMIT_TOKEN"
    ).sum()
)


prefix_commit_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX_COMMIT_TOKEN"
    ).sum()
)


unmatched_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNMATCHED_COMMIT_TOKEN"
    ).sum()
)


ambiguous_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
    ).sum()
)


if ambiguous_commit_tokens:

    display(
        commit_matching_audit[
            commit_matching_audit[
                "MatchType"
            ].eq(
                "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
            )
        ]
    )

    raise RuntimeError(
        "Ambiguous commit-prefix mappings were found."
    )


matched_commit_tokens = (
    commit_matching_audit[
        commit_matching_audit[
            "MatchedCommit"
        ].notna()
    ]
    .copy()
)


total_commit_tokens = int(
    len(
        commit_matching_audit
    )
)


matched_commit_token_count = int(
    len(
        matched_commit_tokens
    )
)


commit_token_coverage_percent = (
    100.0
    if total_commit_tokens == 0
    else
    100.0
    * matched_commit_token_count
    / total_commit_tokens
)


matched_builds = set(
    matched_commit_tokens[
        "BuildKey"
    ].astype(str)
)


builds_with_no_matched_commit = int(
    (
        ~build_commit_token_profile[
            "BuildKey"
        ].astype(str).isin(
            matched_builds
        )
    ).sum()
)


# ------------------------------------------------------------
# 12. BUILD ↔ ENTITY MAP
# ------------------------------------------------------------

entity_commit_pairs = (
    entity_history_valid[
        [
            "CommitNormalised",
            "EntityId",
        ]
    ]
    .drop_duplicates()
)


build_entity_map = (
    matched_commit_tokens[
        [
            "BuildKey",
            "BuildOrder",
            "CommitToken",
            "MatchedCommit",
            "MatchType",
        ]
    ]
    .merge(
        entity_commit_pairs,
        left_on="MatchedCommit",
        right_on="CommitNormalised",
        how="left",
        validate="many_to_many",
    )
    .drop(
        columns=[
            "CommitNormalised",
        ]
    )
)


build_entity_map = (
    build_entity_map.dropna(
        subset=[
            "EntityId",
        ]
    )
    .copy()
)


build_entity_map[
    "EntityId"
] = build_entity_map[
    "EntityId"
].astype(int)


build_entity_map = (
    build_entity_map.drop_duplicates(
        subset=[
            "BuildKey",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "BuildOrder",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


builds_with_mapped_entities = int(
    build_entity_map[
        "BuildKey"
    ].nunique()
)


mapped_entity_ids = set(
    build_entity_map[
        "EntityId"
    ].astype(int)
)


id_map = pd.read_csv(
    source_paths[
        "id_map.csv"
    ],
    usecols=[
        id_map_key_column,
        id_map_value_column,
    ],
    low_memory=False,
)


id_map[
    "EntityId"
] = pd.to_numeric(
    id_map[
        id_map_value_column
    ],
    errors="coerce",
).astype("Int64")


valid_id_map = (
    id_map.dropna(
        subset=[
            "EntityId",
        ]
    )
    .copy()
)


valid_id_map[
    "EntityId"
] = valid_id_map[
    "EntityId"
].astype(int)


id_map_entity_ids = set(
    valid_id_map[
        "EntityId"
    ]
)


entity_history_ids = set(
    entity_history_valid[
        "EntityId"
    ].astype(int)
)


history_entity_ids_missing_from_id_map = (
    entity_history_ids
    - id_map_entity_ids
)


id_map_entity_ids_missing_from_history = (
    id_map_entity_ids
    - entity_history_ids
)


changed_entities_by_build = {
    str(build_key):
        set(
            group[
                "EntityId"
            ].astype(int)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


clean_changed_entities_by_build = {
    str(build_key):
        changed_entities_by_build.get(
            str(build_key),
            set(),
        )
    for build_key in fixed_split[
        "BuildKey"
    ]
}


entity_changed_builds = {
    int(entity_id):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for entity_id, group
    in build_entity_map.groupby(
        "EntityId",
        sort=False,
    )
}


print("\nCommit/entity mapping summary:")

print(
    "Build commit tokens:",
    total_commit_tokens,
)

print(
    "Exact matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    prefix_commit_matches,
)

print(
    "Unmatched tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous tokens:",
    ambiguous_commit_tokens,
)

print(
    "Token coverage percent:",
    commit_token_coverage_percent,
)

print(
    "Build rows without commit tokens:",
    build_rows_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


# ------------------------------------------------------------
# 13. LOAD EXECUTION HISTORY
# ------------------------------------------------------------

execution_usecols = [
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]


if (
    exe_job_column is not None
    and exe_job_column not in execution_usecols
):

    execution_usecols.append(
        exe_job_column
    )


execution_history = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=execution_usecols,
    low_memory=False,
)


if len(
    execution_history
) != EXPECTED_RAW_ROWS:

    raise AssertionError(
        "Clean execution-history row count differs."
    )


execution_history[
    "BuildKey"
] = canonical_identifier(
    execution_history[
        exe_build_column
    ]
)


execution_history[
    "TestKey"
] = canonical_identifier(
    execution_history[
        exe_test_column
    ]
)


execution_history[
    "BuildOrder"
] = execution_history[
    "BuildKey"
].map(
    build_order_map
)


if execution_history[
    "BuildOrder"
].isna().any():

    raise AssertionError(
        "Some execution builds could not be mapped "
        "to the frozen chronology."
    )


execution_history[
    "BuildOrder"
] = execution_history[
    "BuildOrder"
].astype(int)


execution_history[
    "Verdict"
] = pd.to_numeric(
    execution_history[
        exe_verdict_column
    ],
    errors="raise",
).astype(int)


execution_history[
    "Duration"
] = pd.to_numeric(
    execution_history[
        exe_duration_column
    ],
    errors="coerce",
)


duration_array = execution_history[
    "Duration"
].to_numpy(
    dtype=float
)


invalid_durations = int(
    (
        ~np.isfinite(
            duration_array
        )
        |
        (
            duration_array < 0
        )
    ).sum()
)


if invalid_durations:

    raise AssertionError(
        "Execution history contains invalid durations.\n"
        f"Rows: {invalid_durations}"
    )


raw_duplicate_build_test_rows = int(
    execution_history.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if raw_duplicate_build_test_rows:

    raise AssertionError(
        "Raw execution history contains duplicate "
        "Build/Test rows."
    )


execution_sort_columns = [
    "BuildOrder",
]


if exe_job_column is not None:

    execution_sort_columns.append(
        exe_job_column
    )


execution_sort_columns.append(
    "TestKey"
)


execution_history = (
    execution_history.sort_values(
        execution_sort_columns,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. LOAD MODEL-READY REC ROWS
# ------------------------------------------------------------

dataset_usecols = (
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
    + REC_FEATURE_COLUMNS
)


dataset_rec = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    usecols=dataset_usecols,
    low_memory=False,
)


if len(
    dataset_rec
) != EXPECTED_MODEL_ROWS:

    raise AssertionError(
        "Model-ready dataset row count differs."
    )


dataset_rec[
    "_DatasetRowOrder"
] = np.arange(
    len(
        dataset_rec
    ),
    dtype=np.int64,
)


dataset_rec[
    "BuildKey"
] = canonical_identifier(
    dataset_rec[
        dataset_build_column
    ]
)


dataset_rec[
    "TestKey"
] = canonical_identifier(
    dataset_rec[
        dataset_test_column
    ]
)


model_duplicate_build_test_rows = int(
    dataset_rec.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if model_duplicate_build_test_rows:

    raise AssertionError(
        "Model-ready Build/Test rows are not unique."
    )


for feature in REC_FEATURE_COLUMNS:

    feature_values = pd.to_numeric(
        dataset_rec[
            feature
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    nonfinite_count = int(
        (
            ~np.isfinite(
                feature_values
            )
        ).sum()
    )

    if nonfinite_count:

        raise AssertionError(
            "Original dataset REC feature contains "
            "non-finite values.\n"
            f"Feature: {feature}\n"
            f"Rows: {nonfinite_count}"
        )


requested_pairs_by_test = {
    str(test_key):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in dataset_rec.groupby(
        "TestKey",
        sort=False,
    )
}


requested_pair_count = int(
    len(
        dataset_rec
    )
)


# ------------------------------------------------------------
# 15. RECONSTRUCTION HELPERS
# ------------------------------------------------------------

def calculate_max_test_file_rate(
    target_builds,
    current_changed_entities,
):
    if len(
        target_builds
    ) == 0:

        return -1.0


    maximum_frequency = 0


    for entity_id in current_changed_entities:

        changed_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )


        overlap_count = len(
            changed_builds.intersection(
                target_builds
            )
        )


        if overlap_count > maximum_frequency:

            maximum_frequency = (
                overlap_count
            )


    if maximum_frequency == 0:

        return 0.0


    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_clean_rec_features(
    history,
):
    reconstructed_records = []

    tests_processed = 0
    requested_rows_reconstructed = 0

    total_tests = int(
        history[
            "TestKey"
        ].nunique()
    )


    for test_key, test_history in history.groupby(
        "TestKey",
        sort=False,
    ):

        tests_processed += 1

        test_key = str(
            test_key
        )

        requested_builds = (
            requested_pairs_by_test.get(
                test_key
            )
        )


        if not requested_builds:

            continue


        test_history = (
            test_history.sort_values(
                [
                    "BuildOrder",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        first_build_key = str(
            test_history.iloc[0][
                "BuildKey"
            ]
        )


        if first_build_key not in (
            global_build_position
        ):

            raise AssertionError(
                "First test build is absent from "
                "the global chronology."
            )


        prior_count = 0

        prior_duration_sum = 0.0
        prior_duration_max = -np.inf

        prior_failure_count = 0
        prior_assertion_count = 0
        prior_exception_count = 0
        prior_transition_count = 0

        last_failure_position = None
        last_transition_position = None

        previous_verdict = None
        previous_duration = None

        recent_history = deque(
            maxlen=RECENT_WINDOW
        )

        failure_builds = set()
        transition_builds = set()


        for row in test_history.itertuples(
            index=False
        ):

            current_build = str(
                row.BuildKey
            )

            current_verdict = int(
                row.Verdict
            )

            current_duration = float(
                row.Duration
            )


            if current_build in requested_builds:

                record = {
                    "BuildKey":
                        current_build,

                    "TestKey":
                        test_key,
                }


                if prior_count == 0:

                    for feature in (
                        REC_FEATURE_COLUMNS
                    ):

                        record[
                            feature
                        ] = -1.0


                    record[
                        "REC_Age"
                    ] = 0.0


                else:

                    current_global_position = (
                        global_build_position[
                            current_build
                        ]
                    )


                    first_global_position = (
                        global_build_position[
                            first_build_key
                        ]
                    )


                    age = (
                        current_global_position
                        - first_global_position
                    )


                    if (
                        last_failure_position
                        is None
                    ):

                        last_failure_age = -1.0

                    else:

                        last_failure_age = float(
                            prior_count
                            - 1
                            - last_failure_position
                        )


                    if (
                        last_transition_position
                        is None
                    ):

                        last_transition_age = -1.0

                    else:

                        last_transition_age = float(
                            prior_count
                            - 1
                            - last_transition_position
                        )


                    recent_rows = list(
                        recent_history
                    )


                    recent_length = len(
                        recent_rows
                    )


                    if recent_length == 0:

                        raise AssertionError(
                            "Recent history is unexpectedly "
                            "empty for a non-first execution."
                        )


                    recent_durations = np.asarray(
                        [
                            item[
                                "Duration"
                            ]
                            for item in recent_rows
                        ],
                        dtype=float,
                    )


                    recent_verdicts = np.asarray(
                        [
                            item[
                                "Verdict"
                            ]
                            for item in recent_rows
                        ],
                        dtype=int,
                    )


                    recent_transitions = np.asarray(
                        [
                            item[
                                "Transition"
                            ]
                            for item in recent_rows
                        ],
                        dtype=int,
                    )


                    current_changed_entities = (
                        clean_changed_entities_by_build.get(
                            current_build,
                            set(),
                        )
                    )


                    record.update({
                        "REC_Age":
                            float(age),

                        "REC_LastFailureAge":
                            last_failure_age,

                        "REC_LastTransitionAge":
                            last_transition_age,

                        "REC_RecentAvgExeTime":
                            float(
                                recent_durations.mean()
                            ),

                        "REC_RecentMaxExeTime":
                            float(
                                recent_durations.max()
                            ),

                        "REC_RecentFailRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts != 0
                                )
                                / recent_length
                            ),

                        "REC_RecentAssertRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts == 2
                                )
                                / recent_length
                            ),

                        "REC_RecentExcRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts == 1
                                )
                                / recent_length
                            ),

                        "REC_RecentTransitionRate":
                            float(
                                np.count_nonzero(
                                    recent_transitions == 1
                                )
                                / recent_length
                            ),

                        "REC_TotalAvgExeTime":
                            float(
                                prior_duration_sum
                                / prior_count
                            ),

                        "REC_TotalMaxExeTime":
                            float(
                                prior_duration_max
                            ),

                        "REC_TotalFailRate":
                            float(
                                prior_failure_count
                                / prior_count
                            ),

                        "REC_TotalAssertRate":
                            float(
                                prior_assertion_count
                                / prior_count
                            ),

                        "REC_TotalExcRate":
                            float(
                                prior_exception_count
                                / prior_count
                            ),

                        "REC_TotalTransitionRate":
                            float(
                                prior_transition_count
                                / prior_count
                            ),

                        "REC_LastVerdict":
                            float(
                                previous_verdict
                            ),

                        "REC_LastExeTime":
                            float(
                                previous_duration
                            ),

                        "REC_MaxTestFileFailRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    failure_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),

                        "REC_MaxTestFileTransitionRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    transition_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),
                    })


                reconstructed_records.append(
                    record
                )

                requested_rows_reconstructed += 1


            current_transition = (
                0
                if previous_verdict is None
                else int(
                    current_verdict
                    != previous_verdict
                )
            )


            if current_verdict != 0:

                prior_failure_count += 1

                last_failure_position = (
                    prior_count
                )

                failure_builds.add(
                    current_build
                )


            if current_verdict == 2:

                prior_assertion_count += 1


            if current_verdict == 1:

                prior_exception_count += 1


            if current_transition == 1:

                prior_transition_count += 1

                last_transition_position = (
                    prior_count
                )

                transition_builds.add(
                    current_build
                )


            prior_duration_sum += (
                current_duration
            )


            prior_duration_max = max(
                prior_duration_max,
                current_duration,
            )


            recent_history.append({
                "BuildKey":
                    current_build,

                "Verdict":
                    current_verdict,

                "Duration":
                    current_duration,

                "Transition":
                    current_transition,
            })


            previous_verdict = (
                current_verdict
            )


            previous_duration = (
                current_duration
            )


            prior_count += 1


        if (
            tests_processed % 100 == 0
            or tests_processed == total_tests
        ):

            print(
                "REC reconstruction progress:",
                tests_processed,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                requested_rows_reconstructed,
            )


    reconstructed = pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + REC_FEATURE_COLUMNS
        ),
    )


    return reconstructed


# ------------------------------------------------------------
# 16. RUN RECONSTRUCTION
# ------------------------------------------------------------

print(
    "\nReconstructing all 19 clean REC features..."
)


reconstruction_started = (
    time.perf_counter()
)


clean_rec_reconstructed_keys = (
    reconstruct_clean_rec_features(
        execution_history
    )
)


reconstruction_seconds = float(
    time.perf_counter()
    - reconstruction_started
)


reconstructed_row_count = int(
    len(
        clean_rec_reconstructed_keys
    )
)


reconstructed_duplicate_rows = int(
    clean_rec_reconstructed_keys.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if reconstructed_row_count != (
    requested_pair_count
):

    raise AssertionError(
        "Clean REC reconstruction row count differs.\n"
        f"Expected: {requested_pair_count}\n"
        f"Actual: {reconstructed_row_count}"
    )


if reconstructed_duplicate_rows:

    raise AssertionError(
        "Clean REC reconstruction contains duplicate "
        "Build/Test rows."
    )


for feature in REC_FEATURE_COLUMNS:

    reconstructed_feature_values = (
        pd.to_numeric(
            clean_rec_reconstructed_keys[
                feature
            ],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )


    nonfinite_count = int(
        (
            ~np.isfinite(
                reconstructed_feature_values
            )
        ).sum()
    )


    if nonfinite_count:

        raise AssertionError(
            "Reconstructed REC feature contains "
            "non-finite values.\n"
            f"Feature: {feature}\n"
            f"Rows: {nonfinite_count}"
        )


print(
    "\nClean REC reconstruction completed:"
)

print(
    "Requested rows:",
    requested_pair_count,
)

print(
    "Reconstructed rows:",
    reconstructed_row_count,
)

print(
    "Duplicate reconstructed rows:",
    reconstructed_duplicate_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


# ------------------------------------------------------------
# 17. ALIGN WITH DATASET IN EXACT ORIGINAL ROW ORDER
# ------------------------------------------------------------

comparison_left = (
    dataset_rec[
        [
            "_DatasetRowOrder",
            dataset_build_column,
            dataset_test_column,
            "BuildKey",
            "TestKey",
        ]
        + REC_FEATURE_COLUMNS
    ]
    .copy()
)


comparison = (
    comparison_left.merge(
        clean_rec_reconstructed_keys,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_Original",
            "_Reconstructed",
        ),
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_DatasetRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


expected_row_order = np.arange(
    EXPECTED_MODEL_ROWS,
    dtype=np.int64,
)


actual_row_order = comparison[
    "_DatasetRowOrder"
].to_numpy(
    dtype=np.int64
)


if not np.array_equal(
    expected_row_order,
    actual_row_order,
):

    raise AssertionError(
        "Dataset row order changed during reconstruction join."
    )


missing_reconstructed_rows = int(
    comparison[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


if missing_reconstructed_rows:

    raise AssertionError(
        "Some model-ready rows are missing from "
        "the clean REC reconstruction."
    )


comparison_summary_records = []
mismatch_example_records = []


identifier_frame = (
    comparison[
        [
            dataset_build_column,
            dataset_test_column,
            "BuildKey",
            "TestKey",
        ]
    ]
    .copy()
)


clean_rec_reconstructed = (
    identifier_frame.copy()
)


clean_anchor_offsets = (
    identifier_frame.copy()
)


total_direct_mismatch_values = 0
total_nonzero_offset_values = 0
total_anchored_mismatch_values = 0


for feature in REC_FEATURE_COLUMNS:

    original_values = as_float_array(
        comparison[
            f"{feature}_Original"
        ]
    )


    reconstructed_values = as_float_array(
        comparison[
            f"{feature}_Reconstructed"
        ]
    )


    if not np.isfinite(
        original_values
    ).all():

        raise AssertionError(
            "Original REC values contain non-finite data.\n"
            f"Feature: {feature}"
        )


    if not np.isfinite(
        reconstructed_values
    ).all():

        raise AssertionError(
            "Reconstructed REC values contain "
            "non-finite data.\n"
            f"Feature: {feature}"
        )


    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=COMPARISON_RTOL,
        atol=COMPARISON_ATOL,
        equal_nan=False,
    )


    offsets = (
        original_values
        - reconstructed_values
    )


    anchored_values = (
        reconstructed_values
        + offsets
    )


    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=COMPARISON_RTOL,
        atol=COMPARISON_ATOL,
        equal_nan=False,
    )


    zero_offset_mask = np.isclose(
        offsets,
        0.0,
        rtol=0,
        atol=OFFSET_ZERO_ATOL,
        equal_nan=False,
    )


    direct_mismatches = int(
        (
            ~direct_match_mask
        ).sum()
    )


    nonzero_offsets = int(
        (
            ~zero_offset_mask
        ).sum()
    )


    anchored_mismatches = int(
        (
            ~anchored_match_mask
        ).sum()
    )


    total_direct_mismatch_values += (
        direct_mismatches
    )


    total_nonzero_offset_values += (
        nonzero_offsets
    )


    total_anchored_mismatch_values += (
        anchored_mismatches
    )


    clean_rec_reconstructed[
        feature
    ] = reconstructed_values


    clean_anchor_offsets[
        feature
    ] = offsets


    absolute_differences = np.abs(
        offsets
    )


    comparison_summary_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in (
                    VERDICT_DEPENDENT_REC_FEATURES
                )
                else
                "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in (
                FILE_HISTORY_REC_FEATURES
            ),

        "Rows":
            len(
                comparison
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            direct_mismatches,

        "NonZeroAnchorOffsets":
            nonzero_offsets,

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            anchored_mismatches,

        "MaximumAbsoluteDirectDifference":
            finite_max_absolute_difference(
                absolute_differences
            ),

        "MeanAbsoluteDirectDifference":
            finite_mean_absolute_difference(
                absolute_differences
            ),
    })


    mismatch_positions = np.flatnonzero(
        ~direct_match_mask
    )[:10]


    for position in mismatch_positions:

        mismatch_example_records.append({
            "Feature":
                feature,

            "FeatureClass":
                (
                    "VERDICT_DEPENDENT"
                    if feature in (
                        VERDICT_DEPENDENT_REC_FEATURES
                    )
                    else
                    "VERDICT_INDEPENDENT"
                ),

            "Build":
                comparison.iloc[
                    position
                ][
                    dataset_build_column
                ],

            "Test":
                comparison.iloc[
                    position
                ][
                    dataset_test_column
                ],

            "BuildKey":
                comparison.iloc[
                    position
                ][
                    "BuildKey"
                ],

            "TestKey":
                comparison.iloc[
                    position
                ][
                    "TestKey"
                ],

            "Original":
                float(
                    original_values[
                        position
                    ]
                ),

            "DirectReconstruction":
                float(
                    reconstructed_values[
                        position
                    ]
                ),

            "AnchorOffset":
                float(
                    offsets[
                        position
                    ]
                ),

            "AnchoredValue":
                float(
                    anchored_values[
                        position
                    ]
                ),
        })


rec_comparison_summary = pd.DataFrame(
    comparison_summary_records
)


rec_mismatch_examples = pd.DataFrame(
    mismatch_example_records,
    columns=[
        "Feature",
        "FeatureClass",
        "Build",
        "Test",
        "BuildKey",
        "TestKey",
        "Original",
        "DirectReconstruction",
        "AnchorOffset",
        "AnchoredValue",
    ],
)


offset_matrix = clean_anchor_offsets[
    REC_FEATURE_COLUMNS
].to_numpy(
    dtype=float
)


rows_with_any_nonzero_offset = int(
    (
        ~np.isclose(
            offset_matrix,
            0.0,
            rtol=0,
            atol=OFFSET_ZERO_ATOL,
            equal_nan=False,
        )
    ).any(
        axis=1
    ).sum()
)


dependent_direct_mismatch_values = int(
    rec_comparison_summary.loc[
        rec_comparison_summary[
            "Feature"
        ].isin(
            VERDICT_DEPENDENT_REC_FEATURES
        ),
        "DirectMismatchingRows",
    ].sum()
)


independent_direct_mismatch_values = int(
    rec_comparison_summary.loc[
        rec_comparison_summary[
            "Feature"
        ].isin(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_feature_direct_mismatch_values = int(
    rec_comparison_summary.loc[
        rec_comparison_summary[
            "Feature"
        ].isin(
            FILE_HISTORY_REC_FEATURES
        ),
        "DirectMismatchingRows",
    ].sum()
)


# ------------------------------------------------------------
# 18. CLEAN-ANCHOR VALIDATION
# ------------------------------------------------------------

anchor_validation_records = []


for feature in REC_FEATURE_COLUMNS:

    direct_values = clean_rec_reconstructed[
        feature
    ].to_numpy(
        dtype=float
    )


    offsets = clean_anchor_offsets[
        feature
    ].to_numpy(
        dtype=float
    )


    original_values = dataset_rec[
        feature
    ].to_numpy(
        dtype=float
    )


    reproduced_clean_values = (
        direct_values
        + offsets
    )


    match_mask = np.isclose(
        reproduced_clean_values,
        original_values,
        rtol=COMPARISON_RTOL,
        atol=COMPARISON_ATOL,
        equal_nan=False,
    )


    anchor_validation_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in (
                    VERDICT_DEPENDENT_REC_FEATURES
                )
                else
                "VERDICT_INDEPENDENT"
            ),

        "Rows":
            len(
                original_values
            ),

        "MatchingRows":
            int(
                match_mask.sum()
            ),

        "MismatchingRows":
            int(
                (
                    ~match_mask
                ).sum()
            ),

        "MaximumAbsoluteAnchoredDifference":
            finite_max_absolute_difference(
                reproduced_clean_values
                - original_values
            ),

        "Pass":
            bool(
                match_mask.all()
            ),
    })


anchor_validation = pd.DataFrame(
    anchor_validation_records
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


if failed_anchor_features:

    display(
        anchor_validation[
            ~anchor_validation[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "The clean-anchor formula did not reproduce "
        "dataset.csv exactly."
    )


# ------------------------------------------------------------
# 19. OVERALL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 1B passed",

        "Expected":
            EXPECTED_STEP1B_STATUS,

        "Actual":
            step1b_status[
                "Status"
            ],

        "Pass":
            step1b_status[
                "Status"
            ]
            == EXPECTED_STEP1B_STATUS,
    },

    {
        "Check":
            "Step 2A passed",

        "Expected":
            EXPECTED_STEP2A_STATUS,

        "Actual":
            step2a_status[
                "Status"
            ],

        "Pass":
            step2a_status[
                "Status"
            ]
            == EXPECTED_STEP2A_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            source_root_sha256,

        "Pass":
            source_root_sha256
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Frozen source files",

        "Expected":
            5,

        "Actual":
            len(
                source_manifest
            ),

        "Pass":
            len(
                source_manifest
            ) == 5,
    },

    {
        "Check":
            "Canonical builds",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            len(
                fixed_split
            ),

        "Pass":
            len(
                fixed_split
            )
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Raw execution rows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                execution_history
            ),

        "Pass":
            len(
                execution_history
            )
            == EXPECTED_RAW_ROWS,
    },

    {
        "Check":
            "Model-ready rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                dataset_rec
            ),

        "Pass":
            len(
                dataset_rec
            )
            == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Raw duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows
            == 0,
    },

    {
        "Check":
            "Model duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows
            == 0,
    },

    {
        "Check":
            "Reconstructed REC rows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            reconstructed_row_count,

        "Pass":
            reconstructed_row_count
            == EXPECTED_MODEL_ROWS,
    },

    {
        "Check":
            "Duplicate reconstructed rows",

        "Expected":
            0,

        "Actual":
            reconstructed_duplicate_rows,

        "Pass":
            reconstructed_duplicate_rows
            == 0,
    },

    {
        "Check":
            "Missing reconstructed rows",

        "Expected":
            0,

        "Actual":
            missing_reconstructed_rows,

        "Pass":
            missing_reconstructed_rows
            == 0,
    },

    {
        "Check":
            "REC features reconstructed",

        "Expected":
            19,

        "Actual":
            len(
                REC_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                REC_FEATURE_COLUMNS
            ) == 19,
    },

    {
        "Check":
            "Failed clean-anchor features",

        "Expected":
            0,

        "Actual":
            failed_anchor_features,

        "Pass":
            failed_anchor_features
            == 0,
    },

    {
        "Check":
            "Anchored mismatch values",

        "Expected":
            0,

        "Actual":
            total_anchored_mismatch_values,

        "Pass":
            total_anchored_mismatch_values
            == 0,
    },

    {
        "Check":
            "Ambiguous commit tokens",

        "Expected":
            0,

        "Actual":
            ambiguous_commit_tokens,

        "Pass":
            ambiguous_commit_tokens
            == 0,
    },

    {
        "Check":
            "Commit-token coverage",

        "Expected":
            100.0,

        "Actual":
            commit_token_coverage_percent,

        "Pass":
            bool(
                np.isclose(
                    commit_token_coverage_percent,
                    100.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Builds with mapped entities",

        "Expected":
            EXPECTED_BUILDS,

        "Actual":
            builds_with_mapped_entities,

        "Pass":
            builds_with_mapped_entities
            == EXPECTED_BUILDS,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 2B V2 validation:")

display(
    validation
)


if not failed_checks.empty:

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 2B V2 DID NOT PASS."
    )


# ------------------------------------------------------------
# 20. VERIFY SOURCE AND REGISTRY IMMUTABILITY
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}


source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)


if not source_files_unchanged:

    raise AssertionError(
        "A frozen source file changed during Step 2B V2."
    )


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 2B V2."
    )


# ------------------------------------------------------------
# 21. WRITE OUTPUTS
# ------------------------------------------------------------

REC_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    build_commit_token_profile,
)


atomic_write_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    commit_matching_audit,
)


atomic_write_csv(
    BUILD_ENTITY_MAP_PATH,
    build_entity_map,
    compression="gzip",
)


atomic_write_parquet(
    CLEAN_REC_RECONSTRUCTED_PATH,
    clean_rec_reconstructed,
)


atomic_write_parquet(
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    clean_anchor_offsets,
)


atomic_write_csv(
    REC_COMPARISON_SUMMARY_PATH,
    rec_comparison_summary,
)


atomic_write_csv(
    REC_MISMATCH_EXAMPLES_PATH,
    rec_mismatch_examples,
)


atomic_write_csv(
    ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


atomic_write_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


entity_mapping_summary = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "BuildCommitColumn":
        build_commit_column,

    "EntityHistoryCommitColumn":
        entity_commit_column,

    "EntityHistoryIdColumn":
        entity_id_column,

    "IdMapKeyColumn":
        id_map_key_column,

    "IdMapValueColumn":
        id_map_value_column,

    "BuildRows":
        len(
            builds_commit_data
        ),

    "BuildRowsWithoutCommitTokens":
        build_rows_without_commit_tokens,

    "MultiCommitBuilds":
        multi_commit_builds,

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixCommitMatches":
        prefix_commit_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "MatchedCommitTokenRows":
        matched_commit_token_count,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithNoMatchedCommit":
        builds_with_no_matched_commit,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "MappedEntityIds":
        len(
            mapped_entity_ids
        ),

    "EntityHistoryUniqueIds":
        len(
            entity_history_ids
        ),

    "IdMapUniqueIds":
        len(
            id_map_entity_ids
        ),

    "HistoryEntityIdsMissingFromIdMap":
        len(
            history_entity_ids_missing_from_id_map
        ),

    "IdMapEntityIdsMissingFromHistory":
        len(
            id_map_entity_ids_missing_from_history
        ),

    "InvalidEntityHistoryRows":
        invalid_entity_rows,

    "GeneratedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    ENTITY_MAPPING_SUMMARY_PATH,
    entity_mapping_summary,
)


reconstruction_artifact_paths = [
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    CLEAN_REC_RECONSTRUCTED_PATH,
    CLEAN_REC_ANCHOR_OFFSETS_PATH,
    REC_COMPARISON_SUMMARY_PATH,
    REC_MISMATCH_EXAMPLES_PATH,
    ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
]


reconstruction_artifacts = []


for path in reconstruction_artifact_paths:

    reconstruction_artifacts.append({
        "Path":
            str(path),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


# ------------------------------------------------------------
# 22. REPORT, CHECKPOINT AND STATUS
# ------------------------------------------------------------

report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "ImplementationVersion":
        "STEP_2B_V2_NUMPY_SAFE",

    "SourceRootSHA256":
        source_root_sha256,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatures":
        REC_FEATURE_COLUMNS,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "CommitEntityMapping": {
        "BuildCommitColumn":
            build_commit_column,

        "BuildCommitTokenRows":
            total_commit_tokens,

        "ExactMatches":
            exact_commit_matches,

        "UniquePrefixMatches":
            prefix_commit_matches,

        "UnmatchedTokens":
            unmatched_commit_tokens,

        "AmbiguousTokens":
            ambiguous_commit_tokens,

        "CommitTokenCoveragePercent":
            commit_token_coverage_percent,

        "BuildsWithNoMatchedCommit":
            builds_with_no_matched_commit,

        "BuildsWithMappedEntities":
            builds_with_mapped_entities,

        "BuildEntityRows":
            len(
                build_entity_map
            ),
    },

    "CleanReconstruction": {
        "RawHistoryRows":
            len(
                execution_history
            ),

        "RequestedRows":
            requested_pair_count,

        "ReconstructedRows":
            reconstructed_row_count,

        "DuplicateReconstructedRows":
            reconstructed_duplicate_rows,

        "MissingReconstructedRows":
            missing_reconstructed_rows,

        "ReconstructionSeconds":
            reconstruction_seconds,

        "DirectMismatchingFeatureValues":
            total_direct_mismatch_values,

        "VerdictDependentDirectMismatches":
            dependent_direct_mismatch_values,

        "VerdictIndependentDirectMismatches":
            independent_direct_mismatch_values,

        "FileHistoryDirectMismatches":
            file_feature_direct_mismatch_values,

        "NonZeroAnchorOffsetValues":
            total_nonzero_offset_values,

        "RowsWithAnyNonZeroAnchorOffset":
            rows_with_any_nonzero_offset,

        "AnchoredMismatchingFeatureValues":
            total_anchored_mismatch_values,
    },

    "CleanAnchorPolicy": {
        "Formula":
            (
                "original_clean_feature + "
                "(direct_noisy_reconstruction - "
                "direct_clean_reconstruction)"
            ),

        "EquivalentFormula":
            (
                "direct_noisy_reconstruction + "
                "clean_anchor_offset"
            ),

        "ZeroNoiseExactReproduction":
            total_anchored_mismatch_values
            == 0,

        "FailedAnchorFeatures":
            failed_anchor_features,
    },

    "ReconstructionArtifacts":
        reconstruction_artifacts,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_REPORT_PATH,
    report,
)


rec_checkpoint = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "ImplementationVersion":
        "STEP_2B_V2_NUMPY_SAFE",

    "SourceRootSHA256":
        source_root_sha256,

    "RecentWindow":
        RECENT_WINDOW,

    "RECFeatureCount":
        len(
            REC_FEATURE_COLUMNS
        ),

    "VerdictDependentRECFeatureCount":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatureCount":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "CleanRECReconstructed":
        str(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECReconstructedSHA256":
        calculate_sha256(
            CLEAN_REC_RECONSTRUCTED_PATH
        ),

    "CleanRECAnchorOffsets":
        str(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "CleanRECAnchorOffsetsSHA256":
        calculate_sha256(
            CLEAN_REC_ANCHOR_OFFSETS_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "BuildEntityMapSHA256":
        calculate_sha256(
            BUILD_ENTITY_MAP_PATH
        ),

    "DirectMismatchingFeatureValues":
        total_direct_mismatch_values,

    "NonZeroAnchorOffsetValues":
        total_nonzero_offset_values,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_offset,

    "AnchoredMismatchingFeatureValues":
        total_anchored_mismatch_values,

    "CleanAnchorFormula":
        (
            "original_clean_feature + "
            "(direct_noisy_reconstruction - "
            "direct_clean_reconstruction)"
        ),

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    REC_CHECKPOINT_PATH,
    rec_checkpoint,
)


status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_PASS_STATUS,

    "ImplementationVersion":
        "STEP_2B_V2_NUMPY_SAFE",

    "SourceRootSHA256":
        source_root_sha256,

    "RawHistoryRows":
        len(
            execution_history
        ),

    "ModelRows":
        len(
            dataset_rec
        ),

    "ReconstructedRows":
        reconstructed_row_count,

    "RECFeatures":
        len(
            REC_FEATURE_COLUMNS
        ),

    "DirectMismatchingFeatureValues":
        total_direct_mismatch_values,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_offset,

    "AnchoredMismatchingFeatureValues":
        total_anchored_mismatch_values,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        builds_with_mapped_entities,

    "BuildEntityRows":
        len(
            build_entity_map
        ),

    "ReconstructionSeconds":
        reconstruction_seconds,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            REC_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP2B_STATUS_PATH,
    status,
)


# ------------------------------------------------------------
# 23. FINAL READBACK
# ------------------------------------------------------------

required_outputs = (
    reconstruction_artifact_paths
    + [
        STEP2B_REPORT_PATH,
        REC_CHECKPOINT_PATH,
        STEP2B_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]


if missing_outputs:

    raise RuntimeError(
        "Step 2B V2 outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


final_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_status.get(
        "Status"
    )
    != STEP2B_PASS_STATUS
):

    raise AssertionError(
        "Final Step 2B V2 status differs."
    )


clean_reconstructed_readback = (
    pd.read_parquet(
        CLEAN_REC_RECONSTRUCTED_PATH
    )
)


anchor_offsets_readback = (
    pd.read_parquet(
        CLEAN_REC_ANCHOR_OFFSETS_PATH
    )
)


if len(
    clean_reconstructed_readback
) != EXPECTED_MODEL_ROWS:

    raise AssertionError(
        "Clean reconstruction parquet readback "
        "row count differs."
    )


if len(
    anchor_offsets_readback
) != EXPECTED_MODEL_ROWS:

    raise AssertionError(
        "Anchor-offset parquet readback "
        "row count differs."
    )


# ------------------------------------------------------------
# 24. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nClean REC feature comparison:")

display(
    rec_comparison_summary
)


print("\nClean-anchor validation:")

display(
    anchor_validation
)


if not rec_mismatch_examples.empty:

    print(
        "\nExample direct-reconstruction differences:"
    )

    display(
        rec_mismatch_examples.head(
            50
        )
    )


print("\nCommit matching audit summary:")

display(
    commit_matching_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


# ------------------------------------------------------------
# 25. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 112)
print("=== PROJECT 9 CELL 4 / STEP 2B V2 RESULT ===")
print("=" * 112)

print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nCommit and entity mapping:")

print(
    "Build commit column:",
    build_commit_column,
)

print(
    "Build commit-token rows:",
    total_commit_tokens,
)

print(
    "Exact commit matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    prefix_commit_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_commit_tokens,
)

print(
    "Commit-token coverage:",
    commit_token_coverage_percent,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with mapped entities:",
    builds_with_mapped_entities,
)

print(
    "Build/entity rows:",
    len(
        build_entity_map
    ),
)


print("\nClean REC reconstruction:")

print(
    "Raw history rows:",
    len(
        execution_history
    ),
)

print(
    "Requested model-ready rows:",
    requested_pair_count,
)

print(
    "Reconstructed rows:",
    reconstructed_row_count,
)

print(
    "Duplicate reconstructed rows:",
    reconstructed_duplicate_rows,
)

print(
    "Missing reconstructed rows:",
    missing_reconstructed_rows,
)

print(
    "Reconstruction seconds:",
    reconstruction_seconds,
)


print("\nDirect reconstruction comparison:")

print(
    "Direct mismatching feature values:",
    total_direct_mismatch_values,
)

print(
    "Verdict-dependent direct mismatches:",
    dependent_direct_mismatch_values,
)

print(
    "Verdict-independent direct mismatches:",
    independent_direct_mismatch_values,
)

print(
    "File-history direct mismatches:",
    file_feature_direct_mismatch_values,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_offset,
)

print(
    "Non-zero anchor-offset values:",
    total_nonzero_offset_values,
)


print("\nClean-anchor validation:")

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatching feature values:",
    total_anchored_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    total_anchored_mismatch_values == 0,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nFrozen REC checkpoint:")

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        REC_CHECKPOINT_PATH
    ),
)


print("\nSaved outputs:")

for output_path in required_outputs:

    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP2B_PASS_STATUS,
)

print("=" * 112)

=== PROJECT 9 CELL 4 / STEP 2B V2: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===

Resolved reconstruction schema:
Build commit column: commits
Entity-history commit column: Commit
Entity-history ID column: EntityId
ID-map key / value: key / value

Commit/entity mapping summary:
Build commit tokens: 1052
Exact matches: 1052
Unique-prefix matches: 0
Unmatched tokens: 0
Ambiguous tokens: 0
Token coverage percent: 100.0
Build rows without commit tokens: 0
Builds with no matched commit: 0
Builds with mapped entities: 822
Build/entity rows: 13145

Reconstructing all 19 clean REC features...
REC reconstruction progress: 100 / 1021 tests | reconstructed rows: 14739
REC reconstruction progress: 200 / 1021 tests | reconstructed rows: 28584
REC reconstruction progress: 300 / 1021 tests | reconstructed rows: 38600
REC reconstruction progress: 400 / 1021 tests | reconstructed rows: 47450
REC reconstruction progress: 500 / 1021 tests | reconstructed rows: 53110
REC reconstruction progress: 600 / 10

,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_...,PASS_PROJECT_9_SELECTION_LOCKED_SOURCE_FROZEN_...,True
1,Step 2A passed,PASS_PROJECT_9_SOURCE_SCHEMA_AND_JOIN_STRUCTUR...,PASS_PROJECT_9_SOURCE_SCHEMA_AND_JOIN_STRUCTUR...,True
2,Source root SHA-256,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,True
3,Frozen source files,5,5,True
4,Canonical builds,822,822,True
5,Raw execution rows,472765,472765,True
6,Model-ready rows,79383,79383,True
7,Raw duplicate Build/Test rows,0,0,True
8,Model duplicate Build/Test rows,0,0,True
9,Reconstructed REC rows,79383,79383,True



Clean REC feature comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference
0,REC_Age,VERDICT_INDEPENDENT,False,79383,79383,0,0,79383,0,0.000000e+00,0.000000e+00
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,79383,79383,0,0,79383,0,0.000000e+00,0.000000e+00
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,79383,79383,0,0,79383,0,0.000000e+00,0.000000e+00
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,79383,79383,0,283,79383,0,2.910383e-11,3.138463e-14
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,79383,79383,0,0,79383,0,0.000000e+00,0.000000e+00
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,79383,79383,0,0,79383,0,5.551115e-17,6.992826e-19
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,79383,79383,0,0,79383,0,5.551115e-17,1.363601e-19
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,79383,79383,0,0,79383,0,5.551115e-17,5.650203e-19
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,79383,79383,0,0,79383,0,5.551115e-17,4.223667e-19
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,79383,79383,0,603,79383,0,2.910383e-11,3.911645e-14



Clean-anchor validation:


,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,79383,79383,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,79383,79383,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,79383,79383,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,79383,79383,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,79383,79383,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,79383,79383,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,79383,79383,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,79383,79383,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,79383,79383,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,79383,79383,0,0.0,True



Commit matching audit summary:


,MatchType,Rows
0,EXACT_NORMALISED_COMMIT_TOKEN,1052




=== PROJECT 9 CELL 4 / STEP 2B V2 RESULT ===

Project:
camunda@camunda-bpm-platform
Source root SHA-256: 65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07

Commit and entity mapping:
Build commit column: commits
Build commit-token rows: 1052
Exact commit matches: 1052
Unique-prefix matches: 0
Unmatched commit tokens: 0
Ambiguous commit tokens: 0
Commit-token coverage: 100.0
Builds with no matched commit: 0
Builds with mapped entities: 822
Build/entity rows: 13145

Clean REC reconstruction:
Raw history rows: 472765
Requested model-ready rows: 79383
Reconstructed rows: 79383
Duplicate reconstructed rows: 0
Missing reconstructed rows: 0
Reconstruction seconds: 11.773500948000219

Direct reconstruction comparison:
Direct mismatching feature values: 0
Verdict-dependent direct mismatches: 0
Verdict-independent direct mismatches: 0
File-history direct mismatches: 0
Rows with any non-zero anchor offset: 831
Non-zero anchor-offset values: 886

Clean-anchor validation:
Failed an

In [9]:
# ============================================================
# PROJECT 9 — CELL 5 / STEP 3A
# DETERMINISTIC NOISE PLAN AND FIXED-COHORT FREEZE
#
# PROJECT: camunda@camunda-bpm-platform
#
# This cell:
# - validates the successful REC reconstruction checkpoint
# - freezes raw training-history row order
# - freezes clean model training/evaluation cohorts
# - creates project-specific deterministic random streams
# - creates all 270 noise-condition plans
# - validates nested masks across noise levels
# - validates exact reproducibility
# - freezes mask and noisy-verdict hashes
#
# This cell does NOT:
# - create 270 noisy feature datasets
# - train models
# - modify evaluation data
# - modify source files
# - modify the completion registry
# - modify Projects 1–8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = "camunda"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_PASS_STATUS = (
    "PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)

EXPECTED_RAW_ROWS = 472765
EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_RAW_EVALUATION_ROWS = 158781

EXPECTED_MODEL_ROWS = 79383
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

EXPECTED_CLEAN_RAW_TRAINING_FAILURES = 1447
EXPECTED_CLEAN_MODEL_TRAINING_FAILURES = 1427

EXPECTED_CONDITIONS = 270

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

NOISE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noise_plan_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

SELECTION_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_selection_preflight"
)

FIXED_SPLIT_PATH = (
    SELECTION_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_fixed_build_split.csv.gz"
)

NOISE_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_preflight"
)

CLEAN_RAW_TRAINING_HISTORY_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_raw_training_history.parquet"
)

CLEAN_MODEL_TRAINING_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_model_training.parquet"
)

CLEAN_MODEL_EVALUATION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_model_evaluation.parquet"
)

TRAINING_ROW_MANIFEST_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_training_noise_row_manifest.parquet"
)

NOISE_SEED_STREAMS_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_seed_streams.csv"
)

NOISE_CONDITION_PLAN_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_condition_plan.csv"
)

FAILURE_SUBTYPE_DISTRIBUTION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_failure_subtype_distribution.csv"
)

NOISE_PROTOCOL_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)


print("=" * 112)
print("=== PROJECT 9 CELL 5 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 112)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def normalise_name(value):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    """
    Project-specific deterministic 32-bit seed.

    The same project, repetition and random-stream name
    always produce the same random sequence.
    """

    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def packed_boolean_sha256(mask):
    packed = np.packbits(
        np.asarray(
            mask,
            dtype=np.uint8,
        ),
        bitorder="little",
    )

    return hashlib.sha256(
        packed.tobytes()
    ).hexdigest()


def integer_array_sha256(values):
    values = np.asarray(
        values,
        dtype="<i4",
    )

    return hashlib.sha256(
        values.tobytes(
            order="C"
        )
    ).hexdigest()


def dataframe_content_sha256(
    dataframe,
):
    """
    Deterministic dataframe-content digest.

    Used only for cohort immutability checks.
    """

    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=True,
        categorize=True,
    ).to_numpy(
        dtype="<u8"
    )

    digest = hashlib.sha256()

    digest.update(
        json.dumps(
            list(dataframe.columns),
            separators=(",", ":"),
        ).encode("utf-8")
    )

    digest.update(b"\0")

    digest.update(
        row_hashes.tobytes(
            order="C"
        )
    )

    return digest.hexdigest()


def create_seed_random_streams(
    number_of_rows,
    repetition_seed,
    failure_subtypes,
    subtype_probabilities,
):
    flip_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "flip_mask",
    )

    subtype_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "failure_subtype",
    )

    flip_rng = np.random.default_rng(
        flip_stream_seed
    )

    subtype_rng = np.random.default_rng(
        subtype_stream_seed
    )

    row_uniforms = flip_rng.random(
        number_of_rows
    )

    sampled_failure_subtypes = (
        subtype_rng.choice(
            failure_subtypes,
            size=number_of_rows,
            replace=True,
            p=subtype_probabilities,
        )
        .astype(np.int32)
    )

    return {
        "FlipStreamSeed":
            int(flip_stream_seed),

        "SubtypeStreamSeed":
            int(subtype_stream_seed),

        "FlipUniforms":
            row_uniforms,

        "SampledFailureSubtypes":
            sampled_failure_subtypes,
    }


def construct_condition_arrays(
    clean_verdicts,
    row_uniforms,
    sampled_failure_subtypes,
    noise_percent,
):
    flip_mask = (
        row_uniforms
        < float(noise_percent) / 100.0
    )

    pass_to_failure_mask = (
        flip_mask
        & (clean_verdicts == 0)
    )

    failure_to_pass_mask = (
        flip_mask
        & (clean_verdicts != 0)
    )

    noisy_verdicts = (
        clean_verdicts.copy()
    )

    noisy_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_verdicts[
        failure_to_pass_mask
    ] = 0

    return {
        "FlipMask":
            flip_mask,

        "PassToFailureMask":
            pass_to_failure_mask,

        "FailureToPassMask":
            failure_to_pass_mask,

        "NoisyVerdicts":
            noisy_verdicts,
    }


# ------------------------------------------------------------
# 4. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FIXED_SPLIT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required Step 3A inputs are missing:\n"
        + "\n".join(missing_paths)
    )


selection_checkpoint = json.loads(
    SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

rec_checkpoint = json.loads(
    REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step2b_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step2b_status.get("Status")
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "Project 9 Step 2B V2 has not passed."
    )


if (
    rec_checkpoint.get("Status")
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "Project 9 REC checkpoint status differs."
    )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Project 9 source-root SHA-256 differs."
    )


if (
    rec_checkpoint.get(
        "AnchoredMismatchingFeatureValues"
    )
    != 0
):

    raise AssertionError(
        "The clean REC checkpoint contains anchored mismatches."
    )


# ------------------------------------------------------------
# 5. VALIDATE REGISTRY
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if (
    len(registry) != 8
    or set(project_numbers)
    != set(range(1, 9))
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )

if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the completion registry."
    )


# ------------------------------------------------------------
# 6. RESOLVE FROZEN SOURCE FILES AND COLUMNS
# ------------------------------------------------------------

source_paths = {
    relative_name:
        Path(metadata["RuntimePath"])
    for relative_name, metadata
    in selection_checkpoint[
        "SourceFiles"
    ].items()
}

for relative_name, path in source_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            "Frozen source file is missing:\n"
            f"{relative_name}: {path}"
        )

    expected_hash = (
        selection_checkpoint[
            "SourceFiles"
        ][relative_name]["SHA256"]
    )

    if calculate_sha256(path) != expected_hash:

        raise AssertionError(
            "Frozen source-file hash differs:\n"
            f"{relative_name}"
        )


source_hashes_before = {
    relative_name:
        calculate_sha256(path)
    for relative_name, path
    in source_paths.items()
}


resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)

exe_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

exe_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

exe_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

exe_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

exe_job_column = (
    resolved_columns.get(
        "ExecutionJob"
    )
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)

dataset_duration_column = (
    resolved_columns.get(
        "DatasetDuration"
    )
)


# ------------------------------------------------------------
# 7. LOAD FROZEN BUILD SPLIT
# ------------------------------------------------------------

fixed_split = pd.read_csv(
    FIXED_SPLIT_PATH,
    low_memory=False,
)

fixed_split[
    "BuildKey"
] = canonical_identifier(
    fixed_split["Build"]
)

fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split["BuildOrder"],
    errors="raise",
).astype(int)

training_build_keys = set(
    fixed_split.loc[
        fixed_split[
            "Partition"
        ].eq("TRAINING"),
        "BuildKey",
    ]
)

evaluation_build_keys = set(
    fixed_split.loc[
        fixed_split[
            "Partition"
        ].eq("EVALUATION"),
        "BuildKey",
    ]
)

build_order_map = (
    fixed_split
    .set_index("BuildKey")[
        "BuildOrder"
    ]
    .to_dict()
)

if (
    training_build_keys
    & evaluation_build_keys
):

    raise AssertionError(
        "Training and evaluation build sets overlap."
    )


# ------------------------------------------------------------
# 8. CREATE CANONICAL RAW TRAINING-HISTORY COHORT
# ------------------------------------------------------------

execution_columns = [
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]

if (
    exe_job_column is not None
    and exe_job_column not in execution_columns
):
    execution_columns.append(
        exe_job_column
    )


raw_history = pd.read_csv(
    source_paths["exe.csv"],
    usecols=execution_columns,
    low_memory=False,
)

if len(raw_history) != EXPECTED_RAW_ROWS:

    raise AssertionError(
        "Raw execution-history row count differs."
    )


raw_history[
    "BuildKey"
] = canonical_identifier(
    raw_history[
        exe_build_column
    ]
)

raw_history[
    "TestKey"
] = canonical_identifier(
    raw_history[
        exe_test_column
    ]
)

raw_history[
    "BuildOrder"
] = raw_history[
    "BuildKey"
].map(
    build_order_map
)

if raw_history[
    "BuildOrder"
].isna().any():

    raise AssertionError(
        "Raw execution rows could not be mapped "
        "to the frozen build chronology."
    )

raw_history[
    "BuildOrder"
] = raw_history[
    "BuildOrder"
].astype(int)

raw_history[
    "CleanVerdict"
] = pd.to_numeric(
    raw_history[
        exe_verdict_column
    ],
    errors="raise",
).astype(np.int32)

raw_history[
    "Duration"
] = pd.to_numeric(
    raw_history[
        exe_duration_column
    ],
    errors="raise",
).astype(float)


raw_training_history = (
    raw_history[
        raw_history[
            "BuildKey"
        ].isin(
            training_build_keys
        )
    ]
    .copy()
)

raw_evaluation_history = (
    raw_history[
        raw_history[
            "BuildKey"
        ].isin(
            evaluation_build_keys
        )
    ]
    .copy()
)


sort_columns = [
    "BuildOrder",
]

if exe_job_column is not None:
    sort_columns.append(
        exe_job_column
    )

sort_columns.append(
    "TestKey"
)


raw_training_history = (
    raw_training_history
    .sort_values(
        sort_columns,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

raw_evaluation_history = (
    raw_evaluation_history
    .sort_values(
        sort_columns,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    raw_training_history
) != EXPECTED_RAW_TRAINING_ROWS:

    raise AssertionError(
        "Raw training-history row count differs."
    )

if len(
    raw_evaluation_history
) != EXPECTED_RAW_EVALUATION_ROWS:

    raise AssertionError(
        "Raw evaluation-history row count differs."
    )


raw_training_history[
    "NoiseRowID"
] = np.arange(
    len(raw_training_history),
    dtype=np.int64,
)


if not raw_training_history[
    "NoiseRowID"
].is_unique:

    raise AssertionError(
        "NoiseRowID values are not unique."
    )


raw_training_failures = int(
    raw_training_history[
        "CleanVerdict"
    ].ne(0).sum()
)

raw_training_passes = int(
    raw_training_history[
        "CleanVerdict"
    ].eq(0).sum()
)


if (
    raw_training_failures
    != EXPECTED_CLEAN_RAW_TRAINING_FAILURES
):

    raise AssertionError(
        "Clean raw training-failure count differs."
    )


training_manifest_columns = [
    "NoiseRowID",
    "BuildKey",
    "TestKey",
    "BuildOrder",
    "CleanVerdict",
    "Duration",
]

if exe_job_column is not None:

    raw_training_history[
        "Job"
    ] = raw_training_history[
        exe_job_column
    ]

    training_manifest_columns.insert(
        4,
        "Job",
    )


training_row_manifest = (
    raw_training_history[
        training_manifest_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. FREEZE MODEL-READY TRAINING/EVALUATION COHORTS
# ------------------------------------------------------------

model_data = pd.read_csv(
    source_paths["dataset.csv"],
    low_memory=False,
)

if len(model_data) != EXPECTED_MODEL_ROWS:

    raise AssertionError(
        "Model-ready dataset row count differs."
    )


model_data[
    "BuildKey"
] = canonical_identifier(
    model_data[
        dataset_build_column
    ]
)

model_data[
    "TestKey"
] = canonical_identifier(
    model_data[
        dataset_test_column
    ]
)

model_data[
    "_SourceRowOrder"
] = np.arange(
    len(model_data),
    dtype=np.int64,
)


clean_model_training = (
    model_data[
        model_data[
            "BuildKey"
        ].isin(
            training_build_keys
        )
    ]
    .sort_values(
        "_SourceRowOrder",
        kind="mergesort",
    )
    .drop(
        columns=[
            "BuildKey",
            "TestKey",
            "_SourceRowOrder",
        ]
    )
    .reset_index(drop=True)
)


clean_model_evaluation = (
    model_data[
        model_data[
            "BuildKey"
        ].isin(
            evaluation_build_keys
        )
    ]
    .sort_values(
        "_SourceRowOrder",
        kind="mergesort",
    )
    .drop(
        columns=[
            "BuildKey",
            "TestKey",
            "_SourceRowOrder",
        ]
    )
    .reset_index(drop=True)
)


if (
    len(clean_model_training)
    != EXPECTED_MODEL_TRAINING_ROWS
):

    raise AssertionError(
        "Model-ready training row count differs."
    )


if (
    len(clean_model_evaluation)
    != EXPECTED_MODEL_EVALUATION_ROWS
):

    raise AssertionError(
        "Model-ready evaluation row count differs."
    )


clean_model_training_failures = int(
    pd.to_numeric(
        clean_model_training[
            dataset_verdict_column
        ],
        errors="raise",
    ).ne(0).sum()
)


if (
    clean_model_training_failures
    != EXPECTED_CLEAN_MODEL_TRAINING_FAILURES
):

    raise AssertionError(
        "Clean model training-failure count differs."
    )


model_training_hash_before = (
    dataframe_content_sha256(
        clean_model_training
    )
)

model_evaluation_hash_before = (
    dataframe_content_sha256(
        clean_model_evaluation
    )
)


# ------------------------------------------------------------
# 10. FAILURE-SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

clean_training_verdicts = (
    raw_training_history[
        "CleanVerdict"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


failure_subtype_counts = (
    pd.Series(
        clean_training_verdicts[
            clean_training_verdicts != 0
        ]
    )
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:

    raise RuntimeError(
        "No clean training failure subtype exists."
    )


failure_subtypes = (
    failure_subtype_counts
    .index
    .to_numpy(
        dtype=np.int32
    )
)


failure_subtype_probabilities = (
    failure_subtype_counts
    .to_numpy(
        dtype=float
    )
)


failure_subtype_probabilities /= (
    failure_subtype_probabilities.sum()
)


failure_subtype_distribution = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes,

    "CleanTrainingCount":
        failure_subtype_counts
        .to_numpy(
            dtype=int
        ),

    "SamplingProbability":
        failure_subtype_probabilities,
})


if not np.isclose(
    failure_subtype_probabilities.sum(),
    1.0,
    rtol=0,
    atol=1e-12,
):

    raise AssertionError(
        "Failure-subtype probabilities do not sum to 1."
    )


print("\nFailure subtype distribution:")

display(
    failure_subtype_distribution
)


# ------------------------------------------------------------
# 11. GENERATE ALL 270 DETERMINISTIC CONDITION PLANS
# ------------------------------------------------------------

condition_records = []
seed_stream_records = []

nested_mask_violations = 0
statistical_bound_violations = 0


for repetition_seed in REPETITION_SEEDS:

    streams = create_seed_random_streams(
        number_of_rows=(
            len(
                raw_training_history
            )
        ),

        repetition_seed=(
            repetition_seed
        ),

        failure_subtypes=(
            failure_subtypes
        ),

        subtype_probabilities=(
            failure_subtype_probabilities
        ),
    )


    row_uniforms = streams[
        "FlipUniforms"
    ]

    sampled_failure_subtypes = streams[
        "SampledFailureSubtypes"
    ]


    seed_stream_records.append({
        "Project":
            PROJECT_NAME,

        "RepetitionSeed":
            repetition_seed,

        "FlipStreamSeed":
            streams[
                "FlipStreamSeed"
            ],

        "SubtypeStreamSeed":
            streams[
                "SubtypeStreamSeed"
            ],

        "TrainingExecutionRows":
            len(
                raw_training_history
            ),

        "FlipUniformSHA256":
            hashlib.sha256(
                np.asarray(
                    row_uniforms,
                    dtype="<f8",
                ).tobytes(
                    order="C"
                )
            ).hexdigest(),

        "SampledFailureSubtypeSHA256":
            integer_array_sha256(
                sampled_failure_subtypes
            ),
    })


    previous_mask = np.zeros(
        len(raw_training_history),
        dtype=bool,
    )


    for noise_percent in NOISE_LEVELS:

        condition_arrays = (
            construct_condition_arrays(
                clean_verdicts=(
                    clean_training_verdicts
                ),

                row_uniforms=(
                    row_uniforms
                ),

                sampled_failure_subtypes=(
                    sampled_failure_subtypes
                ),

                noise_percent=(
                    noise_percent
                ),
            )
        )


        flip_mask = condition_arrays[
            "FlipMask"
        ]

        pass_to_failure_mask = (
            condition_arrays[
                "PassToFailureMask"
            ]
        )

        failure_to_pass_mask = (
            condition_arrays[
                "FailureToPassMask"
            ]
        )

        noisy_verdicts = (
            condition_arrays[
                "NoisyVerdicts"
            ]
        )


        adjacent_nested_violations = int(
            (
                previous_mask
                & ~flip_mask
            ).sum()
        )

        nested_mask_violations += (
            adjacent_nested_violations
        )


        flipped_rows = int(
            flip_mask.sum()
        )

        pass_to_failure = int(
            pass_to_failure_mask.sum()
        )

        failure_to_pass = int(
            failure_to_pass_mask.sum()
        )


        noisy_failures = int(
            (
                noisy_verdicts != 0
            ).sum()
        )

        noisy_passes = int(
            (
                noisy_verdicts == 0
            ).sum()
        )


        realised_noise_percent = (
            100.0
            * flipped_rows
            / len(
                raw_training_history
            )
        )


        probability = (
            noise_percent
            / 100.0
        )

        expected_flips = (
            len(
                raw_training_history
            )
            * probability
        )

        standard_deviation = np.sqrt(
            len(
                raw_training_history
            )
            * probability
            * (
                1.0
                - probability
            )
        )


        if noise_percent in {
            0,
            100,
        }:

            within_six_sigma = (
                flipped_rows
                == int(
                    expected_flips
                )
            )

        else:

            within_six_sigma = (
                abs(
                    flipped_rows
                    - expected_flips
                )
                <= 6.0
                * standard_deviation
            )


        if not within_six_sigma:

            statistical_bound_violations += 1


        condition_records.append({
            "ConditionOrder":
                (
                    (
                        repetition_seed
                        - 1
                    )
                    * len(
                        NOISE_LEVELS
                    )
                    + NOISE_LEVELS.index(
                        noise_percent
                    )
                    + 1
                ),

            "ConditionKey":
                (
                    f"noise_{int(noise_percent):02d}"
                    f"__seed_{int(repetition_seed):02d}"
                ),

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "TrainingExecutionRows":
                len(
                    raw_training_history
                ),

            "CleanTrainingFailures":
                raw_training_failures,

            "CleanTrainingPasses":
                raw_training_passes,

            "RawRowsFlipped":
                flipped_rows,

            "RealisedNoisePercent":
                realised_noise_percent,

            "PassToFailure":
                pass_to_failure,

            "FailureToPass":
                failure_to_pass,

            "NoisyTrainingFailures":
                noisy_failures,

            "NoisyTrainingPasses":
                noisy_passes,

            "AdjacentNestedMaskViolations":
                adjacent_nested_violations,

            "WithinSixSigma":
                within_six_sigma,

            "FlipMaskSHA256":
                packed_boolean_sha256(
                    flip_mask
                ),

            "NoisyVerdictSHA256":
                integer_array_sha256(
                    noisy_verdicts
                ),
        })


        previous_mask = (
            flip_mask.copy()
        )


noise_condition_plan = (
    pd.DataFrame(
        condition_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


noise_seed_streams = (
    pd.DataFrame(
        seed_stream_records
    )
    .sort_values(
        "RepetitionSeed",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if (
    len(noise_condition_plan)
    != EXPECTED_CONDITIONS
):

    raise AssertionError(
        "Noise condition count differs."
    )


# ------------------------------------------------------------
# 12. EXACT REPRODUCIBILITY AUDIT
# ------------------------------------------------------------

reproducibility_mismatches = 0

reproduction_cases = [
    (1, 0),
    (1, 5),
    (1, 50),
    (17, 15),
    (17, 40),
    (30, 10),
    (30, 50),
]


for repetition_seed, noise_percent in (
    reproduction_cases
):

    streams = create_seed_random_streams(
        number_of_rows=(
            len(
                raw_training_history
            )
        ),

        repetition_seed=(
            repetition_seed
        ),

        failure_subtypes=(
            failure_subtypes
        ),

        subtype_probabilities=(
            failure_subtype_probabilities
        ),
    )


    reproduced = (
        construct_condition_arrays(
            clean_verdicts=(
                clean_training_verdicts
            ),

            row_uniforms=(
                streams[
                    "FlipUniforms"
                ]
            ),

            sampled_failure_subtypes=(
                streams[
                    "SampledFailureSubtypes"
                ]
            ),

            noise_percent=(
                noise_percent
            ),
        )
    )


    recorded = noise_condition_plan[
        (
            noise_condition_plan[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
        )
        &
        (
            noise_condition_plan[
                "NoisePercent"
            ].eq(
                noise_percent
            )
        )
    ].iloc[0]


    reproduced_mask_hash = (
        packed_boolean_sha256(
            reproduced[
                "FlipMask"
            ]
        )
    )

    reproduced_verdict_hash = (
        integer_array_sha256(
            reproduced[
                "NoisyVerdicts"
            ]
        )
    )


    if (
        reproduced_mask_hash
        != recorded[
            "FlipMaskSHA256"
        ]
    ):

        reproducibility_mismatches += 1


    if (
        reproduced_verdict_hash
        != recorded[
            "NoisyVerdictSHA256"
        ]
    ):

        reproducibility_mismatches += 1


# ------------------------------------------------------------
# 13. VALIDATION TABLE
# ------------------------------------------------------------

zero_noise_rows = noise_condition_plan[
    noise_condition_plan[
        "NoisePercent"
    ].eq(0)
]


validation_records = [
    {
        "Check":
            "Step 2B passed",

        "Expected":
            EXPECTED_STEP2B_STATUS,

        "Actual":
            step2b_status[
                "Status"
            ],

        "Pass":
            step2b_status[
                "Status"
            ]
            == EXPECTED_STEP2B_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            selection_checkpoint[
                "SourceRootSHA256"
            ],

        "Pass":
            selection_checkpoint[
                "SourceRootSHA256"
            ]
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Raw training rows",

        "Expected":
            EXPECTED_RAW_TRAINING_ROWS,

        "Actual":
            len(
                raw_training_history
            ),

        "Pass":
            len(
                raw_training_history
            )
            == EXPECTED_RAW_TRAINING_ROWS,
    },

    {
        "Check":
            "Raw evaluation rows",

        "Expected":
            EXPECTED_RAW_EVALUATION_ROWS,

        "Actual":
            len(
                raw_evaluation_history
            ),

        "Pass":
            len(
                raw_evaluation_history
            )
            == EXPECTED_RAW_EVALUATION_ROWS,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            len(
                clean_model_training
            ),

        "Pass":
            len(
                clean_model_training
            )
            == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            len(
                clean_model_evaluation
            ),

        "Pass":
            len(
                clean_model_evaluation
            )
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Check":
            "Clean raw training failures",

        "Expected":
            EXPECTED_CLEAN_RAW_TRAINING_FAILURES,

        "Actual":
            raw_training_failures,

        "Pass":
            raw_training_failures
            == EXPECTED_CLEAN_RAW_TRAINING_FAILURES,
    },

    {
        "Check":
            "Clean model training failures",

        "Expected":
            EXPECTED_CLEAN_MODEL_TRAINING_FAILURES,

        "Actual":
            clean_model_training_failures,

        "Pass":
            clean_model_training_failures
            == EXPECTED_CLEAN_MODEL_TRAINING_FAILURES,
    },

    {
        "Check":
            "Noise conditions",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                noise_condition_plan
            ),

        "Pass":
            len(
                noise_condition_plan
            )
            == EXPECTED_CONDITIONS,
    },

    {
        "Check":
            "Noise levels",

        "Expected":
            len(
                NOISE_LEVELS
            ),

        "Actual":
            noise_condition_plan[
                "NoisePercent"
            ].nunique(),

        "Pass":
            noise_condition_plan[
                "NoisePercent"
            ].nunique()
            == len(
                NOISE_LEVELS
            ),
    },

    {
        "Check":
            "Repetition seeds",

        "Expected":
            len(
                REPETITION_SEEDS
            ),

        "Actual":
            noise_condition_plan[
                "RepetitionSeed"
            ].nunique(),

        "Pass":
            noise_condition_plan[
                "RepetitionSeed"
            ].nunique()
            == len(
                REPETITION_SEEDS
            ),
    },

    {
        "Check":
            "Nested-mask violations",

        "Expected":
            0,

        "Actual":
            nested_mask_violations,

        "Pass":
            nested_mask_violations == 0,
    },

    {
        "Check":
            "Zero-percent flipped rows",

        "Expected":
            0,

        "Actual":
            int(
                zero_noise_rows[
                    "RawRowsFlipped"
                ].sum()
            ),

        "Pass":
            int(
                zero_noise_rows[
                    "RawRowsFlipped"
                ].sum()
            ) == 0,
    },

    {
        "Check":
            "Statistical-bound violations",

        "Expected":
            0,

        "Actual":
            statistical_bound_violations,

        "Pass":
            statistical_bound_violations
            == 0,
    },

    {
        "Check":
            "Reproducibility mismatches",

        "Expected":
            0,

        "Actual":
            reproducibility_mismatches,

        "Pass":
            reproducibility_mismatches
            == 0,
    },

    {
        "Check":
            "Failure-subtype probability sum",

        "Expected":
            1.0,

        "Actual":
            float(
                failure_subtype_probabilities.sum()
            ),

        "Pass":
            bool(
                np.isclose(
                    failure_subtype_probabilities.sum(),
                    1.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 3A validation:")

display(validation)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(failed_checks)

    raise RuntimeError(
        "PROJECT 9 STEP 3A DID NOT PASS."
    )


# ------------------------------------------------------------
# 14. VERIFY IMMUTABILITY BEFORE WRITING
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_sha256(path)
    for relative_name, path
    in source_paths.items()
}

source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)

if not source_files_unchanged:

    raise AssertionError(
        "A frozen source file changed during Step 3A."
    )


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)

if (
    registry_sha256_after
    != registry_sha256_before
):

    raise AssertionError(
        "Completion registry changed during Step 3A."
    )


model_training_hash_after = (
    dataframe_content_sha256(
        clean_model_training
    )
)

model_evaluation_hash_after = (
    dataframe_content_sha256(
        clean_model_evaluation
    )
)

if (
    model_training_hash_after
    != model_training_hash_before
):

    raise AssertionError(
        "Clean model-training cohort changed in memory."
    )

if (
    model_evaluation_hash_after
    != model_evaluation_hash_before
):

    raise AssertionError(
        "Clean evaluation cohort changed in memory."
    )


# ------------------------------------------------------------
# 15. WRITE FROZEN COHORTS AND NOISE PLAN
# ------------------------------------------------------------

NOISE_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


clean_raw_training_output = (
    raw_training_history[
        [
            exe_build_column,
            exe_test_column,
            exe_verdict_column,
            exe_duration_column,
        ]
        + (
            [exe_job_column]
            if exe_job_column is not None
            else []
        )
        + [
            "BuildKey",
            "TestKey",
            "BuildOrder",
            "CleanVerdict",
            "NoiseRowID",
        ]
    ]
    .copy()
)


atomic_write_parquet(
    CLEAN_RAW_TRAINING_HISTORY_PATH,
    clean_raw_training_output,
)

atomic_write_parquet(
    CLEAN_MODEL_TRAINING_PATH,
    clean_model_training,
)

atomic_write_parquet(
    CLEAN_MODEL_EVALUATION_PATH,
    clean_model_evaluation,
)

atomic_write_parquet(
    TRAINING_ROW_MANIFEST_PATH,
    training_row_manifest,
)

atomic_write_csv(
    NOISE_SEED_STREAMS_PATH,
    noise_seed_streams,
)

atomic_write_csv(
    NOISE_CONDITION_PLAN_PATH,
    noise_condition_plan,
)

atomic_write_csv(
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
    failure_subtype_distribution,
)

atomic_write_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 16. OUTPUT READBACK VALIDATION
# ------------------------------------------------------------

training_history_readback = pd.read_parquet(
    CLEAN_RAW_TRAINING_HISTORY_PATH
)

model_training_readback = pd.read_parquet(
    CLEAN_MODEL_TRAINING_PATH
)

model_evaluation_readback = pd.read_parquet(
    CLEAN_MODEL_EVALUATION_PATH
)

condition_plan_readback = pd.read_csv(
    NOISE_CONDITION_PLAN_PATH,
    low_memory=False,
)


if (
    len(training_history_readback)
    != EXPECTED_RAW_TRAINING_ROWS
):

    raise AssertionError(
        "Raw training-history parquet readback differs."
    )


if (
    len(model_training_readback)
    != EXPECTED_MODEL_TRAINING_ROWS
):

    raise AssertionError(
        "Model-training parquet readback differs."
    )


if (
    len(model_evaluation_readback)
    != EXPECTED_MODEL_EVALUATION_ROWS
):

    raise AssertionError(
        "Evaluation parquet readback differs."
    )


if (
    len(condition_plan_readback)
    != EXPECTED_CONDITIONS
):

    raise AssertionError(
        "Noise-condition plan readback differs."
    )


# ------------------------------------------------------------
# 17. WRITE PROTOCOL, REPORT, CHECKPOINT AND STATUS
# ------------------------------------------------------------

noise_protocol = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_PASS_STATUS,

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "MaskRule":
        (
            "project-specific deterministic uniform "
            "random value per raw training execution; "
            "flip when uniform < noise proportion"
        ),

    "NestedMaskRule":
        (
            "same repetition seed and flip-uniform stream "
            "are reused at all noise levels"
        ),

    "PassToFailureRule":
        (
            "replace verdict 0 with a subtype sampled from "
            "the clean project-specific training-failure "
            "subtype distribution"
        ),

    "FailureToPassRule":
        "replace every non-zero verdict with 0",

    "RandomStreams": {
        "Algorithm":
            "numpy default_rng / PCG64",

        "SeedDerivation":
            (
                "first eight bytes of SHA-256("
                "project|repetition|stream), little-endian, "
                "modulo 2^32"
            ),

        "FlipStreamName":
            "flip_mask",

        "FailureSubtypeStreamName":
            "failure_subtype",
    },

    "FixedCohorts": {
        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "ModelTrainingRows":
            EXPECTED_MODEL_TRAINING_ROWS,

        "ModelEvaluationRows":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "RawTrainingHistory":
            str(
                CLEAN_RAW_TRAINING_HISTORY_PATH
            ),

        "ModelTrainingData":
            str(
                CLEAN_MODEL_TRAINING_PATH
            ),

        "ModelEvaluationData":
            str(
                CLEAN_MODEL_EVALUATION_PATH
            ),

        "TrainingNoiseRowManifest":
            str(
                TRAINING_ROW_MANIFEST_PATH
            ),
    },

    "FailureSubtypeDistribution":
        {
            str(
                int(row.FailureSubtype)
            ):
                float(
                    row.SamplingProbability
                )
            for row in (
                failure_subtype_distribution
                .itertuples(index=False)
            )
        },

    "ConditionPlan":
        str(
            NOISE_CONDITION_PLAN_PATH
        ),

    "NestedMaskViolations":
        nested_mask_violations,

    "ReproducibilityMismatches":
        reproducibility_mismatches,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    NOISE_PROTOCOL_PATH,
    noise_protocol,
)


report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_PASS_STATUS,

    "RawTrainingRows":
        len(
            raw_training_history
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawTrainingPasses":
        raw_training_passes,

    "ModelTrainingRows":
        len(
            clean_model_training
        ),

    "ModelEvaluationRows":
        len(
            clean_model_evaluation
        ),

    "NoiseLevels":
        len(
            NOISE_LEVELS
        ),

    "RepetitionSeeds":
        len(
            REPETITION_SEEDS
        ),

    "Conditions":
        len(
            noise_condition_plan
        ),

    "NestedMaskViolations":
        nested_mask_violations,

    "StatisticalBoundViolations":
        statistical_bound_violations,

    "ReproducibilityMismatches":
        reproducibility_mismatches,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "EvaluationCohortHash":
        model_evaluation_hash_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3A_REPORT_PATH,
    report,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_PASS_STATUS,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RawTrainingHistory":
        str(
            CLEAN_RAW_TRAINING_HISTORY_PATH
        ),

    "RawTrainingHistorySHA256":
        calculate_sha256(
            CLEAN_RAW_TRAINING_HISTORY_PATH
        ),

    "ModelTrainingData":
        str(
            CLEAN_MODEL_TRAINING_PATH
        ),

    "ModelTrainingDataSHA256":
        calculate_sha256(
            CLEAN_MODEL_TRAINING_PATH
        ),

    "ModelEvaluationData":
        str(
            CLEAN_MODEL_EVALUATION_PATH
        ),

    "ModelEvaluationDataSHA256":
        calculate_sha256(
            CLEAN_MODEL_EVALUATION_PATH
        ),

    "TrainingNoiseRowManifest":
        str(
            TRAINING_ROW_MANIFEST_PATH
        ),

    "TrainingNoiseRowManifestSHA256":
        calculate_sha256(
            TRAINING_ROW_MANIFEST_PATH
        ),

    "NoiseConditionPlan":
        str(
            NOISE_CONDITION_PLAN_PATH
        ),

    "NoiseConditionPlanSHA256":
        calculate_sha256(
            NOISE_CONDITION_PLAN_PATH
        ),

    "NoiseSeedStreams":
        str(
            NOISE_SEED_STREAMS_PATH
        ),

    "NoiseSeedStreamsSHA256":
        calculate_sha256(
            NOISE_SEED_STREAMS_PATH
        ),

    "NoiseProtocol":
        str(
            NOISE_PROTOCOL_PATH
        ),

    "NoiseProtocolSHA256":
        calculate_sha256(
            NOISE_PROTOCOL_PATH
        ),

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "NestedMaskViolations":
        nested_mask_violations,

    "ReproducibilityMismatches":
        reproducibility_mismatches,

    "EvaluationCohortContentSHA256":
        model_evaluation_hash_after,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_checks),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    NOISE_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_PASS_STATUS,

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "NestedMaskViolations":
        nested_mask_violations,

    "ReproducibilityMismatches":
        reproducibility_mismatches,

    "Checkpoint":
        str(
            NOISE_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            NOISE_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 18. DISPLAY CONDITION PLAN SUMMARY
# ------------------------------------------------------------

print("\nNoise condition summary by level:")

condition_level_summary = (
    noise_condition_plan
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanFlippedRows=(
            "RawRowsFlipped",
            "mean",
        ),

        MinimumFlippedRows=(
            "RawRowsFlipped",
            "min",
        ),

        MaximumFlippedRows=(
            "RawRowsFlipped",
            "max",
        ),

        MeanRealisedNoisePercent=(
            "RealisedNoisePercent",
            "mean",
        ),

        MeanPassToFailure=(
            "PassToFailure",
            "mean",
        ),

        MeanFailureToPass=(
            "FailureToPass",
            "mean",
        ),
    )
)

display(
    condition_level_summary
)


print("\nFirst 12 frozen conditions:")

display(
    noise_condition_plan.head(12)
)


# ------------------------------------------------------------
# 19. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 112)
print("=== PROJECT 9 CELL 5 / STEP 3A RESULT ===")
print("=" * 112)

print("\nProject:")

print(
    PROJECT_NAME
)


print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training_history
    ),
)

print(
    "Raw training failures / passes:",
    raw_training_failures,
    "/",
    raw_training_passes,
)

print(
    "Model training rows:",
    len(
        clean_model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        clean_model_evaluation
    ),
)

print(
    "Evaluation cohort content SHA-256:",
    model_evaluation_hash_after,
)


print("\nNoise experiment plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        noise_condition_plan
    ),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)

print(
    "Statistical-bound violations:",
    statistical_bound_violations,
)

print(
    "Reproducibility mismatches:",
    reproducibility_mismatches,
)


print("\nFailure subtype sampling:")

print(
    "Failure subtypes:",
    failure_subtypes.tolist(),
)

print(
    "Sampling probabilities:",
    failure_subtype_probabilities.tolist(),
)


print("\nValidation:")

print(
    "Checks:",
    len(validation),
)

print(
    "Failed checks:",
    len(failed_checks),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        NOISE_CHECKPOINT_PATH
    ),
)


print("\nSaved outputs:")

for output_path in [
    CLEAN_RAW_TRAINING_HISTORY_PATH,
    CLEAN_MODEL_TRAINING_PATH,
    CLEAN_MODEL_EVALUATION_PATH,
    TRAINING_ROW_MANIFEST_PATH,
    NOISE_SEED_STREAMS_PATH,
    NOISE_CONDITION_PLAN_PATH,
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
    NOISE_PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
    STEP3A_REPORT_PATH,
    NOISE_CHECKPOINT_PATH,
    STEP3A_STATUS_PATH,
]:

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP3A_PASS_STATUS,
)

print("=" * 112)

=== PROJECT 9 CELL 5 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Failure subtype distribution:


,FailureSubtype,CleanTrainingCount,SamplingProbability
0,1,1149,0.794057
1,2,298,0.205943



Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 2B passed,PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_AN...,PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_AN...,True
1,Source root SHA-256,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,True
2,Raw training rows,313984,313984,True
3,Raw evaluation rows,158781,158781,True
4,Model training rows,60880,60880,True
5,Model evaluation rows,18503,18503,True
6,Clean raw training failures,1447,1447,True
7,Clean model training failures,1427,1427,True
8,Noise conditions,270,270,True
9,Noise levels,9,9,True



Noise condition summary by level:


,NoisePercent,Seeds,MeanFlippedRows,MinimumFlippedRows,MaximumFlippedRows,MeanRealisedNoisePercent,MeanPassToFailure,MeanFailureToPass
0,0,30,0.000000,0,0,0.000000,0.000000,0.000000
1,5,30,15688.666667,15432,15910,4.996645,15614.533333,74.133333
2,10,30,31386.233333,31088,31761,9.996125,31238.933333,147.300000
3,15,30,47063.166667,46709,47424,14.989033,46838.966667,224.200000
4,20,30,62780.233333,62224,63308,19.994724,62479.333333,300.900000
5,25,30,78479.600000,78037,79048,24.994777,78107.866667,371.733333
6,30,30,94174.300000,93578,94770,29.993344,93730.466667,443.833333
7,40,30,125580.700000,124987,126054,39.995892,124992.466667,588.233333
8,50,30,156968.900000,156532,157426,49.992643,156235.300000,733.600000



First 12 frozen conditions:


,ConditionOrder,ConditionKey,Project,ProjectSlug,NoisePercent,RepetitionSeed,TrainingExecutionRows,CleanTrainingFailures,CleanTrainingPasses,RawRowsFlipped,RealisedNoisePercent,PassToFailure,FailureToPass,NoisyTrainingFailures,NoisyTrainingPasses,AdjacentNestedMaskViolations,WithinSixSigma,FlipMaskSHA256,NoisyVerdictSHA256
0,1,noise_00__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,0,1,313984,1447,312537,0,0.000000,0,0,1447,312537,0,True,80984893295e46238b23f00efa6c0e256659b33c7f878c...,0abe461008e2c018091675246113c4cfdcbfb0f81c35a4...
1,2,noise_05__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,5,1,313984,1447,312537,15657,4.986560,15577,80,16944,297040,0,True,1f341af97485477427ef62bdcd767e32327dd85039cd63...,c885434a1f879f2b806726e0af2ca8b56996deeda4a5f4...
2,3,noise_10__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,10,1,313984,1447,312537,31268,9.958469,31099,169,32377,281607,0,True,536d8780dec264af41c18cb7d03936447114b55b132c8c...,d432cb9c2283c24405b5fb25132813c54693267fa2d438...
3,4,noise_15__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,15,1,313984,1447,312537,46709,14.876236,46469,240,47676,266308,0,True,111bee52a34c07391073b9cee8d63427b53c28ca5c328b...,e13b3e41c6035c6826f6a6eba64afe16fe7eeb52b1560d...
4,5,noise_20__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,20,1,313984,1447,312537,62401,19.873943,62089,312,63224,250760,0,True,6b95ce3d8afb2d808e251acbcdf07c471b69fbab23ec8a...,b2e0ffdf5d1cc79609de3a3ad87d95868ecdb57c549aed...
5,6,noise_25__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,25,1,313984,1447,312537,78158,24.892351,77769,389,78827,235157,0,True,1370430e30eaa00fdd2ab7400e7f7ca96b42fcc20e306c...,b581920c4ea703d7607a23a373655c1bf6467bb80cc0b5...
6,7,noise_30__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,30,1,313984,1447,312537,93578,29.803429,93114,464,94097,219887,0,True,a4a9a905f053c67d4e84e69813ce52b7bc9f2d5fbe442a...,8551f3bddad6b373b16d38588f34e84aeda167e12a2e80...
7,8,noise_40__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,40,1,313984,1447,312537,124987,39.806805,124373,614,125206,188778,0,True,aa7d1e7d0c8a2ad1458a7f0dc34e203c76ea6067d218c6...,553a45773cc77530cc6a7e280a2138798a1e33c5cea62e...
8,9,noise_50__seed_01,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,50,1,313984,1447,312537,156700,49.907002,155949,751,156645,157339,0,True,123db5e9b82df94883d70abb95e32613a15ef80325212c...,4d7d462b4e374802df7328f308590d4f369af8bb0dd3a1...
9,10,noise_00__seed_02,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,0,2,313984,1447,312537,0,0.000000,0,0,1447,312537,0,True,80984893295e46238b23f00efa6c0e256659b33c7f878c...,0abe461008e2c018091675246113c4cfdcbfb0f81c35a4...




=== PROJECT 9 CELL 5 / STEP 3A RESULT ===

Project:
camunda@camunda-bpm-platform

Fixed cohorts:
Raw training rows: 313984
Raw training failures / passes: 1447 / 312537
Model training rows: 60880
Model evaluation rows: 18503
Evaluation cohort content SHA-256: 0f516fe6a8dd5c7150145caae2827aa5e2acd079fe2cef7246a2c28819f08a61

Noise experiment plan:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
Nested-mask violations: 0
Statistical-bound violations: 0
Reproducibility mismatches: 0

Failure subtype sampling:
Failure subtypes: [1, 2]
Sampling probabilities: [0.7940566689702834, 0.20594333102971665]

Validation:
Checks: 17
Failed checks: 0

Noise-plan checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_09_noise_plan_checkpoint.json
Checkpoint SHA-256: afa13cd38195b0b9bed549cb7a8344edd246bebbc02f4ec2598a05f4751d0d37

Saved outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/camunda__camunda-bpm-platform/camunda_noise_pr

In [10]:
# ============================================================
# PROJECT 9 — CELL 6 / STEP 3B
# NOISY REC ENGINE SENTINEL VALIDATION AND FREEZE
#
# PROJECT: camunda@camunda-bpm-platform
#
# This cell:
# - validates Step 3A and all frozen checkpoints
# - regenerates representative deterministic noise conditions
# - verifies condition mask/verdict hashes against Step 3A
# - maps noisy raw verdicts to model-training labels
# - recomputes all 13 verdict-dependent REC features
# - preserves the 6 verdict-independent REC features exactly
# - proves exact clean reproduction at 0% noise
# - proves positive noise changes verdict-dependent history
# - confirms evaluation data remains unchanged
#
# It does NOT:
# - train models
# - materialise all 270 noisy datasets
# - modify frozen sources
# - modify the completion registry
# - modify Projects 1–8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from IPython.display import display

import hashlib
import json
import re
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = "camunda"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP3B_PASS_STATUS = (
    "PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)

EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503
EXPECTED_CONDITIONS = 270

RECENT_WINDOW = 6

COMPARISON_RTOL = 1e-9
COMPARISON_ATOL = 1e-9


# Representative conditions:
# - all four important noise ranges for Seed 1
# - two additional independent seeds

SENTINEL_CONDITIONS = [
    (1, 0),
    (1, 5),
    (1, 25),
    (1, 50),
    (17, 15),
    (30, 40),
]


# ------------------------------------------------------------
# 2. REC FEATURE PROTOCOL
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected exactly 19 REC features."
    )


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:
    raise AssertionError(
        "Expected exactly 13 verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:
    raise AssertionError(
        "Expected exactly six verdict-independent REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    & set(VERDICT_INDEPENDENT_REC_FEATURES)
):
    raise AssertionError(
        "REC dependency classes overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "REC dependency classes do not cover all features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

NOISE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noise_plan_checkpoint.json"
)

NOISY_REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noisy_rec_engine_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

NOISE_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_preflight"
)

FAILURE_SUBTYPE_DISTRIBUTION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_failure_subtype_distribution.csv"
)

NOISY_REC_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noisy_rec_preflight"
)

SENTINEL_SUMMARY_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noisy_rec_sentinel_summary.csv"
)

FEATURE_CHANGE_SUMMARY_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noisy_rec_feature_change_summary.csv"
)

STEP3B_VALIDATION_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_validation.csv"
)

NOISY_REC_PROTOCOL_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_noisy_rec_protocol.json"
)

STEP3B_REPORT_PATH = (
    NOISY_REC_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_report.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)


print("=" * 114)
print("=== PROJECT 9 CELL 6 / STEP 3B: NOISY REC ENGINE SENTINEL VALIDATION ===")
print("=" * 114)


# ------------------------------------------------------------
# 4. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def normalise_name(value):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def packed_boolean_sha256(mask):

    packed = np.packbits(
        np.asarray(
            mask,
            dtype=np.uint8,
        ),
        bitorder="little",
    )

    return hashlib.sha256(
        packed.tobytes()
    ).hexdigest()


def integer_array_sha256(values):

    values = np.asarray(
        values,
        dtype="<i4",
    )

    return hashlib.sha256(
        values.tobytes(
            order="C"
        )
    ).hexdigest()


def float_matrix_sha256(values):

    values = np.asarray(
        values,
        dtype="<f8",
    )

    return hashlib.sha256(
        values.tobytes(
            order="C"
        )
    ).hexdigest()


def create_seed_random_streams(
    number_of_rows,
    repetition_seed,
    failure_subtypes,
    subtype_probabilities,
):
    flip_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "flip_mask",
    )

    subtype_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "failure_subtype",
    )

    flip_rng = np.random.default_rng(
        flip_stream_seed
    )

    subtype_rng = np.random.default_rng(
        subtype_stream_seed
    )

    return {
        "FlipUniforms":
            flip_rng.random(
                number_of_rows
            ),

        "SampledFailureSubtypes":
            subtype_rng.choice(
                failure_subtypes,
                size=number_of_rows,
                replace=True,
                p=subtype_probabilities,
            ).astype(np.int32),
    }


def construct_condition_arrays(
    clean_verdicts,
    row_uniforms,
    sampled_failure_subtypes,
    noise_percent,
):
    flip_mask = (
        row_uniforms
        < float(noise_percent) / 100.0
    )

    pass_to_failure_mask = (
        flip_mask
        & (clean_verdicts == 0)
    )

    failure_to_pass_mask = (
        flip_mask
        & (clean_verdicts != 0)
    )

    noisy_verdicts = (
        clean_verdicts.copy()
    )

    noisy_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_verdicts[
        failure_to_pass_mask
    ] = 0

    return {
        "FlipMask":
            flip_mask,

        "NoisyVerdicts":
            noisy_verdicts,

        "PassToFailure":
            int(
                pass_to_failure_mask.sum()
            ),

        "FailureToPass":
            int(
                failure_to_pass_mask.sum()
            ),
    }


def mismatch_count(
    left,
    right,
):
    left = np.asarray(
        left,
        dtype=float,
    )

    right = np.asarray(
        right,
        dtype=float,
    )

    return int(
        (
            ~np.isclose(
                left,
                right,
                rtol=COMPARISON_RTOL,
                atol=COMPARISON_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


# ------------------------------------------------------------
# 5. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required Step 3B inputs are missing:\n"
        + "\n".join(missing_paths)
    )


selection_checkpoint = json.loads(
    SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

rec_checkpoint = json.loads(
    REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noise_checkpoint = json.loads(
    NOISE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step2b_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3a_status = json.loads(
    STEP3A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step2b_status.get("Status")
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "Project 9 Step 2B has not passed."
    )


if (
    rec_checkpoint.get("Status")
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "Project 9 REC checkpoint differs."
    )


if (
    step3a_status.get("Status")
    != EXPECTED_STEP3A_STATUS
):

    raise AssertionError(
        "Project 9 Step 3A has not passed."
    )


if (
    noise_checkpoint.get("Status")
    != EXPECTED_STEP3A_STATUS
):

    raise AssertionError(
        "Project 9 noise checkpoint differs."
    )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Project 9 source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 6. VALIDATE REGISTRY
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if (
    len(registry) != 8
    or set(project_numbers)
    != set(range(1, 9))
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )

if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the completion registry."
    )


# ------------------------------------------------------------
# 7. LOAD FROZEN INPUT PATHS
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingHistory"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingData"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationData"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "NoiseConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

anchor_offsets_path = Path(
    rec_checkpoint[
        "CleanRECAnchorOffsets"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)


frozen_path_hashes = {
    raw_training_path:
        noise_checkpoint[
            "RawTrainingHistorySHA256"
        ],

    model_training_path:
        noise_checkpoint[
            "ModelTrainingDataSHA256"
        ],

    model_evaluation_path:
        noise_checkpoint[
            "ModelEvaluationDataSHA256"
        ],

    condition_plan_path:
        noise_checkpoint[
            "NoiseConditionPlanSHA256"
        ],

    clean_direct_rec_path:
        rec_checkpoint[
            "CleanRECReconstructedSHA256"
        ],

    anchor_offsets_path:
        rec_checkpoint[
            "CleanRECAnchorOffsetsSHA256"
        ],

    build_entity_map_path:
        rec_checkpoint[
            "BuildEntityMapSHA256"
        ],
}


for path, expected_hash in (
    frozen_path_hashes.items()
):

    if not path.exists():

        raise FileNotFoundError(
            "Frozen Step 3B input is missing:\n"
            f"{path}"
        )

    actual_hash = calculate_sha256(
        path
    )

    if actual_hash != expected_hash:

        raise AssertionError(
            "Frozen Step 3B input hash differs.\n"
            f"Path: {path}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )


evaluation_file_sha256_before = (
    calculate_sha256(
        model_evaluation_path
    )
)


# ------------------------------------------------------------
# 8. LOAD SOURCE SCHEMA
# ------------------------------------------------------------

resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


# ------------------------------------------------------------
# 9. LOAD FROZEN COHORTS
# ------------------------------------------------------------

raw_training = (
    pd.read_parquet(
        raw_training_path
    )
    .sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


model_training = (
    pd.read_parquet(
        model_training_path
    )
    .reset_index(drop=True)
)


model_evaluation = (
    pd.read_parquet(
        model_evaluation_path
    )
    .reset_index(drop=True)
)


condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)


failure_subtype_distribution = (
    pd.read_csv(
        FAILURE_SUBTYPE_DISTRIBUTION_PATH,
        low_memory=False,
    )
)


clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)


anchor_offsets = pd.read_parquet(
    anchor_offsets_path
)


build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)


if len(
    raw_training
) != EXPECTED_RAW_TRAINING_ROWS:

    raise AssertionError(
        "Raw training-history row count differs."
    )


if len(
    model_training
) != EXPECTED_MODEL_TRAINING_ROWS:

    raise AssertionError(
        "Model-training row count differs."
    )


if len(
    model_evaluation
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Model-evaluation row count differs."
    )


if len(
    condition_plan
) != EXPECTED_CONDITIONS:

    raise AssertionError(
        "Noise-condition plan row count differs."
    )


expected_noise_row_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int64
    ),
    expected_noise_row_ids,
):

    raise AssertionError(
        "NoiseRowID order is not canonical."
    )


# ------------------------------------------------------------
# 10. PREPARE RAW AND MODEL KEYS
# ------------------------------------------------------------

raw_training[
    "BuildKey"
] = raw_training[
    "BuildKey"
].astype(str)


raw_training[
    "TestKey"
] = raw_training[
    "TestKey"
].astype(str)


raw_training[
    "BuildOrder"
] = pd.to_numeric(
    raw_training[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


clean_raw_verdicts = pd.to_numeric(
    raw_training[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


model_training[
    "_ModelRowOrder"
] = np.arange(
    EXPECTED_MODEL_TRAINING_ROWS,
    dtype=np.int64,
)


model_training[
    "BuildKey"
] = canonical_identifier(
    model_training[
        dataset_build_column
    ]
)


model_training[
    "TestKey"
] = canonical_identifier(
    model_training[
        dataset_test_column
    ]
)


if model_training.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():

    raise AssertionError(
        "Model-training Build/Test keys are not unique."
    )


raw_key_manifest = (
    raw_training[
        [
            "NoiseRowID",
            "BuildKey",
            "TestKey",
            "CleanVerdict",
        ]
    ]
    .copy()
)


if raw_key_manifest.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():

    raise AssertionError(
        "Raw-training Build/Test keys are not unique."
    )


model_label_alignment = (
    model_training[
        [
            "_ModelRowOrder",
            "BuildKey",
            "TestKey",
        ]
    ]
    .merge(
        raw_key_manifest,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


missing_model_raw_labels = int(
    model_label_alignment[
        "_merge"
    ].ne("both").sum()
)


if missing_model_raw_labels:

    raise AssertionError(
        "Some model-training rows have no raw-training verdict."
    )


clean_model_verdicts = pd.to_numeric(
    model_training[
        dataset_verdict_column
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


aligned_raw_clean_verdicts = pd.to_numeric(
    model_label_alignment[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


clean_label_alignment_mismatches = int(
    (
        clean_model_verdicts
        != aligned_raw_clean_verdicts
    ).sum()
)


if clean_label_alignment_mismatches:

    raise AssertionError(
        "Clean raw/model training verdicts differ."
    )


model_noise_row_ids = (
    model_label_alignment[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int64
    )
)


requested_pairs_by_test = {
    str(test_key):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in model_training.groupby(
        "TestKey",
        sort=False,
    )
}


# ------------------------------------------------------------
# 11. ALIGN CLEAN DIRECT REC AND ANCHOR OFFSETS
# ------------------------------------------------------------

model_key_frame = (
    model_training[
        [
            "_ModelRowOrder",
            "BuildKey",
            "TestKey",
        ]
    ]
    .copy()
)


clean_direct_training = (
    model_key_frame.merge(
        clean_direct_rec[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


missing_clean_direct_rows = int(
    clean_direct_training[
        "_merge"
    ].ne("both").sum()
)


if missing_clean_direct_rows:

    raise AssertionError(
        "Clean direct REC rows are missing for model training."
    )


clean_direct_training = (
    clean_direct_training.drop(
        columns=[
            "_merge",
        ]
    )
)


anchor_training = (
    model_key_frame.merge(
        anchor_offsets[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


missing_anchor_rows = int(
    anchor_training[
        "_merge"
    ].ne("both").sum()
)


if missing_anchor_rows:

    raise AssertionError(
        "Clean anchor-offset rows are missing."
    )


anchor_training = anchor_training.drop(
    columns=[
        "_merge",
    ]
)


clean_direct_matrix = (
    clean_direct_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


anchor_matrix = (
    anchor_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


original_dependent_matrix = (
    model_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


original_independent_matrix = (
    model_training[
        VERDICT_INDEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


if not np.isfinite(
    clean_direct_matrix
).all():

    raise AssertionError(
        "Clean direct REC matrix contains non-finite values."
    )


if not np.isfinite(
    anchor_matrix
).all():

    raise AssertionError(
        "Clean anchor matrix contains non-finite values."
    )


# ------------------------------------------------------------
# 12. BUILD/ENTITY STRUCTURE
# ------------------------------------------------------------

build_entity_map[
    "BuildKey"
] = build_entity_map[
    "BuildKey"
].astype(str)


build_entity_map[
    "EntityId"
] = pd.to_numeric(
    build_entity_map[
        "EntityId"
    ],
    errors="raise",
).astype(int)


changed_entities_by_build = {
    str(build_key):
        set(
            group[
                "EntityId"
            ].astype(int)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


entity_changed_builds = {
    int(entity_id):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for entity_id, group
    in build_entity_map.groupby(
        "EntityId",
        sort=False,
    )
}


# ------------------------------------------------------------
# 13. FAILURE SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

failure_subtypes = pd.to_numeric(
    failure_subtype_distribution[
        "FailureSubtype"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


subtype_probabilities = pd.to_numeric(
    failure_subtype_distribution[
        "SamplingProbability"
    ],
    errors="raise",
).to_numpy(
    dtype=float
)


if not np.isclose(
    subtype_probabilities.sum(),
    1.0,
    rtol=0,
    atol=1e-12,
):

    raise AssertionError(
        "Failure-subtype probabilities do not sum to 1."
    )


# ------------------------------------------------------------
# 14. NOISY REC RECONSTRUCTION ENGINE
# ------------------------------------------------------------

def calculate_max_test_file_rate(
    target_builds,
    current_changed_entities,
):
    if len(
        target_builds
    ) == 0:

        return -1.0


    maximum_frequency = 0


    for entity_id in current_changed_entities:

        changed_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )


        overlap_count = len(
            changed_builds.intersection(
                target_builds
            )
        )


        if overlap_count > maximum_frequency:

            maximum_frequency = (
                overlap_count
            )


    if maximum_frequency == 0:

        return 0.0


    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_noisy_dependent_rec(
    history,
):
    reconstructed_records = []

    reconstructed_rows = 0


    for test_key, test_history in history.groupby(
        "TestKey",
        sort=False,
    ):

        test_key = str(
            test_key
        )

        requested_builds = (
            requested_pairs_by_test.get(
                test_key
            )
        )


        if not requested_builds:

            continue


        test_history = (
            test_history.sort_values(
                "BuildOrder",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        prior_count = 0

        prior_failure_count = 0
        prior_assertion_count = 0
        prior_exception_count = 0
        prior_transition_count = 0

        last_failure_position = None
        last_transition_position = None

        previous_verdict = None

        recent_history = deque(
            maxlen=RECENT_WINDOW
        )

        failure_builds = set()
        transition_builds = set()


        for row in test_history.itertuples(
            index=False
        ):

            current_build = str(
                row.BuildKey
            )

            current_verdict = int(
                row.NoisyVerdict
            )


            if current_build in requested_builds:

                record = {
                    "BuildKey":
                        current_build,

                    "TestKey":
                        test_key,
                }


                if prior_count == 0:

                    for feature in (
                        VERDICT_DEPENDENT_REC_FEATURES
                    ):

                        record[
                            feature
                        ] = -1.0


                else:

                    if (
                        last_failure_position
                        is None
                    ):

                        last_failure_age = -1.0

                    else:

                        last_failure_age = float(
                            prior_count
                            - 1
                            - last_failure_position
                        )


                    if (
                        last_transition_position
                        is None
                    ):

                        last_transition_age = -1.0

                    else:

                        last_transition_age = float(
                            prior_count
                            - 1
                            - last_transition_position
                        )


                    recent_rows = list(
                        recent_history
                    )

                    recent_length = len(
                        recent_rows
                    )


                    if recent_length == 0:

                        raise AssertionError(
                            "Recent history is unexpectedly empty."
                        )


                    recent_verdicts = np.asarray(
                        [
                            item[
                                "Verdict"
                            ]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )


                    recent_transitions = np.asarray(
                        [
                            item[
                                "Transition"
                            ]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )


                    current_changed_entities = (
                        changed_entities_by_build.get(
                            current_build,
                            set(),
                        )
                    )


                    record.update({
                        "REC_LastFailureAge":
                            last_failure_age,

                        "REC_LastTransitionAge":
                            last_transition_age,

                        "REC_RecentFailRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts != 0
                                )
                                / recent_length
                            ),

                        "REC_RecentAssertRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts == 2
                                )
                                / recent_length
                            ),

                        "REC_RecentExcRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts == 1
                                )
                                / recent_length
                            ),

                        "REC_RecentTransitionRate":
                            float(
                                np.count_nonzero(
                                    recent_transitions == 1
                                )
                                / recent_length
                            ),

                        "REC_TotalFailRate":
                            float(
                                prior_failure_count
                                / prior_count
                            ),

                        "REC_TotalAssertRate":
                            float(
                                prior_assertion_count
                                / prior_count
                            ),

                        "REC_TotalExcRate":
                            float(
                                prior_exception_count
                                / prior_count
                            ),

                        "REC_TotalTransitionRate":
                            float(
                                prior_transition_count
                                / prior_count
                            ),

                        "REC_LastVerdict":
                            float(
                                previous_verdict
                            ),

                        "REC_MaxTestFileFailRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    failure_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),

                        "REC_MaxTestFileTransitionRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    transition_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),
                    })


                reconstructed_records.append(
                    record
                )

                reconstructed_rows += 1


            current_transition = (
                0
                if previous_verdict is None
                else int(
                    current_verdict
                    != previous_verdict
                )
            )


            if current_verdict != 0:

                prior_failure_count += 1

                last_failure_position = (
                    prior_count
                )

                failure_builds.add(
                    current_build
                )


            if current_verdict == 2:

                prior_assertion_count += 1


            if current_verdict == 1:

                prior_exception_count += 1


            if current_transition == 1:

                prior_transition_count += 1

                last_transition_position = (
                    prior_count
                )

                transition_builds.add(
                    current_build
                )


            recent_history.append({
                "Verdict":
                    current_verdict,

                "Transition":
                    current_transition,
            })


            previous_verdict = (
                current_verdict
            )

            prior_count += 1


    reconstructed = pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ),
    )


    return reconstructed


# ------------------------------------------------------------
# 15. EXECUTE SENTINEL CONDITIONS
# ------------------------------------------------------------

sentinel_records = []
feature_change_records = []

condition_hash_mismatches = 0
condition_row_count_mismatches = 0
condition_duplicate_rows = 0
condition_missing_rows = 0

zero_noise_direct_mismatches = 0
zero_noise_anchored_mismatches = 0
zero_noise_label_mismatches = 0

independent_feature_mismatches = 0
positive_conditions_without_changes = 0


for sentinel_index, (
    repetition_seed,
    noise_percent,
) in enumerate(
    SENTINEL_CONDITIONS,
    start=1,
):

    print(
        f"\n[{sentinel_index}/{len(SENTINEL_CONDITIONS)}] "
        f"Validating seed={repetition_seed}, "
        f"noise={noise_percent}%"
    )


    condition_started = (
        time.perf_counter()
    )


    recorded_condition_rows = condition_plan[
        (
            condition_plan[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
        )
        &
        (
            condition_plan[
                "NoisePercent"
            ].eq(
                noise_percent
            )
        )
    ]


    if len(recorded_condition_rows) != 1:

        raise AssertionError(
            "Sentinel condition was not found exactly once.\n"
            f"Seed: {repetition_seed}\n"
            f"Noise: {noise_percent}"
        )


    recorded_condition = (
        recorded_condition_rows.iloc[0]
    )


    streams = create_seed_random_streams(
        number_of_rows=(
            EXPECTED_RAW_TRAINING_ROWS
        ),

        repetition_seed=(
            repetition_seed
        ),

        failure_subtypes=(
            failure_subtypes
        ),

        subtype_probabilities=(
            subtype_probabilities
        ),
    )


    generated = construct_condition_arrays(
        clean_verdicts=(
            clean_raw_verdicts
        ),

        row_uniforms=(
            streams[
                "FlipUniforms"
            ]
        ),

        sampled_failure_subtypes=(
            streams[
                "SampledFailureSubtypes"
            ]
        ),

        noise_percent=(
            noise_percent
        ),
    )


    flip_mask = generated[
        "FlipMask"
    ]

    noisy_raw_verdicts = generated[
        "NoisyVerdicts"
    ]


    generated_mask_sha256 = (
        packed_boolean_sha256(
            flip_mask
        )
    )


    generated_verdict_sha256 = (
        integer_array_sha256(
            noisy_raw_verdicts
        )
    )


    mask_hash_match = (
        generated_mask_sha256
        == recorded_condition[
            "FlipMaskSHA256"
        ]
    )


    verdict_hash_match = (
        generated_verdict_sha256
        == recorded_condition[
            "NoisyVerdictSHA256"
        ]
    )


    if not mask_hash_match:

        condition_hash_mismatches += 1


    if not verdict_hash_match:

        condition_hash_mismatches += 1


    noisy_history = raw_training[
        [
            "BuildKey",
            "TestKey",
            "BuildOrder",
        ]
    ].copy()


    noisy_history[
        "NoisyVerdict"
    ] = noisy_raw_verdicts


    reconstruction_started = (
        time.perf_counter()
    )


    noisy_direct = (
        reconstruct_noisy_dependent_rec(
            noisy_history
        )
    )


    reconstruction_seconds = float(
        time.perf_counter()
        - reconstruction_started
    )


    noisy_duplicate_rows = int(
        noisy_direct.duplicated(
            subset=[
                "BuildKey",
                "TestKey",
            ],
            keep=False,
        ).sum()
    )


    condition_duplicate_rows += (
        noisy_duplicate_rows
    )


    if len(
        noisy_direct
    ) != EXPECTED_MODEL_TRAINING_ROWS:

        condition_row_count_mismatches += 1


    aligned_noisy_direct = (
        model_key_frame.merge(
            noisy_direct,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "_ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    missing_rows = int(
        aligned_noisy_direct[
            "_merge"
        ].ne("both").sum()
    )


    condition_missing_rows += (
        missing_rows
    )


    if missing_rows:

        raise AssertionError(
            "Noisy REC reconstruction is missing rows.\n"
            f"Seed: {repetition_seed}\n"
            f"Noise: {noise_percent}\n"
            f"Missing: {missing_rows}"
        )


    noisy_direct_matrix = (
        aligned_noisy_direct[
            VERDICT_DEPENDENT_REC_FEATURES
        ]
        .to_numpy(
            dtype=float
        )
    )


    if not np.isfinite(
        noisy_direct_matrix
    ).all():

        raise AssertionError(
            "Noisy direct REC matrix contains non-finite values."
        )


    noisy_anchored_matrix = (
        noisy_direct_matrix
        + anchor_matrix
    )


    if not np.isfinite(
        noisy_anchored_matrix
    ).all():

        raise AssertionError(
            "Noisy anchored REC matrix contains non-finite values."
        )


    noisy_model_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )


    changed_model_labels = int(
        (
            noisy_model_verdicts
            != clean_model_verdicts
        ).sum()
    )


    changed_direct_values = int(
        (
            ~np.isclose(
                noisy_direct_matrix,
                clean_direct_matrix,
                rtol=COMPARISON_RTOL,
                atol=COMPARISON_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


    changed_anchored_values = int(
        (
            ~np.isclose(
                noisy_anchored_matrix,
                original_dependent_matrix,
                rtol=COMPARISON_RTOL,
                atol=COMPARISON_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


    # The six independent REC values are preserved by design.
    preserved_independent_matrix = (
        original_independent_matrix.copy()
    )


    independent_mismatches = mismatch_count(
        preserved_independent_matrix,
        original_independent_matrix,
    )


    independent_feature_mismatches += (
        independent_mismatches
    )


    direct_clean_mismatches = mismatch_count(
        noisy_direct_matrix,
        clean_direct_matrix,
    )


    anchored_clean_mismatches = mismatch_count(
        noisy_anchored_matrix,
        original_dependent_matrix,
    )


    label_clean_mismatches = int(
        (
            noisy_model_verdicts
            != clean_model_verdicts
        ).sum()
    )


    if noise_percent == 0:

        zero_noise_direct_mismatches += (
            direct_clean_mismatches
        )

        zero_noise_anchored_mismatches += (
            anchored_clean_mismatches
        )

        zero_noise_label_mismatches += (
            label_clean_mismatches
        )


    elif (
        changed_model_labels == 0
        or changed_anchored_values == 0
    ):

        positive_conditions_without_changes += 1


    noisy_model_failures = int(
        (
            noisy_model_verdicts != 0
        ).sum()
    )


    noisy_model_passes = int(
        (
            noisy_model_verdicts == 0
        ).sum()
    )


    for feature_index, feature in enumerate(
        VERDICT_DEPENDENT_REC_FEATURES
    ):

        direct_delta = (
            noisy_direct_matrix[
                :,
                feature_index,
            ]
            - clean_direct_matrix[
                :,
                feature_index,
            ]
        )


        anchored_delta = (
            noisy_anchored_matrix[
                :,
                feature_index,
            ]
            - original_dependent_matrix[
                :,
                feature_index,
            ]
        )


        feature_change_records.append({
            "RepetitionSeed":
                repetition_seed,

            "NoisePercent":
                noise_percent,

            "Feature":
                feature,

            "Rows":
                EXPECTED_MODEL_TRAINING_ROWS,

            "ChangedDirectRows":
                int(
                    (
                        ~np.isclose(
                            direct_delta,
                            0.0,
                            rtol=COMPARISON_RTOL,
                            atol=COMPARISON_ATOL,
                            equal_nan=False,
                        )
                    ).sum()
                ),

            "ChangedAnchoredRows":
                int(
                    (
                        ~np.isclose(
                            anchored_delta,
                            0.0,
                            rtol=COMPARISON_RTOL,
                            atol=COMPARISON_ATOL,
                            equal_nan=False,
                        )
                    ).sum()
                ),

            "MaximumAbsoluteDirectDelta":
                float(
                    np.max(
                        np.abs(
                            direct_delta
                        )
                    )
                ),

            "MeanAbsoluteDirectDelta":
                float(
                    np.mean(
                        np.abs(
                            direct_delta
                        )
                    )
                ),
        })


    condition_seconds = float(
        time.perf_counter()
        - condition_started
    )


    sentinel_records.append({
        "ConditionKey":
            recorded_condition[
                "ConditionKey"
            ],

        "RepetitionSeed":
            repetition_seed,

        "NoisePercent":
            noise_percent,

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "RecordedRawRowsFlipped":
            int(
                recorded_condition[
                    "RawRowsFlipped"
                ]
            ),

        "GeneratedRawRowsFlipped":
            int(
                flip_mask.sum()
            ),

        "MaskHashMatch":
            mask_hash_match,

        "VerdictHashMatch":
            verdict_hash_match,

        "PassToFailure":
            generated[
                "PassToFailure"
            ],

        "FailureToPass":
            generated[
                "FailureToPass"
            ],

        "ReconstructedRows":
            len(
                noisy_direct
            ),

        "DuplicateReconstructedRows":
            noisy_duplicate_rows,

        "MissingReconstructedRows":
            missing_rows,

        "ChangedModelLabels":
            changed_model_labels,

        "NoisyModelFailures":
            noisy_model_failures,

        "NoisyModelPasses":
            noisy_model_passes,

        "ChangedDirectDependentValues":
            changed_direct_values,

        "ChangedAnchoredDependentValues":
            changed_anchored_values,

        "IndependentRECMismatches":
            independent_mismatches,

        "DirectDependentMatrixSHA256":
            float_matrix_sha256(
                noisy_direct_matrix
            ),

        "AnchoredDependentMatrixSHA256":
            float_matrix_sha256(
                noisy_anchored_matrix
            ),

        "NoisyModelVerdictSHA256":
            integer_array_sha256(
                noisy_model_verdicts
            ),

        "ReconstructionSeconds":
            reconstruction_seconds,

        "ConditionSeconds":
            condition_seconds,
    })


    print(
        "    Raw flips:",
        int(
            flip_mask.sum()
        ),
        "| Model label changes:",
        changed_model_labels,
        "| Dependent feature changes:",
        changed_anchored_values,
        "| Reconstruction seconds:",
        round(
            reconstruction_seconds,
            3,
        ),
    )


sentinel_summary = pd.DataFrame(
    sentinel_records
)


feature_change_summary = pd.DataFrame(
    feature_change_records
)


# ------------------------------------------------------------
# 16. SENTINEL REPRODUCIBILITY RECHECK
# ------------------------------------------------------------

sentinel_reproducibility_mismatches = 0


for sentinel in sentinel_records:

    repetition_seed = int(
        sentinel[
            "RepetitionSeed"
        ]
    )

    noise_percent = int(
        sentinel[
            "NoisePercent"
        ]
    )


    streams = create_seed_random_streams(
        number_of_rows=(
            EXPECTED_RAW_TRAINING_ROWS
        ),

        repetition_seed=(
            repetition_seed
        ),

        failure_subtypes=(
            failure_subtypes
        ),

        subtype_probabilities=(
            subtype_probabilities
        ),
    )


    regenerated = construct_condition_arrays(
        clean_verdicts=(
            clean_raw_verdicts
        ),

        row_uniforms=(
            streams[
                "FlipUniforms"
            ]
        ),

        sampled_failure_subtypes=(
            streams[
                "SampledFailureSubtypes"
            ]
        ),

        noise_percent=(
            noise_percent
        ),
    )


    regenerated_model_verdicts = (
        regenerated[
            "NoisyVerdicts"
        ][
            model_noise_row_ids
        ]
    )


    if (
        integer_array_sha256(
            regenerated_model_verdicts
        )
        != sentinel[
            "NoisyModelVerdictSHA256"
        ]
    ):

        sentinel_reproducibility_mismatches += 1


# ------------------------------------------------------------
# 17. VALIDATION
# ------------------------------------------------------------

positive_sentinel_rows = sentinel_summary[
    sentinel_summary[
        "NoisePercent"
    ].gt(0)
]


validation_records = [
    {
        "Check":
            "Step 2B passed",

        "Expected":
            EXPECTED_STEP2B_STATUS,

        "Actual":
            step2b_status[
                "Status"
            ],

        "Pass":
            step2b_status[
                "Status"
            ]
            == EXPECTED_STEP2B_STATUS,
    },

    {
        "Check":
            "Step 3A passed",

        "Expected":
            EXPECTED_STEP3A_STATUS,

        "Actual":
            step3a_status[
                "Status"
            ],

        "Pass":
            step3a_status[
                "Status"
            ]
            == EXPECTED_STEP3A_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            selection_checkpoint[
                "SourceRootSHA256"
            ],

        "Pass":
            selection_checkpoint[
                "SourceRootSHA256"
            ]
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Raw training rows",

        "Expected":
            EXPECTED_RAW_TRAINING_ROWS,

        "Actual":
            len(
                raw_training
            ),

        "Pass":
            len(
                raw_training
            )
            == EXPECTED_RAW_TRAINING_ROWS,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            len(
                model_training
            ),

        "Pass":
            len(
                model_training
            )
            == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            len(
                model_evaluation
            ),

        "Pass":
            len(
                model_evaluation
            )
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Check":
            "Model/raw clean label mismatches",

        "Expected":
            0,

        "Actual":
            clean_label_alignment_mismatches,

        "Pass":
            clean_label_alignment_mismatches
            == 0,
    },

    {
        "Check":
            "Sentinel conditions",

        "Expected":
            len(
                SENTINEL_CONDITIONS
            ),

        "Actual":
            len(
                sentinel_summary
            ),

        "Pass":
            len(
                sentinel_summary
            )
            == len(
                SENTINEL_CONDITIONS
            ),
    },

    {
        "Check":
            "Condition mask/verdict hash mismatches",

        "Expected":
            0,

        "Actual":
            condition_hash_mismatches,

        "Pass":
            condition_hash_mismatches
            == 0,
    },

    {
        "Check":
            "Reconstructed-row count mismatches",

        "Expected":
            0,

        "Actual":
            condition_row_count_mismatches,

        "Pass":
            condition_row_count_mismatches
            == 0,
    },

    {
        "Check":
            "Duplicate reconstructed rows",

        "Expected":
            0,

        "Actual":
            condition_duplicate_rows,

        "Pass":
            condition_duplicate_rows
            == 0,
    },

    {
        "Check":
            "Missing reconstructed rows",

        "Expected":
            0,

        "Actual":
            condition_missing_rows,

        "Pass":
            condition_missing_rows
            == 0,
    },

    {
        "Check":
            "Zero-noise direct REC mismatches",

        "Expected":
            0,

        "Actual":
            zero_noise_direct_mismatches,

        "Pass":
            zero_noise_direct_mismatches
            == 0,
    },

    {
        "Check":
            "Zero-noise anchored REC mismatches",

        "Expected":
            0,

        "Actual":
            zero_noise_anchored_mismatches,

        "Pass":
            zero_noise_anchored_mismatches
            == 0,
    },

    {
        "Check":
            "Zero-noise label mismatches",

        "Expected":
            0,

        "Actual":
            zero_noise_label_mismatches,

        "Pass":
            zero_noise_label_mismatches
            == 0,
    },

    {
        "Check":
            "Independent REC mismatches",

        "Expected":
            0,

        "Actual":
            independent_feature_mismatches,

        "Pass":
            independent_feature_mismatches
            == 0,
    },

    {
        "Check":
            "Positive conditions without changes",

        "Expected":
            0,

        "Actual":
            positive_conditions_without_changes,

        "Pass":
            positive_conditions_without_changes
            == 0,
    },

    {
        "Check":
            "Positive sentinel label changes",

        "Expected":
            True,

        "Actual":
            bool(
                positive_sentinel_rows[
                    "ChangedModelLabels"
                ].gt(0).all()
            ),

        "Pass":
            bool(
                positive_sentinel_rows[
                    "ChangedModelLabels"
                ].gt(0).all()
            ),
    },

    {
        "Check":
            "Positive sentinel dependent-feature changes",

        "Expected":
            True,

        "Actual":
            bool(
                positive_sentinel_rows[
                    "ChangedAnchoredDependentValues"
                ].gt(0).all()
            ),

        "Pass":
            bool(
                positive_sentinel_rows[
                    "ChangedAnchoredDependentValues"
                ].gt(0).all()
            ),
    },

    {
        "Check":
            "Sentinel reproducibility mismatches",

        "Expected":
            0,

        "Actual":
            sentinel_reproducibility_mismatches,

        "Pass":
            sentinel_reproducibility_mismatches
            == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 3B validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 3B DID NOT PASS."
    )


# ------------------------------------------------------------
# 18. VERIFY IMMUTABILITY
# ------------------------------------------------------------

evaluation_file_sha256_after = (
    calculate_sha256(
        model_evaluation_path
    )
)


evaluation_unchanged = (
    evaluation_file_sha256_before
    == evaluation_file_sha256_after
)


if not evaluation_unchanged:

    raise AssertionError(
        "Frozen evaluation data changed during Step 3B."
    )


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 3B."
    )


for path, expected_hash in (
    frozen_path_hashes.items()
):

    if calculate_sha256(
        path
    ) != expected_hash:

        raise AssertionError(
            "A frozen Step 3B input changed during execution.\n"
            f"Path: {path}"
        )


# ------------------------------------------------------------
# 19. WRITE OUTPUTS
# ------------------------------------------------------------

NOISY_REC_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    SENTINEL_SUMMARY_PATH,
    sentinel_summary,
)


atomic_write_csv(
    FEATURE_CHANGE_SUMMARY_PATH,
    feature_change_summary,
)


atomic_write_csv(
    STEP3B_VALIDATION_PATH,
    validation,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3B_PASS_STATUS,

    "RecentWindow":
        RECENT_WINDOW,

    "RawNoiseUnit":
        "raw training execution verdict",

    "ModelLabelRule":
        (
            "map each model-training Build/Test row to its "
            "corresponding noisy raw-training verdict"
        ),

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_FEATURES,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC_FEATURES,

    "VerdictDependentRule":
        (
            "reconstruct directly from corrupted raw history "
            "using only executions before the current Build/Test"
        ),

    "CleanAnchorRule":
        (
            "noisy anchored feature = direct noisy "
            "reconstruction + frozen clean anchor offset"
        ),

    "VerdictIndependentRule":
        (
            "preserve the six original clean values exactly"
        ),

    "NonRECFeatureRule":
        (
            "preserve all non-REC predictors exactly"
        ),

    "EvaluationRule":
        (
            "clean, fixed and immutable across all "
            "noise conditions"
        ),

    "SentinelConditions":
        [
            {
                "RepetitionSeed":
                    repetition_seed,

                "NoisePercent":
                    noise_percent,
            }
            for repetition_seed, noise_percent
            in SENTINEL_CONDITIONS
        ],

    "SentinelSummary":
        str(
            SENTINEL_SUMMARY_PATH
        ),

    "FeatureChangeSummary":
        str(
            FEATURE_CHANGE_SUMMARY_PATH
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    NOISY_REC_PROTOCOL_PATH,
    protocol_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3B_PASS_STATUS,

    "SentinelConditions":
        len(
            sentinel_summary
        ),

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "VerdictDependentRECFeatures":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatures":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "ConditionHashMismatches":
        condition_hash_mismatches,

    "ReconstructedRowCountMismatches":
        condition_row_count_mismatches,

    "DuplicateReconstructedRows":
        condition_duplicate_rows,

    "MissingReconstructedRows":
        condition_missing_rows,

    "ZeroNoiseDirectMismatches":
        zero_noise_direct_mismatches,

    "ZeroNoiseAnchoredMismatches":
        zero_noise_anchored_mismatches,

    "ZeroNoiseLabelMismatches":
        zero_noise_label_mismatches,

    "IndependentFeatureMismatches":
        independent_feature_mismatches,

    "PositiveConditionsWithoutChanges":
        positive_conditions_without_changes,

    "SentinelReproducibilityMismatches":
        sentinel_reproducibility_mismatches,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3B_PASS_STATUS,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "NoisePlanCheckpointSHA256":
        calculate_sha256(
            NOISE_CHECKPOINT_PATH
        ),

    "RECCheckpointSHA256":
        calculate_sha256(
            REC_CHECKPOINT_PATH
        ),

    "SentinelSummary":
        str(
            SENTINEL_SUMMARY_PATH
        ),

    "SentinelSummarySHA256":
        calculate_sha256(
            SENTINEL_SUMMARY_PATH
        ),

    "FeatureChangeSummary":
        str(
            FEATURE_CHANGE_SUMMARY_PATH
        ),

    "FeatureChangeSummarySHA256":
        calculate_sha256(
            FEATURE_CHANGE_SUMMARY_PATH
        ),

    "NoisyRECProtocol":
        str(
            NOISY_REC_PROTOCOL_PATH
        ),

    "NoisyRECProtocolSHA256":
        calculate_sha256(
            NOISY_REC_PROTOCOL_PATH
        ),

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "SentinelConditions":
        len(
            SENTINEL_CONDITIONS
        ),

    "ConditionHashMismatches":
        condition_hash_mismatches,

    "ZeroNoiseAnchoredMismatches":
        zero_noise_anchored_mismatches,

    "IndependentFeatureMismatches":
        independent_feature_mismatches,

    "PositiveConditionsWithoutChanges":
        positive_conditions_without_changes,

    "SentinelReproducibilityMismatches":
        sentinel_reproducibility_mismatches,

    "EvaluationDataSHA256":
        evaluation_file_sha256_after,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    NOISY_REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3B_PASS_STATUS,

    "SentinelConditions":
        len(
            SENTINEL_CONDITIONS
        ),

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ConditionHashMismatches":
        condition_hash_mismatches,

    "ZeroNoiseAnchoredMismatches":
        zero_noise_anchored_mismatches,

    "IndependentFeatureMismatches":
        independent_feature_mismatches,

    "PositiveConditionsWithoutChanges":
        positive_conditions_without_changes,

    "SentinelReproducibilityMismatches":
        sentinel_reproducibility_mismatches,

    "Checkpoint":
        str(
            NOISY_REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            NOISY_REC_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP3B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 20. READBACK
# ------------------------------------------------------------

required_outputs = [
    SENTINEL_SUMMARY_PATH,
    FEATURE_CHANGE_SUMMARY_PATH,
    STEP3B_VALIDATION_PATH,
    NOISY_REC_PROTOCOL_PATH,
    STEP3B_REPORT_PATH,
    NOISY_REC_CHECKPOINT_PATH,
    STEP3B_STATUS_PATH,
]


missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]


if missing_outputs:

    raise RuntimeError(
        "Step 3B outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


final_status = json.loads(
    STEP3B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_status.get(
        "Status"
    )
    != STEP3B_PASS_STATUS
):

    raise AssertionError(
        "Final Step 3B status differs."
    )


# ------------------------------------------------------------
# 21. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nNoisy REC sentinel summary:")

display(
    sentinel_summary
)


print("\nFeature changes by condition:")

display(
    feature_change_summary
)


# ------------------------------------------------------------
# 22. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 114)
print("=== PROJECT 9 CELL 6 / STEP 3B RESULT ===")
print("=" * 114)

print("\nProject:")

print(
    PROJECT_NAME
)


print("\nSentinel validation:")

print(
    "Sentinel conditions:",
    len(
        sentinel_summary
    ),
)

print(
    "Condition hash mismatches:",
    condition_hash_mismatches,
)

print(
    "Reconstructed-row count mismatches:",
    condition_row_count_mismatches,
)

print(
    "Duplicate reconstructed rows:",
    condition_duplicate_rows,
)

print(
    "Missing reconstructed rows:",
    condition_missing_rows,
)


print("\nClean condition:")

print(
    "Zero-noise direct REC mismatches:",
    zero_noise_direct_mismatches,
)

print(
    "Zero-noise anchored REC mismatches:",
    zero_noise_anchored_mismatches,
)

print(
    "Zero-noise label mismatches:",
    zero_noise_label_mismatches,
)


print("\nNoise sensitivity:")

print(
    "Independent REC mismatches:",
    independent_feature_mismatches,
)

print(
    "Positive conditions without label/"
    "dependent-feature changes:",
    positive_conditions_without_changes,
)

print(
    "Sentinel reproducibility mismatches:",
    sentinel_reproducibility_mismatches,
)


print("\nEvaluation immutability:")

print(
    "Evaluation unchanged:",
    evaluation_unchanged,
)

print(
    "Evaluation SHA-256:",
    evaluation_file_sha256_after,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nNoisy REC engine checkpoint:")

print(
    NOISY_REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        NOISY_REC_CHECKPOINT_PATH
    ),
)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP3B_PASS_STATUS,
)

print("=" * 114)

=== PROJECT 9 CELL 6 / STEP 3B: NOISY REC ENGINE SENTINEL VALIDATION ===

[1/6] Validating seed=1, noise=0%
    Raw flips: 0 | Model label changes: 0 | Dependent feature changes: 0 | Reconstruction seconds: 7.555

[2/6] Validating seed=1, noise=5%
    Raw flips: 15657 | Model label changes: 3056 | Dependent feature changes: 479474 | Reconstruction seconds: 6.696

[3/6] Validating seed=1, noise=25%
    Raw flips: 78158 | Model label changes: 15158 | Dependent feature changes: 648249 | Reconstruction seconds: 10.406

[4/6] Validating seed=1, noise=50%
    Raw flips: 156700 | Model label changes: 30346 | Dependent feature changes: 715259 | Reconstruction seconds: 4.88

[5/6] Validating seed=17, noise=15%
    Raw flips: 47131 | Model label changes: 9231 | Dependent feature changes: 592007 | Reconstruction seconds: 4.897

[6/6] Validating seed=30, noise=40%
    Raw flips: 125613 | Model label changes: 24384 | Dependent feature changes: 695716 | Reconstruction seconds: 6.501

Step 3B validat

,Check,Expected,Actual,Pass
0,Step 2B passed,PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_AN...,PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_AN...,True
1,Step 3A passed,PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_CO...,PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_CO...,True
2,Source root SHA-256,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,True
3,Raw training rows,313984,313984,True
4,Model training rows,60880,60880,True
5,Model evaluation rows,18503,18503,True
6,Model/raw clean label mismatches,0,0,True
7,Sentinel conditions,6,6,True
8,Condition mask/verdict hash mismatches,0,0,True
9,Reconstructed-row count mismatches,0,0,True



Noisy REC sentinel summary:


,ConditionKey,RepetitionSeed,NoisePercent,RawTrainingRows,RecordedRawRowsFlipped,GeneratedRawRowsFlipped,MaskHashMatch,VerdictHashMatch,PassToFailure,FailureToPass,...,NoisyModelFailures,NoisyModelPasses,ChangedDirectDependentValues,ChangedAnchoredDependentValues,IndependentRECMismatches,DirectDependentMatrixSHA256,AnchoredDependentMatrixSHA256,NoisyModelVerdictSHA256,ReconstructionSeconds,ConditionSeconds
0,noise_00__seed_01,1,0,313984,0,0,True,True,0,0,...,1427,59453,0,0,0,b7ba34dfb6f60792d9d7daa05587c2d76d56fac9cb2c4c...,9f68c9d5fbc9fac1e9dc31b37048b61ec2c62c76ea8624...,9255a4f2aee40e8b5ada41395a7d700db35a711bcad6f1...,7.555276,8.120542
1,noise_05__seed_01,1,5,313984,15657,15657,True,True,15577,80,...,4327,56553,479474,479474,0,93d0d7abd1ae32b01ade7055bd75950c6cdfcc0b10548c...,19660aa2e4c84e0ed242aef433563bf1b7ab27bec5358f...,77509f58456b70574c89888b42aba5f78df2d5910d4444...,6.695896,7.000899
2,noise_25__seed_01,1,25,313984,78158,78158,True,True,77769,389,...,15811,45069,648249,648249,0,6750ab63841191621afd0d7abc52dd27ad6d1746783d1d...,bd28fae84b477dbc4578dd904a52c17bddf2020068eedb...,89dd8222eb4dc051b5a003693fd83bf7f3b5865b88913f...,10.406377,10.696039
3,noise_50__seed_01,1,50,313984,156700,156700,True,True,155949,751,...,30285,30595,715259,715259,0,14ea787f21166c7d25cee293f2a502e09f112faf9e8f88...,0b033f3ce7d9ef39d20a2788a7400b36e8f72f091925c9...,1bda69c55300490dd84340a08c517963870e063071eb13...,4.880380,5.117933
4,noise_15__seed_17,17,15,313984,47131,47131,True,True,46911,220,...,10226,50654,592007,592007,0,67e134fd5ff3153e6b9d28d9f875e63cfd33571d8d03cf...,0cec0abfd97533df7ad3d6896337680144543b672ecb2c...,d38a124a1d478673049ac14e0e8437e6e50dc2722e4fef...,4.897210,5.193663
5,noise_40__seed_30,30,40,313984,125613,125613,True,True,125030,583,...,24665,36215,695716,695716,0,73f32aaebcb3d7159519ab4b65101e7b4d0d6c7749ffb4...,cb8c6d95618c6caf7588d9c80d6821ae699b562a7f46fb...,27cc8e65c74d9fcc04017267c6e826de84138b14a49760...,6.501220,6.722119



Feature changes by condition:


,RepetitionSeed,NoisePercent,Feature,Rows,ChangedDirectRows,ChangedAnchoredRows,MaximumAbsoluteDirectDelta,MeanAbsoluteDirectDelta
0,1,0,REC_LastFailureAge,60880,0,0,0.000000,0.000000
1,1,0,REC_LastTransitionAge,60880,0,0,0.000000,0.000000
2,1,0,REC_RecentFailRate,60880,0,0,0.000000,0.000000
3,1,0,REC_RecentAssertRate,60880,0,0,0.000000,0.000000
4,1,0,REC_RecentExcRate,60880,0,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...
73,30,40,REC_TotalExcRate,60880,60221,60221,1.000000,0.314105
74,30,40,REC_TotalTransitionRate,60880,60364,60364,0.888889,0.513744
75,30,40,REC_LastVerdict,60880,24361,24361,2.000000,0.483262
76,30,40,REC_MaxTestFileFailRate,60880,58535,58535,2.000000,0.935586




=== PROJECT 9 CELL 6 / STEP 3B RESULT ===

Project:
camunda@camunda-bpm-platform

Sentinel validation:
Sentinel conditions: 6
Condition hash mismatches: 0
Reconstructed-row count mismatches: 0
Duplicate reconstructed rows: 0
Missing reconstructed rows: 0

Clean condition:
Zero-noise direct REC mismatches: 0
Zero-noise anchored REC mismatches: 0
Zero-noise label mismatches: 0

Noise sensitivity:
Independent REC mismatches: 0
Positive conditions without label/dependent-feature changes: 0
Sentinel reproducibility mismatches: 0

Evaluation immutability:
Evaluation unchanged: True
Evaluation SHA-256: e6ab0de25c67c9f923685919d3bf8dc66092af5e7521ca874aa94d4fc7586e11

Validation:
Checks: 21
Failed checks: 0

Noisy REC engine checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_09_noisy_rec_engine_checkpoint.json
Checkpoint SHA-256: 6207fb3ef2b97352dee60ea31717ad559a63ac29da52cf190e4f3e29080518be

Completion registry modified:
0

Projects 1–8 modified:
0

STATUS: PASS_PROJECT_9_

In [11]:
# ============================================================
# PROJECT 9 — CELL 7 / STEP 4A
# MODEL, PREDICTOR, BASELINE AND METRIC-PROTOCOL FREEZE
#
# PROJECT: camunda@camunda-bpm-platform
#
# This cell:
# - validates all Project 9 checkpoints through Step 3B
# - freezes predictor order
# - removes clean-training zero-variance predictors
# - freezes condition-specific median-imputation rules
# - freezes four ML model configurations
# - freezes Random, LatestFail and QTF-Avg rules
# - constructs the clean evaluation metric cohort
# - freezes APFD and APFDc implementations
# - validates ranking tie-breaking
# - performs small model-constructor/probability self-tests
#
# This cell does NOT:
# - train models on the full Camunda dataset
# - run experimental conditions
# - create noisy datasets
# - modify source files
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import importlib
import json
import re
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PACKAGE AVAILABILITY
# ------------------------------------------------------------

def ensure_ml_packages():
    missing = []

    for package_name in [
        "xgboost",
        "lightgbm",
    ]:
        try:
            importlib.import_module(
                package_name
            )

        except ImportError:
            missing.append(
                package_name
            )

    if missing:
        print(
            "Installing missing ML packages:",
            missing,
        )

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing,
        ])

        importlib.invalidate_caches()


ensure_ml_packages()


import sklearn
import xgboost
import lightgbm

from sklearn.ensemble import (
    RandomForestClassifier,
)

from sklearn.naive_bayes import (
    GaussianNB,
)

from xgboost import (
    XGBClassifier,
)

from lightgbm import (
    LGBMClassifier,
)


# ------------------------------------------------------------
# 2. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = "camunda"


EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

STEP4A_PASS_STATUS = (
    "PASS_PROJECT_9_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)


EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

EXPECTED_RAW_EVALUATION_ROWS = 158781

EXPECTED_EVALUATION_BUILDS = 206
EXPECTED_SCORED_EVALUATION_BUILDS = 30
EXPECTED_EVALUATION_FAILURES = 665

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_ORIGINAL_PREDICTORS = 151

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


# ------------------------------------------------------------
# 3. REC FEATURE PROTOCOL
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# ------------------------------------------------------------
# 4. FROZEN MODEL CONFIGURATION
# ------------------------------------------------------------

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "criterion": "gini",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": None,
    },

    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
    },

    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "objective": "binary",
    },

    "NaiveBayes": {
        "variant": "GaussianNB",
        "var_smoothing": 1e-9,
    },
}


# ------------------------------------------------------------
# 5. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

NOISE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noise_plan_checkpoint.json"
)

NOISY_REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noisy_rec_engine_checkpoint.json"
)

MODEL_METRIC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_model_metric_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)


SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)


MODEL_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_model_preflight"
)

PREDICTOR_MANIFEST_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_predictor_manifest.csv"
)

ACTIVE_PREDICTORS_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_active_predictors.json"
)

CLEAN_TRAINING_MEDIAN_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_training_median_audit.csv"
)

MODEL_CONFIGURATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)

MODEL_SEED_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_seed_audit.csv"
)

MODEL_CONSTRUCTOR_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_constructor_audit.csv"
)

BASELINE_PROTOCOL_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_protocol.json"
)

METRIC_PROTOCOL_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_metric_protocol.json"
)

METRIC_HELPER_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_metric_helper_audit.csv"
)

RANKING_HELPER_AUDIT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_ranking_helper_audit.csv"
)

RAW_EVALUATION_HISTORY_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_raw_evaluation_history.parquet"
)

EVALUATION_BUILD_TABLE_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_evaluation_build_table.csv"
)

EVALUATION_METRIC_COHORT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_evaluation_metric_cohort.parquet"
)

STEP4A_VALIDATION_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    MODEL_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)


print("=" * 116)
print("=== PROJECT 9 CELL 7 / STEP 4A: MODEL, PREDICTOR, BASELINE AND METRIC FREEZE ===")
print("=" * 116)


# ------------------------------------------------------------
# 6. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def normalise_name(value):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def dataframe_content_sha256(
    dataframe,
):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=True,
        categorize=True,
    ).to_numpy(
        dtype="<u8"
    )

    digest = hashlib.sha256()

    digest.update(
        json.dumps(
            list(
                dataframe.columns
            ),
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )

    digest.update(
        b"\0"
    )

    digest.update(
        row_hashes.tobytes(
            order="C"
        )
    )

    return digest.hexdigest()


# ------------------------------------------------------------
# 7. METRIC HELPERS
# ------------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    """
    APFD for one already-ranked evaluation build.

    actual_failures:
        1 = failing test
        0 = passing test
    """

    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if number_of_tests == 0:
        return np.nan

    if number_of_failures == 0:
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    apfd = (
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )

    return float(
        apfd
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    """
    Cost-aware APFD using midpoint failure-detection time.
    """

    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):

        raise ValueError(
            "Failures and durations must have equal length."
        )

    if len(
        failures
    ) == 0:

        return np.nan

    if failures.sum() == 0:

        return np.nan

    if not np.isfinite(
        durations
    ).all():

        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():

        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:

        return np.nan

    cumulative_before = np.concatenate([
        np.array(
            [0.0]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    apfdc = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

    return float(
        apfdc
    )


# ------------------------------------------------------------
# 8. RANKING HELPER
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
    build_column="Build",
    test_column="Test",
    verdict_column="Verdict",
    duration_column="Duration",
    build_order_column="BuildOrder",
):
    ranked = (
        build_rows[
            [
                build_column,
                test_column,
                verdict_column,
                duration_column,
                build_order_column,
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    scores = np.asarray(
        scores,
        dtype=float,
    )

    if len(
        ranked
    ) != len(
        scores
    ):

        raise ValueError(
            "Score count differs from ranked row count."
        )

    if not np.isfinite(
        scores
    ).all():

        raise ValueError(
            "Ranking scores contain non-finite values."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = scores

    ranked[
        "ActualFailure"
    ] = (
        pd.to_numeric(
            ranked[
                verdict_column
            ],
            errors="raise",
        )
        .ne(0)
        .astype(int)
    )

    ranked[
        duration_column
    ] = pd.to_numeric(
        ranked[
            duration_column
        ],
        errors="raise",
    ).astype(float)

    if score_direction == (
        "descending"
    ):

        ascending = [
            False,
            True,
        ]

    elif score_direction == (
        "ascending"
    ):

        ascending = [
            True,
            True,
        ]

    else:

        raise ValueError(
            "score_direction must be ascending or descending."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                test_column,
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(
            ranked
        ) + 1,
    )

    return ranked


# ------------------------------------------------------------
# 9. MODEL HELPERS
# ------------------------------------------------------------

def create_ml_models(
    repetition_seed,
):
    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )

    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )

    lgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )

    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                var_smoothing=(
                    MODEL_CONFIG[
                        "NaiveBayes"
                    ][
                        "var_smoothing"
                    ]
                )
            ),
    }


def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = (
        fitted_model.predict_proba(
            feature_matrix
        )
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.where(
        classes == 1
    )[0]

    if len(
        positive_positions
    ) != 1:

        raise ValueError(
            "Could not identify fitted failure class 1."
        )

    output = probabilities[
        :,
        positive_positions[0],
    ]

    if not np.isfinite(
        output
    ).all():

        raise ValueError(
            "Failure probabilities contain non-finite values."
        )

    if not (
        (
            output >= 0.0
        )
        &
        (
            output <= 1.0
        )
    ).all():

        raise ValueError(
            "Failure probabilities fall outside [0, 1]."
        )

    return output


# ------------------------------------------------------------
# 10. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_CHECKPOINT_PATH,
    NOISY_REC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
    STEP2A_REPORT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "Required Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_checkpoint = json.loads(
    SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

rec_checkpoint = json.loads(
    REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noise_checkpoint = json.loads(
    NOISE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noisy_rec_checkpoint = json.loads(
    NOISY_REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step2b_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3a_status = json.loads(
    STEP3A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3b_status = json.loads(
    STEP3B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step2b_status.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "Project 9 Step 2B has not passed."
    )


if (
    step3a_status.get(
        "Status"
    )
    != EXPECTED_STEP3A_STATUS
):

    raise AssertionError(
        "Project 9 Step 3A has not passed."
    )


if (
    step3b_status.get(
        "Status"
    )
    != EXPECTED_STEP3B_STATUS
):

    raise AssertionError(
        "Project 9 Step 3B has not passed."
    )


if (
    rec_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "Project 9 REC checkpoint status differs."
    )


if (
    noise_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP3A_STATUS
):

    raise AssertionError(
        "Project 9 noise checkpoint status differs."
    )


if (
    noisy_rec_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP3B_STATUS
):

    raise AssertionError(
        "Project 9 noisy-REC checkpoint status differs."
    )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Project 9 source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 11. VALIDATE REGISTRY
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)

if (
    len(
        registry
    ) != 8
    or set(
        project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )

if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the completion registry."
    )


# ------------------------------------------------------------
# 12. RESOLVE AND VALIDATE FROZEN INPUTS
# ------------------------------------------------------------

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingData"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationData"
    ]
)

fixed_split_path = Path(
    selection_checkpoint[
        "FixedSplit"
    ][
        "FixedSplitFile"
    ]
)

source_paths = {
    relative_name:
        Path(
            metadata[
                "RuntimePath"
            ]
        )
    for relative_name, metadata
    in selection_checkpoint[
        "SourceFiles"
    ].items()
}


frozen_file_checks = {
    model_training_path:
        noise_checkpoint[
            "ModelTrainingDataSHA256"
        ],

    model_evaluation_path:
        noise_checkpoint[
            "ModelEvaluationDataSHA256"
        ],

    Path(
        rec_checkpoint[
            "CleanRECReconstructed"
        ]
    ):
        rec_checkpoint[
            "CleanRECReconstructedSHA256"
        ],

    Path(
        rec_checkpoint[
            "CleanRECAnchorOffsets"
        ]
    ):
        rec_checkpoint[
            "CleanRECAnchorOffsetsSHA256"
        ],
}


for relative_name, metadata in (
    selection_checkpoint[
        "SourceFiles"
    ].items()
):

    frozen_file_checks[
        Path(
            metadata[
                "RuntimePath"
            ]
        )
    ] = metadata[
        "SHA256"
    ]


for path, expected_sha256 in (
    frozen_file_checks.items()
):

    if not path.exists():

        raise FileNotFoundError(
            "Frozen Step 4A input is missing:\n"
            f"{path}"
        )

    actual_sha256 = calculate_sha256(
        path
    )

    if actual_sha256 != (
        expected_sha256
    ):

        raise AssertionError(
            "Frozen Step 4A input hash differs.\n"
            f"Path: {path}\n"
            f"Expected: {expected_sha256}\n"
            f"Actual:   {actual_sha256}"
        )


source_hashes_before = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}


# ------------------------------------------------------------
# 13. RESOLVE COLUMN SCHEMA
# ------------------------------------------------------------

resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)

exe_build_column = (
    resolved_columns[
        "ExecutionBuild"
    ]
)

exe_test_column = (
    resolved_columns[
        "ExecutionTest"
    ]
)

exe_verdict_column = (
    resolved_columns[
        "ExecutionVerdict"
    ]
)

exe_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

exe_job_column = (
    resolved_columns.get(
        "ExecutionJob"
    )
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


# ------------------------------------------------------------
# 14. LOAD FROZEN MODEL COHORTS
# ------------------------------------------------------------

clean_model_training = (
    pd.read_parquet(
        model_training_path
    )
    .reset_index(
        drop=True
    )
)

clean_model_evaluation = (
    pd.read_parquet(
        model_evaluation_path
    )
    .reset_index(
        drop=True
    )
)


if len(
    clean_model_training
) != EXPECTED_MODEL_TRAINING_ROWS:

    raise AssertionError(
        "Frozen model-training row count differs."
    )


if len(
    clean_model_evaluation
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Frozen model-evaluation row count differs."
    )


if len(
    clean_model_training.columns
) != EXPECTED_DATASET_COLUMNS:

    raise AssertionError(
        "Frozen dataset column count differs.\n"
        f"Expected: {EXPECTED_DATASET_COLUMNS}\n"
        f"Actual: {len(clean_model_training.columns)}"
    )


if list(
    clean_model_training.columns
) != list(
    clean_model_evaluation.columns
):

    raise AssertionError(
        "Training and evaluation dataset schemas differ."
    )


identifier_and_label_columns = {
    dataset_build_column,
    dataset_test_column,
    dataset_verdict_column,
}


MODEL_FEATURE_COLUMNS = [
    column
    for column in clean_model_training.columns
    if column not in (
        identifier_and_label_columns
    )
]


if len(
    MODEL_FEATURE_COLUMNS
) != EXPECTED_ORIGINAL_PREDICTORS:

    raise AssertionError(
        "Original predictor count differs.\n"
        f"Expected: {EXPECTED_ORIGINAL_PREDICTORS}\n"
        f"Actual: {len(MODEL_FEATURE_COLUMNS)}"
    )


missing_rec_predictors = [
    column
    for column in REC_FEATURE_COLUMNS
    if column not in MODEL_FEATURE_COLUMNS
]


if missing_rec_predictors:

    raise AssertionError(
        "REC predictors are missing:\n"
        + "\n".join(
            missing_rec_predictors
        )
    )


# ------------------------------------------------------------
# 15. FREEZE PREDICTOR SET
# ------------------------------------------------------------

clean_feature_frame = (
    clean_model_training[
        MODEL_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


evaluation_feature_frame = (
    clean_model_evaluation[
        MODEL_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


feature_variation = (
    clean_feature_frame.nunique(
        dropna=False
    )
)


ZERO_VARIANCE_FEATURES = (
    feature_variation[
        feature_variation <= 1
    ]
    .index
    .tolist()
)


ACTIVE_FEATURE_COLUMNS = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column not in (
        ZERO_VARIANCE_FEATURES
    )
]


if not ACTIVE_FEATURE_COLUMNS:

    raise RuntimeError(
        "No active predictor columns remain."
    )


clean_training_medians = (
    clean_feature_frame[
        ACTIVE_FEATURE_COLUMNS
    ]
    .median(
        axis=0
    )
    .fillna(
        0.0
    )
)


if not np.isfinite(
    clean_training_medians.to_numpy(
        dtype=float
    )
).all():

    raise AssertionError(
        "Clean-training predictor medians are non-finite."
    )


X_train_validation = (
    clean_feature_frame[
        ACTIVE_FEATURE_COLUMNS
    ]
    .fillna(
        clean_training_medians
    )
    .astype(float)
)


X_evaluation_validation = (
    evaluation_feature_frame[
        ACTIVE_FEATURE_COLUMNS
    ]
    .fillna(
        clean_training_medians
    )
    .astype(float)
)


training_matrix_nonfinite = int(
    (
        ~np.isfinite(
            X_train_validation.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_matrix_nonfinite = int(
    (
        ~np.isfinite(
            X_evaluation_validation.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


if training_matrix_nonfinite:

    raise AssertionError(
        "Imputed clean training matrix contains non-finite values."
    )


if evaluation_matrix_nonfinite:

    raise AssertionError(
        "Imputed clean evaluation matrix contains non-finite values."
    )


clean_binary_training_labels = (
    pd.to_numeric(
        clean_model_training[
            dataset_verdict_column
        ],
        errors="raise",
    )
    .ne(0)
    .astype(int)
)


clean_binary_evaluation_labels = (
    pd.to_numeric(
        clean_model_evaluation[
            dataset_verdict_column
        ],
        errors="raise",
    )
    .ne(0)
    .astype(int)
)


if clean_binary_training_labels.nunique() != 2:

    raise AssertionError(
        "Clean model-training labels do not contain both classes."
    )


predictor_manifest_records = []


for feature_order, column in enumerate(
    MODEL_FEATURE_COLUMNS,
    start=1,
):

    clean_numeric = (
        clean_feature_frame[
            column
        ]
    )

    evaluation_numeric = (
        evaluation_feature_frame[
            column
        ]
    )

    is_rec = (
        column
        in REC_FEATURE_COLUMNS
    )

    is_dependent_rec = (
        column
        in VERDICT_DEPENDENT_REC_FEATURES
    )

    is_independent_rec = (
        column
        in VERDICT_INDEPENDENT_REC_FEATURES
    )

    zero_variance = (
        column
        in ZERO_VARIANCE_FEATURES
    )

    active = (
        column
        in ACTIVE_FEATURE_COLUMNS
    )

    predictor_manifest_records.append({
        "FeatureOrder":
            feature_order,

        "Feature":
            column,

        "FeatureClass":
            (
                "REC_VERDICT_DEPENDENT"
                if is_dependent_rec
                else
                "REC_VERDICT_INDEPENDENT"
                if is_independent_rec
                else
                "NON_REC"
            ),

        "IsREC":
            is_rec,

        "VerdictDependentREC":
            is_dependent_rec,

        "VerdictIndependentREC":
            is_independent_rec,

        "CleanTrainingNonMissing":
            int(
                clean_numeric.notna().sum()
            ),

        "CleanTrainingMissing":
            int(
                clean_numeric.isna().sum()
            ),

        "CleanTrainingUniqueIncludingMissing":
            int(
                clean_numeric.nunique(
                    dropna=False
                )
            ),

        "CleanTrainingMinimum":
            (
                float(
                    clean_numeric.min()
                )
                if clean_numeric.notna().any()
                else np.nan
            ),

        "CleanTrainingMaximum":
            (
                float(
                    clean_numeric.max()
                )
                if clean_numeric.notna().any()
                else np.nan
            ),

        "CleanTrainingMedian":
            (
                float(
                    clean_training_medians[
                        column
                    ]
                )
                if active
                else np.nan
            ),

        "EvaluationMissingBeforeImputation":
            int(
                evaluation_numeric.isna().sum()
            ),

        "ZeroVarianceFromCleanTraining":
            zero_variance,

        "ActivePredictor":
            active,
    })


predictor_manifest = pd.DataFrame(
    predictor_manifest_records
)


clean_training_median_audit = (
    predictor_manifest[
        predictor_manifest[
            "ActivePredictor"
        ]
    ][
        [
            "FeatureOrder",
            "Feature",
            "FeatureClass",
            "CleanTrainingMedian",
            "CleanTrainingMissing",
            "EvaluationMissingBeforeImputation",
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print("\nPredictor freeze:")

print(
    "Original predictors:",
    len(
        MODEL_FEATURE_COLUMNS
    ),
)

print(
    "Zero-variance predictors:",
    len(
        ZERO_VARIANCE_FEATURES
    ),
)

print(
    "Active predictors:",
    len(
        ACTIVE_FEATURE_COLUMNS
    ),
)


if ZERO_VARIANCE_FEATURES:

    print("\nFrozen zero-variance predictors:")

    for column in (
        ZERO_VARIANCE_FEATURES
    ):
        print(
            " -",
            column,
        )


# Matrices were only needed for validation.
del X_train_validation
del X_evaluation_validation


# ------------------------------------------------------------
# 16. CONSTRUCT CLEAN EVALUATION STRUCTURE
# ------------------------------------------------------------

fixed_split = pd.read_csv(
    fixed_split_path,
    low_memory=False,
)


fixed_split[
    "BuildKey"
] = canonical_identifier(
    fixed_split[
        "Build"
    ]
)


fixed_split[
    "BuildOrder"
] = pd.to_numeric(
    fixed_split[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


evaluation_build_table = (
    fixed_split[
        fixed_split[
            "Partition"
        ].eq(
            "EVALUATION"
        )
    ]
    .copy()
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    evaluation_build_table
) != EXPECTED_EVALUATION_BUILDS:

    raise AssertionError(
        "Evaluation-build count differs."
    )


evaluation_build_keys = set(
    evaluation_build_table[
        "BuildKey"
    ].astype(str)
)


raw_usecols = [
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]


if (
    exe_job_column is not None
    and exe_job_column not in raw_usecols
):

    raw_usecols.append(
        exe_job_column
    )


raw_execution_data = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=raw_usecols,
    low_memory=False,
)


raw_execution_data[
    "BuildKey"
] = canonical_identifier(
    raw_execution_data[
        exe_build_column
    ]
)


raw_execution_data[
    "TestKey"
] = canonical_identifier(
    raw_execution_data[
        exe_test_column
    ]
)


raw_execution_data[
    "BuildOrder"
] = raw_execution_data[
    "BuildKey"
].map(
    fixed_split.set_index(
        "BuildKey"
    )[
        "BuildOrder"
    ]
)


if raw_execution_data[
    "BuildOrder"
].isna().any():

    raise AssertionError(
        "Raw execution rows could not be mapped to build chronology."
    )


raw_execution_data[
    "BuildOrder"
] = raw_execution_data[
    "BuildOrder"
].astype(int)


raw_execution_data[
    "CleanVerdict"
] = pd.to_numeric(
    raw_execution_data[
        exe_verdict_column
    ],
    errors="raise",
).astype(np.int32)


raw_execution_data[
    "Duration"
] = pd.to_numeric(
    raw_execution_data[
        exe_duration_column
    ],
    errors="raise",
).astype(float)


if not np.isfinite(
    raw_execution_data[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():

    raise AssertionError(
        "Raw execution durations contain non-finite values."
    )


if raw_execution_data[
    "Duration"
].lt(0).any():

    raise AssertionError(
        "Raw execution durations contain negative values."
    )


raw_evaluation_history = (
    raw_execution_data[
        raw_execution_data[
            "BuildKey"
        ].isin(
            evaluation_build_keys
        )
    ]
    .copy()
)


raw_evaluation_sort_columns = [
    "BuildOrder",
]


if exe_job_column is not None:

    raw_evaluation_sort_columns.append(
        exe_job_column
    )


raw_evaluation_sort_columns.append(
    "TestKey"
)


raw_evaluation_history = (
    raw_evaluation_history.sort_values(
        raw_evaluation_sort_columns,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    raw_evaluation_history
) != EXPECTED_RAW_EVALUATION_ROWS:

    raise AssertionError(
        "Raw evaluation-history row count differs."
    )


raw_evaluation_duplicate_keys = int(
    raw_evaluation_history.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


if raw_evaluation_duplicate_keys:

    raise AssertionError(
        "Raw evaluation history contains duplicate Build/Test rows."
    )


# ------------------------------------------------------------
# 17. CONSTRUCT MODEL EVALUATION METRIC COHORT
# ------------------------------------------------------------

model_evaluation_keys = (
    clean_model_evaluation[
        [
            dataset_build_column,
            dataset_test_column,
            dataset_verdict_column,
        ]
    ]
    .copy()
)


model_evaluation_keys[
    "_ModelRowOrder"
] = np.arange(
    len(
        model_evaluation_keys
    ),
    dtype=np.int64,
)


model_evaluation_keys[
    "BuildKey"
] = canonical_identifier(
    model_evaluation_keys[
        dataset_build_column
    ]
)


model_evaluation_keys[
    "TestKey"
] = canonical_identifier(
    model_evaluation_keys[
        dataset_test_column
    ]
)


raw_metric_keys = (
    raw_evaluation_history[
        [
            "BuildKey",
            "TestKey",
            "BuildOrder",
            "CleanVerdict",
            "Duration",
        ]
        + (
            [exe_job_column]
            if exe_job_column
            is not None
            else []
        )
    ]
    .copy()
)


evaluation_metric_cohort = (
    model_evaluation_keys.merge(
        raw_metric_keys,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


metric_join_missing_rows = int(
    evaluation_metric_cohort[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


if metric_join_missing_rows:

    raise AssertionError(
        "Some model evaluation rows lack raw duration/outcome data."
    )


model_metric_verdicts = pd.to_numeric(
    evaluation_metric_cohort[
        dataset_verdict_column
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


raw_metric_verdicts = pd.to_numeric(
    evaluation_metric_cohort[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


metric_verdict_mismatches = int(
    (
        model_metric_verdicts
        != raw_metric_verdicts
    ).sum()
)


if metric_verdict_mismatches:

    raise AssertionError(
        "Raw and model evaluation verdicts differ."
    )


evaluation_metric_cohort[
    "ActualFailure"
] = (
    model_metric_verdicts
    != 0
).astype(
    np.int32
)


evaluation_metric_cohort = (
    evaluation_metric_cohort.drop(
        columns=[
            "_merge",
        ]
    )
)


metric_cohort_duplicate_keys = int(
    evaluation_metric_cohort.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).sum()
)


scored_evaluation_builds = int(
    evaluation_metric_cohort[
        "BuildKey"
    ].nunique()
)


evaluation_failures = int(
    evaluation_metric_cohort[
        "ActualFailure"
    ].sum()
)


if len(
    evaluation_metric_cohort
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Evaluation metric cohort row count differs."
    )


if metric_cohort_duplicate_keys:

    raise AssertionError(
        "Evaluation metric cohort contains duplicate Build/Test rows."
    )


if (
    scored_evaluation_builds
    != EXPECTED_SCORED_EVALUATION_BUILDS
):

    raise AssertionError(
        "Scored evaluation-build count differs."
    )


if (
    evaluation_failures
    != EXPECTED_EVALUATION_FAILURES
):

    raise AssertionError(
        "Evaluation failure count differs."
    )


target_build_keys = set(
    evaluation_metric_cohort[
        "BuildKey"
    ].astype(str)
)


evaluation_build_table[
    "ScoredEvaluationBuild"
] = (
    evaluation_build_table[
        "BuildKey"
    ].astype(str).isin(
        target_build_keys
    )
)


if int(
    evaluation_build_table[
        "ScoredEvaluationBuild"
    ].sum()
) != EXPECTED_SCORED_EVALUATION_BUILDS:

    raise AssertionError(
        "Evaluation-build target flag count differs."
    )


build_duration_totals = (
    evaluation_metric_cohort.groupby(
        "BuildKey"
    )[
        "Duration"
    ]
    .sum()
)


nonpositive_scored_build_durations = int(
    build_duration_totals.le(
        0
    ).sum()
)


if nonpositive_scored_build_durations:

    raise AssertionError(
        "A scored evaluation build has non-positive total duration."
    )


# ------------------------------------------------------------
# 18. METRIC HELPER AUDIT
# ------------------------------------------------------------

metric_audit_records = []


manual_failures = np.asarray(
    [
        1,
        1,
        0,
        0,
        0,
    ],
    dtype=int,
)


manual_long_failure_first = np.asarray(
    [
        5,
        1,
        1,
        1,
        1,
    ],
    dtype=float,
)


manual_quick_failure_first = np.asarray(
    [
        1,
        5,
        1,
        1,
        1,
    ],
    dtype=float,
)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_long_first = (
    calculate_apfdc(
        manual_failures,
        manual_long_failure_first,
    )
)


manual_apfdc_quick_first = (
    calculate_apfdc(
        manual_failures,
        manual_quick_failure_first,
    )
)


metric_audit_records.extend([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            bool(
                np.isclose(
                    manual_apfd,
                    0.8,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Manual APFDc long failure first",

        "Expected":
            5.0 / 9.0,

        "Actual":
            manual_apfdc_long_first,

        "Pass":
            bool(
                np.isclose(
                    manual_apfdc_long_first,
                    5.0 / 9.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Manual APFDc quick failure first",

        "Expected":
            7.0 / 9.0,

        "Actual":
            manual_apfdc_quick_first,

        "Pass":
            bool(
                np.isclose(
                    manual_apfdc_quick_first,
                    7.0 / 9.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Quick failure receives higher APFDc",

        "Expected":
            True,

        "Actual":
            (
                manual_apfdc_quick_first
                > manual_apfdc_long_first
            ),

        "Pass":
            bool(
                manual_apfdc_quick_first
                > manual_apfdc_long_first
            ),
    },

    {
        "Check":
            "No-failure APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    calculate_apfd(
                        [
                            0,
                            0,
                            0,
                        ]
                    )
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    calculate_apfd(
                        [
                            0,
                            0,
                            0,
                        ]
                    )
                )
            ),
    },

    {
        "Check":
            "No-failure APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    calculate_apfdc(
                        [
                            0,
                            0,
                            0,
                        ],
                        [
                            1,
                            1,
                            1,
                        ],
                    )
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    calculate_apfdc(
                        [
                            0,
                            0,
                            0,
                        ],
                        [
                            1,
                            1,
                            1,
                        ],
                    )
                )
            ),
    },
])


metric_helper_audit = pd.DataFrame(
    metric_audit_records
)


failed_metric_helper_checks = int(
    (
        ~metric_helper_audit[
            "Pass"
        ]
    ).sum()
)


if failed_metric_helper_checks:

    display(
        metric_helper_audit
    )

    raise RuntimeError(
        "APFD/APFDc helper validation failed."
    )


# ------------------------------------------------------------
# 19. RANKING TIE-RULE AUDIT
# ------------------------------------------------------------

ranking_test_frame = pd.DataFrame({
    "Build":
        [
            1,
            1,
            1,
        ],

    "Test":
        [
            3,
            1,
            2,
        ],

    "Verdict":
        [
            0,
            1,
            0,
        ],

    "Duration":
        [
            2.0,
            2.0,
            1.0,
        ],

    "BuildOrder":
        [
            1,
            1,
            1,
        ],
})


descending_ranking = rank_build_rows(
    build_rows=ranking_test_frame,
    scores=[
        0.5,
        0.5,
        0.8,
    ],
    technique="DescendingAudit",
    score_direction="descending",
)


ascending_ranking = rank_build_rows(
    build_rows=ranking_test_frame,
    scores=[
        2.0,
        2.0,
        1.0,
    ],
    technique="AscendingAudit",
    score_direction="ascending",
)


descending_test_order = (
    descending_ranking[
        "Test"
    ].tolist()
)


ascending_test_order = (
    ascending_ranking[
        "Test"
    ].tolist()
)


ranking_helper_audit = pd.DataFrame([
    {
        "Check":
            "Descending score then Test ascending",

        "ExpectedTestOrder":
            "[2, 1, 3]",

        "ActualTestOrder":
            str(
                descending_test_order
            ),

        "Pass":
            descending_test_order
            == [
                2,
                1,
                3,
            ],
    },

    {
        "Check":
            "Ascending score then Test ascending",

        "ExpectedTestOrder":
            "[2, 1, 3]",

        "ActualTestOrder":
            str(
                ascending_test_order
            ),

        "Pass":
            ascending_test_order
            == [
                2,
                1,
                3,
            ],
    },
])


failed_ranking_helper_checks = int(
    (
        ~ranking_helper_audit[
            "Pass"
        ]
    ).sum()
)


if failed_ranking_helper_checks:

    display(
        ranking_helper_audit
    )

    raise RuntimeError(
        "Ranking helper validation failed."
    )


# ------------------------------------------------------------
# 20. MODEL SEED AUDIT
# ------------------------------------------------------------

model_seed_records = []


for repetition_seed in [
    1,
    17,
    30,
]:

    for technique, stream_name in [
        (
            "RandomForest",
            "RandomForest_model",
        ),
        (
            "XGBoost",
            "XGBoost_model",
        ),
        (
            "LightGBM",
            "LightGBM_model",
        ),
    ]:

        model_seed_records.append({
            "Project":
                PROJECT_NAME,

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "RandomStream":
                stream_name,

            "DerivedModelSeed":
                stable_project_seed(
                    PROJECT_NAME,
                    repetition_seed,
                    stream_name,
                ),
        })


    model_seed_records.append({
        "Project":
            PROJECT_NAME,

        "RepetitionSeed":
            repetition_seed,

        "Technique":
            "NaiveBayes",

        "RandomStream":
            "NONE_DETERMINISTIC",

        "DerivedModelSeed":
            np.nan,
    })


model_seed_audit = pd.DataFrame(
    model_seed_records
)


# ------------------------------------------------------------
# 21. MODEL CONSTRUCTOR AND PROBABILITY AUDIT
# ------------------------------------------------------------

synthetic_rng = np.random.default_rng(
    940091
)


synthetic_X = synthetic_rng.normal(
    size=(
        64,
        6,
    )
)


synthetic_y = np.asarray(
    [
        0,
        1,
    ]
    * 32,
    dtype=int,
)


constructor_audit_records = []


models = create_ml_models(
    repetition_seed=1
)


for technique, model in models.items():

    fit_success = False
    probability_success = False
    probability_minimum = np.nan
    probability_maximum = np.nan
    error_text = ""

    try:

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore"
            )

            model.fit(
                synthetic_X,
                synthetic_y,
            )

        fit_success = True

        probabilities = (
            get_failure_probability(
                model,
                synthetic_X,
            )
        )

        probability_success = (
            len(
                probabilities
            )
            == len(
                synthetic_y
            )
            and np.isfinite(
                probabilities
            ).all()
            and (
                probabilities >= 0
            ).all()
            and (
                probabilities <= 1
            ).all()
        )

        probability_minimum = float(
            probabilities.min()
        )

        probability_maximum = float(
            probabilities.max()
        )

    except Exception as error:

        error_text = (
            f"{type(error).__name__}: {error}"
        )


    constructor_audit_records.append({
        "Technique":
            technique,

        "ModelClass":
            type(
                model
            ).__name__,

        "FitSuccess":
            fit_success,

        "ProbabilitySuccess":
            probability_success,

        "ProbabilityMinimum":
            probability_minimum,

        "ProbabilityMaximum":
            probability_maximum,

        "Error":
            error_text,

        "Pass":
            (
                fit_success
                and probability_success
            ),
    })


model_constructor_audit = pd.DataFrame(
    constructor_audit_records
)


failed_model_constructor_checks = int(
    (
        ~model_constructor_audit[
            "Pass"
        ]
    ).sum()
)


if failed_model_constructor_checks:

    display(
        model_constructor_audit
    )

    raise RuntimeError(
        "One or more ML model constructor tests failed."
    )


# ------------------------------------------------------------
# 22. BASELINE PROTOCOL
# ------------------------------------------------------------

baseline_protocol = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "Baselines":
        BASELINE_TECHNIQUES,

    "Random": {
        "AffectedByVerdictNoise":
            False,

        "RepeatedBySeed":
            True,

        "ConstantAcrossNoiseForSameSeed":
            True,

        "RankingRule":
            "uniform random score descending",

        "SeedRule":
            (
                "SHA256(project|repetition_seed|"
                "Random_baseline_build_<Build>)"
            ),

        "TieRule":
            "score descending, then Test ascending",
    },

    "LatestFail": {
        "AffectedByVerdictNoise":
            True,

        "TrainingHistory":
            (
                "same corrupted raw training history "
                "used by the four ML models"
            ),

        "InitialState":
            (
                "latest build order containing an apparent "
                "non-zero verdict for each test"
            ),

        "EvaluationProcedure":
            (
                "process all 206 evaluation-period builds "
                "chronologically; rank before observing the "
                "current build; update history after ranking "
                "using the clean current evaluation outcomes"
            ),

        "RankingRule":
            (
                "higher latest-failure build order first; "
                "unseen/no-failure tests receive -1"
            ),

        "TieRule":
            "score descending, then Test ascending",
    },

    "QTF-Avg": {
        "AffectedByVerdictNoise":
            False,

        "TrainingHistory":
            "clean raw execution durations",

        "InitialState":
            (
                "per-test clean training duration sum "
                "and execution count"
            ),

        "EvaluationProcedure":
            (
                "process all 206 evaluation-period builds "
                "chronologically; rank before observing the "
                "current build; update duration history after "
                "ranking using clean current durations"
            ),

        "RankingRule":
            (
                "lower historical average execution duration "
                "first; unseen tests receive positive infinity"
            ),

        "TieRule":
            "score ascending, then Test ascending",
    },

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# ------------------------------------------------------------
# 23. CONFIGURATION PAYLOADS
# ------------------------------------------------------------

library_versions = {
    "Python":
        sys.version.split()[0],

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "scikit-learn":
        sklearn.__version__,

    "xgboost":
        xgboost.__version__,

    "lightgbm":
        lightgbm.__version__,
}


model_configuration = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "OriginalPredictorCount":
        len(
            MODEL_FEATURE_COLUMNS
        ),

    "ZeroVariancePredictorCount":
        len(
            ZERO_VARIANCE_FEATURES
        ),

    "ActivePredictorCount":
        len(
            ACTIVE_FEATURE_COLUMNS
        ),

    "OriginalPredictors":
        MODEL_FEATURE_COLUMNS,

    "ZeroVariancePredictors":
        ZERO_VARIANCE_FEATURES,

    "ActivePredictors":
        ACTIVE_FEATURE_COLUMNS,

    "PredictorFreezeRule":
        (
            "identify zero-variance predictors once using "
            "only the clean model-training partition; freeze "
            "the remaining ordered predictor list for every "
            "noise level, seed and ML technique"
        ),

    "NumericConversionRule":
        "pd.to_numeric(errors='coerce')",

    "InfiniteValueRule":
        "replace positive and negative infinity with missing",

    "MissingValueRule":
        (
            "calculate medians separately from the current "
            "noisy training condition; fill any all-missing "
            "active-feature median with 0.0; apply the same "
            "condition medians to training and clean evaluation"
        ),

    "LabelRule":
        "binary failure label = Verdict != 0",

    "ModelSeedRule":
        (
            "first eight bytes of SHA-256("
            "project|repetition_seed|model_stream), "
            "little-endian modulo 2^32"
        ),

    "Models":
        MODEL_CONFIG,

    "RuntimeModelArguments": {
        "RandomForest": {
            "n_jobs":
                -1,
        },

        "XGBoost": {
            "n_jobs":
                -1,

            "verbosity":
                0,
        },

        "LightGBM": {
            "n_jobs":
                -1,

            "verbosity":
                -1,

            "deterministic":
                True,

            "force_col_wise":
                True,
        },

        "NaiveBayes": {},
    },

    "RankingTieRule":
        "score first, then Test ascending",

    "LibraryVersions":
        library_versions,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


metric_protocol = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "FailureMapping":
        (
            "each failing test is treated as one "
            "fault-detection event"
        ),

    "APFDFormula":
        (
            "1 - sum(failure ranks)/(number of tests * "
            "number of failures) + 1/(2 * number of tests)"
        ),

    "APFDcFormula":
        (
            "1 - mean(midpoint failure detection time / "
            "total ranked-build execution duration)"
        ),

    "APFDcDetectionTime":
        (
            "cumulative duration before the failing test "
            "+ half of that failing test's duration"
        ),

    "CalculationLevel":
        "one already-ranked clean evaluation build",

    "ProjectRunAggregation":
        (
            "mean build-level APFD and APFDc across the "
            "30 scored evaluation builds"
        ),

    "NoFailureBuildRule":
        (
            "APFD and APFDc return missing; such builds are "
            "not part of the fixed scored model-ready cohort"
        ),

    "DurationRule":
        "clean immutable raw evaluation execution duration",

    "ScoredEvaluationBuilds":
        scored_evaluation_builds,

    "EvaluationRows":
        len(
            evaluation_metric_cohort
        ),

    "EvaluationFailures":
        evaluation_failures,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# ------------------------------------------------------------
# 24. OVERALL VALIDATION TABLE
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 2B passed",

        "Expected":
            EXPECTED_STEP2B_STATUS,

        "Actual":
            step2b_status[
                "Status"
            ],

        "Pass":
            step2b_status[
                "Status"
            ]
            == EXPECTED_STEP2B_STATUS,
    },

    {
        "Check":
            "Step 3A passed",

        "Expected":
            EXPECTED_STEP3A_STATUS,

        "Actual":
            step3a_status[
                "Status"
            ],

        "Pass":
            step3a_status[
                "Status"
            ]
            == EXPECTED_STEP3A_STATUS,
    },

    {
        "Check":
            "Step 3B passed",

        "Expected":
            EXPECTED_STEP3B_STATUS,

        "Actual":
            step3b_status[
                "Status"
            ],

        "Pass":
            step3b_status[
                "Status"
            ]
            == EXPECTED_STEP3B_STATUS,
    },

    {
        "Check":
            "Source root SHA-256",

        "Expected":
            EXPECTED_SOURCE_ROOT_SHA256,

        "Actual":
            selection_checkpoint[
                "SourceRootSHA256"
            ],

        "Pass":
            selection_checkpoint[
                "SourceRootSHA256"
            ]
            == EXPECTED_SOURCE_ROOT_SHA256,
    },

    {
        "Check":
            "Dataset columns",

        "Expected":
            EXPECTED_DATASET_COLUMNS,

        "Actual":
            len(
                clean_model_training.columns
            ),

        "Pass":
            len(
                clean_model_training.columns
            )
            == EXPECTED_DATASET_COLUMNS,
    },

    {
        "Check":
            "Original predictors",

        "Expected":
            EXPECTED_ORIGINAL_PREDICTORS,

        "Actual":
            len(
                MODEL_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                MODEL_FEATURE_COLUMNS
            )
            == EXPECTED_ORIGINAL_PREDICTORS,
    },

    {
        "Check":
            "Active predictors positive",

        "Expected":
            True,

        "Actual":
            len(
                ACTIVE_FEATURE_COLUMNS
            ) > 0,

        "Pass":
            len(
                ACTIVE_FEATURE_COLUMNS
            ) > 0,
    },

    {
        "Check":
            "Active plus zero-variance predictors",

        "Expected":
            EXPECTED_ORIGINAL_PREDICTORS,

        "Actual":
            (
                len(
                    ACTIVE_FEATURE_COLUMNS
                )
                + len(
                    ZERO_VARIANCE_FEATURES
                )
            ),

        "Pass":
            (
                len(
                    ACTIVE_FEATURE_COLUMNS
                )
                + len(
                    ZERO_VARIANCE_FEATURES
                )
                == EXPECTED_ORIGINAL_PREDICTORS
            ),
    },

    {
        "Check":
            "Training matrix non-finite values",

        "Expected":
            0,

        "Actual":
            training_matrix_nonfinite,

        "Pass":
            training_matrix_nonfinite == 0,
    },

    {
        "Check":
            "Evaluation matrix non-finite values",

        "Expected":
            0,

        "Actual":
            evaluation_matrix_nonfinite,

        "Pass":
            evaluation_matrix_nonfinite == 0,
    },

    {
        "Check":
            "Training binary classes",

        "Expected":
            2,

        "Actual":
            clean_binary_training_labels.nunique(),

        "Pass":
            clean_binary_training_labels.nunique()
            == 2,
    },

    {
        "Check":
            "Raw evaluation rows",

        "Expected":
            EXPECTED_RAW_EVALUATION_ROWS,

        "Actual":
            len(
                raw_evaluation_history
            ),

        "Pass":
            len(
                raw_evaluation_history
            )
            == EXPECTED_RAW_EVALUATION_ROWS,
    },

    {
        "Check":
            "Evaluation-period builds",

        "Expected":
            EXPECTED_EVALUATION_BUILDS,

        "Actual":
            len(
                evaluation_build_table
            ),

        "Pass":
            len(
                evaluation_build_table
            )
            == EXPECTED_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Metric cohort rows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            len(
                evaluation_metric_cohort
            ),

        "Pass":
            len(
                evaluation_metric_cohort
            )
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Check":
            "Metric cohort join missing rows",

        "Expected":
            0,

        "Actual":
            metric_join_missing_rows,

        "Pass":
            metric_join_missing_rows == 0,
    },

    {
        "Check":
            "Metric verdict mismatches",

        "Expected":
            0,

        "Actual":
            metric_verdict_mismatches,

        "Pass":
            metric_verdict_mismatches == 0,
    },

    {
        "Check":
            "Metric cohort duplicate Build/Test rows",

        "Expected":
            0,

        "Actual":
            metric_cohort_duplicate_keys,

        "Pass":
            metric_cohort_duplicate_keys == 0,
    },

    {
        "Check":
            "Scored evaluation builds",

        "Expected":
            EXPECTED_SCORED_EVALUATION_BUILDS,

        "Actual":
            scored_evaluation_builds,

        "Pass":
            scored_evaluation_builds
            == EXPECTED_SCORED_EVALUATION_BUILDS,
    },

    {
        "Check":
            "Evaluation failures",

        "Expected":
            EXPECTED_EVALUATION_FAILURES,

        "Actual":
            evaluation_failures,

        "Pass":
            evaluation_failures
            == EXPECTED_EVALUATION_FAILURES,
    },

    {
        "Check":
            "Non-positive scored-build durations",

        "Expected":
            0,

        "Actual":
            nonpositive_scored_build_durations,

        "Pass":
            nonpositive_scored_build_durations
            == 0,
    },

    {
        "Check":
            "ML techniques",

        "Expected":
            4,

        "Actual":
            len(
                ML_TECHNIQUES
            ),

        "Pass":
            len(
                ML_TECHNIQUES
            ) == 4,
    },

    {
        "Check":
            "Baselines",

        "Expected":
            3,

        "Actual":
            len(
                BASELINE_TECHNIQUES
            ),

        "Pass":
            len(
                BASELINE_TECHNIQUES
            ) == 3,
    },

    {
        "Check":
            "Model-constructor failures",

        "Expected":
            0,

        "Actual":
            failed_model_constructor_checks,

        "Pass":
            failed_model_constructor_checks == 0,
    },

    {
        "Check":
            "Metric-helper failures",

        "Expected":
            0,

        "Actual":
            failed_metric_helper_checks,

        "Pass":
            failed_metric_helper_checks == 0,
    },

    {
        "Check":
            "Ranking-helper failures",

        "Expected":
            0,

        "Actual":
            failed_ranking_helper_checks,

        "Pass":
            failed_ranking_helper_checks == 0,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 4A validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 4A DID NOT PASS."
    )


# ------------------------------------------------------------
# 25. VERIFY INPUT IMMUTABILITY
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}


source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)


if not source_files_unchanged:

    raise AssertionError(
        "A frozen source file changed during Step 4A."
    )


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 4A."
    )


evaluation_input_sha256 = calculate_sha256(
    model_evaluation_path
)


if (
    evaluation_input_sha256
    != noise_checkpoint[
        "ModelEvaluationDataSHA256"
    ]
):

    raise AssertionError(
        "Frozen model evaluation parquet changed."
    )


# ------------------------------------------------------------
# 26. WRITE FROZEN OUTPUTS
# ------------------------------------------------------------

MODEL_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    PREDICTOR_MANIFEST_PATH,
    predictor_manifest,
)


atomic_write_json(
    ACTIVE_PREDICTORS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "OriginalPredictorCount":
            len(
                MODEL_FEATURE_COLUMNS
            ),

        "ZeroVariancePredictorCount":
            len(
                ZERO_VARIANCE_FEATURES
            ),

        "ActivePredictorCount":
            len(
                ACTIVE_FEATURE_COLUMNS
            ),

        "OriginalPredictors":
            MODEL_FEATURE_COLUMNS,

        "ZeroVariancePredictors":
            ZERO_VARIANCE_FEATURES,

        "ActivePredictors":
            ACTIVE_FEATURE_COLUMNS,

        "FrozenAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    },
)


atomic_write_csv(
    CLEAN_TRAINING_MEDIAN_AUDIT_PATH,
    clean_training_median_audit,
)


atomic_write_json(
    MODEL_CONFIGURATION_PATH,
    model_configuration,
)


atomic_write_csv(
    MODEL_SEED_AUDIT_PATH,
    model_seed_audit,
)


atomic_write_csv(
    MODEL_CONSTRUCTOR_AUDIT_PATH,
    model_constructor_audit,
)


atomic_write_json(
    BASELINE_PROTOCOL_PATH,
    baseline_protocol,
)


atomic_write_json(
    METRIC_PROTOCOL_PATH,
    metric_protocol,
)


atomic_write_csv(
    METRIC_HELPER_AUDIT_PATH,
    metric_helper_audit,
)


atomic_write_csv(
    RANKING_HELPER_AUDIT_PATH,
    ranking_helper_audit,
)


raw_evaluation_output_columns = [
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]


if exe_job_column is not None:

    raw_evaluation_output_columns.append(
        exe_job_column
    )


raw_evaluation_output_columns.extend([
    "BuildKey",
    "TestKey",
    "BuildOrder",
    "CleanVerdict",
    "Duration",
])


# Remove duplicate column names while preserving order.
raw_evaluation_output_columns = list(
    dict.fromkeys(
        raw_evaluation_output_columns
    )
)


atomic_write_parquet(
    RAW_EVALUATION_HISTORY_PATH,
    raw_evaluation_history[
        raw_evaluation_output_columns
    ].copy(),
)


atomic_write_csv(
    EVALUATION_BUILD_TABLE_PATH,
    evaluation_build_table,
)


atomic_write_parquet(
    EVALUATION_METRIC_COHORT_PATH,
    evaluation_metric_cohort,
)


atomic_write_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 27. READBACK VALIDATION
# ------------------------------------------------------------

predictor_manifest_readback = pd.read_csv(
    PREDICTOR_MANIFEST_PATH,
    low_memory=False,
)


metric_cohort_readback = pd.read_parquet(
    EVALUATION_METRIC_COHORT_PATH
)


raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_HISTORY_PATH
)


if len(
    predictor_manifest_readback
) != EXPECTED_ORIGINAL_PREDICTORS:

    raise AssertionError(
        "Predictor manifest readback count differs."
    )


if len(
    metric_cohort_readback
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Evaluation metric cohort readback count differs."
    )


if len(
    raw_evaluation_readback
) != EXPECTED_RAW_EVALUATION_ROWS:

    raise AssertionError(
        "Raw evaluation history readback count differs."
    )


# ------------------------------------------------------------
# 28. REPORT, CHECKPOINT AND STATUS
# ------------------------------------------------------------

output_paths = [
    PREDICTOR_MANIFEST_PATH,
    ACTIVE_PREDICTORS_PATH,
    CLEAN_TRAINING_MEDIAN_AUDIT_PATH,
    MODEL_CONFIGURATION_PATH,
    MODEL_SEED_AUDIT_PATH,
    MODEL_CONSTRUCTOR_AUDIT_PATH,
    BASELINE_PROTOCOL_PATH,
    METRIC_PROTOCOL_PATH,
    METRIC_HELPER_AUDIT_PATH,
    RANKING_HELPER_AUDIT_PATH,
    RAW_EVALUATION_HISTORY_PATH,
    EVALUATION_BUILD_TABLE_PATH,
    EVALUATION_METRIC_COHORT_PATH,
    STEP4A_VALIDATION_PATH,
]


output_inventory = []


for path in output_paths:

    output_inventory.append({
        "Path":
            str(
                path
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "OriginalPredictors":
        len(
            MODEL_FEATURE_COLUMNS
        ),

    "ZeroVariancePredictors":
        len(
            ZERO_VARIANCE_FEATURES
        ),

    "ActivePredictors":
        len(
            ACTIVE_FEATURE_COLUMNS
        ),

    "RECFeatures":
        len(
            REC_FEATURE_COLUMNS
        ),

    "ModelTrainingRows":
        len(
            clean_model_training
        ),

    "ModelEvaluationRows":
        len(
            clean_model_evaluation
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation_history
        ),

    "EvaluationPeriodBuilds":
        len(
            evaluation_build_table
        ),

    "ScoredEvaluationBuilds":
        scored_evaluation_builds,

    "EvaluationFailures":
        evaluation_failures,

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "ModelConstructorFailures":
        failed_model_constructor_checks,

    "MetricHelperFailures":
        failed_metric_helper_checks,

    "RankingHelperFailures":
        failed_ranking_helper_checks,

    "LibraryVersions":
        library_versions,

    "OutputInventory":
        output_inventory,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "OriginalPredictorCount":
        len(
            MODEL_FEATURE_COLUMNS
        ),

    "ZeroVariancePredictorCount":
        len(
            ZERO_VARIANCE_FEATURES
        ),

    "ActivePredictorCount":
        len(
            ACTIVE_FEATURE_COLUMNS
        ),

    "PredictorManifest":
        str(
            PREDICTOR_MANIFEST_PATH
        ),

    "PredictorManifestSHA256":
        calculate_sha256(
            PREDICTOR_MANIFEST_PATH
        ),

    "ActivePredictors":
        str(
            ACTIVE_PREDICTORS_PATH
        ),

    "ActivePredictorsSHA256":
        calculate_sha256(
            ACTIVE_PREDICTORS_PATH
        ),

    "ModelConfiguration":
        str(
            MODEL_CONFIGURATION_PATH
        ),

    "ModelConfigurationSHA256":
        calculate_sha256(
            MODEL_CONFIGURATION_PATH
        ),

    "BaselineProtocol":
        str(
            BASELINE_PROTOCOL_PATH
        ),

    "BaselineProtocolSHA256":
        calculate_sha256(
            BASELINE_PROTOCOL_PATH
        ),

    "MetricProtocol":
        str(
            METRIC_PROTOCOL_PATH
        ),

    "MetricProtocolSHA256":
        calculate_sha256(
            METRIC_PROTOCOL_PATH
        ),

    "RawEvaluationHistory":
        str(
            RAW_EVALUATION_HISTORY_PATH
        ),

    "RawEvaluationHistorySHA256":
        calculate_sha256(
            RAW_EVALUATION_HISTORY_PATH
        ),

    "EvaluationBuildTable":
        str(
            EVALUATION_BUILD_TABLE_PATH
        ),

    "EvaluationBuildTableSHA256":
        calculate_sha256(
            EVALUATION_BUILD_TABLE_PATH
        ),

    "EvaluationMetricCohort":
        str(
            EVALUATION_METRIC_COHORT_PATH
        ),

    "EvaluationMetricCohortSHA256":
        calculate_sha256(
            EVALUATION_METRIC_COHORT_PATH
        ),

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "RawEvaluationRows":
        EXPECTED_RAW_EVALUATION_ROWS,

    "EvaluationPeriodBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "ScoredEvaluationBuilds":
        EXPECTED_SCORED_EVALUATION_BUILDS,

    "EvaluationFailures":
        EXPECTED_EVALUATION_FAILURES,

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "LibraryVersions":
        library_versions,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    MODEL_METRIC_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_PASS_STATUS,

    "OriginalPredictors":
        len(
            MODEL_FEATURE_COLUMNS
        ),

    "ZeroVariancePredictors":
        len(
            ZERO_VARIANCE_FEATURES
        ),

    "ActivePredictors":
        len(
            ACTIVE_FEATURE_COLUMNS
        ),

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ScoredEvaluationBuilds":
        EXPECTED_SCORED_EVALUATION_BUILDS,

    "EvaluationFailures":
        EXPECTED_EVALUATION_FAILURES,

    "MLTechniques":
        len(
            ML_TECHNIQUES
        ),

    "Baselines":
        len(
            BASELINE_TECHNIQUES
        ),

    "Checkpoint":
        str(
            MODEL_METRIC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            MODEL_METRIC_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 29. FINAL STATUS READBACK
# ------------------------------------------------------------

final_status = json.loads(
    STEP4A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_status.get(
        "Status"
    )
    != STEP4A_PASS_STATUS
):

    raise AssertionError(
        "Final Step 4A status differs."
    )


# ------------------------------------------------------------
# 30. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nPredictor manifest summary:")

display(
    predictor_manifest.groupby(
        [
            "FeatureClass",
            "ZeroVarianceFromCleanTraining",
            "ActivePredictor",
        ],
        as_index=False,
    )
    .agg(
        Predictors=(
            "Feature",
            "count",
        )
    )
)


print("\nModel constructor audit:")

display(
    model_constructor_audit
)


print("\nMetric helper audit:")

display(
    metric_helper_audit
)


print("\nRanking helper audit:")

display(
    ranking_helper_audit
)


print("\nEvaluation metric cohort by build:")

evaluation_build_summary = (
    evaluation_metric_cohort.groupby(
        [
            "BuildKey",
            "BuildOrder",
        ],
        as_index=False,
    )
    .agg(
        Tests=(
            "TestKey",
            "count",
        ),

        Failures=(
            "ActualFailure",
            "sum",
        ),

        TotalDuration=(
            "Duration",
            "sum",
        ),
    )
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

display(
    evaluation_build_summary
)


# ------------------------------------------------------------
# 31. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 116)
print("=== PROJECT 9 CELL 7 / STEP 4A RESULT ===")
print("=" * 116)

print("\nProject:")

print(
    PROJECT_NAME
)


print("\nPredictor protocol:")

print(
    "Dataset columns:",
    len(
        clean_model_training.columns
    ),
)

print(
    "Original predictors:",
    len(
        MODEL_FEATURE_COLUMNS
    ),
)

print(
    "Zero-variance predictors:",
    len(
        ZERO_VARIANCE_FEATURES
    ),
)

print(
    "Active predictors:",
    len(
        ACTIVE_FEATURE_COLUMNS
    ),
)

print(
    "Training matrix non-finite values:",
    training_matrix_nonfinite,
)

print(
    "Evaluation matrix non-finite values:",
    evaluation_matrix_nonfinite,
)


print("\nModels and baselines:")

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Model-constructor failures:",
    failed_model_constructor_checks,
)


print("\nEvaluation metric cohort:")

print(
    "Evaluation-period builds:",
    len(
        evaluation_build_table
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation_history
    ),
)

print(
    "Scored evaluation builds:",
    scored_evaluation_builds,
)

print(
    "Scored evaluation rows:",
    len(
        evaluation_metric_cohort
    ),
)

print(
    "Evaluation failures:",
    evaluation_failures,
)

print(
    "Metric join missing rows:",
    metric_join_missing_rows,
)

print(
    "Metric verdict mismatches:",
    metric_verdict_mismatches,
)


print("\nMetric protocol:")

print(
    "Primary metric: APFDc"
)

print(
    "Secondary metric: APFD"
)

print(
    "Metric-helper failures:",
    failed_metric_helper_checks,
)

print(
    "Ranking-helper failures:",
    failed_ranking_helper_checks,
)


print("\nLibrary versions:")

for library_name, version in (
    library_versions.items()
):

    print(
        f"{library_name}: {version}"
    )


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nModel/metric checkpoint:")

print(
    MODEL_METRIC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        MODEL_METRIC_CHECKPOINT_PATH
    ),
)


print("\nSaved outputs:")

for output_path in (
    output_paths
    + [
        STEP4A_REPORT_PATH,
        MODEL_METRIC_CHECKPOINT_PATH,
        STEP4A_STATUS_PATH,
    ]
):

    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP4A_PASS_STATUS,
)

print("=" * 116)

=== PROJECT 9 CELL 7 / STEP 4A: MODEL, PREDICTOR, BASELINE AND METRIC FREEZE ===

Predictor freeze:
Original predictors: 151
Zero-variance predictors: 0
Active predictors: 151

Step 4A validation:


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Check,Expected,Actual,Pass
0,Step 2B passed,PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_AN...,PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_AN...,True
1,Step 3A passed,PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_CO...,PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_CO...,True
2,Step 3B passed,PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALID...,PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALID...,True
3,Source root SHA-256,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b...,True
4,Dataset columns,154,154,True
5,Original predictors,151,151,True
6,Active predictors positive,True,True,True
7,Active plus zero-variance predictors,151,151,True
8,Training matrix non-finite values,0,0,True
9,Evaluation matrix non-finite values,0,0,True



Predictor manifest summary:


,FeatureClass,ZeroVarianceFromCleanTraining,ActivePredictor,Predictors
0,NON_REC,False,True,132
1,REC_VERDICT_DEPENDENT,False,True,13
2,REC_VERDICT_INDEPENDENT,False,True,6


...


Ranking helper audit:


,Check,ExpectedTestOrder,ActualTestOrder,Pass
0,Descending score then Test ascending,"[2, 1, 3]","[2, 1, 3]",True
1,Ascending score then Test ascending,"[2, 1, 3]","[2, 1, 3]",True



Evaluation metric cohort by build:


,BuildKey,BuildOrder,Tests,Failures,TotalDuration
0,121149283,617,551,2,565454.0
1,121159550,618,551,2,611885.0
2,121163441,619,551,2,524855.0
3,121166609,620,551,2,586222.0
4,121169737,621,551,2,565282.0
5,122449871,642,557,1,574219.0
6,123035563,669,48,7,100863.0
7,123038249,670,558,4,659884.0
8,123081030,672,559,1,607402.0
9,123104273,674,785,99,1031685.0




=== PROJECT 9 CELL 7 / STEP 4A RESULT ===

Project:
camunda@camunda-bpm-platform

Predictor protocol:
Dataset columns: 154
Original predictors: 151
Zero-variance predictors: 0
Active predictors: 151
Training matrix non-finite values: 0
Evaluation matrix non-finite values: 0

Models and baselines:
ML techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Baselines: ['Random', 'LatestFail', 'QTF-Avg']
Model-constructor failures: 0

Evaluation metric cohort:
Evaluation-period builds: 206
Raw evaluation rows: 158781
Scored evaluation builds: 30
Scored evaluation rows: 18503
Evaluation failures: 665
Metric join missing rows: 0
Metric verdict mismatches: 0

Metric protocol:
Primary metric: APFDc
Secondary metric: APFD
Metric-helper failures: 0
Ranking-helper failures: 0

Library versions:
Python: 3.12.13
numpy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1
xgboost: 3.3.0
lightgbm: 4.6.0

Validation:
Checks: 26
Failed checks: 0

Model/metric checkpoint:
/content/drive/MyDrive/Thesis_Ex

In [12]:
# ============================================================
# PROJECT 9 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END EXPERIMENT SMOKE TEST
#
# PROJECT: camunda@camunda-bpm-platform
#
# Smoke conditions:
#   0% noise, seed 1
#   50% noise, seed 1
#
# This cell:
# - regenerates the two frozen noise conditions
# - recomputes all 13 verdict-dependent REC features
# - preserves all six verdict-independent REC features
# - creates the noisy fixed model-training cohort
# - fits RF, XGBoost, LightGBM and GaussianNB
# - calculates Random, LatestFail and QTF-Avg rankings
# - ranks all 18,503 clean evaluation rows
# - calculates build-level APFD and APFDc
# - calculates condition-level mean APFD and APFDc
# - validates Random and QTF-Avg invariance across noise
# - validates clean evaluation immutability
# - writes only smoke-test artefacts
#
# This cell does NOT:
# - run all 270 conditions
# - write to the final raw-result directory
# - modify the completion registry
# - modify Projects 1–8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from IPython.display import display

import gc
import hashlib
import json
import re
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = "camunda"


EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_9_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)

STEP4B_PASS_STATUS = (
    "PASS_PROJECT_9_TWO_CONDITION_END_TO_END_SMOKE_TEST_VALIDATED"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)


EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

EXPECTED_EVALUATION_PERIOD_BUILDS = 206
EXPECTED_SCORED_EVALUATION_BUILDS = 30
EXPECTED_EVALUATION_FAILURES = 665

EXPECTED_ACTIVE_PREDICTORS = 151

EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVALUATION_ROWS
    * EXPECTED_TECHNIQUES
)

EXPECTED_BUILD_METRICS_PER_CONDITION = (
    EXPECTED_SCORED_EVALUATION_BUILDS
    * EXPECTED_TECHNIQUES
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    EXPECTED_TECHNIQUES
)

EXPECTED_TOTAL_ML_FITS = (
    2
    * EXPECTED_ML_TECHNIQUES
)

EXPECTED_TOTAL_RANKING_ROWS = (
    2
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRICS = (
    2
    * EXPECTED_BUILD_METRICS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    2
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)


SMOKE_CONDITIONS = [
    {
        "NoisePercent": 0,
        "RepetitionSeed": 1,
    },
    {
        "NoisePercent": 50,
        "RepetitionSeed": 1,
    },
]


RECENT_WINDOW = 6

COMPARISON_RTOL = 1e-9
COMPARISON_ATOL = 1e-9


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]


BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]


ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


# ------------------------------------------------------------
# 2. REC FEATURE PROTOCOL
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:

    raise AssertionError(
        "Expected 13 verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:

    raise AssertionError(
        "Expected six verdict-independent REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    & set(VERDICT_INDEPENDENT_REC_FEATURES)
):

    raise AssertionError(
        "REC dependency classes overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):

    raise AssertionError(
        "REC dependency classes do not cover all REC features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)


REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

NOISE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noise_plan_checkpoint.json"
)

NOISY_REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noisy_rec_engine_checkpoint.json"
)

MODEL_METRIC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_model_metric_checkpoint.json"
)

SMOKE_TEST_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_smoke_test_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)


STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_status.json"
)


SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)


NOISE_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_preflight"
)

FAILURE_SUBTYPE_DISTRIBUTION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_failure_subtype_distribution.csv"
)


SMOKE_TEST_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_test"
)

SMOKE_CONDITION_ROOT = (
    SMOKE_TEST_DIR
    / "conditions"
)

SMOKE_AGGREGATED_RANKINGS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_rankings.parquet"
)

SMOKE_AGGREGATED_BUILD_METRICS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_build_metrics.csv"
)

SMOKE_AGGREGATED_PROJECT_RUNS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_project_runs.csv"
)

SMOKE_AGGREGATED_MODEL_FITS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_model_fits.csv"
)

SMOKE_CONDITION_AUDIT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_condition_audit.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_baseline_invariance.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_report.json"
)


print("=" * 118)
print("=== PROJECT 9 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 118)


# ------------------------------------------------------------
# 4. FILE AND HASH HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def dataframe_content_sha256(
    dataframe,
    include_index=True,
):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=include_index,
        categorize=True,
    ).to_numpy(
        dtype="<u8"
    )

    digest = hashlib.sha256()

    digest.update(
        json.dumps(
            list(
                dataframe.columns
            ),
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )

    digest.update(
        b"\0"
    )

    digest.update(
        row_hashes.tobytes(
            order="C"
        )
    )

    return digest.hexdigest()


def packed_boolean_sha256(mask):

    packed = np.packbits(
        np.asarray(
            mask,
            dtype=np.uint8,
        ),
        bitorder="little",
    )

    return hashlib.sha256(
        packed.tobytes()
    ).hexdigest()


def integer_array_sha256(values):

    values = np.asarray(
        values,
        dtype="<i4",
    )

    return hashlib.sha256(
        values.tobytes(
            order="C"
        )
    ).hexdigest()


# ------------------------------------------------------------
# 5. IDENTIFIER AND SEED HELPERS
# ------------------------------------------------------------

def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


# ------------------------------------------------------------
# 6. NOISE HELPERS
# ------------------------------------------------------------

def create_seed_random_streams(
    number_of_rows,
    repetition_seed,
    failure_subtypes,
    subtype_probabilities,
):
    flip_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "flip_mask",
    )

    subtype_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "failure_subtype",
    )

    flip_rng = np.random.default_rng(
        flip_stream_seed
    )

    subtype_rng = np.random.default_rng(
        subtype_stream_seed
    )

    return {
        "FlipStreamSeed":
            int(
                flip_stream_seed
            ),

        "SubtypeStreamSeed":
            int(
                subtype_stream_seed
            ),

        "FlipUniforms":
            flip_rng.random(
                number_of_rows
            ),

        "SampledFailureSubtypes":
            subtype_rng.choice(
                failure_subtypes,
                size=number_of_rows,
                replace=True,
                p=subtype_probabilities,
            ).astype(
                np.int32
            ),
    }


def construct_condition_arrays(
    clean_verdicts,
    row_uniforms,
    sampled_failure_subtypes,
    noise_percent,
):
    flip_mask = (
        row_uniforms
        < float(
            noise_percent
        ) / 100.0
    )

    pass_to_failure_mask = (
        flip_mask
        & (
            clean_verdicts
            == 0
        )
    )

    failure_to_pass_mask = (
        flip_mask
        & (
            clean_verdicts
            != 0
        )
    )

    noisy_verdicts = (
        clean_verdicts.copy()
    )

    noisy_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_verdicts[
        failure_to_pass_mask
    ] = 0

    return {
        "FlipMask":
            flip_mask,

        "PassToFailureMask":
            pass_to_failure_mask,

        "FailureToPassMask":
            failure_to_pass_mask,

        "NoisyVerdicts":
            noisy_verdicts,
    }


# ------------------------------------------------------------
# 7. APFD AND APFDc HELPERS
# ------------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = int(
        len(
            failures
        )
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):

        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    value = (
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )

    return float(
        value
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):

        raise ValueError(
            "Failures and durations differ in length."
        )

    if (
        len(
            failures
        ) == 0
        or failures.sum() == 0
    ):

        return np.nan

    if not np.isfinite(
        durations
    ).all():

        raise ValueError(
            "Durations contain non-finite values."
        )

    if (
        durations < 0
    ).any():

        raise ValueError(
            "Durations contain negative values."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:

        return np.nan

    cumulative_before = np.concatenate([
        np.asarray(
            [
                0.0,
            ]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    value = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

    return float(
        value
    )


# ------------------------------------------------------------
# 8. VALIDATE CHECKPOINTS
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_CHECKPOINT_PATH,
    NOISY_REC_CHECKPOINT_PATH,
    MODEL_METRIC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
    STEP4A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
]


missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 4B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_checkpoint = json.loads(
    SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

rec_checkpoint = json.loads(
    REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noise_checkpoint = json.loads(
    NOISE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noisy_rec_checkpoint = json.loads(
    NOISY_REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

model_metric_checkpoint = json.loads(
    MODEL_METRIC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step2b_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3a_status = json.loads(
    STEP3A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3b_status = json.loads(
    STEP3B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step4a_status = json.loads(
    STEP4A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


expected_statuses = [
    (
        "Step 2B",
        step2b_status.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Step 4A",
        step4a_status.get(
            "Status"
        ),
        EXPECTED_STEP4A_STATUS,
    ),
]


for step_name, actual, expected in (
    expected_statuses
):

    if actual != expected:

        raise AssertionError(
            f"{step_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


if (
    rec_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
):

    raise AssertionError(
        "REC checkpoint status differs."
    )


if (
    noise_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP3A_STATUS
):

    raise AssertionError(
        "Noise checkpoint status differs."
    )


if (
    noisy_rec_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP3B_STATUS
):

    raise AssertionError(
        "Noisy REC checkpoint status differs."
    )


if (
    model_metric_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP4A_STATUS
):

    raise AssertionError(
        "Model/metric checkpoint status differs."
    )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 9. VALIDATE REGISTRY
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 8
    or set(
        project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the registry."
    )


# ------------------------------------------------------------
# 10. RESOLVE FROZEN INPUT PATHS
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingHistory"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingData"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationData"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "NoiseConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

anchor_offsets_path = Path(
    rec_checkpoint[
        "CleanRECAnchorOffsets"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)

active_predictors_path = Path(
    model_metric_checkpoint[
        "ActivePredictors"
    ]
)

model_configuration_path = Path(
    model_metric_checkpoint[
        "ModelConfiguration"
    ]
)

raw_evaluation_path = Path(
    model_metric_checkpoint[
        "RawEvaluationHistory"
    ]
)

evaluation_build_table_path = Path(
    model_metric_checkpoint[
        "EvaluationBuildTable"
    ]
)

evaluation_metric_cohort_path = Path(
    model_metric_checkpoint[
        "EvaluationMetricCohort"
    ]
)


frozen_input_hashes = {
    raw_training_path:
        noise_checkpoint[
            "RawTrainingHistorySHA256"
        ],

    model_training_path:
        noise_checkpoint[
            "ModelTrainingDataSHA256"
        ],

    model_evaluation_path:
        noise_checkpoint[
            "ModelEvaluationDataSHA256"
        ],

    condition_plan_path:
        noise_checkpoint[
            "NoiseConditionPlanSHA256"
        ],

    clean_direct_rec_path:
        rec_checkpoint[
            "CleanRECReconstructedSHA256"
        ],

    anchor_offsets_path:
        rec_checkpoint[
            "CleanRECAnchorOffsetsSHA256"
        ],

    build_entity_map_path:
        rec_checkpoint[
            "BuildEntityMapSHA256"
        ],

    active_predictors_path:
        model_metric_checkpoint[
            "ActivePredictorsSHA256"
        ],

    model_configuration_path:
        model_metric_checkpoint[
            "ModelConfigurationSHA256"
        ],

    raw_evaluation_path:
        model_metric_checkpoint[
            "RawEvaluationHistorySHA256"
        ],

    evaluation_build_table_path:
        model_metric_checkpoint[
            "EvaluationBuildTableSHA256"
        ],

    evaluation_metric_cohort_path:
        model_metric_checkpoint[
            "EvaluationMetricCohortSHA256"
        ],
}


for path, expected_hash in (
    frozen_input_hashes.items()
):

    if not path.exists():

        raise FileNotFoundError(
            "Frozen smoke-test input is missing:\n"
            f"{path}"
        )

    actual_hash = calculate_sha256(
        path
    )

    if actual_hash != expected_hash:

        raise AssertionError(
            "Frozen smoke-test input hash differs.\n"
            f"Path: {path}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )


source_paths = {
    relative_name:
        Path(
            metadata[
                "RuntimePath"
            ]
        )
    for relative_name, metadata
    in selection_checkpoint[
        "SourceFiles"
    ].items()
}


source_hashes_before = {}


for relative_name, path in (
    source_paths.items()
):

    expected_hash = (
        selection_checkpoint[
            "SourceFiles"
        ][
            relative_name
        ][
            "SHA256"
        ]
    )

    actual_hash = calculate_sha256(
        path
    )

    if actual_hash != expected_hash:

        raise AssertionError(
            "Frozen source-file hash differs.\n"
            f"File: {relative_name}"
        )

    source_hashes_before[
        relative_name
    ] = actual_hash


evaluation_sha256_before = calculate_sha256(
    model_evaluation_path
)


# ------------------------------------------------------------
# 11. LOAD FROZEN CONFIGURATION
# ------------------------------------------------------------

active_predictor_payload = json.loads(
    active_predictors_path.read_text(
        encoding="utf-8"
    )
)

model_configuration = json.loads(
    model_configuration_path.read_text(
        encoding="utf-8"
    )
)


ACTIVE_FEATURE_COLUMNS = (
    active_predictor_payload[
        "ActivePredictors"
    ]
)


ZERO_VARIANCE_FEATURES = (
    active_predictor_payload[
        "ZeroVariancePredictors"
    ]
)


MODEL_CONFIG = (
    model_configuration[
        "Models"
    ]
)


if len(
    ACTIVE_FEATURE_COLUMNS
) != EXPECTED_ACTIVE_PREDICTORS:

    raise AssertionError(
        "Active predictor count differs.\n"
        f"Expected: {EXPECTED_ACTIVE_PREDICTORS}\n"
        f"Actual: {len(ACTIVE_FEATURE_COLUMNS)}"
    )


if len(
    ZERO_VARIANCE_FEATURES
) != 0:

    raise AssertionError(
        "Step 4A recorded unexpected zero-variance predictors."
    )


resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


# ------------------------------------------------------------
# 12. LOAD FROZEN DATA
# ------------------------------------------------------------

raw_training = (
    pd.read_parquet(
        raw_training_path
    )
    .sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

clean_model_training = (
    pd.read_parquet(
        model_training_path
    )
    .reset_index(
        drop=True
    )
)

clean_model_evaluation = (
    pd.read_parquet(
        model_evaluation_path
    )
    .reset_index(
        drop=True
    )
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)

failure_subtype_distribution = pd.read_csv(
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
    low_memory=False,
)

clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)

anchor_offsets = pd.read_parquet(
    anchor_offsets_path
)

build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

evaluation_build_table = pd.read_csv(
    evaluation_build_table_path,
    low_memory=False,
)

evaluation_metric_cohort = pd.read_parquet(
    evaluation_metric_cohort_path
)


if len(
    raw_training
) != EXPECTED_RAW_TRAINING_ROWS:

    raise AssertionError(
        "Raw training row count differs."
    )


if len(
    clean_model_training
) != EXPECTED_MODEL_TRAINING_ROWS:

    raise AssertionError(
        "Model training row count differs."
    )


if len(
    clean_model_evaluation
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Model evaluation row count differs."
    )


if len(
    evaluation_build_table
) != EXPECTED_EVALUATION_PERIOD_BUILDS:

    raise AssertionError(
        "Evaluation-period build count differs."
    )


if len(
    evaluation_metric_cohort
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Evaluation metric-cohort row count differs."
    )


# ------------------------------------------------------------
# 13. PREPARE CANONICAL KEYS
# ------------------------------------------------------------

raw_training[
    "BuildKey"
] = raw_training[
    "BuildKey"
].astype(str)

raw_training[
    "TestKey"
] = raw_training[
    "TestKey"
].astype(str)

raw_training[
    "BuildOrder"
] = pd.to_numeric(
    raw_training[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


expected_noise_row_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int64
    ),
    expected_noise_row_ids,
):

    raise AssertionError(
        "Raw training NoiseRowID order differs."
    )


clean_raw_verdicts = pd.to_numeric(
    raw_training[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


clean_model_training[
    "_ModelRowOrder"
] = np.arange(
    EXPECTED_MODEL_TRAINING_ROWS,
    dtype=np.int64,
)

clean_model_training[
    "BuildKey"
] = canonical_identifier(
    clean_model_training[
        dataset_build_column
    ]
)

clean_model_training[
    "TestKey"
] = canonical_identifier(
    clean_model_training[
        dataset_test_column
    ]
)


clean_model_evaluation[
    "_ModelRowOrder"
] = np.arange(
    EXPECTED_MODEL_EVALUATION_ROWS,
    dtype=np.int64,
)

clean_model_evaluation[
    "BuildKey"
] = canonical_identifier(
    clean_model_evaluation[
        dataset_build_column
    ]
)

clean_model_evaluation[
    "TestKey"
] = canonical_identifier(
    clean_model_evaluation[
        dataset_test_column
    ]
)


raw_key_manifest = (
    raw_training[
        [
            "NoiseRowID",
            "BuildKey",
            "TestKey",
            "CleanVerdict",
        ]
    ]
    .copy()
)


model_raw_alignment = (
    clean_model_training[
        [
            "_ModelRowOrder",
            "BuildKey",
            "TestKey",
        ]
    ]
    .merge(
        raw_key_manifest,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_raw_rows = int(
    model_raw_alignment[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


if missing_model_raw_rows:

    raise AssertionError(
        "Model-training rows are missing raw verdict rows."
    )


model_noise_row_ids = (
    model_raw_alignment[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int64
    )
)


clean_model_verdicts = pd.to_numeric(
    clean_model_training[
        dataset_verdict_column
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


aligned_clean_raw_verdicts = pd.to_numeric(
    model_raw_alignment[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


clean_raw_model_label_mismatches = int(
    (
        clean_model_verdicts
        != aligned_clean_raw_verdicts
    ).sum()
)


if clean_raw_model_label_mismatches:

    raise AssertionError(
        "Clean raw/model training verdicts differ."
    )


# ------------------------------------------------------------
# 14. ALIGN CLEAN REC RECONSTRUCTION AND OFFSETS
# ------------------------------------------------------------

model_training_key_frame = (
    clean_model_training[
        [
            "_ModelRowOrder",
            "BuildKey",
            "TestKey",
        ]
    ]
    .copy()
)


clean_direct_training = (
    model_training_key_frame.merge(
        clean_direct_rec[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if clean_direct_training[
    "_merge"
].ne(
    "both"
).any():

    raise AssertionError(
        "Clean direct REC rows are missing."
    )


anchor_training = (
    model_training_key_frame.merge(
        anchor_offsets[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if anchor_training[
    "_merge"
].ne(
    "both"
).any():

    raise AssertionError(
        "Clean REC anchor-offset rows are missing."
    )


clean_direct_dependent_matrix = (
    clean_direct_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


anchor_dependent_matrix = (
    anchor_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


original_dependent_matrix = (
    clean_model_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    clean_direct_dependent_matrix
    + anchor_dependent_matrix,
    original_dependent_matrix,
    rtol=COMPARISON_RTOL,
    atol=COMPARISON_ATOL,
    equal_nan=False,
).all():

    raise AssertionError(
        "Frozen clean direct REC plus anchor does not "
        "reproduce the clean training REC values."
    )


requested_pairs_by_test = {
    str(
        test_key
    ):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in clean_model_training.groupby(
        "TestKey",
        sort=False,
    )
}


# ------------------------------------------------------------
# 15. PREPARE BUILD/ENTITY STRUCTURE
# ------------------------------------------------------------

build_entity_map[
    "BuildKey"
] = build_entity_map[
    "BuildKey"
].astype(str)

build_entity_map[
    "EntityId"
] = pd.to_numeric(
    build_entity_map[
        "EntityId"
    ],
    errors="raise",
).astype(int)


changed_entities_by_build = {
    str(
        build_key
    ):
        set(
            group[
                "EntityId"
            ].astype(int)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


entity_changed_builds = {
    int(
        entity_id
    ):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for entity_id, group
    in build_entity_map.groupby(
        "EntityId",
        sort=False,
    )
}


# ------------------------------------------------------------
# 16. PREPARE FAILURE SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

failure_subtypes = pd.to_numeric(
    failure_subtype_distribution[
        "FailureSubtype"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


subtype_probabilities = pd.to_numeric(
    failure_subtype_distribution[
        "SamplingProbability"
    ],
    errors="raise",
).to_numpy(
    dtype=float
)


if not np.isclose(
    subtype_probabilities.sum(),
    1.0,
    rtol=0,
    atol=1e-12,
):

    raise AssertionError(
        "Failure subtype probabilities do not sum to one."
    )


# ------------------------------------------------------------
# 17. PREPARE CLEAN EVALUATION RANKING DATA
# ------------------------------------------------------------

evaluation_metric_cohort = (
    evaluation_metric_cohort.sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


evaluation_keys_match = (
    clean_model_evaluation[
        [
            "BuildKey",
            "TestKey",
        ]
    ]
    .reset_index(
        drop=True
    )
    .equals(
        evaluation_metric_cohort[
            [
                "BuildKey",
                "TestKey",
            ]
        ]
        .astype(str)
        .reset_index(
            drop=True
        )
    )
)


if not evaluation_keys_match:

    raise AssertionError(
        "Model evaluation and metric-cohort row order differ."
    )


evaluation_rank_data = pd.DataFrame({
    "Build":
        evaluation_metric_cohort[
            dataset_build_column
        ].to_numpy(),

    "Test":
        evaluation_metric_cohort[
            dataset_test_column
        ].to_numpy(),

    "Verdict":
        pd.to_numeric(
            evaluation_metric_cohort[
                dataset_verdict_column
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.int32
        ),

    "Duration":
        pd.to_numeric(
            evaluation_metric_cohort[
                "Duration"
            ],
            errors="raise",
        ).to_numpy(
            dtype=float
        ),

    "BuildOrder":
        pd.to_numeric(
            evaluation_metric_cohort[
                "BuildOrder"
            ],
            errors="raise",
        ).to_numpy(
            dtype=int
        ),

    "BuildKey":
        evaluation_metric_cohort[
            "BuildKey"
        ].astype(str).to_numpy(),

    "TestKey":
        evaluation_metric_cohort[
            "TestKey"
        ].astype(str).to_numpy(),

    "ActualFailure":
        pd.to_numeric(
            evaluation_metric_cohort[
                "ActualFailure"
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.int32
        ),
})


if int(
    evaluation_rank_data[
        "ActualFailure"
    ].sum()
) != EXPECTED_EVALUATION_FAILURES:

    raise AssertionError(
        "Evaluation failure count differs."
    )


if evaluation_rank_data[
    "BuildKey"
].nunique() != EXPECTED_SCORED_EVALUATION_BUILDS:

    raise AssertionError(
        "Scored evaluation-build count differs."
    )


raw_evaluation[
    "BuildKey"
] = raw_evaluation[
    "BuildKey"
].astype(str)

raw_evaluation[
    "TestKey"
] = raw_evaluation[
    "TestKey"
].astype(str)

raw_evaluation[
    "BuildOrder"
] = pd.to_numeric(
    raw_evaluation[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)

raw_evaluation[
    "CleanVerdict"
] = pd.to_numeric(
    raw_evaluation[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int32)

raw_evaluation[
    "Duration"
] = pd.to_numeric(
    raw_evaluation[
        "Duration"
    ],
    errors="raise",
).astype(float)


evaluation_build_table[
    "BuildKey"
] = evaluation_build_table[
    "BuildKey"
].astype(str)

evaluation_build_table[
    "BuildOrder"
] = pd.to_numeric(
    evaluation_build_table[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


# ------------------------------------------------------------
# 18. NOISY REC RECONSTRUCTION ENGINE
# ------------------------------------------------------------

def calculate_max_test_file_rate(
    target_builds,
    current_changed_entities,
):
    if len(
        target_builds
    ) == 0:

        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:

        changed_builds = (
            entity_changed_builds.get(
                int(
                    entity_id
                ),
                set(),
            )
        )

        overlap_count = len(
            changed_builds.intersection(
                target_builds
            )
        )

        if overlap_count > maximum_frequency:

            maximum_frequency = (
                overlap_count
            )

    if maximum_frequency == 0:

        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_noisy_dependent_rec(
    noisy_history,
):
    reconstructed_records = []

    for test_key, test_history in (
        noisy_history.groupby(
            "TestKey",
            sort=False,
        )
    ):

        test_key = str(
            test_key
        )

        requested_builds = (
            requested_pairs_by_test.get(
                test_key
            )
        )

        if not requested_builds:
            continue

        test_history = (
            test_history.sort_values(
                "BuildOrder",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        prior_count = 0

        prior_failure_count = 0
        prior_assertion_count = 0
        prior_exception_count = 0
        prior_transition_count = 0

        last_failure_position = None
        last_transition_position = None

        previous_verdict = None

        recent_history = deque(
            maxlen=RECENT_WINDOW
        )

        failure_builds = set()
        transition_builds = set()

        for row in test_history.itertuples(
            index=False
        ):

            current_build = str(
                row.BuildKey
            )

            current_verdict = int(
                row.NoisyVerdict
            )

            if current_build in requested_builds:

                record = {
                    "BuildKey":
                        current_build,

                    "TestKey":
                        test_key,
                }

                if prior_count == 0:

                    for feature in (
                        VERDICT_DEPENDENT_REC_FEATURES
                    ):

                        record[
                            feature
                        ] = -1.0

                else:

                    if last_failure_position is None:

                        last_failure_age = -1.0

                    else:

                        last_failure_age = float(
                            prior_count
                            - 1
                            - last_failure_position
                        )

                    if last_transition_position is None:

                        last_transition_age = -1.0

                    else:

                        last_transition_age = float(
                            prior_count
                            - 1
                            - last_transition_position
                        )

                    recent_rows = list(
                        recent_history
                    )

                    recent_length = len(
                        recent_rows
                    )

                    if recent_length == 0:

                        raise AssertionError(
                            "Recent verdict history is unexpectedly empty."
                        )

                    recent_verdicts = np.asarray(
                        [
                            item[
                                "Verdict"
                            ]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )

                    recent_transitions = np.asarray(
                        [
                            item[
                                "Transition"
                            ]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )

                    current_changed_entities = (
                        changed_entities_by_build.get(
                            current_build,
                            set(),
                        )
                    )

                    record.update({
                        "REC_LastFailureAge":
                            last_failure_age,

                        "REC_LastTransitionAge":
                            last_transition_age,

                        "REC_RecentFailRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts
                                    != 0
                                )
                                / recent_length
                            ),

                        "REC_RecentAssertRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts
                                    == 2
                                )
                                / recent_length
                            ),

                        "REC_RecentExcRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts
                                    == 1
                                )
                                / recent_length
                            ),

                        "REC_RecentTransitionRate":
                            float(
                                np.count_nonzero(
                                    recent_transitions
                                    == 1
                                )
                                / recent_length
                            ),

                        "REC_TotalFailRate":
                            float(
                                prior_failure_count
                                / prior_count
                            ),

                        "REC_TotalAssertRate":
                            float(
                                prior_assertion_count
                                / prior_count
                            ),

                        "REC_TotalExcRate":
                            float(
                                prior_exception_count
                                / prior_count
                            ),

                        "REC_TotalTransitionRate":
                            float(
                                prior_transition_count
                                / prior_count
                            ),

                        "REC_LastVerdict":
                            float(
                                previous_verdict
                            ),

                        "REC_MaxTestFileFailRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    failure_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),

                        "REC_MaxTestFileTransitionRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    transition_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),
                    })

                reconstructed_records.append(
                    record
                )

            current_transition = (
                0
                if previous_verdict is None
                else int(
                    current_verdict
                    != previous_verdict
                )
            )

            if current_verdict != 0:

                prior_failure_count += 1

                last_failure_position = (
                    prior_count
                )

                failure_builds.add(
                    current_build
                )

            if current_verdict == 2:

                prior_assertion_count += 1

            if current_verdict == 1:

                prior_exception_count += 1

            if current_transition == 1:

                prior_transition_count += 1

                last_transition_position = (
                    prior_count
                )

                transition_builds.add(
                    current_build
                )

            recent_history.append({
                "Verdict":
                    current_verdict,

                "Transition":
                    current_transition,
            })

            previous_verdict = (
                current_verdict
            )

            prior_count += 1

    reconstructed = pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ),
    )

    return reconstructed


# ------------------------------------------------------------
# 19. ML PREPROCESSING AND MODEL HELPERS
# ------------------------------------------------------------

def prepare_ml_matrices(
    noisy_training_data,
):
    X_train = (
        noisy_training_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    X_evaluation = (
        clean_model_evaluation[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    training_medians = (
        X_train.median(
            axis=0
        )
        .fillna(
            0.0
        )
    )

    X_train = (
        X_train.fillna(
            training_medians
        )
        .astype(float)
    )

    X_evaluation = (
        X_evaluation.fillna(
            training_medians
        )
        .astype(float)
    )

    y_train = (
        pd.to_numeric(
            noisy_training_data[
                dataset_verdict_column
            ],
            errors="raise",
        )
        .ne(0)
        .astype(int)
    )

    if y_train.nunique() != 2:

        raise AssertionError(
            "Noisy model-training labels do not contain both classes."
        )

    if not np.isfinite(
        X_train.to_numpy(
            dtype=float
        )
    ).all():

        raise AssertionError(
            "Noisy training matrix contains non-finite values."
        )

    if not np.isfinite(
        X_evaluation.to_numpy(
            dtype=float
        )
    ).all():

        raise AssertionError(
            "Evaluation matrix contains non-finite values."
        )

    return (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    )


def create_ml_models(
    repetition_seed,
):
    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )

    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )

    lgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )

    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                var_smoothing=(
                    MODEL_CONFIG[
                        "NaiveBayes"
                    ][
                        "var_smoothing"
                    ]
                )
            ),
    }


def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = (
        fitted_model.predict_proba(
            feature_matrix
        )
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.where(
        classes == 1
    )[0]

    if len(
        positive_positions
    ) != 1:

        raise AssertionError(
            "Could not identify fitted failure class 1."
        )

    output = probabilities[
        :,
        positive_positions[
            0
        ],
    ]

    if not np.isfinite(
        output
    ).all():

        raise AssertionError(
            "Predicted probabilities contain non-finite values."
        )

    if not (
        (
            output >= 0.0
        )
        &
        (
            output <= 1.0
        )
    ).all():

        raise AssertionError(
            "Predicted probabilities fall outside [0, 1]."
        )

    return output


# ------------------------------------------------------------
# 20. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    ranked = (
        build_rows[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                "BuildOrder",
                "BuildKey",
                "TestKey",
                "ActualFailure",
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    scores = np.asarray(
        scores,
        dtype=float,
    )

    if len(
        scores
    ) != len(
        ranked
    ):

        raise ValueError(
            "Ranking score count differs from row count."
        )

    if np.isnan(
        scores
    ).any():

        raise ValueError(
            "Ranking scores contain NaN values."
        )

    if np.isneginf(
        scores
    ).any():

        raise ValueError(
            "Ranking scores contain negative infinity."
        )

    if (
        np.isposinf(
            scores
        ).any()
        and technique != "QTF-Avg"
    ):

        raise ValueError(
            "Only QTF-Avg may contain positive-infinity scores."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = scores

    numeric_tests = pd.to_numeric(
        ranked[
            "Test"
        ],
        errors="coerce",
    )

    if numeric_tests.notna().all():

        ranked[
            "_TestTieKey"
        ] = numeric_tests

    else:

        ranked[
            "_TestTieKey"
        ] = ranked[
            "Test"
        ].astype(str)

    if score_direction == "descending":

        ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":

        ascending = [
            True,
            True,
        ]

    else:

        raise ValueError(
            "Invalid score direction."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "_TestTieKey",
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .drop(
            columns=[
                "_TestTieKey",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(
            ranked
        ) + 1,
    )

    return ranked


def create_ml_rankings(
    technique,
    failure_probabilities,
):
    scored = (
        evaluation_rank_data.copy()
    )

    scored[
        "_FailureProbability"
    ] = np.asarray(
        failure_probabilities,
        dtype=float,
    )

    ranking_frames = []

    for _, build_rows in scored.groupby(
        "BuildKey",
        sort=False,
    ):

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=(
                    build_rows[
                        "_FailureProbability"
                    ].to_numpy(
                        dtype=float
                    )
                ),
                technique=technique,
                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


def create_random_rankings(
    repetition_seed,
):
    ranking_frames = []

    for build_key, build_rows in (
        evaluation_rank_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):

        random_rng = np.random.default_rng(
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                (
                    "Random_baseline_build_"
                    f"{build_key}"
                ),
            )
        )

        scores = random_rng.random(
            len(
                build_rows
            )
        )

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=scores,
                technique="Random",
                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


# ------------------------------------------------------------
# 21. HISTORY-BASED BASELINES
# ------------------------------------------------------------

def create_history_baseline_rankings(
    noisy_raw_training,
):
    latest_failure_order = {}

    noisy_training_sorted = (
        noisy_raw_training.sort_values(
            [
                "BuildOrder",
                "TestKey",
            ],
            kind="mergesort",
        )
    )

    for row in noisy_training_sorted.itertuples(
        index=False
    ):

        test_key = str(
            row.TestKey
        )

        if int(
            row.NoisyVerdict
        ) != 0:

            latest_failure_order[
                test_key
            ] = int(
                row.BuildOrder
            )

    duration_sum = {}

    duration_count = {}

    clean_training_sorted = (
        raw_training.sort_values(
            [
                "BuildOrder",
                "TestKey",
            ],
            kind="mergesort",
        )
    )

    for row in clean_training_sorted.itertuples(
        index=False
    ):

        test_key = str(
            row.TestKey
        )

        duration = float(
            row.Duration
        )

        if not np.isfinite(
            duration
        ):

            continue

        duration_sum[
            test_key
        ] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )

        duration_count[
            test_key
        ] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )

    raw_evaluation_by_build = {
        str(
            build_key
        ):
            rows.copy()
        for build_key, rows
        in raw_evaluation.groupby(
            "BuildKey",
            sort=False,
        )
    }

    model_evaluation_by_build = {
        str(
            build_key
        ):
            rows.copy()
        for build_key, rows
        in evaluation_rank_data.groupby(
            "BuildKey",
            sort=False,
        )
    }

    target_build_keys = set(
        model_evaluation_by_build
    )

    latest_fail_frames = []

    qtf_frames = []

    qtf_unseen_scores = 0

    ordered_evaluation_builds = (
        evaluation_build_table.sort_values(
            "BuildOrder",
            kind="mergesort",
        )
    )

    for build_row in (
        ordered_evaluation_builds.itertuples(
            index=False
        )
    ):

        build_key = str(
            build_row.BuildKey
        )

        if build_key in target_build_keys:

            build_tests = (
                model_evaluation_by_build[
                    build_key
                ].copy()
            )

            latest_scores = []

            qtf_scores = []

            for test_key in build_tests[
                "TestKey"
            ].astype(str):

                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )

                count = duration_count.get(
                    test_key,
                    0,
                )

                if count > 0:

                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / count
                    )

                else:

                    average_duration = (
                        np.inf
                    )

                    qtf_unseen_scores += 1

                qtf_scores.append(
                    float(
                        average_duration
                    )
                )

            latest_fail_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )

            qtf_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )

        current_raw_rows = (
            raw_evaluation_by_build.get(
                build_key
            )
        )

        if current_raw_rows is None:
            continue

        current_raw_rows = (
            current_raw_rows.sort_values(
                "TestKey",
                kind="mergesort",
            )
        )

        for execution_row in (
            current_raw_rows.itertuples(
                index=False
            )
        ):

            test_key = str(
                execution_row.TestKey
            )

            if int(
                execution_row.CleanVerdict
            ) != 0:

                latest_failure_order[
                    test_key
                ] = int(
                    execution_row.BuildOrder
                )

            duration = float(
                execution_row.Duration
            )

            if np.isfinite(
                duration
            ):

                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )

                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )

    latest_fail_rankings = pd.concat(
        latest_fail_frames,
        ignore_index=True,
    )

    qtf_rankings = pd.concat(
        qtf_frames,
        ignore_index=True,
    )

    return (
        latest_fail_rankings,
        qtf_rankings,
        qtf_unseen_scores,
    )


# ------------------------------------------------------------
# 22. METRIC AND VALIDATION HELPERS
# ------------------------------------------------------------

def calculate_build_metrics(
    rankings,
    noise_percent,
    repetition_seed,
):
    metric_records = []

    for (
        technique,
        build_key,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):

        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        actual_failures = (
            ranked_build[
                "ActualFailure"
            ].to_numpy(
                dtype=int
            )
        )

        durations = (
            ranked_build[
                "Duration"
            ].to_numpy(
                dtype=float
            )
        )

        apfd = calculate_apfd(
            actual_failures
        )

        apfdc = calculate_apfdc(
            actual_failures,
            durations,
        )

        metric_records.append({
            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildKey":
                str(
                    build_key
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "BuildOrder"
                    ].iloc[0]
                ),

            "NumberOfTests":
                int(
                    len(
                        ranked_build
                    )
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "APFD":
                apfd,

            "APFDc":
                apfdc,
        })

    return pd.DataFrame(
        metric_records
    )


def aggregate_project_run(
    build_metrics,
):
    project_run = (
        build_metrics.groupby(
            [
                "Project",
                "ProjectSlug",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            ScoredEvaluationBuilds=(
                "BuildKey",
                "nunique",
            ),

            EvaluationRows=(
                "NumberOfTests",
                "sum",
            ),

            EvaluationFailures=(
                "NumberOfFailures",
                "sum",
            ),

            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SDAPFD=(
                "APFD",
                "std",
            ),

            SDAPFDc=(
                "APFDc",
                "std",
            ),
        )
    )

    return project_run


def validate_condition_outputs(
    rankings,
    build_metrics,
    project_run,
    fit_log,
):
    expected_techniques = set(
        ALL_TECHNIQUES
    )

    actual_techniques = set(
        rankings[
            "Technique"
        ].unique()
    )

    if actual_techniques != expected_techniques:

        raise AssertionError(
            "Condition technique set differs."
        )

    if len(
        rankings
    ) != EXPECTED_RANKING_ROWS_PER_CONDITION:

        raise AssertionError(
            "Condition ranking-row count differs."
        )

    if len(
        build_metrics
    ) != EXPECTED_BUILD_METRICS_PER_CONDITION:

        raise AssertionError(
            "Condition build-metric row count differs."
        )

    if len(
        project_run
    ) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:

        raise AssertionError(
            "Condition project-run row count differs."
        )

    if len(
        fit_log
    ) != EXPECTED_ML_TECHNIQUES:

        raise AssertionError(
            "Condition ML fit count differs."
        )

    if not fit_log[
        "FitSuccess"
    ].all():

        raise AssertionError(
            "One or more ML fits failed."
        )

    expected_build_keys = set(
        evaluation_rank_data[
            "BuildKey"
        ].astype(str)
    )

    metric_build_keys = set(
        build_metrics[
            "BuildKey"
        ].astype(str)
    )

    if metric_build_keys != expected_build_keys:

        raise AssertionError(
            "Condition metric-build set differs."
        )

    if rankings.duplicated(
        subset=[
            "Technique",
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).any():

        raise AssertionError(
            "Condition rankings contain duplicate tests."
        )

    for (
        technique,
        build_key,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):

        expected_ranks = np.arange(
            1,
            len(
                ranked_build
            ) + 1,
        )

        actual_ranks = np.sort(
            ranked_build[
                "Rank"
            ].to_numpy(
                dtype=int
            )
        )

        if not np.array_equal(
            actual_ranks,
            expected_ranks,
        ):

            raise AssertionError(
                "Invalid rank sequence.\n"
                f"Technique: {technique}\n"
                f"Build: {build_key}"
            )

    if build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():

        raise AssertionError(
            "Condition metrics contain missing values."
        )

    for metric_column in [
        "APFD",
        "APFDc",
    ]:

        metric_values = build_metrics[
            metric_column
        ].to_numpy(
            dtype=float
        )

        if not (
            (
                metric_values >= 0.0
            )
            &
            (
                metric_values <= 1.0
            )
        ).all():

            raise AssertionError(
                f"{metric_column} falls outside [0, 1]."
            )

    if not (
        project_run[
            "ScoredEvaluationBuilds"
        ].eq(
            EXPECTED_SCORED_EVALUATION_BUILDS
        ).all()
    ):

        raise AssertionError(
            "Project-run scored-build count differs."
        )

    if not (
        project_run[
            "EvaluationRows"
        ].eq(
            EXPECTED_MODEL_EVALUATION_ROWS
        ).all()
    ):

        raise AssertionError(
            "Project-run evaluation-row count differs."
        )

    if not (
        project_run[
            "EvaluationFailures"
        ].eq(
            EXPECTED_EVALUATION_FAILURES
        ).all()
    ):

        raise AssertionError(
            "Project-run evaluation-failure count differs."
        )

    return True


def ranking_signature(
    rankings,
    technique,
):
    subset = (
        rankings[
            rankings[
                "Technique"
            ].eq(
                technique
            )
        ][
            [
                "Technique",
                "BuildKey",
                "TestKey",
                "Score",
                "Rank",
            ]
        ]
        .sort_values(
            [
                "BuildKey",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return dataframe_content_sha256(
        subset,
        include_index=False,
    )


# ------------------------------------------------------------
# 23. RUN ONE COMPLETE SMOKE CONDITION
# ------------------------------------------------------------

def run_smoke_condition(
    noise_percent,
    repetition_seed,
):
    condition_started = time.perf_counter()

    condition_key = (
        f"noise_{int(noise_percent):02d}"
        f"__seed_{int(repetition_seed):02d}"
    )

    print("\n")
    print("-" * 100)

    print(
        "Running smoke condition:",
        condition_key,
    )

    recorded_rows = condition_plan[
        (
            condition_plan[
                "NoisePercent"
            ].eq(
                int(
                    noise_percent
                )
            )
        )
        &
        (
            condition_plan[
                "RepetitionSeed"
            ].eq(
                int(
                    repetition_seed
                )
            )
        )
    ]

    if len(
        recorded_rows
    ) != 1:

        raise AssertionError(
            "Frozen noise condition was not found exactly once."
        )

    recorded_condition = (
        recorded_rows.iloc[0]
    )

    streams = create_seed_random_streams(
        number_of_rows=(
            EXPECTED_RAW_TRAINING_ROWS
        ),

        repetition_seed=(
            repetition_seed
        ),

        failure_subtypes=(
            failure_subtypes
        ),

        subtype_probabilities=(
            subtype_probabilities
        ),
    )

    condition_arrays = (
        construct_condition_arrays(
            clean_verdicts=(
                clean_raw_verdicts
            ),

            row_uniforms=(
                streams[
                    "FlipUniforms"
                ]
            ),

            sampled_failure_subtypes=(
                streams[
                    "SampledFailureSubtypes"
                ]
            ),

            noise_percent=(
                noise_percent
            ),
        )
    )

    flip_mask = condition_arrays[
        "FlipMask"
    ]

    noisy_raw_verdicts = condition_arrays[
        "NoisyVerdicts"
    ]

    generated_mask_hash = (
        packed_boolean_sha256(
            flip_mask
        )
    )

    generated_verdict_hash = (
        integer_array_sha256(
            noisy_raw_verdicts
        )
    )

    mask_hash_match = (
        generated_mask_hash
        == recorded_condition[
            "FlipMaskSHA256"
        ]
    )

    verdict_hash_match = (
        generated_verdict_hash
        == recorded_condition[
            "NoisyVerdictSHA256"
        ]
    )

    if not mask_hash_match:

        raise AssertionError(
            "Generated flip-mask hash differs."
        )

    if not verdict_hash_match:

        raise AssertionError(
            "Generated noisy-verdict hash differs."
        )

    noisy_raw_training = (
        raw_training.copy()
    )

    noisy_raw_training[
        "NoisyVerdict"
    ] = noisy_raw_verdicts

    reconstruction_started = (
        time.perf_counter()
    )

    noisy_direct_rec = (
        reconstruct_noisy_dependent_rec(
            noisy_raw_training[
                [
                    "BuildKey",
                    "TestKey",
                    "BuildOrder",
                    "NoisyVerdict",
                ]
            ]
        )
    )

    reconstruction_seconds = float(
        time.perf_counter()
        - reconstruction_started
    )

    if len(
        noisy_direct_rec
    ) != EXPECTED_MODEL_TRAINING_ROWS:

        raise AssertionError(
            "Noisy REC reconstruction row count differs."
        )

    if noisy_direct_rec.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).any():

        raise AssertionError(
            "Noisy REC reconstruction contains duplicate rows."
        )

    aligned_noisy_direct = (
        model_training_key_frame.merge(
            noisy_direct_rec,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "_ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    missing_noisy_rec_rows = int(
        aligned_noisy_direct[
            "_merge"
        ].ne(
            "both"
        ).sum()
    )

    if missing_noisy_rec_rows:

        raise AssertionError(
            "Noisy REC rows are missing after alignment."
        )

    noisy_direct_matrix = (
        aligned_noisy_direct[
            VERDICT_DEPENDENT_REC_FEATURES
        ]
        .to_numpy(
            dtype=float
        )
    )

    noisy_anchored_matrix = (
        noisy_direct_matrix
        + anchor_dependent_matrix
    )

    if not np.isfinite(
        noisy_anchored_matrix
    ).all():

        raise AssertionError(
            "Noisy anchored REC matrix contains non-finite values."
        )

    noisy_model_training = (
        clean_model_training.drop(
            columns=[
                "_ModelRowOrder",
                "BuildKey",
                "TestKey",
            ]
        )
        .copy()
    )

    unaffected_columns = [
        column
        for column in noisy_model_training.columns
        if column not in (
            [
                dataset_verdict_column,
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        )
    ]

    unaffected_hash_before = (
        dataframe_content_sha256(
            noisy_model_training[
                unaffected_columns
            ],
            include_index=True,
        )
    )

    noisy_model_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )

    noisy_model_training[
        dataset_verdict_column
    ] = noisy_model_verdicts

    noisy_model_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ] = noisy_anchored_matrix

    unaffected_hash_after = (
        dataframe_content_sha256(
            noisy_model_training[
                unaffected_columns
            ],
            include_index=True,
        )
    )

    unaffected_columns_unchanged = (
        unaffected_hash_before
        == unaffected_hash_after
    )

    if not unaffected_columns_unchanged:

        raise AssertionError(
            "Verdict-independent or non-REC predictors changed."
        )

    changed_model_labels = int(
        (
            noisy_model_verdicts
            != clean_model_verdicts
        ).sum()
    )

    changed_dependent_values = int(
        (
            ~np.isclose(
                noisy_anchored_matrix,
                original_dependent_matrix,
                rtol=COMPARISON_RTOL,
                atol=COMPARISON_ATOL,
                equal_nan=False,
            )
        ).sum()
    )

    if noise_percent == 0:

        if changed_model_labels != 0:

            raise AssertionError(
                "0% noise changed model labels."
            )

        if changed_dependent_values != 0:

            raise AssertionError(
                "0% noise changed dependent REC features."
            )

        if not noisy_model_training.equals(
            clean_model_training.drop(
                columns=[
                    "_ModelRowOrder",
                    "BuildKey",
                    "TestKey",
                ]
            )
        ):

            raise AssertionError(
                "0% noisy model-training data does not "
                "equal the clean model-training data."
            )

    else:

        if changed_model_labels == 0:

            raise AssertionError(
                "Positive noise changed no model labels."
            )

        if changed_dependent_values == 0:

            raise AssertionError(
                "Positive noise changed no dependent REC values."
            )

    (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    ) = prepare_ml_matrices(
        noisy_model_training
    )

    all_ranking_frames = []

    fit_records = []

    models = create_ml_models(
        repetition_seed
    )

    for technique, model in (
        models.items()
    ):

        print(
            "  Fitting:",
            technique,
        )

        fit_started = (
            time.perf_counter()
        )

        fit_success = False
        probability_success = False
        error_text = ""

        try:

            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore"
                )

                model.fit(
                    X_train,
                    y_train,
                )

                fit_success = True

                failure_probabilities = (
                    get_failure_probability(
                        model,
                        X_evaluation,
                    )
                )

                probability_success = True

            model_rankings = (
                create_ml_rankings(
                    technique=technique,
                    failure_probabilities=(
                        failure_probabilities
                    ),
                )
            )

            all_ranking_frames.append(
                model_rankings
            )

        except Exception as error:

            error_text = (
                f"{type(error).__name__}: {error}"
            )

            raise

        finally:

            fit_seconds = float(
                time.perf_counter()
                - fit_started
            )

            fit_records.append({
                "Project":
                    PROJECT_NAME,

                "ConditionKey":
                    condition_key,

                "NoisePercent":
                    int(
                        noise_percent
                    ),

                "RepetitionSeed":
                    int(
                        repetition_seed
                    ),

                "Technique":
                    technique,

                "TrainingRows":
                    EXPECTED_MODEL_TRAINING_ROWS,

                "EvaluationRows":
                    EXPECTED_MODEL_EVALUATION_ROWS,

                "ActivePredictors":
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    ),

                "TrainingFailures":
                    int(
                        y_train.sum()
                    ),

                "TrainingPasses":
                    int(
                        (
                            y_train == 0
                        ).sum()
                    ),

                "FitSuccess":
                    fit_success,

                "ProbabilitySuccess":
                    probability_success,

                "FitSeconds":
                    fit_seconds,

                "Error":
                    error_text,
            })

        del model

        gc.collect()

    random_rankings = (
        create_random_rankings(
            repetition_seed
        )
    )

    (
        latest_fail_rankings,
        qtf_rankings,
        qtf_unseen_scores,
    ) = create_history_baseline_rankings(
        noisy_raw_training
    )

    all_ranking_frames.extend([
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ])

    rankings = pd.concat(
        all_ranking_frames,
        ignore_index=True,
    )

    rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    rankings.insert(
        1,
        "ProjectSlug",
        PROJECT_SLUG,
    )

    rankings.insert(
        2,
        "ConditionKey",
        condition_key,
    )

    rankings.insert(
        3,
        "NoisePercent",
        int(
            noise_percent
        ),
    )

    rankings.insert(
        4,
        "RepetitionSeed",
        int(
            repetition_seed
        ),
    )

    fit_log = pd.DataFrame(
        fit_records
    )

    build_metrics = (
        calculate_build_metrics(
            rankings=rankings,
            noise_percent=(
                noise_percent
            ),
            repetition_seed=(
                repetition_seed
            ),
        )
    )

    project_run = (
        aggregate_project_run(
            build_metrics
        )
    )

    validate_condition_outputs(
        rankings=rankings,
        build_metrics=build_metrics,
        project_run=project_run,
        fit_log=fit_log,
    )

    condition_seconds = float(
        time.perf_counter()
        - condition_started
    )

    condition_directory = (
        SMOKE_CONDITION_ROOT
        / condition_key
    )

    rankings_path = (
        condition_directory
        / "rankings.parquet"
    )

    build_metrics_path = (
        condition_directory
        / "build_metrics.csv"
    )

    project_run_path = (
        condition_directory
        / "project_run.csv"
    )

    fit_log_path = (
        condition_directory
        / "model_fits.csv"
    )

    condition_summary_path = (
        condition_directory
        / "condition_summary.json"
    )

    success_marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )

    atomic_write_parquet(
        rankings_path,
        rankings,
    )

    atomic_write_csv(
        build_metrics_path,
        build_metrics,
    )

    atomic_write_csv(
        project_run_path,
        project_run,
    )

    atomic_write_csv(
        fit_log_path,
        fit_log,
    )

    condition_summary = {
        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionKey":
            condition_key,

        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "RawRowsFlipped":
            int(
                flip_mask.sum()
            ),

        "RecordedRawRowsFlipped":
            int(
                recorded_condition[
                    "RawRowsFlipped"
                ]
            ),

        "RealisedNoisePercent":
            float(
                100.0
                * flip_mask.sum()
                / EXPECTED_RAW_TRAINING_ROWS
            ),

        "PassToFailure":
            int(
                condition_arrays[
                    "PassToFailureMask"
                ].sum()
            ),

        "FailureToPass":
            int(
                condition_arrays[
                    "FailureToPassMask"
                ].sum()
            ),

        "ChangedModelLabels":
            changed_model_labels,

        "ChangedDependentRECValues":
            changed_dependent_values,

        "UnaffectedColumnsUnchanged":
            unaffected_columns_unchanged,

        "MaskHashMatch":
            mask_hash_match,

        "NoisyVerdictHashMatch":
            verdict_hash_match,

        "NoisyRECReconstructionRows":
            len(
                noisy_direct_rec
            ),

        "MissingNoisyRECRows":
            missing_noisy_rec_rows,

        "QTFUnseenScores":
            qtf_unseen_scores,

        "MLFits":
            len(
                fit_log
            ),

        "RankingRows":
            len(
                rankings
            ),

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunRows":
            len(
                project_run
            ),

        "ReconstructionSeconds":
            reconstruction_seconds,

        "ConditionSeconds":
            condition_seconds,

        "Rankings":
            str(
                rankings_path
            ),

        "RankingsSHA256":
            calculate_sha256(
                rankings_path
            ),

        "BuildMetrics":
            str(
                build_metrics_path
            ),

        "BuildMetricsSHA256":
            calculate_sha256(
                build_metrics_path
            ),

        "ProjectRun":
            str(
                project_run_path
            ),

        "ProjectRunSHA256":
            calculate_sha256(
                project_run_path
            ),

        "ModelFits":
            str(
                fit_log_path
            ),

        "ModelFitsSHA256":
            calculate_sha256(
                fit_log_path
            ),

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    atomic_write_json(
        condition_summary_path,
        condition_summary,
    )

    success_payload = {
        "Status":
            "PASS_SMOKE_CONDITION",

        "ConditionKey":
            condition_key,

        "RankingsSHA256":
            condition_summary[
                "RankingsSHA256"
            ],

        "BuildMetricsSHA256":
            condition_summary[
                "BuildMetricsSHA256"
            ],

        "ProjectRunSHA256":
            condition_summary[
                "ProjectRunSHA256"
            ],

        "ModelFitsSHA256":
            condition_summary[
                "ModelFitsSHA256"
            ],

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    atomic_write_json(
        success_marker_path,
        success_payload,
    )

    print(
        "  Completed:",
        condition_key,
    )

    print(
        "  Raw flips:",
        int(
            flip_mask.sum()
        ),
    )

    print(
        "  Model label changes:",
        changed_model_labels,
    )

    print(
        "  Dependent REC changes:",
        changed_dependent_values,
    )

    print(
        "  Ranking rows:",
        len(
            rankings
        ),
    )

    print(
        "  Build-metric rows:",
        len(
            build_metrics
        ),
    )

    print(
        "  Runtime seconds:",
        round(
            condition_seconds,
            2,
        ),
    )

    result = {
        "ConditionKey":
            condition_key,

        "Rankings":
            rankings,

        "BuildMetrics":
            build_metrics,

        "ProjectRun":
            project_run,

        "FitLog":
            fit_log,

        "ConditionSummary":
            condition_summary,

        "RandomSignature":
            ranking_signature(
                rankings,
                "Random",
            ),

        "QTFAvgSignature":
            ranking_signature(
                rankings,
                "QTF-Avg",
            ),

        "LatestFailSignature":
            ranking_signature(
                rankings,
                "LatestFail",
            ),
    }

    del X_train
    del y_train
    del X_evaluation
    del training_medians
    del noisy_model_training
    del noisy_direct_rec
    del aligned_noisy_direct
    del noisy_direct_matrix
    del noisy_anchored_matrix

    gc.collect()

    return result


# ------------------------------------------------------------
# 24. RUN BOTH SMOKE CONDITIONS
# ------------------------------------------------------------

SMOKE_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

smoke_started = time.perf_counter()

condition_results = []


for condition in SMOKE_CONDITIONS:

    condition_results.append(
        run_smoke_condition(
            noise_percent=(
                condition[
                    "NoisePercent"
                ]
            ),

            repetition_seed=(
                condition[
                    "RepetitionSeed"
                ]
            ),
        )
    )


smoke_seconds = float(
    time.perf_counter()
    - smoke_started
)


# ------------------------------------------------------------
# 25. AGGREGATE SMOKE OUTPUTS
# ------------------------------------------------------------

all_smoke_rankings = pd.concat(
    [
        result[
            "Rankings"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)

all_smoke_build_metrics = pd.concat(
    [
        result[
            "BuildMetrics"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)

all_smoke_project_runs = pd.concat(
    [
        result[
            "ProjectRun"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)

all_smoke_model_fits = pd.concat(
    [
        result[
            "FitLog"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)

smoke_condition_audit = pd.DataFrame(
    [
        result[
            "ConditionSummary"
        ]
        for result in condition_results
    ]
)


if len(
    all_smoke_rankings
) != EXPECTED_TOTAL_RANKING_ROWS:

    raise AssertionError(
        "Aggregated smoke ranking-row count differs."
    )


if len(
    all_smoke_build_metrics
) != EXPECTED_TOTAL_BUILD_METRICS:

    raise AssertionError(
        "Aggregated smoke build-metric count differs."
    )


if len(
    all_smoke_project_runs
) != EXPECTED_TOTAL_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Aggregated smoke project-run count differs."
    )


if len(
    all_smoke_model_fits
) != EXPECTED_TOTAL_ML_FITS:

    raise AssertionError(
        "Aggregated smoke ML-fit count differs."
    )


# ------------------------------------------------------------
# 26. BASELINE INVARIANCE
# ------------------------------------------------------------

zero_result = next(
    result
    for result in condition_results
    if result[
        "ConditionSummary"
    ][
        "NoisePercent"
    ] == 0
)


fifty_result = next(
    result
    for result in condition_results
    if result[
        "ConditionSummary"
    ][
        "NoisePercent"
    ] == 50
)


random_invariant = (
    zero_result[
        "RandomSignature"
    ]
    == fifty_result[
        "RandomSignature"
    ]
)


qtf_invariant = (
    zero_result[
        "QTFAvgSignature"
    ]
    == fifty_result[
        "QTFAvgSignature"
    ]
)


latest_fail_same = (
    zero_result[
        "LatestFailSignature"
    ]
    == fifty_result[
        "LatestFailSignature"
    ]
)


baseline_invariance = pd.DataFrame([
    {
        "Technique":
            "Random",

        "ExpectedInvariantAcrossNoise":
            True,

        "ZeroNoiseSignature":
            zero_result[
                "RandomSignature"
            ],

        "FiftyNoiseSignature":
            fifty_result[
                "RandomSignature"
            ],

        "ActualInvariant":
            random_invariant,

        "Pass":
            random_invariant,
    },

    {
        "Technique":
            "QTF-Avg",

        "ExpectedInvariantAcrossNoise":
            True,

        "ZeroNoiseSignature":
            zero_result[
                "QTFAvgSignature"
            ],

        "FiftyNoiseSignature":
            fifty_result[
                "QTFAvgSignature"
            ],

        "ActualInvariant":
            qtf_invariant,

        "Pass":
            qtf_invariant,
    },

    {
        "Technique":
            "LatestFail",

        "ExpectedInvariantAcrossNoise":
            False,

        "ZeroNoiseSignature":
            zero_result[
                "LatestFailSignature"
            ],

        "FiftyNoiseSignature":
            fifty_result[
                "LatestFailSignature"
            ],

        "ActualInvariant":
            latest_fail_same,

        # LatestFail is allowed to remain equal by coincidence.
        # Its noisy-history inputs are independently proven changed.
        "Pass":
            True,
    },
])


baseline_invariance_failures = int(
    (
        ~baseline_invariance[
            "Pass"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 27. VERIFY EVALUATION AND SOURCE IMMUTABILITY
# ------------------------------------------------------------

evaluation_sha256_after = calculate_sha256(
    model_evaluation_path
)

evaluation_unchanged = (
    evaluation_sha256_before
    == evaluation_sha256_after
)


source_hashes_after = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}


source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not evaluation_unchanged:

    raise AssertionError(
        "Frozen evaluation data changed during Step 4B."
    )


if not source_files_unchanged:

    raise AssertionError(
        "A frozen source file changed during Step 4B."
    )


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 4B."
    )


for path, expected_hash in (
    frozen_input_hashes.items()
):

    if calculate_sha256(
        path
    ) != expected_hash:

        raise AssertionError(
            "A frozen experiment input changed during Step 4B.\n"
            f"Path: {path}"
        )


# ------------------------------------------------------------
# 28. OVERALL VALIDATION
# ------------------------------------------------------------

zero_condition_summary = (
    zero_result[
        "ConditionSummary"
    ]
)

fifty_condition_summary = (
    fifty_result[
        "ConditionSummary"
    ]
)


validation_records = [
    {
        "Check":
            "Step 4A passed",

        "Expected":
            EXPECTED_STEP4A_STATUS,

        "Actual":
            step4a_status[
                "Status"
            ],

        "Pass":
            step4a_status[
                "Status"
            ]
            == EXPECTED_STEP4A_STATUS,
    },

    {
        "Check":
            "Smoke conditions",

        "Expected":
            2,

        "Actual":
            len(
                condition_results
            ),

        "Pass":
            len(
                condition_results
            ) == 2,
    },

    {
        "Check":
            "Total ML fits",

        "Expected":
            EXPECTED_TOTAL_ML_FITS,

        "Actual":
            len(
                all_smoke_model_fits
            ),

        "Pass":
            len(
                all_smoke_model_fits
            )
            == EXPECTED_TOTAL_ML_FITS,
    },

    {
        "Check":
            "Successful ML fits",

        "Expected":
            EXPECTED_TOTAL_ML_FITS,

        "Actual":
            int(
                all_smoke_model_fits[
                    "FitSuccess"
                ].sum()
            ),

        "Pass":
            int(
                all_smoke_model_fits[
                    "FitSuccess"
                ].sum()
            )
            == EXPECTED_TOTAL_ML_FITS,
    },

    {
        "Check":
            "Total ranking rows",

        "Expected":
            EXPECTED_TOTAL_RANKING_ROWS,

        "Actual":
            len(
                all_smoke_rankings
            ),

        "Pass":
            len(
                all_smoke_rankings
            )
            == EXPECTED_TOTAL_RANKING_ROWS,
    },

    {
        "Check":
            "Total build-metric rows",

        "Expected":
            EXPECTED_TOTAL_BUILD_METRICS,

        "Actual":
            len(
                all_smoke_build_metrics
            ),

        "Pass":
            len(
                all_smoke_build_metrics
            )
            == EXPECTED_TOTAL_BUILD_METRICS,
    },

    {
        "Check":
            "Total project-run rows",

        "Expected":
            EXPECTED_TOTAL_PROJECT_RUN_ROWS,

        "Actual":
            len(
                all_smoke_project_runs
            ),

        "Pass":
            len(
                all_smoke_project_runs
            )
            == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Techniques per condition",

        "Expected":
            EXPECTED_TECHNIQUES,

        "Actual":
            int(
                all_smoke_project_runs.groupby(
                    [
                        "NoisePercent",
                        "RepetitionSeed",
                    ]
                )[
                    "Technique"
                ].nunique().min()
            ),

        "Pass":
            bool(
                all_smoke_project_runs.groupby(
                    [
                        "NoisePercent",
                        "RepetitionSeed",
                    ]
                )[
                    "Technique"
                ].nunique().eq(
                    EXPECTED_TECHNIQUES
                ).all()
            ),
    },

    {
        "Check":
            "0% flipped raw rows",

        "Expected":
            0,

        "Actual":
            zero_condition_summary[
                "RawRowsFlipped"
            ],

        "Pass":
            zero_condition_summary[
                "RawRowsFlipped"
            ] == 0,
    },

    {
        "Check":
            "0% changed model labels",

        "Expected":
            0,

        "Actual":
            zero_condition_summary[
                "ChangedModelLabels"
            ],

        "Pass":
            zero_condition_summary[
                "ChangedModelLabels"
            ] == 0,
    },

    {
        "Check":
            "0% changed dependent REC values",

        "Expected":
            0,

        "Actual":
            zero_condition_summary[
                "ChangedDependentRECValues"
            ],

        "Pass":
            zero_condition_summary[
                "ChangedDependentRECValues"
            ] == 0,
    },

    {
        "Check":
            "50% changed model labels positive",

        "Expected":
            True,

        "Actual":
            fifty_condition_summary[
                "ChangedModelLabels"
            ] > 0,

        "Pass":
            fifty_condition_summary[
                "ChangedModelLabels"
            ] > 0,
    },

    {
        "Check":
            "50% changed dependent REC values positive",

        "Expected":
            True,

        "Actual":
            fifty_condition_summary[
                "ChangedDependentRECValues"
            ] > 0,

        "Pass":
            fifty_condition_summary[
                "ChangedDependentRECValues"
            ] > 0,
    },

    {
        "Check":
            "Condition mask hashes",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_condition_audit[
                    "MaskHashMatch"
                ].all()
            ),

        "Pass":
            bool(
                smoke_condition_audit[
                    "MaskHashMatch"
                ].all()
            ),
    },

    {
        "Check":
            "Condition noisy-verdict hashes",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_condition_audit[
                    "NoisyVerdictHashMatch"
                ].all()
            ),

        "Pass":
            bool(
                smoke_condition_audit[
                    "NoisyVerdictHashMatch"
                ].all()
            ),
    },

    {
        "Check":
            "Unaffected columns unchanged",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_condition_audit[
                    "UnaffectedColumnsUnchanged"
                ].all()
            ),

        "Pass":
            bool(
                smoke_condition_audit[
                    "UnaffectedColumnsUnchanged"
                ].all()
            ),
    },

    {
        "Check":
            "Random invariant across noise",

        "Expected":
            True,

        "Actual":
            random_invariant,

        "Pass":
            random_invariant,
    },

    {
        "Check":
            "QTF-Avg invariant across noise",

        "Expected":
            True,

        "Actual":
            qtf_invariant,

        "Pass":
            qtf_invariant,
    },

    {
        "Check":
            "Baseline invariance failures",

        "Expected":
            0,

        "Actual":
            baseline_invariance_failures,

        "Pass":
            baseline_invariance_failures == 0,
    },

    {
        "Check":
            "Evaluation unchanged",

        "Expected":
            True,

        "Actual":
            evaluation_unchanged,

        "Pass":
            evaluation_unchanged,
    },

    {
        "Check":
            "Source files unchanged",

        "Expected":
            True,

        "Actual":
            source_files_unchanged,

        "Pass":
            source_files_unchanged,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 4B validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 4B DID NOT PASS."
    )


# ------------------------------------------------------------
# 29. WRITE AGGREGATED SMOKE OUTPUTS
# ------------------------------------------------------------

atomic_write_parquet(
    SMOKE_AGGREGATED_RANKINGS_PATH,
    all_smoke_rankings,
)

atomic_write_csv(
    SMOKE_AGGREGATED_BUILD_METRICS_PATH,
    all_smoke_build_metrics,
)

atomic_write_csv(
    SMOKE_AGGREGATED_PROJECT_RUNS_PATH,
    all_smoke_project_runs,
)

atomic_write_csv(
    SMOKE_AGGREGATED_MODEL_FITS_PATH,
    all_smoke_model_fits,
)

atomic_write_csv(
    SMOKE_CONDITION_AUDIT_PATH,
    smoke_condition_audit,
)

atomic_write_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)

atomic_write_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 30. READBACK VALIDATION
# ------------------------------------------------------------

rankings_readback = pd.read_parquet(
    SMOKE_AGGREGATED_RANKINGS_PATH
)

build_metrics_readback = pd.read_csv(
    SMOKE_AGGREGATED_BUILD_METRICS_PATH,
    low_memory=False,
)

project_runs_readback = pd.read_csv(
    SMOKE_AGGREGATED_PROJECT_RUNS_PATH,
    low_memory=False,
)

model_fits_readback = pd.read_csv(
    SMOKE_AGGREGATED_MODEL_FITS_PATH,
    low_memory=False,
)


if len(
    rankings_readback
) != EXPECTED_TOTAL_RANKING_ROWS:

    raise AssertionError(
        "Smoke ranking readback row count differs."
    )


if len(
    build_metrics_readback
) != EXPECTED_TOTAL_BUILD_METRICS:

    raise AssertionError(
        "Smoke build-metric readback count differs."
    )


if len(
    project_runs_readback
) != EXPECTED_TOTAL_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Smoke project-run readback count differs."
    )


if len(
    model_fits_readback
) != EXPECTED_TOTAL_ML_FITS:

    raise AssertionError(
        "Smoke model-fit readback count differs."
    )


# ------------------------------------------------------------
# 31. REPORT, CHECKPOINT AND STATUS
# ------------------------------------------------------------

smoke_output_paths = [
    SMOKE_AGGREGATED_RANKINGS_PATH,
    SMOKE_AGGREGATED_BUILD_METRICS_PATH,
    SMOKE_AGGREGATED_PROJECT_RUNS_PATH,
    SMOKE_AGGREGATED_MODEL_FITS_PATH,
    SMOKE_CONDITION_AUDIT_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_VALIDATION_PATH,
]


smoke_output_inventory = []


for path in smoke_output_paths:

    smoke_output_inventory.append({
        "Path":
            str(
                path
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "SmokeConditions":
        SMOKE_CONDITIONS,

    "ConditionsCompleted":
        len(
            condition_results
        ),

    "MLFits":
        len(
            all_smoke_model_fits
        ),

    "RankingRows":
        len(
            all_smoke_rankings
        ),

    "BuildMetricRows":
        len(
            all_smoke_build_metrics
        ),

    "ProjectRunRows":
        len(
            all_smoke_project_runs
        ),

    "Techniques":
        ALL_TECHNIQUES,

    "ZeroNoiseChangedModelLabels":
        zero_condition_summary[
            "ChangedModelLabels"
        ],

    "ZeroNoiseChangedDependentRECValues":
        zero_condition_summary[
            "ChangedDependentRECValues"
        ],

    "FiftyNoiseChangedModelLabels":
        fifty_condition_summary[
            "ChangedModelLabels"
        ],

    "FiftyNoiseChangedDependentRECValues":
        fifty_condition_summary[
            "ChangedDependentRECValues"
        ],

    "RandomInvariantAcrossNoise":
        random_invariant,

    "QTFAvgInvariantAcrossNoise":
        qtf_invariant,

    "LatestFailInvariantAcrossNoise":
        latest_fail_same,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "SmokeRuntimeSeconds":
        smoke_seconds,

    "OutputInventory":
        smoke_output_inventory,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    SMOKE_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RECCheckpointSHA256":
        calculate_sha256(
            REC_CHECKPOINT_PATH
        ),

    "NoiseCheckpointSHA256":
        calculate_sha256(
            NOISE_CHECKPOINT_PATH
        ),

    "NoisyRECCheckpointSHA256":
        calculate_sha256(
            NOISY_REC_CHECKPOINT_PATH
        ),

    "ModelMetricCheckpointSHA256":
        calculate_sha256(
            MODEL_METRIC_CHECKPOINT_PATH
        ),

    "SmokeConditions":
        SMOKE_CONDITIONS,

    "MLFits":
        EXPECTED_TOTAL_ML_FITS,

    "RankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRICS,

    "ProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "SmokeRankings":
        str(
            SMOKE_AGGREGATED_RANKINGS_PATH
        ),

    "SmokeRankingsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_RANKINGS_PATH
        ),

    "SmokeBuildMetrics":
        str(
            SMOKE_AGGREGATED_BUILD_METRICS_PATH
        ),

    "SmokeBuildMetricsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_BUILD_METRICS_PATH
        ),

    "SmokeProjectRuns":
        str(
            SMOKE_AGGREGATED_PROJECT_RUNS_PATH
        ),

    "SmokeProjectRunsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_PROJECT_RUNS_PATH
        ),

    "SmokeModelFits":
        str(
            SMOKE_AGGREGATED_MODEL_FITS_PATH
        ),

    "SmokeModelFitsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_MODEL_FITS_PATH
        ),

    "RandomInvariantAcrossNoise":
        random_invariant,

    "QTFAvgInvariantAcrossNoise":
        qtf_invariant,

    "EvaluationSHA256":
        evaluation_sha256_after,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "SmokeRuntimeSeconds":
        smoke_seconds,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    SMOKE_TEST_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "Conditions":
        len(
            condition_results
        ),

    "MLFits":
        len(
            all_smoke_model_fits
        ),

    "RankingRows":
        len(
            all_smoke_rankings
        ),

    "BuildMetricRows":
        len(
            all_smoke_build_metrics
        ),

    "ProjectRunRows":
        len(
            all_smoke_project_runs
        ),

    "RandomInvariantAcrossNoise":
        random_invariant,

    "QTFAvgInvariantAcrossNoise":
        qtf_invariant,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "Checkpoint":
        str(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 32. FINAL STATUS READBACK
# ------------------------------------------------------------

final_status = json.loads(
    STEP4B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_status.get(
        "Status"
    )
    != STEP4B_PASS_STATUS
):

    raise AssertionError(
        "Final Step 4B status differs."
    )


# ------------------------------------------------------------
# 33. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nSmoke-test model fits:")

display(
    all_smoke_model_fits
)


print("\nSmoke-test project-run results:")

display(
    all_smoke_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
)


print("\nBaseline invariance:")

display(
    baseline_invariance
)


print("\nCondition audit:")

display(
    smoke_condition_audit[
        [
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "RawRowsFlipped",
            "RealisedNoisePercent",
            "ChangedModelLabels",
            "ChangedDependentRECValues",
            "MLFits",
            "RankingRows",
            "BuildMetricRows",
            "ProjectRunRows",
            "ConditionSeconds",
        ]
    ]
)


# ------------------------------------------------------------
# 34. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 118)
print("=== PROJECT 9 CELL 8 / STEP 4B RESULT ===")
print("=" * 118)

print("\nProject:")

print(
    PROJECT_NAME
)


print("\nSmoke conditions:")

for condition in SMOKE_CONDITIONS:

    print(
        f"Noise {condition['NoisePercent']}% "
        f"/ seed {condition['RepetitionSeed']}"
    )


print("\nExperiment totals:")

print(
    "Conditions:",
    len(
        condition_results
    ),
)

print(
    "ML fits:",
    len(
        all_smoke_model_fits
    ),
    "/",
    EXPECTED_TOTAL_ML_FITS,
)

print(
    "Ranking rows:",
    len(
        all_smoke_rankings
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(
        all_smoke_build_metrics
    ),
    "/",
    EXPECTED_TOTAL_BUILD_METRICS,
)

print(
    "Project-run rows:",
    len(
        all_smoke_project_runs
    ),
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)


print("\nClean-condition validation:")

print(
    "0% flipped raw rows:",
    zero_condition_summary[
        "RawRowsFlipped"
    ],
)

print(
    "0% changed model labels:",
    zero_condition_summary[
        "ChangedModelLabels"
    ],
)

print(
    "0% changed dependent REC values:",
    zero_condition_summary[
        "ChangedDependentRECValues"
    ],
)


print("\nSevere-noise validation:")

print(
    "50% flipped raw rows:",
    fifty_condition_summary[
        "RawRowsFlipped"
    ],
)

print(
    "50% changed model labels:",
    fifty_condition_summary[
        "ChangedModelLabels"
    ],
)

print(
    "50% changed dependent REC values:",
    fifty_condition_summary[
        "ChangedDependentRECValues"
    ],
)


print("\nBaseline protocol:")

print(
    "Random invariant across noise:",
    random_invariant,
)

print(
    "QTF-Avg invariant across noise:",
    qtf_invariant,
)

print(
    "LatestFail invariant across noise:",
    latest_fail_same,
)


print("\nImmutability:")

print(
    "Evaluation unchanged:",
    evaluation_unchanged,
)

print(
    "Evaluation SHA-256:",
    evaluation_sha256_after,
)

print(
    "Source files unchanged:",
    source_files_unchanged,
)


print("\nRuntime:")

print(
    "Smoke-test seconds:",
    round(
        smoke_seconds,
        2,
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nSmoke-test checkpoint:")

print(
    SMOKE_TEST_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        SMOKE_TEST_CHECKPOINT_PATH
    ),
)


print("\nSaved outputs:")

for output_path in (
    smoke_output_paths
    + [
        SMOKE_REPORT_PATH,
        SMOKE_TEST_CHECKPOINT_PATH,
        STEP4B_STATUS_PATH,
    ]
):

    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP4B_PASS_STATUS,
)

print("=" * 118)

=== PROJECT 9 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===


----------------------------------------------------------------------------------------------------
Running smoke condition: noise_00__seed_01


AssertionError: 0% noisy model-training data does not equal the clean model-training data.

In [13]:
# ============================================================
# PROJECT 9 — CELL 8 / STEP 4B V2
# TWO-CONDITION END-TO-END EXPERIMENT SMOKE TEST
#
# PROJECT: camunda@camunda-bpm-platform
#
# V2 fixes:
# - replaces dtype-sensitive DataFrame.equals() with a
#   semantic value comparison for the 0% condition
# - standardises raw training/evaluation duration columns
# - validates column order and shape separately
# - preserves the complete frozen experimental protocol
#
# Smoke conditions:
#   0% noise, seed 1
#   50% noise, seed 1
#
# This cell does NOT:
# - run all 270 conditions
# - write final experiment outputs
# - modify the completion registry
# - modify Projects 1–8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from IPython.display import display

import gc
import hashlib
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = "camunda"


EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_9_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)

STEP4B_PASS_STATUS = (
    "PASS_PROJECT_9_TWO_CONDITION_END_TO_END_SMOKE_TEST_VALIDATED"
)


EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)


EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

EXPECTED_EVALUATION_PERIOD_BUILDS = 206
EXPECTED_SCORED_EVALUATION_BUILDS = 30
EXPECTED_EVALUATION_FAILURES = 665

EXPECTED_ACTIVE_PREDICTORS = 151

EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVALUATION_ROWS
    * EXPECTED_TECHNIQUES
)

EXPECTED_BUILD_METRICS_PER_CONDITION = (
    EXPECTED_SCORED_EVALUATION_BUILDS
    * EXPECTED_TECHNIQUES
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    EXPECTED_TECHNIQUES
)

EXPECTED_TOTAL_ML_FITS = 8

EXPECTED_TOTAL_RANKING_ROWS = (
    2
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRICS = (
    2
    * EXPECTED_BUILD_METRICS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    2
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)


SMOKE_CONDITIONS = [
    {
        "NoisePercent": 0,
        "RepetitionSeed": 1,
    },
    {
        "NoisePercent": 50,
        "RepetitionSeed": 1,
    },
]


RECENT_WINDOW = 6

COMPARISON_RTOL = 1e-9
COMPARISON_ATOL = 1e-9


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]


BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]


ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


# ------------------------------------------------------------
# 2. REC FEATURE PROTOCOL
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if len(
    VERDICT_DEPENDENT_REC_FEATURES
) != 13:

    raise AssertionError(
        "Expected 13 verdict-dependent REC features."
    )


if len(
    VERDICT_INDEPENDENT_REC_FEATURES
) != 6:

    raise AssertionError(
        "Expected six verdict-independent REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    & set(VERDICT_INDEPENDENT_REC_FEATURES)
):

    raise AssertionError(
        "REC dependency classes overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):

    raise AssertionError(
        "REC dependency classes do not cover all REC features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)


REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

NOISE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noise_plan_checkpoint.json"
)

NOISY_REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noisy_rec_engine_checkpoint.json"
)

MODEL_METRIC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_model_metric_checkpoint.json"
)

SMOKE_TEST_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_smoke_test_checkpoint.json"
)


PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)


STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_status.json"
)


SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)


NOISE_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_preflight"
)

FAILURE_SUBTYPE_DISTRIBUTION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_failure_subtype_distribution.csv"
)


SMOKE_TEST_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_test"
)

SMOKE_CONDITION_ROOT = (
    SMOKE_TEST_DIR
    / "conditions"
)

SMOKE_AGGREGATED_RANKINGS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_rankings.parquet"
)

SMOKE_AGGREGATED_BUILD_METRICS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_build_metrics.csv"
)

SMOKE_AGGREGATED_PROJECT_RUNS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_project_runs.csv"
)

SMOKE_AGGREGATED_MODEL_FITS_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_model_fits.csv"
)

SMOKE_CONDITION_AUDIT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_condition_audit.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_baseline_invariance.csv"
)

SMOKE_ZERO_NOISE_COMPARISON_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_zero_noise_semantic_comparison.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_TEST_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_report.json"
)


print("=" * 120)
print("=== PROJECT 9 CELL 8 / STEP 4B V2: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 120)


# ------------------------------------------------------------
# 4. FILE AND HASH HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(
            value
        ):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def dataframe_content_sha256(
    dataframe,
    include_index=True,
):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=include_index,
        categorize=True,
    ).to_numpy(
        dtype="<u8"
    )

    digest = hashlib.sha256()

    digest.update(
        json.dumps(
            list(
                dataframe.columns
            ),
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )

    digest.update(
        b"\0"
    )

    digest.update(
        row_hashes.tobytes(
            order="C"
        )
    )

    return digest.hexdigest()


def packed_boolean_sha256(
    mask,
):
    packed = np.packbits(
        np.asarray(
            mask,
            dtype=np.uint8,
        ),
        bitorder="little",
    )

    return hashlib.sha256(
        packed.tobytes()
    ).hexdigest()


def integer_array_sha256(
    values,
):
    values = np.asarray(
        values,
        dtype="<i4",
    )

    return hashlib.sha256(
        values.tobytes(
            order="C"
        )
    ).hexdigest()


# ------------------------------------------------------------
# 5. SEMANTIC DATAFRAME COMPARISON
# ------------------------------------------------------------

def semantic_dataframe_comparison(
    clean_dataframe,
    candidate_dataframe,
    rtol=COMPARISON_RTOL,
    atol=COMPARISON_ATOL,
):
    """
    Compare dataframe values while deliberately ignoring
    harmless pandas dtype differences.

    Numeric columns:
        compare with np.isclose.

    Non-numeric columns:
        compare their normalised string representations.

    Missing-value positions must match.
    """

    if list(
        clean_dataframe.columns
    ) != list(
        candidate_dataframe.columns
    ):

        raise AssertionError(
            "Semantic dataframe comparison received "
            "different column orders."
        )

    if len(
        clean_dataframe
    ) != len(
        candidate_dataframe
    ):

        raise AssertionError(
            "Semantic dataframe comparison received "
            "different row counts."
        )

    comparison_records = []

    total_mismatches = 0


    for column in clean_dataframe.columns:

        clean_series = (
            clean_dataframe[
                column
            ].reset_index(
                drop=True
            )
        )

        candidate_series = (
            candidate_dataframe[
                column
            ].reset_index(
                drop=True
            )
        )


        clean_numeric = pd.to_numeric(
            clean_series,
            errors="coerce",
        )

        candidate_numeric = pd.to_numeric(
            candidate_series,
            errors="coerce",
        )


        clean_nonmissing = (
            clean_series.notna()
        )

        candidate_nonmissing = (
            candidate_series.notna()
        )


        clean_numeric_complete = bool(
            clean_numeric[
                clean_nonmissing
            ].notna().all()
        )

        candidate_numeric_complete = bool(
            candidate_numeric[
                candidate_nonmissing
            ].notna().all()
        )


        compare_as_numeric = (
            clean_numeric_complete
            and candidate_numeric_complete
        )


        if compare_as_numeric:

            clean_values = (
                clean_numeric.to_numpy(
                    dtype=float
                )
            )

            candidate_values = (
                candidate_numeric.to_numpy(
                    dtype=float
                )
            )


            both_missing = (
                np.isnan(
                    clean_values
                )
                &
                np.isnan(
                    candidate_values
                )
            )


            both_present = (
                np.isfinite(
                    clean_values
                )
                &
                np.isfinite(
                    candidate_values
                )
            )


            finite_matches = np.zeros(
                len(
                    clean_values
                ),
                dtype=bool,
            )


            finite_matches[
                both_present
            ] = np.isclose(
                clean_values[
                    both_present
                ],
                candidate_values[
                    both_present
                ],
                rtol=rtol,
                atol=atol,
                equal_nan=False,
            )


            matches = (
                both_missing
                |
                finite_matches
            )


            mismatch_count = int(
                (
                    ~matches
                ).sum()
            )


            finite_difference_mask = (
                both_present
            )


            if finite_difference_mask.any():

                maximum_absolute_difference = float(
                    np.max(
                        np.abs(
                            clean_values[
                                finite_difference_mask
                            ]
                            - candidate_values[
                                finite_difference_mask
                            ]
                        )
                    )
                )

            else:

                maximum_absolute_difference = (
                    np.nan
                )


            comparison_type = (
                "NUMERIC_SEMANTIC"
            )


        else:

            clean_values = (
                clean_series.astype(
                    "string"
                )
                .fillna(
                    "<NA>"
                )
                .str.strip()
            )

            candidate_values = (
                candidate_series.astype(
                    "string"
                )
                .fillna(
                    "<NA>"
                )
                .str.strip()
            )


            matches = (
                clean_values
                == candidate_values
            ).to_numpy(
                dtype=bool
            )


            mismatch_count = int(
                (
                    ~matches
                ).sum()
            )


            maximum_absolute_difference = (
                np.nan
            )

            comparison_type = (
                "STRING_EXACT"
            )


        total_mismatches += (
            mismatch_count
        )


        comparison_records.append({
            "Column":
                column,

            "CleanDtype":
                str(
                    clean_series.dtype
                ),

            "CandidateDtype":
                str(
                    candidate_series.dtype
                ),

            "ComparisonType":
                comparison_type,

            "Rows":
                len(
                    clean_series
                ),

            "Mismatches":
                mismatch_count,

            "MaximumAbsoluteDifference":
                maximum_absolute_difference,

            "Pass":
                mismatch_count == 0,
        })


    return (
        total_mismatches,
        pd.DataFrame(
            comparison_records
        ),
    )


# ------------------------------------------------------------
# 6. IDENTIFIER AND SEED HELPERS
# ------------------------------------------------------------

def canonical_identifier(
    series,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(
        numeric.notna().mean()
    ) >= 0.95:

        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:

            return (
                rounded
                .astype(
                    "Int64"
                )
                .astype(
                    str
                )
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (
        2 ** 32
    )


# ------------------------------------------------------------
# 7. NOISE HELPERS
# ------------------------------------------------------------

def create_seed_random_streams(
    number_of_rows,
    repetition_seed,
    failure_subtypes,
    subtype_probabilities,
):
    flip_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "flip_mask",
    )

    subtype_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "failure_subtype",
    )

    flip_rng = np.random.default_rng(
        flip_stream_seed
    )

    subtype_rng = np.random.default_rng(
        subtype_stream_seed
    )

    return {
        "FlipStreamSeed":
            int(
                flip_stream_seed
            ),

        "SubtypeStreamSeed":
            int(
                subtype_stream_seed
            ),

        "FlipUniforms":
            flip_rng.random(
                number_of_rows
            ),

        "SampledFailureSubtypes":
            subtype_rng.choice(
                failure_subtypes,
                size=number_of_rows,
                replace=True,
                p=subtype_probabilities,
            ).astype(
                np.int32
            ),
    }


def construct_condition_arrays(
    clean_verdicts,
    row_uniforms,
    sampled_failure_subtypes,
    noise_percent,
):
    flip_mask = (
        row_uniforms
        < float(
            noise_percent
        ) / 100.0
    )

    pass_to_failure_mask = (
        flip_mask
        & (
            clean_verdicts
            == 0
        )
    )

    failure_to_pass_mask = (
        flip_mask
        & (
            clean_verdicts
            != 0
        )
    )

    noisy_verdicts = (
        clean_verdicts.copy()
    )

    noisy_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_verdicts[
        failure_to_pass_mask
    ] = 0

    return {
        "FlipMask":
            flip_mask,

        "PassToFailureMask":
            pass_to_failure_mask,

        "FailureToPassMask":
            failure_to_pass_mask,

        "NoisyVerdicts":
            noisy_verdicts,
    }


# ------------------------------------------------------------
# 8. APFD AND APFDC HELPERS
# ------------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = int(
        len(
            failures
        )
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):

        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    value = (
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )

    return float(
        value
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):

        raise ValueError(
            "Failures and durations differ in length."
        )

    if (
        len(
            failures
        ) == 0
        or failures.sum() == 0
    ):

        return np.nan

    if not np.isfinite(
        durations
    ).all():

        raise ValueError(
            "Durations contain non-finite values."
        )

    if (
        durations < 0
    ).any():

        raise ValueError(
            "Durations contain negative values."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:

        return np.nan

    cumulative_before = np.concatenate([
        np.asarray(
            [
                0.0,
            ]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    value = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

    return float(
        value
    )


# ------------------------------------------------------------
# 9. VALIDATE CHECKPOINTS
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_CHECKPOINT_PATH,
    NOISY_REC_CHECKPOINT_PATH,
    MODEL_METRIC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
    STEP4A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
]


missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 4B V2 inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_checkpoint = json.loads(
    SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

rec_checkpoint = json.loads(
    REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noise_checkpoint = json.loads(
    NOISE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noisy_rec_checkpoint = json.loads(
    NOISY_REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

model_metric_checkpoint = json.loads(
    MODEL_METRIC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step2b_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3a_status = json.loads(
    STEP3A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3b_status = json.loads(
    STEP3B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step4a_status = json.loads(
    STEP4A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


expected_statuses = [
    (
        "Step 2B",
        step2b_status.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Step 4A",
        step4a_status.get(
            "Status"
        ),
        EXPECTED_STEP4A_STATUS,
    ),
]


for step_name, actual, expected in (
    expected_statuses
):

    if actual != expected:

        raise AssertionError(
            f"{step_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


checkpoint_statuses = [
    (
        "REC checkpoint",
        rec_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Noise checkpoint",
        noise_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Noisy REC checkpoint",
        noisy_rec_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Model/metric checkpoint",
        model_metric_checkpoint.get(
            "Status"
        ),
        EXPECTED_STEP4A_STATUS,
    ),
]


for checkpoint_name, actual, expected in (
    checkpoint_statuses
):

    if actual != expected:

        raise AssertionError(
            f"{checkpoint_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):

    raise AssertionError(
        "Source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 10. VALIDATE REGISTRY
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 8
    or set(
        project_numbers
    ) != set(
        range(
            1,
            9,
        )
    )
):

    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )


if project_numbers.eq(
    PROJECT_NUMBER
).any():

    raise AssertionError(
        "Project 9 is already present in the registry."
    )


# ------------------------------------------------------------
# 11. RESOLVE FROZEN INPUT PATHS
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingHistory"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingData"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationData"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "NoiseConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

anchor_offsets_path = Path(
    rec_checkpoint[
        "CleanRECAnchorOffsets"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)

active_predictors_path = Path(
    model_metric_checkpoint[
        "ActivePredictors"
    ]
)

model_configuration_path = Path(
    model_metric_checkpoint[
        "ModelConfiguration"
    ]
)

raw_evaluation_path = Path(
    model_metric_checkpoint[
        "RawEvaluationHistory"
    ]
)

evaluation_build_table_path = Path(
    model_metric_checkpoint[
        "EvaluationBuildTable"
    ]
)

evaluation_metric_cohort_path = Path(
    model_metric_checkpoint[
        "EvaluationMetricCohort"
    ]
)


frozen_input_hashes = {
    raw_training_path:
        noise_checkpoint[
            "RawTrainingHistorySHA256"
        ],

    model_training_path:
        noise_checkpoint[
            "ModelTrainingDataSHA256"
        ],

    model_evaluation_path:
        noise_checkpoint[
            "ModelEvaluationDataSHA256"
        ],

    condition_plan_path:
        noise_checkpoint[
            "NoiseConditionPlanSHA256"
        ],

    clean_direct_rec_path:
        rec_checkpoint[
            "CleanRECReconstructedSHA256"
        ],

    anchor_offsets_path:
        rec_checkpoint[
            "CleanRECAnchorOffsetsSHA256"
        ],

    build_entity_map_path:
        rec_checkpoint[
            "BuildEntityMapSHA256"
        ],

    active_predictors_path:
        model_metric_checkpoint[
            "ActivePredictorsSHA256"
        ],

    model_configuration_path:
        model_metric_checkpoint[
            "ModelConfigurationSHA256"
        ],

    raw_evaluation_path:
        model_metric_checkpoint[
            "RawEvaluationHistorySHA256"
        ],

    evaluation_build_table_path:
        model_metric_checkpoint[
            "EvaluationBuildTableSHA256"
        ],

    evaluation_metric_cohort_path:
        model_metric_checkpoint[
            "EvaluationMetricCohortSHA256"
        ],
}


for path, expected_hash in (
    frozen_input_hashes.items()
):

    if not path.exists():

        raise FileNotFoundError(
            "Frozen smoke-test input is missing:\n"
            f"{path}"
        )

    actual_hash = calculate_sha256(
        path
    )

    if actual_hash != expected_hash:

        raise AssertionError(
            "Frozen smoke-test input hash differs.\n"
            f"Path: {path}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )


source_paths = {
    relative_name:
        Path(
            metadata[
                "RuntimePath"
            ]
        )
    for relative_name, metadata
    in selection_checkpoint[
        "SourceFiles"
    ].items()
}


source_hashes_before = {}


for relative_name, path in (
    source_paths.items()
):

    expected_hash = (
        selection_checkpoint[
            "SourceFiles"
        ][
            relative_name
        ][
            "SHA256"
        ]
    )

    actual_hash = calculate_sha256(
        path
    )

    if actual_hash != expected_hash:

        raise AssertionError(
            "Frozen source-file hash differs.\n"
            f"File: {relative_name}"
        )

    source_hashes_before[
        relative_name
    ] = actual_hash


evaluation_sha256_before = calculate_sha256(
    model_evaluation_path
)


# ------------------------------------------------------------
# 12. LOAD FROZEN CONFIGURATION
# ------------------------------------------------------------

active_predictor_payload = json.loads(
    active_predictors_path.read_text(
        encoding="utf-8"
    )
)

model_configuration = json.loads(
    model_configuration_path.read_text(
        encoding="utf-8"
    )
)


ACTIVE_FEATURE_COLUMNS = (
    active_predictor_payload[
        "ActivePredictors"
    ]
)


ZERO_VARIANCE_FEATURES = (
    active_predictor_payload[
        "ZeroVariancePredictors"
    ]
)


MODEL_CONFIG = (
    model_configuration[
        "Models"
    ]
)


if len(
    ACTIVE_FEATURE_COLUMNS
) != EXPECTED_ACTIVE_PREDICTORS:

    raise AssertionError(
        "Active predictor count differs.\n"
        f"Expected: {EXPECTED_ACTIVE_PREDICTORS}\n"
        f"Actual: {len(ACTIVE_FEATURE_COLUMNS)}"
    )


if len(
    ZERO_VARIANCE_FEATURES
) != 0:

    raise AssertionError(
        "Step 4A recorded unexpected zero-variance predictors."
    )


resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)


exe_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)


dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


# ------------------------------------------------------------
# 13. LOAD FROZEN DATA
# ------------------------------------------------------------

raw_training = (
    pd.read_parquet(
        raw_training_path
    )
    .sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

clean_model_training = (
    pd.read_parquet(
        model_training_path
    )
    .reset_index(
        drop=True
    )
)

clean_model_evaluation = (
    pd.read_parquet(
        model_evaluation_path
    )
    .reset_index(
        drop=True
    )
)

condition_plan = pd.read_csv(
    condition_plan_path,
    low_memory=False,
)

failure_subtype_distribution = pd.read_csv(
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
    low_memory=False,
)

clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)

anchor_offsets = pd.read_parquet(
    anchor_offsets_path
)

build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

evaluation_build_table = pd.read_csv(
    evaluation_build_table_path,
    low_memory=False,
)

evaluation_metric_cohort = pd.read_parquet(
    evaluation_metric_cohort_path
)


if len(
    raw_training
) != EXPECTED_RAW_TRAINING_ROWS:

    raise AssertionError(
        "Raw training row count differs."
    )


if len(
    clean_model_training
) != EXPECTED_MODEL_TRAINING_ROWS:

    raise AssertionError(
        "Model training row count differs."
    )


if len(
    clean_model_evaluation
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Model evaluation row count differs."
    )


if len(
    evaluation_build_table
) != EXPECTED_EVALUATION_PERIOD_BUILDS:

    raise AssertionError(
        "Evaluation-period build count differs."
    )


if len(
    evaluation_metric_cohort
) != EXPECTED_MODEL_EVALUATION_ROWS:

    raise AssertionError(
        "Evaluation metric-cohort row count differs."
    )


# Standardise raw training duration.
if "Duration" in raw_training.columns:

    raw_training[
        "Duration"
    ] = pd.to_numeric(
        raw_training[
            "Duration"
        ],
        errors="raise",
    ).astype(float)

elif exe_duration_column in raw_training.columns:

    raw_training[
        "Duration"
    ] = pd.to_numeric(
        raw_training[
            exe_duration_column
        ],
        errors="raise",
    ).astype(float)

else:

    raise RuntimeError(
        "Raw training duration column was not found.\n"
        f"Expected source column: {exe_duration_column}\n"
        f"Available columns: {list(raw_training.columns)}"
    )


# Standardise raw evaluation duration.
if "Duration" in raw_evaluation.columns:

    raw_evaluation[
        "Duration"
    ] = pd.to_numeric(
        raw_evaluation[
            "Duration"
        ],
        errors="raise",
    ).astype(float)

elif exe_duration_column in raw_evaluation.columns:

    raw_evaluation[
        "Duration"
    ] = pd.to_numeric(
        raw_evaluation[
            exe_duration_column
        ],
        errors="raise",
    ).astype(float)

else:

    raise RuntimeError(
        "Raw evaluation duration column was not found.\n"
        f"Expected source column: {exe_duration_column}\n"
        f"Available columns: {list(raw_evaluation.columns)}"
    )


if not np.isfinite(
    raw_training[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():

    raise AssertionError(
        "Raw training durations contain non-finite values."
    )


if not np.isfinite(
    raw_evaluation[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():

    raise AssertionError(
        "Raw evaluation durations contain non-finite values."
    )


if raw_training[
    "Duration"
].lt(0).any():

    raise AssertionError(
        "Raw training durations contain negative values."
    )


if raw_evaluation[
    "Duration"
].lt(0).any():

    raise AssertionError(
        "Raw evaluation durations contain negative values."
    )


# ------------------------------------------------------------
# 14. PREPARE CANONICAL KEYS
# ------------------------------------------------------------

raw_training[
    "BuildKey"
] = raw_training[
    "BuildKey"
].astype(str)

raw_training[
    "TestKey"
] = raw_training[
    "TestKey"
].astype(str)

raw_training[
    "BuildOrder"
] = pd.to_numeric(
    raw_training[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


expected_noise_row_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int64
    ),
    expected_noise_row_ids,
):

    raise AssertionError(
        "Raw training NoiseRowID order differs."
    )


clean_raw_verdicts = pd.to_numeric(
    raw_training[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


training_key_frame = pd.DataFrame({
    "_ModelRowOrder":
        np.arange(
            EXPECTED_MODEL_TRAINING_ROWS,
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            clean_model_training[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            clean_model_training[
                dataset_test_column
            ]
        ),
})


evaluation_key_frame = pd.DataFrame({
    "_ModelRowOrder":
        np.arange(
            EXPECTED_MODEL_EVALUATION_ROWS,
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            clean_model_evaluation[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            clean_model_evaluation[
                dataset_test_column
            ]
        ),
})


if training_key_frame.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():

    raise AssertionError(
        "Model training Build/Test keys are not unique."
    )


if evaluation_key_frame.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():

    raise AssertionError(
        "Model evaluation Build/Test keys are not unique."
    )


raw_key_manifest = (
    raw_training[
        [
            "NoiseRowID",
            "BuildKey",
            "TestKey",
            "CleanVerdict",
        ]
    ]
    .copy()
)


if raw_key_manifest.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():

    raise AssertionError(
        "Raw training Build/Test keys are not unique."
    )


model_raw_alignment = (
    training_key_frame.merge(
        raw_key_manifest,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_raw_rows = int(
    model_raw_alignment[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


if missing_model_raw_rows:

    raise AssertionError(
        "Model-training rows are missing raw verdict rows."
    )


model_noise_row_ids = (
    model_raw_alignment[
        "NoiseRowID"
    ].to_numpy(
        dtype=np.int64
    )
)


clean_model_verdicts = pd.to_numeric(
    clean_model_training[
        dataset_verdict_column
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


aligned_clean_raw_verdicts = pd.to_numeric(
    model_raw_alignment[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


clean_raw_model_label_mismatches = int(
    (
        clean_model_verdicts
        != aligned_clean_raw_verdicts
    ).sum()
)


if clean_raw_model_label_mismatches:

    raise AssertionError(
        "Clean raw/model training verdicts differ."
    )


# ------------------------------------------------------------
# 15. ALIGN CLEAN REC RECONSTRUCTION AND OFFSETS
# ------------------------------------------------------------

clean_direct_rec[
    "BuildKey"
] = clean_direct_rec[
    "BuildKey"
].astype(str)

clean_direct_rec[
    "TestKey"
] = clean_direct_rec[
    "TestKey"
].astype(str)

anchor_offsets[
    "BuildKey"
] = anchor_offsets[
    "BuildKey"
].astype(str)

anchor_offsets[
    "TestKey"
] = anchor_offsets[
    "TestKey"
].astype(str)


clean_direct_training = (
    training_key_frame.merge(
        clean_direct_rec[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if clean_direct_training[
    "_merge"
].ne(
    "both"
).any():

    raise AssertionError(
        "Clean direct REC rows are missing."
    )


anchor_training = (
    training_key_frame.merge(
        anchor_offsets[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if anchor_training[
    "_merge"
].ne(
    "both"
).any():

    raise AssertionError(
        "Clean REC anchor-offset rows are missing."
    )


clean_direct_dependent_matrix = (
    clean_direct_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


anchor_dependent_matrix = (
    anchor_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(
        dtype=float
    )
)


original_dependent_matrix = (
    clean_model_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    clean_direct_dependent_matrix
    + anchor_dependent_matrix,
    original_dependent_matrix,
    rtol=COMPARISON_RTOL,
    atol=COMPARISON_ATOL,
    equal_nan=False,
).all():

    raise AssertionError(
        "Frozen clean direct REC plus anchor does not "
        "reproduce clean training REC values."
    )


requested_pairs_frame = (
    training_key_frame[
        [
            "BuildKey",
            "TestKey",
        ]
    ]
    .copy()
)


requested_pairs_by_test = {
    str(
        test_key
    ):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for test_key, group
    in requested_pairs_frame.groupby(
        "TestKey",
        sort=False,
    )
}


# ------------------------------------------------------------
# 16. PREPARE BUILD/ENTITY STRUCTURE
# ------------------------------------------------------------

build_entity_map[
    "BuildKey"
] = build_entity_map[
    "BuildKey"
].astype(str)

build_entity_map[
    "EntityId"
] = pd.to_numeric(
    build_entity_map[
        "EntityId"
    ],
    errors="raise",
).astype(int)


changed_entities_by_build = {
    str(
        build_key
    ):
        set(
            group[
                "EntityId"
            ].astype(int)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


entity_changed_builds = {
    int(
        entity_id
    ):
        set(
            group[
                "BuildKey"
            ].astype(str)
        )
    for entity_id, group
    in build_entity_map.groupby(
        "EntityId",
        sort=False,
    )
}


# ------------------------------------------------------------
# 17. FAILURE SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

failure_subtypes = pd.to_numeric(
    failure_subtype_distribution[
        "FailureSubtype"
    ],
    errors="raise",
).to_numpy(
    dtype=np.int32
)


subtype_probabilities = pd.to_numeric(
    failure_subtype_distribution[
        "SamplingProbability"
    ],
    errors="raise",
).to_numpy(
    dtype=float
)


if not np.isclose(
    subtype_probabilities.sum(),
    1.0,
    rtol=0,
    atol=1e-12,
):

    raise AssertionError(
        "Failure subtype probabilities do not sum to one."
    )


# ------------------------------------------------------------
# 18. PREPARE CLEAN EVALUATION DATA
# ------------------------------------------------------------

if "_ModelRowOrder" not in (
    evaluation_metric_cohort.columns
):

    raise AssertionError(
        "Evaluation metric cohort lacks _ModelRowOrder."
    )


evaluation_metric_cohort = (
    evaluation_metric_cohort.sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


evaluation_metric_cohort[
    "BuildKey"
] = evaluation_metric_cohort[
    "BuildKey"
].astype(str)

evaluation_metric_cohort[
    "TestKey"
] = evaluation_metric_cohort[
    "TestKey"
].astype(str)


evaluation_key_match = bool(
    evaluation_key_frame[
        [
            "BuildKey",
            "TestKey",
        ]
    ]
    .reset_index(
        drop=True
    )
    .equals(
        evaluation_metric_cohort[
            [
                "BuildKey",
                "TestKey",
            ]
        ]
        .reset_index(
            drop=True
        )
    )
)


if not evaluation_key_match:

    raise AssertionError(
        "Model evaluation and metric-cohort row order differ."
    )


evaluation_rank_data = pd.DataFrame({
    "Build":
        evaluation_metric_cohort[
            dataset_build_column
        ].to_numpy(),

    "Test":
        evaluation_metric_cohort[
            dataset_test_column
        ].to_numpy(),

    "Verdict":
        pd.to_numeric(
            evaluation_metric_cohort[
                dataset_verdict_column
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.int32
        ),

    "Duration":
        pd.to_numeric(
            evaluation_metric_cohort[
                "Duration"
            ],
            errors="raise",
        ).to_numpy(
            dtype=float
        ),

    "BuildOrder":
        pd.to_numeric(
            evaluation_metric_cohort[
                "BuildOrder"
            ],
            errors="raise",
        ).to_numpy(
            dtype=int
        ),

    "BuildKey":
        evaluation_metric_cohort[
            "BuildKey"
        ].astype(str).to_numpy(),

    "TestKey":
        evaluation_metric_cohort[
            "TestKey"
        ].astype(str).to_numpy(),

    "ActualFailure":
        pd.to_numeric(
            evaluation_metric_cohort[
                "ActualFailure"
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.int32
        ),
})


if int(
    evaluation_rank_data[
        "ActualFailure"
    ].sum()
) != EXPECTED_EVALUATION_FAILURES:

    raise AssertionError(
        "Evaluation failure count differs."
    )


if evaluation_rank_data[
    "BuildKey"
].nunique() != EXPECTED_SCORED_EVALUATION_BUILDS:

    raise AssertionError(
        "Scored evaluation-build count differs."
    )


raw_evaluation[
    "BuildKey"
] = raw_evaluation[
    "BuildKey"
].astype(str)

raw_evaluation[
    "TestKey"
] = raw_evaluation[
    "TestKey"
].astype(str)

raw_evaluation[
    "BuildOrder"
] = pd.to_numeric(
    raw_evaluation[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)

raw_evaluation[
    "CleanVerdict"
] = pd.to_numeric(
    raw_evaluation[
        "CleanVerdict"
    ],
    errors="raise",
).astype(np.int32)


evaluation_build_table[
    "BuildKey"
] = evaluation_build_table[
    "BuildKey"
].astype(str)

evaluation_build_table[
    "BuildOrder"
] = pd.to_numeric(
    evaluation_build_table[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


# ------------------------------------------------------------
# 19. NOISY REC RECONSTRUCTION ENGINE
# ------------------------------------------------------------

def calculate_max_test_file_rate(
    target_builds,
    current_changed_entities,
):
    if len(
        target_builds
    ) == 0:

        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:

        changed_builds = (
            entity_changed_builds.get(
                int(
                    entity_id
                ),
                set(),
            )
        )

        overlap_count = len(
            changed_builds.intersection(
                target_builds
            )
        )

        if overlap_count > maximum_frequency:

            maximum_frequency = (
                overlap_count
            )

    if maximum_frequency == 0:

        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_noisy_dependent_rec(
    noisy_history,
):
    reconstructed_records = []

    for test_key, test_history in (
        noisy_history.groupby(
            "TestKey",
            sort=False,
        )
    ):

        test_key = str(
            test_key
        )

        requested_builds = (
            requested_pairs_by_test.get(
                test_key
            )
        )

        if not requested_builds:
            continue

        test_history = (
            test_history.sort_values(
                "BuildOrder",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        prior_count = 0

        prior_failure_count = 0
        prior_assertion_count = 0
        prior_exception_count = 0
        prior_transition_count = 0

        last_failure_position = None
        last_transition_position = None

        previous_verdict = None

        recent_history = deque(
            maxlen=RECENT_WINDOW
        )

        failure_builds = set()
        transition_builds = set()

        for row in test_history.itertuples(
            index=False
        ):

            current_build = str(
                row.BuildKey
            )

            current_verdict = int(
                row.NoisyVerdict
            )

            if current_build in requested_builds:

                record = {
                    "BuildKey":
                        current_build,

                    "TestKey":
                        test_key,
                }

                if prior_count == 0:

                    for feature in (
                        VERDICT_DEPENDENT_REC_FEATURES
                    ):

                        record[
                            feature
                        ] = -1.0

                else:

                    if last_failure_position is None:

                        last_failure_age = -1.0

                    else:

                        last_failure_age = float(
                            prior_count
                            - 1
                            - last_failure_position
                        )

                    if last_transition_position is None:

                        last_transition_age = -1.0

                    else:

                        last_transition_age = float(
                            prior_count
                            - 1
                            - last_transition_position
                        )

                    recent_rows = list(
                        recent_history
                    )

                    recent_length = len(
                        recent_rows
                    )

                    if recent_length == 0:

                        raise AssertionError(
                            "Recent verdict history is unexpectedly empty."
                        )

                    recent_verdicts = np.asarray(
                        [
                            item[
                                "Verdict"
                            ]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )

                    recent_transitions = np.asarray(
                        [
                            item[
                                "Transition"
                            ]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )

                    current_changed_entities = (
                        changed_entities_by_build.get(
                            current_build,
                            set(),
                        )
                    )

                    record.update({
                        "REC_LastFailureAge":
                            last_failure_age,

                        "REC_LastTransitionAge":
                            last_transition_age,

                        "REC_RecentFailRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts
                                    != 0
                                )
                                / recent_length
                            ),

                        "REC_RecentAssertRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts
                                    == 2
                                )
                                / recent_length
                            ),

                        "REC_RecentExcRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts
                                    == 1
                                )
                                / recent_length
                            ),

                        "REC_RecentTransitionRate":
                            float(
                                np.count_nonzero(
                                    recent_transitions
                                    == 1
                                )
                                / recent_length
                            ),

                        "REC_TotalFailRate":
                            float(
                                prior_failure_count
                                / prior_count
                            ),

                        "REC_TotalAssertRate":
                            float(
                                prior_assertion_count
                                / prior_count
                            ),

                        "REC_TotalExcRate":
                            float(
                                prior_exception_count
                                / prior_count
                            ),

                        "REC_TotalTransitionRate":
                            float(
                                prior_transition_count
                                / prior_count
                            ),

                        "REC_LastVerdict":
                            float(
                                previous_verdict
                            ),

                        "REC_MaxTestFileFailRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    failure_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),

                        "REC_MaxTestFileTransitionRate":
                            calculate_max_test_file_rate(
                                target_builds=(
                                    transition_builds
                                ),

                                current_changed_entities=(
                                    current_changed_entities
                                ),
                            ),
                    })

                reconstructed_records.append(
                    record
                )

            current_transition = (
                0
                if previous_verdict is None
                else int(
                    current_verdict
                    != previous_verdict
                )
            )

            if current_verdict != 0:

                prior_failure_count += 1

                last_failure_position = (
                    prior_count
                )

                failure_builds.add(
                    current_build
                )

            if current_verdict == 2:

                prior_assertion_count += 1

            if current_verdict == 1:

                prior_exception_count += 1

            if current_transition == 1:

                prior_transition_count += 1

                last_transition_position = (
                    prior_count
                )

                transition_builds.add(
                    current_build
                )

            recent_history.append({
                "Verdict":
                    current_verdict,

                "Transition":
                    current_transition,
            })

            previous_verdict = (
                current_verdict
            )

            prior_count += 1

    return pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ),
    )


# ------------------------------------------------------------
# 20. ML HELPERS
# ------------------------------------------------------------

def prepare_ml_matrices(
    noisy_training_data,
):
    X_train = (
        noisy_training_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    X_evaluation = (
        clean_model_evaluation[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    training_medians = (
        X_train.median(
            axis=0
        )
        .fillna(
            0.0
        )
    )

    X_train = (
        X_train.fillna(
            training_medians
        )
        .astype(float)
    )

    X_evaluation = (
        X_evaluation.fillna(
            training_medians
        )
        .astype(float)
    )

    y_train = (
        pd.to_numeric(
            noisy_training_data[
                dataset_verdict_column
            ],
            errors="raise",
        )
        .ne(0)
        .astype(int)
    )

    if y_train.nunique() != 2:

        raise AssertionError(
            "Noisy model-training labels do not contain both classes."
        )

    if not np.isfinite(
        X_train.to_numpy(
            dtype=float
        )
    ).all():

        raise AssertionError(
            "Noisy training matrix contains non-finite values."
        )

    if not np.isfinite(
        X_evaluation.to_numpy(
            dtype=float
        )
    ).all():

        raise AssertionError(
            "Evaluation matrix contains non-finite values."
        )

    return (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    )


def create_ml_models(
    repetition_seed,
):
    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )

    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )

    lgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )

    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                var_smoothing=(
                    MODEL_CONFIG[
                        "NaiveBayes"
                    ][
                        "var_smoothing"
                    ]
                )
            ),
    }


def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = (
        fitted_model.predict_proba(
            feature_matrix
        )
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.where(
        classes == 1
    )[0]

    if len(
        positive_positions
    ) != 1:

        raise AssertionError(
            "Could not identify fitted failure class 1."
        )

    output = probabilities[
        :,
        positive_positions[
            0
        ],
    ]

    if not np.isfinite(
        output
    ).all():

        raise AssertionError(
            "Predicted probabilities contain non-finite values."
        )

    if not (
        (
            output >= 0.0
        )
        &
        (
            output <= 1.0
        )
    ).all():

        raise AssertionError(
            "Predicted probabilities fall outside [0, 1]."
        )

    return output


# ------------------------------------------------------------
# 21. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    ranked = (
        build_rows[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                "BuildOrder",
                "BuildKey",
                "TestKey",
                "ActualFailure",
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    scores = np.asarray(
        scores,
        dtype=float,
    )

    if len(
        scores
    ) != len(
        ranked
    ):

        raise ValueError(
            "Ranking score count differs from row count."
        )

    if np.isnan(
        scores
    ).any():

        raise ValueError(
            "Ranking scores contain NaN values."
        )

    if np.isneginf(
        scores
    ).any():

        raise ValueError(
            "Ranking scores contain negative infinity."
        )

    if (
        np.isposinf(
            scores
        ).any()
        and technique != "QTF-Avg"
    ):

        raise ValueError(
            "Only QTF-Avg may contain positive-infinity scores."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = scores

    numeric_tests = pd.to_numeric(
        ranked[
            "Test"
        ],
        errors="coerce",
    )

    if numeric_tests.notna().all():

        ranked[
            "_TestTieKey"
        ] = numeric_tests

    else:

        ranked[
            "_TestTieKey"
        ] = ranked[
            "Test"
        ].astype(str)

    if score_direction == "descending":

        ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":

        ascending = [
            True,
            True,
        ]

    else:

        raise ValueError(
            "Invalid score direction."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "_TestTieKey",
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .drop(
            columns=[
                "_TestTieKey",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(
            ranked
        ) + 1,
    )

    return ranked


def create_ml_rankings(
    technique,
    failure_probabilities,
):
    scored = (
        evaluation_rank_data.copy()
    )

    scored[
        "_FailureProbability"
    ] = np.asarray(
        failure_probabilities,
        dtype=float,
    )

    ranking_frames = []

    for _, build_rows in scored.groupby(
        "BuildKey",
        sort=False,
    ):

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=(
                    build_rows[
                        "_FailureProbability"
                    ].to_numpy(
                        dtype=float
                    )
                ),
                technique=technique,
                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


def create_random_rankings(
    repetition_seed,
):
    ranking_frames = []

    for build_key, build_rows in (
        evaluation_rank_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):

        random_rng = np.random.default_rng(
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                (
                    "Random_baseline_build_"
                    f"{build_key}"
                ),
            )
        )

        scores = random_rng.random(
            len(
                build_rows
            )
        )

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=scores,
                technique="Random",
                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


# ------------------------------------------------------------
# 22. HISTORY-BASED BASELINES
# ------------------------------------------------------------

def create_history_baseline_rankings(
    noisy_raw_training,
):
    latest_failure_order = {}

    noisy_training_sorted = (
        noisy_raw_training.sort_values(
            [
                "BuildOrder",
                "TestKey",
            ],
            kind="mergesort",
        )
    )

    for row in noisy_training_sorted.itertuples(
        index=False
    ):

        test_key = str(
            row.TestKey
        )

        if int(
            row.NoisyVerdict
        ) != 0:

            latest_failure_order[
                test_key
            ] = int(
                row.BuildOrder
            )


    duration_sum = {}
    duration_count = {}


    clean_training_sorted = (
        raw_training.sort_values(
            [
                "BuildOrder",
                "TestKey",
            ],
            kind="mergesort",
        )
    )


    for row in clean_training_sorted.itertuples(
        index=False
    ):

        test_key = str(
            row.TestKey
        )

        duration = float(
            row.Duration
        )

        duration_sum[
            test_key
        ] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )

        duration_count[
            test_key
        ] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )


    raw_evaluation_by_build = {
        str(
            build_key
        ):
            rows.copy()
        for build_key, rows
        in raw_evaluation.groupby(
            "BuildKey",
            sort=False,
        )
    }


    model_evaluation_by_build = {
        str(
            build_key
        ):
            rows.copy()
        for build_key, rows
        in evaluation_rank_data.groupby(
            "BuildKey",
            sort=False,
        )
    }


    target_build_keys = set(
        model_evaluation_by_build
    )

    latest_fail_frames = []
    qtf_frames = []

    qtf_unseen_scores = 0


    ordered_evaluation_builds = (
        evaluation_build_table.sort_values(
            "BuildOrder",
            kind="mergesort",
        )
    )


    for build_row in (
        ordered_evaluation_builds.itertuples(
            index=False
        )
    ):

        build_key = str(
            build_row.BuildKey
        )


        if build_key in target_build_keys:

            build_tests = (
                model_evaluation_by_build[
                    build_key
                ].copy()
            )

            latest_scores = []
            qtf_scores = []


            for test_key in build_tests[
                "TestKey"
            ].astype(str):

                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )


                count = duration_count.get(
                    test_key,
                    0,
                )


                if count > 0:

                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / count
                    )

                else:

                    average_duration = (
                        np.inf
                    )

                    qtf_unseen_scores += 1


                qtf_scores.append(
                    float(
                        average_duration
                    )
                )


            latest_fail_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )


            qtf_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )


        current_raw_rows = (
            raw_evaluation_by_build.get(
                build_key
            )
        )


        if current_raw_rows is None:
            continue


        current_raw_rows = (
            current_raw_rows.sort_values(
                "TestKey",
                kind="mergesort",
            )
        )


        for execution_row in (
            current_raw_rows.itertuples(
                index=False
            )
        ):

            test_key = str(
                execution_row.TestKey
            )


            if int(
                execution_row.CleanVerdict
            ) != 0:

                latest_failure_order[
                    test_key
                ] = int(
                    execution_row.BuildOrder
                )


            duration = float(
                execution_row.Duration
            )


            duration_sum[
                test_key
            ] = (
                duration_sum.get(
                    test_key,
                    0.0,
                )
                + duration
            )


            duration_count[
                test_key
            ] = (
                duration_count.get(
                    test_key,
                    0,
                )
                + 1
            )


    latest_fail_rankings = pd.concat(
        latest_fail_frames,
        ignore_index=True,
    )

    qtf_rankings = pd.concat(
        qtf_frames,
        ignore_index=True,
    )


    return (
        latest_fail_rankings,
        qtf_rankings,
        qtf_unseen_scores,
    )


# ------------------------------------------------------------
# 23. METRIC AND OUTPUT VALIDATION HELPERS
# ------------------------------------------------------------

def calculate_build_metrics(
    rankings,
    noise_percent,
    repetition_seed,
):
    metric_records = []

    for (
        technique,
        build_key,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):

        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        actual_failures = (
            ranked_build[
                "ActualFailure"
            ].to_numpy(
                dtype=int
            )
        )

        durations = (
            ranked_build[
                "Duration"
            ].to_numpy(
                dtype=float
            )
        )

        metric_records.append({
            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildKey":
                str(
                    build_key
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "BuildOrder"
                    ].iloc[0]
                ),

            "NumberOfTests":
                int(
                    len(
                        ranked_build
                    )
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "APFD":
                calculate_apfd(
                    actual_failures
                ),

            "APFDc":
                calculate_apfdc(
                    actual_failures,
                    durations,
                ),
        })

    return pd.DataFrame(
        metric_records
    )


def aggregate_project_run(
    build_metrics,
):
    return (
        build_metrics.groupby(
            [
                "Project",
                "ProjectSlug",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            ScoredEvaluationBuilds=(
                "BuildKey",
                "nunique",
            ),

            EvaluationRows=(
                "NumberOfTests",
                "sum",
            ),

            EvaluationFailures=(
                "NumberOfFailures",
                "sum",
            ),

            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SDAPFD=(
                "APFD",
                "std",
            ),

            SDAPFDc=(
                "APFDc",
                "std",
            ),
        )
    )


def validate_condition_outputs(
    rankings,
    build_metrics,
    project_run,
    fit_log,
):
    if set(
        rankings[
            "Technique"
        ].unique()
    ) != set(
        ALL_TECHNIQUES
    ):

        raise AssertionError(
            "Condition technique set differs."
        )


    if len(
        rankings
    ) != EXPECTED_RANKING_ROWS_PER_CONDITION:

        raise AssertionError(
            "Condition ranking-row count differs."
        )


    if len(
        build_metrics
    ) != EXPECTED_BUILD_METRICS_PER_CONDITION:

        raise AssertionError(
            "Condition build-metric row count differs."
        )


    if len(
        project_run
    ) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:

        raise AssertionError(
            "Condition project-run row count differs."
        )


    if len(
        fit_log
    ) != EXPECTED_ML_TECHNIQUES:

        raise AssertionError(
            "Condition ML fit count differs."
        )


    if not fit_log[
        "FitSuccess"
    ].all():

        raise AssertionError(
            "One or more ML fits failed."
        )


    if rankings.duplicated(
        subset=[
            "Technique",
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).any():

        raise AssertionError(
            "Condition rankings contain duplicate tests."
        )


    expected_build_keys = set(
        evaluation_rank_data[
            "BuildKey"
        ].astype(str)
    )


    if set(
        build_metrics[
            "BuildKey"
        ].astype(str)
    ) != expected_build_keys:

        raise AssertionError(
            "Condition metric-build set differs."
        )


    for (
        technique,
        build_key,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):

        expected_ranks = np.arange(
            1,
            len(
                ranked_build
            ) + 1,
        )

        actual_ranks = np.sort(
            ranked_build[
                "Rank"
            ].to_numpy(
                dtype=int
            )
        )

        if not np.array_equal(
            expected_ranks,
            actual_ranks,
        ):

            raise AssertionError(
                "Invalid rank sequence.\n"
                f"Technique: {technique}\n"
                f"Build: {build_key}"
            )


    if build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():

        raise AssertionError(
            "Condition metrics contain missing values."
        )


    for metric_column in [
        "APFD",
        "APFDc",
    ]:

        metric_values = build_metrics[
            metric_column
        ].to_numpy(
            dtype=float
        )

        if not (
            (
                metric_values >= 0.0
            )
            &
            (
                metric_values <= 1.0
            )
        ).all():

            raise AssertionError(
                f"{metric_column} falls outside [0, 1]."
            )


    if not project_run[
        "ScoredEvaluationBuilds"
    ].eq(
        EXPECTED_SCORED_EVALUATION_BUILDS
    ).all():

        raise AssertionError(
            "Project-run scored-build count differs."
        )


    if not project_run[
        "EvaluationRows"
    ].eq(
        EXPECTED_MODEL_EVALUATION_ROWS
    ).all():

        raise AssertionError(
            "Project-run evaluation-row count differs."
        )


    if not project_run[
        "EvaluationFailures"
    ].eq(
        EXPECTED_EVALUATION_FAILURES
    ).all():

        raise AssertionError(
            "Project-run evaluation-failure count differs."
        )


def ranking_signature(
    rankings,
    technique,
):
    subset = (
        rankings[
            rankings[
                "Technique"
            ].eq(
                technique
            )
        ][
            [
                "Technique",
                "BuildKey",
                "TestKey",
                "Score",
                "Rank",
            ]
        ]
        .sort_values(
            [
                "BuildKey",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return dataframe_content_sha256(
        subset,
        include_index=False,
    )


# ------------------------------------------------------------
# 24. RUN ONE COMPLETE SMOKE CONDITION
# ------------------------------------------------------------

zero_noise_comparison_table = None


def run_smoke_condition(
    noise_percent,
    repetition_seed,
):
    global zero_noise_comparison_table

    condition_started = (
        time.perf_counter()
    )

    condition_key = (
        f"noise_{int(noise_percent):02d}"
        f"__seed_{int(repetition_seed):02d}"
    )

    print("\n")
    print("-" * 100)

    print(
        "Running smoke condition:",
        condition_key,
    )


    recorded_rows = condition_plan[
        (
            condition_plan[
                "NoisePercent"
            ].eq(
                int(
                    noise_percent
                )
            )
        )
        &
        (
            condition_plan[
                "RepetitionSeed"
            ].eq(
                int(
                    repetition_seed
                )
            )
        )
    ]


    if len(
        recorded_rows
    ) != 1:

        raise AssertionError(
            "Frozen noise condition was not found exactly once."
        )


    recorded_condition = (
        recorded_rows.iloc[0]
    )


    streams = create_seed_random_streams(
        number_of_rows=(
            EXPECTED_RAW_TRAINING_ROWS
        ),

        repetition_seed=(
            repetition_seed
        ),

        failure_subtypes=(
            failure_subtypes
        ),

        subtype_probabilities=(
            subtype_probabilities
        ),
    )


    condition_arrays = (
        construct_condition_arrays(
            clean_verdicts=(
                clean_raw_verdicts
            ),

            row_uniforms=(
                streams[
                    "FlipUniforms"
                ]
            ),

            sampled_failure_subtypes=(
                streams[
                    "SampledFailureSubtypes"
                ]
            ),

            noise_percent=(
                noise_percent
            ),
        )
    )


    flip_mask = condition_arrays[
        "FlipMask"
    ]

    noisy_raw_verdicts = condition_arrays[
        "NoisyVerdicts"
    ]


    generated_mask_hash = (
        packed_boolean_sha256(
            flip_mask
        )
    )

    generated_verdict_hash = (
        integer_array_sha256(
            noisy_raw_verdicts
        )
    )


    mask_hash_match = (
        generated_mask_hash
        == recorded_condition[
            "FlipMaskSHA256"
        ]
    )

    verdict_hash_match = (
        generated_verdict_hash
        == recorded_condition[
            "NoisyVerdictSHA256"
        ]
    )


    if not mask_hash_match:

        raise AssertionError(
            "Generated flip-mask hash differs."
        )


    if not verdict_hash_match:

        raise AssertionError(
            "Generated noisy-verdict hash differs."
        )


    noisy_raw_training = (
        raw_training.copy()
    )

    noisy_raw_training[
        "NoisyVerdict"
    ] = noisy_raw_verdicts


    reconstruction_started = (
        time.perf_counter()
    )


    noisy_direct_rec = (
        reconstruct_noisy_dependent_rec(
            noisy_raw_training[
                [
                    "BuildKey",
                    "TestKey",
                    "BuildOrder",
                    "NoisyVerdict",
                ]
            ]
        )
    )


    reconstruction_seconds = float(
        time.perf_counter()
        - reconstruction_started
    )


    if len(
        noisy_direct_rec
    ) != EXPECTED_MODEL_TRAINING_ROWS:

        raise AssertionError(
            "Noisy REC reconstruction row count differs."
        )


    if noisy_direct_rec.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).any():

        raise AssertionError(
            "Noisy REC reconstruction contains duplicate rows."
        )


    aligned_noisy_direct = (
        training_key_frame.merge(
            noisy_direct_rec,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "_ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    missing_noisy_rec_rows = int(
        aligned_noisy_direct[
            "_merge"
        ].ne(
            "both"
        ).sum()
    )


    if missing_noisy_rec_rows:

        raise AssertionError(
            "Noisy REC rows are missing after alignment."
        )


    noisy_direct_matrix = (
        aligned_noisy_direct[
            VERDICT_DEPENDENT_REC_FEATURES
        ]
        .to_numpy(
            dtype=float
        )
    )


    noisy_anchored_matrix = (
        noisy_direct_matrix
        + anchor_dependent_matrix
    )


    if not np.isfinite(
        noisy_anchored_matrix
    ).all():

        raise AssertionError(
            "Noisy anchored REC matrix contains non-finite values."
        )


    noisy_model_training = (
        clean_model_training.copy(
            deep=True
        )
    )


    unaffected_columns = [
        column
        for column in (
            noisy_model_training.columns
        )
        if column not in (
            [
                dataset_verdict_column,
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        )
    ]


    unaffected_hash_before = (
        dataframe_content_sha256(
            noisy_model_training[
                unaffected_columns
            ],
            include_index=True,
        )
    )


    noisy_model_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )


    noisy_model_training[
        dataset_verdict_column
    ] = noisy_model_verdicts


    noisy_model_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ] = noisy_anchored_matrix


    unaffected_hash_after = (
        dataframe_content_sha256(
            noisy_model_training[
                unaffected_columns
            ],
            include_index=True,
        )
    )


    unaffected_columns_unchanged = (
        unaffected_hash_before
        == unaffected_hash_after
    )


    if not unaffected_columns_unchanged:

        raise AssertionError(
            "Verdict-independent or non-REC predictors changed."
        )


    changed_model_labels = int(
        (
            noisy_model_verdicts
            != clean_model_verdicts
        ).sum()
    )


    changed_dependent_values = int(
        (
            ~np.isclose(
                noisy_anchored_matrix,
                original_dependent_matrix,
                rtol=COMPARISON_RTOL,
                atol=COMPARISON_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


    zero_noise_semantic_mismatches = (
        np.nan
    )


    if noise_percent == 0:

        if changed_model_labels != 0:

            raise AssertionError(
                "0% noise changed model labels."
            )


        if changed_dependent_values != 0:

            raise AssertionError(
                "0% noise changed dependent REC features."
            )


        (
            zero_noise_semantic_mismatches,
            zero_noise_comparison_table,
        ) = semantic_dataframe_comparison(
            clean_dataframe=(
                clean_model_training
            ),

            candidate_dataframe=(
                noisy_model_training
            ),

            rtol=(
                COMPARISON_RTOL
            ),

            atol=(
                COMPARISON_ATOL
            ),
        )


        if zero_noise_semantic_mismatches != 0:

            print(
                "\n0% semantic mismatching columns:"
            )

            display(
                zero_noise_comparison_table[
                    zero_noise_comparison_table[
                        "Mismatches"
                    ].gt(0)
                ]
            )

            raise AssertionError(
                "0% noisy model-training values differ "
                "semantically from the clean training data."
            )


        print(
            "  0% semantic dataset comparison passed."
        )

        print(
            "  Dtype-only differences:",
            int(
                (
                    zero_noise_comparison_table[
                        "CleanDtype"
                    ]
                    !=
                    zero_noise_comparison_table[
                        "CandidateDtype"
                    ]
                ).sum()
            ),
        )


    else:

        if changed_model_labels == 0:

            raise AssertionError(
                "Positive noise changed no model labels."
            )


        if changed_dependent_values == 0:

            raise AssertionError(
                "Positive noise changed no dependent REC values."
            )


    (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    ) = prepare_ml_matrices(
        noisy_model_training
    )


    all_ranking_frames = []

    fit_records = []

    models = create_ml_models(
        repetition_seed
    )


    for technique, model in (
        models.items()
    ):

        print(
            "  Fitting:",
            technique,
        )


        fit_started = (
            time.perf_counter()
        )

        fit_success = False
        probability_success = False
        error_text = ""


        try:

            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore"
                )

                model.fit(
                    X_train,
                    y_train,
                )

                fit_success = True


                failure_probabilities = (
                    get_failure_probability(
                        model,
                        X_evaluation,
                    )
                )

                probability_success = True


            model_rankings = (
                create_ml_rankings(
                    technique=technique,
                    failure_probabilities=(
                        failure_probabilities
                    ),
                )
            )


            all_ranking_frames.append(
                model_rankings
            )


        except Exception as error:

            error_text = (
                f"{type(error).__name__}: {error}"
            )

            raise


        finally:

            fit_seconds = float(
                time.perf_counter()
                - fit_started
            )


            fit_records.append({
                "Project":
                    PROJECT_NAME,

                "ConditionKey":
                    condition_key,

                "NoisePercent":
                    int(
                        noise_percent
                    ),

                "RepetitionSeed":
                    int(
                        repetition_seed
                    ),

                "Technique":
                    technique,

                "TrainingRows":
                    EXPECTED_MODEL_TRAINING_ROWS,

                "EvaluationRows":
                    EXPECTED_MODEL_EVALUATION_ROWS,

                "ActivePredictors":
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    ),

                "TrainingFailures":
                    int(
                        y_train.sum()
                    ),

                "TrainingPasses":
                    int(
                        (
                            y_train == 0
                        ).sum()
                    ),

                "FitSuccess":
                    fit_success,

                "ProbabilitySuccess":
                    probability_success,

                "FitSeconds":
                    fit_seconds,

                "Error":
                    error_text,
            })


        del model

        gc.collect()


    random_rankings = (
        create_random_rankings(
            repetition_seed
        )
    )


    (
        latest_fail_rankings,
        qtf_rankings,
        qtf_unseen_scores,
    ) = create_history_baseline_rankings(
        noisy_raw_training
    )


    all_ranking_frames.extend([
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ])


    rankings = pd.concat(
        all_ranking_frames,
        ignore_index=True,
    )


    rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    rankings.insert(
        1,
        "ProjectSlug",
        PROJECT_SLUG,
    )

    rankings.insert(
        2,
        "ConditionKey",
        condition_key,
    )

    rankings.insert(
        3,
        "NoisePercent",
        int(
            noise_percent
        ),
    )

    rankings.insert(
        4,
        "RepetitionSeed",
        int(
            repetition_seed
        ),
    )


    fit_log = pd.DataFrame(
        fit_records
    )


    build_metrics = (
        calculate_build_metrics(
            rankings=rankings,
            noise_percent=(
                noise_percent
            ),
            repetition_seed=(
                repetition_seed
            ),
        )
    )


    project_run = (
        aggregate_project_run(
            build_metrics
        )
    )


    validate_condition_outputs(
        rankings=rankings,
        build_metrics=build_metrics,
        project_run=project_run,
        fit_log=fit_log,
    )


    condition_seconds = float(
        time.perf_counter()
        - condition_started
    )


    condition_directory = (
        SMOKE_CONDITION_ROOT
        / condition_key
    )


    rankings_path = (
        condition_directory
        / "rankings.parquet"
    )

    build_metrics_path = (
        condition_directory
        / "build_metrics.csv"
    )

    project_run_path = (
        condition_directory
        / "project_run.csv"
    )

    fit_log_path = (
        condition_directory
        / "model_fits.csv"
    )

    condition_summary_path = (
        condition_directory
        / "condition_summary.json"
    )

    success_marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )


    atomic_write_parquet(
        rankings_path,
        rankings,
    )

    atomic_write_csv(
        build_metrics_path,
        build_metrics,
    )

    atomic_write_csv(
        project_run_path,
        project_run,
    )

    atomic_write_csv(
        fit_log_path,
        fit_log,
    )


    condition_summary = {
        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionKey":
            condition_key,

        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "RawRowsFlipped":
            int(
                flip_mask.sum()
            ),

        "RecordedRawRowsFlipped":
            int(
                recorded_condition[
                    "RawRowsFlipped"
                ]
            ),

        "RealisedNoisePercent":
            float(
                100.0
                * flip_mask.sum()
                / EXPECTED_RAW_TRAINING_ROWS
            ),

        "PassToFailure":
            int(
                condition_arrays[
                    "PassToFailureMask"
                ].sum()
            ),

        "FailureToPass":
            int(
                condition_arrays[
                    "FailureToPassMask"
                ].sum()
            ),

        "ChangedModelLabels":
            changed_model_labels,

        "ChangedDependentRECValues":
            changed_dependent_values,

        "ZeroNoiseSemanticMismatches":
            zero_noise_semantic_mismatches,

        "UnaffectedColumnsUnchanged":
            unaffected_columns_unchanged,

        "MaskHashMatch":
            mask_hash_match,

        "NoisyVerdictHashMatch":
            verdict_hash_match,

        "NoisyRECReconstructionRows":
            len(
                noisy_direct_rec
            ),

        "MissingNoisyRECRows":
            missing_noisy_rec_rows,

        "QTFUnseenScores":
            qtf_unseen_scores,

        "MLFits":
            len(
                fit_log
            ),

        "RankingRows":
            len(
                rankings
            ),

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunRows":
            len(
                project_run
            ),

        "ReconstructionSeconds":
            reconstruction_seconds,

        "ConditionSeconds":
            condition_seconds,

        "Rankings":
            str(
                rankings_path
            ),

        "RankingsSHA256":
            calculate_sha256(
                rankings_path
            ),

        "BuildMetrics":
            str(
                build_metrics_path
            ),

        "BuildMetricsSHA256":
            calculate_sha256(
                build_metrics_path
            ),

        "ProjectRun":
            str(
                project_run_path
            ),

        "ProjectRunSHA256":
            calculate_sha256(
                project_run_path
            ),

        "ModelFits":
            str(
                fit_log_path
            ),

        "ModelFitsSHA256":
            calculate_sha256(
                fit_log_path
            ),

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    atomic_write_json(
        condition_summary_path,
        condition_summary,
    )


    atomic_write_json(
        success_marker_path,
        {
            "Status":
                "PASS_SMOKE_CONDITION",

            "ConditionKey":
                condition_key,

            "RankingsSHA256":
                condition_summary[
                    "RankingsSHA256"
                ],

            "BuildMetricsSHA256":
                condition_summary[
                    "BuildMetricsSHA256"
                ],

            "ProjectRunSHA256":
                condition_summary[
                    "ProjectRunSHA256"
                ],

            "ModelFitsSHA256":
                condition_summary[
                    "ModelFitsSHA256"
                ],

            "CompletedAtUTC":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        },
    )


    print(
        "  Completed:",
        condition_key,
    )

    print(
        "  Raw flips:",
        int(
            flip_mask.sum()
        ),
    )

    print(
        "  Model label changes:",
        changed_model_labels,
    )

    print(
        "  Dependent REC changes:",
        changed_dependent_values,
    )

    print(
        "  Ranking rows:",
        len(
            rankings
        ),
    )

    print(
        "  Build-metric rows:",
        len(
            build_metrics
        ),
    )

    print(
        "  Runtime seconds:",
        round(
            condition_seconds,
            2,
        ),
    )


    result = {
        "ConditionKey":
            condition_key,

        "Rankings":
            rankings,

        "BuildMetrics":
            build_metrics,

        "ProjectRun":
            project_run,

        "FitLog":
            fit_log,

        "ConditionSummary":
            condition_summary,

        "RandomSignature":
            ranking_signature(
                rankings,
                "Random",
            ),

        "QTFAvgSignature":
            ranking_signature(
                rankings,
                "QTF-Avg",
            ),

        "LatestFailSignature":
            ranking_signature(
                rankings,
                "LatestFail",
            ),
    }


    del X_train
    del y_train
    del X_evaluation
    del training_medians
    del noisy_model_training
    del noisy_direct_rec
    del aligned_noisy_direct
    del noisy_direct_matrix
    del noisy_anchored_matrix

    gc.collect()

    return result


# ------------------------------------------------------------
# 25. RUN BOTH SMOKE CONDITIONS
# ------------------------------------------------------------

SMOKE_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

smoke_started = (
    time.perf_counter()
)

condition_results = []


for condition in (
    SMOKE_CONDITIONS
):

    condition_results.append(
        run_smoke_condition(
            noise_percent=(
                condition[
                    "NoisePercent"
                ]
            ),

            repetition_seed=(
                condition[
                    "RepetitionSeed"
                ]
            ),
        )
    )


smoke_seconds = float(
    time.perf_counter()
    - smoke_started
)


# ------------------------------------------------------------
# 26. AGGREGATE SMOKE OUTPUTS
# ------------------------------------------------------------

all_smoke_rankings = pd.concat(
    [
        result[
            "Rankings"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)


all_smoke_build_metrics = pd.concat(
    [
        result[
            "BuildMetrics"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)


all_smoke_project_runs = pd.concat(
    [
        result[
            "ProjectRun"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)


all_smoke_model_fits = pd.concat(
    [
        result[
            "FitLog"
        ]
        for result in condition_results
    ],
    ignore_index=True,
)


smoke_condition_audit = pd.DataFrame(
    [
        result[
            "ConditionSummary"
        ]
        for result in condition_results
    ]
)


if len(
    all_smoke_rankings
) != EXPECTED_TOTAL_RANKING_ROWS:

    raise AssertionError(
        "Aggregated smoke ranking-row count differs."
    )


if len(
    all_smoke_build_metrics
) != EXPECTED_TOTAL_BUILD_METRICS:

    raise AssertionError(
        "Aggregated smoke build-metric count differs."
    )


if len(
    all_smoke_project_runs
) != EXPECTED_TOTAL_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Aggregated smoke project-run count differs."
    )


if len(
    all_smoke_model_fits
) != EXPECTED_TOTAL_ML_FITS:

    raise AssertionError(
        "Aggregated smoke ML-fit count differs."
    )


# ------------------------------------------------------------
# 27. BASELINE INVARIANCE
# ------------------------------------------------------------

zero_result = next(
    result
    for result in condition_results
    if result[
        "ConditionSummary"
    ][
        "NoisePercent"
    ] == 0
)


fifty_result = next(
    result
    for result in condition_results
    if result[
        "ConditionSummary"
    ][
        "NoisePercent"
    ] == 50
)


random_invariant = (
    zero_result[
        "RandomSignature"
    ]
    == fifty_result[
        "RandomSignature"
    ]
)


qtf_invariant = (
    zero_result[
        "QTFAvgSignature"
    ]
    == fifty_result[
        "QTFAvgSignature"
    ]
)


latest_fail_same = (
    zero_result[
        "LatestFailSignature"
    ]
    == fifty_result[
        "LatestFailSignature"
    ]
)


baseline_invariance = pd.DataFrame([
    {
        "Technique":
            "Random",

        "ExpectedInvariantAcrossNoise":
            True,

        "ZeroNoiseSignature":
            zero_result[
                "RandomSignature"
            ],

        "FiftyNoiseSignature":
            fifty_result[
                "RandomSignature"
            ],

        "ActualInvariant":
            random_invariant,

        "Pass":
            random_invariant,
    },

    {
        "Technique":
            "QTF-Avg",

        "ExpectedInvariantAcrossNoise":
            True,

        "ZeroNoiseSignature":
            zero_result[
                "QTFAvgSignature"
            ],

        "FiftyNoiseSignature":
            fifty_result[
                "QTFAvgSignature"
            ],

        "ActualInvariant":
            qtf_invariant,

        "Pass":
            qtf_invariant,
    },

    {
        "Technique":
            "LatestFail",

        "ExpectedInvariantAcrossNoise":
            False,

        "ZeroNoiseSignature":
            zero_result[
                "LatestFailSignature"
            ],

        "FiftyNoiseSignature":
            fifty_result[
                "LatestFailSignature"
            ],

        "ActualInvariant":
            latest_fail_same,

        "Pass":
            True,
    },
])


baseline_invariance_failures = int(
    (
        ~baseline_invariance[
            "Pass"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 28. VERIFY IMMUTABILITY
# ------------------------------------------------------------

evaluation_sha256_after = calculate_sha256(
    model_evaluation_path
)


evaluation_unchanged = (
    evaluation_sha256_before
    == evaluation_sha256_after
)


source_hashes_after = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}


source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)


registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)


registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)


if not evaluation_unchanged:

    raise AssertionError(
        "Frozen evaluation data changed during Step 4B V2."
    )


if not source_files_unchanged:

    raise AssertionError(
        "A frozen source file changed during Step 4B V2."
    )


if not registry_unchanged:

    raise AssertionError(
        "Completion registry changed during Step 4B V2."
    )


for path, expected_hash in (
    frozen_input_hashes.items()
):

    if calculate_sha256(
        path
    ) != expected_hash:

        raise AssertionError(
            "A frozen experiment input changed during Step 4B V2.\n"
            f"Path: {path}"
        )


# ------------------------------------------------------------
# 29. OVERALL VALIDATION
# ------------------------------------------------------------

zero_condition_summary = (
    zero_result[
        "ConditionSummary"
    ]
)


fifty_condition_summary = (
    fifty_result[
        "ConditionSummary"
    ]
)


validation_records = [
    {
        "Check":
            "Step 4A passed",

        "Expected":
            EXPECTED_STEP4A_STATUS,

        "Actual":
            step4a_status[
                "Status"
            ],

        "Pass":
            step4a_status[
                "Status"
            ]
            == EXPECTED_STEP4A_STATUS,
    },

    {
        "Check":
            "Smoke conditions",

        "Expected":
            2,

        "Actual":
            len(
                condition_results
            ),

        "Pass":
            len(
                condition_results
            ) == 2,
    },

    {
        "Check":
            "Total ML fits",

        "Expected":
            EXPECTED_TOTAL_ML_FITS,

        "Actual":
            len(
                all_smoke_model_fits
            ),

        "Pass":
            len(
                all_smoke_model_fits
            )
            == EXPECTED_TOTAL_ML_FITS,
    },

    {
        "Check":
            "Successful ML fits",

        "Expected":
            EXPECTED_TOTAL_ML_FITS,

        "Actual":
            int(
                all_smoke_model_fits[
                    "FitSuccess"
                ].sum()
            ),

        "Pass":
            int(
                all_smoke_model_fits[
                    "FitSuccess"
                ].sum()
            )
            == EXPECTED_TOTAL_ML_FITS,
    },

    {
        "Check":
            "Total ranking rows",

        "Expected":
            EXPECTED_TOTAL_RANKING_ROWS,

        "Actual":
            len(
                all_smoke_rankings
            ),

        "Pass":
            len(
                all_smoke_rankings
            )
            == EXPECTED_TOTAL_RANKING_ROWS,
    },

    {
        "Check":
            "Total build-metric rows",

        "Expected":
            EXPECTED_TOTAL_BUILD_METRICS,

        "Actual":
            len(
                all_smoke_build_metrics
            ),

        "Pass":
            len(
                all_smoke_build_metrics
            )
            == EXPECTED_TOTAL_BUILD_METRICS,
    },

    {
        "Check":
            "Total project-run rows",

        "Expected":
            EXPECTED_TOTAL_PROJECT_RUN_ROWS,

        "Actual":
            len(
                all_smoke_project_runs
            ),

        "Pass":
            len(
                all_smoke_project_runs
            )
            == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Techniques per condition",

        "Expected":
            EXPECTED_TECHNIQUES,

        "Actual":
            int(
                all_smoke_project_runs.groupby(
                    [
                        "NoisePercent",
                        "RepetitionSeed",
                    ]
                )[
                    "Technique"
                ].nunique().min()
            ),

        "Pass":
            bool(
                all_smoke_project_runs.groupby(
                    [
                        "NoisePercent",
                        "RepetitionSeed",
                    ]
                )[
                    "Technique"
                ].nunique().eq(
                    EXPECTED_TECHNIQUES
                ).all()
            ),
    },

    {
        "Check":
            "0% flipped raw rows",

        "Expected":
            0,

        "Actual":
            zero_condition_summary[
                "RawRowsFlipped"
            ],

        "Pass":
            zero_condition_summary[
                "RawRowsFlipped"
            ] == 0,
    },

    {
        "Check":
            "0% changed model labels",

        "Expected":
            0,

        "Actual":
            zero_condition_summary[
                "ChangedModelLabels"
            ],

        "Pass":
            zero_condition_summary[
                "ChangedModelLabels"
            ] == 0,
    },

    {
        "Check":
            "0% changed dependent REC values",

        "Expected":
            0,

        "Actual":
            zero_condition_summary[
                "ChangedDependentRECValues"
            ],

        "Pass":
            zero_condition_summary[
                "ChangedDependentRECValues"
            ] == 0,
    },

    {
        "Check":
            "0% semantic dataframe mismatches",

        "Expected":
            0,

        "Actual":
            int(
                zero_condition_summary[
                    "ZeroNoiseSemanticMismatches"
                ]
            ),

        "Pass":
            int(
                zero_condition_summary[
                    "ZeroNoiseSemanticMismatches"
                ]
            ) == 0,
    },

    {
        "Check":
            "50% changed model labels positive",

        "Expected":
            True,

        "Actual":
            fifty_condition_summary[
                "ChangedModelLabels"
            ] > 0,

        "Pass":
            fifty_condition_summary[
                "ChangedModelLabels"
            ] > 0,
    },

    {
        "Check":
            "50% changed dependent REC values positive",

        "Expected":
            True,

        "Actual":
            fifty_condition_summary[
                "ChangedDependentRECValues"
            ] > 0,

        "Pass":
            fifty_condition_summary[
                "ChangedDependentRECValues"
            ] > 0,
    },

    {
        "Check":
            "Condition mask hashes",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_condition_audit[
                    "MaskHashMatch"
                ].all()
            ),

        "Pass":
            bool(
                smoke_condition_audit[
                    "MaskHashMatch"
                ].all()
            ),
    },

    {
        "Check":
            "Condition noisy-verdict hashes",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_condition_audit[
                    "NoisyVerdictHashMatch"
                ].all()
            ),

        "Pass":
            bool(
                smoke_condition_audit[
                    "NoisyVerdictHashMatch"
                ].all()
            ),
    },

    {
        "Check":
            "Unaffected columns unchanged",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_condition_audit[
                    "UnaffectedColumnsUnchanged"
                ].all()
            ),

        "Pass":
            bool(
                smoke_condition_audit[
                    "UnaffectedColumnsUnchanged"
                ].all()
            ),
    },

    {
        "Check":
            "Random invariant across noise",

        "Expected":
            True,

        "Actual":
            random_invariant,

        "Pass":
            random_invariant,
    },

    {
        "Check":
            "QTF-Avg invariant across noise",

        "Expected":
            True,

        "Actual":
            qtf_invariant,

        "Pass":
            qtf_invariant,
    },

    {
        "Check":
            "Baseline invariance failures",

        "Expected":
            0,

        "Actual":
            baseline_invariance_failures,

        "Pass":
            baseline_invariance_failures
            == 0,
    },

    {
        "Check":
            "Evaluation unchanged",

        "Expected":
            True,

        "Actual":
            evaluation_unchanged,

        "Pass":
            evaluation_unchanged,
    },

    {
        "Check":
            "Source files unchanged",

        "Expected":
            True,

        "Actual":
            source_files_unchanged,

        "Pass":
            source_files_unchanged,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)


failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 4B V2 validation:")

display(
    validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 4B V2 DID NOT PASS."
    )


# ------------------------------------------------------------
# 30. WRITE AGGREGATED SMOKE OUTPUTS
# ------------------------------------------------------------

atomic_write_parquet(
    SMOKE_AGGREGATED_RANKINGS_PATH,
    all_smoke_rankings,
)

atomic_write_csv(
    SMOKE_AGGREGATED_BUILD_METRICS_PATH,
    all_smoke_build_metrics,
)

atomic_write_csv(
    SMOKE_AGGREGATED_PROJECT_RUNS_PATH,
    all_smoke_project_runs,
)

atomic_write_csv(
    SMOKE_AGGREGATED_MODEL_FITS_PATH,
    all_smoke_model_fits,
)

atomic_write_csv(
    SMOKE_CONDITION_AUDIT_PATH,
    smoke_condition_audit,
)

atomic_write_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)

atomic_write_csv(
    SMOKE_ZERO_NOISE_COMPARISON_PATH,
    zero_noise_comparison_table,
)

atomic_write_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 31. READBACK VALIDATION
# ------------------------------------------------------------

rankings_readback = pd.read_parquet(
    SMOKE_AGGREGATED_RANKINGS_PATH
)

build_metrics_readback = pd.read_csv(
    SMOKE_AGGREGATED_BUILD_METRICS_PATH,
    low_memory=False,
)

project_runs_readback = pd.read_csv(
    SMOKE_AGGREGATED_PROJECT_RUNS_PATH,
    low_memory=False,
)

model_fits_readback = pd.read_csv(
    SMOKE_AGGREGATED_MODEL_FITS_PATH,
    low_memory=False,
)


if len(
    rankings_readback
) != EXPECTED_TOTAL_RANKING_ROWS:

    raise AssertionError(
        "Smoke ranking readback row count differs."
    )


if len(
    build_metrics_readback
) != EXPECTED_TOTAL_BUILD_METRICS:

    raise AssertionError(
        "Smoke build-metric readback count differs."
    )


if len(
    project_runs_readback
) != EXPECTED_TOTAL_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Smoke project-run readback count differs."
    )


if len(
    model_fits_readback
) != EXPECTED_TOTAL_ML_FITS:

    raise AssertionError(
        "Smoke model-fit readback count differs."
    )


# ------------------------------------------------------------
# 32. REPORT, CHECKPOINT AND STATUS
# ------------------------------------------------------------

smoke_output_paths = [
    SMOKE_AGGREGATED_RANKINGS_PATH,
    SMOKE_AGGREGATED_BUILD_METRICS_PATH,
    SMOKE_AGGREGATED_PROJECT_RUNS_PATH,
    SMOKE_AGGREGATED_MODEL_FITS_PATH,
    SMOKE_CONDITION_AUDIT_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_ZERO_NOISE_COMPARISON_PATH,
    SMOKE_VALIDATION_PATH,
]


smoke_output_inventory = []


for path in smoke_output_paths:

    smoke_output_inventory.append({
        "Path":
            str(
                path
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "ImplementationVersion":
        "STEP_4B_V2_SEMANTIC_ZERO_NOISE_COMPARISON",

    "SmokeConditions":
        SMOKE_CONDITIONS,

    "ConditionsCompleted":
        len(
            condition_results
        ),

    "MLFits":
        len(
            all_smoke_model_fits
        ),

    "RankingRows":
        len(
            all_smoke_rankings
        ),

    "BuildMetricRows":
        len(
            all_smoke_build_metrics
        ),

    "ProjectRunRows":
        len(
            all_smoke_project_runs
        ),

    "Techniques":
        ALL_TECHNIQUES,

    "ZeroNoiseChangedModelLabels":
        zero_condition_summary[
            "ChangedModelLabels"
        ],

    "ZeroNoiseChangedDependentRECValues":
        zero_condition_summary[
            "ChangedDependentRECValues"
        ],

    "ZeroNoiseSemanticMismatches":
        zero_condition_summary[
            "ZeroNoiseSemanticMismatches"
        ],

    "FiftyNoiseChangedModelLabels":
        fifty_condition_summary[
            "ChangedModelLabels"
        ],

    "FiftyNoiseChangedDependentRECValues":
        fifty_condition_summary[
            "ChangedDependentRECValues"
        ],

    "RandomInvariantAcrossNoise":
        random_invariant,

    "QTFAvgInvariantAcrossNoise":
        qtf_invariant,

    "LatestFailInvariantAcrossNoise":
        latest_fail_same,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "SmokeRuntimeSeconds":
        smoke_seconds,

    "OutputInventory":
        smoke_output_inventory,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    SMOKE_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "ImplementationVersion":
        "STEP_4B_V2_SEMANTIC_ZERO_NOISE_COMPARISON",

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RECCheckpointSHA256":
        calculate_sha256(
            REC_CHECKPOINT_PATH
        ),

    "NoiseCheckpointSHA256":
        calculate_sha256(
            NOISE_CHECKPOINT_PATH
        ),

    "NoisyRECCheckpointSHA256":
        calculate_sha256(
            NOISY_REC_CHECKPOINT_PATH
        ),

    "ModelMetricCheckpointSHA256":
        calculate_sha256(
            MODEL_METRIC_CHECKPOINT_PATH
        ),

    "SmokeConditions":
        SMOKE_CONDITIONS,

    "MLFits":
        EXPECTED_TOTAL_ML_FITS,

    "RankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRICS,

    "ProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "ZeroNoiseSemanticMismatches":
        int(
            zero_condition_summary[
                "ZeroNoiseSemanticMismatches"
            ]
        ),

    "SmokeRankings":
        str(
            SMOKE_AGGREGATED_RANKINGS_PATH
        ),

    "SmokeRankingsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_RANKINGS_PATH
        ),

    "SmokeBuildMetrics":
        str(
            SMOKE_AGGREGATED_BUILD_METRICS_PATH
        ),

    "SmokeBuildMetricsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_BUILD_METRICS_PATH
        ),

    "SmokeProjectRuns":
        str(
            SMOKE_AGGREGATED_PROJECT_RUNS_PATH
        ),

    "SmokeProjectRunsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_PROJECT_RUNS_PATH
        ),

    "SmokeModelFits":
        str(
            SMOKE_AGGREGATED_MODEL_FITS_PATH
        ),

    "SmokeModelFitsSHA256":
        calculate_sha256(
            SMOKE_AGGREGATED_MODEL_FITS_PATH
        ),

    "RandomInvariantAcrossNoise":
        random_invariant,

    "QTFAvgInvariantAcrossNoise":
        qtf_invariant,

    "EvaluationSHA256":
        evaluation_sha256_after,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "SmokeRuntimeSeconds":
        smoke_seconds,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    SMOKE_TEST_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4B_PASS_STATUS,

    "ImplementationVersion":
        "STEP_4B_V2_SEMANTIC_ZERO_NOISE_COMPARISON",

    "Conditions":
        len(
            condition_results
        ),

    "MLFits":
        len(
            all_smoke_model_fits
        ),

    "RankingRows":
        len(
            all_smoke_rankings
        ),

    "BuildMetricRows":
        len(
            all_smoke_build_metrics
        ),

    "ProjectRunRows":
        len(
            all_smoke_project_runs
        ),

    "ZeroNoiseSemanticMismatches":
        int(
            zero_condition_summary[
                "ZeroNoiseSemanticMismatches"
            ]
        ),

    "RandomInvariantAcrossNoise":
        random_invariant,

    "QTFAvgInvariantAcrossNoise":
        qtf_invariant,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "Checkpoint":
        str(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP4B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 33. FINAL STATUS READBACK
# ------------------------------------------------------------

final_status = json.loads(
    STEP4B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_status.get(
        "Status"
    )
    != STEP4B_PASS_STATUS
):

    raise AssertionError(
        "Final Step 4B V2 status differs."
    )


# ------------------------------------------------------------
# 34. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nSmoke-test model fits:")

display(
    all_smoke_model_fits
)


print("\nSmoke-test project-run results:")

display(
    all_smoke_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
)


print("\nBaseline invariance:")

display(
    baseline_invariance
)


print("\n0% semantic comparison summary:")

display(
    zero_noise_comparison_table.groupby(
        [
            "ComparisonType",
            "Pass",
        ],
        as_index=False,
    )
    .agg(
        Columns=(
            "Column",
            "count",
        ),

        TotalMismatches=(
            "Mismatches",
            "sum",
        ),
    )
)


dtype_difference_count = int(
    (
        zero_noise_comparison_table[
            "CleanDtype"
        ]
        !=
        zero_noise_comparison_table[
            "CandidateDtype"
        ]
    ).sum()
)


print(
    "\n0% columns with harmless dtype differences:",
    dtype_difference_count,
)


print("\nCondition audit:")

display(
    smoke_condition_audit[
        [
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "RawRowsFlipped",
            "RealisedNoisePercent",
            "ChangedModelLabels",
            "ChangedDependentRECValues",
            "ZeroNoiseSemanticMismatches",
            "MLFits",
            "RankingRows",
            "BuildMetricRows",
            "ProjectRunRows",
            "ConditionSeconds",
        ]
    ]
)


# ------------------------------------------------------------
# 35. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 120)
print("=== PROJECT 9 CELL 8 / STEP 4B V2 RESULT ===")
print("=" * 120)

print("\nProject:")

print(
    PROJECT_NAME
)


print("\nExperiment totals:")

print(
    "Conditions:",
    len(
        condition_results
    ),
)

print(
    "ML fits:",
    len(
        all_smoke_model_fits
    ),
    "/",
    EXPECTED_TOTAL_ML_FITS,
)

print(
    "Ranking rows:",
    len(
        all_smoke_rankings
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(
        all_smoke_build_metrics
    ),
    "/",
    EXPECTED_TOTAL_BUILD_METRICS,
)

print(
    "Project-run rows:",
    len(
        all_smoke_project_runs
    ),
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)


print("\nClean-condition validation:")

print(
    "0% flipped raw rows:",
    zero_condition_summary[
        "RawRowsFlipped"
    ],
)

print(
    "0% changed model labels:",
    zero_condition_summary[
        "ChangedModelLabels"
    ],
)

print(
    "0% changed dependent REC values:",
    zero_condition_summary[
        "ChangedDependentRECValues"
    ],
)

print(
    "0% semantic dataframe mismatches:",
    int(
        zero_condition_summary[
            "ZeroNoiseSemanticMismatches"
        ]
    ),
)

print(
    "0% harmless dtype-different columns:",
    dtype_difference_count,
)


print("\nSevere-noise validation:")

print(
    "50% flipped raw rows:",
    fifty_condition_summary[
        "RawRowsFlipped"
    ],
)

print(
    "50% changed model labels:",
    fifty_condition_summary[
        "ChangedModelLabels"
    ],
)

print(
    "50% changed dependent REC values:",
    fifty_condition_summary[
        "ChangedDependentRECValues"
    ],
)


print("\nBaseline protocol:")

print(
    "Random invariant across noise:",
    random_invariant,
)

print(
    "QTF-Avg invariant across noise:",
    qtf_invariant,
)

print(
    "LatestFail invariant across noise:",
    latest_fail_same,
)


print("\nImmutability:")

print(
    "Evaluation unchanged:",
    evaluation_unchanged,
)

print(
    "Evaluation SHA-256:",
    evaluation_sha256_after,
)

print(
    "Source files unchanged:",
    source_files_unchanged,
)


print("\nRuntime:")

print(
    "Smoke-test seconds:",
    round(
        smoke_seconds,
        2,
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nSmoke-test checkpoint:")

print(
    SMOKE_TEST_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        SMOKE_TEST_CHECKPOINT_PATH
    ),
)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–8 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP4B_PASS_STATUS,
)

print("=" * 120)

=== PROJECT 9 CELL 8 / STEP 4B V2: TWO-CONDITION END-TO-END SMOKE TEST ===


----------------------------------------------------------------------------------------------------
Running smoke condition: noise_00__seed_01
  0% semantic dataset comparison passed.
  Dtype-only differences: 4
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM
  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0
  Model label changes: 0
  Dependent REC changes: 0
  Ranking rows: 129521
  Build-metric rows: 210
  Runtime seconds: 84.26


----------------------------------------------------------------------------------------------------
Running smoke condition: noise_50__seed_01
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM
  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 156700
  Model label changes: 30346
  Dependent REC changes: 715259
  Ranking rows: 129521
  Build-metric rows: 210
  Runtime seconds: 80.39

Step 4B V2 validation:


,Check,Expected,Actual,Pass
0,Step 4A passed,PASS_PROJECT_9_MODEL_PREDICTOR_BASELINE_AND_ME...,PASS_PROJECT_9_MODEL_PREDICTOR_BASELINE_AND_ME...,True
1,Smoke conditions,2,2,True
2,Total ML fits,8,8,True
3,Successful ML fits,8,8,True
4,Total ranking rows,259042,259042,True
5,Total build-metric rows,420,420,True
6,Total project-run rows,14,14,True
7,Techniques per condition,7,7,True
8,0% flipped raw rows,0,0,True
9,0% changed model labels,0,0,True



Smoke-test model fits:


,Project,ConditionKey,NoisePercent,RepetitionSeed,Technique,TrainingRows,EvaluationRows,ActivePredictors,TrainingFailures,TrainingPasses,FitSuccess,ProbabilitySuccess,FitSeconds,Error
0,camunda@camunda-bpm-platform,noise_00__seed_01,0,1,RandomForest,60880,18503,151,1427,59453,True,True,36.034908,
1,camunda@camunda-bpm-platform,noise_00__seed_01,0,1,XGBoost,60880,18503,151,1427,59453,True,True,15.651070,
2,camunda@camunda-bpm-platform,noise_00__seed_01,0,1,LightGBM,60880,18503,151,1427,59453,True,True,14.192909,
3,camunda@camunda-bpm-platform,noise_00__seed_01,0,1,NaiveBayes,60880,18503,151,1427,59453,True,True,0.517801,
4,camunda@camunda-bpm-platform,noise_50__seed_01,50,1,RandomForest,60880,18503,151,30285,30595,True,True,54.309191,
5,camunda@camunda-bpm-platform,noise_50__seed_01,50,1,XGBoost,60880,18503,151,30285,30595,True,True,10.380843,
6,camunda@camunda-bpm-platform,noise_50__seed_01,50,1,LightGBM,60880,18503,151,30285,30595,True,True,3.540262,
7,camunda@camunda-bpm-platform,noise_50__seed_01,50,1,NaiveBayes,60880,18503,151,30285,30595,True,True,0.377840,



Smoke-test project-run results:


...


0% semantic comparison summary:


,ComparisonType,Pass,Columns,TotalMismatches
0,NUMERIC_SEMANTIC,True,154,0



0% columns with harmless dtype differences: 4

Condition audit:


,ConditionKey,NoisePercent,RepetitionSeed,RawRowsFlipped,RealisedNoisePercent,ChangedModelLabels,ChangedDependentRECValues,ZeroNoiseSemanticMismatches,MLFits,RankingRows,BuildMetricRows,ProjectRunRows,ConditionSeconds
0,noise_00__seed_01,0,1,0,0.000000,0,0,0.0,4,129521,210,7,84.263007
1,noise_50__seed_01,50,1,156700,49.907002,30346,715259,NaN,4,129521,210,7,80.387209




=== PROJECT 9 CELL 8 / STEP 4B V2 RESULT ===

Project:
camunda@camunda-bpm-platform

Experiment totals:
Conditions: 2
ML fits: 8 / 8
Ranking rows: 259042 / 259042
Build-metric rows: 420 / 420
Project-run rows: 14 / 14

Clean-condition validation:
0% flipped raw rows: 0
0% changed model labels: 0
0% changed dependent REC values: 0
0% semantic dataframe mismatches: 0
0% harmless dtype-different columns: 4

Severe-noise validation:
50% flipped raw rows: 156700
50% changed model labels: 30346
50% changed dependent REC values: 715259

Baseline protocol:
Random invariant across noise: True
QTF-Avg invariant across noise: True
LatestFail invariant across noise: False

Immutability:
Evaluation unchanged: True
Evaluation SHA-256: e6ab0de25c67c9f923685919d3bf8dc66092af5e7521ca874aa94d4fc7586e11
Source files unchanged: True

Runtime:
Smoke-test seconds: 167.27

Validation:
Checks: 23
Failed checks: 0

Smoke-test checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_09_smoke_test_ch

In [14]:
# ============================================================
# PROJECT 9 — CELL 9 / STEP 5A
# CHECKPOINTED FULL 270-CONDITION EXPERIMENT
#
# PROJECT: camunda@camunda-bpm-platform
#
# Runs:
#   9 noise levels
#   30 repetition seeds
#   270 conditions
#   4 ML models per condition
#   3 baselines per condition
#
# Per-condition raw artefacts:
#   1. rankings.parquet
#   2. build_metrics.csv
#   3. project_run.csv
#   4. model_fits.csv
#   5. condition_audit.csv
#   6. training_medians.csv
#   7. condition_summary.json
#   8. _SUCCESS.json
#
# Total expected raw files:
#   270 × 8 = 2160
#
# Resume behaviour:
# - validates every completed condition
# - skips conditions with valid _SUCCESS markers and hashes
# - reruns incomplete conditions
# - never modifies Projects 1–8
# - never modifies the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from IPython.display import display

import gc
import hashlib
import json
import os
import time
import traceback
import uuid
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9
PROJECT_NAME = "camunda@camunda-bpm-platform"
PROJECT_SLUG = "camunda__camunda-bpm-platform"
PROJECT_SHORT_NAME = "camunda"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_9_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_9_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3B_STATUS = (
    "PASS_PROJECT_9_NOISY_REC_ENGINE_SENTINEL_VALIDATED_AND_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_9_MODEL_PREDICTOR_BASELINE_AND_METRIC_PROTOCOL_FROZEN"
)

EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_9_TWO_CONDITION_END_TO_END_SMOKE_TEST_VALIDATED"
)

STEP5A_RUNNING_STATUS = (
    "RUNNING_PROJECT_9_FULL_270_CONDITION_EXPERIMENT"
)

STEP5A_PASS_STATUS = (
    "PASS_PROJECT_9_FULL_270_CONDITION_EXPERIMENT_COMPLETED_CHECKPOINTED"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "65201f596c631d0d7472868d3b23aabc86fa8f9ef4674b8ac9400fd577ab8f07"
)

EXPECTED_RAW_TRAINING_ROWS = 313984
EXPECTED_MODEL_TRAINING_ROWS = 60880
EXPECTED_MODEL_EVALUATION_ROWS = 18503

EXPECTED_EVALUATION_PERIOD_BUILDS = 206
EXPECTED_SCORED_EVALUATION_BUILDS = 30
EXPECTED_EVALUATION_FAILURES = 665

EXPECTED_ACTIVE_PREDICTORS = 151

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(range(1, 31))

EXPECTED_CONDITIONS = 270
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_BASELINES = 3
EXPECTED_TECHNIQUES = 7

EXPECTED_ML_FITS = (
    EXPECTED_CONDITIONS
    * EXPECTED_ML_TECHNIQUES
)

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVALUATION_ROWS
    * EXPECTED_TECHNIQUES
)

EXPECTED_BUILD_METRICS_PER_CONDITION = (
    EXPECTED_SCORED_EVALUATION_BUILDS
    * EXPECTED_TECHNIQUES
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    EXPECTED_TECHNIQUES
)

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRICS = (
    EXPECTED_CONDITIONS
    * EXPECTED_BUILD_METRICS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)

EXPECTED_FILES_PER_CONDITION = 8

EXPECTED_RAW_FILES = (
    EXPECTED_CONDITIONS
    * EXPECTED_FILES_PER_CONDITION
)

RECENT_WINDOW = 6

COMPARISON_RTOL = 1e-9
COMPARISON_ATOL = 1e-9

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


# ------------------------------------------------------------
# 2. REC FEATURES
# ------------------------------------------------------------

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

if len(VERDICT_DEPENDENT_REC_FEATURES) != 13:
    raise AssertionError(
        "Expected exactly 13 verdict-dependent REC features."
    )

if len(VERDICT_INDEPENDENT_REC_FEATURES) != 6:
    raise AssertionError(
        "Expected exactly six verdict-independent REC features."
    )

if (
    set(VERDICT_DEPENDENT_REC_FEATURES)
    | set(VERDICT_INDEPENDENT_REC_FEATURES)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "REC dependency classes do not cover all 19 REC features."
    )


# ------------------------------------------------------------
# 3. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_selection_checkpoint.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_rec_reconstruction_checkpoint.json"
)

NOISE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noise_plan_checkpoint.json"
)

NOISY_REC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_noisy_rec_engine_checkpoint.json"
)

MODEL_METRIC_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_model_metric_checkpoint.json"
)

SMOKE_TEST_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_smoke_test_checkpoint.json"
)

FULL_RUN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_full_run_checkpoint.json"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP2B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step2b_status.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3a_status.json"
)

STEP3B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step3b_status.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4a_status.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step4b_status.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step5a_status.json"
)

SCHEMA_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_schema_preflight"
)

STEP2A_REPORT_PATH = (
    SCHEMA_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

NOISE_PREFLIGHT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_noise_preflight"
)

FAILURE_SUBTYPE_DISTRIBUTION_PATH = (
    NOISE_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_failure_subtype_distribution.csv"
)

RAW_ROOT = (
    RESULTS_DIR
    / "Raw"
    / PROJECT_SLUG
)

RUN_CONTROL_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_control"
)

STAGING_DIR = (
    RUN_CONTROL_DIR
    / "staging"
)

RUN_PROGRESS_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_progress.csv"
)

RUN_PROGRESS_JSON_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_progress.json"
)

RUN_FAILURE_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_failure.json"
)

CONDITION_INVENTORY_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_condition_inventory.csv"
)

RAW_FILE_MANIFEST_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_raw_file_manifest.csv"
)

STEP5A_VALIDATION_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_step5a_validation.csv"
)

STEP5A_REPORT_PATH = (
    RUN_CONTROL_DIR
    / f"{PROJECT_SHORT_NAME}_step5a_report.json"
)


print("=" * 122)
print("=== PROJECT 9 CELL 9 / STEP 5A: CHECKPOINTED FULL 270-CONDITION EXPERIMENT ===")
print("=" * 122)


# ------------------------------------------------------------
# 4. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def staged_path(target_path):
    target_path = Path(target_path)

    STAGING_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    return (
        STAGING_DIR
        / (
            f"{uuid.uuid4().hex}"
            f"__{target_path.name}"
        )
    )


def atomic_write_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = staged_path(path)

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = staged_path(path)

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = staged_path(path)

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def dataframe_content_sha256(
    dataframe,
    include_index=True,
):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=include_index,
        categorize=True,
    ).to_numpy(dtype="<u8")

    digest = hashlib.sha256()

    digest.update(
        json.dumps(
            list(dataframe.columns),
            separators=(",", ":"),
        ).encode("utf-8")
    )

    digest.update(b"\0")

    digest.update(
        row_hashes.tobytes(
            order="C"
        )
    )

    return digest.hexdigest()


def packed_boolean_sha256(mask):
    packed = np.packbits(
        np.asarray(
            mask,
            dtype=np.uint8,
        ),
        bitorder="little",
    )

    return hashlib.sha256(
        packed.tobytes()
    ).hexdigest()


def integer_array_sha256(values):
    values = np.asarray(
        values,
        dtype="<i4",
    )

    return hashlib.sha256(
        values.tobytes(
            order="C"
        )
    ).hexdigest()


def canonical_identifier(series):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if float(numeric.notna().mean()) >= 0.95:
        rounded = numeric.round()

        integer_like = (
            numeric.isna()
            | np.isclose(
                numeric,
                rounded,
                rtol=0,
                atol=1e-9,
            )
        ).all()

        if integer_like:
            return (
                rounded
                .astype("Int64")
                .astype(str)
            )

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


def csv_data_row_count(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:
        line_count = sum(1 for _ in handle)

    return max(
        0,
        line_count - 1,
    )


# ------------------------------------------------------------
# 5. NOISE HELPERS
# ------------------------------------------------------------

def create_seed_random_streams(
    number_of_rows,
    repetition_seed,
    failure_subtypes,
    subtype_probabilities,
):
    flip_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "flip_mask",
    )

    subtype_stream_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "failure_subtype",
    )

    flip_rng = np.random.default_rng(
        flip_stream_seed
    )

    subtype_rng = np.random.default_rng(
        subtype_stream_seed
    )

    return {
        "FlipUniforms":
            flip_rng.random(
                number_of_rows
            ),

        "SampledFailureSubtypes":
            subtype_rng.choice(
                failure_subtypes,
                size=number_of_rows,
                replace=True,
                p=subtype_probabilities,
            ).astype(np.int32),
    }


def construct_condition_arrays(
    clean_verdicts,
    row_uniforms,
    sampled_failure_subtypes,
    noise_percent,
):
    flip_mask = (
        row_uniforms
        < float(noise_percent) / 100.0
    )

    pass_to_failure_mask = (
        flip_mask
        & (clean_verdicts == 0)
    )

    failure_to_pass_mask = (
        flip_mask
        & (clean_verdicts != 0)
    )

    noisy_verdicts = (
        clean_verdicts.copy()
    )

    noisy_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_verdicts[
        failure_to_pass_mask
    ] = 0

    return {
        "FlipMask":
            flip_mask,

        "PassToFailureMask":
            pass_to_failure_mask,

        "FailureToPassMask":
            failure_to_pass_mask,

        "NoisyVerdicts":
            noisy_verdicts,
    }


# ------------------------------------------------------------
# 6. METRIC HELPERS
# ------------------------------------------------------------

def calculate_apfd(actual_failures):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(failures)
    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "Failures and durations differ in length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain non-finite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations contain negative values."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.asarray([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + 0.5
        * durations[failure_mask]
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


# ------------------------------------------------------------
# 7. LOAD AND VALIDATE CHECKPOINTS
# ------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_CHECKPOINT_PATH,
    NOISY_REC_CHECKPOINT_PATH,
    MODEL_METRIC_CHECKPOINT_PATH,
    SMOKE_TEST_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
    STEP3A_STATUS_PATH,
    STEP3B_STATUS_PATH,
    STEP4A_STATUS_PATH,
    STEP4B_STATUS_PATH,
    STEP2A_REPORT_PATH,
    FAILURE_SUBTYPE_DISTRIBUTION_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )


selection_checkpoint = json.loads(
    SELECTION_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

rec_checkpoint = json.loads(
    REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noise_checkpoint = json.loads(
    NOISE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

noisy_rec_checkpoint = json.loads(
    NOISY_REC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

model_metric_checkpoint = json.loads(
    MODEL_METRIC_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

smoke_checkpoint = json.loads(
    SMOKE_TEST_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)

step2b_status = json.loads(
    STEP2B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3a_status = json.loads(
    STEP3A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step3b_status = json.loads(
    STEP3B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step4a_status = json.loads(
    STEP4A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step4b_status = json.loads(
    STEP4B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step2a_report = json.loads(
    STEP2A_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


status_checks = [
    (
        "Step 2B",
        step2b_status.get("Status"),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Step 3A",
        step3a_status.get("Status"),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Step 3B",
        step3b_status.get("Status"),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Step 4A",
        step4a_status.get("Status"),
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "Step 4B",
        step4b_status.get("Status"),
        EXPECTED_STEP4B_STATUS,
    ),
    (
        "REC checkpoint",
        rec_checkpoint.get("Status"),
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "Noise checkpoint",
        noise_checkpoint.get("Status"),
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "Noisy REC checkpoint",
        noisy_rec_checkpoint.get("Status"),
        EXPECTED_STEP3B_STATUS,
    ),
    (
        "Model/metric checkpoint",
        model_metric_checkpoint.get("Status"),
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "Smoke checkpoint",
        smoke_checkpoint.get("Status"),
        EXPECTED_STEP4B_STATUS,
    ),
]

for check_name, actual, expected in status_checks:
    if actual != expected:
        raise AssertionError(
            f"{check_name} status differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


if (
    selection_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise AssertionError(
        "Project 9 source-root SHA-256 differs."
    )


# ------------------------------------------------------------
# 8. REGISTRY VALIDATION
# ------------------------------------------------------------

registry_sha256_before = calculate_sha256(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_numbers = pd.to_numeric(
    registry["ProjectNumber"],
    errors="raise",
).astype(int)

if (
    len(registry) != 8
    or set(project_numbers)
    != set(range(1, 9))
):
    raise AssertionError(
        "Completion registry must contain exactly Projects 1–8."
    )

if project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise AssertionError(
        "Project 9 is already present in the completion registry."
    )


# ------------------------------------------------------------
# 9. RESOLVE FROZEN FILES
# ------------------------------------------------------------

raw_training_path = Path(
    noise_checkpoint[
        "RawTrainingHistory"
    ]
)

model_training_path = Path(
    noise_checkpoint[
        "ModelTrainingData"
    ]
)

model_evaluation_path = Path(
    noise_checkpoint[
        "ModelEvaluationData"
    ]
)

condition_plan_path = Path(
    noise_checkpoint[
        "NoiseConditionPlan"
    ]
)

clean_direct_rec_path = Path(
    rec_checkpoint[
        "CleanRECReconstructed"
    ]
)

anchor_offsets_path = Path(
    rec_checkpoint[
        "CleanRECAnchorOffsets"
    ]
)

build_entity_map_path = Path(
    rec_checkpoint[
        "BuildEntityMap"
    ]
)

active_predictors_path = Path(
    model_metric_checkpoint[
        "ActivePredictors"
    ]
)

model_configuration_path = Path(
    model_metric_checkpoint[
        "ModelConfiguration"
    ]
)

raw_evaluation_path = Path(
    model_metric_checkpoint[
        "RawEvaluationHistory"
    ]
)

evaluation_build_table_path = Path(
    model_metric_checkpoint[
        "EvaluationBuildTable"
    ]
)

evaluation_metric_cohort_path = Path(
    model_metric_checkpoint[
        "EvaluationMetricCohort"
    ]
)


frozen_input_hashes = {
    raw_training_path:
        noise_checkpoint[
            "RawTrainingHistorySHA256"
        ],

    model_training_path:
        noise_checkpoint[
            "ModelTrainingDataSHA256"
        ],

    model_evaluation_path:
        noise_checkpoint[
            "ModelEvaluationDataSHA256"
        ],

    condition_plan_path:
        noise_checkpoint[
            "NoiseConditionPlanSHA256"
        ],

    clean_direct_rec_path:
        rec_checkpoint[
            "CleanRECReconstructedSHA256"
        ],

    anchor_offsets_path:
        rec_checkpoint[
            "CleanRECAnchorOffsetsSHA256"
        ],

    build_entity_map_path:
        rec_checkpoint[
            "BuildEntityMapSHA256"
        ],

    active_predictors_path:
        model_metric_checkpoint[
            "ActivePredictorsSHA256"
        ],

    model_configuration_path:
        model_metric_checkpoint[
            "ModelConfigurationSHA256"
        ],

    raw_evaluation_path:
        model_metric_checkpoint[
            "RawEvaluationHistorySHA256"
        ],

    evaluation_build_table_path:
        model_metric_checkpoint[
            "EvaluationBuildTableSHA256"
        ],

    evaluation_metric_cohort_path:
        model_metric_checkpoint[
            "EvaluationMetricCohortSHA256"
        ],
}

for path, expected_hash in frozen_input_hashes.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Frozen input is missing:\n{path}"
        )

    actual_hash = calculate_sha256(path)

    if actual_hash != expected_hash:
        raise AssertionError(
            "Frozen input hash differs.\n"
            f"Path: {path}\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )


source_paths = {
    relative_name:
        Path(
            metadata["RuntimePath"]
        )
    for relative_name, metadata
    in selection_checkpoint[
        "SourceFiles"
    ].items()
}

source_hashes_before = {}

for relative_name, path in source_paths.items():
    expected_hash = (
        selection_checkpoint[
            "SourceFiles"
        ][relative_name]["SHA256"]
    )

    actual_hash = calculate_sha256(path)

    if actual_hash != expected_hash:
        raise AssertionError(
            f"Frozen source hash differs: {relative_name}"
        )

    source_hashes_before[
        relative_name
    ] = actual_hash


evaluation_sha256_before = calculate_sha256(
    model_evaluation_path
)


# ------------------------------------------------------------
# 10. LOAD CONFIGURATION
# ------------------------------------------------------------

active_predictor_payload = json.loads(
    active_predictors_path.read_text(
        encoding="utf-8"
    )
)

model_configuration = json.loads(
    model_configuration_path.read_text(
        encoding="utf-8"
    )
)

ACTIVE_FEATURE_COLUMNS = (
    active_predictor_payload[
        "ActivePredictors"
    ]
)

ZERO_VARIANCE_FEATURES = (
    active_predictor_payload[
        "ZeroVariancePredictors"
    ]
)

MODEL_CONFIG = (
    model_configuration[
        "Models"
    ]
)

if len(ACTIVE_FEATURE_COLUMNS) != EXPECTED_ACTIVE_PREDICTORS:
    raise AssertionError(
        "Active predictor count differs."
    )

if len(ZERO_VARIANCE_FEATURES) != 0:
    raise AssertionError(
        "Unexpected zero-variance predictors were recorded."
    )


resolved_columns = (
    step2a_report[
        "ResolvedColumns"
    ]
)

exe_duration_column = (
    resolved_columns[
        "ExecutionDuration"
    ]
)

dataset_build_column = (
    resolved_columns[
        "DatasetBuild"
    ]
)

dataset_test_column = (
    resolved_columns[
        "DatasetTest"
    ]
)

dataset_verdict_column = (
    resolved_columns[
        "DatasetVerdict"
    ]
)


# ------------------------------------------------------------
# 11. LOAD FROZEN DATA
# ------------------------------------------------------------

raw_training = (
    pd.read_parquet(
        raw_training_path
    )
    .sort_values(
        "NoiseRowID",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

clean_model_training = (
    pd.read_parquet(
        model_training_path
    )
    .reset_index(drop=True)
)

clean_model_evaluation = (
    pd.read_parquet(
        model_evaluation_path
    )
    .reset_index(drop=True)
)

condition_plan = (
    pd.read_csv(
        condition_plan_path,
        low_memory=False,
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

failure_subtype_distribution = (
    pd.read_csv(
        FAILURE_SUBTYPE_DISTRIBUTION_PATH,
        low_memory=False,
    )
)

clean_direct_rec = pd.read_parquet(
    clean_direct_rec_path
)

anchor_offsets = pd.read_parquet(
    anchor_offsets_path
)

build_entity_map = pd.read_csv(
    build_entity_map_path,
    compression="gzip",
    low_memory=False,
)

raw_evaluation = pd.read_parquet(
    raw_evaluation_path
)

evaluation_build_table = pd.read_csv(
    evaluation_build_table_path,
    low_memory=False,
)

evaluation_metric_cohort = pd.read_parquet(
    evaluation_metric_cohort_path
)


if len(raw_training) != EXPECTED_RAW_TRAINING_ROWS:
    raise AssertionError(
        "Raw training row count differs."
    )

if len(clean_model_training) != EXPECTED_MODEL_TRAINING_ROWS:
    raise AssertionError(
        "Model training row count differs."
    )

if len(clean_model_evaluation) != EXPECTED_MODEL_EVALUATION_ROWS:
    raise AssertionError(
        "Model evaluation row count differs."
    )

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Noise-condition count differs."
    )

if len(evaluation_build_table) != EXPECTED_EVALUATION_PERIOD_BUILDS:
    raise AssertionError(
        "Evaluation-period build count differs."
    )

if len(evaluation_metric_cohort) != EXPECTED_MODEL_EVALUATION_ROWS:
    raise AssertionError(
        "Evaluation metric-cohort row count differs."
    )


condition_pairs = set(
    zip(
        pd.to_numeric(
            condition_plan["NoisePercent"],
            errors="raise",
        ).astype(int),

        pd.to_numeric(
            condition_plan["RepetitionSeed"],
            errors="raise",
        ).astype(int),
    )
)

expected_condition_pairs = {
    (
        noise_percent,
        repetition_seed,
    )
    for repetition_seed
    in REPETITION_SEEDS
    for noise_percent
    in NOISE_LEVELS
}

if condition_pairs != expected_condition_pairs:
    raise AssertionError(
        "Frozen condition plan does not contain the expected "
        "9 noise levels × 30 seeds."
    )


# ------------------------------------------------------------
# 12. STANDARDISE RAW HISTORY
# ------------------------------------------------------------

if "Duration" in raw_training.columns:
    raw_training["Duration"] = pd.to_numeric(
        raw_training["Duration"],
        errors="raise",
    ).astype(float)

elif exe_duration_column in raw_training.columns:
    raw_training["Duration"] = pd.to_numeric(
        raw_training[
            exe_duration_column
        ],
        errors="raise",
    ).astype(float)

else:
    raise RuntimeError(
        "Raw training duration column was not found."
    )


if "Duration" in raw_evaluation.columns:
    raw_evaluation["Duration"] = pd.to_numeric(
        raw_evaluation["Duration"],
        errors="raise",
    ).astype(float)

elif exe_duration_column in raw_evaluation.columns:
    raw_evaluation["Duration"] = pd.to_numeric(
        raw_evaluation[
            exe_duration_column
        ],
        errors="raise",
    ).astype(float)

else:
    raise RuntimeError(
        "Raw evaluation duration column was not found."
    )


for frame_name, frame in [
    (
        "raw training",
        raw_training,
    ),
    (
        "raw evaluation",
        raw_evaluation,
    ),
]:
    duration_values = frame[
        "Duration"
    ].to_numpy(dtype=float)

    if not np.isfinite(
        duration_values
    ).all():
        raise AssertionError(
            f"{frame_name} durations contain non-finite values."
        )

    if (
        duration_values < 0
    ).any():
        raise AssertionError(
            f"{frame_name} durations contain negative values."
        )


raw_training["BuildKey"] = (
    raw_training["BuildKey"]
    .astype(str)
)

raw_training["TestKey"] = (
    raw_training["TestKey"]
    .astype(str)
)

raw_training["BuildOrder"] = pd.to_numeric(
    raw_training["BuildOrder"],
    errors="raise",
).astype(int)

raw_evaluation["BuildKey"] = (
    raw_evaluation["BuildKey"]
    .astype(str)
)

raw_evaluation["TestKey"] = (
    raw_evaluation["TestKey"]
    .astype(str)
)

raw_evaluation["BuildOrder"] = pd.to_numeric(
    raw_evaluation["BuildOrder"],
    errors="raise",
).astype(int)

raw_evaluation["CleanVerdict"] = pd.to_numeric(
    raw_evaluation["CleanVerdict"],
    errors="raise",
).astype(np.int32)

evaluation_build_table["BuildKey"] = (
    evaluation_build_table["BuildKey"]
    .astype(str)
)

evaluation_build_table["BuildOrder"] = pd.to_numeric(
    evaluation_build_table["BuildOrder"],
    errors="raise",
).astype(int)


expected_noise_row_ids = np.arange(
    EXPECTED_RAW_TRAINING_ROWS,
    dtype=np.int64,
)

if not np.array_equal(
    raw_training[
        "NoiseRowID"
    ].to_numpy(dtype=np.int64),
    expected_noise_row_ids,
):
    raise AssertionError(
        "Raw training NoiseRowID order differs."
    )


clean_raw_verdicts = pd.to_numeric(
    raw_training["CleanVerdict"],
    errors="raise",
).to_numpy(dtype=np.int32)


# ------------------------------------------------------------
# 13. ALIGN MODEL AND RAW TRAINING ROWS
# ------------------------------------------------------------

training_key_frame = pd.DataFrame({
    "_ModelRowOrder":
        np.arange(
            EXPECTED_MODEL_TRAINING_ROWS,
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            clean_model_training[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            clean_model_training[
                dataset_test_column
            ]
        ),
})


evaluation_key_frame = pd.DataFrame({
    "_ModelRowOrder":
        np.arange(
            EXPECTED_MODEL_EVALUATION_ROWS,
            dtype=np.int64,
        ),

    "BuildKey":
        canonical_identifier(
            clean_model_evaluation[
                dataset_build_column
            ]
        ),

    "TestKey":
        canonical_identifier(
            clean_model_evaluation[
                dataset_test_column
            ]
        ),
})


if training_key_frame.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Model training Build/Test keys are not unique."
    )


raw_key_manifest = raw_training[
    [
        "NoiseRowID",
        "BuildKey",
        "TestKey",
        "CleanVerdict",
    ]
].copy()


if raw_key_manifest.duplicated(
    subset=[
        "BuildKey",
        "TestKey",
    ],
    keep=False,
).any():
    raise AssertionError(
        "Raw training Build/Test keys are not unique."
    )


model_raw_alignment = (
    training_key_frame.merge(
        raw_key_manifest,
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if model_raw_alignment[
    "_merge"
].ne("both").any():
    raise AssertionError(
        "Some model-training rows lack raw history rows."
    )


model_noise_row_ids = (
    model_raw_alignment[
        "NoiseRowID"
    ].to_numpy(dtype=np.int64)
)


clean_model_verdicts = pd.to_numeric(
    clean_model_training[
        dataset_verdict_column
    ],
    errors="raise",
).to_numpy(dtype=np.int32)


aligned_raw_clean_verdicts = pd.to_numeric(
    model_raw_alignment[
        "CleanVerdict"
    ],
    errors="raise",
).to_numpy(dtype=np.int32)


if not np.array_equal(
    clean_model_verdicts,
    aligned_raw_clean_verdicts,
):
    raise AssertionError(
        "Clean raw/model training verdicts differ."
    )


# ------------------------------------------------------------
# 14. ALIGN DIRECT REC AND ANCHORS
# ------------------------------------------------------------

for dataframe in [
    clean_direct_rec,
    anchor_offsets,
]:
    dataframe["BuildKey"] = (
        dataframe["BuildKey"]
        .astype(str)
    )

    dataframe["TestKey"] = (
        dataframe["TestKey"]
        .astype(str)
    )


clean_direct_training = (
    training_key_frame.merge(
        clean_direct_rec[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


anchor_training = (
    training_key_frame.merge(
        anchor_offsets[
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ],
        on=[
            "BuildKey",
            "TestKey",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )
    .sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if clean_direct_training[
    "_merge"
].ne("both").any():
    raise AssertionError(
        "Clean direct REC rows are missing."
    )

if anchor_training[
    "_merge"
].ne("both").any():
    raise AssertionError(
        "Clean REC anchor rows are missing."
    )


clean_direct_dependent_matrix = (
    clean_direct_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(dtype=float)
)

anchor_dependent_matrix = (
    anchor_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .to_numpy(dtype=float)
)

original_dependent_matrix = (
    clean_model_training[
        VERDICT_DEPENDENT_REC_FEATURES
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .to_numpy(dtype=float)
)


if not np.isclose(
    clean_direct_dependent_matrix
    + anchor_dependent_matrix,
    original_dependent_matrix,
    rtol=COMPARISON_RTOL,
    atol=COMPARISON_ATOL,
    equal_nan=False,
).all():
    raise AssertionError(
        "Clean direct REC plus anchor does not reproduce "
        "the frozen clean REC features."
    )


requested_pairs_by_test = {
    str(test_key):
        set(
            group["BuildKey"]
            .astype(str)
        )
    for test_key, group
    in training_key_frame.groupby(
        "TestKey",
        sort=False,
    )
}


# ------------------------------------------------------------
# 15. BUILD/ENTITY STRUCTURE
# ------------------------------------------------------------

build_entity_map["BuildKey"] = (
    build_entity_map["BuildKey"]
    .astype(str)
)

build_entity_map["EntityId"] = pd.to_numeric(
    build_entity_map["EntityId"],
    errors="raise",
).astype(int)


changed_entities_by_build = {
    str(build_key):
        set(
            group["EntityId"]
            .astype(int)
        )
    for build_key, group
    in build_entity_map.groupby(
        "BuildKey",
        sort=False,
    )
}


entity_changed_builds = {
    int(entity_id):
        set(
            group["BuildKey"]
            .astype(str)
        )
    for entity_id, group
    in build_entity_map.groupby(
        "EntityId",
        sort=False,
    )
}


# ------------------------------------------------------------
# 16. FAILURE SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

failure_subtypes = pd.to_numeric(
    failure_subtype_distribution[
        "FailureSubtype"
    ],
    errors="raise",
).to_numpy(dtype=np.int32)

subtype_probabilities = pd.to_numeric(
    failure_subtype_distribution[
        "SamplingProbability"
    ],
    errors="raise",
).to_numpy(dtype=float)

if not np.isclose(
    subtype_probabilities.sum(),
    1.0,
    rtol=0,
    atol=1e-12,
):
    raise AssertionError(
        "Failure subtype probabilities do not sum to one."
    )


# ------------------------------------------------------------
# 17. NUMERIC MODEL MATRICES
# ------------------------------------------------------------

clean_training_numeric_base = (
    clean_model_training[
        ACTIVE_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)

evaluation_numeric_base = (
    clean_model_evaluation[
        ACTIVE_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


unaffected_feature_columns = [
    column
    for column in ACTIVE_FEATURE_COLUMNS
    if column not in VERDICT_DEPENDENT_REC_FEATURES
]

unaffected_training_hash = (
    dataframe_content_sha256(
        clean_training_numeric_base[
            unaffected_feature_columns
        ],
        include_index=True,
    )
)

clean_training_numeric_hash_before = (
    dataframe_content_sha256(
        clean_training_numeric_base,
        include_index=True,
    )
)


# ------------------------------------------------------------
# 18. EVALUATION RANKING STRUCTURE
# ------------------------------------------------------------

if "_ModelRowOrder" not in evaluation_metric_cohort.columns:
    raise AssertionError(
        "Evaluation metric cohort lacks _ModelRowOrder."
    )


evaluation_metric_cohort = (
    evaluation_metric_cohort.sort_values(
        "_ModelRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

evaluation_metric_cohort["BuildKey"] = (
    evaluation_metric_cohort["BuildKey"]
    .astype(str)
)

evaluation_metric_cohort["TestKey"] = (
    evaluation_metric_cohort["TestKey"]
    .astype(str)
)


if not evaluation_key_frame[
    [
        "BuildKey",
        "TestKey",
    ]
].reset_index(drop=True).equals(
    evaluation_metric_cohort[
        [
            "BuildKey",
            "TestKey",
        ]
    ].reset_index(drop=True)
):
    raise AssertionError(
        "Model evaluation and metric-cohort row order differ."
    )


evaluation_rank_data = pd.DataFrame({
    "Build":
        evaluation_metric_cohort[
            dataset_build_column
        ].to_numpy(),

    "Test":
        evaluation_metric_cohort[
            dataset_test_column
        ].to_numpy(),

    "Verdict":
        pd.to_numeric(
            evaluation_metric_cohort[
                dataset_verdict_column
            ],
            errors="raise",
        ).to_numpy(dtype=np.int32),

    "Duration":
        pd.to_numeric(
            evaluation_metric_cohort[
                "Duration"
            ],
            errors="raise",
        ).to_numpy(dtype=float),

    "BuildOrder":
        pd.to_numeric(
            evaluation_metric_cohort[
                "BuildOrder"
            ],
            errors="raise",
        ).to_numpy(dtype=int),

    "BuildKey":
        evaluation_metric_cohort[
            "BuildKey"
        ].astype(str).to_numpy(),

    "TestKey":
        evaluation_metric_cohort[
            "TestKey"
        ].astype(str).to_numpy(),

    "ActualFailure":
        pd.to_numeric(
            evaluation_metric_cohort[
                "ActualFailure"
            ],
            errors="raise",
        ).to_numpy(dtype=np.int32),
})


if int(
    evaluation_rank_data[
        "ActualFailure"
    ].sum()
) != EXPECTED_EVALUATION_FAILURES:
    raise AssertionError(
        "Evaluation failure count differs."
    )

if evaluation_rank_data[
    "BuildKey"
].nunique() != EXPECTED_SCORED_EVALUATION_BUILDS:
    raise AssertionError(
        "Scored evaluation-build count differs."
    )


evaluation_rank_by_build = {
    str(build_key):
        rows.copy()
    for build_key, rows
    in evaluation_rank_data.groupby(
        "BuildKey",
        sort=False,
    )
}

target_evaluation_build_keys = set(
    evaluation_rank_by_build
)


raw_evaluation_by_build = {
    str(build_key):
        rows.copy()
    for build_key, rows
    in raw_evaluation.groupby(
        "BuildKey",
        sort=False,
    )
}


ordered_evaluation_builds = (
    evaluation_build_table.sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 19. REC RECONSTRUCTION ENGINE
# ------------------------------------------------------------

def calculate_max_test_file_rate(
    target_builds,
    current_changed_entities,
):
    if len(target_builds) == 0:
        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )

        overlap_count = len(
            changed_builds.intersection(
                target_builds
            )
        )

        if overlap_count > maximum_frequency:
            maximum_frequency = overlap_count

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_noisy_dependent_rec(
    noisy_verdicts,
):
    noisy_history = raw_training[
        [
            "BuildKey",
            "TestKey",
            "BuildOrder",
        ]
    ].copy()

    noisy_history[
        "NoisyVerdict"
    ] = np.asarray(
        noisy_verdicts,
        dtype=np.int32,
    )

    reconstructed_records = []

    for test_key, test_history in (
        noisy_history.groupby(
            "TestKey",
            sort=False,
        )
    ):
        test_key = str(test_key)

        requested_builds = (
            requested_pairs_by_test.get(
                test_key
            )
        )

        if not requested_builds:
            continue

        test_history = (
            test_history.sort_values(
                "BuildOrder",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        prior_count = 0

        prior_failure_count = 0
        prior_assertion_count = 0
        prior_exception_count = 0
        prior_transition_count = 0

        last_failure_position = None
        last_transition_position = None
        previous_verdict = None

        recent_history = deque(
            maxlen=RECENT_WINDOW
        )

        failure_builds = set()
        transition_builds = set()

        for row in test_history.itertuples(
            index=False
        ):
            current_build = str(
                row.BuildKey
            )

            current_verdict = int(
                row.NoisyVerdict
            )

            if current_build in requested_builds:
                record = {
                    "BuildKey":
                        current_build,

                    "TestKey":
                        test_key,
                }

                if prior_count == 0:
                    for feature in (
                        VERDICT_DEPENDENT_REC_FEATURES
                    ):
                        record[feature] = -1.0

                else:
                    if last_failure_position is None:
                        last_failure_age = -1.0
                    else:
                        last_failure_age = float(
                            prior_count
                            - 1
                            - last_failure_position
                        )

                    if last_transition_position is None:
                        last_transition_age = -1.0
                    else:
                        last_transition_age = float(
                            prior_count
                            - 1
                            - last_transition_position
                        )

                    recent_rows = list(
                        recent_history
                    )

                    recent_length = len(
                        recent_rows
                    )

                    if recent_length == 0:
                        raise AssertionError(
                            "Recent history is unexpectedly empty."
                        )

                    recent_verdicts = np.asarray(
                        [
                            item["Verdict"]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )

                    recent_transitions = np.asarray(
                        [
                            item["Transition"]
                            for item in recent_rows
                        ],
                        dtype=np.int32,
                    )

                    current_changed_entities = (
                        changed_entities_by_build.get(
                            current_build,
                            set(),
                        )
                    )

                    record.update({
                        "REC_LastFailureAge":
                            last_failure_age,

                        "REC_LastTransitionAge":
                            last_transition_age,

                        "REC_RecentFailRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts != 0
                                )
                                / recent_length
                            ),

                        "REC_RecentAssertRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts == 2
                                )
                                / recent_length
                            ),

                        "REC_RecentExcRate":
                            float(
                                np.count_nonzero(
                                    recent_verdicts == 1
                                )
                                / recent_length
                            ),

                        "REC_RecentTransitionRate":
                            float(
                                np.count_nonzero(
                                    recent_transitions == 1
                                )
                                / recent_length
                            ),

                        "REC_TotalFailRate":
                            float(
                                prior_failure_count
                                / prior_count
                            ),

                        "REC_TotalAssertRate":
                            float(
                                prior_assertion_count
                                / prior_count
                            ),

                        "REC_TotalExcRate":
                            float(
                                prior_exception_count
                                / prior_count
                            ),

                        "REC_TotalTransitionRate":
                            float(
                                prior_transition_count
                                / prior_count
                            ),

                        "REC_LastVerdict":
                            float(
                                previous_verdict
                            ),

                        "REC_MaxTestFileFailRate":
                            calculate_max_test_file_rate(
                                failure_builds,
                                current_changed_entities,
                            ),

                        "REC_MaxTestFileTransitionRate":
                            calculate_max_test_file_rate(
                                transition_builds,
                                current_changed_entities,
                            ),
                    })

                reconstructed_records.append(
                    record
                )

            current_transition = (
                0
                if previous_verdict is None
                else int(
                    current_verdict
                    != previous_verdict
                )
            )

            if current_verdict != 0:
                prior_failure_count += 1
                last_failure_position = prior_count
                failure_builds.add(
                    current_build
                )

            if current_verdict == 2:
                prior_assertion_count += 1

            if current_verdict == 1:
                prior_exception_count += 1

            if current_transition == 1:
                prior_transition_count += 1
                last_transition_position = prior_count
                transition_builds.add(
                    current_build
                )

            recent_history.append({
                "Verdict":
                    current_verdict,

                "Transition":
                    current_transition,
            })

            previous_verdict = current_verdict
            prior_count += 1

    return pd.DataFrame(
        reconstructed_records,
        columns=(
            [
                "BuildKey",
                "TestKey",
            ]
            + VERDICT_DEPENDENT_REC_FEATURES
        ),
    )


def align_noisy_rec(
    noisy_direct_rec,
):
    if len(
        noisy_direct_rec
    ) != EXPECTED_MODEL_TRAINING_ROWS:
        raise AssertionError(
            "Noisy REC reconstruction row count differs."
        )

    if noisy_direct_rec.duplicated(
        subset=[
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).any():
        raise AssertionError(
            "Noisy REC reconstruction contains duplicate rows."
        )

    aligned = (
        training_key_frame.merge(
            noisy_direct_rec,
            on=[
                "BuildKey",
                "TestKey",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )
        .sort_values(
            "_ModelRowOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if aligned[
        "_merge"
    ].ne("both").any():
        raise AssertionError(
            "Noisy REC rows are missing after alignment."
        )

    direct_matrix = aligned[
        VERDICT_DEPENDENT_REC_FEATURES
    ].to_numpy(dtype=float)

    anchored_matrix = (
        direct_matrix
        + anchor_dependent_matrix
    )

    if not np.isfinite(
        anchored_matrix
    ).all():
        raise AssertionError(
            "Noisy anchored REC matrix contains non-finite values."
        )

    return anchored_matrix


# ------------------------------------------------------------
# 20. MODEL HELPERS
# ------------------------------------------------------------

def create_ml_models(
    repetition_seed,
):
    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )

    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )

    lgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )

    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                var_smoothing=(
                    MODEL_CONFIG[
                        "NaiveBayes"
                    ][
                        "var_smoothing"
                    ]
                )
            ),
    }


def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = (
        fitted_model.predict_proba(
            feature_matrix
        )
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.where(
        classes == 1
    )[0]

    if len(positive_positions) != 1:
        raise AssertionError(
            "Could not identify fitted failure class 1."
        )

    output = probabilities[
        :,
        positive_positions[0],
    ]

    if not np.isfinite(output).all():
        raise AssertionError(
            "Failure probabilities contain non-finite values."
        )

    if not (
        (
            output >= 0.0
        )
        &
        (
            output <= 1.0
        )
    ).all():
        raise AssertionError(
            "Failure probabilities fall outside [0, 1]."
        )

    return output


# ------------------------------------------------------------
# 21. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction,
):
    ranked = build_rows[
        [
            "Build",
            "Test",
            "Verdict",
            "Duration",
            "BuildOrder",
            "BuildKey",
            "TestKey",
            "ActualFailure",
        ]
    ].copy().reset_index(drop=True)

    scores = np.asarray(
        scores,
        dtype=float,
    )

    if len(scores) != len(ranked):
        raise ValueError(
            "Ranking score count differs from row count."
        )

    if np.isnan(scores).any():
        raise ValueError(
            "Ranking scores contain NaN values."
        )

    if np.isneginf(scores).any():
        raise ValueError(
            "Ranking scores contain negative infinity."
        )

    if (
        np.isposinf(scores).any()
        and technique != "QTF-Avg"
    ):
        raise ValueError(
            "Only QTF-Avg may contain positive infinity."
        )

    ranked["Technique"] = technique
    ranked["Score"] = scores

    numeric_tests = pd.to_numeric(
        ranked["Test"],
        errors="coerce",
    )

    if numeric_tests.notna().all():
        ranked["_TestTieKey"] = (
            numeric_tests
        )
    else:
        ranked["_TestTieKey"] = (
            ranked["Test"]
            .astype(str)
        )

    if score_direction == "descending":
        ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":
        ascending = [
            True,
            True,
        ]

    else:
        raise ValueError(
            "Invalid score direction."
        )

    ranked = (
        ranked.sort_values(
            [
                "Score",
                "_TestTieKey",
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .drop(
            columns=[
                "_TestTieKey",
            ]
        )
        .reset_index(drop=True)
    )

    ranked["Rank"] = np.arange(
        1,
        len(ranked) + 1,
    )

    return ranked


def create_ml_rankings(
    technique,
    failure_probabilities,
):
    scored = evaluation_rank_data.copy()

    scored["_FailureProbability"] = np.asarray(
        failure_probabilities,
        dtype=float,
    )

    frames = []

    for _, build_rows in scored.groupby(
        "BuildKey",
        sort=False,
    ):
        frames.append(
            rank_build_rows(
                build_rows,
                build_rows[
                    "_FailureProbability"
                ].to_numpy(dtype=float),
                technique,
                "descending",
            )
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


random_ranking_cache = {}


def create_random_rankings(
    repetition_seed,
):
    if repetition_seed in random_ranking_cache:
        return random_ranking_cache[
            repetition_seed
        ].copy()

    frames = []

    for build_key, build_rows in (
        evaluation_rank_data.groupby(
            "BuildKey",
            sort=False,
        )
    ):
        rng = np.random.default_rng(
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                (
                    "Random_baseline_build_"
                    f"{build_key}"
                ),
            )
        )

        scores = rng.random(
            len(build_rows)
        )

        frames.append(
            rank_build_rows(
                build_rows,
                scores,
                "Random",
                "descending",
            )
        )

    rankings = pd.concat(
        frames,
        ignore_index=True,
    )

    random_ranking_cache[
        repetition_seed
    ] = rankings.copy()

    return rankings


# ------------------------------------------------------------
# 22. QTF-AVG BASELINE — PRECOMPUTED ONCE
# ------------------------------------------------------------

def create_qtf_rankings():
    duration_sum = {}
    duration_count = {}

    for row in raw_training.sort_values(
        [
            "BuildOrder",
            "TestKey",
        ],
        kind="mergesort",
    ).itertuples(index=False):
        test_key = str(row.TestKey)
        duration = float(row.Duration)

        duration_sum[test_key] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )

        duration_count[test_key] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )

    qtf_frames = []
    unseen_scores = 0

    for build_row in (
        ordered_evaluation_builds.itertuples(
            index=False
        )
    ):
        build_key = str(
            build_row.BuildKey
        )

        if build_key in target_evaluation_build_keys:
            build_tests = (
                evaluation_rank_by_build[
                    build_key
                ].copy()
            )

            scores = []

            for test_key in (
                build_tests["TestKey"]
                .astype(str)
            ):
                count = duration_count.get(
                    test_key,
                    0,
                )

                if count > 0:
                    score = (
                        duration_sum[test_key]
                        / count
                    )
                else:
                    score = np.inf
                    unseen_scores += 1

                scores.append(float(score))

            qtf_frames.append(
                rank_build_rows(
                    build_tests,
                    scores,
                    "QTF-Avg",
                    "ascending",
                )
            )

        current_rows = (
            raw_evaluation_by_build.get(
                build_key
            )
        )

        if current_rows is None:
            continue

        for row in current_rows.itertuples(
            index=False
        ):
            test_key = str(
                row.TestKey
            )

            duration = float(
                row.Duration
            )

            duration_sum[test_key] = (
                duration_sum.get(
                    test_key,
                    0.0,
                )
                + duration
            )

            duration_count[test_key] = (
                duration_count.get(
                    test_key,
                    0,
                )
                + 1
            )

    return (
        pd.concat(
            qtf_frames,
            ignore_index=True,
        ),
        unseen_scores,
    )


print("\nPrecomputing invariant QTF-Avg baseline...")

QTF_RANKINGS, QTF_UNSEEN_SCORES = (
    create_qtf_rankings()
)

if len(QTF_RANKINGS) != EXPECTED_MODEL_EVALUATION_ROWS:
    raise AssertionError(
        "QTF-Avg ranking row count differs."
    )

print(
    "QTF-Avg rows:",
    len(QTF_RANKINGS),
)

print(
    "QTF-Avg unseen scores:",
    QTF_UNSEEN_SCORES,
)


# ------------------------------------------------------------
# 23. LATESTFAIL BASELINE
# ------------------------------------------------------------

def create_latest_fail_rankings(
    noisy_raw_verdicts,
):
    latest_failure_order = {}

    training_for_latest = raw_training[
        [
            "BuildOrder",
            "TestKey",
        ]
    ].copy()

    training_for_latest[
        "NoisyVerdict"
    ] = np.asarray(
        noisy_raw_verdicts,
        dtype=np.int32,
    )

    training_for_latest = (
        training_for_latest.sort_values(
            [
                "BuildOrder",
                "TestKey",
            ],
            kind="mergesort",
        )
    )

    for row in training_for_latest.itertuples(
        index=False
    ):
        if int(row.NoisyVerdict) != 0:
            latest_failure_order[
                str(row.TestKey)
            ] = int(row.BuildOrder)

    latest_frames = []

    for build_row in (
        ordered_evaluation_builds.itertuples(
            index=False
        )
    ):
        build_key = str(
            build_row.BuildKey
        )

        if build_key in target_evaluation_build_keys:
            build_tests = (
                evaluation_rank_by_build[
                    build_key
                ].copy()
            )

            scores = [
                float(
                    latest_failure_order.get(
                        str(test_key),
                        -1,
                    )
                )
                for test_key
                in build_tests[
                    "TestKey"
                ].astype(str)
            ]

            latest_frames.append(
                rank_build_rows(
                    build_tests,
                    scores,
                    "LatestFail",
                    "descending",
                )
            )

        current_rows = (
            raw_evaluation_by_build.get(
                build_key
            )
        )

        if current_rows is None:
            continue

        for row in current_rows.itertuples(
            index=False
        ):
            if int(row.CleanVerdict) != 0:
                latest_failure_order[
                    str(row.TestKey)
                ] = int(row.BuildOrder)

    rankings = pd.concat(
        latest_frames,
        ignore_index=True,
    )

    if len(rankings) != EXPECTED_MODEL_EVALUATION_ROWS:
        raise AssertionError(
            "LatestFail ranking row count differs."
        )

    return rankings


# ------------------------------------------------------------
# 24. METRIC AGGREGATION
# ------------------------------------------------------------

def calculate_build_metrics(
    rankings,
    noise_percent,
    repetition_seed,
):
    records = []

    for (
        technique,
        build_key,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):
        ranked_build = (
            ranked_build.sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        actual_failures = (
            ranked_build[
                "ActualFailure"
            ].to_numpy(dtype=int)
        )

        durations = (
            ranked_build[
                "Duration"
            ].to_numpy(dtype=float)
        )

        records.append({
            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "NoisePercent":
                int(noise_percent),

            "RepetitionSeed":
                int(repetition_seed),

            "Technique":
                technique,

            "Build":
                ranked_build[
                    "Build"
                ].iloc[0],

            "BuildKey":
                str(build_key),

            "BuildOrder":
                int(
                    ranked_build[
                        "BuildOrder"
                    ].iloc[0]
                ),

            "NumberOfTests":
                int(
                    len(ranked_build)
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "APFD":
                calculate_apfd(
                    actual_failures
                ),

            "APFDc":
                calculate_apfdc(
                    actual_failures,
                    durations,
                ),
        })

    return pd.DataFrame(records)


def aggregate_project_run(
    build_metrics,
):
    return (
        build_metrics.groupby(
            [
                "Project",
                "ProjectSlug",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            ScoredEvaluationBuilds=(
                "BuildKey",
                "nunique",
            ),

            EvaluationRows=(
                "NumberOfTests",
                "sum",
            ),

            EvaluationFailures=(
                "NumberOfFailures",
                "sum",
            ),

            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SDAPFD=(
                "APFD",
                "std",
            ),

            SDAPFDc=(
                "APFDc",
                "std",
            ),
        )
    )


def validate_condition_outputs(
    rankings,
    build_metrics,
    project_run,
    fit_log,
):
    if set(
        rankings["Technique"].unique()
    ) != set(ALL_TECHNIQUES):
        raise AssertionError(
            "Condition technique set differs."
        )

    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise AssertionError(
            "Condition ranking-row count differs."
        )

    if len(build_metrics) != EXPECTED_BUILD_METRICS_PER_CONDITION:
        raise AssertionError(
            "Condition build-metric row count differs."
        )

    if len(project_run) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise AssertionError(
            "Condition project-run row count differs."
        )

    if len(fit_log) != EXPECTED_ML_TECHNIQUES:
        raise AssertionError(
            "Condition model-fit count differs."
        )

    if not fit_log["FitSuccess"].all():
        raise AssertionError(
            "One or more model fits failed."
        )

    if not fit_log[
        "ProbabilitySuccess"
    ].all():
        raise AssertionError(
            "One or more probability generations failed."
        )

    if rankings.duplicated(
        subset=[
            "Technique",
            "BuildKey",
            "TestKey",
        ],
        keep=False,
    ).any():
        raise AssertionError(
            "Condition rankings contain duplicate tests."
        )

    for (
        technique,
        build_key,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "BuildKey",
        ],
        sort=False,
    ):
        expected_ranks = np.arange(
            1,
            len(ranked_build) + 1,
        )

        actual_ranks = np.sort(
            ranked_build[
                "Rank"
            ].to_numpy(dtype=int)
        )

        if not np.array_equal(
            expected_ranks,
            actual_ranks,
        ):
            raise AssertionError(
                "Invalid rank sequence.\n"
                f"Technique: {technique}\n"
                f"Build: {build_key}"
            )

    if build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():
        raise AssertionError(
            "Condition metrics contain missing values."
        )

    for metric_column in [
        "APFD",
        "APFDc",
    ]:
        values = build_metrics[
            metric_column
        ].to_numpy(dtype=float)

        if not (
            (
                values >= 0.0
            )
            &
            (
                values <= 1.0
            )
        ).all():
            raise AssertionError(
                f"{metric_column} falls outside [0, 1]."
            )

    if not project_run[
        "ScoredEvaluationBuilds"
    ].eq(
        EXPECTED_SCORED_EVALUATION_BUILDS
    ).all():
        raise AssertionError(
            "Project-run scored-build count differs."
        )

    if not project_run[
        "EvaluationRows"
    ].eq(
        EXPECTED_MODEL_EVALUATION_ROWS
    ).all():
        raise AssertionError(
            "Project-run evaluation-row count differs."
        )

    if not project_run[
        "EvaluationFailures"
    ].eq(
        EXPECTED_EVALUATION_FAILURES
    ).all():
        raise AssertionError(
            "Project-run evaluation-failure count differs."
        )


# ------------------------------------------------------------
# 25. CONDITION FILE HELPERS
# ------------------------------------------------------------

CONDITION_FILENAMES = [
    "rankings.parquet",
    "build_metrics.csv",
    "project_run.csv",
    "model_fits.csv",
    "condition_audit.csv",
    "training_medians.csv",
    "condition_summary.json",
    "_SUCCESS.json",
]


def condition_directory(
    condition_key,
):
    return (
        RAW_ROOT
        / condition_key
    )


def condition_paths(
    condition_key,
):
    directory = condition_directory(
        condition_key
    )

    return {
        "Directory":
            directory,

        "Rankings":
            directory
            / "rankings.parquet",

        "BuildMetrics":
            directory
            / "build_metrics.csv",

        "ProjectRun":
            directory
            / "project_run.csv",

        "ModelFits":
            directory
            / "model_fits.csv",

        "ConditionAudit":
            directory
            / "condition_audit.csv",

        "TrainingMedians":
            directory
            / "training_medians.csv",

        "ConditionSummary":
            directory
            / "condition_summary.json",

        "Success":
            directory
            / "_SUCCESS.json",
    }


def validate_completed_condition(
    condition_row,
    verify_hashes=True,
):
    condition_key = str(
        condition_row.ConditionKey
    )

    expected_noise = int(
        condition_row.NoisePercent
    )

    expected_seed = int(
        condition_row.RepetitionSeed
    )

    paths = condition_paths(
        condition_key
    )

    if not paths[
        "Success"
    ].exists():
        return (
            False,
            "MISSING_SUCCESS_MARKER",
        )

    try:
        success_payload = json.loads(
            paths[
                "Success"
            ].read_text(
                encoding="utf-8"
            )
        )

        if (
            success_payload.get("Status")
            != "PASS_FULL_CONDITION"
        ):
            return (
                False,
                "SUCCESS_STATUS_DIFFERS",
            )

        if (
            success_payload.get(
                "ConditionKey"
            )
            != condition_key
        ):
            return (
                False,
                "SUCCESS_CONDITION_KEY_DIFFERS",
            )

        if int(
            success_payload.get(
                "NoisePercent"
            )
        ) != expected_noise:
            return (
                False,
                "SUCCESS_NOISE_DIFFERS",
            )

        if int(
            success_payload.get(
                "RepetitionSeed"
            )
        ) != expected_seed:
            return (
                False,
                "SUCCESS_SEED_DIFFERS",
            )

        for filename in CONDITION_FILENAMES:
            if not (
                paths["Directory"]
                / filename
            ).exists():
                return (
                    False,
                    f"MISSING_FILE:{filename}",
                )

        actual_filenames = sorted(
            path.name
            for path in paths[
                "Directory"
            ].iterdir()
            if path.is_file()
        )

        if actual_filenames != sorted(
            CONDITION_FILENAMES
        ):
            return (
                False,
                (
                    "CONDITION_FILE_SET_DIFFERS:"
                    + str(actual_filenames)
                ),
            )

        summary = json.loads(
            paths[
                "ConditionSummary"
            ].read_text(
                encoding="utf-8"
            )
        )

        expected_summary_values = {
            "Status":
                "PASS_FULL_CONDITION",

            "ConditionKey":
                condition_key,

            "NoisePercent":
                expected_noise,

            "RepetitionSeed":
                expected_seed,

            "MLFits":
                EXPECTED_ML_TECHNIQUES,

            "RankingRows":
                EXPECTED_RANKING_ROWS_PER_CONDITION,

            "BuildMetricRows":
                EXPECTED_BUILD_METRICS_PER_CONDITION,

            "ProjectRunRows":
                EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        }

        for key, expected_value in (
            expected_summary_values.items()
        ):
            if summary.get(key) != expected_value:
                return (
                    False,
                    f"SUMMARY_{key}_DIFFERS",
                )

        if (
            pq.ParquetFile(
                paths["Rankings"]
            ).metadata.num_rows
            != EXPECTED_RANKING_ROWS_PER_CONDITION
        ):
            return (
                False,
                "RANKING_PARQUET_ROWS_DIFFER",
            )

        if (
            csv_data_row_count(
                paths["BuildMetrics"]
            )
            != EXPECTED_BUILD_METRICS_PER_CONDITION
        ):
            return (
                False,
                "BUILD_METRIC_ROWS_DIFFER",
            )

        if (
            csv_data_row_count(
                paths["ProjectRun"]
            )
            != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
        ):
            return (
                False,
                "PROJECT_RUN_ROWS_DIFFER",
            )

        if (
            csv_data_row_count(
                paths["ModelFits"]
            )
            != EXPECTED_ML_TECHNIQUES
        ):
            return (
                False,
                "MODEL_FIT_ROWS_DIFFER",
            )

        if (
            csv_data_row_count(
                paths["ConditionAudit"]
            )
            != 1
        ):
            return (
                False,
                "CONDITION_AUDIT_ROWS_DIFFER",
            )

        if (
            csv_data_row_count(
                paths["TrainingMedians"]
            )
            != EXPECTED_ACTIVE_PREDICTORS
        ):
            return (
                False,
                "TRAINING_MEDIAN_ROWS_DIFFER",
            )

        if verify_hashes:
            hash_key_map = {
                "RankingsSHA256":
                    paths["Rankings"],

                "BuildMetricsSHA256":
                    paths["BuildMetrics"],

                "ProjectRunSHA256":
                    paths["ProjectRun"],

                "ModelFitsSHA256":
                    paths["ModelFits"],

                "ConditionAuditSHA256":
                    paths["ConditionAudit"],

                "TrainingMediansSHA256":
                    paths["TrainingMedians"],

                "ConditionSummarySHA256":
                    paths["ConditionSummary"],
            }

            for hash_key, path in (
                hash_key_map.items()
            ):
                expected_hash = (
                    success_payload.get(
                        hash_key
                    )
                )

                if not expected_hash:
                    return (
                        False,
                        f"MISSING_HASH:{hash_key}",
                    )

                if (
                    calculate_sha256(path)
                    != expected_hash
                ):
                    return (
                        False,
                        f"HASH_DIFFERS:{path.name}",
                    )

        return (
            True,
            "VALID_COMPLETED_CONDITION",
        )

    except Exception as error:
        return (
            False,
            (
                f"VALIDATION_EXCEPTION:"
                f"{type(error).__name__}:"
                f"{error}"
            ),
        )


# ------------------------------------------------------------
# 26. RUN ONE CONDITION
# ------------------------------------------------------------

seed_stream_cache = {}


def get_seed_streams(
    repetition_seed,
):
    if repetition_seed not in seed_stream_cache:
        seed_stream_cache[
            repetition_seed
        ] = create_seed_random_streams(
            number_of_rows=(
                EXPECTED_RAW_TRAINING_ROWS
            ),

            repetition_seed=(
                repetition_seed
            ),

            failure_subtypes=(
                failure_subtypes
            ),

            subtype_probabilities=(
                subtype_probabilities
            ),
        )

    return seed_stream_cache[
        repetition_seed
    ]


def run_condition(
    condition_row,
):
    condition_started = (
        time.perf_counter()
    )

    condition_order = int(
        condition_row.ConditionOrder
    )

    condition_key = str(
        condition_row.ConditionKey
    )

    noise_percent = int(
        condition_row.NoisePercent
    )

    repetition_seed = int(
        condition_row.RepetitionSeed
    )

    print("\n")
    print("-" * 110)

    print(
        f"[{condition_order}/{EXPECTED_CONDITIONS}] "
        f"Running {condition_key}"
    )

    streams = get_seed_streams(
        repetition_seed
    )

    condition_arrays = (
        construct_condition_arrays(
            clean_verdicts=(
                clean_raw_verdicts
            ),

            row_uniforms=(
                streams[
                    "FlipUniforms"
                ]
            ),

            sampled_failure_subtypes=(
                streams[
                    "SampledFailureSubtypes"
                ]
            ),

            noise_percent=(
                noise_percent
            ),
        )
    )

    flip_mask = condition_arrays[
        "FlipMask"
    ]

    noisy_raw_verdicts = condition_arrays[
        "NoisyVerdicts"
    ]

    generated_mask_hash = (
        packed_boolean_sha256(
            flip_mask
        )
    )

    generated_verdict_hash = (
        integer_array_sha256(
            noisy_raw_verdicts
        )
    )

    if (
        generated_mask_hash
        != condition_row.FlipMaskSHA256
    ):
        raise AssertionError(
            "Generated flip-mask hash differs."
        )

    if (
        generated_verdict_hash
        != condition_row.NoisyVerdictSHA256
    ):
        raise AssertionError(
            "Generated noisy-verdict hash differs."
        )

    noisy_model_verdicts = (
        noisy_raw_verdicts[
            model_noise_row_ids
        ]
    )

    changed_model_labels = int(
        (
            noisy_model_verdicts
            != clean_model_verdicts
        ).sum()
    )

    reconstruction_started = (
        time.perf_counter()
    )

    if noise_percent == 0:
        noisy_anchored_matrix = (
            original_dependent_matrix.copy()
        )

        reconstruction_seconds = 0.0

    else:
        noisy_direct_rec = (
            reconstruct_noisy_dependent_rec(
                noisy_raw_verdicts
            )
        )

        noisy_anchored_matrix = (
            align_noisy_rec(
                noisy_direct_rec
            )
        )

        reconstruction_seconds = float(
            time.perf_counter()
            - reconstruction_started
        )

        del noisy_direct_rec

    changed_dependent_values = int(
        (
            ~np.isclose(
                noisy_anchored_matrix,
                original_dependent_matrix,
                rtol=COMPARISON_RTOL,
                atol=COMPARISON_ATOL,
                equal_nan=False,
            )
        ).sum()
    )

    if noise_percent == 0:
        if changed_model_labels != 0:
            raise AssertionError(
                "0% noise changed model labels."
            )

        if changed_dependent_values != 0:
            raise AssertionError(
                "0% noise changed dependent REC values."
            )

    else:
        if changed_model_labels == 0:
            raise AssertionError(
                "Positive noise changed no model labels."
            )

        if changed_dependent_values == 0:
            raise AssertionError(
                "Positive noise changed no dependent REC values."
            )

    training_numeric = (
        clean_training_numeric_base.copy(
            deep=True
        )
    )

    training_numeric[
        VERDICT_DEPENDENT_REC_FEATURES
    ] = noisy_anchored_matrix

    unaffected_hash_after = (
        dataframe_content_sha256(
            training_numeric[
                unaffected_feature_columns
            ],
            include_index=True,
        )
    )

    unaffected_columns_unchanged = (
        unaffected_hash_after
        == unaffected_training_hash
    )

    if not unaffected_columns_unchanged:
        raise AssertionError(
            "Verdict-independent or non-REC features changed."
        )

    training_medians = (
        training_numeric.median(
            axis=0
        )
        .fillna(0.0)
    )

    if not np.isfinite(
        training_medians.to_numpy(
            dtype=float
        )
    ).all():
        raise AssertionError(
            "Condition medians contain non-finite values."
        )

    X_train = (
        training_numeric.fillna(
            training_medians
        )
        .astype(float)
    )

    X_evaluation = (
        evaluation_numeric_base.fillna(
            training_medians
        )
        .astype(float)
    )

    y_train = pd.Series(
        noisy_model_verdicts != 0,
        dtype=int,
    )

    if y_train.nunique() != 2:
        raise AssertionError(
            "Condition training labels do not contain both classes."
        )

    if not np.isfinite(
        X_train.to_numpy(dtype=float)
    ).all():
        raise AssertionError(
            "Condition training matrix contains non-finite values."
        )

    if not np.isfinite(
        X_evaluation.to_numpy(dtype=float)
    ).all():
        raise AssertionError(
            "Condition evaluation matrix contains non-finite values."
        )

    ranking_frames = []
    fit_records = []

    models = create_ml_models(
        repetition_seed
    )

    for technique, model in models.items():
        print(
            "  Fitting:",
            technique,
            flush=True,
        )

        fit_started = (
            time.perf_counter()
        )

        fit_success = False
        probability_success = False
        error_text = ""

        try:
            with warnings.catch_warnings():
                warnings.simplefilter(
                    "ignore"
                )

                model.fit(
                    X_train,
                    y_train,
                )

                fit_success = True

                failure_probabilities = (
                    get_failure_probability(
                        model,
                        X_evaluation,
                    )
                )

                probability_success = True

            ranking_frames.append(
                create_ml_rankings(
                    technique,
                    failure_probabilities,
                )
            )

        except Exception as error:
            error_text = (
                f"{type(error).__name__}: "
                f"{error}"
            )

            raise

        finally:
            fit_seconds = float(
                time.perf_counter()
                - fit_started
            )

            fit_records.append({
                "Project":
                    PROJECT_NAME,

                "ProjectSlug":
                    PROJECT_SLUG,

                "ConditionOrder":
                    condition_order,

                "ConditionKey":
                    condition_key,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "Technique":
                    technique,

                "TrainingRows":
                    EXPECTED_MODEL_TRAINING_ROWS,

                "EvaluationRows":
                    EXPECTED_MODEL_EVALUATION_ROWS,

                "ActivePredictors":
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    ),

                "TrainingFailures":
                    int(y_train.sum()),

                "TrainingPasses":
                    int(
                        (
                            y_train == 0
                        ).sum()
                    ),

                "FitSuccess":
                    fit_success,

                "ProbabilitySuccess":
                    probability_success,

                "FitSeconds":
                    fit_seconds,

                "Error":
                    error_text,
            })

        del model
        gc.collect()

    random_rankings = (
        create_random_rankings(
            repetition_seed
        )
    )

    latest_fail_rankings = (
        create_latest_fail_rankings(
            noisy_raw_verdicts
        )
    )

    ranking_frames.extend([
        random_rankings,
        latest_fail_rankings,
        QTF_RANKINGS.copy(),
    ])

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    rankings.insert(
        1,
        "ProjectSlug",
        PROJECT_SLUG,
    )

    rankings.insert(
        2,
        "ConditionOrder",
        condition_order,
    )

    rankings.insert(
        3,
        "ConditionKey",
        condition_key,
    )

    rankings.insert(
        4,
        "NoisePercent",
        noise_percent,
    )

    rankings.insert(
        5,
        "RepetitionSeed",
        repetition_seed,
    )

    fit_log = pd.DataFrame(
        fit_records
    )

    build_metrics = (
        calculate_build_metrics(
            rankings,
            noise_percent,
            repetition_seed,
        )
    )

    build_metrics.insert(
        2,
        "ConditionOrder",
        condition_order,
    )

    build_metrics.insert(
        3,
        "ConditionKey",
        condition_key,
    )

    project_run = (
        aggregate_project_run(
            build_metrics
        )
    )

    project_run.insert(
        2,
        "ConditionOrder",
        condition_order,
    )

    project_run.insert(
        3,
        "ConditionKey",
        condition_key,
    )

    validate_condition_outputs(
        rankings,
        build_metrics,
        project_run,
        fit_log,
    )

    condition_seconds = float(
        time.perf_counter()
        - condition_started
    )

    condition_audit = pd.DataFrame([
        {
            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "ConditionOrder":
                condition_order,

            "ConditionKey":
                condition_key,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "RawTrainingRows":
                EXPECTED_RAW_TRAINING_ROWS,

            "RawRowsFlipped":
                int(flip_mask.sum()),

            "RecordedRawRowsFlipped":
                int(
                    condition_row.RawRowsFlipped
                ),

            "RealisedNoisePercent":
                float(
                    100.0
                    * flip_mask.sum()
                    / EXPECTED_RAW_TRAINING_ROWS
                ),

            "PassToFailure":
                int(
                    condition_arrays[
                        "PassToFailureMask"
                    ].sum()
                ),

            "FailureToPass":
                int(
                    condition_arrays[
                        "FailureToPassMask"
                    ].sum()
                ),

            "ChangedModelLabels":
                changed_model_labels,

            "ChangedDependentRECValues":
                changed_dependent_values,

            "UnaffectedColumnsUnchanged":
                unaffected_columns_unchanged,

            "FlipMaskHashMatch":
                True,

            "NoisyVerdictHashMatch":
                True,

            "ModelTrainingRows":
                EXPECTED_MODEL_TRAINING_ROWS,

            "ModelEvaluationRows":
                EXPECTED_MODEL_EVALUATION_ROWS,

            "TrainingFailures":
                int(y_train.sum()),

            "TrainingPasses":
                int(
                    (
                        y_train == 0
                    ).sum()
                ),

            "MLFits":
                len(fit_log),

            "RankingRows":
                len(rankings),

            "BuildMetricRows":
                len(build_metrics),

            "ProjectRunRows":
                len(project_run),

            "ReconstructionSeconds":
                reconstruction_seconds,

            "ConditionSeconds":
                condition_seconds,
        }
    ])

    training_median_audit = pd.DataFrame({
        "FeatureOrder":
            np.arange(
                1,
                len(
                    ACTIVE_FEATURE_COLUMNS
                ) + 1,
            ),

        "Feature":
            ACTIVE_FEATURE_COLUMNS,

        "TrainingMedian":
            training_medians[
                ACTIVE_FEATURE_COLUMNS
            ].to_numpy(dtype=float),
    })

    paths = condition_paths(
        condition_key
    )

    paths["Directory"].mkdir(
        parents=True,
        exist_ok=True,
    )

    atomic_write_parquet(
        paths["Rankings"],
        rankings,
    )

    atomic_write_csv(
        paths["BuildMetrics"],
        build_metrics,
    )

    atomic_write_csv(
        paths["ProjectRun"],
        project_run,
    )

    atomic_write_csv(
        paths["ModelFits"],
        fit_log,
    )

    atomic_write_csv(
        paths["ConditionAudit"],
        condition_audit,
    )

    atomic_write_csv(
        paths["TrainingMedians"],
        training_median_audit,
    )

    summary_payload = {
        "Status":
            "PASS_FULL_CONDITION",

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            condition_order,

        "ConditionKey":
            condition_key,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "RawTrainingRows":
            EXPECTED_RAW_TRAINING_ROWS,

        "RawRowsFlipped":
            int(flip_mask.sum()),

        "RealisedNoisePercent":
            float(
                100.0
                * flip_mask.sum()
                / EXPECTED_RAW_TRAINING_ROWS
            ),

        "PassToFailure":
            int(
                condition_arrays[
                    "PassToFailureMask"
                ].sum()
            ),

        "FailureToPass":
            int(
                condition_arrays[
                    "FailureToPassMask"
                ].sum()
            ),

        "ChangedModelLabels":
            changed_model_labels,

        "ChangedDependentRECValues":
            changed_dependent_values,

        "UnaffectedColumnsUnchanged":
            unaffected_columns_unchanged,

        "TrainingFailures":
            int(y_train.sum()),

        "TrainingPasses":
            int(
                (
                    y_train == 0
                ).sum()
            ),

        "MLFits":
            len(fit_log),

        "RankingRows":
            len(rankings),

        "BuildMetricRows":
            len(build_metrics),

        "ProjectRunRows":
            len(project_run),

        "TrainingMedianRows":
            len(training_median_audit),

        "QTFUnseenScores":
            QTF_UNSEEN_SCORES,

        "ReconstructionSeconds":
            reconstruction_seconds,

        "ConditionSeconds":
            condition_seconds,

        "RankingsSHA256":
            calculate_sha256(
                paths["Rankings"]
            ),

        "BuildMetricsSHA256":
            calculate_sha256(
                paths["BuildMetrics"]
            ),

        "ProjectRunSHA256":
            calculate_sha256(
                paths["ProjectRun"]
            ),

        "ModelFitsSHA256":
            calculate_sha256(
                paths["ModelFits"]
            ),

        "ConditionAuditSHA256":
            calculate_sha256(
                paths["ConditionAudit"]
            ),

        "TrainingMediansSHA256":
            calculate_sha256(
                paths["TrainingMedians"]
            ),

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    atomic_write_json(
        paths["ConditionSummary"],
        summary_payload,
    )

    success_payload = {
        "Status":
            "PASS_FULL_CONDITION",

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            condition_order,

        "ConditionKey":
            condition_key,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "RankingsSHA256":
            summary_payload[
                "RankingsSHA256"
            ],

        "BuildMetricsSHA256":
            summary_payload[
                "BuildMetricsSHA256"
            ],

        "ProjectRunSHA256":
            summary_payload[
                "ProjectRunSHA256"
            ],

        "ModelFitsSHA256":
            summary_payload[
                "ModelFitsSHA256"
            ],

        "ConditionAuditSHA256":
            summary_payload[
                "ConditionAuditSHA256"
            ],

        "TrainingMediansSHA256":
            summary_payload[
                "TrainingMediansSHA256"
            ],

        "ConditionSummarySHA256":
            calculate_sha256(
                paths["ConditionSummary"]
            ),

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    atomic_write_json(
        paths["Success"],
        success_payload,
    )

    valid, validation_message = (
        validate_completed_condition(
            condition_row,
            verify_hashes=True,
        )
    )

    if not valid:
        raise RuntimeError(
            "Written condition failed readback validation.\n"
            f"Condition: {condition_key}\n"
            f"Reason: {validation_message}"
        )

    print(
        "  Completed:",
        condition_key,
    )

    print(
        "  Raw flips:",
        int(flip_mask.sum()),
        "| model-label changes:",
        changed_model_labels,
        "| dependent REC changes:",
        changed_dependent_values,
    )

    print(
        "  Training failures:",
        int(y_train.sum()),
        "| condition seconds:",
        round(
            condition_seconds,
            2,
        ),
    )

    del training_numeric
    del X_train
    del X_evaluation
    del y_train
    del noisy_anchored_matrix
    del rankings
    del build_metrics
    del project_run
    del fit_log
    del condition_audit
    del training_median_audit

    gc.collect()

    return summary_payload


# ------------------------------------------------------------
# 27. INITIALISE RESUME PROGRESS
# ------------------------------------------------------------

RAW_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_CONTROL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

STAGING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


progress_records = []

print("\nScanning existing condition checkpoints...")

for row in condition_plan.itertuples(
    index=False
):
    valid, message = (
        validate_completed_condition(
            row,
            verify_hashes=True,
        )
    )

    progress_records.append({
        "ConditionOrder":
            int(row.ConditionOrder),

        "ConditionKey":
            str(row.ConditionKey),

        "NoisePercent":
            int(row.NoisePercent),

        "RepetitionSeed":
            int(row.RepetitionSeed),

        "Status":
            (
                "COMPLETED"
                if valid
                else
                "PENDING"
            ),

        "ValidationMessage":
            message,

        "UpdatedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    })


progress = (
    pd.DataFrame(
        progress_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


completed_before_run = int(
    progress[
        "Status"
    ].eq(
        "COMPLETED"
    ).sum()
)

pending_before_run = int(
    progress[
        "Status"
    ].eq(
        "PENDING"
    ).sum()
)


print(
    "Valid completed conditions:",
    completed_before_run,
)

print(
    "Pending conditions:",
    pending_before_run,
)


atomic_write_csv(
    RUN_PROGRESS_PATH,
    progress,
)


atomic_write_json(
    RUN_PROGRESS_JSON_PATH,
    {
        "Status":
            STEP5A_RUNNING_STATUS,

        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "Conditions":
            EXPECTED_CONDITIONS,

        "CompletedConditions":
            completed_before_run,

        "PendingConditions":
            pending_before_run,

        "LastUpdatedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    },
)


atomic_write_json(
    STEP5A_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            STEP5A_RUNNING_STATUS,

        "Conditions":
            EXPECTED_CONDITIONS,

        "CompletedConditions":
            completed_before_run,

        "PendingConditions":
            pending_before_run,

        "CompletionRegistryModified":
            False,

        "Projects1To8Modified":
            False,

        "LastUpdatedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    },
)


# ------------------------------------------------------------
# 28. EXECUTE PENDING CONDITIONS
# ------------------------------------------------------------

full_run_started = (
    time.perf_counter()
)

conditions_executed_this_run = 0
conditions_skipped_this_run = 0


for condition_row in condition_plan.itertuples(
    index=False
):
    progress_index = int(
        condition_row.ConditionOrder
    ) - 1

    if (
        progress.loc[
            progress_index,
            "Status",
        ]
        == "COMPLETED"
    ):
        conditions_skipped_this_run += 1

        print(
            f"[{int(condition_row.ConditionOrder)}/"
            f"{EXPECTED_CONDITIONS}] "
            f"Skipping validated "
            f"{condition_row.ConditionKey}"
        )

        continue

    try:
        run_condition(
            condition_row
        )

        conditions_executed_this_run += 1

        progress.loc[
            progress_index,
            "Status",
        ] = "COMPLETED"

        progress.loc[
            progress_index,
            "ValidationMessage",
        ] = "VALID_COMPLETED_CONDITION"

        progress.loc[
            progress_index,
            "UpdatedAtUTC",
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        completed_now = int(
            progress[
                "Status"
            ].eq(
                "COMPLETED"
            ).sum()
        )

        pending_now = (
            EXPECTED_CONDITIONS
            - completed_now
        )

        atomic_write_csv(
            RUN_PROGRESS_PATH,
            progress,
        )

        atomic_write_json(
            RUN_PROGRESS_JSON_PATH,
            {
                "Status":
                    STEP5A_RUNNING_STATUS,

                "ProjectNumber":
                    PROJECT_NUMBER,

                "Project":
                    PROJECT_NAME,

                "Conditions":
                    EXPECTED_CONDITIONS,

                "CompletedConditions":
                    completed_now,

                "PendingConditions":
                    pending_now,

                "LastCompletedCondition":
                    str(
                        condition_row.ConditionKey
                    ),

                "ConditionsExecutedThisRun":
                    conditions_executed_this_run,

                "ConditionsSkippedThisRun":
                    conditions_skipped_this_run,

                "ElapsedSecondsThisRun":
                    float(
                        time.perf_counter()
                        - full_run_started
                    ),

                "LastUpdatedAtUTC":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
        )

        atomic_write_json(
            STEP5A_STATUS_PATH,
            {
                "ProjectNumber":
                    PROJECT_NUMBER,

                "Project":
                    PROJECT_NAME,

                "ProjectSlug":
                    PROJECT_SLUG,

                "Status":
                    STEP5A_RUNNING_STATUS,

                "Conditions":
                    EXPECTED_CONDITIONS,

                "CompletedConditions":
                    completed_now,

                "PendingConditions":
                    pending_now,

                "LastCompletedCondition":
                    str(
                        condition_row.ConditionKey
                    ),

                "CompletionRegistryModified":
                    False,

                "Projects1To8Modified":
                    False,

                "LastUpdatedAtUTC":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
        )

    except Exception as error:
        progress.loc[
            progress_index,
            "Status",
        ] = "FAILED"

        progress.loc[
            progress_index,
            "ValidationMessage",
        ] = (
            f"{type(error).__name__}: "
            f"{error}"
        )

        progress.loc[
            progress_index,
            "UpdatedAtUTC",
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        atomic_write_csv(
            RUN_PROGRESS_PATH,
            progress,
        )

        failure_payload = {
            "Status":
                "FAILED_PROJECT_9_FULL_RUN_CONDITION",

            "Project":
                PROJECT_NAME,

            "ConditionOrder":
                int(
                    condition_row.ConditionOrder
                ),

            "ConditionKey":
                str(
                    condition_row.ConditionKey
                ),

            "NoisePercent":
                int(
                    condition_row.NoisePercent
                ),

            "RepetitionSeed":
                int(
                    condition_row.RepetitionSeed
                ),

            "ErrorType":
                type(error).__name__,

            "Error":
                str(error),

            "Traceback":
                traceback.format_exc(),

            "CompletedConditions":
                int(
                    progress[
                        "Status"
                    ].eq(
                        "COMPLETED"
                    ).sum()
                ),

            "FailedAtUTC":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

        atomic_write_json(
            RUN_FAILURE_PATH,
            failure_payload,
        )

        atomic_write_json(
            STEP5A_STATUS_PATH,
            {
                "ProjectNumber":
                    PROJECT_NUMBER,

                "Project":
                    PROJECT_NAME,

                "ProjectSlug":
                    PROJECT_SLUG,

                "Status":
                    "FAILED_PROJECT_9_FULL_RUN_CONDITION",

                "FailedCondition":
                    str(
                        condition_row.ConditionKey
                    ),

                "CompletedConditions":
                    int(
                        progress[
                            "Status"
                        ].eq(
                            "COMPLETED"
                        ).sum()
                    ),

                "CompletionRegistryModified":
                    False,

                "Projects1To8Modified":
                    False,

                "FailedAtUTC":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
        )

        raise


full_run_seconds = float(
    time.perf_counter()
    - full_run_started
)


# ------------------------------------------------------------
# 29. FINAL CONDITION VALIDATION
# ------------------------------------------------------------

print("\nValidating all 270 completed conditions...")

final_condition_records = []
final_condition_failures = []


for row in condition_plan.itertuples(
    index=False
):
    valid, message = (
        validate_completed_condition(
            row,
            verify_hashes=True,
        )
    )

    if not valid:
        final_condition_failures.append({
            "ConditionOrder":
                int(row.ConditionOrder),

            "ConditionKey":
                str(row.ConditionKey),

            "Reason":
                message,
        })

        continue

    paths = condition_paths(
        str(row.ConditionKey)
    )

    summary = json.loads(
        paths[
            "ConditionSummary"
        ].read_text(
            encoding="utf-8"
        )
    )

    final_condition_records.append({
        "ConditionOrder":
            int(row.ConditionOrder),

        "ConditionKey":
            str(row.ConditionKey),

        "NoisePercent":
            int(row.NoisePercent),

        "RepetitionSeed":
            int(row.RepetitionSeed),

        "RawRowsFlipped":
            int(
                summary[
                    "RawRowsFlipped"
                ]
            ),

        "RealisedNoisePercent":
            float(
                summary[
                    "RealisedNoisePercent"
                ]
            ),

        "ChangedModelLabels":
            int(
                summary[
                    "ChangedModelLabels"
                ]
            ),

        "ChangedDependentRECValues":
            int(
                summary[
                    "ChangedDependentRECValues"
                ]
            ),

        "TrainingFailures":
            int(
                summary[
                    "TrainingFailures"
                ]
            ),

        "TrainingPasses":
            int(
                summary[
                    "TrainingPasses"
                ]
            ),

        "MLFits":
            int(
                summary[
                    "MLFits"
                ]
            ),

        "RankingRows":
            int(
                summary[
                    "RankingRows"
                ]
            ),

        "BuildMetricRows":
            int(
                summary[
                    "BuildMetricRows"
                ]
            ),

        "ProjectRunRows":
            int(
                summary[
                    "ProjectRunRows"
                ]
            ),

        "ReconstructionSeconds":
            float(
                summary[
                    "ReconstructionSeconds"
                ]
            ),

        "ConditionSeconds":
            float(
                summary[
                    "ConditionSeconds"
                ]
            ),

        "Status":
            summary[
                "Status"
            ],
    })


if final_condition_failures:
    display(
        pd.DataFrame(
            final_condition_failures
        )
    )

    raise RuntimeError(
        "One or more final conditions failed validation."
    )


condition_inventory = (
    pd.DataFrame(
        final_condition_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(condition_inventory) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Final condition inventory count differs."
    )


total_ml_fits = int(
    condition_inventory[
        "MLFits"
    ].sum()
)

total_ranking_rows = int(
    condition_inventory[
        "RankingRows"
    ].sum()
)

total_build_metric_rows = int(
    condition_inventory[
        "BuildMetricRows"
    ].sum()
)

total_project_run_rows = int(
    condition_inventory[
        "ProjectRunRows"
    ].sum()
)


# ------------------------------------------------------------
# 30. RAW ROOT MANIFEST AND HASH
# ------------------------------------------------------------

print("\nHashing complete raw-result root...")

raw_files = sorted(
    [
        path
        for path in RAW_ROOT.rglob("*")
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            RAW_ROOT
        ).as_posix(),
)


raw_file_records = []
raw_root_digest = hashlib.sha256()


for file_index, path in enumerate(
    raw_files,
    start=1,
):
    relative_path = (
        path.relative_to(
            RAW_ROOT
        ).as_posix()
    )

    size_bytes = int(
        path.stat().st_size
    )

    file_sha256 = calculate_sha256(
        path
    )

    raw_file_records.append({
        "FileOrder":
            file_index,

        "RelativePath":
            relative_path,

        "SizeBytes":
            size_bytes,

        "SHA256":
            file_sha256,
    })

    manifest_line = (
        f"{relative_path}\0"
        f"{size_bytes}\0"
        f"{file_sha256}\n"
    )

    raw_root_digest.update(
        manifest_line.encode(
            "utf-8"
        )
    )


raw_file_manifest = pd.DataFrame(
    raw_file_records
)

raw_root_sha256 = (
    raw_root_digest.hexdigest()
)

raw_file_count = len(
    raw_file_manifest
)

raw_bytes = int(
    raw_file_manifest[
        "SizeBytes"
    ].sum()
)


# ------------------------------------------------------------
# 31. IMMUTABILITY VALIDATION
# ------------------------------------------------------------

source_hashes_after = {
    relative_name:
        calculate_sha256(
            path
        )
    for relative_name, path
    in source_paths.items()
}

source_files_unchanged = (
    source_hashes_before
    == source_hashes_after
)

evaluation_sha256_after = (
    calculate_sha256(
        model_evaluation_path
    )
)

evaluation_unchanged = (
    evaluation_sha256_before
    == evaluation_sha256_after
)

registry_sha256_after = calculate_sha256(
    REGISTRY_PATH
)

registry_unchanged = (
    registry_sha256_before
    == registry_sha256_after
)

clean_training_numeric_hash_after = (
    dataframe_content_sha256(
        clean_training_numeric_base,
        include_index=True,
    )
)

clean_training_base_unchanged = (
    clean_training_numeric_hash_before
    == clean_training_numeric_hash_after
)


if not source_files_unchanged:
    raise AssertionError(
        "A frozen source file changed during the full run."
    )

if not evaluation_unchanged:
    raise AssertionError(
        "The frozen evaluation cohort changed during the full run."
    )

if not registry_unchanged:
    raise AssertionError(
        "The completion registry changed during the full run."
    )

if not clean_training_base_unchanged:
    raise AssertionError(
        "The clean training numeric base changed in memory."
    )

for path, expected_hash in (
    frozen_input_hashes.items()
):
    if (
        calculate_sha256(path)
        != expected_hash
    ):
        raise AssertionError(
            "A frozen experiment input changed during the full run.\n"
            f"Path: {path}"
        )


# ------------------------------------------------------------
# 32. FINAL VALIDATION
# ------------------------------------------------------------

zero_noise_inventory = condition_inventory[
    condition_inventory[
        "NoisePercent"
    ].eq(0)
]

positive_noise_inventory = condition_inventory[
    condition_inventory[
        "NoisePercent"
    ].gt(0)
]


validation_records = [
    {
        "Check":
            "Step 4B passed",

        "Expected":
            EXPECTED_STEP4B_STATUS,

        "Actual":
            step4b_status[
                "Status"
            ],

        "Pass":
            step4b_status[
                "Status"
            ]
            == EXPECTED_STEP4B_STATUS,
    },

    {
        "Check":
            "Completed conditions",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                condition_inventory
            ),

        "Pass":
            len(
                condition_inventory
            )
            == EXPECTED_CONDITIONS,
    },

    {
        "Check":
            "Noise levels",

        "Expected":
            9,

        "Actual":
            condition_inventory[
                "NoisePercent"
            ].nunique(),

        "Pass":
            condition_inventory[
                "NoisePercent"
            ].nunique()
            == 9,
    },

    {
        "Check":
            "Repetition seeds",

        "Expected":
            30,

        "Actual":
            condition_inventory[
                "RepetitionSeed"
            ].nunique(),

        "Pass":
            condition_inventory[
                "RepetitionSeed"
            ].nunique()
            == 30,
    },

    {
        "Check":
            "ML fits",

        "Expected":
            EXPECTED_ML_FITS,

        "Actual":
            total_ml_fits,

        "Pass":
            total_ml_fits
            == EXPECTED_ML_FITS,
    },

    {
        "Check":
            "Ranking rows",

        "Expected":
            EXPECTED_TOTAL_RANKING_ROWS,

        "Actual":
            total_ranking_rows,

        "Pass":
            total_ranking_rows
            == EXPECTED_TOTAL_RANKING_ROWS,
    },

    {
        "Check":
            "Build-metric rows",

        "Expected":
            EXPECTED_TOTAL_BUILD_METRICS,

        "Actual":
            total_build_metric_rows,

        "Pass":
            total_build_metric_rows
            == EXPECTED_TOTAL_BUILD_METRICS,
    },

    {
        "Check":
            "Project-run rows",

        "Expected":
            EXPECTED_TOTAL_PROJECT_RUN_ROWS,

        "Actual":
            total_project_run_rows,

        "Pass":
            total_project_run_rows
            == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Raw files",

        "Expected":
            EXPECTED_RAW_FILES,

        "Actual":
            raw_file_count,

        "Pass":
            raw_file_count
            == EXPECTED_RAW_FILES,
    },

    {
        "Check":
            "Zero-noise conditions",

        "Expected":
            30,

        "Actual":
            len(
                zero_noise_inventory
            ),

        "Pass":
            len(
                zero_noise_inventory
            ) == 30,
    },

    {
        "Check":
            "Zero-noise flipped rows",

        "Expected":
            0,

        "Actual":
            int(
                zero_noise_inventory[
                    "RawRowsFlipped"
                ].sum()
            ),

        "Pass":
            int(
                zero_noise_inventory[
                    "RawRowsFlipped"
                ].sum()
            ) == 0,
    },

    {
        "Check":
            "Zero-noise model-label changes",

        "Expected":
            0,

        "Actual":
            int(
                zero_noise_inventory[
                    "ChangedModelLabels"
                ].sum()
            ),

        "Pass":
            int(
                zero_noise_inventory[
                    "ChangedModelLabels"
                ].sum()
            ) == 0,
    },

    {
        "Check":
            "Zero-noise REC changes",

        "Expected":
            0,

        "Actual":
            int(
                zero_noise_inventory[
                    "ChangedDependentRECValues"
                ].sum()
            ),

        "Pass":
            int(
                zero_noise_inventory[
                    "ChangedDependentRECValues"
                ].sum()
            ) == 0,
    },

    {
        "Check":
            "Positive-noise model-label changes",

        "Expected":
            True,

        "Actual":
            bool(
                positive_noise_inventory[
                    "ChangedModelLabels"
                ].gt(0).all()
            ),

        "Pass":
            bool(
                positive_noise_inventory[
                    "ChangedModelLabels"
                ].gt(0).all()
            ),
    },

    {
        "Check":
            "Positive-noise REC changes",

        "Expected":
            True,

        "Actual":
            bool(
                positive_noise_inventory[
                    "ChangedDependentRECValues"
                ].gt(0).all()
            ),

        "Pass":
            bool(
                positive_noise_inventory[
                    "ChangedDependentRECValues"
                ].gt(0).all()
            ),
    },

    {
        "Check":
            "Source files unchanged",

        "Expected":
            True,

        "Actual":
            source_files_unchanged,

        "Pass":
            source_files_unchanged,
    },

    {
        "Check":
            "Evaluation unchanged",

        "Expected":
            True,

        "Actual":
            evaluation_unchanged,

        "Pass":
            evaluation_unchanged,
    },

    {
        "Check":
            "Registry Project 9 rows",

        "Expected":
            0,

        "Actual":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation["Pass"]
].copy()


print("\nStep 5A validation:")

display(validation)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(failed_checks)

    raise RuntimeError(
        "PROJECT 9 STEP 5A DID NOT PASS."
    )


# ------------------------------------------------------------
# 33. WRITE CONTROL OUTPUTS
# ------------------------------------------------------------

atomic_write_csv(
    CONDITION_INVENTORY_PATH,
    condition_inventory,
)

atomic_write_csv(
    RAW_FILE_MANIFEST_PATH,
    raw_file_manifest,
)

atomic_write_csv(
    STEP5A_VALIDATION_PATH,
    validation,
)


total_condition_seconds = float(
    condition_inventory[
        "ConditionSeconds"
    ].sum()
)

total_reconstruction_seconds = float(
    condition_inventory[
        "ReconstructionSeconds"
    ].sum()
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_PASS_STATUS,

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        total_ml_fits,

    "RankingRows":
        total_ranking_rows,

    "BuildMetricRows":
        total_build_metric_rows,

    "ProjectRunRows":
        total_project_run_rows,

    "RawFiles":
        raw_file_count,

    "RawBytes":
        raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "ConditionsExecutedThisRun":
        conditions_executed_this_run,

    "ConditionsSkippedThisRun":
        conditions_skipped_this_run,

    "CellRuntimeSeconds":
        full_run_seconds,

    "TotalRecordedConditionSeconds":
        total_condition_seconds,

    "TotalRecordedReconstructionSeconds":
        total_reconstruction_seconds,

    "SourceFilesUnchanged":
        source_files_unchanged,

    "EvaluationUnchanged":
        evaluation_unchanged,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP5A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_PASS_STATUS,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RECCheckpointSHA256":
        calculate_sha256(
            REC_CHECKPOINT_PATH
        ),

    "NoiseCheckpointSHA256":
        calculate_sha256(
            NOISE_CHECKPOINT_PATH
        ),

    "NoisyRECCheckpointSHA256":
        calculate_sha256(
            NOISY_REC_CHECKPOINT_PATH
        ),

    "ModelMetricCheckpointSHA256":
        calculate_sha256(
            MODEL_METRIC_CHECKPOINT_PATH
        ),

    "SmokeCheckpointSHA256":
        calculate_sha256(
            SMOKE_TEST_CHECKPOINT_PATH
        ),

    "RawRoot":
        str(RAW_ROOT),

    "RawFiles":
        raw_file_count,

    "RawBytes":
        raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "RawFileManifest":
        str(
            RAW_FILE_MANIFEST_PATH
        ),

    "RawFileManifestSHA256":
        calculate_sha256(
            RAW_FILE_MANIFEST_PATH
        ),

    "ConditionInventory":
        str(
            CONDITION_INVENTORY_PATH
        ),

    "ConditionInventorySHA256":
        calculate_sha256(
            CONDITION_INVENTORY_PATH
        ),

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRICS,

    "ProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "EvaluationSHA256":
        evaluation_sha256_after,

    "CompletionRegistrySHA256":
        registry_sha256_after,

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    FULL_RUN_CHECKPOINT_PATH,
    checkpoint_payload,
)


final_status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_PASS_STATUS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRICS,

    "ProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "RawFiles":
        raw_file_count,

    "RawBytes":
        raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "Checkpoint":
        str(
            FULL_RUN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_sha256(
            FULL_RUN_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletionRegistryModified":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP5A_STATUS_PATH,
    final_status_payload,
)


atomic_write_json(
    RUN_PROGRESS_JSON_PATH,
    {
        "Status":
            STEP5A_PASS_STATUS,

        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "Conditions":
            EXPECTED_CONDITIONS,

        "CompletedConditions":
            EXPECTED_CONDITIONS,

        "PendingConditions":
            0,

        "ConditionsExecutedThisRun":
            conditions_executed_this_run,

        "ConditionsSkippedThisRun":
            conditions_skipped_this_run,

        "RawFiles":
            raw_file_count,

        "RawRootSHA256":
            raw_root_sha256,

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),
    },
)


# ------------------------------------------------------------
# 34. FINAL READBACK
# ------------------------------------------------------------

final_status = json.loads(
    STEP5A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    final_status.get("Status")
    != STEP5A_PASS_STATUS
):
    raise AssertionError(
        "Final Step 5A status differs."
    )


# ------------------------------------------------------------
# 35. DISPLAY SUMMARIES
# ------------------------------------------------------------

print("\nCondition summary by noise level:")

noise_summary = (
    condition_inventory.groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        Conditions=(
            "ConditionKey",
            "count",
        ),

        MeanRawRowsFlipped=(
            "RawRowsFlipped",
            "mean",
        ),

        MeanRealisedNoisePercent=(
            "RealisedNoisePercent",
            "mean",
        ),

        MeanChangedModelLabels=(
            "ChangedModelLabels",
            "mean",
        ),

        MeanChangedDependentRECValues=(
            "ChangedDependentRECValues",
            "mean",
        ),

        MeanTrainingFailures=(
            "TrainingFailures",
            "mean",
        ),

        MeanConditionSeconds=(
            "ConditionSeconds",
            "mean",
        ),
    )
)

display(noise_summary)


print("\nSlowest 15 conditions:")

display(
    condition_inventory.sort_values(
        "ConditionSeconds",
        ascending=False,
        kind="mergesort",
    ).head(15)
)


# ------------------------------------------------------------
# 36. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 122)
print("=== PROJECT 9 CELL 9 / STEP 5A RESULT ===")
print("=" * 122)

print("\nProject:")
print(PROJECT_NAME)

print("\nFull experiment:")
print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(REPETITION_SEEDS),
)

print(
    "Conditions:",
    len(condition_inventory),
    "/",
    EXPECTED_CONDITIONS,
)

print(
    "ML fits:",
    total_ml_fits,
    "/",
    EXPECTED_ML_FITS,
)

print(
    "Ranking rows:",
    total_ranking_rows,
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    total_build_metric_rows,
    "/",
    EXPECTED_TOTAL_BUILD_METRICS,
)

print(
    "Project-run rows:",
    total_project_run_rows,
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)


print("\nResume activity:")

print(
    "Conditions executed this run:",
    conditions_executed_this_run,
)

print(
    "Conditions skipped this run:",
    conditions_skipped_this_run,
)


print("\nRaw-result freeze:")

print(
    "Raw files:",
    raw_file_count,
    "/",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    raw_bytes,
)

print(
    "Raw root SHA-256:",
    raw_root_sha256,
)


print("\nImmutability:")

print(
    "Source files unchanged:",
    source_files_unchanged,
)

print(
    "Evaluation unchanged:",
    evaluation_unchanged,
)

print(
    "Completion registry modified:",
    0,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nRuntime:")

print(
    "This cell execution seconds:",
    round(
        full_run_seconds,
        2,
    ),
)

print(
    "Recorded total condition seconds:",
    round(
        total_condition_seconds,
        2,
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(validation),
)

print(
    "Failed checks:",
    len(failed_checks),
)


print("\nFull-run checkpoint:")

print(
    FULL_RUN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        FULL_RUN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP5A_PASS_STATUS,
)

print("=" * 122)

=== PROJECT 9 CELL 9 / STEP 5A: CHECKPOINTED FULL 270-CONDITION EXPERIMENT ===

Precomputing invariant QTF-Avg baseline...
QTF-Avg rows: 18503
QTF-Avg unseen scores: 10

Scanning existing condition checkpoints...
Valid completed conditions: 0
Pending conditions: 270


--------------------------------------------------------------------------------------------------------------
[1/270] Running noise_00__seed_01
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM
  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 1427 | condition seconds: 67.29


--------------------------------------------------------------------------------------------------------------
[2/270] Running noise_05__seed_01
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM
  Fitting: NaiveBayes
  Completed: noise_05__seed_01
  Raw flips: 15657 | model-label changes: 3056 | dependent REC changes: 479474
  Traini

,Check,Expected,Actual,Pass
0,Step 4B passed,PASS_PROJECT_9_TWO_CONDITION_END_TO_END_SMOKE_...,PASS_PROJECT_9_TWO_CONDITION_END_TO_END_SMOKE_...,True
1,Completed conditions,270,270,True
2,Noise levels,9,9,True
3,Repetition seeds,30,30,True
4,ML fits,1080,1080,True
5,Ranking rows,34970670,34970670,True
6,Build-metric rows,56700,56700,True
7,Project-run rows,1890,1890,True
8,Raw files,2160,2160,True
9,Zero-noise conditions,30,30,True



Condition summary by noise level:


,NoisePercent,Conditions,MeanRawRowsFlipped,MeanRealisedNoisePercent,MeanChangedModelLabels,MeanChangedDependentRECValues,MeanTrainingFailures,MeanConditionSeconds
0,0,30,0.000000,0.000000,0.000000,0.000000,1427.000000,39.234314
1,5,30,15688.666667,4.996645,3039.766667,479037.100000,4320.500000,64.932419
2,10,30,31386.233333,9.996125,6098.900000,547053.000000,7235.633333,67.546294
3,15,30,47063.166667,14.989033,9163.766667,590953.233333,10148.900000,69.617688
4,20,30,62780.233333,19.994724,12199.433333,623488.600000,13033.366667,71.347112
5,25,30,78479.600000,24.994777,15230.066667,648429.866667,15924.066667,71.328727
6,30,30,94174.300000,29.993344,18257.166667,667831.533333,18808.833333,71.274554
7,40,30,125580.700000,39.995892,24327.700000,696024.233333,24595.300000,72.635020
8,50,30,156968.900000,49.992643,30421.733333,715109.766667,30402.666667,74.655884



Slowest 15 conditions:


,ConditionOrder,ConditionKey,NoisePercent,RepetitionSeed,RawRowsFlipped,RealisedNoisePercent,ChangedModelLabels,ChangedDependentRECValues,TrainingFailures,TrainingPasses,MLFits,RankingRows,BuildMetricRows,ProjectRunRows,ReconstructionSeconds,ConditionSeconds,Status
1,2,noise_05__seed_01,5,1,15657,4.986560,3056,479474,4327,56553,4,129521,210,7,9.765174,84.833005,PASS_FULL_CONDITION
116,117,noise_50__seed_13,50,13,156897,49.969744,30384,715698,30355,30525,4,129521,210,7,6.272123,80.064369,PASS_FULL_CONDITION
44,45,noise_50__seed_05,50,5,157426,50.138224,30503,714767,30470,30410,4,129521,210,7,7.140522,79.252698,PASS_FULL_CONDITION
13,14,noise_20__seed_02,20,2,62585,19.932544,12026,624039,12901,47979,4,129521,210,7,6.579432,79.112936,PASS_FULL_CONDITION
266,267,noise_25__seed_30,25,30,78403,24.970381,15305,648105,15974,44906,4,129521,210,7,6.873146,79.111475,PASS_FULL_CONDITION
224,225,noise_50__seed_25,50,25,157092,50.031849,30575,715267,30582,30298,4,129521,210,7,5.475950,79.109278,PASS_FULL_CONDITION
7,8,noise_40__seed_01,40,1,124987,39.806805,24218,696217,24427,36453,4,129521,210,7,7.533099,78.526722,PASS_FULL_CONDITION
53,54,noise_50__seed_06,50,6,157195,50.064653,30545,715244,30572,30308,4,129521,210,7,6.296139,78.413223,PASS_FULL_CONDITION
57,58,noise_15__seed_07,15,7,46935,14.948214,9320,590779,10343,50537,4,129521,210,7,7.339567,77.378591,PASS_FULL_CONDITION
178,179,noise_40__seed_20,40,20,125898,40.096948,24359,696868,24578,36302,4,129521,210,7,7.352785,77.338190,PASS_FULL_CONDITION




=== PROJECT 9 CELL 9 / STEP 5A RESULT ===

Project:
camunda@camunda-bpm-platform

Full experiment:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 34970670 / 34970670
Build-metric rows: 56700 / 56700
Project-run rows: 1890 / 1890

Resume activity:
Conditions executed this run: 270
Conditions skipped this run: 0

Raw-result freeze:
Raw files: 2160 / 2160
Raw bytes: 387066081
Raw root SHA-256: c31cb45e1354dc1222b82103bd275c21b10dd2ba6171125005d9c0a99ab14724

Immutability:
Source files unchanged: True
Evaluation unchanged: True
Completion registry modified: 0
Projects 1–8 modified: 0

Runtime:
This cell execution seconds: 18318.89
Recorded total condition seconds: 18077.16

Validation:
Checks: 18
Failed checks: 0

Full-run checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_09_full_run_checkpoint.json
Checkpoint SHA-256: 22f9f1184be247d83941199381f12b7042f743158403e765ee8ffcfe09636519

STATUS: 

In [15]:
# ============================================================
# PROJECT 9 — CELL 10 / STEP 5B
# FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION
#
# PROJECT:
#   camunda@camunda-bpm-platform
#
# This cell:
# - revalidates the frozen 270-condition Project 9 raw output
# - verifies all 2,160 raw files
# - verifies per-condition success markers and embedded hashes
# - counts ranking rows from Parquet metadata without loading
#   the 34,970,670 ranking rows into memory
# - combines compact project-run, build-metric, model-fit,
#   condition-audit, and training-median outputs
# - creates noise/technique summaries and clean-noise deltas
# - freezes a Project 9 Step 5B checkpoint
#
# This cell does NOT:
# - rerun any experiment condition
# - write inside the Project 9 raw-result directory
# - access or modify Project 10
# - modify Projects 1–8
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ------------------------------------------------------------
# 1. FROZEN PROJECT 9 CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_SHORT_NAME = "camunda"


EXPECTED_STEP5A_STATUS = (
    "PASS_PROJECT_9_FULL_270_CONDITION_EXPERIMENT_COMPLETED_CHECKPOINTED"
)

STEP5B_PASS_STATUS = (
    "PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
)


EXPECTED_FULL_RUN_CHECKPOINT_SHA256 = (
    "22f9f1184be247d83941199381f12b7042f743158403e765ee8ffcfe09636519"
)

EXPECTED_FROZEN_RAW_ROOT_SHA256 = (
    "c31cb45e1354dc1222b82103bd275c21b10dd2ba6171125005d9c0a99ab14724"
)


NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_TECHNIQUES = 7

EXPECTED_RANKING_ROWS = 34_970_670
EXPECTED_BUILD_METRIC_ROWS = 56_700
EXPECTED_PROJECT_RUN_ROWS = 1_890

EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 387_066_081

EXPECTED_RANKING_ROWS_PER_CONDITION = 129_521
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = 210
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = 7
EXPECTED_MODEL_FITS_PER_CONDITION = 4
EXPECTED_SCORED_BUILDS_PER_RUN = 30
EXPECTED_EVALUATION_FAILURES = 665

EXPECTED_FILES_PER_CONDITION = 8


REQUIRED_CONDITION_FILES = [
    "rankings.parquet",
    "build_metrics.csv",
    "project_run.csv",
    "model_fits.csv",
    "condition_audit.csv",
    "training_medians.csv",
    "condition_summary.json",
    "_SUCCESS.json",
]


# ------------------------------------------------------------
# 2. PATHS — PROJECT 9 ONLY
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


FULL_RUN_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_full_run_checkpoint.json"
)

STEP5B_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_09_step5b_checkpoint.json"
)


RAW_ROOT = (
    RESULTS_DIR
    / "Raw"
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

STEP5B_DIR = (
    PROJECT_AGGREGATED_DIR
    / "camunda_step5b_final_audit"
)


RAW_FILE_MANIFEST_PATH = (
    STEP5B_DIR
    / "camunda_raw_file_manifest.csv"
)

CONDITION_INVENTORY_PATH = (
    STEP5B_DIR
    / "camunda_condition_inventory.csv"
)

PROJECT_RUN_ALL_PATH = (
    STEP5B_DIR
    / "camunda_project_run_all.csv"
)

BUILD_METRICS_ALL_PATH = (
    STEP5B_DIR
    / "camunda_build_metrics_all.parquet"
)

MODEL_FITS_ALL_PATH = (
    STEP5B_DIR
    / "camunda_model_fits_all.csv"
)

CONDITION_AUDIT_ALL_PATH = (
    STEP5B_DIR
    / "camunda_condition_audit_all.csv"
)

TRAINING_MEDIANS_ALL_PATH = (
    STEP5B_DIR
    / "camunda_training_medians_all.parquet"
)

NOISE_TECHNIQUE_SUMMARY_PATH = (
    STEP5B_DIR
    / "camunda_noise_technique_summary.csv"
)

SEED_LEVEL_DELTAS_PATH = (
    STEP5B_DIR
    / "camunda_seed_level_noise_deltas.csv"
)

NOISE_DELTA_SUMMARY_PATH = (
    STEP5B_DIR
    / "camunda_noise_delta_summary.csv"
)

CLEAN_TECHNIQUE_SUMMARY_PATH = (
    STEP5B_DIR
    / "camunda_clean_technique_summary.csv"
)

INVARIANCE_AUDIT_PATH = (
    STEP5B_DIR
    / "camunda_baseline_invariance_audit.csv"
)

STEP5B_VALIDATION_PATH = (
    STEP5B_DIR
    / "camunda_step5b_validation.csv"
)

STEP5B_REPORT_PATH = (
    STEP5B_DIR
    / "camunda_step5b_report.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / "camunda_step5b_status.json"
)


PROJECT_10_ROOT = (
    RESULTS_DIR
    / "Raw"
    / "spring-cloud__spring-cloud-dataflow"
)


print("=" * 126)
print("=== PROJECT 9 CELL 10 / STEP 5B: FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION ===")
print("=" * 126)


# ------------------------------------------------------------
# 3. BASIC HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_parquet(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.stem + ".tmp.parquet"
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        compression="snappy",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_json(
    path,
):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def first_mapping_value(
    mapping,
    keys,
    default=None,
):
    for key in keys:
        if key in mapping:
            value = mapping[
                key
            ]

            if value is not None:
                return value

    return default


def recursive_find_value(
    value,
    candidate_keys,
):
    candidate_keys_lower = {
        str(key).lower()
        for key in candidate_keys
    }

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            if (
                str(key).lower()
                in candidate_keys_lower
            ):
                return child

        for child in value.values():
            found = recursive_find_value(
                child,
                candidate_keys,
            )

            if found is not None:
                return found

    elif isinstance(
        value,
        list,
    ):
        for child in value:
            found = recursive_find_value(
                child,
                candidate_keys,
            )

            if found is not None:
                return found

    return None


def resolve_column(
    dataframe,
    aliases,
    label,
    required=True,
):
    exact_lookup = {
        str(column).lower():
            column
        for column in dataframe.columns
    }

    for alias in aliases:
        if alias in dataframe.columns:
            return alias

        alias_lower = str(
            alias
        ).lower()

        if alias_lower in exact_lookup:
            return exact_lookup[
                alias_lower
            ]

    if required:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Aliases: {aliases}\n"
            f"Columns: {dataframe.columns.tolist()}"
        )

    return None


def parse_condition_key(
    condition_key,
):
    match = re.fullmatch(
        r"noise_(\d+)__seed_(\d+)",
        str(
            condition_key
        ),
    )

    if match is None:
        raise RuntimeError(
            "Could not parse condition key:\n"
            f"{condition_key}"
        )

    return (
        int(
            match.group(1)
        ),
        int(
            match.group(2)
        ),
    )


def expected_condition_order(
    noise_percent,
    repetition_seed,
):
    return (
        (
            int(
                repetition_seed
            )
            - 1
        )
        * len(
            NOISE_LEVELS
        )
        + NOISE_LEVELS.index(
            int(
                noise_percent
            )
        )
        + 1
    )


def condition_status_passes(
    status,
):
    status_text = str(
        status
    ).upper()

    return (
        status_text.startswith(
            "PASS"
        )
        and "CONDITION" in status_text
    )


def normalise_condition_frame(
    dataframe,
    condition_key,
    noise_percent,
    repetition_seed,
):
    dataframe = dataframe.copy()

    condition_column = resolve_column(
        dataframe,
        [
            "ConditionKey",
            "ConditionID",
            "condition_key",
            "condition_id",
        ],
        "condition identifier",
        required=False,
    )

    if condition_column is not None:
        observed = set(
            dataframe[
                condition_column
            ]
            .dropna()
            .astype(str)
            .unique()
        )

        if observed and observed != {
            str(
                condition_key
            )
        }:
            raise RuntimeError(
                f"{condition_key}: condition identifier differs "
                f"inside {condition_column}: {observed}"
            )

    noise_column = resolve_column(
        dataframe,
        [
            "NoisePercent",
            "NoiseLevel",
            "Noise",
            "noise_percent",
        ],
        "noise percent",
        required=False,
    )

    if noise_column is not None:
        observed_noise = set(
            pd.to_numeric(
                dataframe[
                    noise_column
                ],
                errors="raise",
            )
            .astype(int)
            .unique()
        )

        if observed_noise and observed_noise != {
            int(
                noise_percent
            )
        }:
            raise RuntimeError(
                f"{condition_key}: noise value differs "
                f"inside {noise_column}: {observed_noise}"
            )

    seed_column = resolve_column(
        dataframe,
        [
            "RepetitionSeed",
            "Seed",
            "seed",
            "repetition_seed",
        ],
        "repetition seed",
        required=False,
    )

    if seed_column is not None:
        observed_seeds = set(
            pd.to_numeric(
                dataframe[
                    seed_column
                ],
                errors="raise",
            )
            .astype(int)
            .unique()
        )

        if observed_seeds and observed_seeds != {
            int(
                repetition_seed
            )
        }:
            raise RuntimeError(
                f"{condition_key}: seed differs "
                f"inside {seed_column}: {observed_seeds}"
            )

    dataframe[
        "ConditionKey"
    ] = str(
        condition_key
    )

    dataframe[
        "NoisePercent"
    ] = int(
        noise_percent
    )

    dataframe[
        "RepetitionSeed"
    ] = int(
        repetition_seed
    )

    if "ProjectNumber" not in dataframe.columns:
        dataframe[
            "ProjectNumber"
        ] = PROJECT_NUMBER

    if "Project" not in dataframe.columns:
        dataframe[
            "Project"
        ] = PROJECT_NAME

    if "ProjectSlug" not in dataframe.columns:
        dataframe[
            "ProjectSlug"
        ] = PROJECT_SLUG

    return dataframe


def extract_embedded_hashes(
    marker,
):
    output = {}

    hash_container_keys = [
        "FileSHA256",
        "FileHashes",
        "OutputHashes",
        "DataFileSHA256",
        "SHA256",
    ]

    def visit(value):
        if isinstance(
            value,
            dict,
        ):
            for key, child in value.items():
                if (
                    key in hash_container_keys
                    and isinstance(
                        child,
                        dict,
                    )
                ):
                    for filename, digest in child.items():
                        if isinstance(
                            digest,
                            str,
                        ) and re.fullmatch(
                            r"[0-9a-fA-F]{64}",
                            digest,
                        ):
                            output[
                                Path(
                                    filename
                                ).name
                            ] = digest.lower()

                visit(
                    child
                )

        elif isinstance(
            value,
            list,
        ):
            for child in value:
                visit(
                    child
                )

    visit(
        marker
    )

    return output


# ------------------------------------------------------------
# 4. LOAD AND VALIDATE PROJECT 9 STEP 5A
# ------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    FULL_RUN_CHECKPOINT_PATH,
    RAW_ROOT,
]


missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 9 Step 5B inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


if PROJECT_SLUG not in str(
    RAW_ROOT
):
    raise AssertionError(
        "Raw root is not isolated to Project 9."
    )


if str(
    PROJECT_10_ROOT
) in str(
    RAW_ROOT
):
    raise AssertionError(
        "Project 9 raw root overlaps Project 10."
    )


full_run_checkpoint_sha256 = calculate_hash(
    FULL_RUN_CHECKPOINT_PATH
)


if (
    full_run_checkpoint_sha256
    != EXPECTED_FULL_RUN_CHECKPOINT_SHA256
):
    raise AssertionError(
        "Project 9 full-run checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_FULL_RUN_CHECKPOINT_SHA256}\n"
        f"Actual:   {full_run_checkpoint_sha256}"
    )


full_run_checkpoint = read_json(
    FULL_RUN_CHECKPOINT_PATH
)


full_run_status = recursive_find_value(
    full_run_checkpoint,
    [
        "Status",
    ],
)


if full_run_status != EXPECTED_STEP5A_STATUS:
    raise AssertionError(
        "Project 9 Step 5A checkpoint status differs.\n"
        f"Expected: {EXPECTED_STEP5A_STATUS}\n"
        f"Actual:   {full_run_status}"
    )


checkpoint_raw_hash = recursive_find_value(
    full_run_checkpoint,
    [
        "RawRootSHA256",
        "RawResultsRootSHA256",
        "RawResultRootSHA256",
    ],
)


if checkpoint_raw_hash is not None:
    if (
        str(
            checkpoint_raw_hash
        )
        != EXPECTED_FROZEN_RAW_ROOT_SHA256
    ):
        raise AssertionError(
            "Project 9 Step 5A frozen raw-root hash differs.\n"
            f"Expected: {EXPECTED_FROZEN_RAW_ROOT_SHA256}\n"
            f"Actual:   {checkpoint_raw_hash}"
        )


checkpoint_conditions = recursive_find_value(
    full_run_checkpoint,
    [
        "CompletedConditions",
        "Conditions",
    ],
)


if (
    checkpoint_conditions is not None
    and int(
        checkpoint_conditions
    ) != EXPECTED_CONDITIONS
):
    raise AssertionError(
        "Project 9 Step 5A completed-condition count differs."
    )


# ------------------------------------------------------------
# 5. REGISTRY SNAPSHOT — READ ONLY
# ------------------------------------------------------------

registry_sha256_before = calculate_hash(
    REGISTRY_PATH
)

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry,
    [
        "ProjectNumber",
        "project_number",
    ],
    "registry project number",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_before = int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)


if registry_project9_rows_before != 0:
    raise AssertionError(
        "Project 9 is already present in the registry. "
        "Step 5B expects registration to occur only after "
        "the final package is validated."
    )


# ------------------------------------------------------------
# 6. DISCOVER ALL PROJECT 9 CONDITION DIRECTORIES
# ------------------------------------------------------------

success_markers = sorted(
    RAW_ROOT.rglob(
        "_SUCCESS.json"
    ),
    key=lambda path:
        path.relative_to(
            RAW_ROOT
        ).as_posix(),
)


if len(
    success_markers
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Project 9 success-marker count differs.\n"
        f"Expected: {EXPECTED_CONDITIONS}\n"
        f"Actual:   {len(success_markers)}"
    )


condition_records = []
raw_file_records = []

ranking_rows_total = 0
embedded_hash_checks = 0
embedded_hash_mismatches = 0

raw_hash_started = time.perf_counter()


for marker_path in success_markers:
    condition_directory = marker_path.parent

    marker = read_json(
        marker_path
    )

    summary_path = (
        condition_directory
        / "condition_summary.json"
    )

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Missing condition summary: {summary_path}"
        )

    summary = read_json(
        summary_path
    )


    condition_key = first_mapping_value(
        marker,
        [
            "ConditionKey",
            "ConditionID",
        ],
        default=None,
    )

    if condition_key is None:
        condition_key = first_mapping_value(
            summary,
            [
                "ConditionKey",
                "ConditionID",
            ],
            default=None,
        )

    if condition_key is None:
        condition_key = (
            condition_directory.name
        )


    noise_percent, repetition_seed = (
        parse_condition_key(
            condition_key
        )
    )


    marker_noise = first_mapping_value(
        marker,
        [
            "NoisePercent",
            "NoiseLevel",
            "Noise",
        ],
        default=noise_percent,
    )

    marker_seed = first_mapping_value(
        marker,
        [
            "RepetitionSeed",
            "Seed",
        ],
        default=repetition_seed,
    )


    if int(
        marker_noise
    ) != noise_percent:
        raise AssertionError(
            f"{condition_key}: marker noise differs."
        )


    if int(
        marker_seed
    ) != repetition_seed:
        raise AssertionError(
            f"{condition_key}: marker seed differs."
        )


    marker_status = first_mapping_value(
        marker,
        [
            "Status",
        ],
        default="",
    )


    if not condition_status_passes(
        marker_status
    ):
        raise AssertionError(
            f"{condition_key}: success marker status is invalid.\n"
            f"Status: {marker_status}"
        )


    summary_status = first_mapping_value(
        summary,
        [
            "Status",
        ],
        default=marker_status,
    )


    if not condition_status_passes(
        summary_status
    ):
        raise AssertionError(
            f"{condition_key}: condition summary status is invalid.\n"
            f"Status: {summary_status}"
        )


    missing_condition_files = [
        filename
        for filename in REQUIRED_CONDITION_FILES
        if not (
            condition_directory
            / filename
        ).exists()
    ]


    if missing_condition_files:
        raise FileNotFoundError(
            f"{condition_key}: required files are missing:\n"
            + "\n".join(
                missing_condition_files
            )
        )


    actual_condition_files = sorted([
        path.name
        for path in condition_directory.iterdir()
        if path.is_file()
    ])


    unexpected_condition_files = sorted(
        set(
            actual_condition_files
        )
        - set(
            REQUIRED_CONDITION_FILES
        )
    )


    missing_expected_names = sorted(
        set(
            REQUIRED_CONDITION_FILES
        )
        - set(
            actual_condition_files
        )
    )


    if missing_expected_names:
        raise AssertionError(
            f"{condition_key}: condition file set is incomplete:\n"
            + "\n".join(
                missing_expected_names
            )
        )


    if unexpected_condition_files:
        raise AssertionError(
            f"{condition_key}: unexpected raw files were found:\n"
            + "\n".join(
                unexpected_condition_files
            )
        )


    embedded_hashes = extract_embedded_hashes(
        marker
    )


    condition_file_bytes = 0


    for filename in REQUIRED_CONDITION_FILES:
        file_path = (
            condition_directory
            / filename
        )

        relative_path = (
            file_path.relative_to(
                RAW_ROOT
            ).as_posix()
        )

        file_size = int(
            file_path.stat().st_size
        )

        file_sha256 = calculate_hash(
            file_path
        )

        condition_file_bytes += file_size

        embedded_hash_available = (
            filename in embedded_hashes
        )

        embedded_hash_match = None

        if embedded_hash_available:
            embedded_hash_checks += 1

            embedded_hash_match = bool(
                embedded_hashes[
                    filename
                ]
                == file_sha256
            )

            if not embedded_hash_match:
                embedded_hash_mismatches += 1


        parquet_rows = None
        parquet_columns = None

        if filename == "rankings.parquet":
            parquet_file = pq.ParquetFile(
                file_path
            )

            parquet_rows = int(
                parquet_file.metadata.num_rows
            )

            parquet_columns = int(
                parquet_file.metadata.num_columns
            )

            ranking_rows_total += (
                parquet_rows
            )

            if (
                parquet_rows
                != EXPECTED_RANKING_ROWS_PER_CONDITION
            ):
                raise AssertionError(
                    f"{condition_key}: ranking-row count differs.\n"
                    f"Expected: {EXPECTED_RANKING_ROWS_PER_CONDITION}\n"
                    f"Actual:   {parquet_rows}"
                )


        raw_file_records.append({
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "ConditionOrder":
                expected_condition_order(
                    noise_percent,
                    repetition_seed,
                ),

            "ConditionKey":
                str(
                    condition_key
                ),

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "RelativePath":
                relative_path,

            "FileName":
                filename,

            "FileSizeBytes":
                file_size,

            "SHA256":
                file_sha256,

            "EmbeddedHashAvailable":
                embedded_hash_available,

            "EmbeddedHashMatch":
                embedded_hash_match,

            "ParquetRows":
                parquet_rows,

            "ParquetColumns":
                parquet_columns,
        })


    condition_records.append({
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            expected_condition_order(
                noise_percent,
                repetition_seed,
            ),

        "ConditionKey":
            str(
                condition_key
            ),

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "ConditionDirectory":
            str(
                condition_directory
            ),

        "MarkerStatus":
            str(
                marker_status
            ),

        "SummaryStatus":
            str(
                summary_status
            ),

        "Files":
            len(
                actual_condition_files
            ),

        "ConditionBytes":
            condition_file_bytes,

        "RankingRows":
            int(
                pq.ParquetFile(
                    condition_directory
                    / "rankings.parquet"
                ).metadata.num_rows
            ),

        "MarkerSHA256":
            calculate_hash(
                marker_path
            ),

        "SummarySHA256":
            calculate_hash(
                summary_path
            ),
    })


raw_hash_seconds = float(
    time.perf_counter()
    - raw_hash_started
)


raw_file_manifest = (
    pd.DataFrame(
        raw_file_records
    )
    .sort_values(
        [
            "ConditionOrder",
            "FileName",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


condition_inventory = (
    pd.DataFrame(
        condition_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 7. RAW-ROOT INVENTORY AND INDEPENDENT ROOT HASH
# ------------------------------------------------------------

raw_files_on_disk = sorted([
    path
    for path in RAW_ROOT.rglob("*")
    if path.is_file()
], key=lambda path:
    path.relative_to(
        RAW_ROOT
    ).as_posix())


actual_raw_files = len(
    raw_files_on_disk
)

actual_raw_bytes = int(
    sum(
        path.stat().st_size
        for path in raw_files_on_disk
    )
)


manifest_relative_paths = set(
    raw_file_manifest[
        "RelativePath"
    ].astype(str)
)

disk_relative_paths = {
    path.relative_to(
        RAW_ROOT
    ).as_posix()
    for path in raw_files_on_disk
}


manifest_missing_disk_files = len(
    manifest_relative_paths
    - disk_relative_paths
)

manifest_unexpected_disk_files = len(
    disk_relative_paths
    - manifest_relative_paths
)


root_digest = hashlib.sha256()


for row in raw_file_manifest.sort_values(
    "RelativePath",
    kind="mergesort",
).itertuples(
    index=False
):
    root_digest.update(
        (
            f"{row.RelativePath}\0"
            f"{int(row.FileSizeBytes)}\0"
            f"{row.SHA256}\n"
        ).encode(
            "utf-8"
        )
    )


independent_raw_root_sha256 = (
    root_digest.hexdigest()
)


raw_manifest_sha256 = hashlib.sha256(
    raw_file_manifest[
        [
            "RelativePath",
            "FileSizeBytes",
            "SHA256",
        ]
    ]
    .to_csv(
        index=False,
        lineterminator="\n",
    )
    .encode(
        "utf-8"
    )
).hexdigest()


# ------------------------------------------------------------
# 8. CONDITION-COORDINATE VALIDATION
# ------------------------------------------------------------

expected_coordinate_set = {
    (
        noise_percent,
        repetition_seed,
    )
    for repetition_seed in REPETITION_SEEDS
    for noise_percent in NOISE_LEVELS
}


actual_coordinate_set = set(
    zip(
        condition_inventory[
            "NoisePercent"
        ].astype(int),

        condition_inventory[
            "RepetitionSeed"
        ].astype(int),
    )
)


duplicate_condition_keys = int(
    condition_inventory[
        "ConditionKey"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_inventory.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


condition_order_violations = int(
    condition_inventory.apply(
        lambda row:
            int(
                row[
                    "ConditionOrder"
                ]
            )
            != expected_condition_order(
                int(
                    row[
                        "NoisePercent"
                    ]
                ),
                int(
                    row[
                        "RepetitionSeed"
                    ]
                ),
            ),
        axis=1,
    ).sum()
)


files_per_condition_violations = int(
    condition_inventory[
        "Files"
    ].ne(
        EXPECTED_FILES_PER_CONDITION
    ).sum()
)


ranking_rows_per_condition_violations = int(
    condition_inventory[
        "RankingRows"
    ].ne(
        EXPECTED_RANKING_ROWS_PER_CONDITION
    ).sum()
)


# ------------------------------------------------------------
# 9. LOAD AND COMBINE COMPACT CONDITION FILES
# ------------------------------------------------------------

def load_all_condition_csvs(
    filename,
):
    frames = []

    for row in condition_inventory.itertuples(
        index=False
    ):
        condition_directory = Path(
            row.ConditionDirectory
        )

        file_path = (
            condition_directory
            / filename
        )

        frame = pd.read_csv(
            file_path,
            low_memory=False,
        )

        frame = normalise_condition_frame(
            dataframe=frame,
            condition_key=row.ConditionKey,
            noise_percent=row.NoisePercent,
            repetition_seed=row.RepetitionSeed,
        )

        frames.append(
            frame
        )

    return pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )


aggregation_started = time.perf_counter()


project_run_all = load_all_condition_csvs(
    "project_run.csv"
)

build_metrics_all = load_all_condition_csvs(
    "build_metrics.csv"
)

model_fits_all = load_all_condition_csvs(
    "model_fits.csv"
)

condition_audit_all = load_all_condition_csvs(
    "condition_audit.csv"
)

training_medians_all = load_all_condition_csvs(
    "training_medians.csv"
)


aggregation_seconds = float(
    time.perf_counter()
    - aggregation_started
)


# ------------------------------------------------------------
# 10. RESOLVE COMPACT-OUTPUT SCHEMA
# ------------------------------------------------------------

project_run_technique_column = resolve_column(
    project_run_all,
    [
        "Technique",
        "Method",
        "Model",
        "TCPTechnique",
    ],
    "project-run technique",
)

project_run_apfd_column = resolve_column(
    project_run_all,
    [
        "MeanAPFD",
        "APFD",
    ],
    "project-run APFD",
)

project_run_apfdc_column = resolve_column(
    project_run_all,
    [
        "MeanAPFDc",
        "APFDc",
        "MeanAPFDC",
        "APFDC",
    ],
    "project-run APFDc",
)


evaluated_builds_column = resolve_column(
    project_run_all,
    [
        "EvaluatedBuilds",
        "EvaluationBuilds",
        "Builds",
    ],
    "project-run evaluated builds",
    required=False,
)


evaluation_failures_column = resolve_column(
    project_run_all,
    [
        "EvaluationFailures",
        "Failures",
    ],
    "project-run evaluation failures",
    required=False,
)


build_metric_technique_column = resolve_column(
    build_metrics_all,
    [
        "Technique",
        "Method",
        "Model",
        "TCPTechnique",
    ],
    "build-metric technique",
)

build_metric_build_column = resolve_column(
    build_metrics_all,
    [
        "BuildKey",
        "Build",
        "build",
    ],
    "build-metric build",
)

build_metric_apfd_column = resolve_column(
    build_metrics_all,
    [
        "APFD",
    ],
    "build-metric APFD",
)

build_metric_apfdc_column = resolve_column(
    build_metrics_all,
    [
        "APFDc",
        "APFDC",
    ],
    "build-metric APFDc",
)


model_fit_technique_column = resolve_column(
    model_fits_all,
    [
        "Technique",
        "Model",
        "Method",
    ],
    "model-fit technique",
)

model_fit_status_column = resolve_column(
    model_fits_all,
    [
        "FitStatus",
        "Status",
    ],
    "model-fit status",
    required=False,
)


training_median_feature_column = resolve_column(
    training_medians_all,
    [
        "Feature",
        "Predictor",
        "Column",
    ],
    "training-median feature",
)

training_median_value_column = resolve_column(
    training_medians_all,
    [
        "TrainingMedian",
        "Median",
        "Value",
    ],
    "training-median value",
)


# ------------------------------------------------------------
# 11. CANONICAL COMPACT TABLES
# ------------------------------------------------------------

project_run_canonical = pd.DataFrame({
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ConditionKey":
        project_run_all[
            "ConditionKey"
        ].astype(str),

    "NoisePercent":
        pd.to_numeric(
            project_run_all[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(int),

    "RepetitionSeed":
        pd.to_numeric(
            project_run_all[
                "RepetitionSeed"
            ],
            errors="raise",
        ).astype(int),

    "Technique":
        project_run_all[
            project_run_technique_column
        ].astype(str),

    "MeanAPFD":
        pd.to_numeric(
            project_run_all[
                project_run_apfd_column
            ],
            errors="raise",
        ).astype(float),

    "MeanAPFDc":
        pd.to_numeric(
            project_run_all[
                project_run_apfdc_column
            ],
            errors="raise",
        ).astype(float),
})


if evaluated_builds_column is not None:
    project_run_canonical[
        "EvaluatedBuilds"
    ] = pd.to_numeric(
        project_run_all[
            evaluated_builds_column
        ],
        errors="raise",
    ).astype(int)


if evaluation_failures_column is not None:
    project_run_canonical[
        "EvaluationFailures"
    ] = pd.to_numeric(
        project_run_all[
            evaluation_failures_column
        ],
        errors="raise",
    ).astype(int)


build_metrics_canonical = pd.DataFrame({
    "ConditionKey":
        build_metrics_all[
            "ConditionKey"
        ].astype(str),

    "NoisePercent":
        pd.to_numeric(
            build_metrics_all[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(int),

    "RepetitionSeed":
        pd.to_numeric(
            build_metrics_all[
                "RepetitionSeed"
            ],
            errors="raise",
        ).astype(int),

    "Technique":
        build_metrics_all[
            build_metric_technique_column
        ].astype(str),

    "BuildKey":
        build_metrics_all[
            build_metric_build_column
        ].astype(str),

    "APFD":
        pd.to_numeric(
            build_metrics_all[
                build_metric_apfd_column
            ],
            errors="raise",
        ).astype(float),

    "APFDc":
        pd.to_numeric(
            build_metrics_all[
                build_metric_apfdc_column
            ],
            errors="raise",
        ).astype(float),
})


model_fits_canonical = pd.DataFrame({
    "ConditionKey":
        model_fits_all[
            "ConditionKey"
        ].astype(str),

    "NoisePercent":
        pd.to_numeric(
            model_fits_all[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(int),

    "RepetitionSeed":
        pd.to_numeric(
            model_fits_all[
                "RepetitionSeed"
            ],
            errors="raise",
        ).astype(int),

    "Technique":
        model_fits_all[
            model_fit_technique_column
        ].astype(str),
})


if model_fit_status_column is not None:
    model_fits_canonical[
        "FitStatus"
    ] = model_fits_all[
        model_fit_status_column
    ].astype(str)


training_medians_canonical = pd.DataFrame({
    "ConditionKey":
        training_medians_all[
            "ConditionKey"
        ].astype(str),

    "NoisePercent":
        pd.to_numeric(
            training_medians_all[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(int),

    "RepetitionSeed":
        pd.to_numeric(
            training_medians_all[
                "RepetitionSeed"
            ],
            errors="raise",
        ).astype(int),

    "Feature":
        training_medians_all[
            training_median_feature_column
        ].astype(str),

    "TrainingMedian":
        pd.to_numeric(
            training_medians_all[
                training_median_value_column
            ],
            errors="raise",
        ).astype(float),
})


# ------------------------------------------------------------
# 12. STRUCTURAL VALIDATION
# ------------------------------------------------------------

project_run_duplicate_rows = int(
    project_run_canonical.duplicated(
        subset=[
            "ConditionKey",
            "Technique",
        ],
        keep=False,
    ).sum()
)


build_metric_duplicate_rows = int(
    build_metrics_canonical.duplicated(
        subset=[
            "ConditionKey",
            "Technique",
            "BuildKey",
        ],
        keep=False,
    ).sum()
)


model_fit_duplicate_rows = int(
    model_fits_canonical.duplicated(
        subset=[
            "ConditionKey",
            "Technique",
        ],
        keep=False,
    ).sum()
)


condition_audit_duplicate_rows = int(
    condition_audit_all[
        "ConditionKey"
    ].duplicated(
        keep=False
    ).sum()
)


project_run_technique_set = set(
    project_run_canonical[
        "Technique"
    ].unique()
)


model_fit_technique_set = set(
    model_fits_canonical[
        "Technique"
    ].unique()
)


project_run_metric_nonfinite = int(
    (
        ~np.isfinite(
            project_run_canonical[
                [
                    "MeanAPFD",
                    "MeanAPFDc",
                ]
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


project_run_metric_out_of_range = int(
    (
        ~project_run_canonical[
            "MeanAPFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~project_run_canonical[
            "MeanAPFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


build_metric_nonfinite = int(
    (
        ~np.isfinite(
            build_metrics_canonical[
                [
                    "APFD",
                    "APFDc",
                ]
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


build_metric_out_of_range = int(
    (
        ~build_metrics_canonical[
            "APFD"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
    +
    (
        ~build_metrics_canonical[
            "APFDc"
        ].between(
            0.0,
            1.0,
            inclusive="both",
        )
    ).sum()
)


project_runs_per_condition_violations = int(
    project_run_canonical.groupby(
        "ConditionKey"
    ).size().ne(
        EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()
)


build_metrics_per_condition_violations = int(
    build_metrics_canonical.groupby(
        "ConditionKey"
    ).size().ne(
        EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()
)


model_fits_per_condition_violations = int(
    model_fits_canonical.groupby(
        "ConditionKey"
    ).size().ne(
        EXPECTED_MODEL_FITS_PER_CONDITION
    ).sum()
)


builds_per_condition_technique_violations = int(
    build_metrics_canonical.groupby(
        [
            "ConditionKey",
            "Technique",
        ]
    )[
        "BuildKey"
    ].nunique().ne(
        EXPECTED_SCORED_BUILDS_PER_RUN
    ).sum()
)


if evaluated_builds_column is not None:
    evaluated_build_violations = int(
        project_run_canonical[
            "EvaluatedBuilds"
        ].ne(
            EXPECTED_SCORED_BUILDS_PER_RUN
        ).sum()
    )
else:
    evaluated_build_violations = 0


if evaluation_failures_column is not None:
    evaluation_failure_violations = int(
        project_run_canonical[
            "EvaluationFailures"
        ].ne(
            EXPECTED_EVALUATION_FAILURES
        ).sum()
    )
else:
    evaluation_failure_violations = 0


if model_fit_status_column is not None:
    model_fit_failures = int(
        ~model_fits_canonical[
            "FitStatus"
        ]
        .str.upper()
        .str.contains(
            "SUCCESS|PASS",
            regex=True,
        )
    ).sum()
else:
    model_fit_failures = 0


training_median_rows_per_condition = (
    training_medians_canonical.groupby(
        "ConditionKey"
    ).size()
)


training_median_count_values = sorted(
    training_median_rows_per_condition
    .unique()
    .tolist()
)


training_median_count_consistent = bool(
    len(
        training_median_count_values
    ) == 1
)


if not training_median_count_consistent:
    raise RuntimeError(
        "Training-median row counts differ between conditions.\n"
        f"Observed counts: {training_median_count_values}"
    )


active_predictor_count = int(
    training_median_count_values[
        0
    ]
)


expected_training_median_rows = (
    EXPECTED_CONDITIONS
    * active_predictor_count
)


training_median_duplicate_rows = int(
    training_medians_canonical.duplicated(
        subset=[
            "ConditionKey",
            "Feature",
        ],
        keep=False,
    ).sum()
)


training_median_nonfinite = int(
    (
        ~np.isfinite(
            training_medians_canonical[
                "TrainingMedian"
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


reference_feature_set = set(
    training_medians_canonical.loc[
        training_medians_canonical[
            "ConditionKey"
        ].eq(
            condition_inventory[
                "ConditionKey"
            ].iloc[0]
        ),
        "Feature",
    ]
)


feature_set_violations = 0


for _, feature_group in (
    training_medians_canonical.groupby(
        "ConditionKey",
        sort=False,
    )
):
    if set(
        feature_group[
            "Feature"
        ]
    ) != reference_feature_set:
        feature_set_violations += 1


# ------------------------------------------------------------
# 13. CONDITION-AUDIT NOISE PROPAGATION
# ------------------------------------------------------------

audit_noise_column = resolve_column(
    condition_audit_all,
    [
        "NoisePercent",
        "NoiseLevel",
        "Noise",
    ],
    "condition-audit noise percent",
)

audit_raw_flips_column = resolve_column(
    condition_audit_all,
    [
        "RawRowsFlipped",
        "ActualRawFlips",
        "RawFlips",
        "NumberFlipped",
    ],
    "condition-audit raw flips",
)

audit_model_changes_column = resolve_column(
    condition_audit_all,
    [
        "ChangedModelLabels",
        "ActualModelLabelChanges",
        "ModelLabelChanges",
    ],
    "condition-audit model-label changes",
)

audit_rec_changes_column = resolve_column(
    condition_audit_all,
    [
        "ChangedDependentRECValues",
        "DependentRECChangedValues",
        "DependentRECChanges",
    ],
    "condition-audit dependent REC changes",
)


audit_noise = pd.to_numeric(
    condition_audit_all[
        audit_noise_column
    ],
    errors="raise",
).astype(int)

audit_raw_flips = pd.to_numeric(
    condition_audit_all[
        audit_raw_flips_column
    ],
    errors="raise",
).astype(int)

audit_model_changes = pd.to_numeric(
    condition_audit_all[
        audit_model_changes_column
    ],
    errors="raise",
).astype(int)

audit_rec_changes = pd.to_numeric(
    condition_audit_all[
        audit_rec_changes_column
    ],
    errors="raise",
).astype(int)


zero_noise_mask = audit_noise.eq(0)
positive_noise_mask = audit_noise.gt(0)


zero_noise_conditions = int(
    zero_noise_mask.sum()
)

zero_noise_raw_flip_violations = int(
    audit_raw_flips[
        zero_noise_mask
    ].ne(0).sum()
)

zero_noise_model_change_violations = int(
    audit_model_changes[
        zero_noise_mask
    ].ne(0).sum()
)

zero_noise_rec_change_violations = int(
    audit_rec_changes[
        zero_noise_mask
    ].ne(0).sum()
)

positive_noise_without_model_changes = int(
    audit_model_changes[
        positive_noise_mask
    ].le(0).sum()
)

positive_noise_without_rec_changes = int(
    audit_rec_changes[
        positive_noise_mask
    ].le(0).sum()
)


# ------------------------------------------------------------
# 14. BASELINE INVARIANCE VALIDATION
# ------------------------------------------------------------

baseline_invariance_records = []


for technique in [
    "Random",
    "QTF-Avg",
]:
    technique_rows = (
        project_run_canonical[
            project_run_canonical[
                "Technique"
            ].eq(
                technique
            )
        ]
        .copy()
    )

    seed_violations = 0

    for _, seed_rows in technique_rows.groupby(
        "RepetitionSeed",
        sort=True,
    ):
        apfd_unique = seed_rows[
            "MeanAPFD"
        ].nunique(
            dropna=False
        )

        apfdc_unique = seed_rows[
            "MeanAPFDc"
        ].nunique(
            dropna=False
        )

        if (
            apfd_unique != 1
            or apfdc_unique != 1
        ):
            seed_violations += 1

    baseline_invariance_records.append({
        "Technique":
            technique,

        "Expected":
            (
                "Metrics constant across noise "
                "within each repetition seed"
            ),

        "SeedsChecked":
            int(
                technique_rows[
                    "RepetitionSeed"
                ].nunique()
            ),

        "ViolatingSeeds":
            seed_violations,

        "Pass":
            seed_violations == 0,
    })


latest_fail_rows = (
    project_run_canonical[
        project_run_canonical[
            "Technique"
        ].eq(
            "LatestFail"
        )
    ]
    .copy()
)


latest_fail_noise_response_seeds = 0


for _, seed_rows in latest_fail_rows.groupby(
    "RepetitionSeed",
    sort=True,
):
    if (
        seed_rows[
            "MeanAPFD"
        ].nunique(
            dropna=False
        ) > 1
        or seed_rows[
            "MeanAPFDc"
        ].nunique(
            dropna=False
        ) > 1
    ):
        latest_fail_noise_response_seeds += 1


baseline_invariance_records.append({
    "Technique":
        "LatestFail",

    "Expected":
        (
            "At least one repetition seed changes "
            "across noise levels"
        ),

    "SeedsChecked":
        int(
            latest_fail_rows[
                "RepetitionSeed"
            ].nunique()
        ),

    "ViolatingSeeds":
        (
            0
            if latest_fail_noise_response_seeds > 0
            else len(
                REPETITION_SEEDS
            )
        ),

    "Pass":
        latest_fail_noise_response_seeds > 0,
})


baseline_invariance_audit = pd.DataFrame(
    baseline_invariance_records
)


baseline_invariance_failures = int(
    (
        ~baseline_invariance_audit[
            "Pass"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 15. ANALYSIS-READY SUMMARIES
# ------------------------------------------------------------

noise_technique_summary = (
    project_run_canonical.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Runs=(
            "RepetitionSeed",
            "count",
        ),

        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        StdAPFD=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaxAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        StdAPFDc=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaxAPFDc=(
            "MeanAPFDc",
            "max",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


clean_reference = (
    project_run_canonical[
        project_run_canonical[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


seed_level_noise_deltas = (
    project_run_canonical.merge(
        clean_reference,
        on=[
            "RepetitionSeed",
            "Technique",
        ],
        how="left",
        validate="many_to_one",
    )
)


seed_level_noise_deltas[
    "APFDDeltaFromClean"
] = (
    seed_level_noise_deltas[
        "MeanAPFD"
    ]
    - seed_level_noise_deltas[
        "CleanMeanAPFD"
    ]
)


seed_level_noise_deltas[
    "APFDcDeltaFromClean"
] = (
    seed_level_noise_deltas[
        "MeanAPFDc"
    ]
    - seed_level_noise_deltas[
        "CleanMeanAPFDc"
    ]
)


seed_level_noise_deltas[
    "APFDRelativeDegradation"
] = np.where(
    seed_level_noise_deltas[
        "CleanMeanAPFD"
    ].ne(0),

    (
        seed_level_noise_deltas[
            "CleanMeanAPFD"
        ]
        - seed_level_noise_deltas[
            "MeanAPFD"
        ]
    )
    / seed_level_noise_deltas[
        "CleanMeanAPFD"
    ],

    np.nan,
)


seed_level_noise_deltas[
    "APFDcRelativeDegradation"
] = np.where(
    seed_level_noise_deltas[
        "CleanMeanAPFDc"
    ].ne(0),

    (
        seed_level_noise_deltas[
            "CleanMeanAPFDc"
        ]
        - seed_level_noise_deltas[
            "MeanAPFDc"
        ]
    )
    / seed_level_noise_deltas[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


noise_delta_summary = (
    seed_level_noise_deltas.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFDDeltaFromClean=(
            "APFDDeltaFromClean",
            "mean",
        ),

        MedianAPFDDeltaFromClean=(
            "APFDDeltaFromClean",
            "median",
        ),

        MeanAPFDcDeltaFromClean=(
            "APFDcDeltaFromClean",
            "mean",
        ),

        MedianAPFDcDeltaFromClean=(
            "APFDcDeltaFromClean",
            "median",
        ),

        MeanAPFDRelativeDegradation=(
            "APFDRelativeDegradation",
            "mean",
        ),

        MedianAPFDRelativeDegradation=(
            "APFDRelativeDegradation",
            "median",
        ),

        MeanAPFDcRelativeDegradation=(
            "APFDcRelativeDegradation",
            "mean",
        ),

        MedianAPFDcRelativeDegradation=(
            "APFDcRelativeDegradation",
            "median",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


clean_technique_summary = (
    noise_technique_summary[
        noise_technique_summary[
            "NoisePercent"
        ].eq(0)
    ]
    .sort_values(
        "MeanAPFDc",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


noise_summary_rows = len(
    noise_technique_summary
)

noise_delta_summary_rows = len(
    noise_delta_summary
)


# ------------------------------------------------------------
# 16. PRE-WRITE VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 5A passed",

        "Expected":
            EXPECTED_STEP5A_STATUS,

        "Actual":
            full_run_status,

        "Pass":
            full_run_status
            == EXPECTED_STEP5A_STATUS,
    },
    {
        "Check":
            "Step 5A checkpoint SHA-256",

        "Expected":
            EXPECTED_FULL_RUN_CHECKPOINT_SHA256,

        "Actual":
            full_run_checkpoint_sha256,

        "Pass":
            full_run_checkpoint_sha256
            == EXPECTED_FULL_RUN_CHECKPOINT_SHA256,
    },
    {
        "Check":
            "Frozen Step 5A raw-root SHA-256",

        "Expected":
            EXPECTED_FROZEN_RAW_ROOT_SHA256,

        "Actual":
            (
                checkpoint_raw_hash
                if checkpoint_raw_hash is not None
                else EXPECTED_FROZEN_RAW_ROOT_SHA256
            ),

        "Pass":
            (
                checkpoint_raw_hash is None
                or str(
                    checkpoint_raw_hash
                )
                == EXPECTED_FROZEN_RAW_ROOT_SHA256
            ),
    },
    {
        "Check":
            "Condition success markers",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                success_markers
            ),

        "Pass":
            len(
                success_markers
            )
            == EXPECTED_CONDITIONS,
    },
    {
        "Check":
            "Condition inventory rows",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                condition_inventory
            ),

        "Pass":
            len(
                condition_inventory
            )
            == EXPECTED_CONDITIONS,
    },
    {
        "Check":
            "Condition coordinates",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                actual_coordinate_set
            ),

        "Pass":
            actual_coordinate_set
            == expected_coordinate_set,
    },
    {
        "Check":
            "Duplicate condition keys",

        "Expected":
            0,

        "Actual":
            duplicate_condition_keys,

        "Pass":
            duplicate_condition_keys == 0,
    },
    {
        "Check":
            "Duplicate condition coordinates",

        "Expected":
            0,

        "Actual":
            duplicate_condition_coordinates,

        "Pass":
            duplicate_condition_coordinates == 0,
    },
    {
        "Check":
            "Condition-order violations",

        "Expected":
            0,

        "Actual":
            condition_order_violations,

        "Pass":
            condition_order_violations == 0,
    },
    {
        "Check":
            "Files-per-condition violations",

        "Expected":
            0,

        "Actual":
            files_per_condition_violations,

        "Pass":
            files_per_condition_violations == 0,
    },
    {
        "Check":
            "Raw files",

        "Expected":
            EXPECTED_RAW_FILES,

        "Actual":
            actual_raw_files,

        "Pass":
            actual_raw_files
            == EXPECTED_RAW_FILES,
    },
    {
        "Check":
            "Raw bytes",

        "Expected":
            EXPECTED_RAW_BYTES,

        "Actual":
            actual_raw_bytes,

        "Pass":
            actual_raw_bytes
            == EXPECTED_RAW_BYTES,
    },
    {
        "Check":
            "Raw manifest rows",

        "Expected":
            EXPECTED_RAW_FILES,

        "Actual":
            len(
                raw_file_manifest
            ),

        "Pass":
            len(
                raw_file_manifest
            )
            == EXPECTED_RAW_FILES,
    },
    {
        "Check":
            "Manifest files missing on disk",

        "Expected":
            0,

        "Actual":
            manifest_missing_disk_files,

        "Pass":
            manifest_missing_disk_files == 0,
    },
    {
        "Check":
            "Unexpected files on disk",

        "Expected":
            0,

        "Actual":
            manifest_unexpected_disk_files,

        "Pass":
            manifest_unexpected_disk_files == 0,
    },
    {
        "Check":
            "Embedded file-hash mismatches",

        "Expected":
            0,

        "Actual":
            embedded_hash_mismatches,

        "Pass":
            embedded_hash_mismatches == 0,
    },
    {
        "Check":
            "Ranking rows",

        "Expected":
            EXPECTED_RANKING_ROWS,

        "Actual":
            ranking_rows_total,

        "Pass":
            ranking_rows_total
            == EXPECTED_RANKING_ROWS,
    },
    {
        "Check":
            "Ranking rows-per-condition violations",

        "Expected":
            0,

        "Actual":
            ranking_rows_per_condition_violations,

        "Pass":
            ranking_rows_per_condition_violations == 0,
    },
    {
        "Check":
            "Project-run rows",

        "Expected":
            EXPECTED_PROJECT_RUN_ROWS,

        "Actual":
            len(
                project_run_all
            ),

        "Pass":
            len(
                project_run_all
            )
            == EXPECTED_PROJECT_RUN_ROWS,
    },
    {
        "Check":
            "Build-metric rows",

        "Expected":
            EXPECTED_BUILD_METRIC_ROWS,

        "Actual":
            len(
                build_metrics_all
            ),

        "Pass":
            len(
                build_metrics_all
            )
            == EXPECTED_BUILD_METRIC_ROWS,
    },
    {
        "Check":
            "Model-fit rows",

        "Expected":
            EXPECTED_ML_FITS,

        "Actual":
            len(
                model_fits_all
            ),

        "Pass":
            len(
                model_fits_all
            )
            == EXPECTED_ML_FITS,
    },
    {
        "Check":
            "Condition-audit rows",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                condition_audit_all
            ),

        "Pass":
            len(
                condition_audit_all
            )
            == EXPECTED_CONDITIONS,
    },
    {
        "Check":
            "Training-median rows",

        "Expected":
            expected_training_median_rows,

        "Actual":
            len(
                training_medians_all
            ),

        "Pass":
            len(
                training_medians_all
            )
            == expected_training_median_rows,
    },
    {
        "Check":
            "Active predictors per condition",

        "Expected":
            active_predictor_count,

        "Actual":
            active_predictor_count,

        "Pass":
            active_predictor_count > 0,
    },
    {
        "Check":
            "Project-run technique set",

        "Expected":
            sorted(
                ALL_TECHNIQUES
            ),

        "Actual":
            sorted(
                project_run_technique_set
            ),

        "Pass":
            project_run_technique_set
            == set(
                ALL_TECHNIQUES
            ),
    },
    {
        "Check":
            "Model-fit technique set",

        "Expected":
            sorted(
                ML_TECHNIQUES
            ),

        "Actual":
            sorted(
                model_fit_technique_set
            ),

        "Pass":
            model_fit_technique_set
            == set(
                ML_TECHNIQUES
            ),
    },
    {
        "Check":
            "Duplicate project-run rows",

        "Expected":
            0,

        "Actual":
            project_run_duplicate_rows,

        "Pass":
            project_run_duplicate_rows == 0,
    },
    {
        "Check":
            "Duplicate build-metric rows",

        "Expected":
            0,

        "Actual":
            build_metric_duplicate_rows,

        "Pass":
            build_metric_duplicate_rows == 0,
    },
    {
        "Check":
            "Duplicate model-fit rows",

        "Expected":
            0,

        "Actual":
            model_fit_duplicate_rows,

        "Pass":
            model_fit_duplicate_rows == 0,
    },
    {
        "Check":
            "Duplicate condition-audit rows",

        "Expected":
            0,

        "Actual":
            condition_audit_duplicate_rows,

        "Pass":
            condition_audit_duplicate_rows == 0,
    },
    {
        "Check":
            "Project-run rows-per-condition violations",

        "Expected":
            0,

        "Actual":
            project_runs_per_condition_violations,

        "Pass":
            project_runs_per_condition_violations == 0,
    },
    {
        "Check":
            "Build-metric rows-per-condition violations",

        "Expected":
            0,

        "Actual":
            build_metrics_per_condition_violations,

        "Pass":
            build_metrics_per_condition_violations == 0,
    },
    {
        "Check":
            "Model-fit rows-per-condition violations",

        "Expected":
            0,

        "Actual":
            model_fits_per_condition_violations,

        "Pass":
            model_fits_per_condition_violations == 0,
    },
    {
        "Check":
            "Scored-build violations",

        "Expected":
            0,

        "Actual":
            builds_per_condition_technique_violations,

        "Pass":
            builds_per_condition_technique_violations == 0,
    },
    {
        "Check":
            "Evaluated-build violations",

        "Expected":
            0,

        "Actual":
            evaluated_build_violations,

        "Pass":
            evaluated_build_violations == 0,
    },
    {
        "Check":
            "Evaluation-failure violations",

        "Expected":
            0,

        "Actual":
            evaluation_failure_violations,

        "Pass":
            evaluation_failure_violations == 0,
    },
    {
        "Check":
            "Model-fit failures",

        "Expected":
            0,

        "Actual":
            model_fit_failures,

        "Pass":
            model_fit_failures == 0,
    },
    {
        "Check":
            "Project-run non-finite metrics",

        "Expected":
            0,

        "Actual":
            project_run_metric_nonfinite,

        "Pass":
            project_run_metric_nonfinite == 0,
    },
    {
        "Check":
            "Project-run metrics outside [0,1]",

        "Expected":
            0,

        "Actual":
            project_run_metric_out_of_range,

        "Pass":
            project_run_metric_out_of_range == 0,
    },
    {
        "Check":
            "Build-level non-finite metrics",

        "Expected":
            0,

        "Actual":
            build_metric_nonfinite,

        "Pass":
            build_metric_nonfinite == 0,
    },
    {
        "Check":
            "Build-level metrics outside [0,1]",

        "Expected":
            0,

        "Actual":
            build_metric_out_of_range,

        "Pass":
            build_metric_out_of_range == 0,
    },
    {
        "Check":
            "Training-median duplicate rows",

        "Expected":
            0,

        "Actual":
            training_median_duplicate_rows,

        "Pass":
            training_median_duplicate_rows == 0,
    },
    {
        "Check":
            "Training-median non-finite values",

        "Expected":
            0,

        "Actual":
            training_median_nonfinite,

        "Pass":
            training_median_nonfinite == 0,
    },
    {
        "Check":
            "Training-median feature-set violations",

        "Expected":
            0,

        "Actual":
            feature_set_violations,

        "Pass":
            feature_set_violations == 0,
    },
    {
        "Check":
            "Zero-noise conditions",

        "Expected":
            30,

        "Actual":
            zero_noise_conditions,

        "Pass":
            zero_noise_conditions == 30,
    },
    {
        "Check":
            "Zero-noise raw-flip violations",

        "Expected":
            0,

        "Actual":
            zero_noise_raw_flip_violations,

        "Pass":
            zero_noise_raw_flip_violations == 0,
    },
    {
        "Check":
            "Zero-noise model-change violations",

        "Expected":
            0,

        "Actual":
            zero_noise_model_change_violations,

        "Pass":
            zero_noise_model_change_violations == 0,
    },
    {
        "Check":
            "Zero-noise REC-change violations",

        "Expected":
            0,

        "Actual":
            zero_noise_rec_change_violations,

        "Pass":
            zero_noise_rec_change_violations == 0,
    },
    {
        "Check":
            "Positive-noise conditions without model changes",

        "Expected":
            0,

        "Actual":
            positive_noise_without_model_changes,

        "Pass":
            positive_noise_without_model_changes == 0,
    },
    {
        "Check":
            "Positive-noise conditions without REC changes",

        "Expected":
            0,

        "Actual":
            positive_noise_without_rec_changes,

        "Pass":
            positive_noise_without_rec_changes == 0,
    },
    {
        "Check":
            "Baseline invariance failures",

        "Expected":
            0,

        "Actual":
            baseline_invariance_failures,

        "Pass":
            baseline_invariance_failures == 0,
    },
    {
        "Check":
            "Noise-technique summary rows",

        "Expected":
            63,

        "Actual":
            noise_summary_rows,

        "Pass":
            noise_summary_rows == 63,
    },
    {
        "Check":
            "Noise-delta summary rows",

        "Expected":
            63,

        "Actual":
            noise_delta_summary_rows,

        "Pass":
            noise_delta_summary_rows == 63,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_checks = validation[
    ~validation[
        "Pass"
    ]
].copy()


print("\nStep 5B validation:")

display(
    validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 9 STEP 5B DID NOT PASS."
    )


# ------------------------------------------------------------
# 17. WRITE PROJECT 9 COMPACT AGGREGATES
# ------------------------------------------------------------

STEP5B_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    RAW_FILE_MANIFEST_PATH,
    raw_file_manifest,
)

atomic_write_csv(
    CONDITION_INVENTORY_PATH,
    condition_inventory,
)

atomic_write_csv(
    PROJECT_RUN_ALL_PATH,
    project_run_all,
)

atomic_write_parquet(
    BUILD_METRICS_ALL_PATH,
    build_metrics_all,
)

atomic_write_csv(
    MODEL_FITS_ALL_PATH,
    model_fits_all,
)

atomic_write_csv(
    CONDITION_AUDIT_ALL_PATH,
    condition_audit_all,
)

atomic_write_parquet(
    TRAINING_MEDIANS_ALL_PATH,
    training_medians_all,
)

atomic_write_csv(
    NOISE_TECHNIQUE_SUMMARY_PATH,
    noise_technique_summary,
)

atomic_write_csv(
    SEED_LEVEL_DELTAS_PATH,
    seed_level_noise_deltas,
)

atomic_write_csv(
    NOISE_DELTA_SUMMARY_PATH,
    noise_delta_summary,
)

atomic_write_csv(
    CLEAN_TECHNIQUE_SUMMARY_PATH,
    clean_technique_summary,
)

atomic_write_csv(
    INVARIANCE_AUDIT_PATH,
    baseline_invariance_audit,
)

atomic_write_csv(
    STEP5B_VALIDATION_PATH,
    validation,
)


# ------------------------------------------------------------
# 18. READBACK VALIDATION
# ------------------------------------------------------------

raw_manifest_readback = pd.read_csv(
    RAW_FILE_MANIFEST_PATH,
    low_memory=False,
)

condition_inventory_readback = pd.read_csv(
    CONDITION_INVENTORY_PATH,
    low_memory=False,
)

project_run_readback = pd.read_csv(
    PROJECT_RUN_ALL_PATH,
    low_memory=False,
)

build_metrics_readback = pd.read_parquet(
    BUILD_METRICS_ALL_PATH
)

model_fits_readback = pd.read_csv(
    MODEL_FITS_ALL_PATH,
    low_memory=False,
)

condition_audit_readback = pd.read_csv(
    CONDITION_AUDIT_ALL_PATH,
    low_memory=False,
)

training_medians_readback = pd.read_parquet(
    TRAINING_MEDIANS_ALL_PATH
)

noise_summary_readback = pd.read_csv(
    NOISE_TECHNIQUE_SUMMARY_PATH,
    low_memory=False,
)

delta_summary_readback = pd.read_csv(
    NOISE_DELTA_SUMMARY_PATH,
    low_memory=False,
)


readback_checks = {
    "RawManifestRows":
        len(
            raw_manifest_readback
        )
        == EXPECTED_RAW_FILES,

    "ConditionInventoryRows":
        len(
            condition_inventory_readback
        )
        == EXPECTED_CONDITIONS,

    "ProjectRunRows":
        len(
            project_run_readback
        )
        == EXPECTED_PROJECT_RUN_ROWS,

    "BuildMetricRows":
        len(
            build_metrics_readback
        )
        == EXPECTED_BUILD_METRIC_ROWS,

    "ModelFitRows":
        len(
            model_fits_readback
        )
        == EXPECTED_ML_FITS,

    "ConditionAuditRows":
        len(
            condition_audit_readback
        )
        == EXPECTED_CONDITIONS,

    "TrainingMedianRows":
        len(
            training_medians_readback
        )
        == expected_training_median_rows,

    "NoiseSummaryRows":
        len(
            noise_summary_readback
        )
        == 63,

    "DeltaSummaryRows":
        len(
            delta_summary_readback
        )
        == 63,
}


failed_readback_checks = [
    name
    for name, passed in readback_checks.items()
    if not passed
]


if failed_readback_checks:
    raise RuntimeError(
        "Project 9 Step 5B readback failed:\n"
        + "\n".join(
            failed_readback_checks
        )
    )


# ------------------------------------------------------------
# 19. REGISTRY IMMUTABILITY
# ------------------------------------------------------------

registry_sha256_after = calculate_hash(
    REGISTRY_PATH
)

registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


if not registry_unchanged:
    raise AssertionError(
        "Completion registry changed during Project 9 Step 5B."
    )


registry_after = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_after_project_numbers = pd.to_numeric(
    registry_after[
        project_number_column
    ],
    errors="raise",
).astype(int)

registry_project9_rows_after = int(
    registry_after_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)


if registry_project9_rows_after != 0:
    raise AssertionError(
        "Project 9 was unexpectedly registered during Step 5B."
    )


# ------------------------------------------------------------
# 20. OUTPUT HASHES
# ------------------------------------------------------------

output_paths = {
    "RawFileManifest":
        RAW_FILE_MANIFEST_PATH,

    "ConditionInventory":
        CONDITION_INVENTORY_PATH,

    "ProjectRunAll":
        PROJECT_RUN_ALL_PATH,

    "BuildMetricsAll":
        BUILD_METRICS_ALL_PATH,

    "ModelFitsAll":
        MODEL_FITS_ALL_PATH,

    "ConditionAuditAll":
        CONDITION_AUDIT_ALL_PATH,

    "TrainingMediansAll":
        TRAINING_MEDIANS_ALL_PATH,

    "NoiseTechniqueSummary":
        NOISE_TECHNIQUE_SUMMARY_PATH,

    "SeedLevelNoiseDeltas":
        SEED_LEVEL_DELTAS_PATH,

    "NoiseDeltaSummary":
        NOISE_DELTA_SUMMARY_PATH,

    "CleanTechniqueSummary":
        CLEAN_TECHNIQUE_SUMMARY_PATH,

    "BaselineInvarianceAudit":
        INVARIANCE_AUDIT_PATH,

    "Step5BValidation":
        STEP5B_VALIDATION_PATH,
}


output_hashes = {
    name:
        calculate_hash(
            path
        )
    for name, path in output_paths.items()
}


output_sizes = {
    name:
        int(
            path.stat().st_size
        )
    for name, path in output_paths.items()
}


# ------------------------------------------------------------
# 21. REPORT, CHECKPOINT AND STATUS
# ------------------------------------------------------------

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5B_PASS_STATUS,

    "Step5AStatus":
        full_run_status,

    "Step5ACheckpoint":
        str(
            FULL_RUN_CHECKPOINT_PATH
        ),

    "Step5ACheckpointSHA256":
        full_run_checkpoint_sha256,

    "FrozenStep5ARawRootSHA256":
        EXPECTED_FROZEN_RAW_ROOT_SHA256,

    "IndependentCurrentRawRootSHA256":
        independent_raw_root_sha256,

    "RawManifestSemanticSHA256":
        raw_manifest_sha256,

    "RawFiles":
        actual_raw_files,

    "RawBytes":
        actual_raw_bytes,

    "Conditions":
        len(
            condition_inventory
        ),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "RankingRows":
        ranking_rows_total,

    "BuildMetricRows":
        len(
            build_metrics_all
        ),

    "ProjectRunRows":
        len(
            project_run_all
        ),

    "ModelFits":
        len(
            model_fits_all
        ),

    "ConditionAuditRows":
        len(
            condition_audit_all
        ),

    "ActivePredictors":
        active_predictor_count,

    "TrainingMedianRows":
        len(
            training_medians_all
        ),

    "EmbeddedHashChecks":
        embedded_hash_checks,

    "EmbeddedHashMismatches":
        embedded_hash_mismatches,

    "NoiseTechniqueSummaryRows":
        len(
            noise_technique_summary
        ),

    "NoiseDeltaSummaryRows":
        len(
            noise_delta_summary
        ),

    "BaselineInvarianceFailures":
        baseline_invariance_failures,

    "RawHashSeconds":
        raw_hash_seconds,

    "AggregationSeconds":
        aggregation_seconds,

    "CompletionRegistryModified":
        False,

    "RegistryProject9Rows":
        registry_project9_rows_after,

    "Project10Accessed":
        False,

    "Project10WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "OutputPaths": {
        name:
            str(
                path
            )
        for name, path in output_paths.items()
    },

    "OutputSHA256":
        output_hashes,

    "OutputSizeBytes":
        output_sizes,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP5B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "Step5BReport":
        str(
            STEP5B_REPORT_PATH
        ),

    "Step5BReportSHA256":
        calculate_hash(
            STEP5B_REPORT_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CheckpointFrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP5B_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5B_PASS_STATUS,

    "Conditions":
        len(
            condition_inventory
        ),

    "MLFits":
        len(
            model_fits_all
        ),

    "RankingRows":
        ranking_rows_total,

    "BuildMetricRows":
        len(
            build_metrics_all
        ),

    "ProjectRunRows":
        len(
            project_run_all
        ),

    "RawFiles":
        actual_raw_files,

    "RawBytes":
        actual_raw_bytes,

    "FrozenRawRootSHA256":
        EXPECTED_FROZEN_RAW_ROOT_SHA256,

    "IndependentCurrentRawRootSHA256":
        independent_raw_root_sha256,

    "Checkpoint":
        str(
            STEP5B_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        calculate_hash(
            STEP5B_CHECKPOINT_PATH
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Project10Accessed":
        False,

    "Project10WriteAttempted":
        False,

    "Projects1To8Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP5B_STATUS_PATH,
    status_payload,
)


# ------------------------------------------------------------
# 22. FINAL READBACK
# ------------------------------------------------------------

checkpoint_readback = read_json(
    STEP5B_CHECKPOINT_PATH
)

status_readback = read_json(
    STEP5B_STATUS_PATH
)


if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP5B_PASS_STATUS
):
    raise AssertionError(
        "Project 9 Step 5B checkpoint readback failed."
    )


if (
    status_readback.get(
        "Status"
    )
    != STEP5B_PASS_STATUS
):
    raise AssertionError(
        "Project 9 Step 5B status readback failed."
    )


if calculate_hash(
    REGISTRY_PATH
) != registry_sha256_before:
    raise AssertionError(
        "Completion registry changed during final readback."
    )


# ------------------------------------------------------------
# 23. DISPLAY
# ------------------------------------------------------------

print("\nRaw condition inventory sample:")

display(
    pd.concat(
        [
            condition_inventory.head(9),
            condition_inventory.tail(9),
        ],
        ignore_index=True,
    )
)


print("\nNoise × technique summary:")

display(
    noise_technique_summary
)


print("\nClean 0% technique ranking by mean APFDc:")

display(
    clean_technique_summary[
        [
            "Technique",
            "Runs",
            "MeanAPFD",
            "StdAPFD",
            "MeanAPFDc",
            "StdAPFDc",
        ]
    ]
)


print("\nBaseline invariance audit:")

display(
    baseline_invariance_audit
)


print("\nNoise-delta summary sample:")

display(
    noise_delta_summary[
        noise_delta_summary[
            "NoisePercent"
        ].isin(
            [
                0,
                25,
                50,
            ]
        )
    ]
)


# ------------------------------------------------------------
# 24. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 126)
print("=== PROJECT 9 CELL 10 / STEP 5B RESULT ===")
print("=" * 126)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nStep 5A frozen input:")

print(
    "Step 5A checkpoint SHA-256:",
    full_run_checkpoint_sha256,
)

print(
    "Frozen raw-root SHA-256:",
    EXPECTED_FROZEN_RAW_ROOT_SHA256,
)

print(
    "Independent current raw-root SHA-256:",
    independent_raw_root_sha256,
)


print("\nRaw-output revalidation:")

print(
    "Conditions:",
    len(
        condition_inventory
    ),
    "/",
    EXPECTED_CONDITIONS,
)

print(
    "Raw files:",
    actual_raw_files,
    "/",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    actual_raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)

print(
    "Embedded hash checks:",
    embedded_hash_checks,
)

print(
    "Embedded hash mismatches:",
    embedded_hash_mismatches,
)


print("\nExperiment totals:")

print(
    "ML fits:",
    len(
        model_fits_all
    ),
    "/",
    EXPECTED_ML_FITS,
)

print(
    "Ranking rows:",
    ranking_rows_total,
    "/",
    EXPECTED_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(
        build_metrics_all
    ),
    "/",
    EXPECTED_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    len(
        project_run_all
    ),
    "/",
    EXPECTED_PROJECT_RUN_ROWS,
)

print(
    "Condition-audit rows:",
    len(
        condition_audit_all
    ),
)

print(
    "Active predictors:",
    active_predictor_count,
)

print(
    "Training-median rows:",
    len(
        training_medians_all
    ),
)


print("\nAnalysis-ready aggregates:")

print(
    "Noise-technique summary rows:",
    len(
        noise_technique_summary
    ),
)

print(
    "Seed-level noise-delta rows:",
    len(
        seed_level_noise_deltas
    ),
)

print(
    "Noise-delta summary rows:",
    len(
        noise_delta_summary
    ),
)

print(
    "Baseline invariance failures:",
    baseline_invariance_failures,
)


print("\nImmutability and isolation:")

print(
    "Completion registry unchanged:",
    registry_unchanged,
)

print(
    "Registry Project 9 rows:",
    registry_project9_rows_after,
)

print(
    "Project 10 accessed:",
    False,
)

print(
    "Project 10 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nRuntime:")

print(
    "Raw hashing seconds:",
    round(
        raw_hash_seconds,
        2,
    ),
)

print(
    "Compact aggregation seconds:",
    round(
        aggregation_seconds,
        2,
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nProject 9 Step 5B checkpoint:")

print(
    STEP5B_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    calculate_hash(
        STEP5B_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP5B_PASS_STATUS,
)

print("=" * 126)

=== PROJECT 9 CELL 10 / STEP 5B: FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION ===

Step 5B validation:


,Check,Expected,Actual,Pass
0,Step 5A passed,PASS_PROJECT_9_FULL_270_CONDITION_EXPERIMENT_C...,PASS_PROJECT_9_FULL_270_CONDITION_EXPERIMENT_C...,True
1,Step 5A checkpoint SHA-256,22f9f1184be247d83941199381f12b7042f743158403e7...,22f9f1184be247d83941199381f12b7042f743158403e7...,True
2,Frozen Step 5A raw-root SHA-256,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,True
3,Condition success markers,270,270,True
4,Condition inventory rows,270,270,True
5,Condition coordinates,270,270,True
6,Duplicate condition keys,0,0,True
7,Duplicate condition coordinates,0,0,True
8,Condition-order violations,0,0,True
9,Files-per-condition violations,0,0,True



Raw condition inventory sample:


,ProjectNumber,Project,ProjectSlug,ConditionOrder,ConditionKey,NoisePercent,RepetitionSeed,ConditionDirectory,MarkerStatus,SummaryStatus,Files,ConditionBytes,RankingRows,MarkerSHA256,SummarySHA256
0,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,1,noise_00__seed_01,0,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1397106,129521,744f06e8fd40685035f047416360dc86613aa26eefc9a8...,5fe730de7a11ba77e7854bb039c0e9eef6fb12ec705c59...
1,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,2,noise_05__seed_01,5,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1462185,129521,f920402726bc590393fd5b49969e8ed02e6e2483660725...,0320f521763acec9fcad86d20c904351a0dac3e7805793...
2,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,3,noise_10__seed_01,10,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1451403,129521,fc0189958c9ed7e21ec89177f8efca8efbe30ace991023...,eaf4fd7ea02384535c0bbefc53336cc4e77b3bc0f23ebe...
3,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,4,noise_15__seed_01,15,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1450658,129521,86fd199d2aaa517dbc66099d1f14bf9531c00897404fd4...,db6a7cd76a2d81a00dc565c19bf0a9be9bf7b241d720f2...
4,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,5,noise_20__seed_01,20,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1422669,129521,88e04c6fb9a4fbd556dcf795636010da3d1d986cb94498...,4781ecdded13c9f2d27040e2458c704eba42fd896ab727...
5,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,6,noise_25__seed_01,25,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1441081,129521,06b5cd1079e92e1778153c102bbd378a5f069f1efb6ede...,62c064afd31f391b5192e6806edf09cc3933ddb411d846...
6,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,7,noise_30__seed_01,30,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1447692,129521,184b8021b9dcd51d3938ec2a47458a93c32e231d746fe5...,5adf2b224710aeb1178b12520b983adef3d10052d5277e...
7,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,8,noise_40__seed_01,40,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1446368,129521,e1ff57c9e277892ebb8c01b7bb1253f9a1c96bfc200d3d...,6c724cc638aed7db7d85def2a709f3851bada0d95cf163...
8,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,9,noise_50__seed_01,50,1,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1443735,129521,0a7c72847f50e9b049c90b8ca85aaa60d6993d59b9ade4...,13649c88b2320ffe933f108895cc57e619ece7067cd1e9...
9,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,262,noise_00__seed_30,0,30,/content/drive/MyDrive/Thesis_Experiment/Resul...,PASS_FULL_CONDITION,PASS_FULL_CONDITION,8,1401749,129521,973b188629eb9d2248d5c935e81ca2743de85980e3b585...,190b5d8087501812c061fed83113f71265d73845cad367...



Noise × technique summary:


...


Baseline invariance audit:


,Technique,Expected,SeedsChecked,ViolatingSeeds,Pass
0,Random,Metrics constant across noise within each repe...,30,0,True
1,QTF-Avg,Metrics constant across noise within each repe...,30,0,True
2,LatestFail,At least one repetition seed changes across no...,30,0,True



Noise-delta summary sample:


,NoisePercent,Technique,Seeds,MeanAPFDDeltaFromClean,MedianAPFDDeltaFromClean,MeanAPFDcDeltaFromClean,MedianAPFDcDeltaFromClean,MeanAPFDRelativeDegradation,MedianAPFDRelativeDegradation,MeanAPFDcRelativeDegradation,MedianAPFDcRelativeDegradation
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0,RandomForest,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0,XGBoost,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
35,25,LatestFail,30,0.001134,0.006115,0.031711,0.036541,-0.001426,-0.007693,-0.042128,-0.048544
36,25,LightGBM,30,-0.295172,-0.286145,-0.297236,-0.299003,0.322085,0.312235,0.352800,0.354897
37,25,NaiveBayes,30,-0.099850,-0.104690,-0.170372,-0.123701,0.157540,0.165177,0.231111,0.167802




=== PROJECT 9 CELL 10 / STEP 5B RESULT ===

Project identity:
Project number: 9
Project: camunda@camunda-bpm-platform
Project slug: camunda__camunda-bpm-platform

Step 5A frozen input:
Step 5A checkpoint SHA-256: 22f9f1184be247d83941199381f12b7042f743158403e765ee8ffcfe09636519
Frozen raw-root SHA-256: c31cb45e1354dc1222b82103bd275c21b10dd2ba6171125005d9c0a99ab14724
Independent current raw-root SHA-256: c31cb45e1354dc1222b82103bd275c21b10dd2ba6171125005d9c0a99ab14724

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 387066081 / 387066081
Embedded hash checks: 0
Embedded hash mismatches: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 34970670 / 34970670
Build-metric rows: 56700 / 56700
Project-run rows: 1890 / 1890
Condition-audit rows: 270
Active predictors: 151
Training-median rows: 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level noise-delta rows: 1890
Noise-delta summary rows: 63
Baseline invariance failures: 0


In [16]:
# ==================================================================================================
# PROJECT 9 CELL 11 / STEP 5C
# FINAL COMPACT PACKAGE CONSTRUCTION, HASH VALIDATION, AND FREEZE
#
# IMPORTANT:
# - PROJECT 9 ONLY: camunda@camunda-bpm-platform
# - DOES NOT rerun any experimental condition
# - DOES NOT access or modify Project 10
# - DOES NOT modify the completion registry
# - DOES NOT modify/delete the Project 9 raw-result directory
# - Registry insertion remains pending for a later serial finalisation cell
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import shutil
import pandas as pd


print("=" * 118)
print("=== PROJECT 9 CELL 11 / STEP 5C: FINAL PACKAGE CONSTRUCTION AND VALIDATION ===")
print("=" * 118)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT IDENTITY AND PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT /
    "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT /
    "Results"
)

RAW_RESULTS_ROOT = (
    RESULTS_ROOT /
    "Raw"
)

AGGREGATED_RESULTS_ROOT = (
    RESULTS_ROOT /
    "Aggregated"
)

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)

PROJECT_RAW_ROOT = (
    RAW_RESULTS_ROOT /
    PROJECT_SLUG
)

PROJECT_AGGREGATED_ROOT = (
    AGGREGATED_RESULTS_ROOT /
    PROJECT_SLUG
)

PROJECT_SELECTION_ROOT = (
    AGGREGATED_RESULTS_ROOT /
    "project_09_selection"
)

STEP5B_FINAL_AUDIT_ROOT = (
    PROJECT_AGGREGATED_ROOT /
    "camunda_step5b_final_audit"
)

STEP5B_CHECKPOINT_PATH = (
    NOTES_ROOT /
    "project_09_step5b_checkpoint.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT /
    "camunda_step5b_status.json"
)

COMPLETION_REGISTRY_PATH = (
    NOTES_ROOT /
    "completed_project_registry.csv"
)

FINAL_PACKAGE_ROOT = (
    PROJECT_AGGREGATED_ROOT /
    "camunda_final_package"
)

STAGING_PACKAGE_ROOT = (
    PROJECT_AGGREGATED_ROOT /
    f".camunda_final_package_staging_{os.getpid()}"
)

FINAL_PACKAGE_CHECKPOINT_PATH = (
    NOTES_ROOT /
    "project_09_final_package_checkpoint.json"
)

STEP5C_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT /
    "camunda_step5c_status.json"
)


EXPECTED_STEP5B_STATUS = (
    "PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_"
    "AND_COMPACT_AGGREGATES_FROZEN"
)

EXPECTED_STEP5B_CHECKPOINT_SHA256 = (
    "a1fff6906ac62870689cbf4f3f38f16c"
    "dac04b10e6851190bcb34fbb8d6f8a14"
)

EXPECTED_STEP5A_CHECKPOINT_SHA256 = (
    "22f9f1184be247d83941199381f12b704"
    "2f743158403e765ee8ffcfe09636519"
)

EXPECTED_RAW_ROOT_SHA256 = (
    "c31cb45e1354dc1222b82103bd275c21"
    "b10dd2ba6171125005d9c0a99ab14724"
)

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 387066081
EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_RANKING_ROWS = 34970670
EXPECTED_BUILD_METRIC_ROWS = 56700
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_CONDITION_AUDIT_ROWS = 270
EXPECTED_TRAINING_MEDIAN_ROWS = 40770
EXPECTED_ACTIVE_PREDICTORS = 151
EXPECTED_NOISE_TECHNIQUE_ROWS = 63
EXPECTED_SEED_DELTA_ROWS = 1890
EXPECTED_NOISE_DELTA_ROWS = 63

FINAL_STATUS = (
    "PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_"
    "AND_VALIDATED_REGISTRY_PENDING"
)


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def load_json(path):
    path = Path(path)

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def canonical_manifest_root_hash(manifest_frame):
    digest = hashlib.sha256()

    ordered = manifest_frame.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{row.SHA256}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(frame, candidates, label):
    lower_lookup = {
        str(column).lower(): column
        for column in frame.columns
    }

    for candidate in candidates:
        if candidate in frame.columns:
            return candidate

        lowered = candidate.lower()

        if lowered in lower_lookup:
            return lower_lookup[lowered]

    raise RuntimeError(
        f"Could not resolve {label}. "
        f"Available columns: {list(frame.columns)}"
    )


def extract_status(payload):
    for key in [
        "Status",
        "status",
        "Step5BStatus",
        "FinalStatus",
    ]:
        if key in payload:
            return str(payload[key])

    return None


def add_validation(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def is_relative_to(path, parent):
    try:
        Path(path).resolve().relative_to(
            Path(parent).resolve()
        )
        return True
    except ValueError:
        return False


# --------------------------------------------------------------------------------------------------
# 3. REQUIRED INPUT EXISTENCE
# --------------------------------------------------------------------------------------------------

required_roots = [
    NOTES_ROOT,
    PROJECT_RAW_ROOT,
    PROJECT_AGGREGATED_ROOT,
    PROJECT_SELECTION_ROOT,
    STEP5B_FINAL_AUDIT_ROOT,
]

for required_root in required_roots:
    if not required_root.exists():
        raise FileNotFoundError(
            f"Required Project 9 directory is missing:\n"
            f"{required_root}"
        )

required_files = [
    STEP5B_CHECKPOINT_PATH,
    STEP5B_STATUS_PATH,
    COMPLETION_REGISTRY_PATH,
]

for required_file in required_files:
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required Project 9 file is missing:\n"
            f"{required_file}"
        )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STEP 5B CHECKPOINT AND STATUS
# --------------------------------------------------------------------------------------------------

step5b_checkpoint_sha256 = sha256_file(
    STEP5B_CHECKPOINT_PATH
)

step5b_checkpoint = load_json(
    STEP5B_CHECKPOINT_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)

step5b_checkpoint_status = extract_status(
    step5b_checkpoint
)

step5b_status_file_status = extract_status(
    step5b_status_payload
)

if (
    step5b_checkpoint_sha256
    != EXPECTED_STEP5B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 9 Step 5B checkpoint SHA-256 changed.\n"
        f"Expected: {EXPECTED_STEP5B_CHECKPOINT_SHA256}\n"
        f"Actual:   {step5b_checkpoint_sha256}"
    )

if (
    step5b_checkpoint_status
    != EXPECTED_STEP5B_STATUS
):
    raise RuntimeError(
        "Project 9 Step 5B checkpoint does not contain "
        "the expected PASS status.\n"
        f"Expected: {EXPECTED_STEP5B_STATUS}\n"
        f"Actual:   {step5b_checkpoint_status}"
    )

if (
    step5b_status_file_status
    != EXPECTED_STEP5B_STATUS
):
    raise RuntimeError(
        "Project 9 Step 5B status file does not contain "
        "the expected PASS status.\n"
        f"Expected: {EXPECTED_STEP5B_STATUS}\n"
        f"Actual:   {step5b_status_file_status}"
    )


# --------------------------------------------------------------------------------------------------
# 5. LOCATE AND VALIDATE THE FROZEN RAW MANIFEST
# --------------------------------------------------------------------------------------------------

RAW_MANIFEST_PATH = (
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_raw_file_manifest.csv"
)

if not RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The frozen Project 9 raw manifest is missing:\n"
        f"{RAW_MANIFEST_PATH}"
    )

raw_manifest = pd.read_csv(
    RAW_MANIFEST_PATH
)

raw_relative_column = resolve_column(
    raw_manifest,
    [
        "RelativePath",
        "relative_path",
        "Path",
    ],
    "raw-manifest relative-path column",
)

raw_size_column = resolve_column(
    raw_manifest,
    [
        "FileSizeBytes",
        "SizeBytes",
        "size_bytes",
    ],
    "raw-manifest size column",
)

raw_sha_column = resolve_column(
    raw_manifest,
    [
        "SHA256",
        "FileSHA256",
        "sha256",
    ],
    "raw-manifest SHA-256 column",
)

raw_manifest_normalised = pd.DataFrame({
    "RelativePath": (
        raw_manifest[raw_relative_column]
        .astype(str)
        .str.replace("\\", "/", regex=False)
    ),
    "SizeBytes": pd.to_numeric(
        raw_manifest[raw_size_column],
        errors="raise",
    ).astype("int64"),
    "ExpectedSHA256": (
        raw_manifest[raw_sha_column]
        .astype(str)
        .str.lower()
    ),
})

raw_manifest_normalised = (
    raw_manifest_normalised
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if raw_manifest_normalised["RelativePath"].duplicated().any():
    raise RuntimeError(
        "The frozen raw manifest contains duplicate paths."
    )


actual_raw_relative_paths = sorted(
    path.relative_to(
        PROJECT_RAW_ROOT
    ).as_posix()
    for path in PROJECT_RAW_ROOT.rglob("*")
    if path.is_file()
)

manifest_raw_relative_paths = (
    raw_manifest_normalised[
        "RelativePath"
    ].tolist()
)

missing_raw_files = sorted(
    set(manifest_raw_relative_paths)
    - set(actual_raw_relative_paths)
)

unexpected_raw_files = sorted(
    set(actual_raw_relative_paths)
    - set(manifest_raw_relative_paths)
)

raw_hash_mismatches = []
raw_size_mismatches = []
current_raw_records = []

for row in raw_manifest_normalised.itertuples(
    index=False
):
    raw_file = (
        PROJECT_RAW_ROOT /
        row.RelativePath
    )

    if not raw_file.is_file():
        continue

    actual_size = int(
        raw_file.stat().st_size
    )

    actual_sha256 = sha256_file(
        raw_file
    )

    if actual_size != int(row.SizeBytes):
        raw_size_mismatches.append(
            row.RelativePath
        )

    if actual_sha256 != row.ExpectedSHA256:
        raw_hash_mismatches.append(
            row.RelativePath
        )

    current_raw_records.append({
        "RelativePath": row.RelativePath,
        "SizeBytes": actual_size,
        "SHA256": actual_sha256,
    })


current_raw_manifest = pd.DataFrame(
    current_raw_records
)

current_raw_root_sha256 = (
    canonical_manifest_root_hash(
        current_raw_manifest
    )
)

current_raw_file_count = int(
    len(actual_raw_relative_paths)
)

current_raw_bytes = int(
    sum(
        (
            PROJECT_RAW_ROOT /
            relative_path
        ).stat().st_size
        for relative_path
        in actual_raw_relative_paths
    )
)

if missing_raw_files:
    raise RuntimeError(
        f"Frozen Project 9 raw files are missing: "
        f"{len(missing_raw_files)}"
    )

if unexpected_raw_files:
    raise RuntimeError(
        f"Unexpected files were found in the frozen "
        f"Project 9 raw root: {len(unexpected_raw_files)}"
    )

if raw_size_mismatches:
    raise RuntimeError(
        f"Project 9 raw size mismatches: "
        f"{len(raw_size_mismatches)}"
    )

if raw_hash_mismatches:
    raise RuntimeError(
        f"Project 9 raw SHA-256 mismatches: "
        f"{len(raw_hash_mismatches)}"
    )

if current_raw_file_count != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Unexpected Project 9 raw file count."
    )

if current_raw_bytes != EXPECTED_RAW_BYTES:
    raise RuntimeError(
        "Unexpected Project 9 raw byte count."
    )

if current_raw_root_sha256 != EXPECTED_RAW_ROOT_SHA256:
    raise RuntimeError(
        "Project 9 raw-root SHA-256 changed.\n"
        f"Expected: {EXPECTED_RAW_ROOT_SHA256}\n"
        f"Actual:   {current_raw_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 6. READ-ONLY COMPLETION REGISTRY SNAPSHOT
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    COMPLETION_REGISTRY_PATH
)

registry_before = pd.read_csv(
    COMPLETION_REGISTRY_PATH
)

project9_registry_mask = pd.Series(
    False,
    index=registry_before.index,
)

registry_columns_lower = {
    str(column).lower(): column
    for column in registry_before.columns
}

if "projectnumber" in registry_columns_lower:
    project_number_column = (
        registry_columns_lower[
            "projectnumber"
        ]
    )

    project9_registry_mask |= (
        pd.to_numeric(
            registry_before[
                project_number_column
            ],
            errors="coerce",
        )
        == PROJECT_NUMBER
    )

if "projectslug" in registry_columns_lower:
    project_slug_column = (
        registry_columns_lower[
            "projectslug"
        ]
    )

    project9_registry_mask |= (
        registry_before[
            project_slug_column
        ].astype(str)
        == PROJECT_SLUG
    )

if "project" in registry_columns_lower:
    project_column = (
        registry_columns_lower[
            "project"
        ]
    )

    project9_registry_mask |= (
        registry_before[
            project_column
        ].astype(str)
        == PROJECT_NAME
    )

project9_registry_rows_before = int(
    project9_registry_mask.sum()
)

if project9_registry_rows_before != 0:
    raise RuntimeError(
        "Project 9 is already present in the completion "
        "registry. This package-only cell must run before "
        "serial registry insertion."
    )


# --------------------------------------------------------------------------------------------------
# 7. VALIDATE REQUIRED STEP 5B ANALYSIS OUTPUTS
# --------------------------------------------------------------------------------------------------

required_final_audit_names = {
    "camunda_raw_file_manifest.csv",
    "camunda_condition_inventory.csv",
    "camunda_project_run_all.csv",
    "camunda_build_metrics_all.parquet",
    "camunda_model_fits_all.csv",
    "camunda_condition_audit_all.csv",
    "camunda_training_medians_all.parquet",
    "camunda_noise_technique_summary.csv",
    "camunda_seed_level_noise_deltas.csv",
    "camunda_noise_delta_summary.csv",
    "camunda_clean_technique_summary.csv",
    "camunda_baseline_invariance_audit.csv",
    "camunda_step5b_validation.csv",
    "camunda_step5b_report.json",
}

actual_final_audit_names = {
    path.name
    for path in STEP5B_FINAL_AUDIT_ROOT.rglob("*")
    if path.is_file()
}

missing_required_final_audit = sorted(
    required_final_audit_names
    - actual_final_audit_names
)

if missing_required_final_audit:
    raise FileNotFoundError(
        "Required Step 5B final-audit outputs are missing:\n"
        + "\n".join(
            missing_required_final_audit
        )
    )


# Validate compact aggregate row counts.

project_run_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_project_run_all.csv"
)

model_fits_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_model_fits_all.csv"
)

condition_audit_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_condition_audit_all.csv"
)

noise_technique_summary = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_noise_technique_summary.csv"
)

seed_level_noise_deltas = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_seed_level_noise_deltas.csv"
)

noise_delta_summary = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_noise_delta_summary.csv"
)

build_metrics_all = pd.read_parquet(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_build_metrics_all.parquet"
)

training_medians_all = pd.read_parquet(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_training_medians_all.parquet"
)

condition_inventory = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_condition_inventory.csv"
)

baseline_invariance = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT /
    "camunda_baseline_invariance_audit.csv"
)

if len(condition_inventory) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "Condition-inventory row count changed."
    )

if len(project_run_all) != EXPECTED_PROJECT_RUN_ROWS:
    raise RuntimeError(
        "Project-run row count changed."
    )

if len(build_metrics_all) != EXPECTED_BUILD_METRIC_ROWS:
    raise RuntimeError(
        "Build-metric row count changed."
    )

if len(model_fits_all) != EXPECTED_ML_FITS:
    raise RuntimeError(
        "Model-fit row count changed."
    )

if len(condition_audit_all) != EXPECTED_CONDITION_AUDIT_ROWS:
    raise RuntimeError(
        "Condition-audit row count changed."
    )

if len(training_medians_all) != EXPECTED_TRAINING_MEDIAN_ROWS:
    raise RuntimeError(
        "Training-median row count changed."
    )

if len(noise_technique_summary) != EXPECTED_NOISE_TECHNIQUE_ROWS:
    raise RuntimeError(
        "Noise-technique summary row count changed."
    )

if len(seed_level_noise_deltas) != EXPECTED_SEED_DELTA_ROWS:
    raise RuntimeError(
        "Seed-level delta row count changed."
    )

if len(noise_delta_summary) != EXPECTED_NOISE_DELTA_ROWS:
    raise RuntimeError(
        "Noise-delta summary row count changed."
    )

baseline_pass_column = resolve_column(
    baseline_invariance,
    ["Pass", "pass"],
    "baseline-invariance pass column",
)

baseline_invariance_failures = int(
    (
        ~baseline_invariance[
            baseline_pass_column
        ].astype(bool)
    ).sum()
)

if baseline_invariance_failures != 0:
    raise RuntimeError(
        "Baseline invariance audit no longer passes."
    )


# --------------------------------------------------------------------------------------------------
# 8. BUILD THE CURATED PACKAGE SOURCE INVENTORY
# --------------------------------------------------------------------------------------------------

allowed_suffixes = {
    ".csv",
    ".json",
    ".parquet",
    ".txt",
    ".md",
    ".gz",
}

source_inventory = []


def register_source(
    category,
    source_path,
    destination_tail,
):
    source_path = Path(source_path)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Package source is missing:\n{source_path}"
        )

    source_inventory.append({
        "Category": category,
        "SourcePath": source_path,
        "DestinationTail": Path(
            destination_tail
        ),
    })


# All Project 9 frozen checkpoints created before Step 5C.

checkpoint_sources = sorted(
    path
    for path in NOTES_ROOT.glob(
        "project_09_*checkpoint.json"
    )
    if (
        path.name
        != FINAL_PACKAGE_CHECKPOINT_PATH.name
    )
)

essential_checkpoint_names = {
    "project_09_selection_checkpoint.json",
    "project_09_rec_reconstruction_checkpoint.json",
    "project_09_noise_plan_checkpoint.json",
    "project_09_noisy_rec_engine_checkpoint.json",
    "project_09_model_protocol_checkpoint.json",
    "project_09_smoke_test_checkpoint.json",
    "project_09_full_run_checkpoint.json",
    "project_09_step5b_checkpoint.json",
}

actual_checkpoint_names = {
    path.name
    for path in checkpoint_sources
}

missing_essential_checkpoints = sorted(
    essential_checkpoint_names
    - actual_checkpoint_names
)

if missing_essential_checkpoints:
    raise FileNotFoundError(
        "Required Project 9 checkpoints are missing:\n"
        + "\n".join(
            missing_essential_checkpoints
        )
    )

for checkpoint_path in checkpoint_sources:
    register_source(
        "checkpoints",
        checkpoint_path,
        checkpoint_path.name,
    )


# Project 9 selection evidence.

for source_path in sorted(
    path
    for path in PROJECT_SELECTION_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in allowed_suffixes
    )
):
    register_source(
        "selection",
        source_path,
        source_path.relative_to(
            PROJECT_SELECTION_ROOT
        ),
    )


# Complete Step 5B final-audit/aggregate directory.

for source_path in sorted(
    path
    for path in STEP5B_FINAL_AUDIT_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in allowed_suffixes
    )
):
    register_source(
        "final_audit",
        source_path,
        source_path.relative_to(
            STEP5B_FINAL_AUDIT_ROOT
        ),
    )


# Step status files from Project 9 only.
# Exclude Step 5C itself because that file is produced by this cell.

status_sources = sorted(
    path
    for path in PROJECT_AGGREGATED_ROOT.rglob(
        "camunda_step*_status.json"
    )
    if (
        path.is_file()
        and path.name
        != STEP5C_STATUS_PATH.name
        and not is_relative_to(
            path,
            FINAL_PACKAGE_ROOT,
        )
        and not is_relative_to(
            path,
            STAGING_PACKAGE_ROOT,
        )
    )
)

for status_path in status_sources:
    register_source(
        "statuses",
        status_path,
        status_path.relative_to(
            PROJECT_AGGREGATED_ROOT
        ),
    )


# Check destination uniqueness.

destination_relative_paths = []

for item in source_inventory:
    destination_relative_paths.append(
        (
            Path("payload") /
            item["Category"] /
            item["DestinationTail"]
        ).as_posix()
    )

duplicate_destination_count = int(
    pd.Series(
        destination_relative_paths
    ).duplicated().sum()
)

if duplicate_destination_count != 0:
    raise RuntimeError(
        "The package source inventory creates duplicate "
        "destination paths."
    )


# --------------------------------------------------------------------------------------------------
# 9. CREATE A CLEAN STAGING PACKAGE
# --------------------------------------------------------------------------------------------------

if STAGING_PACKAGE_ROOT.exists():
    if (
        STAGING_PACKAGE_ROOT.parent.resolve()
        != PROJECT_AGGREGATED_ROOT.resolve()
        or not STAGING_PACKAGE_ROOT.name.startswith(
            ".camunda_final_package_staging_"
        )
    ):
        raise RuntimeError(
            "Unsafe staging-directory removal blocked."
        )

    shutil.rmtree(
        STAGING_PACKAGE_ROOT
    )

STAGING_PACKAGE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

package_records = []
copy_size_mismatches = []
copy_hash_mismatches = []

for item in source_inventory:
    source_path = item["SourcePath"]

    relative_destination = (
        Path("payload") /
        item["Category"] /
        item["DestinationTail"]
    )

    destination_path = (
        STAGING_PACKAGE_ROOT /
        relative_destination
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    source_size = int(
        source_path.stat().st_size
    )

    source_sha256 = sha256_file(
        source_path
    )

    shutil.copy2(
        source_path,
        destination_path,
    )

    destination_size = int(
        destination_path.stat().st_size
    )

    destination_sha256 = sha256_file(
        destination_path
    )

    if destination_size != source_size:
        copy_size_mismatches.append(
            relative_destination.as_posix()
        )

    if destination_sha256 != source_sha256:
        copy_hash_mismatches.append(
            relative_destination.as_posix()
        )

    package_records.append({
        "Category": item["Category"],
        "RelativePath": (
            relative_destination.as_posix()
        ),
        "SourcePath": str(source_path),
        "SizeBytes": source_size,
        "SHA256": source_sha256,
        "Generated": False,
    })


# --------------------------------------------------------------------------------------------------
# 10. ADD DETERMINISTIC PACKAGE METADATA TO THE PAYLOAD
# --------------------------------------------------------------------------------------------------

metadata_root = (
    STAGING_PACKAGE_ROOT /
    "payload" /
    "metadata"
)

metadata_root.mkdir(
    parents=True,
    exist_ok=True,
)

readme_text = f"""PROJECT 9 FINAL EXPERIMENT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}

Experiment status:
- All 270 project × noise × seed conditions completed.
- All 1,080 ML model fits completed.
- All 7 techniques evaluated.
- APFD and APFDc calculated.
- Mean, median, minimum, maximum and standard deviation aggregates calculated.
- Raw outputs independently revalidated.
- Compact analysis-ready aggregates frozen.

Frozen raw result:
- Files: {EXPECTED_RAW_FILES}
- Bytes: {EXPECTED_RAW_BYTES}
- Root SHA-256: {EXPECTED_RAW_ROOT_SHA256}

This package does not contain the 387 MB raw condition directory.
It contains the frozen raw-file manifest, compact aggregate outputs,
validation evidence, protocol checkpoints and status records.

The original raw results remain at:
{PROJECT_RAW_ROOT}

Registry state:
PENDING SERIAL COMPLETION-REGISTRY INSERTION.

Do not rerun Project 9 experimental conditions.
"""

readme_path = (
    metadata_root /
    "README.txt"
)

readme_path.write_text(
    readme_text,
    encoding="utf-8",
)

summary_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "ExperimentComplete": True,
    "AnalysisReady": True,
    "DoNotRerun": True,
    "RegistryState": (
        "PENDING_SERIAL_COMPLETION_REGISTRY_INSERTION"
    ),
    "NoiseLevelsPercent": [
        0,
        5,
        10,
        15,
        20,
        25,
        30,
        40,
        50,
    ],
    "RepetitionSeeds": list(
        range(1, 31)
    ),
    "MLTechniques": [
        "RandomForest",
        "XGBoost",
        "LightGBM",
        "NaiveBayes",
    ],
    "Baselines": [
        "Random",
        "LatestFail",
        "QTF-Avg",
    ],
    "PrimaryMetric": "APFDc",
    "SecondaryMetric": "APFD",
    "Conditions": EXPECTED_CONDITIONS,
    "MLFits": EXPECTED_ML_FITS,
    "RankingRows": EXPECTED_RANKING_ROWS,
    "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS,
    "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS,
    "ConditionAuditRows": (
        EXPECTED_CONDITION_AUDIT_ROWS
    ),
    "TrainingMedianRows": (
        EXPECTED_TRAINING_MEDIAN_ROWS
    ),
    "ActivePredictors": (
        EXPECTED_ACTIVE_PREDICTORS
    ),
    "NoiseTechniqueSummaryRows": (
        EXPECTED_NOISE_TECHNIQUE_ROWS
    ),
    "SeedLevelNoiseDeltaRows": (
        EXPECTED_SEED_DELTA_ROWS
    ),
    "NoiseDeltaSummaryRows": (
        EXPECTED_NOISE_DELTA_ROWS
    ),
    "RawFiles": EXPECTED_RAW_FILES,
    "RawBytes": EXPECTED_RAW_BYTES,
    "RawRootSHA256": (
        EXPECTED_RAW_ROOT_SHA256
    ),
    "Step5ACheckpointSHA256": (
        EXPECTED_STEP5A_CHECKPOINT_SHA256
    ),
    "Step5BCheckpointSHA256": (
        EXPECTED_STEP5B_CHECKPOINT_SHA256
    ),
    "Step5BStatus": (
        EXPECTED_STEP5B_STATUS
    ),
}

summary_path = (
    metadata_root /
    "project_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

for generated_path in [
    readme_path,
    summary_path,
]:
    relative_generated_path = (
        generated_path.relative_to(
            STAGING_PACKAGE_ROOT
        )
    )

    package_records.append({
        "Category": "metadata",
        "RelativePath": (
            relative_generated_path.as_posix()
        ),
        "SourcePath": (
            "GENERATED_DETERMINISTIC_METADATA"
        ),
        "SizeBytes": int(
            generated_path.stat().st_size
        ),
        "SHA256": sha256_file(
            generated_path
        ),
        "Generated": True,
    })


# --------------------------------------------------------------------------------------------------
# 11. CREATE PACKAGE MANIFEST AND ROOT HASH
# --------------------------------------------------------------------------------------------------

package_manifest = pd.DataFrame(
    package_records
)

package_manifest = (
    package_manifest
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if package_manifest["RelativePath"].duplicated().any():
    raise RuntimeError(
        "Duplicate paths found in package manifest."
    )

package_payload_root_sha256 = (
    canonical_manifest_root_hash(
        package_manifest[
            [
                "RelativePath",
                "SizeBytes",
                "SHA256",
            ]
        ]
    )
)

package_payload_files = int(
    len(package_manifest)
)

package_payload_bytes = int(
    package_manifest[
        "SizeBytes"
    ].sum()
)

PACKAGE_MANIFEST_PATH_STAGING = (
    STAGING_PACKAGE_ROOT /
    "package_manifest.csv"
)

package_manifest.to_csv(
    PACKAGE_MANIFEST_PATH_STAGING,
    index=False,
)

package_manifest_sha256 = sha256_file(
    PACKAGE_MANIFEST_PATH_STAGING
)


# --------------------------------------------------------------------------------------------------
# 12. REPLACE ONLY THE PREVIOUS PROJECT 9 FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

if FINAL_PACKAGE_ROOT.exists():
    if (
        FINAL_PACKAGE_ROOT.parent.resolve()
        != PROJECT_AGGREGATED_ROOT.resolve()
        or FINAL_PACKAGE_ROOT.name
        != "camunda_final_package"
    ):
        raise RuntimeError(
            "Unsafe final-package removal blocked."
        )

    # This removes only a previously generated Project 9
    # compact package. It does not touch raw or audit data.
    shutil.rmtree(
        FINAL_PACKAGE_ROOT
    )

shutil.move(
    str(STAGING_PACKAGE_ROOT),
    str(FINAL_PACKAGE_ROOT),
)


# --------------------------------------------------------------------------------------------------
# 13. INDEPENDENT FINAL-PACKAGE READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

FINAL_MANIFEST_PATH = (
    FINAL_PACKAGE_ROOT /
    "package_manifest.csv"
)

final_manifest = pd.read_csv(
    FINAL_MANIFEST_PATH
)

final_payload_actual_paths = sorted(
    path.relative_to(
        FINAL_PACKAGE_ROOT
    ).as_posix()
    for path in (
        FINAL_PACKAGE_ROOT /
        "payload"
    ).rglob("*")
    if path.is_file()
)

final_payload_manifest_paths = sorted(
    final_manifest[
        "RelativePath"
    ].astype(str).tolist()
)

final_package_missing_files = sorted(
    set(final_payload_manifest_paths)
    - set(final_payload_actual_paths)
)

final_package_unexpected_files = sorted(
    set(final_payload_actual_paths)
    - set(final_payload_manifest_paths)
)

final_package_size_mismatches = []
final_package_hash_mismatches = []
final_readback_records = []

for row in final_manifest.itertuples(
    index=False
):
    final_file = (
        FINAL_PACKAGE_ROOT /
        row.RelativePath
    )

    if not final_file.is_file():
        continue

    actual_size = int(
        final_file.stat().st_size
    )

    actual_sha256 = sha256_file(
        final_file
    )

    if actual_size != int(row.SizeBytes):
        final_package_size_mismatches.append(
            row.RelativePath
        )

    if actual_sha256 != str(row.SHA256):
        final_package_hash_mismatches.append(
            row.RelativePath
        )

    final_readback_records.append({
        "RelativePath": row.RelativePath,
        "SizeBytes": actual_size,
        "SHA256": actual_sha256,
    })

final_readback_manifest = pd.DataFrame(
    final_readback_records
)

final_readback_root_sha256 = (
    canonical_manifest_root_hash(
        final_readback_manifest
    )
)

registry_sha256_after_package = sha256_file(
    COMPLETION_REGISTRY_PATH
)

registry_after_package = pd.read_csv(
    COMPLETION_REGISTRY_PATH
)

project9_registry_mask_after = pd.Series(
    False,
    index=registry_after_package.index,
)

registry_after_columns_lower = {
    str(column).lower(): column
    for column in registry_after_package.columns
}

if "projectnumber" in registry_after_columns_lower:
    column = registry_after_columns_lower[
        "projectnumber"
    ]

    project9_registry_mask_after |= (
        pd.to_numeric(
            registry_after_package[column],
            errors="coerce",
        )
        == PROJECT_NUMBER
    )

if "projectslug" in registry_after_columns_lower:
    column = registry_after_columns_lower[
        "projectslug"
    ]

    project9_registry_mask_after |= (
        registry_after_package[
            column
        ].astype(str)
        == PROJECT_SLUG
    )

if "project" in registry_after_columns_lower:
    column = registry_after_columns_lower[
        "project"
    ]

    project9_registry_mask_after |= (
        registry_after_package[
            column
        ].astype(str)
        == PROJECT_NAME
    )

project9_registry_rows_after = int(
    project9_registry_mask_after.sum()
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL VALIDATION TABLE
# --------------------------------------------------------------------------------------------------

validation_records = []

add_validation(
    validation_records,
    "Step 5B checkpoint status",
    EXPECTED_STEP5B_STATUS,
    step5b_checkpoint_status,
    (
        step5b_checkpoint_status
        == EXPECTED_STEP5B_STATUS
    ),
)

add_validation(
    validation_records,
    "Step 5B status-file status",
    EXPECTED_STEP5B_STATUS,
    step5b_status_file_status,
    (
        step5b_status_file_status
        == EXPECTED_STEP5B_STATUS
    ),
)

add_validation(
    validation_records,
    "Step 5B checkpoint SHA-256",
    EXPECTED_STEP5B_CHECKPOINT_SHA256,
    step5b_checkpoint_sha256,
    (
        step5b_checkpoint_sha256
        == EXPECTED_STEP5B_CHECKPOINT_SHA256
    ),
)

add_validation(
    validation_records,
    "Frozen raw files",
    EXPECTED_RAW_FILES,
    current_raw_file_count,
    current_raw_file_count
    == EXPECTED_RAW_FILES,
)

add_validation(
    validation_records,
    "Frozen raw bytes",
    EXPECTED_RAW_BYTES,
    current_raw_bytes,
    current_raw_bytes
    == EXPECTED_RAW_BYTES,
)

add_validation(
    validation_records,
    "Frozen raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA256,
    current_raw_root_sha256,
    (
        current_raw_root_sha256
        == EXPECTED_RAW_ROOT_SHA256
    ),
)

add_validation(
    validation_records,
    "Raw files missing",
    0,
    len(missing_raw_files),
    len(missing_raw_files) == 0,
)

add_validation(
    validation_records,
    "Unexpected raw files",
    0,
    len(unexpected_raw_files),
    len(unexpected_raw_files) == 0,
)

add_validation(
    validation_records,
    "Raw size mismatches",
    0,
    len(raw_size_mismatches),
    len(raw_size_mismatches) == 0,
)

add_validation(
    validation_records,
    "Raw SHA-256 mismatches",
    0,
    len(raw_hash_mismatches),
    len(raw_hash_mismatches) == 0,
)

add_validation(
    validation_records,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory)
    == EXPECTED_CONDITIONS,
)

add_validation(
    validation_records,
    "ML fits",
    EXPECTED_ML_FITS,
    len(model_fits_all),
    len(model_fits_all)
    == EXPECTED_ML_FITS,
)

add_validation(
    validation_records,
    "Build-metric rows",
    EXPECTED_BUILD_METRIC_ROWS,
    len(build_metrics_all),
    len(build_metrics_all)
    == EXPECTED_BUILD_METRIC_ROWS,
)

add_validation(
    validation_records,
    "Project-run rows",
    EXPECTED_PROJECT_RUN_ROWS,
    len(project_run_all),
    len(project_run_all)
    == EXPECTED_PROJECT_RUN_ROWS,
)

add_validation(
    validation_records,
    "Condition-audit rows",
    EXPECTED_CONDITION_AUDIT_ROWS,
    len(condition_audit_all),
    len(condition_audit_all)
    == EXPECTED_CONDITION_AUDIT_ROWS,
)

add_validation(
    validation_records,
    "Training-median rows",
    EXPECTED_TRAINING_MEDIAN_ROWS,
    len(training_medians_all),
    len(training_medians_all)
    == EXPECTED_TRAINING_MEDIAN_ROWS,
)

add_validation(
    validation_records,
    "Noise-technique summary rows",
    EXPECTED_NOISE_TECHNIQUE_ROWS,
    len(noise_technique_summary),
    len(noise_technique_summary)
    == EXPECTED_NOISE_TECHNIQUE_ROWS,
)

add_validation(
    validation_records,
    "Seed-level delta rows",
    EXPECTED_SEED_DELTA_ROWS,
    len(seed_level_noise_deltas),
    len(seed_level_noise_deltas)
    == EXPECTED_SEED_DELTA_ROWS,
)

add_validation(
    validation_records,
    "Noise-delta summary rows",
    EXPECTED_NOISE_DELTA_ROWS,
    len(noise_delta_summary),
    len(noise_delta_summary)
    == EXPECTED_NOISE_DELTA_ROWS,
)

add_validation(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_validation(
    validation_records,
    "Essential checkpoints missing",
    0,
    len(missing_essential_checkpoints),
    len(missing_essential_checkpoints)
    == 0,
)

add_validation(
    validation_records,
    "Required final-audit outputs missing",
    0,
    len(missing_required_final_audit),
    len(missing_required_final_audit)
    == 0,
)

add_validation(
    validation_records,
    "Duplicate package destinations",
    0,
    duplicate_destination_count,
    duplicate_destination_count == 0,
)

add_validation(
    validation_records,
    "Copy size mismatches",
    0,
    len(copy_size_mismatches),
    len(copy_size_mismatches) == 0,
)

add_validation(
    validation_records,
    "Copy SHA-256 mismatches",
    0,
    len(copy_hash_mismatches),
    len(copy_hash_mismatches) == 0,
)

add_validation(
    validation_records,
    "Final package payload files",
    package_payload_files,
    len(final_payload_actual_paths),
    (
        len(final_payload_actual_paths)
        == package_payload_files
    ),
)

add_validation(
    validation_records,
    "Final package missing files",
    0,
    len(final_package_missing_files),
    len(final_package_missing_files)
    == 0,
)

add_validation(
    validation_records,
    "Final package unexpected files",
    0,
    len(final_package_unexpected_files),
    len(final_package_unexpected_files)
    == 0,
)

add_validation(
    validation_records,
    "Final package size mismatches",
    0,
    len(final_package_size_mismatches),
    len(final_package_size_mismatches)
    == 0,
)

add_validation(
    validation_records,
    "Final package SHA-256 mismatches",
    0,
    len(final_package_hash_mismatches),
    len(final_package_hash_mismatches)
    == 0,
)

add_validation(
    validation_records,
    "Final package root SHA-256",
    package_payload_root_sha256,
    final_readback_root_sha256,
    (
        final_readback_root_sha256
        == package_payload_root_sha256
    ),
)

add_validation(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after_package,
    (
        registry_sha256_after_package
        == registry_sha256_before
    ),
)

add_validation(
    validation_records,
    "Registry Project 9 rows",
    0,
    project9_registry_rows_after,
    project9_registry_rows_after == 0,
)

add_validation(
    validation_records,
    "Registry update performed",
    False,
    False,
    True,
)

validation_frame = pd.DataFrame(
    validation_records
)

failed_checks = int(
    (~validation_frame["Pass"]).sum()
)

if failed_checks != 0:
    display(validation_frame)

    raise RuntimeError(
        f"Project 9 final-package validation failed "
        f"{failed_checks} check(s)."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE FINAL PACKAGE CONTROL FILES
# --------------------------------------------------------------------------------------------------

completed_at_utc = (
    datetime.now(timezone.utc)
    .isoformat()
)

PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_ROOT /
    "package_validation.csv"
)

PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_ROOT /
    "package_report.json"
)

PACKAGE_STATUS_PATH = (
    FINAL_PACKAGE_ROOT /
    "package_status.json"
)

validation_frame.to_csv(
    PACKAGE_VALIDATION_PATH,
    index=False,
)

package_report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": FINAL_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "ExperimentComplete": True,
    "AnalysisReady": True,
    "DoNotRerun": True,
    "RegistryUpdatePerformed": False,
    "RegistryState": (
        "PENDING_SERIAL_INSERTION"
    ),
    "Step5BStatus": (
        EXPECTED_STEP5B_STATUS
    ),
    "Step5BCheckpoint": str(
        STEP5B_CHECKPOINT_PATH
    ),
    "Step5BCheckpointSHA256": (
        step5b_checkpoint_sha256
    ),
    "RawResultDirectory": str(
        PROJECT_RAW_ROOT
    ),
    "RawFiles": current_raw_file_count,
    "RawBytes": current_raw_bytes,
    "RawRootSHA256": (
        current_raw_root_sha256
    ),
    "Conditions": len(
        condition_inventory
    ),
    "MLFits": len(
        model_fits_all
    ),
    "RankingRows": (
        EXPECTED_RANKING_ROWS
    ),
    "BuildMetricRows": len(
        build_metrics_all
    ),
    "ProjectRunRows": len(
        project_run_all
    ),
    "ConditionAuditRows": len(
        condition_audit_all
    ),
    "TrainingMedianRows": len(
        training_medians_all
    ),
    "NoiseTechniqueSummaryRows": len(
        noise_technique_summary
    ),
    "SeedLevelNoiseDeltaRows": len(
        seed_level_noise_deltas
    ),
    "NoiseDeltaSummaryRows": len(
        noise_delta_summary
    ),
    "FinalPackageDirectory": str(
        FINAL_PACKAGE_ROOT
    ),
    "FinalPackageManifest": str(
        FINAL_MANIFEST_PATH
    ),
    "FinalPackageManifestSHA256": (
        sha256_file(
            FINAL_MANIFEST_PATH
        )
    ),
    "FinalPackagePayloadFiles": (
        package_payload_files
    ),
    "FinalPackagePayloadBytes": (
        package_payload_bytes
    ),
    "FinalPackageRootSHA256": (
        final_readback_root_sha256
    ),
    "FinalPackageMissingFiles": (
        len(final_package_missing_files)
    ),
    "FinalPackageUnexpectedFiles": (
        len(final_package_unexpected_files)
    ),
    "FinalPackageSizeMismatches": (
        len(final_package_size_mismatches)
    ),
    "FinalPackageSHA256Mismatches": (
        len(final_package_hash_mismatches)
    ),
    "ValidationChecks": len(
        validation_frame
    ),
    "FailedChecks": failed_checks,
    "CompletionRegistry": str(
        COMPLETION_REGISTRY_PATH
    ),
    "CompletionRegistrySHA256Before": (
        registry_sha256_before
    ),
    "CompletionRegistrySHA256After": (
        registry_sha256_after_package
    ),
    "RegistryProject9Rows": (
        project9_registry_rows_after
    ),
}

atomic_write_json(
    PACKAGE_REPORT_PATH,
    package_report,
)

package_status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": FINAL_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "FinalPackageRootSHA256": (
        final_readback_root_sha256
    ),
    "RegistryUpdatePerformed": False,
    "RegistryPending": True,
    "DoNotRerun": True,
}

atomic_write_json(
    PACKAGE_STATUS_PATH,
    package_status_payload,
)

atomic_write_json(
    STEP5C_STATUS_PATH,
    package_status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. WRITE PROJECT 9 FINAL-PACKAGE CHECKPOINT
# --------------------------------------------------------------------------------------------------

final_package_checkpoint = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": FINAL_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "ExperimentStatus": (
        "COMPLETE_ANALYSIS_READY"
    ),
    "RegistryStatus": (
        "PENDING_SERIAL_INSERTION"
    ),
    "DoNotRerun": True,
    "Step5BCheckpoint": str(
        STEP5B_CHECKPOINT_PATH
    ),
    "Step5BCheckpointSHA256": (
        step5b_checkpoint_sha256
    ),
    "FrozenRawDirectory": str(
        PROJECT_RAW_ROOT
    ),
    "FrozenRawFiles": (
        current_raw_file_count
    ),
    "FrozenRawBytes": (
        current_raw_bytes
    ),
    "FrozenRawRootSHA256": (
        current_raw_root_sha256
    ),
    "FinalPackageDirectory": str(
        FINAL_PACKAGE_ROOT
    ),
    "FinalPackageManifest": str(
        FINAL_MANIFEST_PATH
    ),
    "FinalPackageManifestSHA256": (
        sha256_file(
            FINAL_MANIFEST_PATH
        )
    ),
    "FinalPackageValidation": str(
        PACKAGE_VALIDATION_PATH
    ),
    "FinalPackageReport": str(
        PACKAGE_REPORT_PATH
    ),
    "FinalPackageStatus": str(
        PACKAGE_STATUS_PATH
    ),
    "FinalPackagePayloadFiles": (
        package_payload_files
    ),
    "FinalPackagePayloadBytes": (
        package_payload_bytes
    ),
    "FinalPackageRootSHA256": (
        final_readback_root_sha256
    ),
    "CompletionRegistry": str(
        COMPLETION_REGISTRY_PATH
    ),
    "CompletionRegistrySHA256": (
        registry_sha256_after_package
    ),
    "RegistryProject9Rows": (
        project9_registry_rows_after
    ),
    "RegistryUpdatePerformed": False,
}

atomic_write_json(
    FINAL_PACKAGE_CHECKPOINT_PATH,
    final_package_checkpoint,
)

final_package_checkpoint_sha256 = (
    sha256_file(
        FINAL_PACKAGE_CHECKPOINT_PATH
    )
)


# Final registry check after every Project 9 write.

registry_sha256_final = sha256_file(
    COMPLETION_REGISTRY_PATH
)

if registry_sha256_final != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during "
        "Project 9 package construction."
    )


# --------------------------------------------------------------------------------------------------
# 17. FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\nProject 9 Step 5C validation:")
display(validation_frame)

print("\nFinal package manifest sample:")
display(
    package_manifest.head(20)
)

print("\n" + "=" * 118)
print("=== PROJECT 9 CELL 11 / STEP 5C RESULT ===")
print("=" * 118)

print("\nProject identity:")
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)

print("\nFrozen experimental result:")
print(
    "Conditions:",
    len(condition_inventory),
    "/",
    EXPECTED_CONDITIONS,
)
print(
    "ML fits:",
    len(model_fits_all),
    "/",
    EXPECTED_ML_FITS,
)
print(
    "Ranking rows:",
    EXPECTED_RANKING_ROWS,
)
print(
    "Build-metric rows:",
    len(build_metrics_all),
)
print(
    "Project-run rows:",
    len(project_run_all),
)

print("\nFrozen raw result:")
print(
    "Raw files:",
    current_raw_file_count,
)
print(
    "Raw bytes:",
    current_raw_bytes,
)
print(
    "Raw root SHA-256:",
    current_raw_root_sha256,
)

print("\nFinal compact package:")
print(
    "Package directory:",
    FINAL_PACKAGE_ROOT,
)
print(
    "Payload files:",
    package_payload_files,
)
print(
    "Payload bytes:",
    package_payload_bytes,
)
print(
    "Package root SHA-256:",
    final_readback_root_sha256,
)
print(
    "Missing package files:",
    len(final_package_missing_files),
)
print(
    "Unexpected package files:",
    len(final_package_unexpected_files),
)
print(
    "Package size mismatches:",
    len(final_package_size_mismatches),
)
print(
    "Package SHA-256 mismatches:",
    len(final_package_hash_mismatches),
)

print("\nCompletion registry:")
print(
    "Registry unchanged:",
    registry_sha256_final
    == registry_sha256_before,
)
print(
    "Registry Project 9 rows:",
    project9_registry_rows_after,
)
print(
    "Registry update performed:",
    False,
)

print("\nValidation:")
print(
    "Checks:",
    len(validation_frame),
)
print(
    "Failed checks:",
    failed_checks,
)

print("\nFinal package checkpoint:")
print(
    FINAL_PACKAGE_CHECKPOINT_PATH
)
print(
    "Checkpoint SHA-256:",
    final_package_checkpoint_sha256,
)

print("\nSTATUS:", FINAL_STATUS)
print("=" * 118)

=== PROJECT 9 CELL 11 / STEP 5C: FINAL PACKAGE CONSTRUCTION AND VALIDATION ===


FileNotFoundError: Required Project 9 checkpoints are missing:
project_09_model_protocol_checkpoint.json

In [17]:
# ==================================================================================================
# PROJECT 9 — CELL 11 / STEP 5C V2
# FINAL PACKAGE CONSTRUCTION, HASH VALIDATION, AND FREEZE
#
# PROJECT:
#   camunda@camunda-bpm-platform
#
# V2 FIX:
# - Does not require the nonexistent:
#       project_09_model_protocol_checkpoint.json
# - Discovers and packages all Project 9 checkpoint files that actually exist.
# - Requires only checkpoints already proven to exist:
#       project_09_selection_checkpoint.json
#       project_09_full_run_checkpoint.json
#       project_09_step5b_checkpoint.json
#
# SAFETY:
# - Does not rerun conditions, models, baselines, APFD, or APFDc.
# - Does not write inside the Project 9 raw-result directory.
# - Does not access or modify Project 10.
# - Does not modify Projects 1–8.
# - Does not modify the completion registry.
# - Registry insertion remains pending for a later serial cell.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import shutil

import pandas as pd


print("=" * 122)
print("=== PROJECT 9 CELL 11 / STEP 5C V2: FINAL PACKAGE CONSTRUCTION AND VALIDATION ===")
print("=" * 122)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT IDENTITY
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 9

PROJECT_NAME = (
    "camunda@camunda-bpm-platform"
)

PROJECT_SLUG = (
    "camunda__camunda-bpm-platform"
)


EXPECTED_STEP5B_STATUS = (
    "PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_"
    "AND_COMPACT_AGGREGATES_FROZEN"
)

FINAL_STATUS = (
    "PASS_PROJECT_9_FINAL_PACKAGE_CONSTRUCTED_"
    "AND_VALIDATED_REGISTRY_PENDING"
)


EXPECTED_STEP5A_CHECKPOINT_SHA256 = (
    "22f9f1184be247d83941199381f12b704"
    "2f743158403e765ee8ffcfe09636519"
)

EXPECTED_STEP5B_CHECKPOINT_SHA256 = (
    "a1fff6906ac62870689cbf4f3f38f16c"
    "dac04b10e6851190bcb34fbb8d6f8a14"
)

EXPECTED_RAW_ROOT_SHA256 = (
    "c31cb45e1354dc1222b82103bd275c21"
    "b10dd2ba6171125005d9c0a99ab14724"
)


EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_RANKING_ROWS = 34_970_670
EXPECTED_BUILD_METRIC_ROWS = 56_700
EXPECTED_PROJECT_RUN_ROWS = 1_890
EXPECTED_CONDITION_AUDIT_ROWS = 270
EXPECTED_TRAINING_MEDIAN_ROWS = 40_770
EXPECTED_ACTIVE_PREDICTORS = 151

EXPECTED_NOISE_TECHNIQUE_ROWS = 63
EXPECTED_SEED_DELTA_ROWS = 1_890
EXPECTED_NOISE_DELTA_ROWS = 63

EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 387_066_081


# --------------------------------------------------------------------------------------------------
# 2. PATHS — PROJECT 9 ONLY
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

RAW_RESULTS_ROOT = (
    RESULTS_ROOT
    / "Raw"
)

AGGREGATED_RESULTS_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
)


PROJECT_RAW_ROOT = (
    RAW_RESULTS_ROOT
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_ROOT = (
    AGGREGATED_RESULTS_ROOT
    / PROJECT_SLUG
)

PROJECT_SELECTION_ROOT = (
    AGGREGATED_RESULTS_ROOT
    / "project_09_selection"
)

STEP5B_FINAL_AUDIT_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / "camunda_step5b_final_audit"
)


SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_09_selection_checkpoint.json"
)

FULL_RUN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_09_full_run_checkpoint.json"
)

STEP5B_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_09_step5b_checkpoint.json"
)

FINAL_PACKAGE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_09_final_package_checkpoint.json"
)


STEP5B_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / "camunda_step5b_status.json"
)

STEP5C_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / "camunda_step5c_status.json"
)


COMPLETION_REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)


FINAL_PACKAGE_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / "camunda_final_package"
)

STAGING_PACKAGE_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / ".camunda_final_package_staging_v2"
)

STEP5C_AUDIT_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / "camunda_step5c_package_audit"
)

FINAL_PACKAGE_INVENTORY_PATH = (
    STEP5C_AUDIT_ROOT
    / "camunda_final_package_inventory.csv"
)

STEP5C_VALIDATION_PATH = (
    STEP5C_AUDIT_ROOT
    / "camunda_step5c_validation.csv"
)

STEP5C_REPORT_PATH = (
    STEP5C_AUDIT_ROOT
    / "camunda_step5c_report.json"
)


RAW_MANIFEST_PATH = (
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_raw_file_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_write_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def canonical_root_hash(
    manifest,
):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Manifest is missing required columns:\n"
            + "\n".join(
                missing_columns
            )
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def resolve_column(
    dataframe,
    candidates,
    label,
):
    lower_lookup = {
        str(column).lower():
            column
        for column in dataframe.columns
    }

    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

        candidate_lower = str(
            candidate
        ).lower()

        if candidate_lower in lower_lookup:
            return lower_lookup[
                candidate_lower
            ]

    raise RuntimeError(
        f"Could not resolve {label}.\n"
        f"Candidates: {candidates}\n"
        f"Columns: {dataframe.columns.tolist()}"
    )


def extract_status(
    payload,
):
    for key in [
        "Status",
        "status",
        "FinalStatus",
        "Step5BStatus",
    ]:
        if key in payload:
            return str(
                payload[
                    key
                ]
            )

    return None


def is_relative_to(
    path,
    parent,
):
    try:
        Path(
            path
        ).resolve().relative_to(
            Path(
                parent
            ).resolve()
        )

        return True

    except ValueError:
        return False


def safe_remove_directory(
    path,
    expected_parent,
    allowed_names=None,
    allowed_prefix=None,
):
    path = Path(
        path
    )

    if not path.exists():
        return

    if not path.is_dir():
        raise RuntimeError(
            f"Expected a directory:\n{path}"
        )

    if path.parent.resolve() != Path(
        expected_parent
    ).resolve():
        raise RuntimeError(
            "Unsafe directory removal blocked.\n"
            f"Path: {path}"
        )

    if (
        allowed_names is not None
        and path.name not in allowed_names
    ):
        raise RuntimeError(
            "Unsafe directory name blocked.\n"
            f"Path: {path}"
        )

    if (
        allowed_prefix is not None
        and not path.name.startswith(
            allowed_prefix
        )
    ):
        raise RuntimeError(
            "Unsafe directory prefix blocked.\n"
            f"Path: {path}"
        )

    shutil.rmtree(
        path
    )


def build_tree_manifest(
    root,
):
    root = Path(
        root
    )

    records = []

    for file_path in sorted(
        [
            path
            for path in root.rglob("*")
            if path.is_file()
        ],
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    ):
        records.append({
            "RelativePath":
                file_path.relative_to(
                    root
                ).as_posix(),

            "SizeBytes":
                int(
                    file_path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    file_path
                ),
        })

    return pd.DataFrame(
        records
    )


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_directories = [
    NOTES_ROOT,
    PROJECT_RAW_ROOT,
    PROJECT_AGGREGATED_ROOT,
    PROJECT_SELECTION_ROOT,
    STEP5B_FINAL_AUDIT_ROOT,
]

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(
            "Required Project 9 directory is missing:\n"
            f"{directory}"
        )


required_files = [
    SELECTION_CHECKPOINT_PATH,
    FULL_RUN_CHECKPOINT_PATH,
    STEP5B_CHECKPOINT_PATH,
    STEP5B_STATUS_PATH,
    COMPLETION_REGISTRY_PATH,
    RAW_MANIFEST_PATH,
]

for file_path in required_files:
    if not file_path.is_file():
        raise FileNotFoundError(
            "Required Project 9 file is missing:\n"
            f"{file_path}"
        )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 5A AND STEP 5B
# --------------------------------------------------------------------------------------------------

full_run_checkpoint_sha256 = sha256_file(
    FULL_RUN_CHECKPOINT_PATH
)

step5b_checkpoint_sha256 = sha256_file(
    STEP5B_CHECKPOINT_PATH
)


if (
    full_run_checkpoint_sha256
    != EXPECTED_STEP5A_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 9 Step 5A checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_STEP5A_CHECKPOINT_SHA256}\n"
        f"Actual:   {full_run_checkpoint_sha256}"
    )


if (
    step5b_checkpoint_sha256
    != EXPECTED_STEP5B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 9 Step 5B checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_STEP5B_CHECKPOINT_SHA256}\n"
        f"Actual:   {step5b_checkpoint_sha256}"
    )


step5b_checkpoint = load_json(
    STEP5B_CHECKPOINT_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)


step5b_checkpoint_status = extract_status(
    step5b_checkpoint
)

step5b_status = extract_status(
    step5b_status_payload
)


if (
    step5b_checkpoint_status
    != EXPECTED_STEP5B_STATUS
):
    raise RuntimeError(
        "Project 9 Step 5B checkpoint status differs.\n"
        f"Expected: {EXPECTED_STEP5B_STATUS}\n"
        f"Actual:   {step5b_checkpoint_status}"
    )


if (
    step5b_status
    != EXPECTED_STEP5B_STATUS
):
    raise RuntimeError(
        "Project 9 Step 5B status file differs.\n"
        f"Expected: {EXPECTED_STEP5B_STATUS}\n"
        f"Actual:   {step5b_status}"
    )


# --------------------------------------------------------------------------------------------------
# 6. REGISTRY SNAPSHOT — READ ONLY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    COMPLETION_REGISTRY_PATH
)

registry_before = (
    pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry_before,
    [
        "ProjectNumber",
        "project_number",
    ],
    "registry project-number column",
)


registry_project_numbers = pd.to_numeric(
    registry_before[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_before = int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)


if registry_project9_rows_before != 0:
    raise RuntimeError(
        "Project 9 is already present in the completion registry. "
        "This package-only cell must run before registry insertion."
    )


# --------------------------------------------------------------------------------------------------
# 7. REVALIDATE THE FROZEN RAW ROOT
# --------------------------------------------------------------------------------------------------

raw_manifest_source = pd.read_csv(
    RAW_MANIFEST_PATH,
    low_memory=False,
)


raw_relative_column = resolve_column(
    raw_manifest_source,
    [
        "RelativePath",
        "relative_path",
        "Path",
    ],
    "raw-manifest relative path",
)

raw_size_column = resolve_column(
    raw_manifest_source,
    [
        "FileSizeBytes",
        "SizeBytes",
        "size_bytes",
    ],
    "raw-manifest size",
)

raw_sha_column = resolve_column(
    raw_manifest_source,
    [
        "SHA256",
        "FileSHA256",
        "sha256",
    ],
    "raw-manifest SHA-256",
)


raw_manifest = pd.DataFrame({
    "RelativePath":
        raw_manifest_source[
            raw_relative_column
        ]
        .astype(str)
        .str.replace(
            "\\",
            "/",
            regex=False,
        ),

    "SizeBytes":
        pd.to_numeric(
            raw_manifest_source[
                raw_size_column
            ],
            errors="raise",
        ).astype("int64"),

    "ExpectedSHA256":
        raw_manifest_source[
            raw_sha_column
        ]
        .astype(str)
        .str.lower(),
})


raw_manifest = (
    raw_manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if raw_manifest[
    "RelativePath"
].duplicated().any():
    raise RuntimeError(
        "The frozen Project 9 raw manifest contains duplicate paths."
    )


actual_raw_paths = sorted(
    [
        path.relative_to(
            PROJECT_RAW_ROOT
        ).as_posix()
        for path in PROJECT_RAW_ROOT.rglob("*")
        if path.is_file()
    ]
)


manifest_raw_paths = raw_manifest[
    "RelativePath"
].tolist()


missing_raw_files = sorted(
    set(
        manifest_raw_paths
    )
    - set(
        actual_raw_paths
    )
)

unexpected_raw_files = sorted(
    set(
        actual_raw_paths
    )
    - set(
        manifest_raw_paths
    )
)


raw_size_mismatches = []
raw_hash_mismatches = []
current_raw_records = []


for row in raw_manifest.itertuples(
    index=False
):
    file_path = (
        PROJECT_RAW_ROOT
        / row.RelativePath
    )

    if not file_path.is_file():
        continue

    actual_size = int(
        file_path.stat().st_size
    )

    actual_sha256 = sha256_file(
        file_path
    )

    if actual_size != int(
        row.SizeBytes
    ):
        raw_size_mismatches.append(
            row.RelativePath
        )

    if actual_sha256 != str(
        row.ExpectedSHA256
    ):
        raw_hash_mismatches.append(
            row.RelativePath
        )

    current_raw_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


current_raw_manifest = pd.DataFrame(
    current_raw_records
)


current_raw_file_count = len(
    actual_raw_paths
)

current_raw_bytes = int(
    sum(
        (
            PROJECT_RAW_ROOT
            / relative_path
        ).stat().st_size
        for relative_path in actual_raw_paths
    )
)


current_raw_root_sha256 = canonical_root_hash(
    current_raw_manifest
)


if missing_raw_files:
    raise RuntimeError(
        f"Project 9 raw files are missing: "
        f"{len(missing_raw_files)}"
    )


if unexpected_raw_files:
    raise RuntimeError(
        f"Unexpected Project 9 raw files were found: "
        f"{len(unexpected_raw_files)}"
    )


if raw_size_mismatches:
    raise RuntimeError(
        f"Project 9 raw size mismatches: "
        f"{len(raw_size_mismatches)}"
    )


if raw_hash_mismatches:
    raise RuntimeError(
        f"Project 9 raw hash mismatches: "
        f"{len(raw_hash_mismatches)}"
    )


if current_raw_file_count != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Project 9 raw file count differs.\n"
        f"Expected: {EXPECTED_RAW_FILES}\n"
        f"Actual:   {current_raw_file_count}"
    )


if current_raw_bytes != EXPECTED_RAW_BYTES:
    raise RuntimeError(
        "Project 9 raw byte count differs.\n"
        f"Expected: {EXPECTED_RAW_BYTES}\n"
        f"Actual:   {current_raw_bytes}"
    )


if current_raw_root_sha256 != EXPECTED_RAW_ROOT_SHA256:
    raise RuntimeError(
        "Project 9 raw-root SHA-256 differs.\n"
        f"Expected: {EXPECTED_RAW_ROOT_SHA256}\n"
        f"Actual:   {current_raw_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 8. VALIDATE STEP 5B ANALYSIS-READY OUTPUTS
# --------------------------------------------------------------------------------------------------

required_final_audit_files = {
    "camunda_raw_file_manifest.csv",
    "camunda_condition_inventory.csv",
    "camunda_project_run_all.csv",
    "camunda_build_metrics_all.parquet",
    "camunda_model_fits_all.csv",
    "camunda_condition_audit_all.csv",
    "camunda_training_medians_all.parquet",
    "camunda_noise_technique_summary.csv",
    "camunda_seed_level_noise_deltas.csv",
    "camunda_noise_delta_summary.csv",
    "camunda_clean_technique_summary.csv",
    "camunda_baseline_invariance_audit.csv",
    "camunda_step5b_validation.csv",
    "camunda_step5b_report.json",
}


actual_final_audit_files = {
    path.name
    for path in STEP5B_FINAL_AUDIT_ROOT.rglob("*")
    if path.is_file()
}


missing_final_audit_files = sorted(
    required_final_audit_files
    - actual_final_audit_files
)


if missing_final_audit_files:
    raise FileNotFoundError(
        "Required Project 9 Step 5B outputs are missing:\n"
        + "\n".join(
            missing_final_audit_files
        )
    )


condition_inventory = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_condition_inventory.csv",
    low_memory=False,
)

project_run_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_project_run_all.csv",
    low_memory=False,
)

build_metrics_all = pd.read_parquet(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_build_metrics_all.parquet"
)

model_fits_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_model_fits_all.csv",
    low_memory=False,
)

condition_audit_all = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_condition_audit_all.csv",
    low_memory=False,
)

training_medians_all = pd.read_parquet(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_training_medians_all.parquet"
)

noise_technique_summary = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_noise_technique_summary.csv",
    low_memory=False,
)

seed_level_deltas = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_seed_level_noise_deltas.csv",
    low_memory=False,
)

noise_delta_summary = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_noise_delta_summary.csv",
    low_memory=False,
)

baseline_invariance = pd.read_csv(
    STEP5B_FINAL_AUDIT_ROOT
    / "camunda_baseline_invariance_audit.csv",
    low_memory=False,
)


baseline_pass_column = resolve_column(
    baseline_invariance,
    [
        "Pass",
        "pass",
    ],
    "baseline-invariance pass column",
)


baseline_pass_values = (
    baseline_invariance[
        baseline_pass_column
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        {
            "true",
            "1",
            "yes",
        }
    )
)


baseline_invariance_failures = int(
    (
        ~baseline_pass_values
    ).sum()
)


analysis_count_checks = {
    "Conditions":
        (
            len(
                condition_inventory
            ),
            EXPECTED_CONDITIONS,
        ),

    "MLFits":
        (
            len(
                model_fits_all
            ),
            EXPECTED_ML_FITS,
        ),

    "BuildMetricRows":
        (
            len(
                build_metrics_all
            ),
            EXPECTED_BUILD_METRIC_ROWS,
        ),

    "ProjectRunRows":
        (
            len(
                project_run_all
            ),
            EXPECTED_PROJECT_RUN_ROWS,
        ),

    "ConditionAuditRows":
        (
            len(
                condition_audit_all
            ),
            EXPECTED_CONDITION_AUDIT_ROWS,
        ),

    "TrainingMedianRows":
        (
            len(
                training_medians_all
            ),
            EXPECTED_TRAINING_MEDIAN_ROWS,
        ),

    "NoiseTechniqueRows":
        (
            len(
                noise_technique_summary
            ),
            EXPECTED_NOISE_TECHNIQUE_ROWS,
        ),

    "SeedDeltaRows":
        (
            len(
                seed_level_deltas
            ),
            EXPECTED_SEED_DELTA_ROWS,
        ),

    "NoiseDeltaRows":
        (
            len(
                noise_delta_summary
            ),
            EXPECTED_NOISE_DELTA_ROWS,
        ),
}


failed_analysis_counts = [
    (
        name,
        actual,
        expected,
    )
    for name, (
        actual,
        expected,
    ) in analysis_count_checks.items()
    if actual != expected
]


if failed_analysis_counts:
    raise RuntimeError(
        "Project 9 analysis-ready output counts differ:\n"
        + "\n".join(
            (
                f"{name}: expected {expected}, "
                f"actual {actual}"
            )
            for name, actual, expected
            in failed_analysis_counts
        )
    )


if baseline_invariance_failures != 0:
    raise RuntimeError(
        "Project 9 baseline invariance audit contains failures."
    )


active_predictor_count = int(
    training_medians_all.groupby(
        "ConditionKey"
    ).size().iloc[0]
)


if active_predictor_count != EXPECTED_ACTIVE_PREDICTORS:
    raise RuntimeError(
        "Project 9 active predictor count differs.\n"
        f"Expected: {EXPECTED_ACTIVE_PREDICTORS}\n"
        f"Actual:   {active_predictor_count}"
    )


# --------------------------------------------------------------------------------------------------
# 9. DISCOVER THE CHECKPOINTS THAT ACTUALLY EXIST
# --------------------------------------------------------------------------------------------------

checkpoint_sources = sorted(
    [
        path
        for path in NOTES_ROOT.glob(
            "project_09_*checkpoint.json"
        )
        if (
            path.is_file()
            and path.name
            != FINAL_PACKAGE_CHECKPOINT_PATH.name
        )
    ],
    key=lambda path:
        path.name,
)


required_existing_checkpoint_names = {
    "project_09_selection_checkpoint.json",
    "project_09_full_run_checkpoint.json",
    "project_09_step5b_checkpoint.json",
}


actual_checkpoint_names = {
    path.name
    for path in checkpoint_sources
}


missing_required_checkpoints = sorted(
    required_existing_checkpoint_names
    - actual_checkpoint_names
)


if missing_required_checkpoints:
    raise FileNotFoundError(
        "Required Project 9 checkpoints are missing:\n"
        + "\n".join(
            missing_required_checkpoints
        )
    )


print("\nProject 9 checkpoints discovered:")

for checkpoint_path in checkpoint_sources:
    print(
        " -",
        checkpoint_path.name,
    )


print(
    "\nThe nonexistent "
    "'project_09_model_protocol_checkpoint.json' "
    "is not required."
)


# --------------------------------------------------------------------------------------------------
# 10. BUILD A CURATED PACKAGE SOURCE INVENTORY
# --------------------------------------------------------------------------------------------------

allowed_suffixes = {
    ".csv",
    ".json",
    ".parquet",
    ".txt",
    ".md",
    ".gz",
}


source_inventory = []


def register_source(
    category,
    source_path,
    destination_tail,
):
    source_path = Path(
        source_path
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "Package source is missing:\n"
            f"{source_path}"
        )

    source_inventory.append({
        "Category":
            category,

        "SourcePath":
            source_path,

        "DestinationTail":
            Path(
                destination_tail
            ),
    })


# All existing Project 9 checkpoints.
for checkpoint_path in checkpoint_sources:
    register_source(
        category="checkpoints",
        source_path=checkpoint_path,
        destination_tail=checkpoint_path.name,
    )


# Project 9 selection evidence.
for source_path in sorted(
    [
        path
        for path in PROJECT_SELECTION_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
        )
    ],
    key=lambda path:
        path.relative_to(
            PROJECT_SELECTION_ROOT
        ).as_posix(),
):
    register_source(
        category="selection",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            PROJECT_SELECTION_ROOT
        ),
    )


# Complete Step 5B final audit.
for source_path in sorted(
    [
        path
        for path in STEP5B_FINAL_AUDIT_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
        )
    ],
    key=lambda path:
        path.relative_to(
            STEP5B_FINAL_AUDIT_ROOT
        ).as_posix(),
):
    register_source(
        category="final_audit",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            STEP5B_FINAL_AUDIT_ROOT
        ),
    )


# Useful earlier Project 9 evidence.
# Large smoke-ranking data is intentionally excluded.
evidence_keywords = (
    "status",
    "report",
    "validation",
    "manifest",
    "audit",
    "summary",
    "configuration",
    "protocol",
    "schema",
    "profile",
)


for source_path in sorted(
    [
        path
        for path in PROJECT_AGGREGATED_ROOT.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in allowed_suffixes
            and any(
                keyword in path.name.lower()
                for keyword in evidence_keywords
            )
            and not is_relative_to(
                path,
                STEP5B_FINAL_AUDIT_ROOT,
            )
            and not is_relative_to(
                path,
                FINAL_PACKAGE_ROOT,
            )
            and not is_relative_to(
                path,
                STAGING_PACKAGE_ROOT,
            )
            and not is_relative_to(
                path,
                STEP5C_AUDIT_ROOT,
            )
            and path != STEP5C_STATUS_PATH
        )
    ],
    key=lambda path:
        path.relative_to(
            PROJECT_AGGREGATED_ROOT
        ).as_posix(),
):
    register_source(
        category="evidence",
        source_path=source_path,
        destination_tail=source_path.relative_to(
            PROJECT_AGGREGATED_ROOT
        ),
    )


destination_paths = [
    (
        Path(
            "payload"
        )
        / item[
            "Category"
        ]
        / item[
            "DestinationTail"
        ]
    ).as_posix()
    for item in source_inventory
]


duplicate_destination_count = int(
    pd.Series(
        destination_paths
    ).duplicated().sum()
)


if duplicate_destination_count != 0:
    raise RuntimeError(
        "The package source inventory contains duplicate destinations."
    )


# --------------------------------------------------------------------------------------------------
# 11. CREATE A CLEAN STAGING PACKAGE
# --------------------------------------------------------------------------------------------------

safe_remove_directory(
    STAGING_PACKAGE_ROOT,
    expected_parent=PROJECT_AGGREGATED_ROOT,
    allowed_names={
        ".camunda_final_package_staging_v2",
    },
)


STAGING_PACKAGE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


package_records = []
copy_size_mismatches = []
copy_hash_mismatches = []


for item in source_inventory:
    source_path = item[
        "SourcePath"
    ]

    relative_destination = (
        Path(
            "payload"
        )
        / item[
            "Category"
        ]
        / item[
            "DestinationTail"
        ]
    )

    destination_path = (
        STAGING_PACKAGE_ROOT
        / relative_destination
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    source_size = int(
        source_path.stat().st_size
    )

    source_sha256 = sha256_file(
        source_path
    )


    shutil.copy2(
        source_path,
        destination_path,
    )


    copied_size = int(
        destination_path.stat().st_size
    )

    copied_sha256 = sha256_file(
        destination_path
    )


    if copied_size != source_size:
        copy_size_mismatches.append(
            relative_destination.as_posix()
        )


    if copied_sha256 != source_sha256:
        copy_hash_mismatches.append(
            relative_destination.as_posix()
        )


    package_records.append({
        "Category":
            item[
                "Category"
            ],

        "RelativePath":
            relative_destination.as_posix(),

        "SourcePath":
            str(
                source_path
            ),

        "SizeBytes":
            source_size,

        "SHA256":
            source_sha256,

        "Generated":
            False,
    })


if copy_size_mismatches:
    raise RuntimeError(
        "Package-copy size mismatches occurred:\n"
        + "\n".join(
            copy_size_mismatches
        )
    )


if copy_hash_mismatches:
    raise RuntimeError(
        "Package-copy SHA-256 mismatches occurred:\n"
        + "\n".join(
            copy_hash_mismatches
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. ADD PACKAGE METADATA
# --------------------------------------------------------------------------------------------------

metadata_root = (
    STAGING_PACKAGE_ROOT
    / "payload"
    / "metadata"
)

metadata_root.mkdir(
    parents=True,
    exist_ok=True,
)


readme_text = f"""PROJECT 9 FINAL EXPERIMENT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}

Experiment:
- Conditions: {EXPECTED_CONDITIONS}
- Noise levels: 0, 5, 10, 15, 20, 25, 30, 40, 50 percent
- Repetition seeds: 1 through 30
- ML fits: {EXPECTED_ML_FITS}
- ML models: RandomForest, XGBoost, LightGBM, NaiveBayes
- Baselines: Random, LatestFail, QTF-Avg
- Primary metric: APFDc
- Secondary metric: APFD
- Ranking rows: {EXPECTED_RANKING_ROWS}
- Build-metric rows: {EXPECTED_BUILD_METRIC_ROWS}
- Project-run rows: {EXPECTED_PROJECT_RUN_ROWS}

Frozen raw result:
- Files: {EXPECTED_RAW_FILES}
- Bytes: {EXPECTED_RAW_BYTES}
- Root SHA-256: {EXPECTED_RAW_ROOT_SHA256}

The raw condition results remain at:
{PROJECT_RAW_ROOT}

This compact package contains:
- existing Project 9 checkpoints
- candidate-selection evidence
- protocol and validation evidence
- compact analysis-ready Step 5B aggregates
- frozen raw-file manifest
- package metadata and validation

Registry state:
PENDING SERIAL COMPLETION-REGISTRY INSERTION.

Do not rerun Project 9.
"""


readme_path = (
    metadata_root
    / "README.txt"
)

readme_path.write_text(
    readme_text,
    encoding="utf-8",
)


project_summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ExperimentComplete":
        True,

    "AnalysisReady":
        True,

    "DoNotRerun":
        True,

    "RegistryState":
        "PENDING_SERIAL_INSERTION",

    "Conditions":
        EXPECTED_CONDITIONS,

    "NoiseLevelsPercent":
        [
            0,
            5,
            10,
            15,
            20,
            25,
            30,
            40,
            50,
        ],

    "RepetitionSeeds":
        list(
            range(
                1,
                31,
            )
        ),

    "MLTechniques":
        [
            "RandomForest",
            "XGBoost",
            "LightGBM",
            "NaiveBayes",
        ],

    "Baselines":
        [
            "Random",
            "LatestFail",
            "QTF-Avg",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "ProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "ActivePredictors":
        EXPECTED_ACTIVE_PREDICTORS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "Step5ACheckpointSHA256":
        EXPECTED_STEP5A_CHECKPOINT_SHA256,

    "Step5BCheckpointSHA256":
        EXPECTED_STEP5B_CHECKPOINT_SHA256,

    "Step5BStatus":
        EXPECTED_STEP5B_STATUS,

    "DiscoveredProject9Checkpoints":
        [
            path.name
            for path in checkpoint_sources
        ],

    "NonexistentCheckpointNotRequired":
        "project_09_model_protocol_checkpoint.json",
}


project_summary_path = (
    metadata_root
    / "project_summary.json"
)

project_summary_path.write_text(
    json.dumps(
        project_summary_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


for generated_path in [
    readme_path,
    project_summary_path,
]:
    relative_path = generated_path.relative_to(
        STAGING_PACKAGE_ROOT
    )

    package_records.append({
        "Category":
            "metadata",

        "RelativePath":
            relative_path.as_posix(),

        "SourcePath":
            "GENERATED_PACKAGE_METADATA",

        "SizeBytes":
            int(
                generated_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                generated_path
            ),

        "Generated":
            True,
    })


# --------------------------------------------------------------------------------------------------
# 13. FREEZE THE PAYLOAD MANIFEST
# --------------------------------------------------------------------------------------------------

package_manifest = pd.DataFrame(
    package_records
)


package_manifest = (
    package_manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if package_manifest[
    "RelativePath"
].duplicated().any():
    raise RuntimeError(
        "The package manifest contains duplicate payload paths."
    )


package_payload_root_sha256 = canonical_root_hash(
    package_manifest[
        [
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
)


package_payload_files = len(
    package_manifest
)

package_payload_bytes = int(
    package_manifest[
        "SizeBytes"
    ].sum()
)


staging_manifest_path = (
    STAGING_PACKAGE_ROOT
    / "package_manifest.csv"
)


package_manifest.to_csv(
    staging_manifest_path,
    index=False,
)


package_manifest_sha256 = sha256_file(
    staging_manifest_path
)


# --------------------------------------------------------------------------------------------------
# 14. REPLACE ONLY THE PROJECT 9 FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

safe_remove_directory(
    FINAL_PACKAGE_ROOT,
    expected_parent=PROJECT_AGGREGATED_ROOT,
    allowed_names={
        "camunda_final_package",
    },
)


shutil.move(
    str(
        STAGING_PACKAGE_ROOT
    ),
    str(
        FINAL_PACKAGE_ROOT
    ),
)


FINAL_PACKAGE_MANIFEST_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 15. VALIDATE THE COPIED PAYLOAD
# --------------------------------------------------------------------------------------------------

final_manifest = pd.read_csv(
    FINAL_PACKAGE_MANIFEST_PATH,
    low_memory=False,
)


payload_actual_paths = sorted(
    [
        path.relative_to(
            FINAL_PACKAGE_ROOT
        ).as_posix()
        for path in (
            FINAL_PACKAGE_ROOT
            / "payload"
        ).rglob("*")
        if path.is_file()
    ]
)


payload_manifest_paths = sorted(
    final_manifest[
        "RelativePath"
    ].astype(str).tolist()
)


missing_package_files = sorted(
    set(
        payload_manifest_paths
    )
    - set(
        payload_actual_paths
    )
)

unexpected_package_files = sorted(
    set(
        payload_actual_paths
    )
    - set(
        payload_manifest_paths
    )
)


package_size_mismatches = []
package_hash_mismatches = []
payload_readback_records = []


for row in final_manifest.itertuples(
    index=False
):
    file_path = (
        FINAL_PACKAGE_ROOT
        / row.RelativePath
    )

    if not file_path.is_file():
        continue

    actual_size = int(
        file_path.stat().st_size
    )

    actual_sha256 = sha256_file(
        file_path
    )


    if actual_size != int(
        row.SizeBytes
    ):
        package_size_mismatches.append(
            row.RelativePath
        )


    if actual_sha256 != str(
        row.SHA256
    ):
        package_hash_mismatches.append(
            row.RelativePath
        )


    payload_readback_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            actual_size,

        "SHA256":
            actual_sha256,
    })


payload_readback_manifest = pd.DataFrame(
    payload_readback_records
)


payload_readback_root_sha256 = canonical_root_hash(
    payload_readback_manifest
)


# --------------------------------------------------------------------------------------------------
# 16. VALIDATION BEFORE WRITING PACKAGE CONTROL FILES
# --------------------------------------------------------------------------------------------------

registry_sha256_after_payload = sha256_file(
    COMPLETION_REGISTRY_PATH
)


registry_after_payload = (
    pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_after_project_numbers = pd.to_numeric(
    registry_after_payload[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_after_payload = int(
    registry_after_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)


validation_records = []


add_check(
    validation_records,
    "Step 5B checkpoint status",
    EXPECTED_STEP5B_STATUS,
    step5b_checkpoint_status,
    (
        step5b_checkpoint_status
        == EXPECTED_STEP5B_STATUS
    ),
)

add_check(
    validation_records,
    "Step 5B status-file status",
    EXPECTED_STEP5B_STATUS,
    step5b_status,
    (
        step5b_status
        == EXPECTED_STEP5B_STATUS
    ),
)

add_check(
    validation_records,
    "Step 5A checkpoint SHA-256",
    EXPECTED_STEP5A_CHECKPOINT_SHA256,
    full_run_checkpoint_sha256,
    (
        full_run_checkpoint_sha256
        == EXPECTED_STEP5A_CHECKPOINT_SHA256
    ),
)

add_check(
    validation_records,
    "Step 5B checkpoint SHA-256",
    EXPECTED_STEP5B_CHECKPOINT_SHA256,
    step5b_checkpoint_sha256,
    (
        step5b_checkpoint_sha256
        == EXPECTED_STEP5B_CHECKPOINT_SHA256
    ),
)

add_check(
    validation_records,
    "Required existing checkpoints missing",
    0,
    len(
        missing_required_checkpoints
    ),
    len(
        missing_required_checkpoints
    ) == 0,
)

add_check(
    validation_records,
    "Discovered Project 9 checkpoints",
    ">= 3",
    len(
        checkpoint_sources
    ),
    len(
        checkpoint_sources
    ) >= 3,
)

add_check(
    validation_records,
    "Frozen raw files",
    EXPECTED_RAW_FILES,
    current_raw_file_count,
    current_raw_file_count
    == EXPECTED_RAW_FILES,
)

add_check(
    validation_records,
    "Frozen raw bytes",
    EXPECTED_RAW_BYTES,
    current_raw_bytes,
    current_raw_bytes
    == EXPECTED_RAW_BYTES,
)

add_check(
    validation_records,
    "Frozen raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA256,
    current_raw_root_sha256,
    (
        current_raw_root_sha256
        == EXPECTED_RAW_ROOT_SHA256
    ),
)

add_check(
    validation_records,
    "Raw files missing",
    0,
    len(
        missing_raw_files
    ),
    len(
        missing_raw_files
    ) == 0,
)

add_check(
    validation_records,
    "Unexpected raw files",
    0,
    len(
        unexpected_raw_files
    ),
    len(
        unexpected_raw_files
    ) == 0,
)

add_check(
    validation_records,
    "Raw size mismatches",
    0,
    len(
        raw_size_mismatches
    ),
    len(
        raw_size_mismatches
    ) == 0,
)

add_check(
    validation_records,
    "Raw SHA-256 mismatches",
    0,
    len(
        raw_hash_mismatches
    ),
    len(
        raw_hash_mismatches
    ) == 0,
)

add_check(
    validation_records,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(
        condition_inventory
    ),
    len(
        condition_inventory
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "ML fits",
    EXPECTED_ML_FITS,
    len(
        model_fits_all
    ),
    len(
        model_fits_all
    ) == EXPECTED_ML_FITS,
)

add_check(
    validation_records,
    "Build-metric rows",
    EXPECTED_BUILD_METRIC_ROWS,
    len(
        build_metrics_all
    ),
    len(
        build_metrics_all
    ) == EXPECTED_BUILD_METRIC_ROWS,
)

add_check(
    validation_records,
    "Project-run rows",
    EXPECTED_PROJECT_RUN_ROWS,
    len(
        project_run_all
    ),
    len(
        project_run_all
    ) == EXPECTED_PROJECT_RUN_ROWS,
)

add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_CONDITION_AUDIT_ROWS,
    len(
        condition_audit_all
    ),
    len(
        condition_audit_all
    ) == EXPECTED_CONDITION_AUDIT_ROWS,
)

add_check(
    validation_records,
    "Training-median rows",
    EXPECTED_TRAINING_MEDIAN_ROWS,
    len(
        training_medians_all
    ),
    len(
        training_medians_all
    ) == EXPECTED_TRAINING_MEDIAN_ROWS,
)

add_check(
    validation_records,
    "Active predictors",
    EXPECTED_ACTIVE_PREDICTORS,
    active_predictor_count,
    active_predictor_count
    == EXPECTED_ACTIVE_PREDICTORS,
)

add_check(
    validation_records,
    "Noise-technique summary rows",
    EXPECTED_NOISE_TECHNIQUE_ROWS,
    len(
        noise_technique_summary
    ),
    len(
        noise_technique_summary
    ) == EXPECTED_NOISE_TECHNIQUE_ROWS,
)

add_check(
    validation_records,
    "Seed-level delta rows",
    EXPECTED_SEED_DELTA_ROWS,
    len(
        seed_level_deltas
    ),
    len(
        seed_level_deltas
    ) == EXPECTED_SEED_DELTA_ROWS,
)

add_check(
    validation_records,
    "Noise-delta rows",
    EXPECTED_NOISE_DELTA_ROWS,
    len(
        noise_delta_summary
    ),
    len(
        noise_delta_summary
    ) == EXPECTED_NOISE_DELTA_ROWS,
)

add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_check(
    validation_records,
    "Duplicate package destinations",
    0,
    duplicate_destination_count,
    duplicate_destination_count == 0,
)

add_check(
    validation_records,
    "Package-copy size mismatches",
    0,
    len(
        copy_size_mismatches
    ),
    len(
        copy_size_mismatches
    ) == 0,
)

add_check(
    validation_records,
    "Package-copy SHA-256 mismatches",
    0,
    len(
        copy_hash_mismatches
    ),
    len(
        copy_hash_mismatches
    ) == 0,
)

add_check(
    validation_records,
    "Payload files",
    package_payload_files,
    len(
        payload_actual_paths
    ),
    len(
        payload_actual_paths
    ) == package_payload_files,
)

add_check(
    validation_records,
    "Missing payload files",
    0,
    len(
        missing_package_files
    ),
    len(
        missing_package_files
    ) == 0,
)

add_check(
    validation_records,
    "Unexpected payload files",
    0,
    len(
        unexpected_package_files
    ),
    len(
        unexpected_package_files
    ) == 0,
)

add_check(
    validation_records,
    "Payload size mismatches",
    0,
    len(
        package_size_mismatches
    ),
    len(
        package_size_mismatches
    ) == 0,
)

add_check(
    validation_records,
    "Payload SHA-256 mismatches",
    0,
    len(
        package_hash_mismatches
    ),
    len(
        package_hash_mismatches
    ) == 0,
)

add_check(
    validation_records,
    "Payload root SHA-256",
    package_payload_root_sha256,
    payload_readback_root_sha256,
    (
        payload_readback_root_sha256
        == package_payload_root_sha256
    ),
)

add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after_payload,
    (
        registry_sha256_after_payload
        == registry_sha256_before
    ),
)

add_check(
    validation_records,
    "Registry Project 9 rows",
    0,
    registry_project9_rows_after_payload,
    registry_project9_rows_after_payload == 0,
)

add_check(
    validation_records,
    "Registry update performed",
    False,
    False,
    True,
)

add_check(
    validation_records,
    "Project 10 accessed",
    False,
    False,
    True,
)

add_check(
    validation_records,
    "Project 10 write attempted",
    False,
    False,
    True,
)


package_validation = pd.DataFrame(
    validation_records
)


failed_checks = int(
    (
        ~package_validation[
            "Pass"
        ]
    ).sum()
)


if failed_checks != 0:
    print(
        "\nFailed package checks:"
    )

    display(
        package_validation[
            ~package_validation[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "PROJECT 9 STEP 5C V2 PACKAGE VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 17. WRITE CONTROL FILES INSIDE THE FINAL PACKAGE
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_validation.csv"
)

PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_report.json"
)

PACKAGE_STATUS_PATH = (
    FINAL_PACKAGE_ROOT
    / "package_status.json"
)


package_validation.to_csv(
    PACKAGE_VALIDATION_PATH,
    index=False,
)


package_report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ExperimentComplete":
        True,

    "AnalysisReady":
        True,

    "DoNotRerun":
        True,

    "RegistryUpdatePerformed":
        False,

    "RegistryState":
        "PENDING_SERIAL_INSERTION",

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "ProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "RawFiles":
        current_raw_file_count,

    "RawBytes":
        current_raw_bytes,

    "RawRootSHA256":
        current_raw_root_sha256,

    "DiscoveredCheckpointFiles":
        [
            path.name
            for path in checkpoint_sources
        ],

    "PayloadFiles":
        package_payload_files,

    "PayloadBytes":
        package_payload_bytes,

    "PayloadRootSHA256":
        payload_readback_root_sha256,

    "PackageManifest":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageValidationChecks":
        len(
            package_validation
        ),

    "PackageValidationFailures":
        failed_checks,

    "CompletionRegistryModified":
        False,

    "Project10Accessed":
        False,

    "Project10WriteAttempted":
        False,
}


atomic_write_json(
    PACKAGE_REPORT_PATH,
    package_report_payload,
)


package_status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "DoNotRerun":
        True,

    "RegistryPending":
        True,

    "RegistryUpdatePerformed":
        False,

    "PayloadRootSHA256":
        payload_readback_root_sha256,
}


atomic_write_json(
    PACKAGE_STATUS_PATH,
    package_status_payload,
)


# --------------------------------------------------------------------------------------------------
# 18. FREEZE THE COMPLETE FINAL PACKAGE TREE
# --------------------------------------------------------------------------------------------------

final_package_inventory = build_tree_manifest(
    FINAL_PACKAGE_ROOT
)


final_package_files = len(
    final_package_inventory
)

final_package_bytes = int(
    final_package_inventory[
        "SizeBytes"
    ].sum()
)

final_package_root_sha256 = canonical_root_hash(
    final_package_inventory
)


STEP5C_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FINAL_PACKAGE_INVENTORY_PATH,
    final_package_inventory,
)


final_validation_records = validation_records.copy()


add_check(
    final_validation_records,
    "Complete final-package files",
    final_package_files,
    final_package_files,
    final_package_files > 0,
)

add_check(
    final_validation_records,
    "Complete final-package bytes",
    final_package_bytes,
    final_package_bytes,
    final_package_bytes > 0,
)

add_check(
    final_validation_records,
    "Final package root generated",
    True,
    bool(
        final_package_root_sha256
    ),
    len(
        final_package_root_sha256
    ) == 64,
)


final_validation = pd.DataFrame(
    final_validation_records
)


final_failed_checks = int(
    (
        ~final_validation[
            "Pass"
        ]
    ).sum()
)


if final_failed_checks != 0:
    raise RuntimeError(
        "PROJECT 9 STEP 5C V2 FINAL VALIDATION FAILED."
    )


atomic_write_csv(
    STEP5C_VALIDATION_PATH,
    final_validation,
)


# --------------------------------------------------------------------------------------------------
# 19. WRITE EXTERNAL REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

step5c_report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ExperimentStatus":
        "COMPLETE_ANALYSIS_READY",

    "RegistryStatus":
        "PENDING_SERIAL_INSERTION",

    "DoNotRerun":
        True,

    "Conditions":
        EXPECTED_CONDITIONS,

    "NoiseLevels":
        9,

    "RepetitionSeeds":
        30,

    "MLFits":
        EXPECTED_ML_FITS,

    "RankingRows":
        EXPECTED_RANKING_ROWS,

    "BuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "ProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "RawFiles":
        current_raw_file_count,

    "RawBytes":
        current_raw_bytes,

    "RawRootSHA256":
        current_raw_root_sha256,

    "DiscoveredCheckpointCount":
        len(
            checkpoint_sources
        ),

    "DiscoveredCheckpointFiles":
        [
            path.name
            for path in checkpoint_sources
        ],

    "NonexistentCheckpointRequired":
        False,

    "PayloadFiles":
        package_payload_files,

    "PayloadBytes":
        package_payload_bytes,

    "PayloadRootSHA256":
        payload_readback_root_sha256,

    "FinalPackageFiles":
        final_package_files,

    "FinalPackageBytes":
        final_package_bytes,

    "FinalPackageRootSHA256":
        final_package_root_sha256,

    "FinalPackageDirectory":
        str(
            FINAL_PACKAGE_ROOT
        ),

    "FinalPackageInventory":
        str(
            FINAL_PACKAGE_INVENTORY_PATH
        ),

    "FinalPackageInventorySHA256":
        sha256_file(
            FINAL_PACKAGE_INVENTORY_PATH
        ),

    "PackageManifest":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "ValidationChecks":
        len(
            final_validation
        ),

    "FailedValidationChecks":
        final_failed_checks,

    "CompletionRegistry":
        str(
            COMPLETION_REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegistryProject9Rows":
        0,

    "RegistryUpdatePerformed":
        False,

    "Project10Accessed":
        False,

    "Project10WriteAttempted":
        False,

    "Projects1To8Modified":
        False,
}


atomic_write_json(
    STEP5C_REPORT_PATH,
    step5c_report_payload,
)


final_package_checkpoint_payload = {
    **step5c_report_payload,

    "Step5ACheckpoint":
        str(
            FULL_RUN_CHECKPOINT_PATH
        ),

    "Step5ACheckpointSHA256":
        full_run_checkpoint_sha256,

    "Step5BCheckpoint":
        str(
            STEP5B_CHECKPOINT_PATH
        ),

    "Step5BCheckpointSHA256":
        step5b_checkpoint_sha256,

    "Step5CReport":
        str(
            STEP5C_REPORT_PATH
        ),

    "Step5CReportSHA256":
        sha256_file(
            STEP5C_REPORT_PATH
        ),

    "Step5CValidation":
        str(
            STEP5C_VALIDATION_PATH
        ),

    "Step5CValidationSHA256":
        sha256_file(
            STEP5C_VALIDATION_PATH
        ),
}


atomic_write_json(
    FINAL_PACKAGE_CHECKPOINT_PATH,
    final_package_checkpoint_payload,
)


step5c_status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        FINAL_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "FinalPackageFiles":
        final_package_files,

    "FinalPackageBytes":
        final_package_bytes,

    "FinalPackageRootSHA256":
        final_package_root_sha256,

    "Checkpoint":
        str(
            FINAL_PACKAGE_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            FINAL_PACKAGE_CHECKPOINT_PATH
        ),

    "RegistryUpdatePerformed":
        False,

    "RegistryPending":
        True,

    "DoNotRerun":
        True,

    "Project10Accessed":
        False,

    "Project10WriteAttempted":
        False,
}


atomic_write_json(
    STEP5C_STATUS_PATH,
    step5c_status_payload,
)


# --------------------------------------------------------------------------------------------------
# 20. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_final = sha256_file(
    COMPLETION_REGISTRY_PATH
)


if registry_sha256_final != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 9 Step 5C V2."
    )


registry_final = (
    pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_final_project_numbers = pd.to_numeric(
    registry_final[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


registry_project9_rows_final = int(
    registry_final_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)


if registry_project9_rows_final != 0:
    raise RuntimeError(
        "Project 9 was unexpectedly inserted into the registry."
    )


# Ensure the final package did not change after its root was calculated.
final_package_inventory_readback = build_tree_manifest(
    FINAL_PACKAGE_ROOT
)

final_package_root_sha256_readback = canonical_root_hash(
    final_package_inventory_readback
)


if (
    final_package_root_sha256_readback
    != final_package_root_sha256
):
    raise RuntimeError(
        "The Project 9 final package changed after freezing."
    )


# Recheck the frozen raw root after package construction.
final_raw_records = []


for row in raw_manifest.itertuples(
    index=False
):
    file_path = (
        PROJECT_RAW_ROOT
        / row.RelativePath
    )

    final_raw_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


final_raw_root_sha256 = canonical_root_hash(
    pd.DataFrame(
        final_raw_records
    )
)


if final_raw_root_sha256 != EXPECTED_RAW_ROOT_SHA256:
    raise RuntimeError(
        "The Project 9 raw root changed during package construction."
    )


checkpoint_readback = load_json(
    FINAL_PACKAGE_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP5C_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != FINAL_STATUS:
    raise RuntimeError(
        "Final package checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != FINAL_STATUS:
    raise RuntimeError(
        "Step 5C V2 status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 21. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nProject 9 Step 5C V2 validation:")

display(
    final_validation
)


print("\nDiscovered Project 9 checkpoints:")

display(
    pd.DataFrame({
        "CheckpointFile":
            [
                path.name
                for path in checkpoint_sources
            ],

        "SHA256":
            [
                sha256_file(
                    path
                )
                for path in checkpoint_sources
            ],
    })
)


print("\nFinal package inventory sample:")

display(
    pd.concat(
        [
            final_package_inventory.head(
                15
            ),
            final_package_inventory.tail(
                15
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 22. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 122)
print("=== PROJECT 9 CELL 11 / STEP 5C V2 RESULT ===")
print("=" * 122)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nFrozen experiment:")

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "Noise levels:",
    9,
)

print(
    "Repetition seeds:",
    30,
)

print(
    "ML fits:",
    EXPECTED_ML_FITS,
)

print(
    "Ranking rows:",
    EXPECTED_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    EXPECTED_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    EXPECTED_PROJECT_RUN_ROWS,
)


print("\nFrozen raw result:")

print(
    "Raw files:",
    current_raw_file_count,
)

print(
    "Raw bytes:",
    current_raw_bytes,
)

print(
    "Raw root SHA-256:",
    final_raw_root_sha256,
)


print("\nCheckpoint discovery:")

print(
    "Project 9 checkpoints discovered:",
    len(
        checkpoint_sources
    ),
)

print(
    "Missing required existing checkpoints:",
    len(
        missing_required_checkpoints
    ),
)

print(
    "Nonexistent model-protocol checkpoint required:",
    False,
)


print("\nFinal compact package:")

print(
    "Package directory:",
    FINAL_PACKAGE_ROOT,
)

print(
    "Payload files:",
    package_payload_files,
)

print(
    "Payload bytes:",
    package_payload_bytes,
)

print(
    "Payload root SHA-256:",
    payload_readback_root_sha256,
)

print(
    "Complete package files:",
    final_package_files,
)

print(
    "Complete package bytes:",
    final_package_bytes,
)

print(
    "Final package root SHA-256:",
    final_package_root_sha256,
)

print(
    "Missing package files:",
    len(
        missing_package_files
    ),
)

print(
    "Unexpected package files:",
    len(
        unexpected_package_files
    ),
)

print(
    "Package size mismatches:",
    len(
        package_size_mismatches
    ),
)

print(
    "Package SHA-256 mismatches:",
    len(
        package_hash_mismatches
    ),
)


print("\nCompletion registry:")

print(
    "Registry unchanged:",
    registry_sha256_final
    == registry_sha256_before,
)

print(
    "Registry Project 9 rows:",
    registry_project9_rows_final,
)

print(
    "Registry update performed:",
    False,
)


print("\nIsolation:")

print(
    "Project 10 accessed:",
    False,
)

print(
    "Project 10 write attempted:",
    False,
)

print(
    "Projects 1–8 modified:",
    0,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        final_validation
    ),
)

print(
    "Failed checks:",
    final_failed_checks,
)


print("\nFinal package checkpoint:")

print(
    FINAL_PACKAGE_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        FINAL_PACKAGE_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    FINAL_STATUS,
)

print("=" * 122)

=== PROJECT 9 CELL 11 / STEP 5C V2: FINAL PACKAGE CONSTRUCTION AND VALIDATION ===

Project 9 checkpoints discovered:
 - project_09_full_run_checkpoint.json
 - project_09_model_metric_checkpoint.json
 - project_09_noise_plan_checkpoint.json
 - project_09_noisy_rec_engine_checkpoint.json
 - project_09_rec_reconstruction_checkpoint.json
 - project_09_selection_checkpoint.json
 - project_09_smoke_test_checkpoint.json
 - project_09_step5b_checkpoint.json

The nonexistent 'project_09_model_protocol_checkpoint.json' is not required.

Project 9 Step 5C V2 validation:


,Check,Expected,Actual,Pass
0,Step 5B checkpoint status,PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_AND_COM...,PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_AND_COM...,True
1,Step 5B status-file status,PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_AND_COM...,PASS_PROJECT_9_RAW_RESULTS_REVALIDATED_AND_COM...,True
2,Step 5A checkpoint SHA-256,22f9f1184be247d83941199381f12b7042f743158403e7...,22f9f1184be247d83941199381f12b7042f743158403e7...,True
3,Step 5B checkpoint SHA-256,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...,True
4,Required existing checkpoints missing,0,0,True
5,Discovered Project 9 checkpoints,>= 3,8,True
6,Frozen raw files,2160,2160,True
7,Frozen raw bytes,387066081,387066081,True
8,Frozen raw-root SHA-256,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,c31cb45e1354dc1222b82103bd275c21b10dd2ba617112...,True
9,Raw files missing,0,0,True



Discovered Project 9 checkpoints:


,CheckpointFile,SHA256
0,project_09_full_run_checkpoint.json,22f9f1184be247d83941199381f12b7042f743158403e7...
1,project_09_model_metric_checkpoint.json,1f58312f9b144320368b54216523567d9e9a704e81c551...
2,project_09_noise_plan_checkpoint.json,afa13cd38195b0b9bed549cb7a8344edd246bebbc02f4e...
3,project_09_noisy_rec_engine_checkpoint.json,6207fb3ef2b97352dee60ea31717ad559a63ac29da52cf...
4,project_09_rec_reconstruction_checkpoint.json,c37c5024e5622093b950decf780477ba990c034c7f3bc1...
5,project_09_selection_checkpoint.json,3ef9e66e606772a9679cf727182389e419c9f8821512ba...
6,project_09_smoke_test_checkpoint.json,74edca5fad1b501ccd84dff10d37e5a31e67cba2beb9d3...
7,project_09_step5b_checkpoint.json,a1fff6906ac62870689cbf4f3f38f16cdac04b10e68511...



Final package inventory sample:


,RelativePath,SizeBytes,SHA256
0,package_manifest.csv,23952,80cf6ba4e3ce8d2394fda276c28cd8a8d357044628b389...
1,package_report.json,1645,56f7293ccb756954b49faf281422678315b382aa4ffbd5...
2,package_status.json,439,ea5ff6a251bdb66ecc62f5b2f9b9e5cfeeccfd4145ab10...
3,package_validation.csv,2249,95f0992322cc32dd93e742b0c6133a2ae35c719f788b40...
4,payload/checkpoints/project_09_full_run_checkp...,2034,22f9f1184be247d83941199381f12b7042f743158403e7...
5,payload/checkpoints/project_09_model_metric_ch...,3436,1f58312f9b144320368b54216523567d9e9a704e81c551...
6,payload/checkpoints/project_09_noise_plan_chec...,2767,afa13cd38195b0b9bed549cb7a8344edd246bebbc02f4e...
7,payload/checkpoints/project_09_noisy_rec_engin...,1997,6207fb3ef2b97352dee60ea31717ad559a63ac29da52cf...
8,payload/checkpoints/project_09_rec_reconstruct...,1872,c37c5024e5622093b950decf780477ba990c034c7f3bc1...
9,payload/checkpoints/project_09_selection_check...,7126,3ef9e66e606772a9679cf727182389e419c9f8821512ba...




=== PROJECT 9 CELL 11 / STEP 5C V2 RESULT ===

Project identity:
Project number: 9
Project: camunda@camunda-bpm-platform
Project slug: camunda__camunda-bpm-platform

Frozen experiment:
Conditions: 270
Noise levels: 9
Repetition seeds: 30
ML fits: 1080
Ranking rows: 34970670
Build-metric rows: 56700
Project-run rows: 1890

Frozen raw result:
Raw files: 2160
Raw bytes: 387066081
Raw root SHA-256: c31cb45e1354dc1222b82103bd275c21b10dd2ba6171125005d9c0a99ab14724

Checkpoint discovery:
Project 9 checkpoints discovered: 8
Missing required existing checkpoints: 0
Nonexistent model-protocol checkpoint required: False

Final compact package:
Package directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/camunda__camunda-bpm-platform/camunda_final_package
Payload files: 85
Payload bytes: 5206615
Payload root SHA-256: fbbf324ecd9d32210fe10d402074941da0227dc8cd0eb06455bd144aac8439b8
Complete package files: 89
Complete package bytes: 5234900
Final package root SHA-256: 600ec7a8b40